# How 3D movies work — Telugu, cinematic character edition

This is the **only video notebook upgraded to the new characters**. It uses your reviewed Arjun and Meera artwork, the existing Telugu dialogue and voices, cinema/lobby scenes, and measured speech timing.

Before Run all:
1. Prepare and approve both character atlases with the separate Character Studio workflow. Upload the complete `characters/arjun/cinematic-v2/` and `characters/meera/cinematic-v2/` folders to `MyDrive/ai-video/_assets/`. Reference pictures alone cannot animate the video.
2. Copy `colab/assets/environments/cinema/v1/` and `colab/assets/environments/lobby/v1/` into the same `_assets/environments/` hierarchy.
3. Accept the Indic Parler model terms at https://huggingface.co/ai4bharat/indic-parler-tts and add a read token named `HF_TOKEN` in **Colab Secrets**, with notebook access enabled.
4. Select an allocated **T4 GPU** in Runtime → Change runtime type. Check Settings, then Run all.

Output: `MyDrive/ai-video/three_d_friends_te_v3_cinematic/render/final.mp4`, subtitles, metadata and QC reports. The notebook saves a plan with the exact approved artwork hashes. If you replace artwork or change the story, choose a new `PROJECT` folder; an existing project cannot silently switch assets.

**Free-tier preset:** 1280×720 at 24 fps, one drawing worker, up to two speech attempts per line, a 120-second finished-video limit and a 60-minute local session budget. Parler uses GPU; small multilingual Whisper uses CPU int8; Pillow animation/text use CPU; FFmpeg tries NVENC and falls back to CPU. No image or video diffusion models run here. Colab quota, hardware availability and session duration vary; these limits do not guarantee completion in one session. Large model weights use local scratch by default.

If disconnected, reconnect and Run all; completed matching narration and render segments are reused. CPU-only rendering is available with `RENDER_CACHED_AUDIO_ONLY=True` once this project has all matching narration WAVs **and their JSON sidecars**. A missing/stale line stops instead of starting a large CPU speech model.

Read `docs/GOOGLE_COLAB.md` in the repository for the Drive folder tree and first-time artwork workflow. Inspect the rendered characters and listen to the Telugu audio before sharing. CPU integration tests do not certify reference fidelity or Colab CUDA compatibility.

### Current script revision: v4

The notebook now embeds `colab/stories/three_d_friends_te_v4/story.json` via its updated plan. Default output is `MyDrive/ai-video/three_d_friends_te_v4_cinematic/`. Only cinema/v1 is required; the earlier lobby instruction applies to v3. The new script stays in the cinema, uses original Telugu humor, and selects CPU-generated curiosity music. Review the new dialogue/audio before sharing; previous script approvals do not apply.

### Current script revision: v5 (retention edit)

The notebook now uses the v5 story and writes to `MyDrive/ai-video/three_d_friends_te_v5_cinematic/`. The main question opens the video; the eye and glasses diagrams remain visible through their explanations; the one-eye qualification follows the mechanism; a two-line joke closes the video. This supersedes the v4 output-path note above. Character packs, cinema background and local Parler requirements remain the same. Actual speech duration, naturalness and audience retention still need verification.


In [ ]:
# ==== 1. Settings (you can edit these) ====
NOTEBOOK_VERSION = "Telugu 3D cinematic characters — Colab Free preset v1"
print("Notebook version:", NOTEBOOK_VERSION)
PROJECT = "three_d_friends_te_v5_cinematic"
DRIVE_ROOT  = "/content/drive/MyDrive/ai-video"
VOICE       = "am_michael"                    # only used to test the voice model; each character's voice is in the plan
SPEED_SCALE = 1.0                            # multiplies every line's speed (0.95 = a bit slower overall)
WER_RETRY   = 0.15                           # re-make a line if the speech check finds >15% wrong words
ASR_MODEL   = "small.en"                     # speech-check / subtitle-timing model
EXPRESSIVE  = True                           # scene-aware delivery: each line gets a direction from the story
VOICE_PROFILE = "science_explainer"          # science_explainer | beginner_education | cinematic_documentary |
                                             # mystery_documentary | historical_narration | inspirational_story
VOICE_DIRECTOR = "rules"                     # "rules": plan tones + story roles (no model) | "llm": a local model
DIRECTOR_MODEL = "Qwen/Qwen3-4B-Instruct-2507"   # used only when VOICE_DIRECTOR = "llm"
VOICE_BACKEND = "kokoro"                     # "kokoro" (fast; follows pace + pauses only) | "qwen3-tts" (follows the
                                             # whole direction as an instruction; 1.7B, ~3 min setup; measure first
                                             # with VOICE_BENCH)
NARRATOR_QWEN_SPEAKER = "aiden"              # Qwen3-TTS narrator identity: "aiden" or "ryan" (native English)
EXPRESSIVE_MAX_INTENSITY = 0.5              # cap for instruction-following voices (measured: strong styles shifted
                                             # the narrator's timbre by ~5 semitones and changed a word)
QWEN_TTS_DTYPE = "float32"                   # T4 has no fast bf16; fp16 can be unstable on older GPUs (benchmark it)
VOICE_BENCH = False                          # True: neutral vs expressive narration A/B for each backend, then stop
MAX_VOICE_TAKES_NORMAL = 1                   # takes per ordinary line (a retry still happens on a technical failure)
MAX_VOICE_TAKES_HERO = 1
                                             # or Parler (Kokoro is deterministic: one take), both kept for A/B
ASR_MODEL_MULTI = "small"
SPEECH_RETRY_OTHER = 0.35                    # non-English: re-make a line if > 35 % of its LETTERS are heard wrong
                                             # (unverified starting point: Whisper's Telugu is weak; see the log values)
PARLER_SEC_PER_CHAR = (0.025, 0.25)          # Telugu voice: a take outside this length per letter is cut off/rambling
VIDEO_SPEED = "fast"
CLIP_STEPS  = 25                             # steps per clip in "fast"/"quality"
CACHE_THRESHOLD = 0.08                       # "fast" mode only
MAX_SLOWMO  = 1.4                            # a clip may be slowed down this much to fit a longer line; longer
                                             # lines get extra clips instead (heavy slow motion looks smeary)
SMOOTH_MOTION = "blend"                      # frame-rate change / slow motion: "blend" (soft, never warps),
                                             # "mci" (motion-interpolated: smoother but can warp shapes), "none"
SHARPEN     = 0.5                            # light sharpening after upscaling the small AI clips (0 = off)
FRAMES_ON_DEMAND = True                      # make only as many video frames as each shot needs (~1.6x faster)
CONTINUITY_CHECK = False
CONTINUITY_RETRIES = 1                       # if not, remake the clip with a new seed this many times (keeps the best)
# ---- logical consistency (shots with a scene-state "state" block in the plan) ----
VIDEO_RECIPE = "auto"                        # "auto": shots with a scene-state are generated with VACE layout control
                                             # (Wan2.1-VACE-1.3B: boxes from the plan steer positions and direction);
                                             # "t2v": plain text-to-video everywhere (no spatial control)
LOGIC_CHECK = False
                                             # check positions, direction, stops, hand-offs, order against the plan
VLM_CHECK = False
REPAIR_BUDGET = 2                            # targeted repairs per shot before it goes to review
VAE_TILING = "auto"                          # VACE's VAE: "auto" = untiled (faster), tiles only after out-of-memory;
                                             # True = always tiles (old behaviour, slowest, lowest memory)
VAE_ZERO_CACHE = True                        # reuse VACE's all-zero "inactive" latent (bit-identical; one VAE pass less)
BENCHMARK = False                            # True: time ONE scene-state shot under each BENCH_CONFIGS setting, write
                                             # render/benchmark.md, then stop (no video is made). Takes ~15–40 min.
BENCH_SHOT = "sh11"                          # the shot to benchmark (a VACE walking scene is representative)
BENCH_CONFIGS = ["optimized", "turbo_tiled", "baseline"]   # baseline = the old fast mode, 25 steps (~25 min alone)
MUSIC_STYLE = "curious"                      # "library": licensed tracks per chapter (credited in the description);
                                             # "curious" / "ambient": music generated in the notebook (offline fallback)
MUSIC_UNDER_VOICE_DB = 10                    # music bed this many dB under the voice (ducking adds ~6–10 dB while someone speaks)
BURN_SUBS   = True                           # burn coloured subtitles into the video (an .srt is saved either way)
SUB_STYLE   = "karaoke"                      # "karaoke": each word lights up as it is spoken (ASS); "plain": classic
MUSIC_GAIN  = 0.30                           # music level before ducking under the voices
DUCK = {"threshold": 0.02, "ratio": 8, "attack": 20, "release": 500}   # music ducking under the voice (sidechaincompress)
RUN_QC_STRICT = True                         # final report: approved only if EVERY mandatory check is PASS
SFX_GAIN_DB = 0.0                            # raise/lower all sound effects together
# ---- two-friend storytelling, licences and budgets (TARGET_DURATION = TARGET_SECONDS in the topic notebook;
#      MAX_REPAIR_ATTEMPTS = REPAIR_BUDGET above; RUN_MODE below) ----
STORY_MODE  = "two_friends_cinematic"               # "two_friends_cinematic": two original 2D friends talk it through |
                                             # "narrator": the classic narrated explainer (set by the plan)
VISUAL_STYLE = "cinematic_atlas"
LICENSE_STRICT = True          # True: refuse every model whose licence is marked "review" in the
                                             # registry (turbo LoRA → fast mode, SDXL → drawn stills, OpenAI CLIP check off)
LIP_SYNC    = "energy"                       # "energy": the mouth opens with the speaker's own loudness | "none": closed
MAX_AI_VIDEO_SHOTS = 0
MAX_GPU_MINUTES = 60
# ---- Telugu voices via Google Gemini FREE TIER (optional; the only hosted exception, for speech only) ----
STRICT_OPEN_SOURCE = True  # True: local open-source voices only (Indic Parler-TTS for Telugu);
                                             # Gemini (proprietary, even on the free tier) runs ONLY if you also set
                                             # TELUGU_TTS_BACKEND = "gemini_free" yourself — never via "auto"
TELUGU_TTS_BACKEND = "indic_parler"
GEMINI_TRANSCRIPT_FORM = "display"           # what Gemini reads: "display" (English words in English letters, as in the
                                             # subtitles) | "telugu_script" (respelled) — the listening benchmark decides
TELUGU_PRON_BENCH = False                    # True: one short exchange in every pronunciation form (Parler: respelled vs
                                             # Latin; Gemini, if enabled: display vs respelled) → blind listening sheet
GEMINI_TTS_MODEL = "gemini-3.8-flash-tts"    # or "gemini-3.8-flash-lite-tts" (only if listening says it is as good)
GEMINI_FREE_TIER_CONFIRMED = False           # True ONLY after AI Studio → Projects shows this key's project with
                                             # Billing Tier "Free" (no billing account). A key alone proves nothing.
GEMINI_FREE_TIER_ONLY = True                 # never a paid route (no Batch/Flex/Priority, no billed key)
GEMINI_MODE = "scene"                        # "scene": two-speaker conversation chunks | "turn": one request per line
MAX_VIDEOS_PER_DAY = 1                       # local safety budget (NOT Google's quota, which AI Studio shows)
MAX_VIDEO_DURATION_SECONDS = 120
MAX_GEMINI_TTS_CALLS_PER_VIDEO = 6
MAX_GEMINI_RETRIES_PER_SEGMENT = 1
TELUGU_VOICE_BENCH = False                   # True: Parler vs Gemini (scene / per-turn) on a short dialogue, blind A/B
TELUGU_VOICE_AUDITION = False                # True: Gemini voice candidates for Arjun and Meera on one Telugu exchange
TELUGU_BENCH_LITE = False                    # also try gemini-3.8-flash-lite-tts in the benchmark (1 more call)
MAX_GEMINI_BENCH_CALLS = 14                  # separate local budget for the benchmark / audition (one-off)
FPS, W, H = 24, 1280, 720
RUN_MODE = "draft"
                                             # "cpu_fallback": 1280x720, no AI video (drawn / licensed images only)
if RUN_MODE in ("draft", "cpu_fallback"):
    W, H = 1280, 720
X264_PRESET = "veryfast"                     # final encode on CPU (if no GPU encoder)
MODEL_CACHE_ON_DRIVE = False

# This target reuses prepared artwork. These controls apply only to the Telugu 3D video.
import time, os
SESSION_STARTED = time.monotonic()
CHARACTER_VERSIONS = {'arjun': 'cinematic-v2', 'meera': 'cinematic-v2'}
MAX_SESSION_MINUTES = 60  # local wall-time guard; Colab can disconnect sooner
CPU_THREADS = min(2, os.cpu_count() or 1)
RENDER_CACHED_AUDIO_ONLY = False  # True: never start a TTS model; require matching cached speech
MAX_TTS_ATTEMPTS = 2
ALLOW_CALMER_RETAKE = False


In [ ]:
# Check inputs before installing packages or downloading voice models.
import os, json, shutil, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ASSETS = Path(DRIVE_ROOT) / '_assets'
for cid, version in CHARACTER_VERSIONS.items():
    if not version or Path(version).name != version or version in ('.', '..'):
        raise ValueError('Character versions must be simple folder names.')
    manifest = ASSETS / 'characters' / cid / version / 'manifest.json'
    if not manifest.is_file():
        raise FileNotFoundError(f'Missing {manifest}. Prepare and approve the cinematic character pack first; '
                                'see docs/GOOGLE_COLAB.md. Reference pictures alone are not animation packs.')
    info = json.loads(manifest.read_text())
    if info.get('status') != 'approved' or info.get('asset_type') != 'atlas_v1':
        raise ValueError(f'{cid}/{version} is not an approved cinematic atlas.')
for env in ('cinema',):
    if not (ASSETS / 'environments' / env / 'v1/manifest.json').is_file():
        raise FileNotFoundError(f'Copy colab/assets/environments/{env}/v1 to {ASSETS}/environments/{env}/v1.')
if shutil.disk_usage('/content').free < 15 * 2**30:
    raise RuntimeError('Need at least 15 GiB free local scratch space before this run.')
import psutil, torch
if psutil.virtual_memory().available < 5 * 2**30:
    raise RuntimeError('Need at least 5 GiB available system RAM. Restart the runtime to release other models.')
if not RENDER_CACHED_AUDIO_ONLY:
    if not torch.cuda.is_available():
        raise RuntimeError('New Telugu speech needs an allocated GPU. Select T4 GPU; if unavailable, resume later. '
                           'Set RENDER_CACHED_AUDIO_ONLY=True only when this project already has matching narration.')
    if torch.cuda.get_device_properties(0).total_memory < 12 * 2**30:
        raise RuntimeError('This speech preset requires at least 12 GiB VRAM.')
# CUDA allocator choice must precede model loading; T4 uses float32 for Parler's T5 stability.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['OMP_NUM_THREADS'] = str(CPU_THREADS)
os.environ['MKL_NUM_THREADS'] = str(CPU_THREADS)
torch.set_num_threads(CPU_THREADS)
print('Inputs found. Full manifest/content verification follows before speech generation.')
print('CPU: animation, Telugu text, small int8 speech check. GPU: Parler speech and NVENC if available.')
print('Colab Free availability and session length vary; the local budget does not grant extra quota.')


In [ ]:
# ==== 1b. Run log ====
# Everything this notebook prints, every ffmpeg command, every error (with the full traceback), how long each
# cell took, and RAM/GPU use every 30 s go to a log file: MyDrive/ai-video/<project>/logs/run_<date>.log
# If something goes wrong, send that file to Claude.
import sys, os, time, threading, traceback, subprocess
from datetime import datetime

class RunLog:
    def __init__(self, path):
        self.lock = threading.Lock()
        self._open(path)
    def _open(self, path):
        self.path = str(path)
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        self.f = open(self.path, "a", encoding="utf-8", buffering=1)
    def write(self, text, level="INFO"):
        stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with self.lock:
            for line in str(text).splitlines() or [""]:
                self.f.write(f"{stamp} {level:<5} {line}\n")
            self.f.flush()
    def move_to(self, path):
        """Continue the log in a new place (local disk → Google Drive once it is mounted)."""
        with self.lock:
            self.f.close()
            old = open(self.path, encoding="utf-8").read()
            os.remove(self.path)
            self._open(path)
            self.f.write(old); self.f.flush()

class Tee:
    """Copies everything written to the screen into the log (progress bars: only their final state)."""
    _is_tee = True
    def __init__(self, stream, level):
        self.stream, self.level, self.buf = stream, level, ""
    def write(self, s):
        self.stream.write(s)
        self.buf += s
        if "\n" in self.buf:
            *lines, self.buf = self.buf.split("\n")
            for ln in lines:
                ln = ln.split("\r")[-1].rstrip()
                if ln:
                    LOG.write(ln, self.level)
        self.buf = self.buf[self.buf.rfind("\r") + 1:]
        return len(s)
    def flush(self):
        self.stream.flush()
    def __getattr__(self, k):
        return getattr(self.stream, k)

def log(msg, level="INFO"):
    """Write to the log file only (not the screen)."""
    LOG.write(msg, level)

def log_exc(context):
    """Full traceback of the exception being handled → log file; the screen only shows a short message."""
    LOG.write(f"{context}\n{traceback.format_exc()}", "ERROR")

def resources():
    msg = ""
    try:
        import psutil
        vm = psutil.virtual_memory()
        msg += f"RAM {(vm.total - vm.available) / 2**30:.1f}/{vm.total / 2**30:.1f} GB"
    except ImportError:
        pass
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                            "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10)
        used, tot, util = [float(x) for x in r.stdout.strip().splitlines()[0].split(",")]
        msg += f" | GPU {used / 1024:.1f}/{tot / 1024:.1f} GB, {util:.0f}% busy"
    except Exception:
        pass
    try:
        du = shutil.disk_usage("/content" if os.path.exists("/content") else "/")
        msg += f" | disk free {du.free / 2**30:.0f} GB"
    except Exception:
        pass
    return msg

import shutil
for name in ("stdout", "stderr"):                     # re-running this cell: unwrap the old copy first
    while getattr(getattr(sys, name), "_is_tee", False):
        setattr(sys, name, getattr(sys, name).stream)
if "LOG" not in globals():
    LOG = RunLog(f"/content/run_logs/run_{datetime.now():%Y%m%d_%H%M%S}.log" if os.path.exists("/content")
                 else os.path.join(os.environ.get("WORK_DIR", "."), "run_logs", f"run_{datetime.now():%Y%m%d_%H%M%S}.log"))
sys.stdout, sys.stderr = Tee(sys.stdout, "INFO"), Tee(sys.stderr, "WARN")

# RAM / GPU / disk every 30 s, to the log only (shows what was happening right before a crash)
if "_MONITOR_STOP" in globals():
    _MONITOR_STOP.set()
_MONITOR_STOP = threading.Event()
def _monitor(stop):
    """Every 30 s: resources + the highest system RAM use seen in between (sampled every 2 s, so short peaks that a
    30-second snapshot would miss are still logged)."""
    try:
        import psutil
    except ImportError:
        psutil = None
    peak, n = 0.0, 0
    while not stop.wait(2):
        if psutil is not None:
            vm = psutil.virtual_memory()
            peak = max(peak, (vm.total - vm.available) / 2**30)
            RUN_PEAKS["ram_gb"] = max(RUN_PEAKS["ram_gb"], round(peak, 2))
        n += 1
        if n % 5 == 0:                                   # every 10 s: GPU memory of ALL processes (voice workers too)
            try:
                r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                   capture_output=True, text=True, timeout=5)
                RUN_PEAKS["gpu_gb"] = max(RUN_PEAKS["gpu_gb"], round(float(r.stdout.split()[0]) / 1024, 2))
            except Exception:
                pass
        if n % 15 == 0:
            log(resources() + (f" | RAM peak {peak:.1f} GB in the last 30 s" if psutil is not None else ""), "RES")
            peak = 0.0
RUN_PEAKS = globals().get("RUN_PEAKS") or {"ram_gb": 0.0, "gpu_gb": 0.0}   # whole-run peaks, for the final report
DEVICE_REPORT = globals().get("DEVICE_REPORT") or {}                      # step → where it actually ran
threading.Thread(target=_monitor, args=(_MONITOR_STOP,), daemon=True).start()

# Each cell: start, end, how long it took, and the full error if it failed
try:
    _ip = get_ipython()
    for ev, fn in globals().get("_LOG_HOOKS", []):
        try:
            _ip.events.unregister(ev, fn)
        except ValueError:
            pass
    _CELL = {}
    def _pre(info):
        if not getattr(sys.stdout, "_is_tee", False):   # something replaced the screen output: re-attach
            sys.stdout = Tee(sys.stdout, "INFO")
        if not getattr(sys.stderr, "_is_tee", False):
            sys.stderr = Tee(sys.stderr, "WARN")
        title = next((l.strip("# =") for l in (info.raw_cell or "").splitlines() if l.strip()), "")[:80]
        _CELL.update(title=title, t0=time.time())
        log(f"──── CELL START: {title} | {resources()}", "CELL")
    def _post(result):
        dt = time.time() - _CELL.get("t0", time.time())
        err = result.error_in_exec or result.error_before_exec
        if err is not None:
            LOG.write(f"CELL FAILED after {dt:.1f}s: {_CELL.get('title')}\n"
                      + "".join(traceback.format_exception(type(err), err, err.__traceback__)), "ERROR")
        log(f"──── CELL END ({'error' if err else 'ok'}, {dt:.1f}s): {_CELL.get('title')} | {resources()}", "CELL")
    _LOG_HOOKS = [("pre_run_cell", _pre), ("post_run_cell", _post)]
    for ev, fn in _LOG_HOOKS:
        _ip.events.register(ev, fn)
except NameError:
    pass                                                # not inside a notebook

log(f"Run started | notebook {NOTEBOOK_VERSION} | Python {sys.version.split()[0]} | {resources()}")
log("Settings: " + ", ".join(f"{k}={globals()[k]!r}" for k in
    ["PROJECT", "VOICE", "SPEED_SCALE", "WER_RETRY", "ASR_MODEL", "CLIP_STEPS", "MAX_SLOWMO", "SMOOTH_MOTION", "SHARPEN", "CONTINUITY_CHECK", "CONTINUITY_RETRIES", "FRAMES_ON_DEMAND",
     "MUSIC_STYLE", "BURN_SUBS", "VIDEO_SPEED", "CACHE_THRESHOLD", "MODEL_CACHE_ON_DRIVE", "MUSIC_GAIN", "SFX_GAIN_DB", "FPS", "W", "H", "X264_PRESET"]))
print("Logging to:", LOG.path, "(moves to Google Drive in step 3)")


In [ ]:
# Install only dependencies used by this drawn Telugu target; keep Colab's matching torch/CUDA.
import subprocess, sys
packages = ['numpy>=1.26,<3', 'Pillow>=10,<14', 'soundfile>=0.12,<1',
            'faster-whisper>=1.1,<2', 'pydantic>=2.7,<3', 'scipy>=1.11,<2',
            'librosa>=0.10,<2', 'huggingface_hub>=0.34,<1', 'psutil>=5.9,<8',
            'imageio>=2.34,<3', 'imageio-ffmpeg>=0.5,<1', 'ftfy>=6,<7']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
# Indic Parler's isolated environment is installed only if uncached speech is needed.
# No diffusion, FLUX, Wan, Qwen, or hosted service is installed for rendering this video.
subprocess.run(['ffmpeg', '-version'], check=True, stdout=subprocess.DEVNULL)
from PIL import features
if not features.check('raqm'):
    raise RuntimeError('Pillow needs RAQM for Telugu text. Install libraqm0/libfribidi0 and restart the runtime.')


In [ ]:
# Persistent results on Drive; disposable scratch on Colab's local disk.
import os, json, subprocess, shutil, math, random, gc, time, re, difflib, hashlib
from pathlib import Path
PROJ = Path(DRIVE_ROOT) / PROJECT
WORK = Path(os.environ.get('WORK_DIR', '/content/work')) / PROJECT
for d in ('narration', 'keyframes', 'clips', 'render'):
    (PROJ / d).mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
# Keep large model weights on local scratch by default to protect free Drive space.
MODEL_CACHE = (Path(DRIVE_ROOT) if MODEL_CACHE_ON_DRIVE else WORK.parent) / '_model_cache'
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
if not LOG.path.startswith(str(PROJ)):
    LOG.move_to(PROJ / 'logs' / Path(LOG.path).name)
GPU = {'name': None, 'vram_gb': 0.0, 'bf16': False}
import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    GPU = {'name': p.name, 'vram_gb': round(p.total_memory / 2**30, 1),
           'bf16': torch.cuda.get_device_capability(0)[0] >= 8}
BIG_GPU = False
VIDEO_ENGINE, IMAGE_MODEL = None, 'procedural'  # All visuals use approved assets and drawn scenes.

def free_disk_gb():
    return shutil.disk_usage(WORK).free / 2**30

def need_disk(gb, what, purge=()):
    if free_disk_gb() < gb:
        raise RuntimeError(f'Need {gb} GiB free scratch for {what}. Completed outputs remain on Drive.')

def free_tier_estimate(n_ai, n_drawn):
    print(f'{n_drawn} drawn scenes; {n_ai} AI clips. No diffusion model downloads. '
          'Runtime depends on cached speech and the hardware Colab allocates; quota is not guaranteed.')

print('Project:', PROJ, '| GPU:', GPU['name'] or 'none', '| local model cache:', MODEL_CACHE)


In [ ]:
# ==== 3b. Engine library: validation, drawn scenes, music library ====
# These are ordinary Python modules (colab/engine/lib/ in the project), copied into the notebook so it stays a
# single file to upload. They are unit-tested outside Colab (colab/tests/).
import sys, importlib
LIB_DIR = WORK.parent / "_engine_lib"
LIB_DIR.mkdir(parents=True, exist_ok=True)
for _name, _src in json.loads(r"""{"acceptance.py": "\"\"\"Acceptance, targeted repair and cache identities for generated clips.\n\ndecide(results) -> Decision\n  ACCEPT  every critical check PASS (non-critical FAIL/UNKNOWN are reported, never hidden)\n  REPAIR  a critical FAIL with a known targeted repair left in the shot's budget\n  REVIEW  a critical UNKNOWN, or failures with no repair left: a person must approve (approvals.json) before the\n          clip may be used; otherwise the editor uses the shot's deterministic fallback\nNothing is chosen \"because it scored highest\": a candidate is usable only if ACCEPTED or approved.\n\nCache identities (separate on purpose):\n  generation key: everything that shapes the pixels (model, recipe, steps, LoRA, frames, size, seed, prompt, control\n                  video, references, generator code version). Change \u2192 regenerate.\n  validation key: generation key + the constraints the clip is judged against (scene-state, expected tracks) +\n                  validator versions and thresholds. Change \u2192 re-validate the existing clip (no regeneration);\n                  until then its status is UNKNOWN, not \"accepted\".\nPure Python; unit-tested in tests/test_acceptance.py.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib, json\nfrom dataclasses import dataclass, field\n\n# failure (check name) -> repairs to try, in order. Each repair is something the executor can actually do.\nREPAIRS = {\n    \"direction\":          [\"layout_strength\", \"layout_simplify\", \"fallback\"],\n    \"start_position\":     [\"layout_strength\", \"first_frame\", \"fallback\"],\n    \"end_position\":       [\"layout_strength\", \"layout_simplify\", \"fallback\"],\n    \"stationary\":         [\"layout_strength\", \"static_camera\", \"fallback\"],\n    \"camera_static\":      [\"static_camera\", \"layout_strength\", \"fallback\"],\n    \"teleport\":           [\"reseed\", \"fewer_frames\", \"fallback\"],\n    \"presence\":           [\"reference_identity\", \"reseed\", \"fallback\"],\n    \"stop_before_action\": [\"split_action\", \"fallback\"],\n    \"object_transfer\":    [\"split_action\", \"layout_strength\", \"fallback\"],\n    \"event_order\":        [\"split_action\", \"fallback\"],\n    \"event\":              [\"split_action\", \"reseed\", \"fallback\"],     # VLM: the action didn't happen\n    # walking/entering didn't happen as planned (geometry FAIL, or not measurable + VLM \"no\"): steer the layout\n    # harder / simplify it \u2014 the layout boxes ARE what drives the walk; a new seed alone rarely fixes a structure\n    \"event_motion\":       [\"layout_strength\", \"layout_simplify\", \"fallback\"],\n    \"completes_before_cut\": [\"earlier_action\", \"fallback\"],\n    \"identity\":           [\"reference_identity\", \"fallback\"],\n}\n# what each repair changes (documented; the executor implements the ones marked automatic)\nREPAIR_EFFECT = {\n    \"layout_strength\":    \"automatic: VACE conditioning_scale +0.25 (max 1.5) and the layout boxes drawn thicker\",\n    \"layout_simplify\":    \"automatic: drop non-critical subjects from the layout; keep only the moving subject and its target\",\n    \"static_camera\":      \"automatic: remove camera-move words from the prompt and add 'static camera, locked-off shot'\",\n    \"reseed\":             \"automatic: new seed, same controls (only for random glitches such as a teleport frame)\",\n    \"fewer_frames\":       \"automatic: generate fewer frames (shorter motion is more stable), edit stretches gently\",\n    \"first_frame\":        \"needs a composition keyframe (L4+): condition the first frame on it\",\n    \"reference_identity\": \"needs a character reference image (VACE reference_images) \u2014 L4+ or a prepared reference\",\n    \"split_action\":       \"plan change: split the shot into shorter shots with one action each \u2192 REVIEW\",\n    \"earlier_action\":     \"plan change: end the action earlier (event t) \u2192 REVIEW\",\n    \"fallback\":           \"use the shot's deterministic fallback (drawn scene / still + move) and flag it\",\n}\nAUTOMATIC = {\"layout_strength\", \"layout_simplify\", \"static_camera\", \"reseed\", \"fewer_frames\"}\n\n\n@dataclass\nclass Decision:\n    status: str                                   # ACCEPT | REPAIR | REVIEW\n    repair: str | None = None\n    reasons: list = field(default_factory=list)\n    unknown: list = field(default_factory=list)\n\n\ndef decide(results, tried: list[str], budget: int = 2, available=frozenset(AUTOMATIC)):\n    \"\"\"results: list of CheckResult (or dicts). tried: repairs already applied to this shot.\"\"\"\n    rs = [r if isinstance(r, dict) else r.to_dict() for r in results]\n    crit = [r for r in rs if r.get(\"critical\", True)]\n    fails = [r for r in crit if r[\"status\"] == \"FAIL\"]\n    unknown = [r for r in crit if r[\"status\"] == \"UNKNOWN\"]\n    reasons = [f\"{r['check']}({r.get('entity') or '-'}): {r['reason']}\" for r in fails]\n    unk = [f\"{r['check']}({r.get('entity') or '-'}): {r['reason']}\" for r in unknown]\n    if not fails and not unknown:\n        return Decision(\"ACCEPT\", reasons=[], unknown=[])\n    if fails and len(tried) < budget:\n        for r in fails:\n            for rep in REPAIRS.get(r[\"check\"], [\"fallback\"]):\n                if rep in available and rep not in tried:\n                    return Decision(\"REPAIR\", rep, reasons, unk)\n    return Decision(\"REVIEW\", None, reasons, unk)\n\n\ndef _h(obj) -> str:\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]\n\n\ndef generation_key(*, model: str, recipe: str, steps: int, lora: str | None, frames: int, size: tuple, seed: int,\n                   prompt: str, negative: str, control: str | None = None, references: list | None = None,\n                   code_version: int = 1, extra: dict | None = None) -> str:\n    return _h({\"model\": model, \"recipe\": recipe, \"steps\": steps, \"lora\": lora, \"frames\": frames, \"size\": list(size),\n               \"seed\": seed, \"prompt\": prompt, \"negative\": negative, \"control\": control,\n               \"references\": references or [], \"code\": code_version, \"extra\": extra or {}})\n\n\ndef validation_key(gen_key: str, constraints: dict, validator_versions: dict, thresholds: dict) -> str:\n    return _h({\"gen\": gen_key, \"constraints\": constraints, \"validators\": validator_versions, \"thresholds\": thresholds})\n\n\ndef usable(sidecar: dict, gen_key: str, val_key: str, approvals: dict) -> tuple[bool, str]:\n    \"\"\"May the editor use this clip? (True/False, why).\"\"\"\n    if not sidecar or sidecar.get(\"gen_key\") != gen_key:\n        return False, \"missing or made with different settings: regenerate\"\n    if sidecar.get(\"val_key\") != val_key:\n        return False, \"constraints or validators changed since it was checked: re-validate\"\n    st = sidecar.get(\"decision\")\n    if st == \"ACCEPT\":\n        return True, \"accepted by the checks\"\n    ap = approvals.get(sidecar.get(\"shot_id\", \"\"), {})\n    if ap.get(\"approved\") and ap.get(\"gen_key\") == gen_key:\n        return True, f\"approved by a person ({ap.get('note', '')})\"\n    return False, f\"{st or 'unchecked'}: needs approval in approvals.json, otherwise the fallback is used\"\n", "assets.py": "\"\"\"Licensed real images from Wikimedia Commons, ranked for relevance to what the narration says.\n\nsearch (Commons API, polite client from research.Wiki) \u2192 licence policy (registry.asset_license: PD / CC0 / CC BY;\nBY-SA only on opt-in; unknown rejected) \u2192 size/type filter \u2192 download (cached, sha256) \u2192 SigLIP 2 relevance ranking\n\u2192 decision with evidence \u2192 assets_manifest.json (creator, licence, URLs, hash, query, scores).\nA candidate is accepted only if it matches the beat's visual description better than generic distractors AND clears\na calibrated floor; otherwise the beat falls back to a deterministic card. Never \"any picture rather than none\".\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib, html, json, re, urllib.request\nfrom pathlib import Path\n\nfrom registry import asset_license\n\nCOMMONS_API = \"https://commons.wikimedia.org/w/api.php\"\nRANK_MODEL = \"google/siglip2-base-patch16-224\"\nTH = {\"min_width\": 1000, \"floor_logit\": -6.0, \"margin\": 1.5,   # calibrated on real Commons images (right +3.9..+6.5,\n      \"narration_margin\": 0.0}                                   # wrong \u2264 -2.7): see docs/10 and tests/test_assets.py\nDISTRACTORS = [\"a page of text\", \"a logo or icon\", \"a chart or table of numbers\", \"a photo of a person's face\",\n               \"a map\", \"a screenshot of a website\", \"a building\", \"an empty white background\"]\n\n\ndef _clean(h):\n    return re.sub(r\"\\s+\", \" \", html.unescape(re.sub(r\"<[^>]+>\", \" \", h or \"\"))).strip()\n\n\ndef search(wiki, query, n=12, allow_sharealike=False):\n    \"\"\"Commons files for a query, each with its licence decision (rejected ones kept, with the reason).\"\"\"\n    d = wiki.get(action=\"query\", generator=\"search\", gsrsearch=f\"{query} filetype:bitmap\", gsrnamespace=\"6\",\n                 gsrlimit=str(n), prop=\"imageinfo\", iiprop=\"url|size|mime|extmetadata\", iiurlwidth=\"1600\",\n                 iiextmetadatafilter=\"LicenseShortName|LicenseUrl|Artist|AttributionRequired|ImageDescription|Credit\")\n    out = []\n    for pg in sorted(d.get(\"query\", {}).get(\"pages\", []), key=lambda p: p.get(\"index\", 0)):\n        ii = (pg.get(\"imageinfo\") or [{}])[0]\n        md = ii.get(\"extmetadata\", {})\n        get = lambda k: (md.get(k) or {}).get(\"value\", \"\")\n        ok, why = asset_license(_clean(get(\"LicenseShortName\")), get(\"LicenseUrl\"), allow_sharealike)\n        if ok and ii.get(\"width\", 0) < TH[\"min_width\"]:\n            ok, why = False, f\"too small ({ii.get('width')} px wide)\"\n        if ok and ii.get(\"mime\") not in (\"image/jpeg\", \"image/png\", \"image/svg+xml\", \"image/tiff\", \"image/webp\"):\n            ok, why = False, f\"not a picture ({ii.get('mime')})\"\n        out.append({\"id\": \"C\" + hashlib.sha1(pg[\"title\"].encode()).hexdigest()[:10], \"title\": pg[\"title\"],\n                    \"page\": ii.get(\"descriptionurl\"), \"file_url\": ii.get(\"thumburl\") or ii.get(\"url\"),\n                    \"width\": ii.get(\"width\"), \"height\": ii.get(\"height\"), \"mime\": ii.get(\"mime\"),\n                    \"license\": _clean(get(\"LicenseShortName\")), \"license_url\": get(\"LicenseUrl\"),\n                    \"author\": _clean(get(\"Artist\"))[:200] or \"unknown\", \"credit\": _clean(get(\"Credit\"))[:200],\n                    \"attribution_required\": str(get(\"AttributionRequired\")).lower() in (\"true\", \"1\", \"yes\"),\n                    \"description\": _clean(get(\"ImageDescription\"))[:300], \"query\": query,\n                    \"license_ok\": ok, \"license_why\": why})\n    return out\n\n\ndef download(asset, folder, ua):\n    \"\"\"Fetch the (1600 px) rendition once; the file name is its content hash (cache + provenance).\"\"\"\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\n    known = folder / f\"{asset['id']}.json\"\n    if known.exists():\n        meta = json.loads(known.read_text())\n        if (folder / meta[\"file\"]).exists():\n            asset.update(file=meta[\"file\"], sha256=meta[\"sha256\"]); return folder / meta[\"file\"]\n    req = urllib.request.Request(asset[\"file_url\"], headers={\"User-Agent\": ua})\n    data = urllib.request.urlopen(req, timeout=60).read()\n    sha = hashlib.sha256(data).hexdigest()\n    ext = {\"image/png\": \".png\", \"image/svg+xml\": \".png\"}.get(asset[\"mime\"], \".jpg\")   # SVGs come as PNG renditions\n    f = folder / f\"{sha[:16]}{ext}\"\n    f.write_bytes(data)\n    asset.update(file=f.name, sha256=sha)\n    known.write_text(json.dumps({\"file\": f.name, \"sha256\": sha}))\n    return f\n\n\nclass Ranker:\n    \"\"\"SigLIP 2 image\u2013text logits. Used RELATIVELY: the intended description must beat generic distractors by a\n    margin and clear a floor. A high score is evidence of relevance, never proof that the picture is correct.\"\"\"\n    def __init__(self, model=RANK_MODEL, device=\"cpu\", cache_dir=None):\n        import torch\n        from transformers import AutoModel, AutoProcessor\n        self.torch, self.device = torch, device\n        self.proc = AutoProcessor.from_pretrained(model, cache_dir=cache_dir)\n        self.model = AutoModel.from_pretrained(model, cache_dir=cache_dir).to(device).eval()\n\n    def logits(self, images, texts):\n        \"\"\"[PIL images] \u00d7 [texts] \u2192 matrix of logits (rows: images).\"\"\"\n        with self.torch.inference_mode():\n            inp = self.proc(text=texts, images=images, padding=\"max_length\", max_length=64, return_tensors=\"pt\").to(self.device)\n            return self.model(**inp).logits_per_image.float().cpu().numpy()\n\n\ndef rank(ranker, files, description, others=(), th=TH, narration=None, other_narration=()):\n    \"\"\"Score candidate images for a beat. Accepted only if the picture fits (1) the beat's visual description AND\n    (2) what the narration actually SAYS, each better than generic distractors and the OTHER beats (a picture that\n    fits another beat better is not this beat's picture). Returns [(index, logit, margin, accepted, why)] best first.\"\"\"\n    from PIL import Image\n    if not files:\n        return []\n    imgs = [Image.open(f).convert(\"RGB\") for f in files]\n    others = [o for o in others if o and o != description]\n    texts = [description] + list(DISTRACTORS) + others\n    n_t = len(texts)\n    if narration:\n        texts += [narration] + [o for o in other_narration if o and o != narration]\n    L = ranker.logits(imgs, texts)\n    out = []\n    for i, row in enumerate(L):\n        own, best_other = float(row[0]), float(max(row[1:n_t])) if n_t > 1 else -1e9\n        margin = own - best_other\n        ok = own >= th[\"floor_logit\"] and margin >= th[\"margin\"]\n        why = (\"matches the beat\" if ok else\n               f\"below the floor ({own:.1f} < {th['floor_logit']})\" if own < th[\"floor_logit\"] else\n               f\"fits '{texts[1 + int(row[1:n_t].argmax())][:40]}' as well or better (margin {margin:.1f})\")\n        if ok and narration:\n            said = float(row[n_t])\n            rivals = list(row[1:1 + len(DISTRACTORS)]) + list(row[n_t + 1:])\n            n_margin = said - (max(rivals) if rivals else -1e9)\n            if n_margin < th[\"narration_margin\"]:\n                ok, why = False, f\"fits its description but not what is SAID (narration margin {n_margin:.1f})\"\n        out.append((i, round(own, 2), round(margin, 2), ok, why))\n    return sorted(out, key=lambda r: (-r[3], -r[1]))\n\n\ndef credit_line(a):\n    \"\"\"Short on-screen / description credit, as CC BY requires (TASL: title, author, source, licence).\"\"\"\n    title = re.sub(r\"^File:|\\.\\w+$\", \"\", a[\"title\"])\n    return f\"{title} \u2014 {a['author']} \u2014 {a['license']} \u2014 Wikimedia Commons\"\n", "characters.py": "\"\"\"Character Bible: the two recurring friends of the storytelling videos.\n\nBoth characters are ORIGINAL artwork drawn by code (charanim.py) from the canonical parameters below \u2014 no image\nmodel, no third-party character design, no real person's likeness \u2014 so the artwork is ours to use. Reference\nimages are rendered once per (character, expression, gesture) and stored with stable content-hash asset IDs: a\ncharacter is never \"re-imagined\" from a text prompt. Outfits/settings may vary per video; identity fields may not.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib, json\nfrom typing import Literal, Optional\n\nfrom pydantic import BaseModel, Field, field_validator\n\nEXPRESSIONS = (\"neutral\", \"smile\", \"curious\", \"surprised\", \"confused\", \"thinking\", \"happy\", \"listening\", \"wink\")\nGESTURES = (\"idle\", \"explain\", \"point\", \"think\", \"open_hands\", \"nod\", \"duck\")   # duck: dodge something flying at you\n\n\nclass Voice(BaseModel):\n    \"\"\"One character's voice in one language. A preset of an approved model (no cloning, no real person).\"\"\"\n    backend: Literal[\"kokoro\", \"parler\", \"qwen3-tts\"]\n    voice: str                                   # Kokoro voice / Parler trained speaker / Qwen speaker\n    style: str = \"\"                              # Parler description of how they speak\n    pitch: float = 0.0                           # identity pitch offset (semitones), never used for emotion\n    qwen_speaker: Optional[str] = None           # Qwen3-TTS CustomVoice preset used when VOICE_BACKEND=\"qwen3-tts\"\n    gemini_voice: Optional[str] = None           # Gemini prebuilt voice (backend \"gemini_free\"); provisional until\n    gemini_voice_status: str = \"\"                # \"provisional\" | \"auditioned\" (Telugu audition 2026-10-19: the user\n                                                 # chose Arjun = Achird, Meera = Sulafat)\n    emotion_control: Literal[\"none\", \"instruction\", \"description\"] = \"none\"   # what the backend can follow\n    rate_range: tuple[float, float] = (0.92, 1.08)                          # allowed pace multipliers\n    intensity_range: tuple[float, float] = (0.1, 0.5)                       # validated expressive range\n    accent: str = \"\"\n    baseline: str = \"\"                           # baseline delivery, in words\n    reference_sample: str = \"\"                   # speaker-verification reference (made on the first run)\n    pronunciation: dict[str, str] = Field(default_factory=dict)            # written word \u2192 what the TTS reads\n\n\nclass Look(BaseModel):\n    skin: str\n    hair: str\n    hair_style: Literal[\"short_messy\", \"long_tied\", \"bob\", \"short_neat\"]\n    top: str\n    top_detail: str                              # collar / trim colour\n    eyes: str = \"#2B1D14\"\n    glasses: bool = False\n\n\nclass Character(BaseModel):\n    id: str = Field(pattern=r\"^[a-z][a-z0-9_]{1,23}$\")\n    name: str\n    names: dict[str, str] = Field(default_factory=dict)         # per language: {\"te\": \"\u0c05\u0c30\u0c4d\u0c1c\u0c41\u0c28\u0c4d\"}\n    role: Literal[\"curious\", \"expert\"]\n    personality: str\n    look: Look\n    voices: dict[str, Voice]                                    # per language code\n    expressions: list[str] = Field(default_factory=lambda: list(EXPRESSIONS))\n    gestures: list[str] = Field(default_factory=lambda: list(GESTURES))\n    tag: str                                                    # detector query (scene checks)\n    color: str                                                  # subtitle colour\n\n    @field_validator(\"expressions\")\n    @classmethod\n    def known_expr(cls, v):\n        bad = [e for e in v if e not in EXPRESSIONS]\n        if bad:\n            raise ValueError(f\"unknown expressions {bad}\")\n        return v\n\n    @field_validator(\"gestures\")\n    @classmethod\n    def known_gest(cls, v):\n        bad = [g for g in v if g not in GESTURES]\n        if bad:\n            raise ValueError(f\"unknown gestures {bad}\")\n        return v\n\n\nclass Bible(BaseModel):\n    version: int = 1\n    relationship: str\n    characters: dict[str, Character]\n\n    @field_validator(\"characters\")\n    @classmethod\n    def one_of_each(cls, v):\n        roles = sorted(c.role for c in v.values())\n        if roles != [\"curious\", \"expert\"]:\n            raise ValueError(f\"the bible needs exactly one curious and one expert friend, got {roles}\")\n        for k, c in v.items():\n            if k != c.id:\n                raise ValueError(f\"character key {k} != id {c.id}\")\n        return v\n\n    def by_role(self, role):\n        return next(c for c in self.characters.values() if c.role == role)\n\n\nDEFAULT_BIBLE = {\n    \"version\": 1,\n    \"relationship\": \"Close friends of the same age who hang out often. Arjun notices things and asks; Meera loves \"\n                    \"explaining with real examples and small demonstrations. She never mocks him and never lectures \"\n                    \"for long; he asks real follow-ups, sometimes with a plausible wrong guess, never just 'Wow'.\",\n    \"characters\": {\n        \"arjun\": {\n            \"id\": \"arjun\", \"name\": \"Arjun\", \"names\": {\"te\": \"\u0c05\u0c30\u0c4d\u0c1c\u0c41\u0c28\u0c4d\"}, \"role\": \"curious\",\n            \"personality\": \"curious, observant, relatable, occasionally funny; asks the question the viewer is thinking\",\n            \"look\": {\"skin\": \"#C68B59\", \"hair\": \"#1F1A17\", \"hair_style\": \"short_messy\", \"top\": \"#F4A23B\",\n                     \"top_detail\": \"#C97A12\"},\n            \"voices\": {\"en\": {\"backend\": \"kokoro\", \"voice\": \"am_michael\", \"qwen_speaker\": \"aiden\", \"accent\": \"American\",\n                              \"baseline\": \"friendly, curious, a little energetic; questions rise naturally, never cartoonish\",\n                              \"rate_range\": [0.95, 1.08], \"intensity_range\": [0.1, 0.5]},\n                       \"te\": {\"backend\": \"parler\", \"voice\": \"Prakash\", \"emotion_control\": \"description\",\n                              \"gemini_voice\": \"Achird\", \"gemini_voice_status\": \"auditioned\",\n                              \"style\": \"in a young, lively and curious voice at a moderate pace\",\n                              \"baseline\": \"friendly and curious\", \"pronunciation\": {\"3D\": \"\u0c24\u0c4d\u0c30\u0c40\u0c21\u0c40\"}}},\n            \"tag\": \"a young man in an orange hoodie\", \"color\": \"#FFD166\"},\n        \"meera\": {\n            \"id\": \"meera\", \"name\": \"Meera\", \"names\": {\"te\": \"\u0c2e\u0c40\u0c30\u0c3e\"}, \"role\": \"expert\",\n            \"personality\": \"knowledgeable, friendly, patient; simplifies with familiar examples and demonstrations\",\n            \"look\": {\"skin\": \"#A86B4A\", \"hair\": \"#2B1B14\", \"hair_style\": \"long_tied\", \"top\": \"#2E8C8C\",\n                     \"top_detail\": \"#F2F2F2\", \"glasses\": True},\n            # Qwen3-TTS CustomVoice has no native-English female preset (only aiden/ryan are English natives):\n            # Meera stays on Kokoro af_heart until a female preset is measured and approved (see the voice benchmark)\n            \"voices\": {\"en\": {\"backend\": \"kokoro\", \"voice\": \"af_heart\", \"accent\": \"American\",\n                              \"baseline\": \"warm, patient, confident; calm explanations that brighten on discoveries\",\n                              \"rate_range\": [0.93, 1.05], \"intensity_range\": [0.1, 0.45]},\n                       \"te\": {\"backend\": \"parler\", \"voice\": \"Lalitha\", \"emotion_control\": \"description\",\n                              \"gemini_voice\": \"Sulafat\", \"gemini_voice_status\": \"auditioned\",\n                              \"style\": \"in a warm, clear and friendly explaining voice at a moderate pace\",\n                              \"baseline\": \"warm and clear\", \"pronunciation\": {\"3D\": \"\u0c24\u0c4d\u0c30\u0c40\u0c21\u0c40\"}}},\n            \"tag\": \"a young woman in a teal jacket\", \"color\": \"#7FDBDA\"},\n    },\n}\n\n\ndef load(bible=None) -> Bible:\n    return Bible.model_validate(bible or DEFAULT_BIBLE)\n\n\ndef asset_id(char: Character, expression: str, gesture: str, extra: str = \"\") -> str:\n    \"\"\"Stable ID of one reference image: same look + expression + gesture \u2192 same ID, on every machine.\"\"\"\n    key = json.dumps([char.id, char.look.model_dump(), expression, gesture, extra], sort_keys=True)\n    return f\"{char.id}_{expression}_{gesture}_{hashlib.sha1(key.encode()).hexdigest()[:10]}\"\n\n\ndef cast_characters(bible: Bible, language: str = \"en\") -> dict:\n    \"\"\"The plan's `characters` block (existing cast format) for this bible and language.\"\"\"\n    out = {}\n    for c in bible.characters.values():\n        v = c.voices.get(language)\n        if v is None:\n            raise ValueError(f\"{c.name} has no voice for language '{language}' (add it to the Character Bible)\")\n        entry = {\"name\": c.names.get(language, c.name), \"voice\": v.voice, \"speed\": 1.0, \"pitch\": v.pitch,\n                 \"fx\": \"none\", \"color\": c.color, \"look\": f\"original 2D character {c.name}\", \"tag\": c.tag,\n                 \"role\": c.role}\n        if v.style:\n            entry[\"voice_style\"] = v.style\n        if v.qwen_speaker:\n            entry[\"qwen_speaker\"] = v.qwen_speaker\n        if v.gemini_voice:\n            entry[\"gemini_voice\"] = v.gemini_voice\n        out[c.id] = entry\n    return out\n\n\ndef speak_text(text: str, voice: Voice) -> str | None:\n    \"\"\"The TTS string for a line from the character's pronunciation profile (None when it equals the text).\"\"\"\n    out = text\n    for w, say in voice.pronunciation.items():\n        out = out.replace(w, say)\n    return out if out != text else None\n", "charanim.py": "\"\"\"Lightweight 2D character scenes on the CPU: the two friends of the Character Bible talking in a place.\n\nNo AI model and no GPU. The characters are original vector-style drawings made from the Bible's canonical look\n(characters.py), so they look the same in every shot of every video. Per frame only small things change:\nbreathing, blinking, the speaking mouth (from the speaker's real audio level), a nod, an eyebrow, a gesture.\n\nSpeed: each body part is drawn once at 2x (smooth edges), shrunk, and cached by its state; a frame is a cached\nbackground plus a few cached sprites pasted at offsets (~20\u201340 ms at 1080p on one CPU core).\n\nMouth: \"energy\" lip movement follows the loudness of the speaker's own voice track, so the mouth is open when sound\nis heard and closed in pauses. It is not phoneme lip sync (Rhubarb would be; optional, not bundled). With no audio\nlevel the mouth stays closed \u2014 never random flapping.\n\nSpec (drawkit kind \"character\"), all times are fractions 0..1 of the visual:\n  {\"world\": \"living_room|cinema_seats|cinema_screen|lobby|studio\",\n   \"shot\": \"two_shot|wide|closeup|reaction|ots|establishing\", \"subject\": \"<char id>\" (closeup/reaction/ots),\n   \"chars\": {\"arjun\": {\"expr\": \"curious\", \"gesture\": \"point\", \"wear\": [\"3d_glasses\"], \"hold\": \"3d_glasses\"}, ...},\n   \"screen\": \"double|clean|plain\" (cinema worlds), \"changes\": [{\"t\": 0.5, \"chars\": {...}, \"screen\": \"clean\"}],\n   \"label\": \"Illustration\" (optional corner tag), \"push\": true (slow push-in),\n   \"_speech\": {\"arjun\": [0..1 per frame]}, \"_t0\": seconds (start of this visual in the video), \"_dur\": seconds}\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib, math\nfrom collections import OrderedDict\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFilter\n\nimport characters as charbible\n\nSS = 2                                    # supersampling factor for smooth edges\nW0, H0 = 700, 1000                        # local design box of one character (waist up), at scale 1\nWORLDS = (\"living_room\", \"cinema_seats\", \"cinema_screen\", \"lobby\", \"studio\")\nSHOTS = (\"two_shot\", \"wide\", \"closeup\", \"reaction\", \"ots\", \"establishing\")\nWEARABLES = (\"3d_glasses\",)\nHOLDABLES = (\"3d_glasses\", \"popcorn\")\n_BIBLE = None\n_cache: \"OrderedDict[tuple, Image.Image]\" = OrderedDict()\nCACHE_MAX = 160\n\n\ndef bible():\n    global _BIBLE\n    if _BIBLE is None:\n        _BIBLE = charbible.load()\n    return _BIBLE\n\n\ndef set_bible(b):\n    global _BIBLE\n    _BIBLE = b\n    _cache.clear()\n\n\ndef _cached(key, make):\n    if key in _cache:\n        _cache.move_to_end(key)\n        return _cache[key]\n    im = make()\n    _cache[key] = im\n    if len(_cache) > CACHE_MAX:\n        _cache.popitem(last=False)\n    return im\n\n\ndef rgb(c):\n    c = c.lstrip(\"#\")\n    return tuple(int(c[i:i + 2], 16) for i in (0, 2, 4))\n\n\ndef shade(c, k):\n    \"\"\"k < 0 darker, k > 0 lighter.\"\"\"\n    c = rgb(c) if isinstance(c, str) else c\n    return tuple(int(max(0, min(255, v + (255 - v) * k if k > 0 else v * (1 + k)))) for v in c)\n\n\ndef _canvas(w, h):\n    im = Image.new(\"RGBA\", (int(w * SS), int(h * SS)), (0, 0, 0, 0))\n    return im, ImageDraw.Draw(im)\n\n\ndef _down(im, scale):\n    w, h = im.size\n    return im.resize((max(1, round(w / SS * scale)), max(1, round(h / SS * scale))), Image.LANCZOS)\n\n\n# ------------------------------------------------------------------------------------------- body parts\ndef _P(*xy):\n    return [v * SS for v in xy]\n\n\ndef torso_sprite(cid, scale):\n    \"\"\"Neck + shoulders + top (waist up), drawn once per character and scale.\"\"\"\n    def make():\n        c = bible().characters[cid]\n        L = c.look\n        im, d = _canvas(W0, H0)\n        skin, top, det = rgb(L.skin), rgb(L.top), rgb(L.top_detail)\n        d.rounded_rectangle(_P(312, 440, 388, 600), radius=30 * SS, fill=shade(skin, -0.12))     # neck\n        if L.hair_style == \"short_messy\":                                                    # hood behind the neck\n            d.ellipse(_P(205, 520, 495, 660), fill=shade(top, -0.22))\n        d.rounded_rectangle(_P(120, 560, 580, 1100), radius=150 * SS, fill=top)               # shoulders/torso\n        d.rounded_rectangle(_P(120, 560, 580, 1100), radius=150 * SS, outline=shade(top, -0.25), width=4 * SS)\n        if L.hair_style == \"short_messy\":                                                    # hoodie: collar + strings\n            d.chord(_P(255, 520, 445, 640), 0, 180, fill=shade(top, -0.3))\n            for x in (318, 382):\n                d.line(_P(x, 600, x - 4, 760), fill=det, width=7 * SS)\n                d.ellipse(_P(x - 13, 752, x + 5, 770), fill=det)\n            d.rounded_rectangle(_P(250, 860, 450, 960), radius=30 * SS, outline=shade(top, -0.25), width=5 * SS)\n        else:                                                                                # jacket over a T-shirt\n            d.polygon(_P(285, 575, 415, 575, 350, 760), fill=det)\n            d.line(_P(285, 575, 350, 770, 350, 1000), fill=shade(top, -0.3), width=6 * SS)\n            d.line(_P(415, 575, 350, 770), fill=shade(top, -0.3), width=6 * SS)\n            d.polygon(_P(270, 565, 300, 560, 345, 720, 320, 735), fill=shade(top, 0.12))   # lapels\n            d.polygon(_P(430, 565, 400, 560, 355, 720, 380, 735), fill=shade(top, 0.12))\n        return _down(im, scale)\n    return _cached((\"torso\", cid, round(scale, 3)), make)\n\n\nEXPR = {   # eyebrow lift (px), eyebrow tilt (deg, + = inner end up), mouth shape, eye openness, blush\n    \"neutral\":   (0, 0, \"flat\", 1.0, False),\n    \"smile\":     (2, 0, \"smile\", 0.95, False),\n    \"happy\":     (6, 0, \"grin\", 0.85, True),\n    \"curious\":   (10, 8, \"small\", 1.05, False),\n    \"surprised\": (22, 4, \"o\", 1.2, False),\n    \"confused\":  (6, -10, \"wavy\", 1.0, False),\n    \"thinking\":  (4, -6, \"side\", 0.9, False),\n    \"listening\": (3, 3, \"soft\", 1.0, False),\n    \"wink\":      (4, 0, \"side\", 1.0, False),     # one eye closed (e.g. the one-eye test)\n}\n\n\ndef head_sprite(cid, scale, expr, mouth, eyes_open, facing, wear):\n    \"\"\"Head with hair, face and worn props. mouth 0..3 (0 closed), facing -1 left / +1 right / 0 front.\"\"\"\n    key = (\"head\", cid, round(scale, 3), expr, mouth, eyes_open, facing, tuple(wear))\n\n    def make():\n        c = bible().characters[cid]\n        L = c.look\n        im, d = _canvas(W0, 560)\n        skin, hair, eye = rgb(L.skin), rgb(L.hair), rgb(L.eyes)\n        fx = 16 * facing                                       # features turn a little toward the other person\n        lift, tilt, mshape, eo, blush = EXPR.get(expr, EXPR[\"neutral\"])\n        # hair behind the head\n        if L.hair_style == \"long_tied\":\n            d.ellipse(_P(470 - 60 * (facing > 0), 250, 600 - 60 * (facing > 0), 470), fill=shade(hair, -0.1))  # bun/tail\n            d.ellipse(_P(185, 120, 515, 470), fill=hair)\n        elif L.hair_style == \"bob\":\n            d.rounded_rectangle(_P(175, 130, 525, 520), radius=140 * SS, fill=hair)\n        # ears + head\n        for ex in (200, 500):\n            d.ellipse(_P(ex - 24, 300, ex + 24, 380), fill=shade(skin, -0.08))\n        d.ellipse(_P(200, 150, 500, 505), fill=skin)\n        # front hair\n        if L.hair_style == \"short_messy\":\n            pts = [(195, 300), (190, 210), (230, 150), (290, 118), (360, 108), (430, 122), (480, 160), (508, 230),\n                   (505, 300), (470, 235), (450, 250), (420, 205), (385, 235), (350, 200), (315, 238), (280, 205),\n                   (245, 250), (225, 230)]\n            d.polygon(_P(*[v for p in pts for v in p]), fill=hair)\n        elif L.hair_style in (\"long_tied\", \"short_neat\"):\n            d.chord(_P(195, 120, 505, 420), 180, 360, fill=hair)\n            d.polygon(_P(205, 270, 260, 175, 350, 150, 330, 215, 270, 250), fill=hair)       # side parting sweep\n            d.polygon(_P(495, 270, 440, 175, 360, 150, 380, 205, 450, 240), fill=hair)\n        # eyebrows\n        for side, ex in ((-1, 290 + fx), (1, 410 + fx)):\n            y = 288 - lift\n            a = math.radians(tilt) * (-side)\n            dx, dy = 34 * math.cos(a), 34 * math.sin(a)\n            if expr == \"confused\" and side == 1:\n                dy, y = -dy, y - 10\n            d.line(_P(ex - dx, y + dy, ex + dx, y - dy), fill=shade(hair, -0.1), width=11 * SS)\n        # eyes\n        gaze = 7 * facing\n        for ex in (290 + fx, 410 + fx):\n            if eyes_open and not (expr == \"wink\" and ex == 290 + fx):\n                ry = 30 * eo\n                d.ellipse(_P(ex - 27, 345 - ry, ex + 27, 345 + ry), fill=(250, 250, 250))\n                d.ellipse(_P(ex - 16 + gaze, 345 - 17, ex + 16 + gaze, 345 + 17), fill=eye)\n                d.ellipse(_P(ex - 5 + gaze, 335, ex + 3 + gaze, 343), fill=(255, 255, 255))\n                if expr == \"happy\":                          # smiling eyes: lower lid up\n                    d.chord(_P(ex - 30, 345, ex + 30, 395), 180, 360, fill=skin)\n            else:\n                d.arc(_P(ex - 26, 330, ex + 26, 360), 20, 160, fill=shade(skin, -0.5), width=7 * SS)\n        # glasses (part of Meera's identity) or 3D glasses (a prop)\n        if \"3d_glasses\" in wear:                           # tinted, see-through lenses: the eyes stay readable\n            lens = Image.new(\"RGBA\", im.size, (0, 0, 0, 0)); ld = ImageDraw.Draw(lens)\n            for x0 in (240, 360):\n                ld.rounded_rectangle(_P(x0 + fx, 305, x0 + 100 + fx, 385), radius=14 * SS, fill=(40, 44, 52, 120))\n            im.alpha_composite(lens)\n            d = ImageDraw.Draw(im)\n            d.rounded_rectangle(_P(240 + fx, 305, 340 + fx, 385), radius=14 * SS, outline=(15, 15, 18), width=8 * SS)\n            d.rounded_rectangle(_P(360 + fx, 305, 460 + fx, 385), radius=14 * SS, outline=(15, 15, 18), width=8 * SS)\n            d.line(_P(340 + fx, 330, 360 + fx, 330), fill=(15, 15, 18), width=8 * SS)\n            d.line(_P(240 + fx, 330, 200, 320), fill=(15, 15, 18), width=7 * SS)\n            d.line(_P(460 + fx, 330, 500, 320), fill=(15, 15, 18), width=7 * SS)\n        elif L.glasses:\n            for ex in (290 + fx, 410 + fx):\n                d.ellipse(_P(ex - 44, 303, ex + 44, 387), outline=(60, 40, 30), width=6 * SS)\n            d.line(_P(334 + fx, 335, 366 + fx, 335), fill=(60, 40, 30), width=6 * SS)\n        # nose\n        d.arc(_P(330 + fx * 1.4, 370, 370 + fx * 1.4, 420), 30, 150, fill=shade(skin, -0.28), width=6 * SS)\n        if blush:\n            for ex in (265 + fx, 435 + fx):\n                d.ellipse(_P(ex - 26, 395, ex + 26, 420), fill=(230, 120, 120, 70))\n        # mouth\n        mx, my = 350 + fx * 1.2, 450\n        lip, inside = shade(skin, -0.45), (120, 40, 50)\n        if mouth > 0:\n            wdt = 34 + (8 if mshape in (\"smile\", \"grin\") else 0)\n            hgt = (0, 10, 20, 32)[mouth]\n            d.ellipse(_P(mx - wdt, my - hgt / 2, mx + wdt, my + hgt / 2 + 6), fill=inside, outline=lip, width=4 * SS)\n            if mouth >= 2:\n                d.chord(_P(mx - wdt + 10, my + hgt / 2 - 6, mx + wdt - 10, my + hgt / 2 + 10), 180, 360,\n                        fill=(205, 95, 105))\n        elif mshape == \"o\":\n            d.ellipse(_P(mx - 16, my - 16, mx + 16, my + 20), fill=inside, outline=lip, width=4 * SS)\n        elif mshape == \"grin\":\n            d.chord(_P(mx - 46, my - 22, mx + 46, my + 34), 0, 180, fill=inside, outline=lip, width=4 * SS)\n            d.chord(_P(mx - 36, my - 18, mx + 36, my + 4), 0, 180, fill=(250, 250, 250))\n        elif mshape in (\"smile\", \"soft\"):\n            a = 40 if mshape == \"smile\" else 30\n            d.arc(_P(mx - a, my - 30, mx + a, my + 16), 25, 155, fill=lip, width=7 * SS)\n        elif mshape == \"wavy\":\n            d.line(_P(mx - 30, my + 4, mx - 10, my - 4, mx + 10, my + 4, mx + 30, my - 4), fill=lip, width=6 * SS)\n        elif mshape == \"side\":\n            d.line(_P(mx - 24, my + 2, mx + 26, my - 6), fill=lip, width=7 * SS)\n        elif mshape == \"small\":\n            d.ellipse(_P(mx - 10, my - 6, mx + 10, my + 10), fill=inside, outline=lip, width=3 * SS)\n        else:\n            d.line(_P(mx - 28, my, mx + 28, my), fill=lip, width=7 * SS)\n        return _down(im, scale)\n    return _cached(key, make)\n\n\n# arm poses: (shoulder, elbow, hand) in local coords for the arm on the \"front\" side (+1 = character's right of\n# the image) \u2014 mirrored for facing; \"rest\" is the idle pose every gesture starts from\nPOSES = {\n    \"idle\":       {\"R\": ((520, 640), (575, 860), (560, 1060)), \"L\": ((180, 640), (125, 860), (140, 1060))},\n    \"explain\":    {\"R\": ((520, 640), (585, 860), (470, 760)), \"L\": ((180, 640), (125, 860), (140, 1060))},\n    \"point\":      {\"R\": ((520, 640), (640, 650), (760, 600)), \"L\": ((180, 640), (125, 860), (140, 1060))},\n    \"think\":      {\"R\": ((520, 640), (520, 820), (395, 520)), \"L\": ((180, 640), (230, 860), (420, 840))},\n    \"open_hands\": {\"R\": ((520, 640), (600, 840), (600, 760)), \"L\": ((180, 640), (100, 840), (100, 760))},\n    \"nod\":        {\"R\": ((520, 640), (575, 860), (560, 1060)), \"L\": ((180, 640), (125, 860), (140, 1060))},\n    \"show\":       {\"R\": ((520, 640), (620, 800), (560, 600)), \"L\": ((180, 640), (125, 860), (140, 1060))},\n    \"duck\":       {\"R\": ((520, 640), (640, 520), (470, 330)), \"L\": ((180, 640), (60, 520), (230, 330))},\n}                  # \"duck\": both forearms up in front of the face, the body drops (character_frame)                  # \"show\": a held prop lifted next to the face (used automatically in close-ups)\n\n\ndef arms_sprite(cid, scale, gesture, phase, facing, hold):\n    \"\"\"Both arms for a gesture at phase 0..1 (0 = resting), mirrored so gestures point toward the other person.\"\"\"\n    q = round(phase * 8) / 8\n    key = (\"arms\", cid, round(scale, 3), gesture, q, facing, hold)\n\n    def make():\n        c = bible().characters[cid]\n        top, skin = rgb(c.look.top), rgb(c.look.skin)\n        im, d = _canvas(W0 + 260, H0)                       # wider: a pointing hand reaches out of the body box\n        ox = 130\n        mirror = facing < 0\n        for side in (\"R\", \"L\"):\n            pts = []\n            for a, b in zip(POSES[\"idle\"][side], POSES.get(gesture, POSES[\"idle\"])[side]):\n                pts.append((a[0] + (b[0] - a[0]) * q, a[1] + (b[1] - a[1]) * q))\n            if mirror:\n                pts = [(W0 - x, y) for x, y in pts]\n            pts = [(x + ox, y) for x, y in pts]\n            (sx, sy), (ex, ey), (hx, hy) = pts\n            col = shade(top, -0.08 if side == \"L\" else 0)\n            d.line(_P(sx, sy, ex, ey), fill=col, width=86 * SS)\n            d.line(_P(ex, ey, hx, hy), fill=col, width=76 * SS)\n            for x, y, r in ((sx, sy, 43), (ex, ey, 40)):\n                d.ellipse(_P(x - r, y - r, x + r, y + r), fill=col)\n            d.ellipse(_P(hx - 36, hy - 36, hx + 36, hy + 36), fill=skin)\n            if gesture == \"point\" and side == \"R\" and q > 0.5:     # index finger\n                dirx = -1 if mirror else 1\n                d.line(_P(hx, hy - 8, hx + 70 * dirx, hy - 20), fill=skin, width=20 * SS)\n            if hold and side == \"R\" and q > 0.3 and gesture in (\"explain\", \"open_hands\", \"point\", \"show\"):\n                _held(d, hold, hx, hy)\n        return _down(im, scale)\n    return _cached(key, make)\n\n\ndef _held(d, what, hx, hy):\n    if what == \"3d_glasses\":\n        y = hy - 70\n        for x0 in (hx - 105, hx + 5):\n            d.rounded_rectangle(_P(x0, y, x0 + 100, y + 70), radius=12 * SS, fill=(40, 44, 52), outline=(15, 15, 18),\n                                width=8 * SS)\n        d.line(_P(hx - 5, y + 25, hx + 5, y + 25), fill=(15, 15, 18), width=8 * SS)\n    elif what == \"popcorn\":\n        d.polygon(_P(hx - 60, hy - 150, hx + 60, hy - 150, hx + 45, hy + 10, hx - 45, hy + 10), fill=(230, 60, 60))\n        for k in range(-2, 3):\n            d.line(_P(hx + 22 * k, hy - 150, hx + 18 * k, hy + 10), fill=(250, 250, 250), width=10 * SS)\n        for k in range(7):\n            x = hx - 54 + 18 * k\n            d.ellipse(_P(x - 18, hy - 185 - 12 * (k % 2), x + 18, hy - 150), fill=(255, 238, 180))\n\n\ndef back_sprite(cid, scale):\n    \"\"\"A character seen from behind (over-the-shoulder foreground, audience in the cinema).\"\"\"\n    def make():\n        c = bible().characters[cid]\n        L = c.look\n        im, d = _canvas(W0, H0)\n        hair, top, skin = rgb(L.hair), rgb(L.top), rgb(L.skin)\n        d.rounded_rectangle(_P(120, 560, 580, 1100), radius=150 * SS, fill=shade(top, -0.35))\n        d.rounded_rectangle(_P(312, 440, 388, 600), radius=30 * SS, fill=shade(skin, -0.35))\n        for ex in (200, 500):\n            d.ellipse(_P(ex - 24, 300, ex + 24, 380), fill=shade(skin, -0.3))\n        d.ellipse(_P(195, 140, 505, 500), fill=shade(hair, -0.05))\n        if L.hair_style == \"long_tied\":\n            d.ellipse(_P(300, 250, 400, 360), fill=shade(hair, 0.06))\n        return _down(im, scale)\n    return _cached((\"back\", cid, round(scale, 3)), make)\n\n\n# ------------------------------------------------------------------------------------------- worlds\ndef _grad(w, h, top, bot):\n    y = np.linspace(0, 1, h)[:, None, None]\n    return (np.array(top, float) * (1 - y) + np.array(bot, float) * y) * np.ones((1, w, 1))\n\n\ndef movie_frame(w, h, mode=\"plain\"):\n    \"\"\"What is on the cinema screen: a mountain valley with a bird and a ball flying toward the audience.\n    \"double\": two slightly shifted copies (what you see WITHOUT the glasses); \"clean\": one sharp image with depth.\"\"\"\n    mode, _, prop = mode.partition(\"+\")                 # \"clean+popcorn\": a popcorn bucket instead of the ball\n\n    def one(shift=0):\n        im = Image.fromarray(_grad(w, h, (110, 170, 230), (250, 205, 150)).astype(np.uint8)).convert(\"RGBA\")\n        d = ImageDraw.Draw(im)\n        d.polygon([(0, h * .75), (w * .25, h * .35), (w * .45, h * .7), (w * .65, h * .3), (w, h * .72), (w, h), (0, h)],\n                  fill=(70, 110, 120))\n        d.polygon([(0, h * .9), (w * .3, h * .65), (w * .6, h * .85), (w, h * .7), (w, h), (0, h)], fill=(50, 120, 70))\n        d.ellipse([w * .78, h * .1, w * .9, h * .1 + w * .12], fill=(255, 245, 200))\n        cx = w * .38 + shift\n        if prop == \"popcorn\":                                                               # bucket + flying kernels\n            d.polygon([(cx - h * .15, h * .42), (cx + h * .15, h * .42), (cx + h * .11, h * .74), (cx - h * .11, h * .74)],\n                      fill=(230, 60, 60))\n            for k in range(-2, 3):\n                d.line([(cx + h * .055 * k, h * .42), (cx + h * .04 * k, h * .74)], fill=(250, 250, 250), width=max(3, h // 50))\n            for k, (u, v, r) in enumerate([(-.12, .36, .05), (-.04, .33, .055), (.05, .35, .05), (.12, .37, .045),\n                                           (-.2, .22, .04), (.0, .16, .045), (.18, .2, .04), (.3, .1, .035)]):\n                x, y = cx + h * u * (1.6 if v < .3 else 1) + shift * (0.5 if v < .3 else 0), h * v\n                d.ellipse([x - h * r, y - h * r, x + h * r, y + h * r], fill=(255, 238, 180))\n        else:\n            d.ellipse([cx - h * .17, h * .38, cx + h * .17, h * .72], fill=(235, 70, 60))   # the ball (near: shifts most)\n            d.ellipse([cx - h * .09, h * .43, cx - h * .02, h * .5], fill=(255, 170, 160))\n        bx = w * .62 + shift * 0.3\n        d.line([(bx - 40, h * .22), (bx, h * .25), (bx + 40, h * .21)], fill=(30, 30, 40), width=max(3, h // 60))\n        return im\n    if mode == \"double\":\n        a, b = one(-h * .06), one(h * .06)\n        return Image.blend(a, b, 0.5)\n    im = one()\n    if mode == \"clean\":                                   # the ball \"comes out\": a soft shadow and a brighter rim\n        d = ImageDraw.Draw(im)\n        cx = w * .38\n        d.ellipse([cx - h * .2, h * .74, cx + h * .2, h * .8], fill=(20, 40, 30, 90))\n        if prop != \"popcorn\":                             # the ball's bright rim (the bucket has its own stripes)\n            d.arc([cx - h * .17, h * .38, cx + h * .17, h * .72], 200, 340, fill=(255, 230, 220), width=max(3, h // 70))\n    return im\n\n\ndef world_image(world, screen=\"plain\", blur=0.0):\n    \"\"\"1920x1080 background of a place (cached). blur > 0: shallow depth of field for close-ups.\"\"\"\n    def make():\n        W, H = 1920, 1080\n        if world == \"living_room\":\n            im = Image.fromarray(_grad(W, H, (246, 222, 190), (226, 196, 160)).astype(np.uint8)).convert(\"RGBA\")\n            d = ImageDraw.Draw(im)\n            d.rectangle([0, 760, W, H], fill=(170, 120, 85))                               # floor\n            d.rectangle([0, 740, W, 770], fill=(150, 100, 70))\n            d.rounded_rectangle([140, 170, 560, 560], radius=12, fill=(150, 200, 235), outline=(255, 255, 255), width=16)\n            d.line([350, 170, 350, 560], fill=(255, 255, 255), width=10)                   # window\n            d.line([140, 365, 560, 365], fill=(255, 255, 255), width=10)\n            d.ellipse([400, 210, 470, 280], fill=(255, 240, 190))\n            d.rectangle([1380, 180, 1640, 380], fill=(120, 90, 70)); d.rectangle([1395, 195, 1625, 365], fill=(250, 238, 210))\n            d.polygon([(1420, 340), (1490, 250), (1550, 320), (1580, 280), (1610, 340)], fill=(110, 160, 120))\n            d.rounded_rectangle([240, 600, 1680, 860], radius=60, fill=(90, 120, 150))       # sofa\n            d.rounded_rectangle([240, 520, 1680, 700], radius=60, fill=(105, 138, 170))\n            d.rectangle([1760, 300, 1780, 760], fill=(80, 70, 60)); d.polygon([(1700, 300), (1840, 300), (1800, 200), (1740, 200)],\n                                                                                fill=(255, 225, 160))\n            d.ellipse([60, 640, 200, 780], fill=(70, 130, 80)); d.rectangle([110, 760, 150, 800], fill=(140, 90, 60))\n        elif world == \"cinema_seats\":                      # facing the audience: dark hall, seat rows behind them\n            im = Image.fromarray(_grad(W, H, (22, 18, 40), (40, 18, 30)).astype(np.uint8)).convert(\"RGBA\")\n            d = ImageDraw.Draw(im)\n            for r, (y, s, k) in enumerate([(420, 70, 0.35), (560, 90, 0.5), (720, 115, 0.7)]):\n                col = shade((150, 30, 45), -0.6 + 0.3 * k)\n                n = int(W / (s * 1.25)) + 2\n                for i in range(n):\n                    x = i * s * 1.25 - (r % 2) * s * 0.6\n                    d.rounded_rectangle([x, y, x + s, y + s * 1.4], radius=int(s * 0.3), fill=col)\n            beam = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0)); bd = ImageDraw.Draw(beam)\n            bd.polygon([(960, 60), (0, 1080), (1920, 1080)], fill=(200, 210, 255, 26))         # light from the screen\n            im.alpha_composite(beam)\n            # light from the screen on the audience (they face it); it changes with the film\n            glow = Image.new(\"RGBA\", (W, H), (120, 140, 200, 30) if screen.split(\"+\")[0] != \"double\" else (150, 120, 170, 30))\n            im.alpha_composite(glow)\n        elif world == \"cinema_screen\":                     # from behind the seats toward the big screen\n            im = Image.fromarray(_grad(W, H, (12, 10, 22), (25, 15, 25)).astype(np.uint8)).convert(\"RGBA\")\n            d = ImageDraw.Draw(im)\n            sx0, sy0, sx1, sy1 = 260, 80, 1660, 700\n            glow = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0)); gd = ImageDraw.Draw(glow)\n            gd.rectangle([sx0 - 60, sy0 - 40, sx1 + 60, sy1 + 60], fill=(180, 190, 255, 60))\n            im.alpha_composite(glow.filter(ImageFilter.GaussianBlur(40)))\n            im.alpha_composite(movie_frame(sx1 - sx0, sy1 - sy0, screen), (sx0, sy0))\n            d.rectangle([sx0, sy0, sx1, sy1], outline=(30, 30, 40), width=6)\n            for r, (y, s) in enumerate([(780, 80), (900, 110)]):\n                for i in range(int(W / (s * 1.2)) + 2):\n                    x = i * s * 1.2 - (r % 2) * s * 0.6\n                    d.ellipse([x, y, x + s, y + s * 1.1], fill=(18, 14, 24))                # heads of the audience\n        elif world == \"lobby\":\n            im = Image.fromarray(_grad(W, H, (60, 30, 70), (120, 50, 70)).astype(np.uint8)).convert(\"RGBA\")\n            d = ImageDraw.Draw(im)\n            d.rectangle([0, 780, W, H], fill=(70, 30, 40))\n            for i, x in enumerate((160, 560, 1360)):                                        # movie posters\n                d.rectangle([x, 160, x + 300, 600], fill=(250, 220, 120), outline=(255, 250, 230), width=10)\n                d.rectangle([x + 25, 185, x + 275, 470], fill=[(60, 120, 200), (200, 80, 80), (80, 160, 120)][i])\n                d.rectangle([x + 40, 500, x + 260, 520], fill=(90, 60, 40)); d.rectangle([x + 70, 540, x + 230, 556], fill=(90, 60, 40))\n            for k in range(14):                                                             # marquee lights\n                d.ellipse([60 + k * 135, 70, 84 + k * 135, 94], fill=(255, 230, 150))\n            d.rounded_rectangle([1720, 480, 1920, 800], radius=20, fill=(200, 40, 50))       # popcorn stand\n        else:                                                                               # studio: soft plain\n            im = Image.fromarray(_grad(W, H, (40, 70, 110), (20, 40, 70)).astype(np.uint8)).convert(\"RGBA\")\n        if blur > 0:\n            im = im.filter(ImageFilter.GaussianBlur(blur))\n        return im.convert(\"RGB\")\n    return _cached((\"world\", world, screen, blur), make)\n\n\n# ------------------------------------------------------------------------------------------- shots\ndef _layout(shot, ids, subject):\n    \"\"\"[(char id, x centre, baseline y, scale, facing, view)] back to front, for 1920x1080.\"\"\"\n    a, b = ids[0], ids[-1]\n    other = b if subject == a else a\n    if shot == \"wide\":\n        return [(a, 700, 1130, 0.62, 1, \"front\"), (b, 1220, 1130, 0.62, -1, \"front\")]\n    if shot == \"establishing\":\n        return [(a, 820, 1220, 0.55, 0, \"back\"), (b, 1100, 1220, 0.55, 0, \"back\")]\n    if shot in (\"closeup\", \"reaction\"):\n        left = subject == a\n        return [(subject, 860 if left else 1060, 1460, 1.32, 1 if left else -1, \"front\")]\n    if shot == \"ots\":                                       # over the listener's shoulder at the speaker\n        left = subject == a                                 # speaker on the left, listener's back on the right\n        return [(subject, 760 if left else 1160, 1250, 0.95, 1 if left else -1, \"front\"),\n                (other, 1600 if left else 320, 1500, 1.45, 0, \"back\")]\n    return [(a, 640, 1180, 0.84, 1, \"front\"), (b, 1280, 1180, 0.84, -1, \"front\")]        # two_shot\n\n\ndef _blinks(cid, t_abs):\n    \"\"\"Deterministic blinks: every 2.6\u20135.2 s (per-character schedule), 0.13 s long.\"\"\"\n    rng = np.random.default_rng(int(hashlib.sha1(cid.encode()).hexdigest()[:8], 16))\n    t = rng.uniform(0.5, 2.0)\n    while t < t_abs + 1:\n        if t <= t_abs < t + 0.13:\n            return False\n        t += rng.uniform(2.6, 5.2)\n    return True\n\n\ndef _state(spec, t):\n    \"\"\"Character states and screen at shot fraction t (applies 'changes' in order).\"\"\"\n    chars = {k: dict(v) for k, v in spec.get(\"chars\", {}).items()}\n    screen, since = spec.get(\"screen\", \"plain\"), {k: 0.0 for k in chars}\n    for ch in sorted(spec.get(\"changes\", []), key=lambda c: c[\"t\"]):\n        if ch[\"t\"] <= t:\n            for k, v in ch.get(\"chars\", {}).items():\n                chars.setdefault(k, {}).update(v)\n                if \"gesture\" in v:\n                    since[k] = ch[\"t\"]\n            screen = ch.get(\"screen\", screen)\n    return chars, screen, since\n\n\ndef mouth_level(e):\n    return 0 if e < 0.12 else 1 if e < 0.35 else 2 if e < 0.65 else 3\n\n\ndef character_frame(spec, plan, t, w, h, bg, th):\n    \"\"\"drawkit renderer for kind \"character\" (w, h are the 1920x1080 design size).\n    Plans with visual_style_version \"storybook_v1\" are drawn by storybook_scene (rigged illustrated characters in\n    environment packs); every other plan keeps this drawing exactly as before.\"\"\"\n    if (plan or {}).get(\"visual_style_version\") == \"storybook_v1\":\n        import storybook_scene\n        return storybook_scene.character_frame(spec, plan, t, w, h, bg, th)\n    dur = float(spec.get(\"_dur\", 4.0))\n    n = max(int(spec.get(\"_n\", 1)), 1)\n    fi = min(int(round(t * (n - 1))), n - 1)\n    t_abs = float(spec.get(\"_t0\", 0.0)) + t * dur\n    chars, screen, since = _state(spec, t)\n    ids = list(chars) or list(bible().characters)\n    shot, subject = spec.get(\"shot\", \"two_shot\"), spec.get(\"subject\") or ids[0]\n    world = spec.get(\"world\", \"studio\")\n    close = shot in (\"closeup\", \"reaction\", \"ots\")\n    frame = world_image(world, screen, blur=6.0 if close else (2.0 if shot == \"two_shot\" else 0.0)).copy().convert(\"RGBA\")\n    speech = spec.get(\"_speech\", {})\n    for cid, cx, base, sc, facing, view in _layout(shot, ids, subject):\n        st = chars.get(cid, {})\n        e = (speech.get(cid) or [0.0] * n)[fi] if speech.get(cid) else 0.0\n        breathe = 5 * sc * math.sin(2 * math.pi * t_abs / 3.6 + (0.0 if cid == ids[0] else 1.7))\n        if view == \"back\":\n            spr = back_sprite(cid, sc)\n            y = base - spr.height + breathe\n            frame.alpha_composite(spr, (int(cx - spr.width / 2), int(y)))\n            continue\n        gest = st.get(\"gesture\", \"idle\")\n        if close and st.get(\"hold\") and gest in (\"idle\", \"explain\", \"open_hands\", \"nod\"):\n            gest = \"show\"                                # otherwise the prop is below the frame\n        g_age = (t - since.get(cid, 0.0)) * dur\n        phase = min(g_age / 0.5, 1.0) if gest not in (\"idle\", \"nod\") else 0.0\n        nod = 0.0\n        if gest == \"nod\" and g_age < 1.6:\n            nod = 10 * sc * abs(math.sin(math.pi * g_age / 0.4))\n        talk_bob = 4 * sc * e * math.sin(2 * math.pi * t_abs * 3.1)\n        expr = st.get(\"expr\", \"listening\" if e < 0.05 else \"neutral\")\n        torso = torso_sprite(cid, sc)\n        x0 = int(cx - torso.width / 2)\n        y0 = int(base - torso.height + breathe)\n        if gest == \"duck\":                               # dodge: drop ~12% of the body fast, then stay low\n            y0 += int(0.12 * torso.height * phase)\n        frame.alpha_composite(torso, (x0, y0))\n        head = head_sprite(cid, sc, expr, mouth_level(e), _blinks(cid, t_abs), facing, tuple(st.get(\"wear\", [])))\n        frame.alpha_composite(head, (x0, int(y0 + nod + talk_bob + 1.5 * breathe)))\n        arms = arms_sprite(cid, sc, gest, phase, facing, st.get(\"hold\"))\n        frame.alpha_composite(arms, (int(cx - arms.width / 2), y0))\n    if spec.get(\"push\"):\n        z = 1.0 + 0.05 * t\n        cw, ch_ = int(1920 / z), int(1080 / z)\n        frame = frame.crop(((1920 - cw) // 2, (1080 - ch_) // 2, (1920 - cw) // 2 + cw, (1080 - ch_) // 2 + ch_)).resize(\n            (1920, 1080), Image.BILINEAR)\n    if spec.get(\"label\"):\n        _tag(frame, spec[\"label\"])\n    return np.asarray(frame.convert(\"RGB\"))\n\n\ndef _tag(frame, text):\n    import drawkit\n    d = ImageDraw.Draw(frame)\n    f = drawkit.font(30, True)\n    tw = d.textlength(text, font=f)\n    d.rounded_rectangle([1920 - tw - 80, 36, 1880, 92], radius=14, fill=(0, 0, 0, 150))\n    d.text((1920 - tw - 58, 64), text, font=f, fill=(255, 255, 255), anchor=\"lm\")\n\n\ndef check_spec(spec, plan=None):\n    if (plan or {}).get(\"visual_style_version\") == \"storybook_v1\":\n        import storybook_scene\n        return storybook_scene.check_spec(spec, plan)\n    out = []\n    b = bible()\n    if spec.get(\"world\", \"studio\") not in WORLDS:\n        out.append(f\"unknown world '{spec.get('world')}' (use {', '.join(WORLDS)})\")\n    if spec.get(\"shot\", \"two_shot\") not in SHOTS:\n        out.append(f\"unknown character shot '{spec.get('shot')}' (use {', '.join(SHOTS)})\")\n    chars = spec.get(\"chars\", {})\n    if not chars:\n        out.append(\"character shot needs 'chars'\")\n    for cid, st in list(chars.items()) + [(k, v) for c in spec.get(\"changes\", []) for k, v in c.get(\"chars\", {}).items()]:\n        if cid not in b.characters:\n            out.append(f\"unknown character '{cid}' (Character Bible: {', '.join(b.characters)})\")\n            continue\n        if st.get(\"expr\", \"neutral\") not in b.characters[cid].expressions:\n            out.append(f\"{cid}: unknown expression '{st.get('expr')}'\")\n        if st.get(\"gesture\", \"idle\") not in b.characters[cid].gestures:\n            out.append(f\"{cid}: unknown gesture '{st.get('gesture')}'\")\n        for x in st.get(\"wear\", []):\n            if x not in WEARABLES:\n                out.append(f\"{cid}: unknown wearable '{x}'\")\n        if st.get(\"hold\") and st[\"hold\"] not in HOLDABLES:\n            out.append(f\"{cid}: unknown held prop '{st['hold']}'\")\n    if spec.get(\"shot\") in (\"closeup\", \"reaction\", \"ots\") and spec.get(\"subject\") not in chars:\n        out.append(f\"{spec.get('shot')} needs a 'subject' that is one of the chars\")\n    for c in spec.get(\"changes\", []):\n        if not 0 <= c.get(\"t\", -1) <= 1:\n            out.append(f\"change time t={c.get('t')} must be a fraction 0..1\")\n    return out\n\n\ndef reference_sheet(path, cid=None):\n    \"\"\"Asset library: one PNG per character with every expression (front) + IDs (stable content hashes).\"\"\"\n    b = bible()\n    ids, rows = [cid] if cid else list(b.characters), []\n    for c in ids:\n        ch = b.characters[c]\n        exprs = ch.expressions\n        sheet = Image.new(\"RGB\", (len(exprs) * 260, 330), (235, 238, 242))\n        d = ImageDraw.Draw(sheet)\n        for i, e in enumerate(exprs):\n            tor, hd = torso_sprite(c, 0.36), head_sprite(c, 0.36, e, 0, True, 0, ())\n            sheet.paste(tor, (i * 260 + 4, 330 - tor.height + 40), tor)\n            sheet.paste(hd, (i * 260 + 4, 330 - tor.height + 40), hd)\n            d.text((i * 260 + 130, 12), e, fill=(30, 30, 40), anchor=\"mt\")\n            rows.append({\"character\": c, \"expression\": e, \"gesture\": \"idle\", \"asset_id\": charbible.asset_id(ch, e, \"idle\")})\n        out = str(path).replace(\".png\", f\"_{c}.png\")\n        sheet.save(out)\n    return rows\n", "cinematic_workflow.py": "\"\"\"Reference and approval contracts for opt-in cinematic characters; no image model imports.\"\"\"\nimport copy\nimport hashlib\nimport json\nimport shutil\nfrom pathlib import Path\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef record_review(path, files, kind, reviewer, approved=False):\n    \"\"\"Call only after the named person reviews actual files, never from an automatic score.\"\"\"\n    if approved is not True or not reviewer.strip() or not files:\n        raise ValueError(\"Explicit human approval, reviewer and reviewed files are required\")\n    files = [Path(f) for f in files]\n    if not all(p.is_file() for p in files):\n        raise FileNotFoundError(\"Review inputs are missing\")\n    import datetime\n    data = {\"kind\": kind, \"reviewer\": reviewer, \"approved\": True,\n            \"at\": datetime.datetime.now(datetime.timezone.utc).isoformat(),\n            \"files\": {str(p.resolve()): sha(p) for p in files}}\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(\".partial\")\n    tmp.write_text(json.dumps(data, indent=2))\n    tmp.replace(path)\n    return data\n\n\ndef require_review(path, files, kind):\n    path = Path(path)\n    if not path.is_file():\n        raise ValueError(f\"Human {kind} review is pending: {path}\")\n    data = json.loads(path.read_text())\n    if data.get(\"approved\") is not True or data.get(\"kind\") != kind or not data.get(\"reviewer\"):\n        raise ValueError(f\"Invalid {kind} approval\")\n    required = {str(Path(p).resolve()) for p in files}\n    if not required.issubset(data.get(\"files\", {})):\n        raise ValueError(\"Approval does not cover the current master\")\n    for name, expected in data[\"files\"].items():\n        if not Path(name).is_file() or sha(name) != expected:\n            raise ValueError(f\"Reviewed file changed or disappeared: {name}\")\n    return data\n\n\ndef preflight(references, output):\n    \"\"\"CPU-safe reference validation; report actual hardware without downloading a model.\"\"\"\n    from PIL import Image\n    refs = {}\n    for cid in (\"arjun\", \"meera\"):\n        p = Path(references[cid])\n        if not p.is_file():\n            raise FileNotFoundError(f\"Upload the supplied {cid} reference to {p}\")\n        with Image.open(p) as im:\n            im.verify()\n        with Image.open(p) as im:\n            refs[cid] = {\"file\": str(p), \"sha256\": sha(p), \"size\": list(im.size)}\n    import platform\n    report = {\"python\": platform.python_version(), \"references\": refs, \"gpu\": None,\n              \"disk_free_gb\": round(shutil.disk_usage(Path(output).parent).free / 2**30, 2)}\n    try:\n        import torch\n        report.update(torch=torch.__version__, cuda=torch.version.cuda)\n        if torch.cuda.is_available():\n            p = torch.cuda.get_device_properties(0)\n            report.update(gpu=p.name, vram_gb=p.total_memory / 2**30,\n                          capability=[p.major, p.minor], native_bf16=p.major >= 8)\n    except ImportError:\n        report[\"torch\"] = \"not installed\"\n    try:\n        import psutil\n        report[\"ram_available_gb\"] = psutil.virtual_memory().available / 2**30\n    except ImportError:\n        report[\"ram_available_gb\"] = None\n    Path(output).write_text(json.dumps(report, indent=2))\n    return report\n\n\ndef validate_plan(plan, root):\n    \"\"\"New cinematic plans require approved, hash-pinned atlases. Legacy plans are unaffected.\"\"\"\n    if not plan.get(\"cinematic_characters\"):\n        return\n    import storybook as SB\n    if plan.get(\"visual_style_version\") != \"storybook_v1\":\n        raise ValueError(\"Cinematic characters require the Storybook renderer\")\n    pins = plan.get(\"character_assets\", {})\n    if not {\"arjun\", \"meera\"}.issubset(pins):\n        raise ValueError(\"Both cinematic characters must be pinned\")\n    for cid, pin in pins.items():\n        if pin.get(\"status\") != \"approved\" or not pin.get(\"manifest_sha256\") or not pin.get(\"content_sha256\"):\n            raise ValueError(f\"{cid}: approved manifest and content pins required\")\n        ch = SB.load(cid, pin[\"version\"], root, pin[\"manifest_sha256\"], pin[\"content_sha256\"])\n        if ch.man.get(\"status\") != \"approved\" or ch.man.get(\"asset_type\") != \"atlas_v1\":\n            raise ValueError(f\"{cid}: approved cinematic atlas required; procedural art is not accepted\")\n        if not ch.man.get(\"provenance\", {}).get(\"approved_by\"):\n            raise ValueError(f\"{cid}: human artwork approval missing\")\n\n\ndef bind_plan(plan, root, versions):\n    \"\"\"Create a new plan, preserving every story/audio field and the original plan.\"\"\"\n    import storybook as SB\n    out = copy.deepcopy(plan)\n    out[\"visual_style_version\"] = \"storybook_v1\"\n    out[\"cinematic_characters\"] = True\n    out[\"character_assets\"] = {}\n    for cid, version in versions.items():\n        ch = SB.load(cid, version, root)\n        out[\"character_assets\"][cid] = {\"version\": version, \"status\": ch.man[\"status\"],\n            \"art_version\": ch.man[\"art_version\"], \"manifest_sha256\": SB.manifest_sha(ch.dir / \"manifest.json\"),\n            \"content_sha256\": SB.content_fingerprint(ch.man)}\n    validate_plan(out, root)\n    return out\n\n\nRHUBARB = {\"A\": \"closed\", \"B\": \"teeth\", \"C\": \"small\", \"D\": \"wide\", \"E\": \"round\",\n           \"F\": \"round\", \"G\": \"teeth\", \"H\": \"small\", \"X\": \"rest\"}\n\n\ndef rhubarb_visemes(cues, fps, frames):\n    \"\"\"Optional Rhubarb JSON to existing mouth IDs; does not replace audio-derived visemes.\"\"\"\n    import math\n    if fps <= 0 or frames < 0:\n        raise ValueError(\"Invalid timing\")\n    out = [\"rest\"] * frames\n    last = 0.0\n    for cue in cues:\n        start, end = float(cue[\"start\"]), float(cue[\"end\"])\n        if not math.isfinite(start + end) or start < last or end < start or cue[\"value\"] not in RHUBARB:\n            raise ValueError(\"Invalid or overlapping mouth cue\")\n        for i in range(max(0, math.ceil(start * fps)), min(frames, math.ceil(end * fps))):\n            out[i] = RHUBARB[cue[\"value\"]]\n        last = end\n    return out\n", "colab_budget.py": "\"\"\"Local resource limits for the Telugu cinematic target, not Colab quota predictions.\"\"\"\nimport time\n\n\nclass BudgetExceeded(RuntimeError):\n    pass\n\n\nclass SessionBudget:\n    def __init__(self, minutes=60, clock=time.monotonic):\n        if not 0 < minutes <= 90:\n            raise ValueError('Session budget must be between 0 and 90 minutes.')\n        self.on_exhausted = None\n        self.clock = clock\n        self.deadline = clock() + minutes * 60\n\n    def check(self, stage):\n        if self.clock() >= self.deadline:\n            if self.on_exhausted:\n                self.on_exhausted()\n            raise BudgetExceeded(f'Local session budget reached before {stage}. Completed files are on Drive; '\n                                 'resume in a later allocated session with Run all.')\n\n    def timeout(self, maximum=600):\n        self.check('next operation')\n        return max(0.01, min(maximum, self.deadline - self.clock()))\n\n\ndef validate_target(plan):\n    if plan.get('language') != 'te' or plan.get('tts_backend') != 'indic_parler':\n        raise ValueError('This target requires Telugu and local Indic Parler speech.')\n    if not plan.get('strict_open_source'):\n        raise ValueError('Hosted voices are disabled for this target.')\n    shots = plan.get('shots', [])\n    if not shots or len(shots) > 12:\n        raise ValueError('Expected 1\u201312 drawn scenes for this short video.')\n    if any(s.get('recipe') not in {'story', 'character', 'demo', 'diagram', 'card'} for s in shots):\n        raise ValueError('This target permits drawn scenes only; AI video downloads are disabled.')\n    if sum(len(s.get('lines', [])) for s in shots) > 60:\n        raise ValueError('This short-video target permits at most 60 dialogue lines.')\n    for s in shots:\n        for v in s.get('story', {}).get('visuals', []):\n            if v.get('recipe') not in {'character', 'demo', 'diagram', 'card'}:\n                raise ValueError('Nested story visuals must also be drawn; no generated video.')\n\n\ndef validate_v4_continuity(plan):\n    \"\"\"Deterministic constraints for this specific seated-cinema script, not general semantic proof.\"\"\"\n    if plan.get('story_revision') not in ('three_d_friends_te_v4', 'three_d_friends_te_v5'):\n        return\n    claims = {c['id'] for c in plan.get('claims', [])}\n    for shot in plan['shots']:\n        if shot.get('location') != 'cinema_seats':\n            raise ValueError('v4 stays in one cinema: an unexplained location change is forbidden.')\n        for line in shot['lines']:\n            if set(line.get('claims', [])) - claims:\n                raise ValueError('Unknown factual claim reference.')\n            if line.get('kind') in ('fact', 'correction') and not line.get('claims'):\n                raise ValueError('Factual dialogue needs cited claims.')\n        for visual in shot['story']['visuals']:\n            if visual.get('recipe') == 'character':\n                spec = visual['character']\n                states = [spec.get('chars', {})] + [c.get('chars', {}) for c in spec.get('changes', [])]\n                for state in states:\n                    for char in state.values():\n                        if char.get('hold'):\n                            raise ValueError('v4 has no held props or handoffs.')\n                        if 'wear' in char and char['wear'] != ['3d_glasses']:\n                            raise ValueError('v4 glasses stay worn throughout.')\n", "demos.py": "\"\"\"Physical demonstrations and licensed photos as drawn shots (no AI, no GPU): the \"show, don't tell\" visuals of\nthe two-friend stories. They draw real objects (a hand, eyes, a projector, a screen, glasses, light) acting out\nthe idea \u2014 not boxes and arrows. Short on-screen words only where they label what you see (\"LEFT EYE\").\n\ndrawkit kinds:\n  \"demo\":  {\"template\": \"finger_test\" | \"eye_views\" | \"projector_glasses\", \"caption\": optional short text}\n  \"photo\": {\"file\": absolute path, \"credit\": \"title \u2014 author, licence\", \"move\": \"push_in|pan_left|pan_right\"}\nAll animation is driven by t (0..1 of the visual), so a demo fits whatever length the dialogue gives it.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFilter\n\nimport charanim\nimport drawkit\n\nTEMPLATES = (\"finger_test\", \"eye_views\", \"projector_glasses\")\nLABELS = {\"te\": {\"LEFT EYE\": \"\u0c0e\u0c21\u0c2e \u0c15\u0c28\u0c4d\u0c28\u0c41\", \"RIGHT EYE\": \"\u0c15\u0c41\u0c21\u0c3f \u0c15\u0c28\u0c4d\u0c28\u0c41\", \"PROJECTOR\": \"\u0c2a\u0c4d\u0c30\u0c4a\u0c1c\u0c46\u0c15\u0c4d\u0c1f\u0c30\u0c4d\",\n                 \"SILVER SCREEN\": \"\u0c38\u0c3f\u0c32\u0c4d\u0c35\u0c30\u0c4d \u0c38\u0c4d\u0c15\u0c4d\u0c30\u0c40\u0c28\u0c4d\", \"BRAIN: ONE PICTURE WITH DEPTH\": \"\u0c2e\u0c46\u0c26\u0c21\u0c41: \u0c32\u0c4b\u0c24\u0c41 \u0c09\u0c28\u0c4d\u0c28 \u0c12\u0c15\u0c47 \u0c1a\u0c3f\u0c24\u0c4d\u0c30\u0c02\",\n                 \"EACH LENS LETS ONLY ITS OWN PICTURE THROUGH\": \"\u0c2a\u0c4d\u0c30\u0c24\u0c3f \u0c32\u0c46\u0c28\u0c4d\u0c38\u0c4d \u0c24\u0c28 \u0c1a\u0c3f\u0c24\u0c4d\u0c30\u0c3e\u0c28\u0c4d\u0c28\u0c3f \u0c2e\u0c3e\u0c24\u0c4d\u0c30\u0c2e\u0c47 \u0c2a\u0c02\u0c2a\u0c41\u0c24\u0c41\u0c02\u0c26\u0c3f\"},\n          # code-mixed Telugu plans (plan \"code_mix\"): the words the friends actually say\n          \"te_mix\": {\"LEFT EYE\": \"Left eye\", \"RIGHT EYE\": \"Right eye\", \"PROJECTOR\": \"Projector\",\n                     \"SILVER SCREEN\": \"Silver screen\", \"BRAIN: ONE PICTURE WITH DEPTH\": \"Brain: \u0c30\u0c46\u0c02\u0c21\u0c41 views = \u0c12\u0c15 3D picture\",\n                     \"EACH LENS LETS ONLY ITS OWN PICTURE THROUGH\": \"\u0c12\u0c15\u0c4d\u0c15\u0c4b lens \u0c24\u0c28 picture\u0c28\u0c3f \u0c2e\u0c3e\u0c24\u0c4d\u0c30\u0c2e\u0c47 \u0c2a\u0c02\u0c2a\u0c41\u0c24\u0c41\u0c02\u0c26\u0c3f\"}}\n_LANG = [\"en\"]                                            # set per frame from the plan's language\n\n\ndef tr(text):\n    \"\"\"On-screen words in the video's language (drawn with the script font, so Telugu shapes correctly).\"\"\"\n    return LABELS.get(_LANG[0], {}).get(text) or LABELS.get(_LANG[0].split(\"_\")[0], {}).get(text, text)\nLEFT_C, RIGHT_C = (255, 150, 60), (70, 170, 255)          # the left-eye picture / the right-eye picture\n\n\ndef _ease(x):\n    return drawkit.ease(x)\n\n\ndef _label(d, xy, text, col=(255, 255, 255), size=40, bg=(0, 0, 0, 150)):\n    text = tr(text)\n    f = drawkit.font(size, True)\n    tw = d.textlength(text, font=f)\n    x, y = xy\n    d.rounded_rectangle([x - tw / 2 - 18, y - size * 0.75, x + tw / 2 + 18, y + size * 0.75], radius=14, fill=bg)\n    d.text((x, y), text, font=f, fill=col, anchor=\"mm\")\n\n\ndef _eye_icon(d, cx, cy, s, open_, col=(255, 255, 255)):\n    if open_:\n        d.ellipse([cx - s, cy - s * 0.6, cx + s, cy + s * 0.6], fill=col, outline=(30, 30, 40), width=4)\n        d.ellipse([cx - s * 0.35, cy - s * 0.35, cx + s * 0.35, cy + s * 0.35], fill=(40, 60, 90))\n    else:\n        d.arc([cx - s, cy - s * 0.7, cx + s, cy + s * 0.5], 20, 160, fill=col, width=7)\n\n\n# ------------------------------------------------------------------ 1. the finger test (try it yourself)\ndef finger_test(spec, t, w, h):\n    \"\"\"Your thumb in front of a far window: close one eye, then the other \u2014 the near thumb jumps, the far window\n    hardly moves. Alternates left/right eye 4 times over the shot.\"\"\"\n    phase = int(t * 4) % 2                                    # 0: left eye open, 1: right eye open\n    k = (t * 4) % 1\n    blend = _ease(min(k / 0.18, 1.0))                         # quick switch at the start of each quarter\n    x_near = (1 if phase == 0 else -1) * 130           # seen by the LEFT eye, a near thumb appears to the RIGHT\n    x_prev = -x_near\n    near_dx = x_prev + (x_near - x_prev) * blend\n    far_dx = near_dx * 0.08\n    world = spec.get(\"world\", \"living_room\")\n    bg = charanim.world_image(world if world in (\"living_room\", \"lobby\") else \"living_room\").copy()\n    bg = bg.crop((int(160 + far_dx), 100, int(160 + far_dx) + 1600, 1000)).resize((w, h), Image.BILINEAR)\n    bg = bg.filter(ImageFilter.GaussianBlur(3)).convert(\"RGBA\")\n    d = ImageDraw.Draw(bg)\n    skin = charanim.rgb(charanim.bible().by_role(\"curious\").look.skin)\n    cx = w / 2 + near_dx\n    d.rounded_rectangle([cx - 70, 470, cx + 70, h + 40], radius=60, fill=charanim.shade(skin, -0.05))     # thumb\n    d.rounded_rectangle([cx - 52, 500, cx + 52, 590], radius=40, fill=charanim.shade(skin, 0.25))        # nail\n    d.rounded_rectangle([cx - 190, 760, cx + 120, h + 80], radius=110, fill=skin)                        # fist\n    for j in range(3):\n        d.arc([cx - 180, 790 + j * 70, cx + 40, 880 + j * 70], 200, 330, fill=charanim.shade(skin, -0.25), width=6)\n    over = Image.new(\"RGBA\", (w, h), (0, 0, 0, 0)); od = ImageDraw.Draw(over)\n    od.rounded_rectangle([w / 2 - 260, 40, w / 2 + 260, 170], radius=30, fill=(0, 0, 0, 150))\n    _eye_icon(od, w / 2 - 110, 105, 55, phase == 0)\n    _eye_icon(od, w / 2 + 110, 105, 55, phase == 1)\n    bg.alpha_composite(over)\n    _label(ImageDraw.Draw(bg), (w / 2, 230), \"LEFT EYE\" if phase == 0 else \"RIGHT EYE\", size=42)\n    return bg\n\n\n# ------------------------------------------------------------------ 2. each eye sees its own picture \u2192 one 3D view\ndef _scene_view(w, h, ball_dx, tint=None):\n    \"\"\"A ball in front of a tree, as one eye sees it (the near ball shifts, the far tree barely).\"\"\"\n    im = Image.new(\"RGBA\", (w, h), (205, 230, 250, 255)); d = ImageDraw.Draw(im)\n    d.rectangle([0, h * 0.72, w, h], fill=(120, 185, 110))\n    tx = w * 0.62 + ball_dx * 0.1\n    d.rectangle([tx - w * 0.03, h * 0.35, tx + w * 0.03, h * 0.75], fill=(120, 80, 50))\n    d.ellipse([tx - w * 0.16, h * 0.08, tx + w * 0.16, h * 0.48], fill=(60, 140, 70))\n    bx = w * 0.42 + ball_dx\n    d.ellipse([bx - w * 0.11, h * 0.45, bx + w * 0.11, h * 0.45 + w * 0.22], fill=(235, 70, 60))\n    d.ellipse([bx - w * 0.06, h * 0.49, bx - w * 0.01, h * 0.49 + w * 0.05], fill=(255, 170, 160))\n    if tint:\n        im.alpha_composite(Image.new(\"RGBA\", (w, h), tint + (40,)))\n    return im\n\n\ndef eye_views(spec, t, w, h):\n    \"\"\"Top: a head from above with sight lines from each eye to the ball. Then the two pictures appear side by side\n    (ball in a different place in each), then slide together into one picture where the ball stands out in depth.\"\"\"\n    im = Image.new(\"RGBA\", (w, h), (24, 32, 52, 255)); d = ImageDraw.Draw(im)\n    a = _ease(min(t / 0.3, 1))                         # sight lines\n    b = drawkit.prog(t, 0.25, 0.45)                    # two pictures appear\n    c = drawkit.prog(t, 0.65, 0.85)                    # they merge\n    hx, hy = 330, 560                                  # head from above\n    skin = charanim.rgb(charanim.bible().by_role(\"curious\").look.skin)\n    hair = charanim.rgb(charanim.bible().by_role(\"curious\").look.hair)\n    d.ellipse([hx - 150, hy - 170, hx + 150, hy + 170], fill=hair)\n    d.ellipse([hx + 60, hy - 120, hx + 175, hy + 120], fill=skin)\n    eyes = [(hx + 160, hy - 60), (hx + 160, hy + 60)]\n    for (ex, ey), col in zip(eyes, (LEFT_C, RIGHT_C)):\n        d.ellipse([ex - 20, ey - 20, ex + 20, ey + 20], fill=(255, 255, 255), outline=col, width=6)\n    ball = (760, 560)\n    d.ellipse([ball[0] - 50, ball[1] - 50, ball[0] + 50, ball[1] + 50], fill=(235, 70, 60))\n    d.ellipse([985 - 40, 520, 985 + 40, 600], fill=(60, 140, 70))          # the far tree, further along the view\n    if a > 0:\n        for (ex, ey), col in zip(eyes, (LEFT_C, RIGHT_C)):\n            x1, y1 = ex + (ball[0] - ex) * a, ey + (ball[1] - ey) * a\n            d.line([ex, ey, x1, y1], fill=col, width=7)\n    pw, ph = 380, 300\n    if c < 1:\n        # the left eye sees the near ball further RIGHT against the far tree, the right eye further left\n        for k, (col, dx, name) in enumerate(((LEFT_C, 50, \"LEFT EYE\"), (RIGHT_C, -50, \"RIGHT EYE\"))):\n            if b <= 0:\n                continue\n            x0 = 1080 + k * (pw + 60) - (k * (pw + 60) - (pw + 60) / 2) * c\n            y0 = 260 + 40 * (1 - b)\n            pic = _scene_view(pw, ph, dx)\n            pic.putalpha(int(255 * b * (1 - 0.5 * c)))\n            im.alpha_composite(pic, (int(x0), int(y0)))\n            fade = tuple(int(bgc + (cc - bgc) * b) for cc, bgc in zip(col, (24, 32, 52)))   # fades in with the picture\n            d.rectangle([x0, y0, x0 + pw, y0 + ph], outline=fade, width=8)\n            if c <= 0:                                 # names go before the pictures start to merge\n                _label(d, (x0 + pw / 2, y0 + ph + 50), name, col=col, size=34)\n    if c > 0:                                          # one picture, with depth: the ball floats in front\n        x0, y0 = 1080 + (pw + 60) / 2, 260\n        pic = _scene_view(pw, ph, 0)\n        pd = ImageDraw.Draw(pic)\n        pd.ellipse([pw * 0.42 - pw * 0.13, ph * 0.83, pw * 0.42 + pw * 0.13, ph * 0.9], fill=(30, 60, 30, 110))\n        big = pic.resize((int(pw * (1 + 0.15 * c)), int(ph * (1 + 0.15 * c))), Image.BILINEAR)\n        big.putalpha(int(255 * c))\n        im.alpha_composite(big, (int(x0 - (big.width - pw) / 2), int(y0 - (big.height - ph) / 2)))\n        _label(d, (x0 + pw / 2, y0 + ph + 70), \"BRAIN: ONE PICTURE WITH DEPTH\", size=34)\n    return im\n\n\n# ------------------------------------------------------------------ 3. projector \u2192 screen \u2192 glasses \u2192 eyes\ndef projector_glasses(spec, t, w, h):\n    \"\"\"Seen from above: the projector (behind the audience) throws two pictures onto the same screen at once \u2014\n    orange for the left eye, blue for the right eye \u2014 each with its own light \"twist\" (polarisation). The silver\n    screen reflects both to the viewer. Each lens lets one twist through and blocks the other, so each eye gets\n    only its own picture.\"\"\"\n    im = Image.new(\"RGBA\", (w, h), (16, 14, 28, 255)); d = ImageDraw.Draw(im)\n    a = drawkit.prog(t, 0.0, 0.25)                     # beams to the screen\n    b = drawkit.prog(t, 0.25, 0.5)                     # reflected back to both lenses\n    c = drawkit.prog(t, 0.5, 0.72)                     # filtered at the lenses\n    px, py = 170, 190                                  # projector, above and behind the audience\n    d.rounded_rectangle([px - 120, py - 75, px + 80, py + 75], radius=18, fill=(70, 72, 84))\n    d.ellipse([px + 50, py - 48, px + 146, py + 48], fill=(40, 42, 50), outline=(180, 190, 210), width=6)\n    _label(d, (px, py + 125), \"PROJECTOR\", size=28)\n    sx = 1760\n    d.rectangle([sx, 110, sx + 36, 980], fill=(220, 225, 235))\n    _label(d, (sx - 30, 70), \"SILVER SCREEN\", size=28)\n    vx, vy = 720, 600                                  # viewer's head from above, facing the screen (right)\n    look = charanim.bible().by_role(\"curious\").look\n    skin, hair = charanim.rgb(look.skin), charanim.rgb(look.hair)\n    d.ellipse([vx - 170, vy - 190, vx + 150, vy + 190], fill=hair)\n    d.ellipse([vx + 40, vy - 150, vx + 190, vy + 150], fill=skin)\n    eyes = [(vx + 175, vy - 70), (vx + 175, vy + 70)]  # upper = LEFT eye (facing right, left is up)\n    lens = [(vx + 250, vy - 70), (vx + 250, vy + 70)]\n    for (ex, ey), col in zip(eyes, (LEFT_C, RIGHT_C)):\n        d.ellipse([ex - 22, ey - 22, ex + 22, ey + 22], fill=(255, 255, 255), outline=col, width=6)\n    d.line([lens[0][0], lens[0][1], lens[1][0], lens[1][1]], fill=(15, 15, 18), width=10)\n    for (lx, ly), col in zip(lens, (LEFT_C, RIGHT_C)):\n        d.rounded_rectangle([lx - 16, ly - 50, lx + 16, ly + 50], radius=10, fill=(40, 44, 52), outline=col, width=7)\n\n    def beam(p, q, col, k, twist, width=10):\n        n = 24\n        for i in range(int(n * k)):\n            u0, u1 = i / n, (i + 1) / n\n            x0, y0 = p[0] + (q[0] - p[0]) * u0, p[1] + (q[1] - p[1]) * u0\n            x1, y1 = p[0] + (q[0] - p[0]) * u1, p[1] + (q[1] - p[1]) * u1\n            d.line([x0, y0, x1, y1], fill=col, width=width)\n            if i % 4 == 2:                              # little rotating marks = the light's twist\n                ang = twist * (i * 0.6 + t * 12)\n                d.line([x0 - 18 * math.cos(ang), y0 - 18 * math.sin(ang), x0 + 18 * math.cos(ang),\n                        y0 + 18 * math.sin(ang)], fill=(255, 255, 255), width=4)\n\n    hit = [(sx, 520), (sx, 580)]\n    for (hx, hy), col, tw in zip(hit, (LEFT_C, RIGHT_C), (1, -1)):\n        beam((px + 146, py + (-10 if tw > 0 else 10)), (hx, hy), col, a, tw)\n    if b > 0:\n        for (hx, hy), col, tw in zip(hit, (LEFT_C, RIGHT_C), (1, -1)):\n            for (lx, ly) in lens:                      # both pictures reach BOTH lenses\n                beam((hx, hy), (lx + 18, ly + (-12 if tw > 0 else 12)), col, b, tw, width=7)\n    if c > 0:\n        for k, ((lx, ly), (ex, ey), col) in enumerate(zip(lens, eyes, (LEFT_C, RIGHT_C))):\n            d.line([lx - 16, ly, lx - 16 + (ex + 22 - lx + 16) * c, ly], fill=col, width=12)\n            bad = RIGHT_C if k == 0 else LEFT_C        # the other picture is stopped at this lens\n            x0 = lx + 40\n            d.line([x0, ly - 22, x0 + 44, ly + 22], fill=bad, width=9)\n            d.line([x0, ly + 22, x0 + 44, ly - 22], fill=bad, width=9)\n        _label(d, (w / 2, 60), \"EACH LENS LETS ONLY ITS OWN PICTURE THROUGH\", size=38)   # clear of the subtitles\n    return im\n\n\ndef demo_frame(spec, plan, t, w, h, bg, th):\n    _LANG[0] = (plan or {}).get(\"language\", \"en\") + (\"_mix\" if (plan or {}).get(\"code_mix\") else \"\")\n    fn = {\"finger_test\": finger_test, \"eye_views\": eye_views, \"projector_glasses\": projector_glasses}[spec[\"template\"]]\n    im = fn(spec, t, w, h)\n    if spec.get(\"caption\"):                            # top: the bottom ~20% belongs to the subtitles\n        _label(ImageDraw.Draw(im), (w / 2, 300), spec[\"caption\"], size=40)\n    return np.asarray(im.convert(\"RGB\"))\n\n\n# ------------------------------------------------------------------ licensed photo, fitted + slow camera move\n_photo = {}\n\n\ndef photo_frame(spec, plan, t, w, h, bg, th):\n    key = (spec[\"file\"], w, h)\n    if key not in _photo:\n        _photo.clear()\n        src = Image.open(spec[\"file\"]).convert(\"RGB\")\n        big = Image.new(\"RGB\", (int(w * 1.1), int(h * 1.1)))\n        back = src.resize(big.size, Image.BILINEAR).filter(ImageFilter.GaussianBlur(30))\n        big.paste(Image.eval(back, lambda v: int(v * 0.75)))\n        fg = src.copy(); fg.thumbnail(big.size, Image.LANCZOS)\n        big.paste(fg, ((big.width - fg.width) // 2, (big.height - fg.height) // 2))\n        _photo[key] = big\n    big = _photo[key]\n    move = spec.get(\"move\", \"push_in\")\n    z = 1.0 + 0.08 * t if move == \"push_in\" else 1.04\n    cw, ch = int(big.width / z / 1.1 * 1.0), int(big.height / z / 1.1)\n    x = (big.width - cw) / 2 + {\"pan_left\": -1, \"pan_right\": 1}.get(move, 0) * (t - 0.5) * (big.width - cw) * 0.8\n    y = (big.height - ch) / 2\n    im = big.crop((int(x), int(y), int(x) + cw, int(y) + ch)).resize((w, h), Image.BILINEAR).convert(\"RGBA\")\n    d = ImageDraw.Draw(im)\n    f = drawkit.font(24)\n    text = spec.get(\"credit\", \"\")\n    while d.textlength(text, font=f) > w * 0.6 and len(text) > 20:\n        text = text[:-4].rstrip() + \"\u2026\"\n    if text:\n        tw = d.textlength(text, font=f)\n        d.rounded_rectangle([w - tw - 60, 24, w - 24, 66], radius=8, fill=(0, 0, 0, 150))\n        d.text((w - tw - 42, 45), text, font=f, fill=(235, 235, 240), anchor=\"lm\")\n    return np.asarray(im.convert(\"RGB\"))\n\n\ndef check_spec(kind, spec, plan=None):\n    out = []\n    if kind == \"demo\" and spec.get(\"template\") not in TEMPLATES:\n        out.append(f\"unknown demo template '{spec.get('template')}' (use {', '.join(TEMPLATES)})\")\n    if kind == \"photo\":\n        import os\n        if not spec.get(\"file\") or not os.path.exists(spec[\"file\"]):\n            out.append(f\"photo file missing: {spec.get('file')}\")\n        if not spec.get(\"credit\"):\n            out.append(\"photo needs its credit line (title, author, licence)\")\n    return out\n", "drawkit.py": "\"\"\"Drawn scenes (no AI, no GPU): app screens, a \"journey\" travel metaphor, and title/recap cards.\n\nWhy: AI video models garble text and can't show UI or abstract processes; flowcharts bore non-technical viewers\n(the backend video was 13/17 box-and-arrow diagrams). These recipes show the same ideas the way people meet\nthem in real life: a phone screen, a parcel/letter travelling between places, a calendar, a short card.\n\nAll timings in a spec are FRACTIONS of the shot (0..1), so a spec works whatever the voice length is.\nPure Pillow + numpy; frames are piped to ffmpeg by encode(). Unit-tested in tests/test_drawkit.py.\n\n    render(kind, spec, plan, n_frames, fps, w, h) -> iterator of RGB frames (numpy uint8, h x w x 3)\n\"\"\"\nimport math\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont, ImageFilter\n\nTHEME = {\"bg_top\": (17, 34, 64), \"bg_bottom\": (28, 64, 92), \"card\": (250, 250, 252), \"ink\": (24, 30, 42),\n         \"muted\": (110, 118, 132), \"accent\": (25, 195, 166), \"accent2\": (255, 107, 107), \"gold\": (255, 196, 61),\n         \"line\": (226, 230, 236), \"good\": (46, 170, 90), \"white\": (255, 255, 255), \"shadow\": (0, 0, 0, 90)}\nFONT_PATHS = [\"/usr/share/fonts/truetype/dejavu/DejaVuSans{b}.ttf\",\n              \"/usr/share/fonts/truetype/liberation/LiberationSans{lb}.ttf\"]\n_fonts = {}\nSCRIPT_FONT = {}        # {\"regular\": path, \"bold\": path}: set for non-Latin videos (e.g. Telugu \u2192 Hind Guntur)\n\n\ndef set_script_font(regular, bold=None):\n    \"\"\"Draw ALL text (cards, drawn scenes, diagrams, titles) with one font that has the video's script AND Latin\n    letters (Telugu narration keeps PNR, WL 12, \u20b9 ...). Pillow shapes it with raqm (Telugu conjuncts/vowel signs).\"\"\"\n    SCRIPT_FONT.clear()\n    SCRIPT_FONT.update(regular=str(regular), bold=str(bold or regular))\n    _fonts.clear()\n\n\ndef font_file(name):\n    \"\"\"Path of a DejaVu font file (e.g. 'DejaVuSans-Bold.ttf'): the system copy, else the copy matplotlib ships\n    (matplotlib is preinstalled on Colab), else None. Never fails. With a script font set, that font instead.\"\"\"\n    import os\n    if SCRIPT_FONT:\n        return SCRIPT_FONT[\"bold\" if \"Bold\" in name else \"regular\"]\n    cands = [f\"/usr/share/fonts/truetype/dejavu/{name}\"]\n    try:\n        import matplotlib\n        cands.append(os.path.join(os.path.dirname(matplotlib.__file__), \"mpl-data\", \"fonts\", \"ttf\", name))\n    except Exception:\n        pass\n    return next((c for c in cands if os.path.exists(c)), None)\n\n\ndef font(size, bold=False):\n    k = (size, bold)\n    if k not in _fonts:\n        f = None\n        mpl = font_file(\"DejaVuSans-Bold.ttf\" if bold else \"DejaVuSans.ttf\")\n        for p in ([mpl] if SCRIPT_FONT else FONT_PATHS + ([mpl] if mpl else [])):\n            try:\n                f = ImageFont.truetype(p.format(b=\"-Bold\" if bold else \"\", lb=\"-Bold\" if bold else \"-Regular\"), size)\n                break\n            except OSError:\n                continue\n        _fonts[k] = f or ImageFont.load_default(size=size)\n    return _fonts[k]\n\n\ndef ease(x):\n    x = min(max(x, 0.0), 1.0)\n    return 4 * x ** 3 if x < 0.5 else 1 - (-2 * x + 2) ** 3 / 2\n\n\ndef prog(t, a, b):\n    \"\"\"0 before a, 1 after b, eased in between (all in shot fractions).\"\"\"\n    return ease((t - a) / max(b - a, 1e-6))\n\n\ndef rgb(c):\n    if isinstance(c, str):\n        c = c.lstrip(\"#\")\n        return tuple(int(c[i:i + 2], 16) for i in (0, 2, 4))\n    return tuple(c)\n\n\ndef theme(plan):\n    th = dict(THEME, currency=\"$\")\n    for k, v in (plan.get(\"look\") or {}).items():\n        th[k] = rgb(v) if isinstance(v, str) and v.startswith(\"#\") else v     # colours, plus e.g. \"currency\": \"\u20b9\"\n    return th\n\n\ndef background(w, h, th):\n    y = np.linspace(0, 1, h)[:, None, None]\n    top, bot = np.array(th[\"bg_top\"], float), np.array(th[\"bg_bottom\"], float)\n    img = (top * (1 - y) + bot * y) * np.ones((1, w, 1))\n    yy, xx = np.mgrid[0:h, 0:w]                                  # soft vignette light in the middle\n    glow = np.exp(-(((xx - w * 0.5) / (w * 0.6)) ** 2 + ((yy - h * 0.45) / (h * 0.7)) ** 2))[..., None] * 18\n    return Image.fromarray(np.clip(img + glow, 0, 255).astype(np.uint8))\n\n\n_shadow_cache = {}\ndef shadowed(base, box, radius, fill, blur=18, offset=(0, 10)):\n    \"\"\"Rounded rectangle with a soft drop shadow, drawn onto base (RGBA). The blur is done on a small patch\n    around the box and cached by size (blurring the full frame made drawn shots ~10x slower).\"\"\"\n    x0, y0, x1, y1 = [int(round(v)) for v in box]\n    m = blur * 3\n    key = (x1 - x0, y1 - y0, radius, blur)\n    if key not in _shadow_cache:\n        sh = Image.new(\"RGBA\", (x1 - x0 + 2 * m, y1 - y0 + 2 * m), (0, 0, 0, 0))\n        ImageDraw.Draw(sh).rounded_rectangle([m, m, m + x1 - x0, m + y1 - y0], radius, fill=(0, 0, 0, 110))\n        _shadow_cache[key] = sh.filter(ImageFilter.GaussianBlur(blur))\n    pos = (x0 - m + offset[0], y0 - m + offset[1])\n    patch = _shadow_cache[key]\n    cx0, cy0 = max(pos[0], 0), max(pos[1], 0)                     # clip to the frame\n    cx1, cy1 = min(pos[0] + patch.width, base.width), min(pos[1] + patch.height, base.height)\n    if cx1 > cx0 and cy1 > cy0:\n        base.alpha_composite(patch.crop((cx0 - pos[0], cy0 - pos[1], cx1 - pos[0], cy1 - pos[1])), (cx0, cy0))\n    ImageDraw.Draw(base).rounded_rectangle([x0, y0, x1, y1], radius, fill=fill)\n\n\ndef text_fit(d, text, max_w, size, bold=False, min_size=18):\n    while size > min_size and d.textlength(text, font=font(size, bold)) > max_w:\n        size -= 2\n    return font(size, bold)\n\n\ndef wrap(d, text, f, max_w):\n    lines, cur = [], \"\"\n    for wd in text.split():\n        t = (cur + \" \" + wd).strip()\n        if cur and d.textlength(t, font=f) > max_w:\n            lines.append(cur); cur = wd\n        else:\n            cur = t\n    return lines + ([cur] if cur else [])\n\n\n# ---------------------------------------------------------------- icons (simple, friendly, consistent)\ndef icon(d, kind, cx, cy, s, th, active=True, on_light=False):\n    \"\"\"Flat icon centred at (cx, cy), size s. kinds: phone, store, warehouse, truck, bank, card, network,\n    processor, person, box, money, clock, calendar, check, question.\"\"\"\n    col = th[\"accent\"] if active else (150, 160, 175)\n    ink, white = th[\"ink\"], (th[\"white\"] if not on_light else (196, 214, 226))\n    r = s / 2\n    if kind == \"phone\":\n        d.rounded_rectangle([cx - r * .5, cy - r * .9, cx + r * .5, cy + r * .9], r * .12, fill=ink, outline=col, width=4)\n        d.rounded_rectangle([cx - r * .4, cy - r * .72, cx + r * .4, cy + r * .66], r * .06, fill=col)\n    elif kind in (\"store\", \"shop\"):\n        d.rectangle([cx - r * .8, cy - r * .2, cx + r * .8, cy + r * .8], fill=white)\n        for i in range(5):                                       # striped awning\n            x0 = cx - r * .9 + i * r * .36\n            d.rectangle([x0, cy - r * .55, x0 + r * .36, cy - r * .15], fill=col if i % 2 == 0 else white)\n        d.rectangle([cx - r * .2, cy + r * .2, cx + r * .2, cy + r * .8], fill=col)\n    elif kind in (\"warehouse\", \"box\"):\n        if kind == \"warehouse\":\n            d.polygon([(cx - r * .9, cy - r * .1), (cx, cy - r * .7), (cx + r * .9, cy - r * .1)], fill=col)\n            d.rectangle([cx - r * .8, cy - r * .1, cx + r * .8, cy + r * .8], fill=white)\n            d.rectangle([cx - r * .45, cy + r * .2, cx + r * .45, cy + r * .8], fill=(200, 205, 215))\n        else:\n            d.rectangle([cx - r * .7, cy - r * .5, cx + r * .7, cy + r * .7], fill=(214, 168, 112))\n            d.rectangle([cx - r * .12, cy - r * .5, cx + r * .12, cy + r * .7], fill=(236, 205, 160))\n    elif kind == \"truck\":\n        d.rectangle([cx - r * .9, cy - r * .5, cx + r * .25, cy + r * .4], fill=white)\n        d.polygon([(cx + r * .25, cy - r * .2), (cx + r * .7, cy - r * .2), (cx + r * .9, cy + r * .1),\n                   (cx + r * .9, cy + r * .4), (cx + r * .25, cy + r * .4)], fill=col)\n        for x in (cx - r * .55, cx + r * .55):\n            d.ellipse([x - r * .18, cy + r * .3, x + r * .18, cy + r * .66], fill=ink)\n    elif kind == \"bank\":\n        d.polygon([(cx - r * .95, cy - r * .35), (cx, cy - r * .9), (cx + r * .95, cy - r * .35)], fill=col)\n        for i in range(4):\n            x = cx - r * .65 + i * r * .43\n            d.rectangle([x - r * .1, cy - r * .3, x + r * .1, cy + r * .55], fill=white)\n        d.rectangle([cx - r * .95, cy + r * .55, cx + r * .95, cy + r * .75], fill=col)\n    elif kind == \"card\":\n        d.rounded_rectangle([cx - r * .9, cy - r * .55, cx + r * .9, cy + r * .55], r * .12, fill=col)\n        d.rectangle([cx - r * .9, cy - r * .3, cx + r * .9, cy - r * .1], fill=ink)\n        d.rounded_rectangle([cx - r * .7, cy + r * .1, cx - r * .25, cy + r * .35], r * .05, fill=th[\"gold\"])\n    elif kind == \"network\":\n        d.ellipse([cx - r * .85, cy - r * .85, cx + r * .85, cy + r * .85], outline=col, width=6)\n        d.ellipse([cx - r * .4, cy - r * .85, cx + r * .4, cy + r * .85], outline=col, width=4)\n        d.line([cx - r * .85, cy, cx + r * .85, cy], fill=col, width=4)\n        for y in (cy - r * .45, cy + r * .45):\n            d.line([cx - r * .72, y, cx + r * .72, y], fill=col, width=3)\n    elif kind == \"processor\":\n        d.rounded_rectangle([cx - r * .65, cy - r * .65, cx + r * .65, cy + r * .65], r * .15, fill=white)\n        for i in range(-2, 3):                                   # chip pins\n            o = i * r * .25\n            for a, b in (((cx + o, cy - r * .65), (cx + o, cy - r * .9)), ((cx + o, cy + r * .65), (cx + o, cy + r * .9)),\n                         ((cx - r * .65, cy + o), (cx - r * .9, cy + o)), ((cx + r * .65, cy + o), (cx + r * .9, cy + o))):\n                d.line([a, b], fill=col, width=5)\n        d.rounded_rectangle([cx - r * .3, cy - r * .3, cx + r * .3, cy + r * .3], r * .08, fill=col)\n    elif kind == \"person\":\n        d.ellipse([cx - r * .32, cy - r * .85, cx + r * .32, cy - r * .2], fill=col)\n        d.pieslice([cx - r * .7, cy - r * .1, cx + r * .7, cy + r * 1.2], 180, 360, fill=col)\n    elif kind == \"money\":\n        d.rounded_rectangle([cx - r * .9, cy - r * .5, cx + r * .9, cy + r * .5], r * .1, fill=th[\"good\"])\n        d.ellipse([cx - r * .3, cy - r * .3, cx + r * .3, cy + r * .3], outline=white, width=5)\n        f = font(int(r * .5), True)\n        d.text((cx, cy), th.get(\"currency\", \"$\"), font=f, fill=white, anchor=\"mm\")\n    elif kind == \"clock\":\n        d.ellipse([cx - r * .85, cy - r * .85, cx + r * .85, cy + r * .85], fill=white, outline=col, width=8)\n        d.line([cx, cy, cx, cy - r * .55], fill=ink, width=7)\n        d.line([cx, cy, cx + r * .4, cy + r * .1], fill=ink, width=7)\n    elif kind == \"check\":\n        d.ellipse([cx - r * .85, cy - r * .85, cx + r * .85, cy + r * .85], fill=th[\"good\"])\n        d.line([(cx - r * .4, cy), (cx - r * .1, cy + r * .32), (cx + r * .45, cy - r * .35)], fill=white, width=int(r * .18))\n    elif kind == \"question\":\n        d.ellipse([cx - r * .85, cy - r * .85, cx + r * .85, cy + r * .85], fill=th[\"accent2\"])\n        d.text((cx, cy + r * .05), \"?\", font=font(int(r * 1.2), True), fill=white, anchor=\"mm\")\n    elif kind == \"calendar\":\n        d.rounded_rectangle([cx - r * .8, cy - r * .7, cx + r * .8, cy + r * .8], r * .12, fill=white)\n        d.rectangle([cx - r * .8, cy - r * .7, cx + r * .8, cy - r * .3], fill=th[\"accent2\"])\n    else:\n        d.ellipse([cx - r * .6, cy - r * .6, cx + r * .6, cy + r * .6], fill=col)\n\n\n# ---------------------------------------------------------------- UI recipe: a phone screen\nPHONE = {\"w\": 500, \"h\": 820, \"top\": 36}   # ends at y=856: the subtitles below (y\u2248900\u20131040) never cover it\n\n\ndef _widget_height(d, wdg, inner_w):\n    k = wdg[\"type\"]\n    if k == \"header\":\n        return 96\n    if k == \"product\":\n        return 150\n    if k == \"button\":\n        return 92\n    if k == \"choice\":\n        return 66 * len(wdg[\"options\"]) + 20\n    if k == \"status\":\n        return 74 * len(wdg[\"steps\"]) + 20\n    if k == \"statement\":\n        return 64 * len(wdg[\"rows\"]) + 70\n    if k == \"notice\":\n        return 24 + 40 * len(wrap(d, wdg[\"text\"], font(26), inner_w - 90))\n    if k == \"text\":\n        return 16 + int(wdg.get(\"size\", 28) * 1.35) * len(wrap(d, wdg[\"text\"], font(wdg.get(\"size\", 28), wdg.get(\"bold\", False)), inner_w))\n    return 40\n\n\ndef _draw_widget(img, d, wdg, x, y, iw, t, ev, th):\n    \"\"\"Draws one widget at (x, y) width iw; ev = {\"show\": p, \"tap\": p, \"status\": value, \"hl\": p} at time t.\"\"\"\n    k = wdg[\"type\"]\n    ink, mut = th[\"ink\"], th[\"muted\"]\n    if k == \"header\":\n        d.rectangle([x - 24, y - 6, x + iw + 24, y + 90], fill=th[\"white\"])\n        d.text((x, y + 42), wdg[\"text\"], font=text_fit(d, wdg[\"text\"], iw - 60, 36, True), fill=ink, anchor=\"lm\")\n        if wdg.get(\"back\", True):\n            d.text((x + iw, y + 42), \"\u22ee\", font=font(34, True), fill=mut, anchor=\"rm\")\n        d.line([x - 24, y + 90, x + iw + 24, y + 90], fill=th[\"line\"], width=2)\n    elif k == \"product\":\n        d.rounded_rectangle([x, y + 10, x + 120, y + 130], 18, fill=(236, 240, 246))\n        icon(d, wdg.get(\"icon\", \"box\"), x + 60, y + 70, 92, th, on_light=True)\n        d.text((x + 140, y + 30), wdg[\"name\"], font=text_fit(d, wdg[\"name\"], iw - 150, 30, True), fill=ink)\n        if wdg.get(\"detail\"):\n            d.text((x + 140, y + 74), wdg[\"detail\"], font=text_fit(d, wdg[\"detail\"], iw - 150, 24), fill=mut)\n        if wdg.get(\"price\"):\n            d.text((x + 140, y + 108), wdg[\"price\"], font=font(28, True), fill=ink)\n    elif k == \"button\":\n        tap = ev.get(\"tap\", 0.0)\n        primary = wdg.get(\"style\", \"primary\") == \"primary\"\n        base = th[\"accent\"] if primary else th[\"white\"]\n        pressed = 1 - abs(2 * tap - 1) if 0 < tap < 1 else 0\n        fill = tuple(int(c * (1 - 0.18 * pressed)) for c in base)\n        d.rounded_rectangle([x, y + 8, x + iw, y + 80], 36, fill=fill, outline=None if primary else th[\"accent\"], width=3)\n        d.text((x + iw / 2, y + 44), wdg[\"text\"], font=text_fit(d, wdg[\"text\"], iw - 40, 30, True),\n               fill=th[\"white\"] if primary else th[\"accent\"], anchor=\"mm\")\n        if 0 < tap < 1:                                            # finger-tap ripple\n            rr = 20 + 120 * tap\n            ov = Image.new(\"RGBA\", img.size, (0, 0, 0, 0))\n            ImageDraw.Draw(ov).ellipse([x + iw / 2 - rr, y + 44 - rr, x + iw / 2 + rr, y + 44 + rr],\n                                       fill=(255, 255, 255, int(150 * (1 - tap))))\n            img.alpha_composite(ov)\n    elif k == \"choice\":\n        sel = wdg.get(\"selected\")\n        p = ev.get(\"hl\", 1.0 if sel is not None else 0.0)\n        for i, opt in enumerate(wdg[\"options\"]):\n            yy = y + 10 + 66 * i\n            on = i == sel and p > 0.5\n            d.rounded_rectangle([x, yy, x + iw, yy + 56], 14, fill=(232, 250, 246) if on else th[\"white\"],\n                                outline=th[\"accent\"] if on else th[\"line\"], width=3)\n            d.ellipse([x + 16, yy + 16, x + 40, yy + 40], outline=th[\"accent\"] if on else mut, width=3)\n            if on:\n                d.ellipse([x + 22, yy + 22, x + 34, yy + 34], fill=th[\"accent\"])\n            d.text((x + 56, yy + 28), opt, font=text_fit(d, opt, iw - 70, 26), fill=ink, anchor=\"lm\")\n    elif k == \"status\":\n        cur = ev.get(\"status\", wdg.get(\"current\", 0))              # float: 1.5 = halfway from step 1 to 2\n        for i, st in enumerate(wdg[\"steps\"]):\n            yy = y + 20 + 74 * i\n            done = cur >= i\n            if i < len(wdg[\"steps\"]) - 1:                          # connector line, fills as status advances\n                d.line([x + 22, yy + 22, x + 22, yy + 96], fill=th[\"line\"], width=6)\n                fillp = min(max(cur - i, 0), 1)\n                if fillp > 0:\n                    d.line([x + 22, yy + 22, x + 22, yy + 22 + 74 * fillp], fill=th[\"good\"], width=6)\n            d.ellipse([x + 6, yy + 6, x + 38, yy + 38], fill=th[\"good\"] if done else th[\"white\"],\n                      outline=th[\"good\"] if done else th[\"line\"], width=4)\n            if done:\n                d.line([(x + 14, yy + 22), (x + 20, yy + 29), (x + 31, yy + 15)], fill=th[\"white\"], width=4)\n            d.text((x + 56, yy + 22), st, font=text_fit(d, st, iw - 60, 27, done and abs(cur - i) < 0.5),\n                   fill=ink if done else mut, anchor=\"lm\")\n    elif k == \"statement\":\n        d.text((x, y + 24), wdg.get(\"title\", \"Recent activity\"), font=font(26, True), fill=mut, anchor=\"lm\")\n        for i, row in enumerate(wdg[\"rows\"]):\n            yy = y + 56 + 64 * i\n            show = ev.get(f\"row{i}\", 1.0)\n            if show <= 0:\n                continue\n            hl = row.get(\"highlight\") and show > 0\n            if hl:\n                d.rounded_rectangle([x - 10, yy, x + iw + 10, yy + 58], 12, fill=(232, 250, 240))\n            d.text((x, yy + 18), row.get(\"date\", \"\"), font=font(20), fill=mut, anchor=\"lm\")\n            d.text((x, yy + 42), row[\"text\"], font=text_fit(d, row[\"text\"], iw - 170, 24, hl), fill=ink, anchor=\"lm\")\n            amt = row.get(\"amount\", \"\")\n            d.text((x + iw, yy + 30), amt, font=font(26, True),\n                   fill=th[\"good\"] if amt.startswith(\"+\") else ink, anchor=\"rm\")\n    elif k == \"notice\":\n        lines = wrap(d, wdg[\"text\"], font(26), iw - 90)\n        hgt = 24 + 40 * len(lines)\n        d.rounded_rectangle([x, y, x + iw, y + hgt], 16, fill=(240, 247, 255))\n        icon(d, wdg.get(\"icon\", \"clock\"), x + 36, y + hgt / 2, 44, th, on_light=True)\n        for i, ln in enumerate(lines):\n            d.text((x + 72, y + 32 + 40 * i), ln, font=font(26), fill=ink, anchor=\"lm\")\n    elif k == \"text\":\n        f = font(wdg.get(\"size\", 28), wdg.get(\"bold\", False))\n        for i, ln in enumerate(wrap(d, wdg[\"text\"], f, iw)):\n            d.text((x, y + 8 + i * int(wdg.get(\"size\", 28) * 1.35)), ln, font=f, fill=ink if wdg.get(\"bold\") else mut)\n\n\ndef _ui_events(spec, t):\n    \"\"\"State of every widget at time t (fraction of the shot) from the spec's timeline.\"\"\"\n    st = {}\n    for e in spec.get(\"events\", []):\n        a = e[\"t\"]\n        if e[\"do\"] == \"tap\":\n            st.setdefault(e[\"target\"], {})[\"tap\"] = min(max((t - a) / 0.08, 0), 1) if t >= a else 0\n        elif e[\"do\"] == \"show\":\n            st.setdefault(e[\"target\"], {})[\"show\"] = prog(t, a, a + 0.08)\n        elif e[\"do\"] == \"select\":\n            st.setdefault(e[\"target\"], {})[\"hl\"] = 1.0 if t >= a else 0.0\n        elif e[\"do\"] == \"status\":\n            prev = st.get(e[\"target\"], {}).get(\"status\", e.get(\"from\", 0))\n            st.setdefault(e[\"target\"], {})[\"status\"] = prev + (e[\"to\"] - prev) * prog(t, a, a + e.get(\"dur\", 0.15))\n        elif e[\"do\"] == \"row\":\n            st.setdefault(e[\"target\"], {})[f\"row{e['row']}\"] = prog(t, a, a + 0.06)\n    return st\n\n\ndef ui_frame(spec, plan, t, w, h, bg, th):\n    \"\"\"One frame of a UI shot: a phone with the current screen, optional notification, optional caption.\"\"\"\n    img = bg.copy().convert(\"RGBA\")\n    d = ImageDraw.Draw(img)\n    pw, ph, top = PHONE[\"w\"], PHONE[\"h\"], PHONE[\"top\"]\n    # a step label sits top-left: move the phone right so they never overlap (the caption then goes on the left)\n    off = spec.get(\"offset_x\", 300 if spec.get(\"_has_label\") else 0)\n    px = w // 2 - pw // 2 + off\n    shadowed(img, [px - 18, top - 18, px + pw + 18, top + ph + 18], 64, (14, 16, 22, 255))\n    d = ImageDraw.Draw(img)\n    screens = spec[\"screens\"]\n    # which screen is showing: \"screen\" events switch with a short slide\n    cur, nxt, sp = 0, None, 0.0\n    for e in spec.get(\"events\", []):\n        if e[\"do\"] == \"screen\":\n            if t >= e[\"t\"] + 0.07:\n                cur = e[\"to\"]\n            elif t >= e[\"t\"]:\n                nxt, sp = e[\"to\"], prog(t, e[\"t\"], e[\"t\"] + 0.07)\n    ev = _ui_events(spec, t)\n    scr = Image.new(\"RGBA\", (pw, ph), th[\"card\"] + (255,))\n    for idx, slide in ([(cur, 0)] if nxt is None else [(cur, -sp * pw), (nxt, (1 - sp) * pw)]):\n        layer = Image.new(\"RGBA\", (pw, ph), th[\"card\"] + (255,))\n        ld = ImageDraw.Draw(layer)\n        ld.text((32, 22), spec.get(\"clock\", \"9:41\"), font=font(22, True), fill=th[\"ink\"], anchor=\"lm\")\n        ld.text((pw - 32, 22), \"\u25cf\u25cf\u25cf \u25ae\", font=font(18), fill=th[\"ink\"], anchor=\"rm\")\n        y, iw = 44, pw - 64\n        for wdg in screens[idx]:\n            e = ev.get(wdg.get(\"id\"), {})\n            show = e.get(\"show\", 0.0 if wdg.get(\"hidden\") else 1.0)\n            hgt = _widget_height(ld, wdg, iw)\n            if show >= 1:\n                _draw_widget(layer, ld, wdg, 32, y, iw, t, e, th)\n            elif show > 0:\n                wl = Image.new(\"RGBA\", (pw, ph), (0, 0, 0, 0))\n                _draw_widget(wl, ImageDraw.Draw(wl), wdg, 32, y + int(24 * (1 - show)), iw, t, e, th)\n                a = np.array(wl); a[..., 3] = (a[..., 3] * show).astype(np.uint8)\n                layer.alpha_composite(Image.fromarray(a))\n                ld = ImageDraw.Draw(layer)\n            y += hgt + 18\n        scr.alpha_composite(layer, (int(slide), 0))\n    mask = Image.new(\"L\", (pw, ph), 0)\n    ImageDraw.Draw(mask).rounded_rectangle([0, 0, pw - 1, ph - 1], 48, fill=255)\n    img.paste(scr, (px, top), mask)\n    for e in spec.get(\"events\", []):                            # notification banner slides down over the screen\n        if e[\"do\"] == \"notify\":\n            p = prog(t, e[\"t\"], e[\"t\"] + 0.06) * (1 - prog(t, e.get(\"until\", 2.0), e.get(\"until\", 2.0) + 0.06))\n            if p > 0:\n                bh = 150\n                by = top + 14 - int((1 - p) * (bh + 30))\n                shadowed(img, [px + 16, by, px + pw - 16, by + bh], 26, (255, 255, 255, 255), blur=12, offset=(0, 6))\n                d = ImageDraw.Draw(img)\n                icon(d, e.get(\"icon\", \"bank\"), px + 66, by + 52, 58, th, on_light=True)\n                d.text((px + 112, by + 36), e[\"title\"], font=text_fit(d, e[\"title\"], pw - 150, 26, True), fill=th[\"ink\"], anchor=\"lm\")\n                for i, ln in enumerate(wrap(d, e[\"body\"], font(23), pw - 150)[:3]):\n                    d.text((px + 112, by + 74 + 30 * i), ln, font=font(23), fill=th[\"muted\"], anchor=\"lm\")\n    if spec.get(\"caption\"):\n        _caption(img, spec[\"caption\"], t, w, h, th, side=\"right\" if off <= 0 else \"left\")\n    return np.asarray(img.convert(\"RGB\"))\n\n\ndef _caption(img, cap, t, w, h, th, side=\"right\"):\n    \"\"\"Short callout next to the phone (\u2264 6 words): what the viewer should notice.\"\"\"\n    p = prog(t, cap.get(\"t\", 0.1), cap.get(\"t\", 0.1) + 0.08)\n    if p <= 0:\n        return\n    d = ImageDraw.Draw(img)\n    f = font(46, True)\n    lines = wrap(d, cap[\"text\"], f, 520)\n    x = w // 2 + PHONE[\"w\"] // 2 + 80 if side == \"right\" else w // 2 - PHONE[\"w\"] // 2 - 600\n    y = cap.get(\"y\", 380)\n    layer = Image.new(\"RGBA\", img.size, (0, 0, 0, 0))\n    ld = ImageDraw.Draw(layer)\n    for i, ln in enumerate(lines):\n        ld.text((x + int(30 * (1 - p)), y + 62 * i), ln, font=f, fill=th[\"white\"] + (int(255 * p),))\n    ld.rectangle([x, y - 26, x + 90 * p, y - 18], fill=th[\"gold\"] + (int(255 * p),))\n    img.alpha_composite(layer)\n\n\n# ---------------------------------------------------------------- journey recipe: travel metaphor\ndef journey_layout(stations, w):\n    n = len(stations)\n    xs = [int(w * (0.12 + 0.76 * i / max(n - 1, 1))) for i in range(n)]\n    return {s[\"id\"]: (x, 500) for s, x in zip(stations, xs)}\n\n\n_static = {}\ndef journey_static(spec, plan, w, bg, th):\n    \"\"\"Road, station tiles and labels: identical in every frame of a shot, so drawn once.\"\"\"\n    key = (id(spec), id(bg))\n    if key in _static:\n        return _static[key]\n    J = plan[\"journey\"]\n    stations = J[\"stations\"]\n    pos = journey_layout(stations, w)\n    img = bg.copy().convert(\"RGBA\")\n    d = ImageDraw.Draw(img)\n    path = spec.get(\"path\", [])\n    # road\n    xs = [p[0] for p in pos.values()]\n    d.rounded_rectangle([min(xs) - 40, 488, max(xs) + 40, 512], 12, fill=(255, 255, 255, 40))\n    for x in range(min(xs), max(xs), 46):\n        d.line([x, 500, x + 20, 500], fill=(255, 255, 255, 120), width=4)\n    # which stations matter in this shot\n    focus = set(spec.get(\"focus\", path))\n    for s in stations:\n        x, y = pos[s[\"id\"]]\n        on = s[\"id\"] in focus\n        r = 104 if on else 84\n        shadowed(img, [x - r, y - r - 120, x + r, y + r - 120], 40, (255, 255, 255, 255) if on else (205, 212, 224, 255),\n                 blur=14, offset=(0, 8))\n        d = ImageDraw.Draw(img)\n        icon(d, s.get(\"icon\", \"store\"), x, y - 120, r * 1.25, th, active=on, on_light=True)\n        lab = s[\"label\"]\n        f = text_fit(d, lab, 300, 34 if on else 28, on)\n        for i, ln in enumerate(wrap(d, lab, f, 280)):\n            d.text((x, y + 70 + 40 * i), ln, font=f, fill=th[\"white\"] if on else (180, 190, 205), anchor=\"mm\")\n    _static.clear()\n    _static[key] = (img, pos)\n    return img, pos\n\n\ndef journey_frame(spec, plan, t, w, h, bg, th):\n    \"\"\"Stations in a row (same for every journey shot = consistent), the money/parcel token travelling along\n    spec[\"path\"], optional caption and business-day calendar.\"\"\"\n    J = plan[\"journey\"]\n    static, pos = journey_static(spec, plan, w, bg, th)\n    img = static.copy()\n    d = ImageDraw.Draw(img)\n    path = spec.get(\"path\", [])\n    t0, t1 = spec.get(\"move\", [0.1, 0.85])\n    # the token travelling along the path\n    if len(path) >= 2:\n        seg = len(path) - 1\n        p = prog(t, t0, t1) * seg\n        i = min(int(p), seg - 1)\n        f = p - i\n        (ax, ay), (bx, by) = pos[path[i]], pos[path[i + 1]]\n        tx, ty = ax + (bx - ax) * f, ay + (by - ay) * f - 10 - 26 * math.sin(math.pi * f)\n    elif path:\n        tx, ty = pos[path[0]][0], pos[path[0]][1] - 10\n    else:\n        tx = ty = None\n    if tx is not None:\n        tok = spec.get(\"token\", J.get(\"token\", \"money\"))\n        shadowed(img, [tx - 62, ty - 40, tx + 62, ty + 40], 14, (255, 255, 255, 255), blur=10, offset=(0, 6))\n        d = ImageDraw.Draw(img)\n        icon(d, tok, tx, ty, 108, th, on_light=True)\n    # caption\n    if spec.get(\"caption\"):\n        p = prog(t, 0.03, 0.12)\n        layer = Image.new(\"RGBA\", img.size, (0, 0, 0, 0))\n        ld = ImageDraw.Draw(layer)\n        f = text_fit(ld, spec[\"caption\"], w - 400, 54, True)\n        ld.text((w // 2, 215 + int(20 * (1 - p))), spec[\"caption\"], font=f, fill=th[\"white\"] + (int(255 * p),), anchor=\"mm\")\n        img.alpha_composite(layer)\n    # business-day calendar: days tick over; weekends greyed and skipped\n    cal = spec.get(\"calendar\")\n    if cal:\n        _calendar(img, cal, t, w, h, th)\n    return np.asarray(img.convert(\"RGB\"))\n\n\nDAYS = [\"Mon\", \"Tue\", \"Wed\", \"Thu\", \"Fri\", \"Sat\", \"Sun\"]\n\n\ndef _calendar(img, cal, t, w, h, th):\n    d = ImageDraw.Draw(img)\n    start = DAYS.index(cal.get(\"start\", \"Mon\"))\n    n = cal[\"days\"]\n    x0, y0, cw = w // 2 - 7 * 92 // 2, 700, 92\n    shown = int(prog(t, cal.get(\"t0\", 0.15), cal.get(\"t1\", 0.9)) * n + 0.0001)\n    biz = 0\n    for i in range(min(n, 14)):\n        dn = DAYS[(start + i) % 7]\n        weekend = dn in (\"Sat\", \"Sun\")\n        x = x0 + (i % 7) * cw\n        y = y0 + (i // 7) * 110\n        fill = (70, 82, 104) if weekend else (255, 255, 255)\n        if i < shown and not weekend:\n            fill = th[\"gold\"]\n        d.rounded_rectangle([x, y, x + cw - 12, y + 96], 14, fill=fill)\n        d.text((x + (cw - 12) / 2, y + 30), dn, font=font(24, True), fill=th[\"ink\"] if not weekend else (160, 170, 185), anchor=\"mm\")\n        if not weekend and i < shown:\n            biz += 1\n            d.text((x + (cw - 12) / 2, y + 66), str(biz), font=font(30, True), fill=th[\"ink\"], anchor=\"mm\")\n        elif weekend:\n            d.text((x + (cw - 12) / 2, y + 66), \"closed\", font=font(18), fill=(160, 170, 185), anchor=\"mm\")\n    if cal.get(\"label\"):\n        d.text((w // 2, y0 - 34), cal[\"label\"], font=font(30, True), fill=th[\"white\"], anchor=\"mm\")\n\n\n# ---------------------------------------------------------------- card recipe: hook question / recap\ndef card_frame(spec, plan, t, w, h, bg, th):\n    img = bg.copy().convert(\"RGBA\")\n    layer = Image.new(\"RGBA\", img.size, (0, 0, 0, 0))\n    d = ImageDraw.Draw(layer)\n    lines = spec.get(\"lines\", [])\n    reveal = spec.get(\"reveal\", len(lines) > 1)\n    y = h * 0.5 - (len(lines) * 92) / 2 + (40 if spec.get(\"icon\") else 0)\n    if spec.get(\"icon\"):\n        p = prog(t, 0.02, 0.1)\n        icon(d, spec[\"icon\"], w // 2, int(y - 120 - 20 * (1 - p)), int(150 * (0.6 + 0.4 * p)), th)\n    for i, ln in enumerate(lines):\n        a = (0.06 + 0.8 * i / max(len(lines), 1)) if reveal else 0.03\n        p = prog(t, a, a + 0.08)\n        if p <= 0:\n            continue\n        big = i == 0 and spec.get(\"headline\", True)\n        f = text_fit(d, ln, w - 300, 74 if big else 52, big)\n        col = th[\"white\"] if big else (225, 232, 242)\n        d.text((w // 2, y + 92 * i + int(18 * (1 - p))), ln, font=f, fill=col + (int(255 * p),), anchor=\"mm\")\n        if spec.get(\"bullets\") and not big:\n            tw = d.textlength(ln, font=f)\n            d.ellipse([w // 2 - tw / 2 - 40, y + 92 * i - 10, w // 2 - tw / 2 - 20, y + 92 * i + 10],\n                      fill=th[\"gold\"] + (int(255 * p),))\n    img.alpha_composite(layer)\n    return np.asarray(img.convert(\"RGB\"))\n\n\nRENDERERS = {\"ui\": ui_frame, \"journey\": journey_frame, \"card\": card_frame}\n\n\nDESIGN_W, DESIGN_H = 1920, 1080                      # every layout is designed at 1080p, then scaled\n\n\ndef renderer(kind):\n    \"\"\"Frame function of a drawn kind. The two-friend kinds live in their own modules (loaded on first use).\"\"\"\n    if kind not in RENDERERS:\n        if kind == \"character\":\n            import charanim\n            RENDERERS[kind] = charanim.character_frame\n        elif kind in (\"demo\", \"photo\"):\n            import demos\n            RENDERERS.update(demo=demos.demo_frame, photo=demos.photo_frame)\n    return RENDERERS[kind]\n\n\ndef render(kind, spec, plan, n_frames, fps, w, h):\n    \"\"\"Yields n_frames RGB frames (h x w x 3, uint8) for a drawn shot.\n    kind \"story\": spec[\"visuals\"] is a list of sub-visuals, each {\"recipe\": kind, kind: spec, \"frames\": n}, drawn\n    one after the other (the visual timeline of one dialogue scene; the audio is not affected).\"\"\"\n    if kind == \"story\":\n        done = 0\n        for v in spec[\"visuals\"]:\n            n = min(int(v[\"frames\"]), n_frames - done)\n            if n > 0:\n                yield from render(v[\"recipe\"], dict(v[v[\"recipe\"]], _n=n), plan, n, fps, w, h)\n                done += n\n        if done < n_frames:                              # never short: hold the last visual's final frame\n            last = spec[\"visuals\"][-1]\n            f = list(render(last[\"recipe\"], dict(last[last[\"recipe\"]], _n=1), plan, 1, fps, w, h))[0]\n            for _ in range(n_frames - done):\n                yield f\n        return\n    th = theme(plan)\n    bg = background(DESIGN_W, DESIGN_H, th)\n    fn = renderer(kind)\n    spec = dict(spec, _n=spec.get(\"_n\", n_frames))\n    native = kind == \"character\" and bool((plan or {}).get(\"visual_style_version\"))   # storybook: drawn at w x h\n    for i in range(n_frames):\n        if native:\n            yield fn(spec, plan, i / max(n_frames - 1, 1), w, h, bg, th)\n            continue\n        f = fn(spec, plan, i / max(n_frames - 1, 1), DESIGN_W, DESIGN_H, bg, th)\n        if (w, h) != (DESIGN_W, DESIGN_H):\n            f = np.asarray(Image.fromarray(f).resize((w, h), Image.LANCZOS))\n        yield f\n\n\ndef check_spec(kind, spec, plan):\n    \"\"\"Problems with a drawn-shot spec (used by the plan validation). Returns a list of messages.\"\"\"\n    out = []\n    if kind == \"ui\":\n        if not spec.get(\"screens\"):\n            out.append(\"ui shot needs 'screens' (a list of screens, each a list of widgets)\")\n        ids = {w.get(\"id\") for sc in spec.get(\"screens\", []) for w in sc}\n        for e in spec.get(\"events\", []):\n            if e[\"do\"] in (\"tap\", \"show\", \"select\", \"status\", \"row\") and e.get(\"target\") not in ids:\n                out.append(f\"event '{e['do']}' targets unknown widget '{e.get('target')}'\")\n            if not 0 <= e.get(\"t\", -1) <= 1:\n                out.append(f\"event time t={e.get('t')} must be a fraction 0..1 of the shot\")\n            if e[\"do\"] == \"screen\" and not 0 <= e.get(\"to\", -1) < len(spec.get(\"screens\", [])):\n                out.append(f\"screen event goes to missing screen {e.get('to')}\")\n        for sc in spec.get(\"screens\", []):\n            for wdg in sc:\n                if wdg.get(\"type\") not in (\"header\", \"product\", \"button\", \"choice\", \"status\", \"statement\", \"notice\", \"text\"):\n                    out.append(f\"unknown widget type '{wdg.get('type')}'\")\n    elif kind == \"journey\":\n        st = {s[\"id\"] for s in plan.get(\"journey\", {}).get(\"stations\", [])}\n        if not st:\n            out.append(\"journey shots need a plan-level 'journey': {'stations': [...]}\")\n        for sid in spec.get(\"path\", []) + spec.get(\"focus\", []):\n            if sid not in st:\n                out.append(f\"journey path uses unknown station '{sid}'\")\n    elif kind == \"character\":\n        import charanim\n        out += charanim.check_spec(spec, plan)\n    elif kind in (\"demo\", \"photo\"):\n        import demos\n        out += demos.check_spec(kind, spec, plan)\n    elif kind == \"story\":\n        if not spec.get(\"visuals\"):\n            out.append(\"story shot needs 'visuals'\")\n        for v in spec.get(\"visuals\", []):\n            r = v.get(\"recipe\")\n            if r not in (\"character\", \"demo\", \"photo\", \"card\", \"ui\", \"journey\"):\n                out.append(f\"story visual '{v.get('id')}': recipe '{r}' can't be drawn inside a scene\")\n            else:\n                out += [f\"visual {v.get('id')}: {m}\" for m in check_spec(r, v.get(r, {}), plan)]\n    elif kind == \"card\":\n        if not spec.get(\"lines\"):\n            out.append(\"card shot needs 'lines'\")\n        for ln in spec.get(\"lines\", []):\n            if len(ln.split()) > 9:\n                out.append(f\"card line too long for the screen ({len(ln.split())} words): '{ln}'\")\n    return out\n", "envfactory.py": "\"\"\"Cinematic Environment Factory: reusable, versioned, layered places for the storybook characters.\n\nAn environment pack (one folder per version, never rewritten):\n  <root>/environments/<env>/<version>/manifest.json   layers (file, depth, sha256), camera presets, lighting,\n                                                      character slots, subtitle/diagram safe regions, provenance\n  <root>/environments/<env>/<version>/layers/*.png    master-size RGBA layers, back to front\n\nEach pack has one or more SETS (a viewpoint of the same place: e.g. the cinema seen from the screen toward the\nseats, and from behind the seats toward the screen). A camera preset frames a set: a crop of the master, a\ndefault movement, the character slots (position, scale, view, seated), and the parallax depth of each layer.\nThe same place is reused for every shot of a video and across videos: different framings, never a different\ncinema per shot. Provisional packs are drawn by code here (status \"provisional\"); the Environment Studio can\nreplace them with approved generated art in the same format (each layer generated or cut separately \u2014 a\nflattened image is never split into fake parallax layers).\n\"\"\"\nfrom __future__ import annotations\n\nimport datetime, hashlib, json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFilter\n\nENV_ART_VERSION = \"storybook-env-procedural-1\"\nMASTER = (2400, 1350)                    # master size of every set (16:9; presets crop it)\n\n\ndef _sha(b):\n    return hashlib.sha256(b).hexdigest()\n\n\ndef _grad(w, h, top, bot, power=1.0):\n    t = (np.linspace(0, 1, h) ** power)[:, None, None]\n    a = np.array(top, np.float32)[None, None, :]\n    b = np.array(bot, np.float32)[None, None, :]\n    return Image.fromarray(np.repeat((a + (b - a) * t).astype(np.uint8), w, 1), \"RGB\").convert(\"RGBA\")\n\n\ndef _blur_layer(im, r):\n    return im.filter(ImageFilter.GaussianBlur(r))\n\n\ndef _glow(size, box, color, alpha, blur):\n    g = Image.new(\"RGBA\", size, color + (0,))\n    m = Image.new(\"L\", size, 0)\n    ImageDraw.Draw(m).ellipse(box, fill=int(255 * alpha))\n    g.putalpha(m.filter(ImageFilter.GaussianBlur(blur)))\n    return g\n\n\ndef _seat_row(w, y, seat_w, seat_h, color, n_off=0.0, arm=True, back_shine=0.12):\n    \"\"\"One row of cinema seats (soft rounded backs with a light rim from the screen).\"\"\"\n    im = Image.new(\"RGBA\", (w, int(y + seat_h * 1.6)), (0, 0, 0, 0))\n    d = ImageDraw.Draw(im)\n    pitch = seat_w * 1.14\n    x = -pitch * n_off\n    dark = tuple(int(c * 0.62) for c in color)\n    while x < w + pitch:\n        d.rounded_rectangle([x, y, x + seat_w, y + seat_h], radius=int(seat_w * 0.32), fill=color)\n        d.rounded_rectangle([x + seat_w * 0.1, y + seat_h * 0.62, x + seat_w * 0.9, y + seat_h * 1.15],\n                            radius=int(seat_w * 0.2), fill=dark)\n        d.rounded_rectangle([x + seat_w * 0.12, y + seat_h * 0.06, x + seat_w * 0.88, y + seat_h * 0.16],\n                            radius=int(seat_w * 0.1), fill=tuple(min(255, int(c * (1 + back_shine * 3))) for c in color))\n        x += pitch\n    return im\n\n\n# ------------------------------------------------------------------------------------------- the packs\ndef draw_cinema():\n    W, H = MASTER\n    L = {}\n    # SET \"seats\": we look from the screen toward the audience; the friends sit in a middle row\n    back = _grad(W, H, (26, 20, 44), (54, 24, 40), 1.2)\n    d = ImageDraw.Draw(back)\n    for i in range(9):                                     # wall panels with warm sconces\n        x = i * 290 - 60\n        d.rounded_rectangle([x, 70, x + 250, 520], radius=30, fill=(40, 30, 58))\n        back.alpha_composite(_glow((W, H), [x + 95, 120, x + 155, 180], (255, 196, 120), 0.9, 30))\n    back.alpha_composite(_glow((W, H), [W * 0.1, -400, W * 0.9, 500], (120, 150, 255), 0.35, 160))    # projector haze\n    beam = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))\n    ImageDraw.Draw(beam).polygon([(W * 0.46, 0), (W * 0.54, 0), (W * 0.95, H), (W * 0.05, H)], fill=(170, 190, 255, 22))\n    back.alpha_composite(_blur_layer(beam, 20))\n    L[\"seats_back\"] = back\n    rows = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))\n    for k, (y, sw, sh_, col) in enumerate([(430, 120, 150, (96, 28, 48)), (560, 150, 190, (120, 32, 54))]):\n        r = _seat_row(W, y, sw, sh_, col, n_off=0.5 * k)\n        rows.alpha_composite(_blur_layer(r, 3 - k), (0, 0))\n    for x in range(40, W, 210):                            # a few audience heads in the back rows\n        rows.alpha_composite(_glow((W, H), [x, 430, x + 95, 530], (16, 12, 24), 1.0, 2))\n    L[\"seats_rows\"] = rows\n    own = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))          # the friends' own row (behind them)\n    own.alpha_composite(_seat_row(W, 720, 300, 330, (150, 38, 62), n_off=0.25, back_shine=0.16))\n    L[\"seats_own_row\"] = own\n    fg = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))           # the row in front of them: hides legs, gives depth\n    fg.alpha_composite(_seat_row(W, 1110, 420, 380, (70, 18, 32), n_off=0.1, back_shine=0.05))\n    L[\"seats_fg\"] = _blur_layer(fg, 6)\n    light = _glow((W, H), [W * 0.15, 300, W * 0.85, 1500], (140, 170, 255), 0.10, 220)   # screen light on them\n    L[\"seats_light\"] = light\n    # SET \"screen\": from behind the friends toward the big screen (the movie is drawn live into `screen_rect`)\n    sb = _grad(W, H, (10, 8, 20), (24, 14, 26))\n    d = ImageDraw.Draw(sb)\n    d.rectangle([0, 0, W, 120], fill=(36, 14, 30))\n    for x in (0, W - 260):                                 # side curtains\n        for k in range(6):\n            d.rectangle([x + k * 44, 0, x + k * 44 + 30, H], fill=(110 - 6 * k, 18, 34))\n    sb.alpha_composite(_glow((W, H), [300, 60, W - 300, 900], (170, 185, 255), 0.45, 120))\n    L[\"screen_back\"] = sb\n    aud = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))\n    for r, (y, s) in enumerate([(930, 70), (1010, 92), (1110, 120)]):\n        for i in range(int(W / (s * 1.25)) + 2):\n            x = i * s * 1.25 - (r % 2) * s * 0.6\n            aud.alpha_composite(_glow((W, H), [x, y, x + s, y + s * 1.2], (14, 10, 20), 1.0, 2))\n    L[\"screen_audience\"] = aud\n    fgs = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))\n    fgs.alpha_composite(_seat_row(W, 1240, 460, 300, (40, 12, 22), back_shine=0.03))\n    L[\"screen_fg\"] = _blur_layer(fgs, 8)\n    return L\n\n\ndef draw_lobby():\n    W, H = MASTER\n    L = {}\n    back = _grad(W, H, (64, 32, 76), (130, 56, 78), 1.1)\n    d = ImageDraw.Draw(back)\n    d.rectangle([0, 960, W, H], fill=(84, 34, 46))\n    for k in range(17):                                    # marquee bulbs\n        back.alpha_composite(_glow((W, H), [60 + k * 140, 70, 92 + k * 140, 102], (255, 225, 150), 1.0, 6))\n    for i, (x, col) in enumerate(((180, (60, 120, 200)), (640, (200, 80, 90)), (1560, (80, 160, 120)), (2020, (230, 160, 60)))):\n        d.rounded_rectangle([x, 180, x + 300, 640], radius=14, fill=(250, 222, 130))\n        d.rounded_rectangle([x + 22, 204, x + 278, 520], radius=8, fill=col)\n        d.ellipse([x + 90, 280, x + 210, 400], fill=tuple(min(255, c + 60) for c in col))\n        d.rectangle([x + 40, 545, x + 260, 562], fill=(110, 70, 50)); d.rectangle([x + 80, 585, x + 220, 598], fill=(110, 70, 50))\n    for x in range(0, W, 160):                             # floor tiles (perspective lines)\n        d.line([(W / 2 + (x - W / 2) * 0.55, 960), (x - (W / 2 - x) * 0.8, H)], fill=(96, 40, 52), width=3)\n    back.alpha_composite(_glow((W, H), [700, 0, 1700, 700], (255, 200, 170), 0.25, 160))\n    L[\"lobby_back\"] = back\n    fg = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))\n    d = ImageDraw.Draw(fg)\n    d.rounded_rectangle([W - 330, 620, W + 40, 1180], radius=28, fill=(200, 42, 56))     # popcorn stand (foreground)\n    d.rounded_rectangle([W - 300, 560, W - 40, 660], radius=20, fill=(250, 220, 120))\n    for k in range(9):\n        fg.alpha_composite(_glow((W, H), [W - 290 + k * 28, 520 - (k % 3) * 18, W - 250 + k * 28, 566], (255, 240, 190), 1.0, 1))\n    L[\"lobby_fg\"] = _blur_layer(fg, 4)\n    return L\n\n\ndef draw_studio():\n    W, H = MASTER\n    back = _grad(W, H, (48, 82, 130), (22, 44, 76))\n    d = ImageDraw.Draw(back)\n    d.rectangle([0, 980, W, H], fill=(30, 52, 86))\n    back.alpha_composite(_glow((W, H), [600, 100, 1800, 900], (150, 200, 255), 0.35, 200))\n    for x in range(120, W, 340):\n        back.alpha_composite(_glow((W, H), [x, 160, x + 120, 280], (255, 255, 255), 0.18, 30))\n    return {\"studio_back\": back}\n\n\ndef draw_space_observatory():\n    W, H = MASTER\n    rng = np.random.default_rng(7)\n    sky = _grad(W, H, (6, 8, 26), (22, 14, 48))\n    d = ImageDraw.Draw(sky)\n    for _ in range(900):\n        x, y, r = rng.uniform(0, W), rng.uniform(0, H * 0.8), rng.choice([1, 1, 1, 2, 3])\n        b = int(rng.uniform(150, 255))\n        d.ellipse([x - r, y - r, x + r, y + r], fill=(b, b, min(255, b + 20), 255))\n    sky.alpha_composite(_glow((W, H), [1300, 120, 2100, 600], (170, 90, 220), 0.55, 120))   # a distant galaxy\n    sky.alpha_composite(_glow((W, H), [1550, 260, 1850, 420], (255, 220, 255), 0.8, 50))\n    room = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))\n    d = ImageDraw.Draw(room)\n    for i in range(5):                                     # the big window frame\n        x = 120 + i * 540\n        d.rectangle([x - 22, 0, x + 22, 980], fill=(40, 48, 78))\n    d.arc([-200, -500, W + 200, 900], 200, 340, fill=(40, 48, 78), width=40)\n    d.rectangle([0, 960, W, H], fill=(34, 40, 66))\n    for x in range(0, W, 120):\n        d.line([(x, 960), (x + (x - W / 2) * 0.6, H)], fill=(44, 52, 84), width=3)\n    L = {\"obs_sky\": sky, \"obs_room\": room}\n    holo = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))         # holographic display on a pedestal (centre)\n    d = ImageDraw.Draw(holo)\n    d.rounded_rectangle([1080, 900, 1320, 1010], radius=18, fill=(60, 72, 110))\n    holo.alpha_composite(_glow((W, H), [960, 480, 1440, 960], (90, 220, 255), 0.45, 40))\n    for k in range(6):\n        d.ellipse([1200 - 40 * k, 720 - 14 * k, 1200 + 40 * k, 720 + 14 * k], outline=(140, 240, 255, 170), width=3)\n    L[\"obs_holo\"] = holo\n    fg = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))\n    ImageDraw.Draw(fg).rounded_rectangle([-100, 1180, 700, 1500], radius=40, fill=(24, 28, 50))   # console edge\n    L[\"obs_fg\"] = _blur_layer(fg, 6)\n    return L\n\n\nPACKS = {\n    \"cinema\": {\n        \"draw\": draw_cinema, \"topic\": \"movies, 3D films, light, vision\",\n        \"sets\": {\n            \"seats\": {\"layers\": [(\"seats_back\", 0.15), (\"seats_rows\", 0.35), (\"seats_own_row\", 0.6), (\"@characters\", 0.75),\n                                 (\"seats_light\", 0.75), (\"seats_fg\", 1.25)],\n                      \"lighting\": {\"key\": \"front (the screen)\", \"tint\": [205, 215, 255], \"tint_strength\": 0.15,\n                                   \"rim\": [150, 175, 255], \"temperature\": \"cool\"}},\n            \"screen\": {\"layers\": [(\"screen_back\", 0.1), (\"@screen\", 0.1), (\"screen_audience\", 0.5), (\"@characters\", 0.8),\n                                  (\"screen_fg\", 1.2)],\n                       \"screen_rect\": [360, 110, 2040, 900],\n                       \"lighting\": {\"key\": \"back (the screen)\", \"tint\": [150, 160, 220], \"tint_strength\": 0.4,\n                                    \"rim\": [200, 210, 255], \"temperature\": \"cool\"}},\n        },\n        \"presets\": {\n            \"seats_two_shot\": {\"set\": \"seats\", \"crop\": [180, 180, 2040, 1148], \"move\": \"slow_push_in\",\n                               \"slots\": {\"arjun\": [820, 1746, 0.85], \"meera\": [1580, 1746, 0.85]}, \"seated\": True},\n            \"seats_wide\": {\"set\": \"seats\", \"crop\": [0, 0, 2400, 1350], \"move\": \"static\",\n                           \"slots\": {\"arjun\": [900, 1471, 0.7], \"meera\": [1500, 1471, 0.7]}, \"seated\": True},\n            \"seats_closeup_arjun\": {\"set\": \"seats\", \"crop\": [520, 330, 1300, 731], \"move\": \"slow_push_in\",\n                                    \"slots\": {\"arjun\": [1170, 1435, 0.6], \"meera\": [1990, 1435, 0.6]}, \"seated\": True},\n            \"seats_closeup_meera\": {\"set\": \"seats\", \"crop\": [960, 330, 1300, 731], \"move\": \"slow_push_in\",\n                                    \"slots\": {\"meera\": [1610, 1435, 0.6], \"arjun\": [790, 1435, 0.6]}, \"seated\": True},\n            \"screen_back\": {\"set\": \"screen\", \"crop\": [0, 0, 2400, 1350], \"move\": \"slow_push_in\",\n                            \"slots\": {\"arjun\": [1020, 1820, 0.55], \"meera\": [1380, 1820, 0.55]}, \"view\": \"back\"},\n        },\n        \"map\": {\"cinema_seats\": {\"two_shot\": \"seats_two_shot\", \"wide\": \"seats_wide\", \"establishing\": \"seats_wide\",\n                                 \"closeup\": \"seats_closeup_{subject}\", \"reaction\": \"seats_closeup_{subject}\",\n                                 \"ots\": \"seats_closeup_{subject}\"},\n                \"cinema_screen\": {\"*\": \"screen_back\"}},\n    },\n    \"lobby\": {\n        \"draw\": draw_lobby, \"topic\": \"after the movie, everyday places\",\n        \"sets\": {\"main\": {\"layers\": [(\"lobby_back\", 0.2), (\"@characters\", 0.8), (\"lobby_fg\", 1.3)],\n                          \"lighting\": {\"key\": \"warm overhead\", \"tint\": [255, 226, 205], \"tint_strength\": 0.14,\n                                       \"rim\": [255, 210, 170], \"temperature\": \"warm\"}}},\n        \"presets\": {\n            \"lobby_two_shot\": {\"set\": \"main\", \"crop\": [240, 120, 1920, 1080], \"move\": \"slow_push_in\",\n                               \"slots\": {\"arjun\": [900, 1506, 0.75], \"meera\": [1500, 1506, 0.75]}},\n            \"lobby_wide\": {\"set\": \"main\", \"crop\": [0, 0, 2400, 1350], \"move\": \"static\",\n                           \"slots\": {\"arjun\": [950, 1298, 0.55], \"meera\": [1450, 1298, 0.55]}},\n            \"lobby_closeup_arjun\": {\"set\": \"main\", \"crop\": [520, 220, 1300, 731], \"move\": \"slow_push_in\",\n                                    \"slots\": {\"arjun\": [1170, 1325, 0.6], \"meera\": [1990, 1325, 0.6]}},\n            \"lobby_closeup_meera\": {\"set\": \"main\", \"crop\": [960, 220, 1300, 731], \"move\": \"slow_push_in\",\n                                    \"slots\": {\"meera\": [1610, 1325, 0.6], \"arjun\": [790, 1325, 0.6]}},\n        },\n        \"map\": {\"lobby\": {\"two_shot\": \"lobby_two_shot\", \"wide\": \"lobby_wide\", \"establishing\": \"lobby_wide\",\n                          \"closeup\": \"lobby_closeup_{subject}\", \"reaction\": \"lobby_closeup_{subject}\",\n                          \"ots\": \"lobby_closeup_{subject}\"}},\n    },\n    \"studio\": {\n        \"draw\": draw_studio, \"topic\": \"neutral explainer room\",\n        \"sets\": {\"main\": {\"layers\": [(\"studio_back\", 0.2), (\"@characters\", 0.8)],\n                          \"lighting\": {\"key\": \"soft front\", \"tint\": [230, 238, 255], \"tint_strength\": 0.08,\n                                       \"rim\": [200, 220, 255], \"temperature\": \"neutral\"}}},\n        \"presets\": {\n            \"studio_two_shot\": {\"set\": \"main\", \"crop\": [240, 120, 1920, 1080], \"move\": \"static\",\n                                \"slots\": {\"arjun\": [900, 1506, 0.75], \"meera\": [1500, 1506, 0.75]}},\n            \"studio_closeup_arjun\": {\"set\": \"main\", \"crop\": [520, 220, 1300, 731], \"move\": \"slow_push_in\",\n                                     \"slots\": {\"arjun\": [1170, 1325, 0.6], \"meera\": [1990, 1325, 0.6]}},\n            \"studio_closeup_meera\": {\"set\": \"main\", \"crop\": [960, 220, 1300, 731], \"move\": \"slow_push_in\",\n                                     \"slots\": {\"meera\": [1610, 1325, 0.6], \"arjun\": [790, 1325, 0.6]}},\n        },\n        \"map\": {\"studio\": {\"*\": \"studio_two_shot\", \"closeup\": \"studio_closeup_{subject}\",\n                           \"reaction\": \"studio_closeup_{subject}\"},\n                \"living_room\": {\"*\": \"studio_two_shot\", \"closeup\": \"studio_closeup_{subject}\",\n                                \"reaction\": \"studio_closeup_{subject}\"}},\n    },\n    \"space_observatory\": {\n        \"draw\": draw_space_observatory, \"topic\": \"space, black holes, gravity, stars\",\n        \"sets\": {\"main\": {\"layers\": [(\"obs_sky\", 0.05), (\"obs_room\", 0.3), (\"obs_holo\", 0.45), (\"@characters\", 0.8),\n                                     (\"obs_fg\", 1.3)],\n                          \"lighting\": {\"key\": \"cool from the hologram\", \"tint\": [170, 200, 255], \"tint_strength\": 0.3,\n                                       \"rim\": [120, 230, 255], \"temperature\": \"cool\"}}},\n        \"presets\": {\n            \"obs_two_shot\": {\"set\": \"main\", \"crop\": [120, 80, 2160, 1215], \"move\": \"slow_push_in\",\n                             \"slots\": {\"arjun\": [760, 1540, 0.75], \"meera\": [1640, 1540, 0.75]}},\n            \"obs_wide\": {\"set\": \"main\", \"crop\": [0, 0, 2400, 1350], \"move\": \"pan_right\",\n                         \"slots\": {\"arjun\": [700, 1380, 0.6], \"meera\": [1700, 1380, 0.6]}},\n        },\n        \"map\": {\"space_observatory\": {\"*\": \"obs_two_shot\", \"wide\": \"obs_wide\", \"establishing\": \"obs_wide\"}},\n    },\n}\nSAFE = {\"subtitles\": [0, 0.80, 1.0, 1.0], \"diagram\": [0.08, 0.06, 0.92, 0.76]}   # fractions of the frame\n\n\ndef build_environment(root, env, version=\"v1\", log=print):\n    pack = PACKS[env]\n    vdir = Path(root) / \"environments\" / env / version\n    if (vdir / \"manifest.json\").exists():\n        raise FileExistsError(f\"{vdir} already exists: environment versions are immutable\")\n    tmp = vdir.with_name(version + \".partial\")\n    (tmp / \"layers\").mkdir(parents=True, exist_ok=True)\n    layers = {}\n    for name, im in pack[\"draw\"]().items():\n        f = tmp / \"layers\" / f\"{name}.png\"\n        im.save(f, optimize=True)\n        layers[name] = {\"file\": f\"layers/{name}.png\", \"size\": list(im.size), \"sha256\": _sha(f.read_bytes())}\n    man = {\"environment\": env, \"version\": version, \"status\": \"provisional\", \"art_version\": ENV_ART_VERSION,\n           \"master\": list(MASTER), \"topic\": pack[\"topic\"], \"sets\": pack[\"sets\"], \"presets\": pack[\"presets\"],\n           \"map\": pack[\"map\"], \"safe_regions\": SAFE, \"layers\": layers,\n           \"provenance\": {\"method\": \"procedural drawing (colab/engine/lib/envfactory.py)\", \"model\": None,\n                          \"license\": \"project-original (code-drawn)\",\n                          \"created\": datetime.datetime.now().isoformat(timespec=\"seconds\"), \"approved_by\": None}}\n    (tmp / \"manifest.json\").write_text(json.dumps(man, indent=1))\n    tmp.rename(vdir)\n    log(f\"environment {env} {version}: {len(layers)} layers written to {vdir}\")\n    return vdir\n\n\ndef verify(vdir):\n    vdir = Path(vdir)\n    try:\n        man = json.loads((vdir / \"manifest.json\").read_text())\n    except Exception as e:\n        return [f\"manifest unreadable: {e!r}\"]\n    bad = []\n    for name, L in man[\"layers\"].items():\n        f = vdir / L[\"file\"]\n        if not f.exists():\n            bad.append(f\"missing layer {name}\")\n        elif _sha(f.read_bytes()) != L[\"sha256\"]:\n            bad.append(f\"layer {name} changed since the manifest was written\")\n    for s, st in man[\"sets\"].items():\n        for lname, _ in st[\"layers\"]:\n            if not lname.startswith(\"@\") and lname not in man[\"layers\"]:\n                bad.append(f\"set {s} uses missing layer {lname}\")\n    return bad\n\n\nclass Environment:\n    def __init__(self, vdir):\n        self.dir = Path(vdir)\n        self.man = json.loads((self.dir / \"manifest.json\").read_text())\n        self.id, self.version = self.man[\"environment\"], self.man[\"version\"]\n        self._img, self._scaled = {}, {}\n\n    def img(self, name):\n        if name not in self._img:\n            self._img[name] = Image.open(self.dir / self.man[\"layers\"][name][\"file\"]).convert(\"RGBA\")\n        return self._img[name]\n\n    def preset_for(self, world, shot, subject):\n        \"\"\"Which camera preset frames (world, shot) \u2014 or None if this pack doesn't cover that world.\"\"\"\n        m = self.man[\"map\"].get(world)\n        if m is None:\n            return None\n        p = m.get(shot) or m.get(\"*\")\n        if p is None:\n            return None\n        p = p.format(subject=subject or \"arjun\")\n        return p if p in self.man[\"presets\"] else None\n\n\n_ENV = {}\n\n\ndef load(env, version, root):\n    key = (str(root), env, version)\n    if key in _ENV:\n        return _ENV[key]\n    vdir = Path(root) / \"environments\" / env / version\n    if not (vdir / \"manifest.json\").exists():\n        raise FileNotFoundError(f\"environment {env} {version} not found in {root}\")\n    probs = verify(vdir)\n    if probs:\n        raise ValueError(f\"environment {env} {version} is damaged: {probs[:3]}\")\n    _ENV[key] = Environment(vdir)\n    return _ENV[key]\n\n\ndef ensure(env, version, root, log=print):\n    vdir = Path(root) / \"environments\" / env / version\n    if not (vdir / \"manifest.json\").exists():\n        build_environment(root, env, version, log=log)\n    return load(env, version, root)\n\n\nWORLD_ENV = {\"cinema_seats\": \"cinema\", \"cinema_screen\": \"cinema\", \"lobby\": \"lobby\", \"studio\": \"studio\",\n             \"living_room\": \"studio\", \"space_observatory\": \"space_observatory\"}\n\n\ndef camera(preset, u, move=None):\n    \"\"\"Crop box (x, y, w, h) in master pixels at shot fraction u, for a restrained camera move.\"\"\"\n    x, y, w, h = preset[\"crop\"]\n    move = move or preset.get(\"move\", \"static\")\n    if move == \"slow_push_in\":\n        z = 1.0 + 0.05 * u\n        return (x + w * (1 - 1 / z) / 2, y + h * (1 - 1 / z) / 2, w / z, h / z)\n    if move == \"slow_pull_back\":\n        z = 1.05 - 0.05 * u\n        return (x + w * (1 - 1 / z) / 2, y + h * (1 - 1 / z) / 2, w / z, h / z)\n    if move in (\"pan_right\", \"pan_left\"):\n        dx = 0.04 * w * (u - 0.5) * (1 if move == \"pan_right\" else -1)\n        return (x + dx, y, w, h)\n    return (x, y, w, h)\n\n\n# ------------------------------------------------------------------------------------------- generated plates (v2+)\nPLATE_PROMPTS = {   # one plate per set; the camera presets of the base version are kept (same crops and slots)\n    (\"cinema\", \"seats\"): (\"Cinematic 3D animated-film background: the inside of a cosy movie theatre seen from the \"\n                          \"screen toward rows of plush red velvet seats, warm wall sconces, soft blue light from the \"\n                          \"projector screen on the seats, shallow depth of field, no people, no text. 16:9.\"),\n    (\"cinema\", \"screen\"): (\"Cinematic 3D animated-film background: a cosy movie theatre seen from behind the seats \"\n                           \"toward a large glowing BLANK white cinema screen with red curtains, silhouettes of seat \"\n                           \"backs in the foreground, no people, no text. 16:9.\"),\n    (\"lobby\", \"main\"): (\"Cinematic 3D animated-film background: a warm, colourful cinema lobby with movie posters \"\n                        \"(no readable text), marquee light bulbs, a popcorn stand, polished floor, no people. 16:9.\"),\n    (\"studio\", \"main\"): (\"Cinematic 3D animated-film background: a calm, softly lit modern learning studio with \"\n                         \"warm wood and teal accents, soft window light, no people, no text. 16:9.\"),\n    (\"space_observatory\", \"main\"): (\"Cinematic 3D animated-film background: a futuristic observatory with a huge \"\n                                    \"window onto stars and a colourful galaxy, a glowing hologram pedestal in the \"\n                                    \"middle, no people, no text. 16:9.\"),\n}\n\n\ndef import_plates(root, env, version, plates, base_version=\"v1\", provenance=None, screen_rects=None, log=print):\n    \"\"\"A NEW environment version whose back layers are generated plates ({set: path to a 16:9 image}), keeping the\n    base version's camera presets, character slots, lighting, world map and safe regions \u2014 existing plans and\n    camera framing keep working. Sets without a plate keep the base layers. The code-drawn foreground of a set\n    that gets a plate is dropped (it would not match the painted plate); screen_rects can re-place the live movie.\"\"\"\n    import datetime, shutil\n    base = json.loads((Path(root) / \"environments\" / env / base_version / \"manifest.json\").read_text())\n    vdir = Path(root) / \"environments\" / env / version\n    if (vdir / \"manifest.json\").exists():\n        raise FileExistsError(f\"{vdir} exists: environment versions are immutable\")\n    tmp = vdir.with_name(version + \".partial\")\n    shutil.rmtree(tmp, ignore_errors=True)\n    (tmp / \"layers\").mkdir(parents=True)\n    layers, sets = {}, json.loads(json.dumps(base[\"sets\"]))\n    for set_name, st in sets.items():\n        if set_name in plates:\n            im = Image.open(plates[set_name]).convert(\"RGBA\").resize(MASTER, Image.LANCZOS)\n            name = f\"{set_name}_plate\"\n            st[\"layers\"] = [(name, 0.2)] + ([(\"@screen\", 0.2)] if any(l == \"@screen\" for l, _ in st[\"layers\"]) else []) \\\n                + [(\"@characters\", 0.8)]\n            if screen_rects and set_name in screen_rects:\n                st[\"screen_rect\"] = screen_rects[set_name]\n            imgs = {name: im}\n        else:\n            imgs = {l: Image.open(Path(root) / \"environments\" / env / base_version / base[\"layers\"][l][\"file\"])\n                    for l, _ in st[\"layers\"] if not l.startswith(\"@\")}\n        for n, im in imgs.items():\n            f = tmp / \"layers\" / f\"{n}.png\"\n            im.save(f, optimize=True)\n            layers[n] = {\"file\": f\"layers/{n}.png\", \"size\": list(im.size), \"sha256\": _sha(f.read_bytes())}\n    man = dict(base, version=version, status=\"provisional\", art_version=\"generated-plates\", sets=sets, layers=layers,\n               provenance=dict(provenance or {}, base_version=base_version,\n                               created=datetime.datetime.now().isoformat(timespec=\"seconds\"), approved_by=None))\n    (tmp / \"manifest.json\").write_text(json.dumps(man, indent=1))\n    tmp.rename(vdir)\n    log(f\"environment {env} {version}: plates for {sorted(plates)} on top of {base_version}'s presets\")\n    return vdir\n", "expressive_tts.py": "\"\"\"Expressive TTS backends behind one interface, and an HONEST mapping of voice direction to what each backend can do.\n\nEvery direction field is reported per line as one of:\n  \"native\"       the backend has a real control for it (e.g. Kokoro speed),\n  \"instruction\"  requested in a natural-language instruction the model was trained to follow \u2014 an effect is likely\n                 but not guaranteed; the voice QC measures what actually came out,\n  \"description\"  the same for description-conditioned models (Parler),\n  \"silence\"      done by inserting real silence between chunks (pauses),\n  \"unsupported\"  the backend can't do it: NOT applied, and reported as such (never claimed as applied).\nDSP pitch shifting is never used to fake emotion.\n\"\"\"\nfrom __future__ import annotations\n\nfrom voice_direction import EMOTION_WORDS, VoiceDirection, instruction\n\nFIELDS = (\"emotion\", \"intensity\", \"energy\", \"pace_multiplier\", \"emphasis_words\", \"pause_before_ms\", \"pause_after_ms\")\nCAPS = {\n    \"kokoro\":    {\"emotion\": \"unsupported\", \"intensity\": \"unsupported\", \"energy\": \"unsupported\",\n                  \"pace_multiplier\": \"native\", \"emphasis_words\": \"unsupported\",\n                  \"pause_before_ms\": \"silence\", \"pause_after_ms\": \"silence\"},\n    \"qwen3-tts\": {\"emotion\": \"instruction\", \"intensity\": \"instruction\", \"energy\": \"instruction\",\n                  \"pace_multiplier\": \"instruction\", \"emphasis_words\": \"instruction\",\n                  \"pause_before_ms\": \"silence\", \"pause_after_ms\": \"silence\"},\n    \"gemini\":    {\"emotion\": \"style\", \"intensity\": \"unsupported\", \"energy\": \"unsupported\",\n                  \"pace_multiplier\": \"unsupported\", \"emphasis_words\": \"unsupported\",\n                  \"pause_before_ms\": \"natural\", \"pause_after_ms\": \"silence\"},     # style = speech_metadata.style;\n                  # natural = the model's own gap between turns inside a two-speaker request (kept as measured)\n    \"parler\":    {\"emotion\": \"description\", \"intensity\": \"description\", \"energy\": \"description\",\n                  \"pace_multiplier\": \"description\", \"emphasis_words\": \"unsupported\",\n                  \"pause_before_ms\": \"silence\", \"pause_after_ms\": \"silence\"},\n}\nBACKENDS = {\n    \"gemini\": {\"model\": \"gemini-3.8-flash-tts (Google Gemini API, free tier: hosted exception, speech only)\",\n               \"license\": \"proprietary hosted service (free-tier terms)\", \"languages\": [\"te\", \"en\", \"...\"],\n               \"identity\": \"prebuilt voice per character\", \"instruction\": True, \"params\": \"n/a (hosted)\",\n               \"vram_gb\": \"0 (remote)\"},\n    \"kokoro\": {\"model\": \"hexgrad/Kokoro-82M (kokoro-onnx)\", \"license\": \"apache-2.0\", \"languages\": [\"en\"],\n               \"identity\": \"fixed stock voice\", \"instruction\": False, \"params\": \"82M\", \"vram_gb\": \"~0.3 (or CPU)\"},\n    \"qwen3-tts\": {\"model\": \"Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\", \"license\": \"apache-2.0\",\n                  \"languages\": [\"en\", \"zh\", \"ja\", \"ko\", \"de\", \"fr\", \"ru\", \"pt\", \"es\", \"it\"],\n                  \"identity\": \"fixed premium speaker (e.g. aiden, ryan)\", \"instruction\": True, \"params\": \"1.7B\",\n                  \"vram_gb\": \"~7 fp32 (T4: measure with VOICE_BENCH)\"},\n    \"parler\": {\"model\": \"ai4bharat/indic-parler-tts\", \"license\": \"apache-2.0\", \"languages\": [\"te\", \"hi\", \"en\", \"...\"],\n               \"identity\": \"trained named speaker + description\", \"instruction\": \"description\", \"params\": \"0.9B\",\n               \"vram_gb\": \"~4 fp32\"},\n}\n\n\ndef report(d: VoiceDirection | None, backend: str) -> dict:\n    \"\"\"What happened to each direction field on this backend (see the module docstring).\"\"\"\n    if d is None:\n        return {\"direction\": \"none\"}\n    caps = CAPS[backend]\n    out = {}\n    for f in FIELDS:\n        v = getattr(d, f)\n        neutral = (f == \"emotion\" and v == \"neutral\") or (f == \"intensity\" and v == 0) or \\\n                  (f == \"pace_multiplier\" and v == 1.0) or (f == \"emphasis_words\" and not v) or \\\n                  (f == \"pause_before_ms\" and v == 0) or (f == \"energy\" and v == \"medium\")\n        out[f] = {\"value\": v, \"how\": \"not needed\" if neutral else caps[f]}\n    out[\"unsupported\"] = [f for f, x in out.items() if isinstance(x, dict) and x[\"how\"] == \"unsupported\"]\n    return out\n\n\ndef kokoro_params(d: VoiceDirection | None, base_speed: float) -> dict:\n    \"\"\"Kokoro can only take the pace natively (speed); everything else is reported unsupported.\"\"\"\n    return {\"speed\": round(base_speed * (d.pace_multiplier if d else 1.0), 3)}\n\n\ndef qwen_params(d: VoiceDirection | None, speaker: str) -> dict:\n    \"\"\"Qwen3-TTS CustomVoice: one fixed speaker (identity) + the direction as an instruction (style).\"\"\"\n    directed = d is not None and (d.emotion != \"neutral\" or d.intensity > 0)\n    return {\"speaker\": speaker, \"language\": \"English\",\n            \"instruct\": (d.style_instruction or instruction(d)) if directed else \"\"}\n\n\ndef parler_style(d: VoiceDirection | None) -> str:\n    \"\"\"Indic Parler-TTS: the direction becomes part of the speaker description (emotion support for Telugu is\n    'not extensively tested' per its model card: the QC decides).\"\"\"\n    if d is None or d.emotion == \"neutral\":\n        return \"\"\n    pace = \"slightly slowly\" if d.pace_multiplier < 0.97 else \"slightly quickly\" if d.pace_multiplier > 1.03 else \"at a moderate pace\"\n    expr = \"slightly expressive\" if d.intensity < 0.4 else \"expressive\"\n    return f\"with {EMOTION_WORDS[d.emotion]}, {expr}, {pace}\"\n", "factcheck.py": "\"\"\"Is each narrated sentence supported by the facts it cites? Two deterministic gates + one model gate.\n\n  1. numbers: every number in the sentence (digits, \"^\" exponents, number words) appears in its evidence;\n  2. names: every proper noun / acronym in the sentence appears in its evidence (or the topic);\n  3. entailment: an NLI cross-encoder (cross-encoder/nli-deberta-v3-base, Apache-2.0, runs on the CPU) reads\n     evidence \u2192 sentence: PASS if entailed, FAIL if contradicted, UNKNOWN if neither.\nQuestions (hooks) make no claim: only gates 1\u20132 apply. Analogies: gates 1\u20132 + they must be worded as a comparison\n(NLI can't judge them: a correct analogy is \"about something else\", which it scores as contradiction). FAIL and UNKNOWN both block automatic approval; the planner\ngets one rewrite, then the sentence is dropped. No model can guarantee a claim is true \u2014 this guarantees an\nunsupported claim can't reach an approved render silently. Thresholds are settings (TH), to be calibrated.\n\"\"\"\nfrom __future__ import annotations\n\nimport re\n\nNLI_MODEL = \"cross-encoder/nli-deberta-v3-base\"     # chosen on a labelled set: 13/14, 0 false passes (small: 12/14)\nTH = {\"entail_min\": 0.5, \"contradict_max\": 0.5}\nNUM_WORDS = {\"zero\": 0, \"one\": 1, \"two\": 2, \"three\": 3, \"four\": 4, \"five\": 5, \"six\": 6, \"seven\": 7, \"eight\": 8,\n             \"nine\": 9, \"ten\": 10, \"eleven\": 11, \"twelve\": 12, \"thirteen\": 13, \"fourteen\": 14, \"fifteen\": 15,\n             \"twenty\": 20, \"thirty\": 30, \"forty\": 40, \"fifty\": 50, \"hundred\": 100, \"thousand\": 1000,\n             \"dozen\": 12, \"half\": 0.5}\nSCALE = {\"thousand\": 1e3, \"million\": 1e6, \"billion\": 1e9, \"trillion\": 1e12}\nGENERIC = {\"I\", \"We\", \"You\", \"It\", \"This\", \"That\", \"These\", \"Those\", \"There\", \"Here\", \"What\", \"Why\", \"How\", \"When\",\n           \"Where\", \"Which\", \"Who\", \"But\", \"And\", \"So\", \"Then\", \"Now\", \"Imagine\", \"Think\", \"Ever\", \"Picture\"}\n\n\ndef numbers(text):\n    \"\"\"Values mentioned: '2\u00d710^30' \u2192 2e30, '4.6 billion' \u2192 4.6e9, '1,045' \u2192 1045, 'three' \u2192 3. Years kept as is.\"\"\"\n    t = text.replace(\"\u2212\", \"-\").replace(\",\", \"\")\n    out = set()\n    for m in re.finditer(r\"(\\d+(?:\\.\\d+)?)\\s*(?:\u00d7|x|\\*)\\s*10\\^(-?\\d+)\", t):\n        out.add(float(m.group(1)) * 10 ** int(m.group(2)))\n    t2 = re.sub(r\"(\\d+(?:\\.\\d+)?)\\s*(?:\u00d7|x|\\*)\\s*10\\^(-?\\d+)\", \" \", t)\n    for m in re.finditer(r\"10\\^(-?\\d+)\", t2):\n        out.add(10.0 ** int(m.group(1)))\n    t2 = re.sub(r\"10\\^(-?\\d+)\", \" \", t2)\n    for m in re.finditer(r\"(\\d+(?:\\.\\d+)?)\\s*(thousand|million|billion|trillion)?\", t2, re.I):\n        v = float(m.group(1)) * (SCALE[m.group(2).lower()] if m.group(2) else 1)\n        out.add(v)\n    for w, v in NUM_WORDS.items():\n        for m in re.finditer(rf\"\\b{w}\\b\\s*(thousand|million|billion|trillion)?\", t2, re.I):\n            out.add(v * (SCALE[m.group(1).lower()] if m.group(1) else 1))\n    return out\n\n\ndef names(text):\n    \"\"\"Proper nouns and acronyms, not counting the first word of a sentence or generic capitalised words.\"\"\"\n    found = set()\n    for sent in re.split(r\"(?<=[.!?])\\s+\", text):\n        toks = re.findall(r\"[A-Za-z][A-Za-z'*-]*\", sent)\n        for i, w in enumerate(toks):\n            if (w.isupper() and len(w) >= 2) or (i > 0 and w[0].isupper() and w not in GENERIC):\n                found.add(w.strip(\"'s\").rstrip(\"'\"))\n    return found\n\n\ndef _close(a, b):\n    return abs(a - b) <= 1e-9 * max(abs(a), abs(b), 1) or (b and abs(a / b - 1) < 0.02)\n\n\ndef static_checks(sentence, evidence, topic=\"\"):\n    ev_nums, ev_low = numbers(evidence), (evidence + \" \" + topic).lower()\n    bad_nums = sorted(n for n in numbers(sentence) if not any(_close(n, e) for e in ev_nums))\n    bad_names = sorted(n for n in names(sentence) if n.lower() not in ev_low)\n    return bad_nums, bad_names\n\n\nclass NLI:\n    def __init__(self, model=NLI_MODEL, device=\"cpu\", cache_dir=None):\n        import torch\n        from transformers import AutoModelForSequenceClassification, AutoTokenizer\n        self.torch, self.device = torch, device\n        self.tok = AutoTokenizer.from_pretrained(model, cache_dir=cache_dir)\n        self.model = AutoModelForSequenceClassification.from_pretrained(model, cache_dir=cache_dir).to(device).eval()\n        lab = {v.lower(): int(k) for k, v in self.model.config.id2label.items()}\n        self.ix = (lab[\"entailment\"], lab[\"contradiction\"], lab[\"neutral\"])\n\n    def scores(self, pairs):\n        \"\"\"[(premise, hypothesis)] \u2192 [(p_entail, p_contradict, p_neutral)], batched.\"\"\"\n        if not pairs:\n            return []\n        with self.torch.inference_mode():\n            enc = self.tok([p for p, _ in pairs], [h for _, h in pairs], truncation=True, max_length=512,\n                           padding=True, return_tensors=\"pt\").to(self.device)\n            pr = self.model(**enc).logits.softmax(-1).cpu().tolist()\n        e, c, n = self.ix\n        return [(r[e], r[c], r[n]) for r in pr]\n\n\ndef check_sentences(items, nli, topic=\"\", th=TH):\n    \"\"\"items: [{\"id\", \"text\", \"evidence\": [fact texts], \"question\": bool}] \u2192 {id: {\"status\", \"why\", \"scores\"...}}.\"\"\"\n    out, todo = {}, []\n    for it in items:\n        ev = \" \".join(it[\"evidence\"])\n        bad_nums, bad_names = static_checks(it[\"text\"], ev, topic)\n        r = {\"text\": it[\"text\"], \"evidence\": it[\"evidence\"], \"bad_numbers\": bad_nums, \"unsupported_names\": bad_names}\n        if bad_nums or bad_names:\n            r.update(status=\"FAIL\", why=\"numbers/names not in the cited facts\")\n        elif it.get(\"question\"):\n            r.update(status=\"PASS\", why=\"question: makes no claim; numbers and names are sourced\")\n        elif it.get(\"analogy\"):\n            # a comparison is about something ELSE, which NLI reads as \"contradiction\" (measured: 0.84 for a correct\n            # waterfall analogy), so NLI can't judge it. It must SAY it's a comparison, and gates 1\u20132 already keep\n            # out new numbers and names, so no factual claim can be smuggled in this way.\n            if re.search(r\"\\b(like|as if|as though|imagine|think of|picture|similar to|compare|what if|suppose)\\b\", it[\"text\"], re.I):\n                r.update(status=\"PASS\", why=\"analogy: marked as a comparison, no unsourced numbers or names\")\n            else:\n                r.update(status=\"FAIL\", why=\"called an analogy but not worded as a comparison ('like', 'imagine'...)\")\n        elif not it[\"evidence\"]:\n            r.update(status=\"FAIL\", why=\"a statement with no cited fact\")\n        else:\n            todo.append((it[\"id\"], ev, it[\"text\"]))\n        out[it[\"id\"]] = r\n    for (sid, ev, text), (pe, pc, pn) in zip(todo, nli.scores([(ev, t) for _, ev, t in todo])):\n        r = out[sid]\n        r[\"scores\"] = {\"entail\": round(pe, 3), \"contradict\": round(pc, 3), \"neutral\": round(pn, 3)}\n        if pc >= th[\"contradict_max\"]:\n            r.update(status=\"FAIL\", why=\"contradicted by its cited facts\")\n        elif pe >= th[\"entail_min\"]:\n            r.update(status=\"PASS\", why=\"entailed by its cited facts\")\n        else:\n            r.update(status=\"UNKNOWN\", why=\"not clearly supported by its cited facts\")\n    return out\n\n\ndef same_claim(nli, a, b, th=0.9):\n    \"\"\"A repeat for the viewer: either sentence strongly entails the other (measured: a reworded repeat scores\n    1.00 one way and 0.01 the other, because it adds a detail; unrelated sentences score 0.00 both ways).\"\"\"\n    (e1, _, _), (e2, _, _) = nli.scores([(a, b), (b, a)])\n    return max(e1, e2) >= th\n", "gemini_tts.py": "\"\"\"Google Gemini free-tier TTS: the project's ONE hosted exception, allowed for speech synthesis only.\n\nVerified against the official docs on 2026-10-19 (ai.google.dev/gemini-api/docs: speech-generation, pricing,\nrate-limits, billing, terms):\n  * models gemini-3.8-flash-tts / gemini-3.8-flash-lite-tts: Telugu + up to 2 speakers (prebuilt voices) per request;\n  * REST POST /v1beta/interactions (= SDK client.interactions.create; the SDK itself is not installed, see\n    RestClient), per-turn `speech_metadata` (speaker, style),\n    generation_config.speech_config = [{\"voice\"}] (one speaker) or {\"speakers\": [{\"speaker\", \"voice\"}]} (two);\n  * unary responses are WAV (44-byte RIFF header, 24 kHz, mono, 16-bit) \u2014 never wrapped in a second header here;\n  * Free tier = Standard (and Priority) only; Batch/Flex are NOT free \u2192 this module never sets a service tier;\n  * free-tier input/output \"used to improve our products\" (human review possible) \u2192 only the approved dialogue text\n    and short style words are sent; no research, no files, no credentials;\n  * free-tier rate limits are per PROJECT and shown in AI Studio, not published per model \u2192 local budget below;\n  * there is NO documented API to read a project's billing status \u2192 the user must confirm it (fails closed).\n\nStatuses: PASS, FAIL, NOT_TESTED, RESOURCE_UNAVAILABLE, FREE_QUOTA_EXHAUSTED, ACCESS_REQUIRED, MANUAL_REVIEW_REQUIRED,\nPOLICY_BLOCKED.\n\"\"\"\nfrom __future__ import annotations\n\nimport base64, datetime, hashlib, io, json, os, re, time, wave\nfrom pathlib import Path\n\nFREE_TTS_MODELS = (\"gemini-3.8-flash-tts\", \"gemini-3.8-flash-lite-tts\")     # free tier verified on the pricing page\nMIN_SDK = (2, 25, 0)\nPREBUILT_VOICES = (\"Zephyr\", \"Puck\", \"Charon\", \"Kore\", \"Fenrir\", \"Leda\", \"Orus\", \"Aoede\", \"Callirrhoe\", \"Autonoe\",\n                   \"Enceladus\", \"Iapetus\", \"Umbriel\", \"Algieba\", \"Despina\", \"Erinome\", \"Algenib\", \"Rasalgethi\",\n                   \"Laomedeia\", \"Achernar\", \"Alnilam\", \"Schedar\", \"Gacrux\", \"Pulcherrima\", \"Achird\", \"Zubenelgenubi\",\n                   \"Vindemiatrix\", \"Sadachbia\", \"Sadaltager\", \"Sulafat\")\nINLINE_TAGS = (\"<short pause>\", \"<long pause>\", \"<laugh>\", \"<laughter>\", \"<chuckle>\", \"<giggle>\", \"<sigh>\", \"<gasp>\")\n# conversational intention \u2192 short turn-level style (an EXPERIMENT; empty = the model's own natural reading)\nSTYLE_FOR_INTENT = {\"explaining_observation\": \"casual and conversational\", \"genuine_curiosity\": \"naturally curious\",\n                    \"reflective_explanation\": \"warm and relaxed\", \"playful_disbelief\": \"lightly amused\",\n                    \"quiet_realization\": \"thoughtful and reflective\", \"enthusiastic_discovery\": \"subtly excited\",\n                    \"gentle_correction\": \"calm and reassuring\", \"satisfying_revelation\": \"warm and relaxed\",\n                    \"guiding_demo\": \"casual and conversational\", \"playful_remark\": \"lightly amused\",\n                    \"asking_confirmation\": \"naturally curious\", \"thinking_aloud\": \"thoughtful and reflective\"}\n\n\nclass GeminiBlocked(Exception):\n    \"\"\"A policy / access / quota condition: the caller falls back to the local voice (never to a paid route).\"\"\"\n    def __init__(self, status, why):\n        super().__init__(f\"{status}: {why}\")\n        self.status, self.why = status, why\n\n\ndef redact(text, secret):\n    \"\"\"Never let the key appear in a message, log or file.\"\"\"\n    text = str(text)\n    if secret:\n        text = text.replace(secret, \"***\")\n    return re.sub(r\"AIza[0-9A-Za-z_\\-]{20,}\", \"AIza***\", text)\n\n\n# ------------------------------------------------------------------ preflight (no synthesis call)\ndef sdk_version():\n    try:\n        import importlib.metadata as md\n        v = md.version(\"google-genai\")\n        return tuple(int(x) for x in re.findall(r\"\\d+\", v)[:3]), v\n    except Exception:\n        return None, None\n\n\ndef preflight(cfg, key, ledger, project, client=None):\n    \"\"\"Every condition that must hold BEFORE any Gemini request. Returns {\"status\", \"checks\": {name: (status, why)}}.\n    cfg: {\"enabled\", \"model\", \"free_tier_confirmed\", \"free_tier_only\", \"max_videos_per_day\", \"max_calls_per_video\"}.\n    The key is only checked for presence (never printed). Model availability: one metadata call (models.get) when a\n    client is given \u2014 not a synthesis call.\"\"\"\n    ck = {}\n    ck[\"enabled in settings\"] = (\"PASS\", \"\") if cfg.get(\"enabled\") else (\"NOT_TESTED\", \"Gemini TTS is off in the settings\")\n    ok_model = cfg.get(\"model\") in FREE_TTS_MODELS\n    ck[\"approved free-tier model\"] = (\"PASS\", cfg.get(\"model\")) if ok_model else (\n        \"POLICY_BLOCKED\", f\"{cfg.get('model')!r} is not an approved free-tier TTS model {FREE_TTS_MODELS}\")\n    ck[\"free tier confirmed by the user\"] = (\"PASS\", \"\") if cfg.get(\"free_tier_confirmed\") and cfg.get(\"free_tier_only\", True) \\\n        else (\"POLICY_BLOCKED\", \"set GEMINI_FREE_TIER_CONFIRMED = True only after AI Studio \u2192 Projects shows this \"\n                                \"project's Billing Tier as Free (a key alone does not prove the project is unbilled)\")\n    ck[\"API key present\"] = (\"PASS\", \"\") if key else (\"ACCESS_REQUIRED\", \"add the Colab secret GEMINI_API_KEY\")\n    try:\n        import requests  # noqa: F401  (REST client: no extra package, Colab's environment stays untouched)\n        ck[\"HTTP client\"] = (\"PASS\", \"requests\")\n    except ImportError:\n        ck[\"HTTP client\"] = (\"RESOURCE_UNAVAILABLE\", \"requests is not installed\")\n    today = ledger.today()\n    # a voice audition / benchmark (\"bench:\u2026\") is a one-off test, not a video: it never uses up the daily video\n    others = [p for p in today.get(\"videos\", []) if p != project and not p.startswith(\"bench:\")]\n    ck[\"daily video budget\"] = (\"PASS\", f\"{len(others)} other videos today\") if len(others) < cfg.get(\"max_videos_per_day\", 1) \\\n        else (\"FREE_QUOTA_EXHAUSTED\", f\"MAX_VIDEOS_PER_DAY={cfg.get('max_videos_per_day', 1)} already used today: {others}\")\n    used = ledger.calls(project)\n    ck[\"calls left for this video\"] = (\"PASS\", f\"{used} used\") if used < cfg.get(\"max_calls_per_video\", 6) \\\n        else (\"FREE_QUOTA_EXHAUSTED\", f\"MAX_GEMINI_TTS_CALLS_PER_VIDEO={cfg.get('max_calls_per_video', 6)} used today: \"\n                                      \"only audio already made (cached) can be used; no new request\")\n    if client is not None and all(s == \"PASS\" for s, _ in ck.values()):\n        try:\n            client.models.get(model=cfg[\"model\"])\n            ck[\"model available to this key\"] = (\"PASS\", \"\")\n        except Exception as e:\n            st = classify(e)                       # a bad key / no access stops here; anything else is decided by\n            ck[\"model available to this key\"] = (  # the first real request (its errors are classified the same way)\n                \"ACCESS_REQUIRED\" if st == \"ACCESS_REQUIRED\" else \"NOT_TESTED\", redact(e, key)[:200])\n    # \"calls left\" is information: audio already made (cached on Drive) is reused for free, and every NEW request is\n    # still refused by synthesize() once the budget is used \u2014 so a used-up budget can't block re-using cached audio\n    soft = {(\"model available to this key\", \"NOT_TESTED\"), (\"calls left for this video\", \"FREE_QUOTA_EXHAUSTED\")}\n    bad = [s for k, (s, _) in ck.items() if s != \"PASS\" and (k, s) not in soft]\n    return {\"status\": \"PASS\" if not bad else bad[0], \"checks\": ck}\n\n\ndef classify(e):\n    \"\"\"Map an API error to a status (429 \u2192 FREE_QUOTA_EXHAUSTED; 401/403 \u2192 ACCESS_REQUIRED; 404 \u2192 POLICY_BLOCKED).\"\"\"\n    code = getattr(e, \"code\", None) or getattr(e, \"status_code\", None)\n    msg = str(e)\n    if code == 429 or \"RESOURCE_EXHAUSTED\" in msg or re.search(r\"\\b429\\b\", msg):\n        return \"FREE_QUOTA_EXHAUSTED\"\n    if code in (401, 403) or re.search(r\"\\b(401|403)\\b|PERMISSION_DENIED|API key not valid\", msg):\n        return \"ACCESS_REQUIRED\"\n    if code == 404 or \"NOT_FOUND\" in msg:\n        return \"POLICY_BLOCKED\"\n    return \"FAIL\"\n\n\ndef retry_delay_s(e):\n    \"\"\"The delay the API asks for (RetryInfo 'retryDelay': '23s'), if any.\"\"\"\n    m = re.search(r\"retry(?:Delay|_delay)['\\\"]?\\s*[:=]\\s*['\\\"]?(\\d+(?:\\.\\d+)?)s\", str(e))\n    return float(m.group(1)) if m else None\n\n\n# ------------------------------------------------------------------ usage ledger (persistent, on Drive)\nclass Ledger:\n    \"\"\"Local safety budget, NOT Google's quota: per day the videos that used Gemini and every request made.\"\"\"\n    def __init__(self, path, clock=None):\n        self.path, self.clock = Path(path), clock or (lambda: datetime.datetime.now())\n        self.data = json.loads(self.path.read_text()) if self.path.exists() else {}\n\n    def _day(self):\n        return self.clock().strftime(\"%Y-%m-%d\")\n\n    def today(self):\n        return self.data.setdefault(self._day(), {\"videos\": [], \"requests\": []})\n\n    def calls(self, project):\n        return sum(1 for r in self.today()[\"requests\"] if r[\"project\"] == project and not r.get(\"cache_hit\"))\n\n    def record(self, project, **info):\n        d = self.today()\n        if project not in d[\"videos\"] and not info.get(\"cache_hit\") and not project.startswith(\"bench:\"):\n            d[\"videos\"].append(project)\n        d[\"requests\"].append({\"project\": project, \"time\": self.clock().strftime(\"%H:%M:%S\"), **info})\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=1))\n\n\n# ------------------------------------------------------------------ request / response\ndef build_request(model, turns, voices, style_on=True):\n    \"\"\"turns: [{\"speaker\": \"arjun\", \"text\": \"...\", \"style\": \"...\"}] (1\u20132 distinct speakers); voices: {speaker: voice}.\n    Only the transcript and short style words are sent (privacy).\"\"\"\n    if model not in FREE_TTS_MODELS:\n        raise GeminiBlocked(\"POLICY_BLOCKED\", f\"model {model!r} is not an approved free-tier TTS model\")\n    speakers = list(dict.fromkeys(t[\"speaker\"] for t in turns))\n    if len(speakers) > 2:\n        raise ValueError(\"Gemini multi-speaker TTS takes at most 2 speakers per request\")\n    for s in speakers:\n        if voices.get(s) not in PREBUILT_VOICES:\n            raise ValueError(f\"speaker {s}: {voices.get(s)!r} is not a prebuilt Gemini voice\")\n    content = []\n    for t in turns:\n        meta = {\"type\": \"speech_metadata\"}\n        if len(speakers) == 2:\n            meta[\"speaker\"] = t[\"speaker\"]\n        if style_on and t.get(\"style\"):\n            meta[\"style\"] = t[\"style\"]\n        item = {\"type\": \"text\", \"text\": t[\"text\"]}\n        if len(meta) > 1:\n            item[\"annotations\"] = [meta]\n        content.append(item)\n    speech = ({\"mode\": \"conversational\", \"speakers\": [{\"speaker\": s, \"voice\": voices[s]} for s in speakers]}\n              if len(speakers) == 2 else [{\"voice\": voices[speakers[0]]}])   # as in Google's two-speaker example\n    return {\"model\": model, \"input\": [{\"type\": \"user_input\", \"content\": content}],\n            \"response_format\": {\"type\": \"audio\"}, \"generation_config\": {\"speech_config\": speech}}\n\n\ndef decode_audio(interaction):\n    \"\"\"The response audio as WAV bytes + its real format. Unary responses are WAV; raw PCM (l16) is wrapped ONCE.\"\"\"\n    out = getattr(interaction, \"output_audio\", None)\n    data = getattr(out, \"data\", None) if out is not None else None\n    if data is None and isinstance(interaction, dict):\n        data = (interaction.get(\"output_audio\") or {}).get(\"data\")\n    if not data:\n        raise ValueError(\"the response has no audio\")\n    raw = base64.b64decode(data) if isinstance(data, str) else bytes(data)\n    if raw[:4] != b\"RIFF\":\n        mime = getattr(out, \"mime_type\", \"\") or \"\"\n        if \"l16\" not in mime and \"pcm\" not in mime:\n            raise ValueError(f\"unexpected audio format (first bytes {raw[:4]!r}, mime {mime!r})\")\n        b = io.BytesIO()\n        with wave.open(b, \"wb\") as w:\n            w.setnchannels(1); w.setsampwidth(2); w.setframerate(24000); w.writeframes(raw)\n        raw = b.getvalue()\n    return raw, wav_info(raw)\n\n\ndef wav_info(raw):\n    \"\"\"Validate WAV bytes; return the actual format (never assumed).\"\"\"\n    if raw[:4] != b\"RIFF\" or raw[8:12] != b\"WAVE\":\n        raise ValueError(\"not a WAV file\")\n    with wave.open(io.BytesIO(raw)) as w:\n        n, sr, ch, sw = w.getnframes(), w.getframerate(), w.getnchannels(), w.getsampwidth()\n    if n == 0:\n        raise ValueError(\"empty audio\")\n    if raw[44:48] == b\"RIFF\":\n        raise ValueError(\"double WAV header (PCM was wrapped twice)\")\n    return {\"sample_rate\": sr, \"channels\": ch, \"bit_depth\": 8 * sw, \"frames\": n, \"duration_s\": round(n / sr, 3)}\n\n\n# ------------------------------------------------------------------ cached synthesis with budget + 429 policy\ndef cache_key(model, turns, voices, style_on):\n    return hashlib.sha1(json.dumps([model, [(t[\"speaker\"], t[\"text\"], t.get(\"style\", \"\") if style_on else \"\")\n                                            for t in turns], {k: voices[k] for k in sorted(voices)}, style_on],\n                                   ensure_ascii=False).encode()).hexdigest()[:16]\n\n\ndef synthesize(client, cfg, turns, voices, cache_dir, ledger, project, key=\"\", log=print, sleep=time.sleep,\n               style_on=True, label=\"\"):\n    \"\"\"One request (cached). Returns (wav path, info). Raises GeminiBlocked on policy/quota/access, so the caller\n    falls back to the local voice. Retries at most cfg['max_retries'] times, only on 429 with a short server delay.\"\"\"\n    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)\n    k = cache_key(cfg[\"model\"], turns, voices, style_on)\n    wav_path, meta_path = cache_dir / f\"{k}.wav\", cache_dir / f\"{k}.json\"\n    if wav_path.exists() and meta_path.exists():\n        info = json.loads(meta_path.read_text())\n        ledger.record(project, label=label, model=cfg[\"model\"], key=k, cache_hit=True, status=\"PASS\")\n        return wav_path, dict(info, cache_hit=True)\n    if ledger.calls(project) >= cfg.get(\"max_calls_per_video\", 6):\n        raise GeminiBlocked(\"FREE_QUOTA_EXHAUSTED\", f\"local budget: {cfg.get('max_calls_per_video', 6)} calls for this video used\")\n    req = build_request(cfg[\"model\"], turns, voices, style_on)\n    retries = 0\n    while True:\n        t0 = time.time()\n        try:\n            inter = client.interactions.create(**req)\n        except Exception as e:\n            st, why = classify(e), redact(e, key)[:300]\n            ledger.record(project, label=label, model=cfg[\"model\"], key=k, status=st, error=why,\n                          retry=retries, transcript_hash=k, speakers=voices)\n            d = retry_delay_s(e)\n            if st == \"FREE_QUOTA_EXHAUSTED\" and retries < cfg.get(\"max_retries\", 1) and d is not None and d <= 60:\n                retries += 1\n                log(f\"Gemini 429: waiting {d:.0f}s as the API asks (retry {retries})\")\n                sleep(d)\n                if ledger.calls(project) >= cfg.get(\"max_calls_per_video\", 6):\n                    raise GeminiBlocked(\"FREE_QUOTA_EXHAUSTED\", \"local budget used during retries\")\n                continue\n            if st in (\"FREE_QUOTA_EXHAUSTED\", \"ACCESS_REQUIRED\", \"POLICY_BLOCKED\"):\n                raise GeminiBlocked(st, why)\n            raise GeminiBlocked(\"FAIL\", why)\n        raw, info = decode_audio(inter)\n        wav_path.write_bytes(raw)                          # the ORIGINAL audio, before any resampling\n        info.update(model=cfg[\"model\"], voices=voices, turns=[(t[\"speaker\"], t[\"text\"]) for t in turns],\n                    style_on=style_on, gen_s=round(time.time() - t0, 2), created=time.strftime(\"%Y-%m-%d %H:%M:%S\"))\n        meta_path.write_text(json.dumps(info, ensure_ascii=False, indent=1))\n        ledger.record(project, label=label, model=cfg[\"model\"], key=k, status=\"PASS\", transcript_hash=k,\n                      speakers=voices, duration_s=info[\"duration_s\"], retry=retries)\n        return wav_path, dict(info, cache_hit=False)\n\n\ndef plan_chunks(scenes, max_calls, reserve=2, max_s=40.0, sec_per_char=0.075):\n    \"\"\"Group consecutive scenes into conversation chunks: as few requests as possible (each \u2264 ~max_s seconds of\n    estimated speech), leaving `reserve` calls of the budget for repairs. scenes: [[(line_id, speaker, text)]].\"\"\"\n    est = lambda lines: sum(len(re.sub(r\"\\s\", \"\", t)) for _, _, t in lines) * sec_per_char\n    chunks, cur = [], []\n    for sc in scenes:\n        if cur and est(cur + sc) > max_s:\n            chunks.append(cur); cur = []\n        cur = cur + sc\n    if cur:\n        chunks.append(cur)\n    while len(chunks) > max(max_calls - reserve, 1) and len(chunks) > 1:      # merge the two shortest neighbours\n        i = min(range(len(chunks) - 1), key=lambda j: est(chunks[j]) + est(chunks[j + 1]))\n        chunks[i:i + 2] = [chunks[i] + chunks[i + 1]]\n    return chunks\n\n\nclass ApiError(Exception):\n    \"\"\"An HTTP error from the Gemini API (code = HTTP status; the message carries Google's error JSON).\"\"\"\n    def __init__(self, code, message):\n        super().__init__(f\"{code} {message}\")\n        self.code = code\n\n\nclass RestClient:\n    \"\"\"The documented REST form of the same API (POST /v1beta/interactions), with Colab's own `requests`.\n    Why not the google-genai SDK: every version with TTS (2.25\u20132.31) needs google-auth >= 2.56, while Colab's\n    google-colab pins google-auth == 2.49.0 \u2014 installing it breaks Colab's own package. The key travels only in\n    the x-goog-api-key header (never in a URL, so it can't end up in a log line).\"\"\"\n    BASE = \"https://generativelanguage.googleapis.com/v1beta\"\n\n    def __init__(self, key, timeout=240, session=None):\n        import types\n        import requests\n        self._key, self._timeout = key, timeout\n        self._http = session or requests.Session()\n        self.interactions = types.SimpleNamespace(create=self._create)\n        self.models = types.SimpleNamespace(get=self._get)\n\n    def _call(self, method, path, body=None):\n        r = self._http.request(method, f\"{self.BASE}/{path}\", json=body, timeout=self._timeout,\n                               headers={\"x-goog-api-key\": self._key, \"Content-Type\": \"application/json\"})\n        if r.status_code >= 400:\n            raise ApiError(r.status_code, r.text[:2000])\n        return r.json()\n\n    def _get(self, model):\n        return self._call(\"GET\", f\"models/{model}\")\n\n    def _create(self, **req):\n        import types\n        d = self._call(\"POST\", \"interactions\", req)\n        audio = [c for st in d.get(\"steps\", []) if st.get(\"type\") == \"model_output\"\n                 for c in st.get(\"content\", []) if c.get(\"type\") == \"audio\"]\n        if not audio:\n            raise ValueError(\"the response has no audio (no model_output step with audio content)\")\n        a = audio[-1]\n        return types.SimpleNamespace(output_audio=types.SimpleNamespace(\n            data=a.get(\"data\"), mime_type=a.get(\"mime_type\") or a.get(\"mimeType\") or \"audio/wav\"))\n\n\ndef make_client(key):\n    \"\"\"The Gemini client used by the notebook (REST; see RestClient for why not the SDK).\"\"\"\n    return RestClient(key)\n\n\n# ------------------------------------------------------------------ the whole voice pass for one video\ndef _read_wav(path):\n    import numpy as np, soundfile as sf\n    w, sr = sf.read(str(path), dtype=\"float32\")\n    return (w.mean(1) if w.ndim > 1 else w).astype(np.float32), sr\n\n\ndef letter_error(expected, heard):\n    \"\"\"Share of the expected letters the recogniser did not hear (any script).\"\"\"\n    import difflib\n    from turn_align import letters\n    e, h = letters(expected), letters(heard or \"\")\n    if not e:\n        return 0.0\n    m = sum(b.size for b in difflib.SequenceMatcher(None, e, h, autojunk=False).get_matching_blocks())\n    return round(1 - m / len(e), 3)\n\n\ndef run_voices(lines, voices, cfg, client, cache_dir, ledger, project, out_dir, asr_fn=None, key=\"\", log=print,\n               retry_at=0.35, speaker_fn=None, sleep=time.sleep, err_fn=None):\n    \"\"\"lines: [{\"id\", \"scene\", \"speaker\", \"text\" (what is SPOKEN), \"intent\"}] in order. Makes every line with\n    Gemini (two-speaker chunks first, per-turn as the repair), splits and checks it, and writes\n    out_dir/<id>.wav (original sample rate) \u2014 ONLY if every line succeeded (one voice per character for the whole\n    video). Returns the report: {\"status\": PASS | FALLBACK, \"why\", \"lines\": {...}, \"chunks\": [...], \"calls\", \"gaps\"}.\n    cfg[\"asr_repairs\"] = False: a line the recogniser doubts is kept and flagged for listening, never re-requested\n    (code-switched speech fools Whisper; an API call is not spent on the recogniser's doubt). err_fn(expected, heard)\n    replaces the letter error (e.g. telugu_dialogue.bilingual_error for code-mixed lines).\"\"\"\n    import numpy as np, soundfile as sf\n    import turn_align\n    letter_error_ = err_fn or letter_error\n    asr_repairs = cfg.get(\"asr_repairs\", True)\n    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)\n    tmp = Path(cache_dir) / \"split\"; tmp.mkdir(parents=True, exist_ok=True)\n    style = lambda l: STYLE_FOR_INTENT.get(l.get(\"intent\"), \"\") if cfg.get(\"styles\", True) else \"\"\n    scenes = []\n    for l in lines:\n        if not scenes or scenes[-1][0][\"scene\"] != l[\"scene\"]:\n            scenes.append([])\n        scenes[-1].append(l)\n    by_id = {l[\"id\"]: l for l in lines}\n    groups = plan_chunks([[(l[\"id\"], l[\"speaker\"], l[\"text\"]) for l in sc] for sc in scenes],\n                         cfg.get(\"max_calls_per_video\", 6), reserve=cfg.get(\"reserve_calls\", 2))\n    report = {\"status\": \"PASS\", \"why\": \"\", \"lines\": {}, \"chunks\": [], \"gaps\": {}, \"mode\": cfg.get(\"mode\", \"scene\")}\n    pieces = {}\n\n    def check_piece(lid, wav, sr, words):\n        heard = \" \".join(w[0] for w in words) if words is not None else None\n        err = None if heard is None else letter_error_(by_id[lid][\"text\"], heard)\n        return heard, err\n\n    def per_turn(lid):\n        l = by_id[lid]\n        for attempt in range(1 + cfg.get(\"max_retries\", 1)):\n            p, info = synthesize(client, cfg, [{\"speaker\": l[\"speaker\"], \"text\": l[\"text\"], \"style\": style(l)}],\n                                 {l[\"speaker\"]: voices[l[\"speaker\"]]}, cache_dir, ledger, project, key, log, sleep,\n                                 label=f\"turn {lid}\" + (f\" retry {attempt}\" if attempt else \"\"),\n                                 style_on=attempt == 0)            # the retry drops the style: plain reading\n            wav, sr = _read_wav(p)\n            res = turn_align.align(wav, sr, [l[\"text\"]])\n            seg = turn_align.cut(wav, sr, res[\"spans\"])[0]\n            f = tmp / f\"{lid}_turn.wav\"; sf.write(f, seg, sr)\n            heard, words = asr_fn(f) if asr_fn else (None, None)\n            err = None if heard is None else letter_error_(l[\"text\"], heard)\n            if err is None or err <= retry_at:\n                return seg, sr, {\"how\": \"per-turn\", \"err\": err, \"heard\": heard, \"words\": words, \"cache\": info.get(\"cache_hit\")}\n            if not asr_repairs:                            # the recogniser's doubt costs no second request\n                log(f\"Gemini {lid}: {err:.0%} of the letters not heard \u2192 kept, flagged for listening\")\n                return seg, sr, {\"how\": \"per-turn\", \"err\": err, \"heard\": heard, \"words\": words,\n                                 \"cache\": info.get(\"cache_hit\"), \"review\": True}\n            log(f\"Gemini {lid}: {err:.0%} of the letters not heard; \" + (\"retrying without style\" if attempt == 0 else \"giving up\"))\n        return None, None, {\"how\": \"per-turn\", \"err\": err, \"heard\": heard}\n\n    for g in groups:\n        ids = [lid for lid, _, _ in g]\n        rec = {\"lines\": ids, \"mode\": cfg.get(\"mode\", \"scene\")}\n        speakers = {by_id[i][\"speaker\"] for i in ids}\n        ok_chunk = False\n        if cfg.get(\"mode\", \"scene\") == \"scene\" and len(speakers) <= 2 and len(ids) > 1:\n            try:\n                p, info = synthesize(client, cfg, [{\"speaker\": by_id[i][\"speaker\"], \"text\": by_id[i][\"text\"],\n                                                    \"style\": style(by_id[i])} for i in ids],\n                                     {s: voices[s] for s in speakers}, cache_dir, ledger, project, key, log, sleep,\n                                     label=f\"scene {ids[0]}..{ids[-1]}\")\n                wav, sr = _read_wav(p)\n                heard, words = asr_fn(p) if asr_fn else (None, None)\n                res = turn_align.align(wav, sr, [by_id[i][\"text\"] for i in ids], words=words)\n                rec.update(align=res[\"status\"], source=res.get(\"source\"), why=res.get(\"why\"), duration_s=info[\"duration_s\"],\n                           cache_hit=info.get(\"cache_hit\"))\n                if res[\"status\"] == \"PASS\":\n                    segs = turn_align.cut(wav, sr, res[\"spans\"])\n                    bad = []\n                    for i, (lid, seg, (a, b)) in enumerate(zip(ids, segs, res[\"spans\"])):\n                        ws = [w for w in (words or []) if a - 0.05 <= w[1] < b] if words is not None else None\n                        ht = \" \".join(w[0] for w in ws) if ws is not None else None\n                        err = None if ht is None else letter_error_(by_id[lid][\"text\"], ht)\n                        pieces[lid] = (seg, sr, {\"how\": \"two-speaker chunk\", \"err\": err, \"heard\": ht,\n                                                 \"words\": [[w[0], round(w[1] - a, 3), round(w[2] - a, 3)] for w in ws] if ws else ws})\n                        if i:\n                            report[\"gaps\"][lid] = int(1000 * res[\"gaps\"][i - 1])\n                        if err is not None and err > retry_at:\n                            bad.append(lid)\n                    ok_chunk = True\n                    rec[\"cuts_without_pause\"] = res.get(\"cuts_without_pause\", [])\n                    left = cfg.get(\"max_calls_per_video\", 6) - ledger.calls(project)\n                    if bad and (len(bad) > left or not asr_repairs):\n                        # not enough calls to repair them all: keep the conversation take, a person listens to these\n                        # lines (Whisper's Telugu is weak: a \"miss\" is often the recogniser, not the voice)\n                        for lid in bad:\n                            pieces[lid][2][\"review\"] = True\n                        rec[\"flagged_for_review\"] = bad\n                        log(f\"Gemini: {len(bad)} lines not clearly heard, {left} calls left \u2192 kept, flagged for listening\")\n                        bad = []\n                    for lid in bad:                        # targeted repair of one turn, not the whole chunk\n                        seg, sr2, m = per_turn(lid)\n                        if seg is None:\n                            ok_chunk = False; pieces.pop(lid, None)\n                        else:\n                            pieces[lid] = (seg, sr2, dict(m, how=\"per-turn repair\"))\n                    rec[\"repaired\"] = bad\n            except GeminiBlocked as e:\n                report.update(status=\"FALLBACK\", why=f\"{e.status}: {e.why}\"); report[\"chunks\"].append(rec); break\n        if not ok_chunk and report[\"status\"] == \"PASS\":\n            need = len([i for i in ids if i not in pieces])\n            left = cfg.get(\"max_calls_per_video\", 6) - ledger.calls(project)\n            if need > left:                                 # don't spend calls on a job that can't be finished\n                report.update(status=\"FALLBACK\", why=f\"chunk {ids[0]}..{ids[-1]} could not be split \"\n                              f\"({rec.get('why', '')}) and per-line requests need {need} calls, only {left} left\")\n                report[\"chunks\"].append(rec)\n                break\n            for lid in ids:                                 # the chunk couldn't be split reliably: per turn\n                if lid in pieces:\n                    continue\n                try:\n                    seg, sr2, m = per_turn(lid)\n                except GeminiBlocked as e:\n                    report.update(status=\"FALLBACK\", why=f\"{e.status}: {e.why}\"); break\n                if seg is None:\n                    report.update(status=\"FALLBACK\", why=f\"{lid}: words not heard correctly after a retry\"); break\n                pieces[lid] = (seg, sr2, m)\n            rec[\"mode\"] = \"per-turn\"\n        report[\"chunks\"].append(rec)\n        if report[\"status\"] != \"PASS\":\n            break\n    if report[\"status\"] == \"PASS\" and speaker_fn is not None:     # who is speaking? (voice embeddings)\n        embs = {lid: speaker_fn(seg, sr) for lid, (seg, sr, _) in pieces.items()}\n        cents = {}\n        for s in {l[\"speaker\"] for l in lines}:\n            cents[s] = [embs[i] for i in embs if by_id[i][\"speaker\"] == s]\n        wrong = []\n        for lid, e in embs.items():\n            sims = {}\n            for s, E in cents.items():\n                others = [x for x, i in zip(E, [i for i in embs if by_id[i][\"speaker\"] == s]) if i != lid]\n                if others:\n                    c = np.mean(others, axis=0); sims[s] = float(e @ (c / (np.linalg.norm(c) + 1e-9)))\n            if sims and max(sims, key=sims.get) != by_id[lid][\"speaker\"]:\n                wrong.append((lid, sims))\n        report[\"attribution\"] = {\"misattributed\": wrong}\n        if wrong:\n            report.update(status=\"FALLBACK\", why=f\"speaker attribution unclear for {[w[0] for w in wrong]}\")\n    if report[\"status\"] == \"PASS\" and len(pieces) == len(lines):\n        for lid, (seg, sr, m) in pieces.items():\n            sf.write(out_dir / f\"{lid}.wav\", seg, sr)\n            report[\"lines\"][lid] = {k: v for k, v in m.items() if k != \"words\"} | {\"words\": m.get(\"words\")}\n    elif report[\"status\"] == \"PASS\":\n        report.update(status=\"FALLBACK\", why=\"not every line could be made\")\n    report[\"calls\"] = ledger.calls(project)\n    return report\n", "layout_control.py": "\"\"\"From a shot's scene-state to (1) the expected track of every entity and (2) a layout control video.\n\nThe SAME expected tracks steer generation (VACE layout control: white background, one coloured 2-px box per\nsubject, the format of VACE's own `layout_bbox` annotator) and are the contract the generated clip is checked\nagainst (motion_check.py). Objects held by a character ride along with the holder's hand; a \"place\" event moves\nthe object from the hand to its target box during the event.\n\nPure numpy + Pillow; unit-tested in tests/test_scene_state.py.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nfrom scene_state import ShotState, Box, LOCOMOTION, cx\n\nPALETTE = [(220, 30, 30), (30, 90, 220), (20, 160, 60), (230, 140, 0), (150, 40, 190), (0, 170, 170),\n           (120, 120, 120), (200, 0, 120)]\n\n\ndef _lerp(a: Box, b: Box, u: float) -> Box:\n    u = min(max(u, 0.0), 1.0)\n    u = u * u * (3 - 2 * u)                                        # ease in/out (people accelerate and stop)\n    return tuple(a[i] + (b[i] - a[i]) * u for i in range(4))\n\n\ndef hand_box(owner: Box, facing: str | None, size=0.06) -> Box:\n    \"\"\"Where a held object sits: at hand height, on the side the person faces.\"\"\"\n    hx = owner[2] - size * 0.2 if facing == \"right\" else owner[0] + size * 0.2 if facing == \"left\" else cx(owner)\n    hy = owner[1] + (owner[3] - owner[1]) * 0.55\n    return (hx - size / 2, hy - size / 2, hx + size / 2, hy + size / 2)\n\n\ndef expected_tracks(state: ShotState, n: int) -> dict[str, list[Box | None]]:\n    \"\"\"Box of every entity at each of n frames (None = not on screen).\"\"\"\n    out = {k: [] for k in state.start}\n    persons = {e.actor for e in state.events} | {k for k, v in state.start.items() if v.facing}\n    for e in state.events:\n        out.setdefault(e.actor, [])\n        if e.obj:\n            out.setdefault(e.obj, [])\n    for i in range(n):\n        t = i / max(n - 1, 1)\n        boxes = {k: v.box for k, v in state.start.items()}\n        facing = {k: v.facing for k, v in state.start.items()}\n        holders = dict(state.holders)\n        for e in state.events:\n            if t < e.t[0]:\n                break\n            u = (t - e.t[0]) / (e.t[1] - e.t[0])\n            if e.type in LOCOMOTION and e.to_box is not None and e.actor in boxes:\n                if cx(e.to_box) != cx(boxes[e.actor]):\n                    facing[e.actor] = \"right\" if cx(e.to_box) > cx(boxes[e.actor]) else \"left\"\n                boxes[e.actor] = _lerp(boxes[e.actor], e.to_box, u)\n            elif e.type in (\"take_out\", \"pick_up\") and e.obj:\n                if u >= 1:\n                    holders[e.obj] = e.actor\n            elif e.type in (\"place\", \"hand_over\") and e.obj and e.actor in boxes:\n                start = hand_box(boxes[e.actor], facing.get(e.actor))\n                if e.type == \"place\":\n                    end = e.to_box or (boxes.get(e.target) and _on_top(boxes[e.target])) or start\n                else:\n                    end = hand_box(boxes[e.target], facing.get(e.target)) if e.target in boxes else start\n                if u >= 1:\n                    holders[e.obj] = e.target\n                    boxes[e.obj] = end\n                else:\n                    holders[e.obj] = \"__moving__\"\n                    boxes[e.obj] = _lerp(start, end, u)\n        for obj, h in holders.items():                             # held objects follow the holder's hand\n            if obj not in out or h == \"__moving__\" or h not in boxes:\n                continue\n            if h in persons:\n                boxes[obj] = hand_box(boxes[h], facing.get(h))\n            elif obj not in boxes:                                 # resting on a counter/table\n                boxes[obj] = _on_top(boxes[h])\n        for k in out:\n            out[k].append(boxes.get(k))\n    return out\n\n\ndef _on_top(anchor: Box, size=0.06) -> Box:\n    x = cx(anchor)\n    return (x - size / 2, anchor[1] - size, x + size / 2, anchor[1])\n\n\ndef control_video(tracks: dict[str, list[Box | None]], w: int, h: int, colors: dict[str, tuple] | None = None,\n                  skip: set[str] = frozenset()):\n    \"\"\"VACE layout control frames (PIL images): white background, one coloured 2-px box outline per entity.\"\"\"\n    names = [k for k in tracks if k not in skip]\n    # colour by position in the FULL track list, so a subject keeps its colour when others are left out\n    colors = colors or {k: PALETTE[i % len(PALETTE)] for i, k in enumerate(tracks)}\n    n = len(next(iter(tracks.values()))) if tracks else 0\n    frames = []\n    for i in range(n):\n        im = Image.new(\"RGB\", (w, h), (255, 255, 255))\n        d = ImageDraw.Draw(im)\n        for k in names:\n            b = tracks[k][i]\n            if b is None:\n                continue\n            x0, y0, x1, y1 = [int(round(v * s)) for v, s in zip(b, (w, h, w, h))]\n            d.rectangle([x0, y0, x1, y1], outline=colors[k], width=2)\n        frames.append(im)\n    return frames\n\n\ndef composition_guide(state: ShotState, tracks, w: int, h: int, names: dict[str, str]):\n    \"\"\"Human-readable storyboard frame: first and last positions with arrows (saved next to the review images).\"\"\"\n    im = Image.new(\"RGB\", (w, h), (245, 246, 250))\n    d = ImageDraw.Draw(im)\n    for i, (k, seq) in enumerate(tracks.items()):\n        col = PALETTE[i % len(PALETTE)]\n        first = next((b for b in seq if b), None)\n        last = next((b for b in reversed(seq) if b), None)\n        if not first:\n            continue\n        p0 = [int(v * s) for v, s in zip(first, (w, h, w, h))]\n        d.rectangle(p0, outline=col, width=2)\n        d.text((p0[0] + 4, p0[1] + 4), names.get(k, k), fill=col)\n        if last and last != first:\n            p1 = [int(v * s) for v, s in zip(last, (w, h, w, h))]\n            d.rectangle(p1, outline=col, width=4)\n            d.line([((p0[0] + p0[2]) // 2, (p0[1] + p0[3]) // 2), ((p1[0] + p1[2]) // 2, (p1[1] + p1[3]) // 2)],\n                   fill=col, width=3)\n    d.text((8, h - 18), f\"{state.shot_id}  camera {state.camera.setup} ({state.camera.side}, {state.camera.move})\",\n           fill=(40, 40, 40))\n    return im\n", "motion_check.py": "\"\"\"Deterministic spatial / temporal checks of a generated clip against its scene-state contract.\n\nInputs\n  expected: {entity: [box or None per sampled frame]}     from layout_control.expected_tracks\n  observed: {entity: [(box, score) or None per frame]}    from a tracker (trackers.py)\n  camera:   per-frame cumulative camera shift (dx, dy, reliability) from estimate_camera_motion()\nEvery check returns a CheckResult: PASS, FAIL or UNKNOWN, with evidence. UNKNOWN is never treated as PASS\n(acceptance.py decides what happens).\n\nCamera compensation: when the camera pans right, everything static drifts LEFT on screen. The world motion of a\nsubject is therefore its screen motion PLUS the camera's own motion (both in frame widths). A clip in which the\ncamera pans while the person stands still must not count as \"the person walked\".\n\nPure numpy; unit-tested on synthetic videos in tests/test_motion_check.py.\n\"\"\"\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field, asdict\nimport numpy as np\n\nVERSION = 2                       # part of the validation cache key: bump when a rule or threshold changes\nTH = {\"region\": 0.18,             # start/end position tolerance (frame widths)\n      \"travel_share\": 0.4,        # observed travel must be \u2265 40 % of the planned travel, same direction\n      \"still\": 0.08,              # a stationary subject may drift at most this much\n      \"jump\": 0.22,               # one-sample jump larger than this = teleport\n      \"near\": 0.10,               # \"next to\" distance for hand-offs\n      \"visible\": 0.6,             # share of a window in which the subject must be detected\n      \"cam_reliable\": 0.08,       # phase-correlation peak ratio below this = camera estimate unreliable\n      \"stop_speed\": 0.012}        # per-sample speed (frame widths) that counts as \"stopped\"\n\n\n@dataclass\nclass CheckResult:\n    check: str\n    status: str                   # \"PASS\" | \"FAIL\" | \"UNKNOWN\"\n    entity: str | None = None\n    reason: str = \"\"\n    evidence: dict = field(default_factory=dict)\n    critical: bool = True\n\n    def to_dict(self):\n        return _plain(asdict(self))\n\n\ndef _plain(x):\n    \"\"\"numpy numbers -> plain Python (sidecars are JSON).\"\"\"\n    if isinstance(x, dict):\n        return {k: _plain(v) for k, v in x.items()}\n    if isinstance(x, (list, tuple)):\n        return [_plain(v) for v in x]\n    if isinstance(x, np.generic):\n        return x.item()\n    return x\n\n\ndef _c(b):\n    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)\n\n\n# ------------------------------------------------------------------------------------------- camera motion\nANCHOR_GAP = 8                    # frames between camera-motion anchor frames\n\n\ndef estimate_camera_motion(frames: np.ndarray, masks: list[list[tuple]] | None = None, scale: int = 4,\n                           margin: float = 0.03):\n    \"\"\"Cumulative global shift per frame (fractions of width/height) by phase correlation of grey frames.\n    Moving subjects are blanked so a walking person doesn't look like a camera pan; for each compared pair the union\n    of both frames' subject boxes (+ margin) is blanked in BOTH images. Callers pass detected AND planned subject\n    boxes: when the detector misses the subject and nothing is blanked, a walking person alone measures as a ~0.3\n    frame-width \"camera move\" on a static shot (tests/test_speed.py) \u2014 the false 'camera moves 0.25 frame widths'.\n    Sub-pixel peaks + anchor frames make slow real pans measurable (whole-pixel peaks read them as 0).\n    masks[i]: boxes (x0, y0, x1, y1, 0..1) to ignore in frame i. Returns (shift [n,2], reliability [n]).\"\"\"\n    n, h, w = frames.shape[:3]\n    g = frames[:, ::scale, ::scale].astype(np.float32).mean(axis=3)\n    hh, ww = g.shape[1:]\n    win = np.outer(np.hanning(hh), np.hanning(ww)).astype(np.float32)\n    shift = np.zeros((n, 2), np.float32)\n    rel = np.ones(n, np.float32)\n\n    def blank(img, boxes):\n        img = img.copy()\n        if boxes:\n            keep = np.ones(img.shape, bool)\n            for b in boxes:\n                x0, y0, x1, y1 = [int(round(v * s)) for v, s in\n                                  zip((b[0] - margin, b[1] - margin, b[2] + margin, b[3] + margin), (ww, hh, ww, hh))]\n                keep[max(y0, 0):max(y1, 0), max(x0, 0):max(x1, 0)] = False\n            fill = img[keep].mean() if keep.any() else img.mean()\n            img[~keep] = fill\n        return np.fft.fft2((img - img.mean()) * win)\n\n    def peak(r):\n        \"\"\"Correlation peak with sub-pixel precision (parabola through the peak and its two neighbours per axis):\n        on a frame shrunk 8x a slow pan moves ~0.2 px per frame, which a whole-pixel peak rounds to 0 every time.\"\"\"\n        py, px = np.unravel_index(np.argmax(r), r.shape)\n        def sub(a, b, c):\n            den = a - 2 * b + c\n            return 0.0 if abs(den) < 1e-12 else float(np.clip(0.5 * (a - c) / den, -0.5, 0.5))\n        fy = py + sub(r[(py - 1) % hh, px], r[py, px], r[(py + 1) % hh, px])\n        fx = px + sub(r[py, (px - 1) % ww], r[py, px], r[py, (px + 1) % ww])\n        return (fx - ww if fx > ww / 2 else fx), (fy - hh if fy > hh / 2 else fy)\n\n    anchor = 0\n    for i in range(1, n):\n        # compare with an anchor frame up to ANCHOR_GAP frames back (not just the previous frame): a slow pan adds up\n        # to a measurable shift before it is read, so rounding errors don't accumulate frame after frame\n        boxes = []\n        if masks:\n            for j in (anchor, i):\n                if j < len(masks):\n                    boxes += list(masks[j] or [])\n        A, B = blank(g[anchor], boxes), blank(g[i], boxes)\n        R = B * np.conj(A)\n        R /= np.abs(R) + 1e-6\n        r = np.abs(np.fft.ifft2(R))\n        dx, dy = peak(r)\n        # content moved by (dx, dy) pixels: the camera moved the opposite way\n        shift[i] = shift[anchor] + np.array([-dx / ww, -dy / hh], np.float32)\n        rel[i] = float(r.max() / (r.sum() + 1e-6) * r.size / 50.0)\n        if i - anchor >= ANCHOR_GAP:\n            anchor = i\n    return shift, rel\n\n\n# ------------------------------------------------------------------------------------------- helpers\ndef _window(seq, a, b):\n    n = len(seq)\n    return seq[int(a * (n - 1)): max(int(b * (n - 1)) + 1, int(a * (n - 1)) + 1)]\n\n\ndef _centers(obs):\n    return [(_c(o[0]) if o else None) for o in obs]\n\n\ndef _visible_share(obs):\n    return sum(o is not None for o in obs) / max(len(obs), 1)\n\n\ndef _mean_x(seq):\n    xs = [c[0] for c in seq if c]\n    return float(np.mean(xs)) if xs else None\n\n\n# ------------------------------------------------------------------------------------------- checks\ndef check_presence(name, exp, obs, critical=True, small=False):\n    want = [i for i, e in enumerate(exp) if e is not None]\n    if not want:\n        return CheckResult(\"presence\", \"PASS\", name, \"not expected on screen\", critical=critical)\n    share = sum(obs[i] is not None for i in want) / len(want)\n    ev = {\"visible_share\": round(share, 2), \"frames_expected\": len(want)}\n    if share >= TH[\"visible\"]:\n        return CheckResult(\"presence\", \"PASS\", name, f\"visible in {share:.0%} of the frames\", ev, critical)\n    if small:                                       # small objects are often missed by detectors: not proof of absence\n        return CheckResult(\"presence\", \"UNKNOWN\", name, f\"small object detected in only {share:.0%} of frames\", ev, critical)\n    return CheckResult(\"presence\", \"FAIL\", name, f\"only visible in {share:.0%} of the frames it should be in\", ev, critical)\n\n\ndef check_region(name, which, exp, obs, critical=True):\n    \"\"\"Start or end position: the subject's centre is where the plan puts it (\u00b1TH['region']).\"\"\"\n    a, b = (0.0, 0.15) if which == \"start\" else (0.85, 1.0)\n    ce = _mean_x([_c(x) if x else None for x in _window(exp, a, b)])\n    co = _mean_x(_window(_centers(obs), a, b))\n    if ce is None:\n        return CheckResult(f\"{which}_position\", \"PASS\", name, \"not on screen then\", critical=critical)\n    if co is None:\n        return CheckResult(f\"{which}_position\", \"UNKNOWN\", name, f\"not detected at the {which}\", critical=critical)\n    ev = {\"expected_x\": round(ce, 2), \"observed_x\": round(co, 2)}\n    ok = abs(ce - co) <= TH[\"region\"]\n    side = lambda x: \"left\" if x < 0.4 else \"right\" if x > 0.6 else \"centre\"\n    return CheckResult(f\"{which}_position\", \"PASS\" if ok else \"FAIL\", name,\n                       f\"{which}s on the {side(co)} (planned: {side(ce)})\", ev, critical)\n\n\ndef check_direction(name, exp, obs, cam_shift, cam_rel, critical=True):\n    \"\"\"Travel direction and amount, after removing camera motion.\"\"\"\n    e0, e1 = next((x for x in exp if x), None), next((x for x in reversed(exp) if x), None)\n    if e0 is None:\n        return CheckResult(\"direction\", \"PASS\", name, \"not on screen\", critical=critical)\n    planned = _c(e1)[0] - _c(e0)[0]\n    cs = _centers(obs)\n    idx = [i for i, c in enumerate(cs) if c is not None]\n    if len(idx) < 2:\n        return CheckResult(\"direction\", \"UNKNOWN\", name, \"not tracked in enough frames\", critical=critical)\n    i0, i1 = idx[0], idx[-1]\n    screen = cs[i1][0] - cs[i0][0]\n    cam = float(cam_shift[i1][0] - cam_shift[i0][0]) if cam_shift is not None else 0.0\n    world = screen + cam\n    reliable = cam_rel is None or float(np.median(cam_rel[i0 + 1:i1 + 1])) >= TH[\"cam_reliable\"]\n    ev = {\"planned_dx\": round(planned, 3), \"screen_dx\": round(screen, 3), \"camera_dx\": round(cam, 3),\n          \"world_dx\": round(world, 3), \"camera_estimate_reliable\": reliable}\n    if not reliable and abs(cam) > 0.05:\n        return CheckResult(\"direction\", \"UNKNOWN\", name, \"camera motion can't be measured reliably\", ev, critical)\n    if abs(planned) < 0.05:                          # should stay put\n        ok = abs(world) <= TH[\"still\"]\n        return CheckResult(\"stationary\", \"PASS\" if ok else \"FAIL\", name,\n                           \"stays in place\" if ok else f\"drifts {world:+.2f} frame widths (should stay still)\", ev, critical)\n    if np.sign(world) != np.sign(planned) and abs(world) > TH[\"still\"]:\n        return CheckResult(\"direction\", \"FAIL\", name, f\"moves {'right' if world > 0 else 'left'} but should move \"\n                           f\"{'right' if planned > 0 else 'left'}\", ev, critical)\n    if abs(world) < TH[\"travel_share\"] * abs(planned):\n        return CheckResult(\"direction\", \"FAIL\", name, f\"travels only {abs(world):.2f} of the planned \"\n                           f\"{abs(planned):.2f} frame widths\", ev, critical)\n    return CheckResult(\"direction\", \"PASS\", name, f\"moves {'right' if world > 0 else 'left'} as planned\", ev, critical)\n\n\ndef check_teleport(name, obs, critical=True):\n    cs = [c for c in _centers(obs)]\n    jumps = [abs(b[0] - a[0]) + abs(b[1] - a[1]) for a, b in zip(cs, cs[1:]) if a and b]\n    big = max(jumps) if jumps else 0.0\n    ok = big <= TH[\"jump\"]\n    return CheckResult(\"teleport\", \"PASS\" if ok else \"FAIL\", name,\n                       \"moves continuously\" if ok else f\"jumps {big:.2f} frame widths between two frames\",\n                       {\"max_jump\": round(big, 3)}, critical)\n\n\ndef check_stop_before(actor, obs, t_action, critical=True):\n    \"\"\"The actor has (nearly) stopped just before a hand action starts.\"\"\"\n    cs = _centers(obs)\n    n = len(cs)\n    i = int(t_action * (n - 1))\n    seg = [c for c in cs[max(i - 2, 0): i + 1] if c]\n    if len(seg) < 2:\n        return CheckResult(\"stop_before_action\", \"UNKNOWN\", actor, \"not tracked around the action\", critical=critical)\n    speed = float(np.mean([abs(b[0] - a[0]) for a, b in zip(seg, seg[1:])]))\n    ok = speed <= TH[\"stop_speed\"] * 2\n    return CheckResult(\"stop_before_action\", \"PASS\" if ok else \"FAIL\", actor,\n                       \"stops before acting\" if ok else \"is still moving when the action starts\",\n                       {\"speed\": round(speed, 4)}, critical)\n\n\ndef _box_dist(c, box):\n    \"\"\"Distance from a point to a box (0 inside).\"\"\"\n    dx = max(box[0] - c[0], 0, c[0] - box[2]); dy = max(box[1] - c[1], 0, c[1] - box[3])\n    return float(np.hypot(dx, dy))\n\n\ndef check_transfer(obj, obs_obj, obs_from, exp_obj, critical=True):\n    \"\"\"Object placed / handed over: it starts at the giver and ENDS WHERE THE PLAN PUTS IT, away from the giver's\n    hand. (Being merely 'near' the target is not enough: a person standing at the counter with the case still in\n    her hand is near the counter too.)\"\"\"\n    co = _centers(obs_obj)\n    n = len(co)\n    k = max(n // 5, 1)\n    first = [c for c in co[:k] if c]\n    last = [c for c in co[-k:] if c]\n    if not first or not last:\n        return CheckResult(\"object_transfer\", \"UNKNOWN\", obj, \"object not tracked at the start and the end\",\n                           critical=critical)\n    giver = [b[0] for b in obs_from[:k] if b]\n    plan_end = next((b for b in reversed(exp_obj) if b), None)\n    if not giver or plan_end is None:\n        return CheckResult(\"object_transfer\", \"UNKNOWN\", obj, \"giver not tracked or no planned end position\",\n                           critical=critical)\n    s0 = np.mean(first, axis=0); s1 = np.mean(last, axis=0)\n    d_start = min(_box_dist(s0, g) for g in giver)\n    d_end = float(np.hypot(s1[0] - _c(plan_end)[0], s1[1] - _c(plan_end)[1]))\n    giver_end = [b[0] for b in obs_from[-k:] if b]\n    left_hand = not giver_end or min(_box_dist(s1, g) for g in giver_end) > 0.01 or d_end <= TH[\"near\"] / 2\n    ev = {\"start_distance_to_giver\": round(d_start, 3), \"end_distance_to_planned_spot\": round(d_end, 3),\n          \"left_the_hand\": bool(left_hand)}\n    ok = d_start <= TH[\"near\"] and d_end <= TH[\"near\"] and left_hand\n    return CheckResult(\"object_transfer\", \"PASS\" if ok else \"FAIL\", obj,\n                       \"goes from the giver to the planned spot\" if ok else\n                       (\"never leaves the giver\" if not left_hand or d_end > 3 * TH[\"near\"] else \"ends in the wrong place\"),\n                       ev, critical)\n\n\ndef check_object_support(obj, obs_obj, obs_people: dict, critical=True):\n    \"\"\"Objects don't move by themselves: whenever the object moves, some person must be within reach of it.\"\"\"\n    co = _centers(obs_obj)\n    bad, moving = 0, 0\n    for i in range(1, len(co)):\n        if not co[i] or not co[i - 1]:\n            continue\n        if abs(co[i][0] - co[i - 1][0]) + abs(co[i][1] - co[i - 1][1]) <= 0.015:\n            continue\n        moving += 1\n        near_someone = any(seq[i] and _box_dist(co[i], seq[i][0]) <= TH[\"near\"] for seq in obs_people.values()\n                           if i < len(seq))\n        bad += not near_someone\n    ok = bad <= 1\n    return CheckResult(\"object_support\", \"PASS\" if ok else \"FAIL\", obj,\n                       \"only moves when someone carries it\" if ok else\n                       f\"moves on its own in {bad} of {moving} moving frames (nobody holding it)\",\n                       {\"unsupported_moving_frames\": bad, \"moving_frames\": moving}, critical)\n\n\ndef check_order(times: dict[str, float | None], order: list[str], critical=True):\n    \"\"\"Events observed in the planned order (times = when each event was seen to complete).\"\"\"\n    seen = [(k, times.get(k)) for k in order]\n    if any(t is None for _, t in seen):\n        return CheckResult(\"event_order\", \"UNKNOWN\", None, \"not every event could be timed\",\n                           {\"times\": {k: t for k, t in seen}}, critical)\n    ok = all(a[1] <= b[1] + 1e-6 for a, b in zip(seen, seen[1:]))\n    return CheckResult(\"event_order\", \"PASS\" if ok else \"FAIL\", None,\n                       \"events in the planned order\" if ok else \"events happen in the wrong order\",\n                       {\"times\": {k: round(t, 3) for k, t in seen}}, critical)\n\n\ndef arrival_time(obs, target_x, tol=0.08):\n    \"\"\"Fraction of the clip at which a subject first reaches target_x (None if never).\"\"\"\n    cs = _centers(obs)\n    for i, c in enumerate(cs):\n        if c and abs(c[0] - target_x) <= tol:\n            return i / max(len(cs) - 1, 1)\n    return None\n\n\ndef check_completes_before_cut(name, t_done, clip_seconds, used_seconds, stretch, critical=True):\n    \"\"\"Editing keeps the clip up to `used_seconds` of shot time; the action must be complete before that.\"\"\"\n    if t_done is None:\n        return CheckResult(\"completes_before_cut\", \"UNKNOWN\", name, \"completion time unknown\", critical=critical)\n    done_at = t_done * clip_seconds * stretch\n    ok = done_at <= used_seconds - 0.2\n    return CheckResult(\"completes_before_cut\", \"PASS\" if ok else \"FAIL\", name,\n                       f\"done at {done_at:.1f}s of {used_seconds:.1f}s\" + (\"\" if ok else \" (would be cut off)\"),\n                       {\"done_at_s\": round(done_at, 2), \"kept_s\": round(used_seconds, 2)}, critical)\n\n\ndef validate_clip(state, expected, observed, cam_shift, cam_rel, entities, clip_seconds=5.0, used_seconds=5.0,\n                  stretch=1.0):\n    \"\"\"All deterministic checks for one clip. Returns a list of CheckResult.\"\"\"\n    from scene_state import LOCOMOTION, MANIPULATION\n    res = []\n    small = {k for k, e in entities.items() if getattr(e, \"small\", False)}\n    anchors = {k for k, e in entities.items() if getattr(e, \"kind\", \"\") == \"anchor\"}\n    for name, exp in expected.items():\n        if name in anchors:\n            continue\n        obs = observed.get(name) or [None] * len(exp)\n        res.append(check_presence(name, exp, obs, small=name in small))\n        if name in small:\n            continue                                         # positions of tiny objects: left to the VLM / review\n        res.append(check_region(name, \"start\", exp, obs))\n        res.append(check_region(name, \"end\", exp, obs))\n        res.append(check_direction(name, exp, obs, cam_shift, cam_rel))\n        res.append(check_teleport(name, obs))\n        if getattr(entities.get(name), \"kind\", \"\") == \"prop\":\n            people = {k: observed.get(k) or [] for k, e in entities.items() if getattr(e, \"kind\", \"\") == \"character\"}\n            res.append(check_object_support(name, obs, people))\n    times, order = {}, []\n    for e in state.events:\n        if e.type in LOCOMOTION and e.to_box:\n            t = arrival_time(observed.get(e.actor) or [], (e.to_box[0] + e.to_box[2]) / 2)\n            times[e.id] = t; order.append(e.id)\n        if e.type in (\"place\", \"hand_over\", \"pick_up\") and e.critical:\n            obs_a = observed.get(e.actor) or []\n            res.append(check_stop_before(e.actor, obs_a, e.t[0]))\n            if e.obj and e.obj not in small:\n                res.append(check_transfer(e.obj, observed.get(e.obj) or [], obs_a, expected.get(e.obj) or []))\n            times[e.id] = None if (e.obj in small or not observed.get(e.obj)) else e.t[1]\n            order.append(e.id)\n    if len(order) >= 2:\n        res.append(check_order(times, order))\n    for e in state.events:\n        if e.critical and e.id in times:\n            res.append(check_completes_before_cut(e.id, times[e.id], clip_seconds, used_seconds, stretch))\n    if cam_shift is not None and state.camera.move == \"static\":\n        drift = float(np.abs(cam_shift[-1]).max())\n        res.append(CheckResult(\"camera_static\", \"PASS\" if drift <= 0.06 else \"FAIL\", None,\n                               \"camera stays still\" if drift <= 0.06 else f\"camera moves {drift:.2f} frame widths \"\n                               \"(plan: static) \u2014 subjects may only seem to move\", {\"camera_drift\": round(drift, 3)}))\n    return res\n\n\ndef run_checks(fn, *args, **kw):\n    \"\"\"Run a validator; an exception becomes an explicit UNKNOWN result (never a silent pass, never a crash).\"\"\"\n    try:\n        return fn(*args, **kw)\n    except Exception as e:                                   # noqa: BLE001 - report every failure as evidence\n        return [CheckResult(\"validator_error\", \"UNKNOWN\", None, f\"{type(e).__name__}: {e}\"[:300])]\n", "musiclib.py": "\"\"\"Background music from a licensed library, arranged by chapter.\n\nWhy: the old score was synthesised drones and plucks; it sounded amateur and didn't follow the story.\nResearch (see docs/08 section 22): explainer music should be simple, instrumental, mid-tempo, sit ~12\u201318 dB\nunder the voice, and change only at chapter boundaries with smooth crossfades.\n\nLibrary: Kevin MacLeod (incompetech.com), Creative Commons Attribution 4.0. incompetech's FAQ: \"Can I use your\nmusic for YouTube videos? Yes, AND you can monetize the videos. Be sure to credit me.\" The credit text below is\nadded to the YouTube description automatically. Tracks were chosen from incompetech's own catalogue data\n(feel / bpm / description). Download URLs were checked (HTTP 200) on 2026-10-15.\n\nPure Python: arrange() and credits() are unit-tested locally (tests/test_musiclib.py).\n\"\"\"\nimport urllib.parse\n\nBASE = \"https://incompetech.com/music/royalty-free/mp3-royaltyfree/\"\nLICENSE = \"Creative Commons: By Attribution 4.0 https://creativecommons.org/licenses/by/4.0/\"\n\n# title -> catalogue facts (from incompetech's pieces.json) + the moods we use it for\nTRACKS = {\n    \"Easy Lemon\":     {\"bpm\": 82,  \"bytes\": 5057420, \"feel\": \"Bright, Calming, Relaxed\",\n                       \"note\": \"\\\"Takes voice-over very well\\\" (catalogue)\", \"moods\": [\"calm\", \"explain\", \"curious\"]},\n    \"Wallpaper\":      {\"bpm\": 92,  \"bytes\": 8804339, \"feel\": \"Bouncy, Bright, Calming, Uplifting\",\n                       \"note\": \"\\\"Very very clean mix that is non-offensive\\\" (catalogue)\", \"moods\": [\"explain\", \"busy\", \"calm\"]},\n    \"Investigations\": {\"bpm\": 94,  \"bytes\": 3745876, \"feel\": \"Humorous, Mysterious\",\n                       \"note\": \"light mystery: hooks and 'where is it?' moments\", \"moods\": [\"mystery\", \"curious\", \"tension\"]},\n    \"Carefree\":       {\"bpm\": 96,  \"bytes\": 6566713, \"feel\": \"Bouncy, Bright, Calming, Uplifting\",\n                       \"note\": \"\\\"Nothing says 'It'll be okay' quite like a ukulele\\\" (catalogue)\", \"moods\": [\"joy\", \"payoff\", \"resolution\"]},\n    \"Inspired\":       {\"bpm\": 120, \"bytes\": 9156393, \"feel\": \"Bright, Relaxed, Calming, Uplifting\",\n                       \"note\": \"\\\"corporate films of intricate industrial processes\\\" (catalogue)\", \"moods\": [\"process\", \"busy\"]},\n    \"Life of Riley\":  {\"bpm\": 102, \"bytes\": 7533831, \"feel\": \"Bright, Relaxed, Uplifting\",\n                       \"note\": \"cheery mid-tempo\", \"moods\": [\"joy\", \"busy\"]},\n    \"Local Forecast - Elevator\": {\"bpm\": 82, \"bytes\": 7570404, \"feel\": \"Bouncy, Bright, Grooving\",\n                       \"note\": \"waiting / elevator feel\", \"moods\": [\"waiting\", \"reflective\"]},\n    \"Thinking Music\": {\"bpm\": 125, \"bytes\": 6270785, \"feel\": \"Dark, Humorous, Suspenseful\",\n                       \"note\": \"comedic 'hard thinking'\", \"moods\": [\"thinking\", \"tension\"]},\n}\nDEFAULT_BY_MOOD = {\"mystery\": \"Investigations\", \"curious\": \"Investigations\", \"tension\": \"Investigations\",\n                   \"calm\": \"Easy Lemon\", \"explain\": \"Easy Lemon\", \"busy\": \"Wallpaper\", \"process\": \"Wallpaper\",\n                   \"waiting\": \"Local Forecast - Elevator\", \"reflective\": \"Easy Lemon\", \"joy\": \"Carefree\",\n                   \"payoff\": \"Carefree\", \"resolution\": \"Carefree\", \"awe\": \"Easy Lemon\", \"sad\": \"Easy Lemon\"}\n\n\ndef url(title):\n    return BASE + urllib.parse.quote(title) + \".mp3\"\n\n\ndef arrange(plan, timeline, total):\n    \"\"\"Music cues: one track per chapter, crossfading at chapter boundaries.\n\n    plan[\"music\"][\"cues\"]: [{\"from\": \"sh01\", \"track\": \"Investigations\", \"gain_db\": 0, \"offset\": 0}, ...]\n    Without cues: a new cue starts wherever the shot's \"chapter\" changes, using the chapter's mood.\n    timeline: [{\"shot_id\", \"start\", \"dur\", \"mood\"}]. Returns [{\"track\", \"start\", \"end\", \"gain_db\", \"offset\"}].\"\"\"\n    music = plan.get(\"music\", {})\n    start_of = {r[\"shot_id\"]: r[\"start\"] for r in timeline}\n    shots = {s[\"shot_id\"]: s for s in plan.get(\"shots\", [])}\n    cues = []\n    if music.get(\"cues\"):\n        for c in music[\"cues\"]:\n            if c[\"from\"] not in start_of:\n                raise ValueError(f\"music cue starts at unknown shot {c['from']}\")\n            if c[\"track\"] not in TRACKS:\n                raise ValueError(f\"music track '{c['track']}' is not in the licensed library {sorted(TRACKS)}\")\n            cues.append({\"track\": c[\"track\"], \"start\": start_of[c[\"from\"]], \"gain_db\": c.get(\"gain_db\", 0.0),\n                         \"offset\": c.get(\"offset\", 0.0)})\n    else:\n        # no cues in the plan: one track per chapter if the plan has chapters, otherwise follow the scene moods;\n        # either way a track plays at least MIN_CUE seconds (switching more often feels restless)\n        MIN_CUE = 12.0\n        runs = []\n        for r in timeline:\n            sh = shots.get(r[\"shot_id\"], {})\n            mood = sh.get(\"music_mood\") or r.get(\"mood\", \"calm\")\n            key = sh.get(\"chapter\") or DEFAULT_BY_MOOD.get(mood, \"Easy Lemon\")\n            if runs and runs[-1][\"key\"] == key:\n                runs[-1][\"end\"] = r[\"start\"] + r[\"dur\"]\n            else:\n                runs.append({\"key\": key, \"track\": DEFAULT_BY_MOOD.get(mood, \"Easy Lemon\"), \"start\": r[\"start\"],\n                             \"end\": r[\"start\"] + r[\"dur\"]})\n        merged_runs = []\n        for run in runs:\n            if merged_runs and run[\"end\"] - run[\"start\"] < MIN_CUE:\n                merged_runs[-1][\"end\"] = run[\"end\"]            # too short: the previous track keeps playing\n            else:\n                merged_runs.append(run)\n        if len(merged_runs) > 1 and merged_runs[0][\"end\"] - merged_runs[0][\"start\"] < MIN_CUE:\n            merged_runs[1][\"start\"] = merged_runs[0][\"start\"]; merged_runs.pop(0)\n        cues = [{\"track\": r[\"track\"], \"start\": r[\"start\"], \"gain_db\": 0.0, \"offset\": 0.0} for r in merged_runs]\n    cues.sort(key=lambda c: c[\"start\"])\n    merged = []                                         # two chapters in a row with the same track: keep playing\n    for c in cues:\n        if merged and merged[-1][\"track\"] == c[\"track\"] and merged[-1][\"gain_db\"] == c[\"gain_db\"]:\n            continue\n        merged.append(c)\n    for a, b in zip(merged, merged[1:]):\n        a[\"end\"] = b[\"start\"]\n    if merged:\n        merged[0][\"start\"] = 0.0\n        merged[-1][\"end\"] = total\n    return merged\n\n\ndef credits(cues):\n    \"\"\"The attribution text CC BY 4.0 requires (incompetech's format), one block per track used.\"\"\"\n    seen, out = set(), []\n    for c in cues:\n        if c[\"track\"] not in seen:\n            seen.add(c[\"track\"])\n            out.append(f\"\\\"{c['track']}\\\" Kevin MacLeod (incompetech.com)\\nLicensed under {LICENSE}\")\n    return \"\\n\".join(out)\n", "parler_worker.py": "\"\"\"Telugu (and other Indic) voices: Indic Parler-TTS, run in its OWN virtual environment.\n\nparler-tts pins transformers==4.46.1; the notebook needs transformers 5.x (Qwen3-VL). So this file is NOT imported by\nthe notebook: the narration cell starts it once with the TTS environment's python, keeps it running, and sends one\nJSON request per line on stdin; the model loads once.\n\n  request:  {\"text\": \"...\", \"description\": \"Lalitha speaks ...\", \"seed\": 123, \"out\": \"/path/x.wav\"}\n  response: {\"ok\": true, \"out\": \"...\", \"seconds\": 3.2, \"gen_s\": 4.1}  |  {\"ok\": false, \"error\": \"...\"}\n\nThe first line printed is {\"ready\": true, \"sr\": ..., \"device\": ..., \"dtype\": ..., \"load_s\": ...}. Everything else\n(warnings, progress) goes to stderr, which the notebook writes to its log file.\n\nRun: python parler_worker.py ai4bharat/indic-parler-tts cuda float32\n\"\"\"\nimport json, sys, time\n\nMODEL = \"ai4bharat/indic-parler-tts\"            # Apache-2.0; gated (free, auto-approved): needs a Hugging Face token\nPARLER_GIT = \"git+https://github.com/huggingface/parler-tts.git@d108732cd57788ec86bc857d99a6cabd66663d68\"\n# Telugu speakers the model was trained on (model card: \"Prakash, Lalitha, Kiran\"; recommended: Prakash, Lalitha)\nSPEAKERS = {\"te\": [\"Prakash\", \"Lalitha\", \"Kiran\"]}\n# Everything parler-tts needs on top of Colab, resolved for Python 3.13 / x86_64 Linux with\n#   uv pip compile --python-version 3.13 --python-platform x86_64-manylinux_2_28  (parler-tts @ PARLER_GIT)\n# then torch, torchaudio, triton, numpy, nvidia-* and cuda-* REMOVED: the environment sees Colab's own (matching) copies.\n# Installed with --no-deps, so nothing else is downloaded or upgraded.\nREQUIREMENTS = \"\"\"\nabsl-py==2.5.1\nargbind==0.3.9\nasttokens==3.0.2\ncertifi==2026.7.22\ncffi==2.1.1\ncharset-normalizer==3.5.2\ncloudpickle==3.1.2\ncontourpy==1.4.0\ncycler==0.12.1\ndecorator==5.3.1\ndescript-audio-codec==1.0.0\ndescript-audiotools @ git+https://github.com/descriptinc/audiotools@348ebf2034ce24e2a91a553e3171cb00c0c71678\ndocstring-parser==0.18.0\neinops==0.8.2\nexecuting==2.3.0\nffmpy==1.0.0\nfilelock==4.2.0\nfire==0.7.1\nflatten-dict==0.5.0\nfonttools==4.67.0\nfsspec==2026.9.0\ngrpcio==1.84.0\nhf-xet==1.7.0\nhuggingface-hub==0.36.2\nidna==3.20\nimportlib-resources==7.1.0\nipython==9.17.1\nipython-pygments-lexers==1.1.1\njedi==0.20.1\njinja2==3.1.6\njoblib==1.6.0\njulius==0.2.8\nkiwisolver==1.5.1\nlazy-loader==0.6\nlibrosa==1.0.0\nllvmlite==0.50.0\nmarkdown==3.11.1\nmarkdown-it-py==4.2.0\nmarkdown2==2.5.5\nmarkupsafe==3.0.4\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nmdurl==0.1.2\nmpmath==1.3.0\nmsgpack==1.2.3\nnarwhals==2.27.1\nnetworkx==3.7\nnumba==0.68.0\npackaging==26.3\nparler-tts @ git+https://github.com/huggingface/parler-tts.git@d108732cd57788ec86bc857d99a6cabd66663d68\nparso==0.8.7\npexpect==4.9.0\npillow==13.0.0\nplatformdirs==4.13.3\npooch==1.9.0\nprompt-toolkit==3.0.53\nprotobuf==4.25.9\npsutil==7.2.2\nptyprocess==0.7.0\npure-eval==0.2.4\npycparser==3.11\npygments==2.21.0\npyloudnorm==0.2.0\npyparsing==3.3.3\npystoi==0.4.1\npython-dateutil==2.9.0.post0\npyyaml==6.0.3\nrandomname==0.2.1\nregex==2026.9.29\nrequests==2.34.2\nrich==15.0.0\nsafetensors==0.8.0\nscikit-learn==1.9.1\nscipy==1.18.1\nsentencepiece==0.2.2\nsetuptools==84.0.0\nsix==1.17.0\nsoundfile==0.14.0\nsoxr==1.1.0\nstack-data==0.6.3\nsympy==1.14.0\ntensorboard==2.20.0\ntensorboard-data-server==0.7.2\ntermcolor==3.3.0\nthreadpoolctl==3.7.0\ntokenizers==0.20.3\ntorch-stoi==0.2.3\ntqdm==4.70.1\ntraitlets==5.16.1\ntransformers==4.46.1\ntyping-extensions==4.16.0\nurllib3==2.8.0\nwcwidth==0.9.2\nwerkzeug==3.1.9\n\"\"\"\n\n\ndef main():\n    mid, device, dtype = sys.argv[1], sys.argv[2], sys.argv[3]\n    t0 = time.time()\n    proto, sys.stdout = sys.stdout, sys.stderr        # library prints must never mix into the JSON channel\n    def say(obj):\n        proto.write(json.dumps(obj) + \"\\n\"); proto.flush()\n    import numpy as np, soundfile as sf, torch\n    from parler_tts import ParlerTTSForConditionalGeneration\n    from transformers import AutoTokenizer, set_seed\n    dt = {\"float32\": torch.float32, \"float16\": torch.float16, \"bfloat16\": torch.bfloat16}[dtype]\n    model = ParlerTTSForConditionalGeneration.from_pretrained(mid, torch_dtype=dt).to(device).eval()\n    tok = AutoTokenizer.from_pretrained(mid)\n    desc_tok = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)\n    sr = model.config.sampling_rate\n    say({\"ready\": True, \"sr\": sr, \"device\": device, \"dtype\": dtype, \"load_s\": round(time.time() - t0, 1),\n         \"transformers\": __import__(\"transformers\").__version__})\n    for raw in sys.stdin:\n        if not raw.strip():\n            continue\n        try:\n            req = json.loads(raw)\n            t1 = time.time()\n            set_seed(int(req.get(\"seed\", 0)))\n            d = desc_tok(req[\"description\"], return_tensors=\"pt\").to(device)\n            p = tok(req[\"text\"], return_tensors=\"pt\").to(device)\n            with torch.no_grad():\n                gen = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,\n                                     prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)\n            audio = gen.float().cpu().numpy().squeeze()\n            if audio.size == 0 or not np.isfinite(audio).all():\n                raise RuntimeError(\"the voice model produced empty or broken audio\")\n            sf.write(req[\"out\"], audio.astype(np.float32), sr)\n            say({\"ok\": True, \"out\": req[\"out\"], \"seconds\": round(audio.size / sr, 2), \"gen_s\": round(time.time() - t1, 1)})\n        except Exception as e:\n            import traceback\n            traceback.print_exc(file=sys.stderr)\n            say({\"ok\": False, \"error\": f\"{type(e).__name__}: {str(e)[:300]}\"})\n\n\nif __name__ == \"__main__\":\n    main()\n", "planner.py": "\"\"\"Topic \u2192 fact-checked script \u2192 plan JSON, with a LOCAL open model (no paid API).\n\nThe LLM only writes DATA (a Draft: title, hook candidates, beats with narration + cited fact ids + a visual intent).\nEverything else is fixed, tested code:\n  parse + validate (Pydantic, with the error fed back for up to 2 retries)\n  \u2192 fact check every sentence against the facts it cites (factcheck.py), one rewrite for failures, then drop\n  \u2192 choose the hook by rules (sourced, short, a question or a surprising fact, names the topic)\n  \u2192 build the plan deterministically (cards / licensed images, claims with source URLs, music, metadata),\n    which the notebook's existing validator then checks again.\nDefault model: Qwen/Qwen3-4B-Instruct-2507 (Apache-2.0, ~8 GB fp16 \u2192 fits a T4 on its own);\nQwen/Qwen3-1.7B for low memory. Both verified Apache-2.0 on the Hub.\n\"\"\"\nfrom __future__ import annotations\n\nimport json, re, textwrap\nfrom typing import Literal\n\nfrom pydantic import BaseModel, Field, ValidationError, field_validator\n\nDEFAULT_MODEL = \"Qwen/Qwen3-4B-Instruct-2507\"\nWORDS_PER_SECOND = 2.5           # Kokoro narration pace incl. pauses (measured on the existing videos \u2248 2.4\u20132.7)\n\n\n# ------------------------------------------------------------------ the data the LLM writes\nclass Sentence(BaseModel):\n    text: str\n    kind: Literal[\"fact\", \"question\", \"analogy\"] = \"fact\"\n    fact_ids: list[str] = Field(default_factory=list)\n\n    @field_validator(\"text\")\n    @classmethod\n    def short(cls, v):\n        v = v.strip()\n        if not v or len(v.split()) > 24:\n            raise ValueError(\"each sentence must have 1\u201324 words\")\n        return v\n\n\nclass Visual(BaseModel):\n    kind: Literal[\"photo\", \"card\"]\n    description: str = \"\"                      # photo: what the picture shows (concrete nouns)\n    card_lines: list[str] = Field(default_factory=list)\n\n\nclass Beat(BaseModel):\n    narration: list[Sentence] = Field(min_length=1, max_length=3)\n    visual: Visual\n\n\nclass Draft(BaseModel):\n    title: str\n    hooks: list[Sentence] = Field(min_length=2, max_length=4)\n    beats: list[Beat] = Field(min_length=3, max_length=10)\n    takeaway: Sentence\n\n    @field_validator(\"title\")\n    @classmethod\n    def title_len(cls, v):\n        if len(v.split()) > 10:\n            raise ValueError(\"title: at most 10 words\")\n        return v.strip()\n\n\nclass Rewrite(BaseModel):\n    text: str\n    fact_ids: list[str] = Field(default_factory=list)       # missing \u2192 the original sentence's facts\n\n\nMIN_LENGTH_FRACTION = 0.5        # a script under half the target length after the fact check is refused, not shipped\n\n\ndef symbols(text):\n    \"\"\"Notation a beginner can't read aloud or follow (M\u2609, \u221d, \u2248 ...).\"\"\"\n    import unicodedata\n    return sorted({ch for ch in text if unicodedata.category(ch) in (\"So\", \"Sm\") and ch not in \"+=%\u00b0<>\"})\n\n\ndef _content(text):\n    return {w for w in re.findall(r\"[a-z]{4,}\", text.lower())}\n\n\ndef similar(a, b):\n    \"\"\"Word overlap (Jaccard) of two sentences' content words.\"\"\"\n    x, y = _content(a), _content(b)\n    return len(x & y) / max(len(x | y), 1)\n\n\nSYSTEM = \"\"\"You write scripts for short educational explainer videos for complete beginners.\nRules:\n- Use ONLY the numbered facts given. Every \"fact\" sentence must be fully supported by the facts it cites in fact_ids.\n- Never add a number, name, date or claim that is not in the cited facts. If unsure, leave it out.\n- Simple words. Explain an unfamiliar term in the same sentence. At most 20 words per sentence.\n- An everyday comparison is allowed as kind \"analogy\" (it must not add new facts). Questions use kind \"question\".\n- Build curiosity: question \u2192 explanation \u2192 payoff. No greetings, no filler, no repetition.\n- Answer with ONE JSON object and nothing else.\"\"\"\n\nSCHEMA_HINT = \"\"\"{\n  \"title\": \"short title, max 8 words\",\n  \"hooks\": [{\"text\": \"a gripping opening question or surprising sourced fact\", \"kind\": \"question|fact\", \"fact_ids\": [\"F2\"]}, ... 3 candidates],\n  \"beats\": [\n    {\"narration\": [{\"text\": \"...\", \"kind\": \"fact|question|analogy\", \"fact_ids\": [\"F1\"]}],\n     \"visual\": {\"kind\": \"photo\", \"description\": \"what a real photo/illustration should show, concrete nouns\"}},\n    {\"narration\": [...], \"visual\": {\"kind\": \"card\", \"card_lines\": [\"max 6 words\", \"max 6 words\"]}}\n  ],\n  \"takeaway\": {\"text\": \"one closing sentence\", \"kind\": \"fact\", \"fact_ids\": [\"F3\"]}\n}\"\"\"\n\n\ndef draft_prompt(topic, facts, target_s):\n    words = int(target_s * WORDS_PER_SECOND)\n    lines = \"\\n\".join(f\"{f.id}: {f.text}\" for f in facts)\n    return (f\"Topic: {topic}\\nTarget length: about {target_s} seconds = about {words} spoken words in total, \"\n            f\"in 5\u20137 beats of 1\u20132 sentences.\\nUse \"\n            f\"\\\"photo\\\" visuals for things a real photo or illustration can show, \\\"card\\\" for numbers, steps or \"\n            f\"comparisons.\\n\\nFacts:\\n{lines}\\n\\nReturn JSON in exactly this shape:\\n{SCHEMA_HINT}\")\n\n\n# ------------------------------------------------------------------ the local model\nclass LocalLLM:\n    def __init__(self, model_id=DEFAULT_MODEL, device=\"cuda\", cache_dir=None, log=print):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n        self.torch, self.device, self.log, self.model_id = torch, device, log, model_id\n        self.tok = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)\n        dt = torch.float16 if device == \"cuda\" else torch.float32\n        kw = {\"device_map\": {\"\": 0}} if device == \"cuda\" else {}\n        self.model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dt, cache_dir=cache_dir, **kw).eval()\n        if device != \"cuda\":\n            self.model.to(device)\n\n    def chat(self, system, user, max_new_tokens=1600):\n        msgs = [{\"role\": \"system\", \"content\": system}, {\"role\": \"user\", \"content\": user}]\n        try:\n            text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        except TypeError:\n            text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n        inp = self.tok(text, return_tensors=\"pt\").to(self.model.device)\n        with self.torch.inference_mode():\n            out = self.model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,\n                                      pad_token_id=self.tok.eos_token_id)\n        ans = self.tok.decode(out[0, inp[\"input_ids\"].shape[1]:], skip_special_tokens=True)\n        return re.sub(r\"(?s)<think>.*?</think>\", \"\", ans).strip()\n\n\ndef parse_json(text, model_cls):\n    \"\"\"The first complete JSON object in the answer, validated by the Pydantic model.\"\"\"\n    s = text.find(\"{\")\n    if s < 0:\n        raise ValueError(\"no JSON object in the answer\")\n    depth, inside, esc = 0, False, False\n    for i, ch in enumerate(text[s:], s):\n        if inside:\n            esc = (ch == \"\\\\\") and not esc\n            if ch == '\"' and not esc:\n                inside = False\n            continue\n        if ch == '\"':\n            inside = True\n        elif ch == \"{\":\n            depth += 1\n        elif ch == \"}\":\n            depth -= 1\n            if depth == 0:\n                return model_cls.model_validate(json.loads(text[s:i + 1]))\n    raise ValueError(\"the JSON object is not closed (answer cut off?)\")\n\n\ndef ask_json(llm, system, user, model_cls, tries=3, log=print, max_new_tokens=1600):\n    err = None\n    for k in range(tries):\n        prompt = user if err is None else (f\"{user}\\n\\nYour previous answer was invalid: {err}\\n\"\n                                           \"Return corrected JSON only.\")\n        ans = llm.chat(system, prompt, max_new_tokens)\n        try:\n            return parse_json(ans, model_cls)\n        except (ValueError, ValidationError, json.JSONDecodeError) as e:\n            err = str(e)[:600]\n            log(f\"planner: answer {k + 1} invalid: {err[:200]}\")\n    raise RuntimeError(f\"the planner model gave no valid JSON after {tries} tries: {err}\")\n\n\n# ------------------------------------------------------------------ fact-checked script\ndef make_script(topic, sheet, llm, nli, target_s=60, log=print, max_facts=30):\n    \"\"\"Draft \u2192 fact check \u2192 one rewrite \u2192 drop what is still unsupported. Returns (script dict, checks dict).\"\"\"\n    import factcheck\n    facts = sheet.facts[:max_facts]\n    by_id = {f.id: f for f in facts}\n    draft = ask_json(llm, SYSTEM, draft_prompt(topic, facts, target_s), Draft, log=log)\n    sentences = []                                             # (key, Sentence)\n    for i, h in enumerate(draft.hooks):\n        sentences.append((f\"hook{i}\", h))\n    for b, beat in enumerate(draft.beats):\n        for j, s in enumerate(beat.narration):\n            sentences.append((f\"b{b}s{j}\", s))\n    sentences.append((\"take\", draft.takeaway))\n\n    def items(pairs):\n        out = []\n        for key, s in pairs:\n            ids = [x for x in s.fact_ids if x in by_id]\n            out.append({\"id\": key, \"text\": s.text, \"question\": s.kind == \"question\" or s.text.endswith(\"?\"),\n                        \"analogy\": s.kind == \"analogy\",\n                        \"evidence\": [by_id[x].text for x in ids] or ([] if s.kind == \"fact\" else [f.text for f in facts[:8]])})\n        return out\n\n    checks = factcheck.check_sentences(items(sentences), nli, topic)\n    for key, s in sentences:                                 # beginner readability: no unexplained notation\n        if checks[key][\"status\"] == \"PASS\" and symbols(s.text):\n            checks[key].update(status=\"FAIL\", why=f\"symbols a beginner can't read: {' '.join(symbols(s.text))}\")\n    bad = [(k, s) for k, s in sentences if checks[k][\"status\"] != \"PASS\" and not k.startswith(\"hook\")]\n    for key, s in bad:                                         # one rewrite, restricted to the sentence's facts\n        ids = [x for x in s.fact_ids if x in by_id] or [f.id for f in facts[:6]]\n        user = (\"Rewrite this sentence for a beginner so that it says ONLY what these facts say. Keep it short.\\n\"\n                f\"Sentence: {s.text}\\nFacts:\\n\" + \"\\n\".join(f\"{x}: {by_id[x].text}\" for x in ids) +\n                '\\nReturn JSON: {\"text\": \"...\", \"fact_ids\": [\"...\"]}')\n        try:\n            rw = ask_json(llm, SYSTEM, user, Rewrite, tries=2, log=log, max_new_tokens=200)\n        except RuntimeError:\n            continue\n        # the rewrite must stay on the SAME facts: citing other facts would change what the beat says\n        new = Sentence(text=rw.text, kind=\"fact\", fact_ids=[x for x in rw.fact_ids if x in ids] or ids)\n        r = factcheck.check_sentences(items([(key, new)]), nli, topic)[key]\n        if r[\"status\"] == \"PASS\" and symbols(new.text):\n            r.update(status=\"FAIL\", why=f\"symbols a beginner can't read: {' '.join(symbols(new.text))}\")\n        r[\"rewritten_from\"] = s.text\n        checks[key] = r\n        if r[\"status\"] == \"PASS\":\n            _replace(draft, key, new)\n    # hook: the best SUPPORTED candidate (questions and sourced facts), short, naming the topic\n    tw = set(re.findall(r\"[a-z]{4,}\", topic.lower()))\n    def hook_score(i, h):\n        c = checks[f\"hook{i}\"]\n        return (c[\"status\"] == \"PASS\", h.kind == \"question\" or bool(re.search(r\"\\d\", h.text)),\n                bool(tw & set(re.findall(r\"[a-z]{4,}\", h.text.lower()))), -len(h.text.split()))\n    hi, hook = max(enumerate(draft.hooks), key=lambda x: hook_score(*x))\n    if checks[f\"hook{hi}\"][\"status\"] != \"PASS\":\n        raise RuntimeError(\"no hook candidate is supported by the sources; re-run or choose another topic\")\n    beats, said = [], [hook.text]\n    for b, beat in enumerate(draft.beats):\n        keep = []\n        for j, s in enumerate(beat.narration):\n            key = f\"b{b}s{j}\"\n            if checks[key][\"status\"] != \"PASS\":\n                log(f\"planner: dropped unsupported sentence in beat {b + 1}: {s.text!r} ({checks[key]['why']})\")\n            elif any(similar(s.text, x) >= 0.6 for x in said) or (\n                    s.kind == \"fact\" and any(factcheck.same_claim(nli, s.text, x) for x in said)):   # a repeat\n                checks[key].update(status=\"DROPPED\", why=\"repeats an earlier sentence\")\n                log(f\"planner: dropped repetition in beat {b + 1}: {s.text!r}\")\n            else:\n                keep.append(s); said.append(s.text)\n        if keep:\n            beats.append({\"narration\": [s.model_dump() for s in keep], \"visual\": beat.visual.model_dump()})\n    take = draft.takeaway.model_dump() if checks[\"take\"][\"status\"] == \"PASS\" and not any(\n        similar(draft.takeaway.text, x) >= 0.6 for x in said) else None\n    words = sum(len(s[\"text\"].split()) for b in beats for s in b[\"narration\"]) + len(hook.text.split()) + \\\n        (len(take[\"text\"].split()) if take else 0)\n    target_words = target_s * WORDS_PER_SECOND\n    if len(beats) < 2 or words < MIN_LENGTH_FRACTION * target_words:\n        raise RuntimeError(f\"only {words} words ({len(beats)} beats) survived the fact check, for a target of \"\n                           f\"~{target_words:.0f}: too thin to publish. Use the 4B planner model, a shorter target, \"\n                           \"or a topic Wikipedia covers better.\")\n    script = {\"title\": draft.title, \"hook\": hook.model_dump(), \"beats\": beats, \"takeaway\": take,\n              \"model\": getattr(llm, \"model_id\", \"?\")}\n    return script, checks\n\n\ndef _replace(draft, key, new):\n    if key == \"take\":\n        draft.takeaway = new\n    elif key.startswith(\"b\"):\n        b, j = (int(x) for x in re.findall(r\"\\d+\", key))\n        draft.beats[b].narration[j] = new\n\n\n# ------------------------------------------------------------------ deterministic plan builder\nMOODS = {\"curious\": {\"pause_after\": 0.55, \"grade\": \"cool\", \"move_amount\": 0.06,\n                     \"music\": {\"base\": 0.5, \"curious\": 0.7, \"tension\": 0.1, \"shimmer\": 0.4}},\n         \"calm\": {\"pause_after\": 0.5, \"grade\": \"neutral\", \"move_amount\": 0.05,\n                  \"music\": {\"base\": 0.4, \"curious\": 0.8, \"tension\": 0.0, \"shimmer\": 0.2}},\n         \"reflective\": {\"pause_after\": 0.6, \"grade\": \"neutral\", \"move_amount\": 0.05,\n                        \"music\": {\"base\": 0.5, \"curious\": 0.6, \"tension\": 0.0, \"shimmer\": 0.4}}}\nTONES = {\"curious\": {\"speed\": 0.96, \"pitch\": 0.2, \"gain_db\": 0.0, \"pause_after\": 0.55},\n         \"explainer\": {\"speed\": 1.0, \"pitch\": 0.0, \"gain_db\": 0.0, \"pause_after\": 0.45},\n         \"warm\": {\"speed\": 0.95, \"pitch\": 0.0, \"gain_db\": -0.5, \"pause_after\": 0.6}}\nLOOK = {\"bg_top\": \"#10233F\", \"bg_bottom\": \"#1B4060\", \"accent\": \"#19C3A6\", \"accent2\": \"#FF6B6B\", \"gold\": \"#FFC43D\"}\n\n\ndef card_lines(text, n=3, width=6):\n    \"\"\"Up to n short lines (\u2264 9 words each, the card limit) of WHOLE clauses from a sentence: clauses are joined\n    until there are \u2265 4 words (\"However,\" alone isn't a card), and a clause that doesn't fit is left out rather\n    than cut mid-phrase.\"\"\"\n    clauses, cur = [], []\n    for part in re.split(r\"(?<=[,;:])\\s+\", re.sub(r\"[.!]$\", \"\", text.strip())):\n        cur += part.rstrip(\",;:\").split()\n        if len(cur) >= 4:\n            clauses.append(cur); cur = []\n    if cur:\n        clauses.append(cur)\n    words = []\n    for c in clauses:\n        if len(words) + len(c) > n * width and words:\n            break\n        words += c\n    words = words[:n * width]\n    return [\" \".join(words[i:i + width]) for i in range(0, len(words), width)]\n\n\ndef card_supported(line, narration):\n    \"\"\"On-screen text must say what the narration says: its numbers and names appear in the narration, and it\n    shares at least one content word with it (\"Nothing escapes\" on a beat about gas clouds is refused).\"\"\"\n    import factcheck\n    bad_nums, bad_names = factcheck.static_checks(line, narration)\n    words = {w for w in re.findall(r\"[a-z]{4,}\", line.lower())}\n    said = {w for w in re.findall(r\"[a-z]{4,}\", narration.lower())}\n    stem = lambda w: w[:5]\n    return not bad_nums and not bad_names and bool({stem(w) for w in words} & {stem(w) for w in said})\n\n\ndef build_plan(topic, script, sheet, beat_assets, voice=\"af_heart\", project=None):\n    \"\"\"Script + chosen images \u2192 the cast-format plan the notebook renders. beat_assets[i]: manifest entry or None.\"\"\"\n    by_id = sheet.by_id()\n    used, shots = [], []\n\n    def line(s, tone):\n        ids = [x for x in s.get(\"fact_ids\", []) if x in by_id]\n        used.extend(ids)\n        return {\"speaker\": \"narrator\", \"tone\": tone, \"text\": s[\"text\"], **({\"story\": True} if s[\"kind\"] != \"fact\" else {})}, ids\n\n    h, ids = line(script[\"hook\"], \"curious\")\n    shots.append({\"shot_id\": \"sh01\", \"chapter\": \"hook\", \"recipe\": \"card\", \"mood\": \"curious\", \"seed\": 1,\n                  \"card\": {\"icon\": \"question\", \"lines\": card_lines(script[\"hook\"][\"text\"], 2, 7), \"bullets\": False},\n                  \"lines\": [h], \"claims\": ids, \"sfx\": [{\"type\": \"pop\", \"at\": 0.5, \"dur\": 0.4, \"gain_db\": -14}]})\n    for i, beat in enumerate(script[\"beats\"]):\n        sid = f\"sh{len(shots) + 1:02d}\"\n        tone = \"curious\" if beat[\"narration\"][0][\"kind\"] == \"question\" else \"explainer\"\n        ls, cid = [], []\n        for s in beat[\"narration\"]:\n            l, ids = line(s, tone); ls.append(l); cid += ids\n        shot = {\"shot_id\": sid, \"chapter\": f\"beat{i + 1}\", \"mood\": \"calm\" if i % 2 else \"curious\", \"seed\": i + 2,\n                \"lines\": ls, \"claims\": sorted(set(cid))}\n        a = beat_assets[i] if i < len(beat_assets) else None\n        if a is not None:\n            shot.update(recipe=\"stock\", move=\"push_in\" if i % 2 else \"pan_right\",\n                        stock={\"file\": a[\"file\"], \"credit\": a[\"credit_line\"], \"license\": a[\"license\"],\n                               \"license_ok\": a[\"license_ok\"], \"asset\": a[\"id\"], \"page\": a[\"page\"],\n                               \"description\": beat[\"visual\"].get(\"description\", \"\")})\n        else:\n            said = \" \".join(x[\"text\"] for x in beat[\"narration\"])\n            lines = [ln for ln in beat[\"visual\"].get(\"card_lines\", []) if 0 < len(ln.split()) <= 9][:4]\n            if not lines or not all(card_supported(ln, said) for ln in lines):\n                lines = card_lines(beat[\"narration\"][0][\"text\"])      # text taken from what is actually said\n            shot.update(recipe=\"card\", card={\"lines\": lines, \"bullets\": len(lines) > 2})\n        shots.append(shot)\n    if script.get(\"takeaway\"):\n        t, ids = line(script[\"takeaway\"], \"warm\")\n        shots.append({\"shot_id\": f\"sh{len(shots) + 1:02d}\", \"chapter\": \"payoff\", \"recipe\": \"card\", \"mood\": \"reflective\",\n                      \"seed\": 99, \"card\": {\"icon\": \"check\", \"lines\": card_lines(script[\"takeaway\"][\"text\"]),\n                                           \"bullets\": False}, \"lines\": [t], \"claims\": ids})\n    claims = [{\"id\": f.id, \"text\": f.text, \"sources\": [f.source_url], \"confidence\": \"sourced\",\n               \"source_revision\": f.revid} for f in sheet.facts if f.id in set(used)]\n    images = [a for a in beat_assets if a]\n    slug = re.sub(r\"[^a-z0-9]+\", \"_\", topic.lower()).strip(\"_\")[:40]\n    return {\n        \"plan_version\": 1, \"project\": project or f\"topic_{slug}\", \"language\": \"en\", \"title\": script[\"title\"],\n        \"logline\": f\"Auto-generated explainer: {topic}\", \"made_by\": f\"local planner ({script.get('model')})\",\n        \"style\": {\"suffix\": \"clean educational explainer\", \"negative\": \"text, watermark, logo\"},\n        \"look\": LOOK, \"characters\": {\"narrator\": {\"name\": \"Narrator\", \"voice\": voice, \"speed\": 1.0, \"pitch\": 0,\n                                                 \"fx\": \"none\", \"color\": \"#FFFFFF\", \"look\": None}},\n        \"tones\": TONES, \"moods\": MOODS, \"claims\": claims, \"entities\": {}, \"glossary\": {},\n        \"vocabulary\": sorted({w for f in sheet.facts for w in re.findall(r\"\\b[A-Z][a-z]{3,}\\b\", f.text)})[:40],\n        \"music\": {\"cues\": [{\"from\": \"sh01\", \"track\": \"Investigations\"}, {\"from\": \"sh02\", \"track\": \"Easy Lemon\"},\n                           {\"from\": shots[-1][\"shot_id\"], \"track\": \"Carefree\"}]},\n        \"shots\": shots,\n        \"sources\": sheet.sources,\n        \"metadata\": {\n            \"title\": script[\"title\"],\n            \"summary\": \" \".join(s[\"text\"] for b in script[\"beats\"] for s in b[\"narration\"])[:600],\n            \"disclaimer\": (\"Automatically researched from Wikipedia and fact-checked against the cited sentences by a \"\n                           \"local model; simplified for beginners. Check the sources listed below.\"),\n            \"category\": \"Education\", \"audience\": \"Not made for kids\",\n            \"tags\": sorted(set(re.findall(r\"[a-z]{4,}\", topic.lower())))[:10],\n            \"sources\": [s[\"url\"] for s in sheet.sources],\n            \"image_credits\": [a[\"credit_line\"] + f\" ({a['page']})\" for a in images]}}\n\n\ndef script_text(script):\n    \"\"\"Readable script (for the person reviewing it).\"\"\"\n    out = [script[\"title\"].upper(), \"\", \"HOOK: \" + script[\"hook\"][\"text\"], \"\"]\n    for i, b in enumerate(script[\"beats\"], 1):\n        v = b[\"visual\"]\n        out.append(f\"[{i}] \" + \" \".join(s[\"text\"] for s in b[\"narration\"]))\n        out.append(\"     visual: \" + (v.get(\"description\") or \" / \".join(v.get(\"card_lines\", []))))\n    if script.get(\"takeaway\"):\n        out += [\"\", \"TAKEAWAY: \" + script[\"takeaway\"][\"text\"]]\n    return \"\\n\".join(textwrap.fill(x, 110, subsequent_indent=\"     \") if x else x for x in out)\n", "qclib.py": "\"\"\"Final quality control: measurements of the rendered video (FFmpeg / ffprobe) and the approval gate.\n\nEvery check is a dict {\"status\": PASS|FAIL|UNKNOWN, \"mandatory\": bool, \"evidence\": ..., \"shots\": [...], \"fix\": str}.\n`approved(checks)` is True only when every MANDATORY check is PASS: an UNKNOWN (couldn't be measured) blocks\napproval exactly like a FAIL, so nothing unverified slips into an \"approved\" render.\nThresholds are engineering settings (THRESHOLDS), not universal truths.\n\"\"\"\nfrom __future__ import annotations\n\nimport json, re, subprocess\nfrom pathlib import Path\n\nTHRESHOLDS = {\"true_peak_max_dbtp\": -1.0,      # above this the mix may clip after AAC encoding\n              \"loudness_target_lufs\": -14.0,  # configurable target, not a platform requirement\n              \"loudness_tolerance_lu\": 1.5,\n              \"silence_db\": -45.0,            # quieter than this for...\n              \"silence_min_s\": 1.5,           # ...this long = unintended silence (the music bed should fill pauses)\n              \"black_min_s\": 0.5,\n              \"freeze_noise\": 0.003, \"freeze_min_s\": 0.8,\n              \"duration_drift_s\": 0.5,\n              \"duck_min_db\": 4.0}             # music must be at least this much lower while someone speaks\n\n\ndef check(status, mandatory=True, evidence=None, shots=None, fix=\"\"):\n    assert status in (\"PASS\", \"FAIL\", \"UNKNOWN\"), status\n    return {\"status\": status, \"mandatory\": mandatory, \"evidence\": evidence, \"shots\": shots or [], \"fix\": fix}\n\n\ndef approved(checks: dict) -> bool:\n    return all(c[\"status\"] == \"PASS\" for c in checks.values() if c[\"mandatory\"])\n\n\ndef _run(cmd):\n    return subprocess.run([str(c) for c in cmd], capture_output=True, text=True)\n\n\ndef probe(path) -> dict:\n    \"\"\"Streams of a media file: video codec/size/fps/pix_fmt, audio codec/rate/channels, durations.\"\"\"\n    r = _run([\"ffprobe\", \"-v\", \"error\", \"-show_entries\",\n              \"format=duration:stream=codec_type,codec_name,width,height,r_frame_rate,pix_fmt,sample_rate,channels,duration\",\n              \"-of\", \"json\", path])\n    if r.returncode:\n        raise RuntimeError(f\"ffprobe failed: {r.stderr.strip()[:300]}\")\n    d = json.loads(r.stdout)\n    out = {\"duration\": float(d[\"format\"][\"duration\"]), \"video\": None, \"audio\": None}\n    for s in d[\"streams\"]:\n        if s[\"codec_type\"] == \"video\" and out[\"video\"] is None:\n            num, den = (int(x) for x in s[\"r_frame_rate\"].split(\"/\"))\n            out[\"video\"] = {\"codec\": s[\"codec_name\"], \"width\": s[\"width\"], \"height\": s[\"height\"],\n                            \"fps\": round(num / max(den, 1), 3), \"pix_fmt\": s.get(\"pix_fmt\"),\n                            \"duration\": float(s.get(\"duration\") or out[\"duration\"])}\n        elif s[\"codec_type\"] == \"audio\" and out[\"audio\"] is None:\n            out[\"audio\"] = {\"codec\": s[\"codec_name\"], \"sample_rate\": int(s[\"sample_rate\"]), \"channels\": s[\"channels\"],\n                            \"duration\": float(s.get(\"duration\") or out[\"duration\"])}\n    return out\n\n\ndef video_events(path, th=THRESHOLDS) -> dict:\n    \"\"\"Black and frozen stretches, in ONE decoding pass.\"\"\"\n    r = _run([\"ffmpeg\", \"-hide_banner\", \"-i\", path, \"-vf\",\n              f\"blackdetect=d={th['black_min_s']}:pix_th=0.10,freezedetect=n={th['freeze_noise']}:d={th['freeze_min_s']}\",\n              \"-an\", \"-f\", \"null\", \"-\"])\n    black = [(float(a), float(b)) for a, b in re.findall(r\"black_start:(\\S+) black_end:(\\S+)\", r.stderr)]\n    fs = [float(x) for x in re.findall(r\"freeze_start: (\\S+)\", r.stderr)]\n    fe = [float(x) for x in re.findall(r\"freeze_end: (\\S+)\", r.stderr)]\n    dur = probe(path)[\"duration\"]\n    freezes = [(a, fe[i] if i < len(fe) else dur) for i, a in enumerate(fs)]   # a freeze to the end has no end line\n    return {\"black\": black, \"freezes\": freezes}\n\n\ndef audio_stats(path, th=THRESHOLDS) -> dict:\n    \"\"\"Integrated loudness, true peak and silent stretches, in ONE decoding pass.\"\"\"\n    r = _run([\"ffmpeg\", \"-hide_banner\", \"-i\", path, \"-vn\", \"-af\",\n              f\"ebur128=peak=true,silencedetect=n={th['silence_db']}dB:d={th['silence_min_s']}\", \"-f\", \"null\", \"-\"])\n    lufs = re.findall(r\"I:\\s+(-?[\\d.]+) LUFS\", r.stderr)\n    peaks = re.findall(r\"Peak:\\s+(-?[\\d.]+|-inf) dBFS\", r.stderr)\n    ss = [float(x) for x in re.findall(r\"silence_start: (\\S+)\", r.stderr)]\n    se = [float(x) for x in re.findall(r\"silence_end: (\\S+)\", r.stderr)]\n    dur = probe(path)[\"duration\"]\n    return {\"integrated_lufs\": float(lufs[-1]) if lufs else None,\n            \"true_peak_dbtp\": (max(float(p) for p in peaks if p != \"-inf\") if any(p != \"-inf\" for p in peaks) else None),\n            \"silences\": [(a, se[i] if i < len(se) else dur) for i, a in enumerate(ss)]}\n\n\ndef _levels(path, sr=8000, hop=0.1):\n    \"\"\"RMS level in dB every `hop` seconds (mono, via FFmpeg \u2192 raw float samples).\"\"\"\n    import numpy as np\n    r = subprocess.run([\"ffmpeg\", \"-hide_banner\", \"-loglevel\", \"error\", \"-i\", str(path), \"-ac\", \"1\", \"-ar\", str(sr),\n                        \"-f\", \"f32le\", \"-\"], capture_output=True)\n    x = np.frombuffer(r.stdout, np.float32)\n    n = int(sr * hop)\n    x = x[: len(x) // n * n].reshape(-1, n)\n    return 20 * np.log10(np.sqrt((x ** 2).mean(axis=1)) + 1e-9)\n\n\ndef ducking(voice_wav, ducked_music_wav, voice_on_db=-35.0) -> dict:\n    \"\"\"How much quieter the (already ducked) music bed is while the voice speaks than in the pauses.\n    Measured, not assumed: a broken sidechain shows ~0 dB.\"\"\"\n    import numpy as np\n    v, m = _levels(voice_wav), _levels(ducked_music_wav)\n    n = min(len(v), len(m))\n    v, m = v[:n], m[:n]\n    speak, pause = v > voice_on_db, v < voice_on_db - 15\n    if speak.sum() < 5 or pause.sum() < 5:\n        return {\"reduction_db\": None, \"why\": \"not enough speech or pauses to compare\"}\n    return {\"reduction_db\": round(float(np.median(m[pause]) - np.median(m[speak])), 1),\n            \"speech_s\": round(float(speak.sum()) * 0.1, 1), \"pause_s\": round(float(pause.sum()) * 0.1, 1)}\n\n\ndef contact_sheet(video, out, n=12, cols=4, width=1600):\n    \"\"\"n evenly spaced frames in a grid, for a person to look over before publishing.\"\"\"\n    dur = probe(video)[\"duration\"]\n    step = max(dur / n, 0.1)\n    rows = -(-n // cols)\n    r = _run([\"ffmpeg\", \"-hide_banner\", \"-loglevel\", \"error\", \"-y\", \"-i\", video, \"-vf\",\n              f\"fps=1/{step:.3f},scale={width // cols}:-2,tile={cols}x{rows}\", \"-frames:v\", \"1\", \"-q:v\", \"3\", out])\n    if r.returncode:\n        raise RuntimeError(r.stderr.strip()[:300])\n    return Path(out)\n\n\ndef overlaps(spans, a, b, min_s=0.0):\n    \"\"\"Total seconds of `spans` [(start, end)] inside [a, b]; True if more than min_s.\"\"\"\n    return sum(max(0.0, min(e, b) - max(s, a)) for s, e in spans) > min_s\n", "qwen_tts_worker.py": "\"\"\"Expressive English narration: Qwen3-TTS CustomVoice, run in its OWN virtual environment.\n\nqwen-tts pins transformers==4.57.3; the notebook needs 5.x. So, like parler_worker.py, this file is started once by\nthe narration cell with the TTS environment's python, keeps the model loaded, and answers one JSON request per line:\n\n  request:  {\"text\": \"...\", \"speaker\": \"Ryan\", \"language\": \"English\", \"instruct\": \"Speak with quiet curiosity...\",\n             \"seed\": 123, \"out\": \"/path/x.wav\", \"gen\": {\"temperature\": 0.9}}\n  response: {\"ok\": true, \"out\": \"...\", \"seconds\": 3.2, \"gen_s\": 4.1}  |  {\"ok\": false, \"error\": \"...\"}\n\nThe first line is {\"ready\": true, \"sr\": ..., \"device\": ..., \"dtype\": ..., \"speakers\": [...], \"instruct\": bool}.\nPrecision is a parameter: a T4 has no native bf16 and Qwen3-TTS has reported fp16 instability on older GPUs, so the\nnotebook benchmarks fp32/fp16 instead of assuming. Attention: PyTorch SDPA (FlashAttention-2 is never required).\n\nRun: python qwen_tts_worker.py Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice cuda float32\n\"\"\"\nimport json, sys, time\n\nMODEL = \"Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\"   # Apache-2.0; instruction control (the 0.6B has none)\nINSTRUCT_MODELS = {\"Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\"}\n# Everything qwen-tts 0.1.1 needs on top of Colab, resolved for Python 3.13 / x86_64 Linux\n# (uv pip compile --python-version 3.13 --python-platform x86_64-manylinux_2_28), then torch, torchaudio, triton,\n# numpy, nvidia-* and cuda-* REMOVED (Colab's matching copies are used). Installed with --no-deps.\nREQUIREMENTS = \"\"\"\naccelerate==1.12.0\nannotated-doc==0.0.5\nannotated-types==0.8.0\nanyio==4.15.1\naudioop-lts==0.2.2\nbrotli==1.2.0\ncertifi==2026.7.22\ncffi==2.1.1\ncharset-normalizer==3.5.2\nclick==8.5.0\ncloudpickle==3.1.2\ndecorator==5.3.1\neinops==0.8.2\nfastapi==0.143.3\nfilelock==4.2.1\nflatbuffers==25.12.19\nfsspec==2026.9.0\ngradio==6.17.3\ngradio-client==2.5.0\ngroovy==0.1.2\nh11==0.16.0\nhf-gradio==0.4.1\nhf-xet==1.7.0\nhttpcore==1.0.9\nhttpx==0.28.1\nhuggingface-hub==0.36.2\nidna==3.20\njinja2==3.1.6\njoblib==1.6.0\nlazy-loader==0.6\nlibrosa==1.0.0\nllvmlite==0.50.0\nmarkdown-it-py==4.2.0\nmarkupsafe==3.0.4\nmdurl==0.1.2\nmpmath==1.3.0\nmsgpack==1.2.3\nnarwhals==2.27.1\nnetworkx==3.7\nnumba==0.68.0\nonnxruntime==1.31.0\nopentelemetry-api==1.45.1\norjson==3.13.0\npackaging==26.3\npandas==3.0.6\npillow==12.3.0\nplatformdirs==4.13.4\npooch==1.9.0\nprotobuf==7.36.2\npsutil==7.2.2\npycparser==3.11\npydantic==2.14.1\npydantic-core==2.50.1\npydub==0.25.1\npygments==2.21.0\npython-dateutil==2.9.0.post0\npython-multipart==0.0.32\npytz==2026.5\npyyaml==6.0.3\nqwen-tts==0.1.1\nregex==2026.9.29\nrequests==2.34.2\nrich==15.0.0\nsafehttpx==0.1.7\nsafetensors==0.8.0\nscikit-learn==1.9.1\nscipy==1.18.1\nsemantic-version==2.10.0\nsetuptools==84.0.0\nshellingham==1.5.4\nsix==1.17.0\nsoundfile==0.14.0\nsox==1.5.0\nsoxr==1.1.0\nstarlette==1.8.0\nsympy==1.14.0\nthreadpoolctl==3.7.0\ntokenizers==0.22.2\ntomlkit==0.14.0\ntqdm==4.70.1\ntransformers==4.57.3\ntyper==0.27.3\ntyping-extensions==4.16.0\ntyping-inspection==0.4.4\nurllib3==2.8.0\nuvicorn==0.54.0\n\"\"\"\n\n\ndef main():\n    mid, device, dtype = sys.argv[1], sys.argv[2], sys.argv[3]\n    t0 = time.time()\n    proto, sys.stdout = sys.stdout, sys.stderr        # library prints must never mix into the JSON channel\n    def say(obj):\n        proto.write(json.dumps(obj) + \"\\n\"); proto.flush()\n    import numpy as np, soundfile as sf, torch\n    from qwen_tts import Qwen3TTSModel\n    dt = {\"float32\": torch.float32, \"float16\": torch.float16, \"bfloat16\": torch.bfloat16}[dtype]\n    model = Qwen3TTSModel.from_pretrained(mid, device_map=(\"cuda:0\" if device == \"cuda\" else \"cpu\"), dtype=dt,\n                                          attn_implementation=\"sdpa\")\n    try:\n        speakers = list(model.get_supported_speakers())\n    except Exception:\n        speakers = []\n    say({\"ready\": True, \"device\": device, \"dtype\": dtype, \"load_s\": round(time.time() - t0, 1), \"speakers\": speakers,\n         \"instruct\": mid in INSTRUCT_MODELS, \"transformers\": __import__(\"transformers\").__version__,\n         \"vram_gb\": round(torch.cuda.memory_allocated() / 2**30, 2) if device == \"cuda\" else None})\n    for raw in sys.stdin:\n        if not raw.strip():\n            continue\n        try:\n            req = json.loads(raw)\n            t1 = time.time()\n            torch.manual_seed(int(req.get(\"seed\", 0)))\n            if device == \"cuda\":\n                torch.cuda.manual_seed_all(int(req.get(\"seed\", 0))); torch.cuda.reset_peak_memory_stats()\n            kw = dict(text=req[\"text\"], language=req.get(\"language\", \"English\"), speaker=req[\"speaker\"])\n            if req.get(\"instruct\"):\n                kw[\"instruct\"] = req[\"instruct\"]\n            with torch.inference_mode():\n                wavs, sr = model.generate_custom_voice(**kw, **(req.get(\"gen\") or {}))\n            audio = np.asarray(wavs[0], dtype=np.float32).squeeze()\n            if audio.size == 0 or not np.isfinite(audio).all():\n                raise RuntimeError(\"the voice model produced empty or broken audio (try float32 on this GPU)\")\n            sf.write(req[\"out\"], audio, sr)\n            say({\"ok\": True, \"out\": req[\"out\"], \"seconds\": round(audio.size / sr, 2), \"sr\": sr,\n                 \"gen_s\": round(time.time() - t1, 1),\n                 \"peak_vram_gb\": round(torch.cuda.max_memory_allocated() / 2**30, 2) if device == \"cuda\" else None})\n        except Exception as e:\n            import traceback\n            traceback.print_exc(file=sys.stderr)\n            say({\"ok\": False, \"error\": f\"{type(e).__name__}: {str(e)[:300]}\"})\n\n\nif __name__ == \"__main__\":\n    main()\n", "registry.py": "\"\"\"Model and asset licence registry: every model the engine can load, with its VERIFIED weight licence, and the\nlicence policy for third-party images.\n\nVerified against the Hugging Face API / publisher pages on 2026-10-18 (tests/test_registry.py --online re-checks).\nstatus: \"ok\" = permissive licence (Apache-2.0 / MIT), \"review\" = usable but with a condition a person should know\nabout (see note), \"blocked\" = must not be used. LICENSE_STRICT in the notebook refuses \"review\" models.\n\"\"\"\nfrom __future__ import annotations\n\nimport re\n\nMODELS = {\n    # id: (use, licence of the weights, status, note)\n    \"Wan-AI/Wan2.1-T2V-1.3B-Diffusers\": (\"text-to-video (T4)\", \"apache-2.0\", \"ok\", \"\"),\n    \"Wan-AI/Wan2.1-VACE-1.3B-diffusers\": (\"layout-controlled video (scene-state shots)\", \"apache-2.0\", \"ok\", \"\"),\n    \"Wan-AI/Wan2.2-TI2V-5B-Diffusers\": (\"video on big GPUs; its UMT5 text encoder is also used on the T4\", \"apache-2.0\", \"ok\", \"\"),\n    \"Kijai/WanVideo_comfy\": (\"turbo LoRA (Self-Forcing DMD, 4 steps)\", \"apache-2.0 (upstream gdhe17/Self-Forcing)\", \"review\",\n                             \"LoRA extracted from the Apache-2.0 Self-Forcing weights; the repackaging repo has no licence \"\n                             \"tag of its own\"),\n    \"google/owlv2-base-patch16-ensemble\": (\"open-vocabulary detector (scene checks)\", \"apache-2.0\", \"ok\", \"\"),\n    \"Qwen/Qwen3-VL-2B-Instruct\": (\"video-language judge (scene checks)\", \"apache-2.0\", \"ok\", \"\"),\n    \"openai/clip-vit-base-patch32\": (\"legacy continuity check (plain text-to-video shots only)\", \"mit (OpenAI CLIP repo)\", \"review\",\n                                     \"no licence tag on the Hub; OpenAI's model card lists deployed use as out of scope\"),\n    \"google/siglip2-base-patch16-224\": (\"image-text relevance ranking (licensed images)\", \"apache-2.0\", \"ok\", \"\"),\n    \"stabilityai/stable-diffusion-xl-base-1.0\": (\"optional keyframe images\", \"openrail++\", \"review\",\n                                                 \"OpenRAIL++: commercial use allowed with use-based restrictions\"),\n    \"Tongyi-MAI/Z-Image-Turbo\": (\"optional keyframe images (big GPUs)\", \"apache-2.0\", \"ok\", \"\"),\n    \"hexgrad/Kokoro-82M\": (\"English voices (kokoro-onnx export of these weights)\", \"apache-2.0\", \"ok\", \"\"),\n    \"ai4bharat/indic-parler-tts\": (\"Telugu voices\", \"apache-2.0\", \"ok\", \"gated: accept the terms, needs a token\"),\n    # Character / Environment Studio (one-time asset production, never loaded during a routine video render)\n    \"black-forest-labs/FLUX.2-klein-4B\": (\"character concepts, reference-conditioned expressions/poses (Studio)\",\n                                          \"apache-2.0\", \"ok\", \"\"),\n    \"black-forest-labs/FLUX.2-klein-9B\": (\"(not used)\", \"flux-non-commercial-license\", \"blocked\",\n                                          \"non-commercial weights: refused (use the Apache-2.0 4B model)\"),\n    \"ZhengPeng7/BiRefNet\": (\"background removal / layer cut-out of approved artwork (Studio)\", \"mit\", \"ok\", \"\"),\n    \"briaai/RMBG-2.0\": (\"(not used)\", \"cc-by-nc-4.0 (bria-rmbg-2.0)\", \"blocked\", \"non-commercial weights: refused\"),\n    \"briaai/RMBG-1.4\": (\"(not used)\", \"bria-rmbg-1.4 (commercial use needs an agreement)\", \"blocked\",\n                        \"not commercially usable without a Bria agreement: refused\"),\n    \"Systran/faster-whisper-small.en\": (\"speech check / word timing (English)\", \"mit\", \"ok\", \"\"),\n    \"Systran/faster-whisper-tiny\": (\"speech check in tests\", \"mit\", \"ok\", \"\"),\n    \"Systran/faster-whisper-tiny.en\": (\"speech check in tests (English)\", \"mit\", \"ok\", \"\"),\n    \"mobiuslabsgmbh/faster-whisper-large-v3-turbo\": (\"speech check (Telugu)\", \"mit\", \"ok\",\n                                                     \"the Hub redirects it to dropbox-dash/faster-whisper-large-v3-turbo\"),\n    \"Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\": (\"expressive English narration (instruction-controlled delivery)\",\n                                             \"apache-2.0\", \"ok\", \"runs in its own environment (qwen-tts pins transformers 4.57.3)\"),\n    \"Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice\": (\"narration without instruction control (tests only)\", \"apache-2.0\", \"ok\", \"\"),\n    \"speechbrain/spkrec-ecapa-voxceleb\": (\"narrator identity check (speaker embeddings)\", \"apache-2.0\", \"ok\", \"\"),\n    \"Qwen/Qwen3-4B-Instruct-2507\": (\"topic \u2192 script planner (default)\", \"apache-2.0\", \"ok\", \"\"),\n    \"Qwen/Qwen3-1.7B\": (\"topic \u2192 script planner (low memory)\", \"apache-2.0\", \"ok\", \"\"),\n    \"Qwen/Qwen3-0.6B\": (\"planner in CPU tests\", \"apache-2.0\", \"ok\", \"\"),\n    \"cross-encoder/nli-deberta-v3-base\": (\"fact check: is a sentence entailed by its source? (default)\", \"apache-2.0\", \"ok\", \"\"),\n    \"cross-encoder/nli-deberta-v3-small\": (\"fact check, faster/weaker alternative\", \"apache-2.0\", \"ok\", \"\"),\n    \"ResembleAI/chatterbox-turbo\": (\"evaluated TTS candidate, NOT integrated\", \"mit\", \"review\",\n                                    \"weights MIT; chatterbox-tts pins torch==2.6.0 + transformers==5.2.0 (own env, ~2.5 GB); \"\n                                    \"distinct voices need reference audio (only synthetic/authorised); outputs carry a \"\n                                    \"Perth watermark; not benchmarked here\"),\n    \"ResembleAI/chatterbox-nano\": (\"evaluated TTS candidate, NOT integrated\", \"mit\", \"review\",\n                                   \"same package and conditions as chatterbox-turbo; not benchmarked here\"),\n    \"gdhe17/Self-Forcing\": (\"upstream of the turbo LoRA (checkpoints/self_forcing_dmd.pt); not loaded yet\",\n                            \"apache-2.0\", \"ok\", \"a licence-clean turbo needs these weights converted to a diffusers \"\n                            \"LoRA locally (not done; the Kijai repackaging stays 'review')\"),\n}\n# Hub commit of every model at the time its licence was verified (2026-10-18). Recorded so a later change of the\n# weights or of the licence on the Hub is visible (tests/test_registry.py --online reports a moved revision).\n# Not yet passed as revision= at every load site.\nREVISIONS = {\n    \"Systran/faster-whisper-tiny.en\": \"0d3d19a32d3338f10357c0889762bd8d64bbdeba\",\n    \"ResembleAI/chatterbox-turbo\": \"749d1c1a46eb10492095d68fbcf55691ccf137cd\",\n    \"ResembleAI/chatterbox-nano\": \"71ccd1d0081b430592cea481f4307e764e07bc64\",\n    \"Wan-AI/Wan2.1-T2V-1.3B-Diffusers\": \"0fad780a534b6463e45facd96134c9f345acfa5b\",\n    \"Wan-AI/Wan2.1-VACE-1.3B-diffusers\": \"ec4d2cb062b548996b179d493fdd05340de702a1\",\n    \"Wan-AI/Wan2.2-TI2V-5B-Diffusers\": \"b8fff7315c768468a5333511427288870b2e9635\",\n    \"Kijai/WanVideo_comfy\": \"8260d429d19fd7a72304cad059160b95d843913f\",\n    \"google/owlv2-base-patch16-ensemble\": \"cfd3195ba4ea9592eec887ded089f4c08eff231d\",\n    \"Qwen/Qwen3-VL-2B-Instruct\": \"89644892e4d85e24eaac8bacfd4f463576704203\",\n    \"openai/clip-vit-base-patch32\": \"3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268\",\n    \"google/siglip2-base-patch16-224\": \"75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2\",\n    \"stabilityai/stable-diffusion-xl-base-1.0\": \"462165984030d82259a11f4367a4eed129e94a7b\",\n    \"Tongyi-MAI/Z-Image-Turbo\": \"f332072aa78be7aecdf3ee76d5c247082da564a6\",\n    \"hexgrad/Kokoro-82M\": \"f3ff3571791e39611d31c381e3a41a3af07b4987\",\n    \"ai4bharat/indic-parler-tts\": \"660bc53c64ae83150613264111708a0465211237\",\n    \"black-forest-labs/FLUX.2-klein-4B\": \"e7b7dc27f91deacad38e78976d1f2b499d76a294\",\n    \"black-forest-labs/FLUX.2-klein-9B\": \"92196c8e11f7b6cf2b7493e037d8c5345c559216\",\n    \"ZhengPeng7/BiRefNet\": \"e2bf8e4460fc8fa32bba5ea4d94b3233d367b0e4\",\n    \"briaai/RMBG-2.0\": \"5df4c9c76d8170882c34f6986e848ee07fd0ba43\",\n    \"briaai/RMBG-1.4\": \"2ceba5a5efaec153162aedea169f76caf9b46cf8\",\n    \"Systran/faster-whisper-small.en\": \"d1d751a5f8271d482d14ca55d9e2deeebbae577f\",\n    \"Systran/faster-whisper-tiny\": \"d90ca5fe260221311c53c58e660288d3deb8d356\",\n    \"mobiuslabsgmbh/faster-whisper-large-v3-turbo\": \"0a363e9161cbc7ed1431c9597a8ceaf0c4f78fcf\",\n    \"Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\": \"0c0e3051f131929182e2c023b9537f8b1c68adfe\",\n    \"Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice\": \"85e237c12c027371202489a0ec509ded67b5e4b5\",\n    \"speechbrain/spkrec-ecapa-voxceleb\": \"0f99f2d0ebe89ac095bcc5903c4dd8f72b367286\",\n    \"Qwen/Qwen3-4B-Instruct-2507\": \"cdbee75f17c01a7cc42f958dc650907174af0554\",\n    \"Qwen/Qwen3-1.7B\": \"70d244cc86ccca08cf5af4e1e306ecf908b1ad5e\",\n    \"Qwen/Qwen3-0.6B\": \"c1899de289a04d12100db370d81485cdf75e47ca\",\n    \"cross-encoder/nli-deberta-v3-base\": \"6c749ce3425cd33b46d187e45b92bbf96ee12ec7\",\n    \"cross-encoder/nli-deberta-v3-small\": \"fa2804872c3b4bd748f38c0185cc85775361e735\",\n    \"gdhe17/Self-Forcing\": \"2f8b779212da279d212c22a509b66ad6552f350e\",\n}\n# Non-model assets and their licences (media policy: PD / CC0 / CC BY; BY-SA opt-in; NC/ND/unknown rejected)\nMEDIA_SOURCES = {\n    \"characters\": (\"Arjun and Meera: original artwork drawn by code (engine/lib/charanim.py) from the Character Bible\",\n                   \"original work of this project\"),\n    \"demonstrations\": (\"finger test, eye views, projector + glasses: drawn by code (engine/lib/demos.py)\",\n                       \"original work of this project\"),\n    \"fonts\": (\"DejaVu Sans (Bitstream Vera licence) / Hind Guntur (SIL OFL 1.1) for Telugu\", \"permissive font licences\"),\n    \"music\": (\"Kevin MacLeod, incompetech.com tracks (engine/lib/musiclib.py), credited in the description\", \"CC BY 4.0\"),\n    \"sound effects\": (\"synthesised in the notebook (sfx cell)\", \"original work of this project\"),\n    \"photos\": (\"Wikimedia Commons, per image: licence checked by asset_license(), credit drawn on screen + description\",\n               \"PD / CC0 / CC BY (BY-SA only with ALLOW_SHAREALIKE)\"),\n}\nALLOWED_MODEL_LICENSES = (\"apache-2.0\", \"mit\", \"bsd\")\n\n# The ONE approved hosted exception (2026-10-19): Google Gemini free-tier TTS, for SPEECH SYNTHESIS ONLY. It is not\n# open-source and not an open-weight model; it is listed apart from MODELS so it can never pass as one.\nHOSTED_EXCEPTIONS = {\n    \"gemini-3.8-flash-tts\": {\"provider\": \"Google AI Studio / Gemini API\", \"scope\": \"tts\", \"tier\": \"free only\",\n                             \"kind\": \"proprietary hosted free-tier TTS exception\",\n                             \"data_use\": \"free-tier input/output used to improve Google products; human review possible\",\n                             \"docs\": \"https://ai.google.dev/gemini-api/docs/speech-generation\",\n                             \"verified\": \"2026-10-19: Telugu + 2 speakers; Standard tier free of charge\"},\n    \"gemini-3.8-flash-lite-tts\": {\"provider\": \"Google AI Studio / Gemini API\", \"scope\": \"tts\", \"tier\": \"free only\",\n                                  \"kind\": \"proprietary hosted free-tier TTS exception\",\n                                  \"data_use\": \"free-tier input/output used to improve Google products; human review possible\",\n                                  \"docs\": \"https://ai.google.dev/gemini-api/docs/speech-generation\",\n                                  \"verified\": \"2026-10-19: Telugu + 2 speakers; Standard tier free of charge\"},\n}\n\n\ndef hosted_ok(model: str, use: str) -> tuple[bool, str]:\n    \"\"\"A hosted model is allowed only if it is a listed exception AND used for its approved scope (speech only).\"\"\"\n    h = HOSTED_EXCEPTIONS.get(model)\n    if h is None:\n        return False, f\"{model}: not an approved hosted exception (no hosted API may be used for {use})\"\n    if use != h[\"scope\"]:\n        return False, f\"{model}: approved for {h['scope']} only, not for {use}\"\n    return True, f\"{model}: {h['kind']} ({h['tier']})\"\n\n\n# services that must never be required (paid or closed); the engine must not reference them\nFORBIDDEN_SERVICES = [\"api.openai.com\", \"api.anthropic.com\", \"api.elevenlabs.io\", \"api.runwayml.com\",\n                      \"api.replicate.com\", \"modal.com\", \"runpod.io\", \"api.pexels.com\", \"pixabay.com/api\"]\nFORBIDDEN_IMPORTS = [\"openai\", \"anthropic\", \"elevenlabs\", \"runwayml\", \"replicate\", \"modal\", \"runpod\"]\n\n\ndef model_ok(mid: str, strict: bool = False) -> tuple[bool, str]:\n    m = MODELS.get(mid)\n    if m is None:\n        return False, f\"{mid}: not in the licence registry (add it with its verified weight licence)\"\n    use, lic, status, note = m\n    if status == \"blocked\" or (strict and status == \"review\"):\n        return False, f\"{mid}: {lic} \u2014 {note or status}\"\n    return True, f\"{mid}: {lic}\" + (f\" (review: {note})\" if status == \"review\" else \"\")\n\n\n# ------------------------------------------------------------------ images (Wikimedia Commons extmetadata)\ndef asset_license(short_name: str, license_url: str = \"\", allow_sharealike: bool = False) -> tuple[bool, str]:\n    \"\"\"Decide from Commons' LicenseShortName (free text, so normalised conservatively). Unknown = rejected.\n    Allowed by default: public domain, CC0, CC BY (any version). CC BY-SA only when allow_sharealike (an image the\n    video builds on can make the whole video a ShareAlike adaptation). NC / ND / GFDL-only / fair use: never.\"\"\"\n    s = re.sub(r\"[\\s_]+\", \" \", (short_name or \"\").strip().lower())\n    u = (license_url or \"\").lower()\n    if not s:\n        return False, \"no licence stated\"\n    if any(x in s for x in (\"fair use\", \"non-free\", \"copyrighted\", \"all rights reserved\")):\n        return False, f\"not free: {short_name}\"\n    if re.search(r\"\\bnc\\b|noncommercial|non-commercial\", s) or \"-nc\" in u:\n        return False, f\"non-commercial: {short_name}\"\n    if re.search(r\"\\bnd\\b|noderiv\", s) or \"-nd\" in u:\n        return False, f\"no derivatives: {short_name}\"\n    if s in (\"public domain\", \"pd\", \"cc0\", \"cc0 1.0\", \"cc-zero\", \"no restrictions\") or s.startswith((\"pd-\", \"public domain\")) \\\n            or \"publicdomain\" in u:\n        return True, \"public domain / CC0\"\n    if re.fullmatch(r\"cc[ -]by[ -]?(\\d\\.\\d)?( [a-z]+)?\", s):\n        return True, f\"attribution: {short_name}\"\n    if re.fullmatch(r\"cc[ -]by[ -]sa[ -]?(\\d\\.\\d)?( [a-z]+)?\", s):\n        return (allow_sharealike, f\"share-alike: {short_name}\" + (\"\" if allow_sharealike else \" (ALLOW_SHAREALIKE is off)\"))\n    if s.startswith(\"gfdl\"):\n        return False, f\"GFDL (impractical in video): {short_name}\"\n    return False, f\"unknown licence (rejected): {short_name}\"\n\n\nACCESS_HELP = (\"Indic Parler-TTS is gated (free, Apache-2.0): 1) log in to huggingface.co with the account of your \"\n               \"HF_TOKEN, 2) open https://huggingface.co/ai4bharat/indic-parler-tts and accept the terms \"\n               \"('Agree and access repository'), 3) make sure the token is a READ token in Colab Secrets as HF_TOKEN \"\n               \"with notebook access ON, 4) run the cell again. The pipeline never bypasses the gate.\")\n\n\ndef gated_access(repo: str, token: str | None, http=None, revision: str | None = None) -> dict:\n    \"\"\"Can this token download this gated model? (a HEAD request for config.json; nothing is downloaded or\n    logged \u2014 the token is only sent to huggingface.co). \u2192 {\"status\": PASS | TOKEN_MISSING | TOKEN_INVALID |\n    ACCESS_REQUIRED | UNKNOWN, \"why\", \"help\"}. UNKNOWN (offline / unexpected answer) is never treated as PASS.\"\"\"\n    if not token:\n        return {\"status\": \"TOKEN_MISSING\", \"why\": \"no HF_TOKEN secret\", \"help\": ACCESS_HELP}\n    if http is None:\n        import requests as http\n    rev = revision or REVISIONS.get(repo) or \"main\"\n    url = f\"https://huggingface.co/{repo}/resolve/{rev}/config.json\"\n    try:\n        r = http.head(url, headers={\"Authorization\": f\"Bearer {token}\"}, allow_redirects=False, timeout=20)\n        code = r.status_code\n    except Exception as e:                                  # offline: say so, don't guess\n        return {\"status\": \"UNKNOWN\", \"why\": f\"could not reach huggingface.co ({type(e).__name__})\", \"help\": ACCESS_HELP}\n    if code in (200, 302, 307):\n        return {\"status\": \"PASS\", \"why\": f\"HTTP {code}\", \"help\": \"\"}\n    if code == 401:\n        return {\"status\": \"TOKEN_INVALID\", \"why\": \"HTTP 401: the token is not valid (new READ token?)\", \"help\": ACCESS_HELP}\n    if code == 403:\n        return {\"status\": \"ACCESS_REQUIRED\", \"why\": \"HTTP 403: this account has not accepted the model's terms\",\n                \"help\": ACCESS_HELP}\n    return {\"status\": \"UNKNOWN\", \"why\": f\"HTTP {code}\", \"help\": ACCESS_HELP}\n\n\n# Tools (not model weights) the storybook system evaluated or can use, with their licences.\nTOOLS = {\n    \"godot\": (\"2D cutout rig renderer (evaluated, not used: needs a display + OpenGL in Colab)\", \"mit\"),\n    \"rhubarb\": (\"optional mouth-shape timing (English recogniser; phonetic mode for other languages)\", \"mit\"),\n    \"pillow\": (\"storybook rig compositing (used)\", \"mit-cmu (HPND)\"),\n    \"ffmpeg\": (\"video assembly (used)\", \"lgpl-2.1+/gpl (binary build)\"),\n}\n", "research.py": "\"\"\"Topic research from Wikipedia (free, public; text CC BY-SA 4.0). Facts are paraphrased in the script and every\none keeps its source URL + revision id, so each narrated claim can be traced and re-checked.\n\nWikimedia API etiquette (https://www.mediawiki.org/wiki/Wikimedia_APIs/Access_policy): a User-Agent with contact\ndetails, maxlag=5 with a pause when the servers are lagged, one request at a time, answers cached on disk.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib, json, math, re, time, urllib.parse, urllib.request\nfrom collections import Counter\nfrom pathlib import Path\n\nfrom pydantic import BaseModel, Field\n\nAPI = \"https://{lang}.wikipedia.org/w/api.php\"\nSTOP = set(\"\"\"a an the and or but if of to in on at by for with from as is are was were be been being it its this that\nthese those which who whom what when where why how than then there here also into over under such can could may might\nwill would not no do does did their they them his her he she we you your our has have had more most other some any\nabout after before during between both each few many much one two three first second\"\"\".split())\n\n\nclass Fact(BaseModel):\n    id: str\n    text: str\n    source_url: str\n    title: str\n    revid: int | None = None\n\n\nclass FactSheet(BaseModel):\n    topic: str\n    facts: list[Fact] = Field(default_factory=list)\n    sources: list[dict] = Field(default_factory=list)      # {title, url, revid, license}\n\n    def by_id(self):\n        return {f.id: f for f in self.facts}\n\n\nclass Wiki:\n    def __init__(self, contact: str, cache_dir, lang: str = \"en\", log=print, min_interval=1.0, api: str | None = None):\n        if not contact or \"@\" not in contact and \"http\" not in contact:\n            raise ValueError(\"Wikimedia asks for contact details in the User-Agent: set WIKI_CONTACT to your e-mail \"\n                             \"address or a web page (https://www.mediawiki.org/wiki/Wikimedia_APIs/Access_policy)\")\n        self.ua = f\"AIExplainerMaker/1.0 ({contact}) python-urllib\"\n        self.lang, self.cache, self.log, self.min_interval = lang, Path(cache_dir), log, min_interval\n        self.api = api or API.format(lang=lang)             # e.g. https://commons.wikimedia.org/w/api.php\n        self.cache.mkdir(parents=True, exist_ok=True)\n        self._last = 0.0\n\n    def get(self, **params):\n        params = {\"format\": \"json\", \"formatversion\": \"2\", \"maxlag\": \"5\", **params}\n        url = self.api + \"?\" + urllib.parse.urlencode(sorted(params.items()))\n        key = self.cache / (hashlib.sha1(url.encode()).hexdigest()[:16] + \".json\")\n        if key.exists():\n            return json.loads(key.read_text())\n        for attempt in range(5):\n            wait = self.min_interval - (time.time() - self._last)\n            if wait > 0:\n                time.sleep(wait)                                 # serial and paced: one request at a time\n            self._last = time.time()\n            req = urllib.request.Request(url, headers={\"User-Agent\": self.ua})\n            try:\n                with urllib.request.urlopen(req, timeout=30) as r:\n                    d = json.load(r)\n            except urllib.error.HTTPError as e:\n                if e.code in (429, 503):\n                    pause = int(e.headers.get(\"Retry-After\") or 5 * (attempt + 1))\n                    self.log(f\"wikipedia: HTTP {e.code}, waiting {pause}s\"); time.sleep(pause); continue\n                raise\n            if d.get(\"error\", {}).get(\"code\") == \"maxlag\":\n                self.log(\"wikipedia: servers lagged (maxlag), waiting 5s\"); time.sleep(5 * (attempt + 1)); continue\n            if \"error\" in d:\n                raise RuntimeError(f\"Wikipedia API error: {d['error']}\")\n            key.write_text(json.dumps(d))\n            return d\n        raise RuntimeError(\"Wikipedia API: still busy after 5 tries\")\n\n    def search(self, query, n=3):\n        d = self.get(action=\"query\", list=\"search\", srsearch=query, srlimit=str(n), srnamespace=\"0\")\n        return [h[\"title\"] for h in d[\"query\"][\"search\"]]\n\n    def article(self, title):\n        # HTML, not the plain-text extract: plain text drops superscripts, so \"2\u00d710<sup>30</sup> kg\" would become\n        # \"2\u00d71030 kg\" \u2014 a wrong number that a fact check would then happily \"confirm\"\n        d = self.get(action=\"query\", prop=\"extracts|info|revisions\", titles=title, inprop=\"url\", rvprop=\"ids\",\n                     redirects=\"1\")\n        pg = d[\"query\"][\"pages\"][0]\n        if pg.get(\"missing\"):\n            raise KeyError(title)\n        return {\"title\": pg[\"title\"], \"url\": pg[\"fullurl\"], \"revid\": pg[\"revisions\"][0][\"revid\"],\n                \"text\": html_to_text(pg.get(\"extract\", \"\"))}\n\n\ndef html_to_text(h):\n    \"\"\"Extract HTML \u2192 paragraphs of plain text, keeping exponents (10<sup>30</sup> \u2192 10^30) and dropping\n    citation marks, tables and headings.\"\"\"\n    import html as H\n    h = re.sub(r\"(?is)<(table|style|script|figure)\\b.*?</\\1>\", \" \", h)\n    h = re.sub(r\"(?is)<h\\d\\b.*?</h\\d>\", \"\\n\", h)\n    h = re.sub(r\"(?is)<sup[^>]*>\\s*\\[[^<]*\\]\\s*</sup>\", \"\", h)            # [1] citation markers\n    h = re.sub(r\"(?is)<sup[^>]*>(.*?)</sup>\", lambda m: \"^\" + re.sub(r\"<[^>]+>\", \"\", m.group(1)).strip(), h)\n    h = re.sub(r\"(?is)<sub[^>]*>(.*?)</sub>\", lambda m: re.sub(r\"<[^>]+>\", \"\", m.group(1)).strip(), h)\n    h = re.sub(r\"(?i)</p>|<br\\s*/?>|</li>\", \"\\n\", h)\n    t = H.unescape(re.sub(r\"<[^>]+>\", \"\", h))\n    t = t.replace(\"\\u2212\", \"-\").replace(\"\\xa0\", \" \")                     # Unicode minus, no-break space\n    return \"\\n\".join(re.sub(r\"[ \\t]+\", \" \", line).strip() for line in t.split(\"\\n\"))\n\n\ndef sentences(text):\n    \"\"\"Plain article text \u2192 sentences (headings, lists and very short or very long lines dropped).\"\"\"\n    out = []\n    for para in text.split(\"\\n\"):\n        para = para.strip()\n        if len(para) < 40 or not para.endswith((\".\", \"!\", \"?\")):\n            continue                                             # headings, captions, list fragments\n        para = re.sub(r\"\\s*\\((?:[^()\\d]*|[^()]*(?:pronounced|listen|IPA)[^()]*)\\)\", \"\", para)  # pronunciations, alt. names\n        for s in re.split(r\"(?<=[.!?])\\s+(?=[A-Z0-9])\", para):\n            s = s.strip()\n            if 40 <= len(s) <= 300 and not re.search(r\"\\b(?:See also|References|Further reading)\\b\", s):\n                out.append(s)\n    return out\n\n\ndef _words(s):\n    return [w for w in re.findall(r\"[a-z0-9]+\", s.lower()) if w not in STOP and len(w) > 2]\n\n\ndef relevance(sentence, topic_words, df, n_docs, position, rank=0):\n    \"\"\"Topic-word overlap (idf-weighted: rare topic words such as \"form\" count more than \"black\"), a small bonus\n    for early sentences (the lead summarises) and numbers, scaled down for lower-ranked search results.\"\"\"\n    ws = Counter(_words(sentence))\n    score = sum(math.log(1 + n_docs / (1 + df[w])) for w in ws if w in topic_words)\n    has_number = bool(re.search(r\"\\d\", sentence))\n    return (score + 0.2 * has_number + 0.5 / (1 + position / 10)) / (1 + 0.35 * rank)\n\n\ndef research(topic: str, wiki: Wiki, n_articles=3, max_facts=40, extra_titles=()):\n    \"\"\"Search Wikipedia for the topic, read the best articles, keep the most relevant sentences as numbered facts.\"\"\"\n    titles = list(dict.fromkeys(list(extra_titles) + wiki.search(topic, n_articles)))[:n_articles + len(extra_titles)]\n    arts = []\n    for t in titles:\n        try:\n            arts.append(wiki.article(t))\n        except KeyError:\n            wiki.log(f\"wikipedia: '{t}' not found\")\n    sheet = FactSheet(topic=topic, sources=[{\"title\": a[\"title\"], \"url\": a[\"url\"], \"revid\": a[\"revid\"],\n                                             \"license\": \"CC BY-SA 4.0 (text); facts paraphrased\"} for a in arts])\n    cands = []\n    for rank, a in enumerate(arts):\n        a[\"rank\"] = rank\n        for i, s in enumerate(sentences(a[\"text\"])):\n            cands.append((a, i, s))\n    if not cands:\n        return sheet\n    df = Counter(w for _, _, s in cands for w in set(_words(s)))\n    tw = set(_words(topic))                                     # the question's words (not every article title)\n    scored = sorted(cands, key=lambda c: -relevance(c[2], tw, df, len(cands), c[1], c[0][\"rank\"]))\n    seen = set()\n    for a, i, s in scored:\n        key = \" \".join(sorted(set(_words(s))))[:120]\n        if key in seen:\n            continue\n        seen.add(key)\n        sheet.facts.append(Fact(id=f\"F{len(sheet.facts) + 1}\", text=s, source_url=a[\"url\"], title=a[\"title\"],\n                                revid=a[\"revid\"]))\n        if len(sheet.facts) >= max_facts:\n            break\n    return sheet\n", "scene_state.py": "\"\"\"Scene-state and spatial-consistency engine.\n\nEvery AI shot can carry a typed \"state\" block: who is where on screen, which way they face, where the camera is,\nwhich side of the action line it stands on, who holds which object, and an ordered list of SMALL events\n(enter \u2192 walk_to \u2192 stop \u2192 place \u2026). From that one description the engine\n  1. simulates the world state shot by shot and rejects impossible or contradictory plans (this module),\n  2. draws the layout control video that steers generation (layout_control.py),\n  3. produces the expected tracks the generated clip is checked against (layout_control.expected_tracks \u2192\n     motion_check.py).\nThe plan is the single source of truth for generation AND validation.\n\nCoordinates: normalised screen space, x from 0 (left edge) to 1 (right edge), y from 0 (top) to 1 (bottom).\nEvent times: fractions of the shot (0..1). Pure Python + pydantic v2; unit-tested in tests/test_scene_state.py.\n\"\"\"\nfrom __future__ import annotations\n\nfrom typing import Literal, Optional\nfrom pydantic import BaseModel, Field, field_validator, model_validator\n\nBox = tuple[float, float, float, float]          # x0, y0, x1, y1 (normalised)\nFacing = Literal[\"left\", \"right\", \"camera\", \"away\"]\nLOCOMOTION = {\"enter\", \"walk_to\", \"exit\"}\nMANIPULATION = {\"take_out\", \"place\", \"pick_up\", \"hand_over\", \"scan\", \"open\"}\nVERSION = 1                                      # bump when the rules change (part of the validation cache key)\n\n\ndef cx(b: Box) -> float:\n    return (b[0] + b[2]) / 2\n\n\ndef cy(b: Box) -> float:\n    return (b[1] + b[3]) / 2\n\n\ndef near(a: Box, b: Box, gap: float = 0.08) -> bool:\n    \"\"\"Boxes touch or are within `gap` of each other (normalised).\"\"\"\n    return not (a[2] + gap < b[0] or b[2] + gap < a[0] or a[3] + gap < b[1] or b[3] + gap < a[1])\n\n\nclass Camera(BaseModel):\n    setup: str                                   # camera position id, e.g. \"store_wide\"\n    axis: str                                    # action line in this location, e.g. \"store_line\"\n    side: Literal[\"front\", \"back\"] = \"front\"     # which side of the action line the camera is on (180\u00b0 rule)\n    size: Literal[\"wide\", \"medium\", \"close\", \"insert\"] = \"medium\"\n    move: Literal[\"static\", \"pan_left\", \"pan_right\", \"tilt_up\", \"tilt_down\", \"push_in\", \"pull_out\",\n                  \"track_left\", \"track_right\"] = \"static\"\n\n\nclass Placement(BaseModel):\n    box: Box\n    facing: Optional[Facing] = None\n\n    @field_validator(\"box\")\n    @classmethod\n    def _box_ok(cls, b):\n        if not (b[0] < b[2] and b[1] < b[3]):\n            raise ValueError(f\"box {b} must be x0 < x1 and y0 < y1\")\n        if not all(-0.6 <= v <= 1.6 for v in b):           # slightly off-screen is allowed (entering/exiting)\n            raise ValueError(f\"box {b} is far outside the frame\")\n        return b\n\n\nclass Event(BaseModel):\n    id: str\n    type: Literal[\"enter\", \"walk_to\", \"stop\", \"take_out\", \"place\", \"pick_up\", \"hand_over\", \"scan\", \"look_at\",\n                  \"talk\", \"sit\", \"stand\", \"open\", \"exit\", \"idle\", \"gesture\"]\n    actor: str\n    obj: Optional[str] = None                    # object acted on (place / pick_up / hand_over / take_out / scan)\n    target: Optional[str] = None                 # entity or anchor (counter, table\u2026) the action is aimed at\n    t: tuple[float, float]                       # start, end (fractions of the shot)\n    to_box: Optional[Box] = None                 # where the actor ends (locomotion) or where the object ends (place)\n    critical: bool = True                        # must be visibly completed; checked after generation\n    describe: str = \"\"                           # wording for the prompt (generated if empty)\n\n    @field_validator(\"t\")\n    @classmethod\n    def _t_ok(cls, t):\n        if not (0.0 <= t[0] < t[1] <= 1.0):\n            raise ValueError(f\"event time {t} must satisfy 0 <= start < end <= 1\")\n        return t\n\n\nclass ShotState(BaseModel):\n    shot_id: str\n    location: str\n    camera: Camera\n    start: dict[str, Placement]                  # visible entities at the first frame (characters, props, anchors)\n    holders: dict[str, str] = Field(default_factory=dict)   # object -> who/what holds it at the first frame\n    events: list[Event] = Field(default_factory=list)\n    axis_change: Optional[str] = None            # reason, when the camera deliberately crosses the action line\n    transition: Optional[str] = None             # \"cut\", \"time_skip\": allows positions (not ownership) to reset\n    moved_offscreen: dict[str, str] = Field(default_factory=dict)   # object -> new holder, changed between shots\n    offscreen_reason: Optional[str] = None       # required with moved_offscreen (e.g. \"shipped to the returns centre\")\n    tail: float = 0.15                           # share of the shot kept AFTER the last critical event\n\n    @model_validator(mode=\"after\")\n    def _events_ordered(self):\n        last = 0.0\n        for e in self.events:\n            if e.t[0] < last - 1e-6 and e.type in LOCOMOTION | MANIPULATION:\n                raise ValueError(f\"{self.shot_id}: event {e.id} starts before the previous action ends\")\n            if e.type in LOCOMOTION | MANIPULATION:\n                last = e.t[1]\n        return self\n\n\nclass Entity(BaseModel):\n    kind: Literal[\"character\", \"prop\", \"anchor\"]\n    name: str = \"\"\n    small: bool = False                          # too small for reliable automatic detection (earbuds, cards)\n\n\nclass World(BaseModel):\n    \"\"\"What is true after a shot: positions are only comparable within the same camera axis and side.\"\"\"\n    location: Optional[str] = None\n    axis: Optional[str] = None\n    side: Optional[str] = None\n    setup: Optional[str] = None\n    boxes: dict[str, Box] = Field(default_factory=dict)\n    facing: dict[str, Optional[str]] = Field(default_factory=dict)\n    holders: dict[str, str] = Field(default_factory=dict)\n    moving: dict[str, int] = Field(default_factory=dict)       # entity -> screen direction it was travelling (+1/-1)\n\n\nclass Issue(BaseModel):\n    level: Literal[\"error\", \"warn\"]\n    rule: str\n    shot: Optional[str]\n    message: str\n    fix: str = \"\"\n\n\ndef _issue(level, rule, shot, msg, fix=\"\"):\n    return Issue(level=level, rule=rule, shot=shot, message=msg, fix=fix)\n\n\ndef simulate(state: ShotState, before: World, entities: dict[str, Entity], max_actions=(1, 1)):\n    \"\"\"Apply one shot's events to the world. Returns (world after the shot, issues).\"\"\"\n    iss = []\n    sid = state.shot_id\n    same_view = (before.location == state.location and before.axis == state.camera.axis and not state.transition)\n    # ---- 180\u00b0 rule: crossing the action line swaps left and right on screen\n    if same_view and before.side and before.side != state.camera.side and not state.axis_change:\n        iss.append(_issue(\"error\", \"X1\", sid, f\"the camera crosses the action line ({before.side} \u2192 \"\n                          f\"{state.camera.side}) without a reason: left and right swap for the viewer\",\n                          \"keep the camera on the same side, or set \\\"axis_change\\\" with the reason (e.g. a reveal)\"))\n    for name in list(state.start) + list(state.holders):\n        if name not in entities:\n            iss.append(_issue(\"error\", \"X0\", sid, f\"unknown entity '{name}'\", \"declare it in plan \\\"entities\\\"\"))\n    # ---- ownership continues across cuts: objects only change hands through events (or a declared off-screen move)\n    if state.moved_offscreen and not state.offscreen_reason:\n        iss.append(_issue(\"error\", \"X2\", sid, \"\\\"moved_offscreen\\\" needs an \\\"offscreen_reason\\\"\",\n                          \"say why the object changed place between shots (e.g. shipped)\"))\n    before = before.model_copy(update={\"holders\": {**before.holders, **state.moved_offscreen}})\n    for obj, holder in state.holders.items():\n        prev = before.holders.get(obj)\n        if prev and prev != holder:\n            iss.append(_issue(\"error\", \"X2\", sid, f\"'{obj}' was held by '{prev}' but this shot starts with \"\n                              f\"'{holder}' holding it, and no event moved it\", \"add the hand-over/place event\"))\n    holders = dict(before.holders); holders.update(state.holders)\n    # ---- left/right order and travel direction continue within the same view\n    if same_view and state.camera.side == before.side:\n        common = [e for e in state.start if e in before.boxes and entities.get(e, Entity(kind=\"prop\")).kind != \"anchor\"]\n        for a in common:\n            for b in common:\n                if a < b and (cx(before.boxes[a]) - cx(before.boxes[b])) * (cx(state.start[a].box) - cx(state.start[b].box)) < 0:\n                    iss.append(_issue(\"error\", \"X3\", sid, f\"'{a}' and '{b}' swap left/right sides between shots\",\n                                      \"keep their screen order, or change the axis on purpose (\\\"axis_change\\\")\"))\n    boxes = {k: v.box for k, v in state.start.items()}\n    facing = {k: v.facing for k, v in state.start.items()}\n    moving = {}\n    n_loco = n_manip = 0\n    for e in state.events:\n        if e.actor not in boxes and e.type != \"enter\":\n            iss.append(_issue(\"error\", \"E1\", sid, f\"event {e.id}: '{e.actor}' is not in the shot at that time\",\n                              \"add them to \\\"start\\\" or begin with an \\\"enter\\\" event\"))\n            continue\n        if e.type in LOCOMOTION:\n            n_loco += 1\n            if e.to_box is None:\n                iss.append(_issue(\"error\", \"E2\", sid, f\"event {e.id} ({e.type}) needs \\\"to_box\\\" (where they end)\"))\n                continue\n            if e.type == \"enter\":\n                b0 = boxes.get(e.actor)\n                if b0 is None:\n                    iss.append(_issue(\"error\", \"E3\", sid, f\"event {e.id}: '{e.actor}' enters, so give their first, \"\n                                      \"partly off-screen box in \\\"start\\\"\"))\n                    continue\n                if not (b0[0] < 0.05 or b0[2] > 0.95):\n                    iss.append(_issue(\"error\", \"E3\", sid, f\"event {e.id}: '{e.actor}' enters but starts in the middle \"\n                                      \"of the frame, not at an edge\", \"start the box at the left or right edge\"))\n            dx = cx(e.to_box) - cx(boxes[e.actor])\n            if abs(dx) > 0.05:\n                d = 1 if dx > 0 else -1\n                moving[e.actor] = d\n                want = \"right\" if d > 0 else \"left\"\n                if facing.get(e.actor) in (\"left\", \"right\") and facing[e.actor] != want:\n                    iss.append(_issue(\"error\", \"E4\", sid, f\"event {e.id}: '{e.actor}' faces {facing[e.actor]} but \"\n                                      f\"walks {want} (walking backwards)\", f\"set facing \\\"{want}\\\"\"))\n                facing[e.actor] = want\n                prev_dir = before.moving.get(e.actor) if same_view and state.camera.side == before.side else None\n                if prev_dir and prev_dir != d and not state.axis_change:\n                    iss.append(_issue(\"error\", \"X4\", sid, f\"'{e.actor}' was moving {'right' if prev_dir > 0 else 'left'} \"\n                                      f\"at the end of the last shot but moves {want} here (reversal)\",\n                                      \"keep the direction, add a turn-around event, or set \\\"axis_change\\\"\"))\n            boxes[e.actor] = e.to_box\n        elif e.type == \"stop\":\n            moving.pop(e.actor, None)\n        elif e.type in MANIPULATION:\n            n_manip += 1\n            if not e.obj:\n                iss.append(_issue(\"error\", \"E5\", sid, f\"event {e.id} ({e.type}) needs \\\"obj\\\"\"))\n                continue\n            holder = holders.get(e.obj)\n            if e.actor in moving and e.type in (\"place\", \"pick_up\", \"hand_over\"):\n                iss.append(_issue(\"error\", \"E6\", sid, f\"event {e.id}: '{e.actor}' is still walking while trying to \"\n                                  f\"{e.type} '{e.obj}'\", \"add a \\\"stop\\\" event before it\"))\n            if e.type in (\"place\", \"hand_over\"):\n                if holder != e.actor:\n                    iss.append(_issue(\"error\", \"E7\", sid, f\"event {e.id}: '{e.actor}' {e.type}s '{e.obj}' but \"\n                                      f\"'{holder}' holds it\", \"add the take_out/pick_up/hand_over that gives it to them\"))\n                tgt = boxes.get(e.target) if e.target else None\n                if e.target and tgt is None:\n                    iss.append(_issue(\"error\", \"E8\", sid, f\"event {e.id}: target '{e.target}' is not in the shot\"))\n                elif tgt and not near(boxes[e.actor], tgt):\n                    iss.append(_issue(\"error\", \"E9\", sid, f\"event {e.id}: '{e.actor}' is not next to '{e.target}' \"\n                                      f\"(can't {e.type} across the room)\", \"walk there first (walk_to + stop)\"))\n                holders[e.obj] = e.target if e.type == \"place\" else e.target\n                if e.to_box:\n                    boxes[e.obj] = e.to_box\n            elif e.type in (\"pick_up\", \"take_out\"):\n                if e.type == \"pick_up\":\n                    src = holder\n                    if src and src in boxes and not near(boxes[e.actor], boxes[src]):\n                        iss.append(_issue(\"error\", \"E9\", sid, f\"event {e.id}: '{e.actor}' picks up '{e.obj}' from \"\n                                          f\"'{src}' without being next to it\"))\n                elif holder not in (e.actor, None):\n                    iss.append(_issue(\"error\", \"E7\", sid, f\"event {e.id}: '{e.actor}' takes out '{e.obj}' but \"\n                                      f\"'{holder}' holds it\"))\n                holders[e.obj] = e.actor\n            elif e.type == \"scan\":\n                tgt = e.target or holder\n                if tgt in boxes and not near(boxes[e.actor], boxes[tgt]):\n                    iss.append(_issue(\"error\", \"E9\", sid, f\"event {e.id}: '{e.actor}' scans '{e.obj}' from too far away\"))\n    # ---- keep each AI clip simple: one movement + one hand action at most (models drop or merge more)\n    if n_loco > max_actions[0] or n_manip > max_actions[1]:\n        iss.append(_issue(\"error\", \"E10\", sid, f\"{n_loco} movements and {n_manip} hand actions in one clip \"\n                          f\"(max {max_actions[0]} + {max_actions[1]}): the model will skip or merge some\",\n                          \"split the shot into shorter shots, one action each\"))\n    crit = [e for e in state.events if e.critical]\n    if crit and max(e.t[1] for e in crit) > 1 - state.tail + 1e-6:\n        iss.append(_issue(\"error\", \"E11\", sid, \"the last required action ends too close to the cut \"\n                          f\"(needs {state.tail:.0%} of the shot after it, so editing never cuts it off)\",\n                          \"end the action earlier\"))\n    after = World(location=state.location, axis=state.camera.axis, side=state.camera.side, setup=state.camera.setup,\n                  boxes=boxes, facing=facing, holders=holders, moving=moving)\n    return after, iss\n\n\ndef check_sequence(states: list[ShotState], entities: dict[str, Entity], max_actions=(1, 1)):\n    \"\"\"Simulate a list of shots in order. Returns (worlds after each shot, all issues).\"\"\"\n    world, worlds, issues = World(), [], []\n    for st in states:\n        if st.location != world.location or st.transition:\n            # new place or time skip: positions reset, ownership carries over\n            world = World(holders=world.holders)\n        world, iss = simulate(st, world, entities, max_actions)\n        worlds.append(world)\n        issues += iss\n    return worlds, issues\n\n\ndef parse_plan(plan: dict):\n    \"\"\"(states in shot order, entities, problems) from a plan's \"entities\" and per-shot \"state\" blocks.\"\"\"\n    problems = []\n    ents = {}\n    for k, v in (plan.get(\"scene_entities\") or {}).items():\n        try:\n            ents[k] = Entity(**v)\n        except Exception as e:                                      # pydantic ValidationError\n            problems.append(_issue(\"error\", \"X0\", None, f\"entity '{k}': {e}\"))\n    states = []\n    for s in plan.get(\"shots\", []):\n        if \"state\" not in s:\n            continue\n        try:\n            states.append(ShotState(shot_id=s[\"shot_id\"], **s[\"state\"]))\n        except Exception as e:\n            problems.append(_issue(\"error\", \"X0\", s[\"shot_id\"], f\"state block is invalid: {e}\"))\n    return states, ents, problems\n\n\ndef prompt_hints(state: ShotState, entities: dict[str, Entity]) -> str:\n    \"\"\"Plain-language spatial wording added to the prompt (the layout video does the real steering).\"\"\"\n    def where(b):\n        x = cx(b)\n        return \"on the left\" if x < 0.4 else \"on the right\" if x > 0.6 else \"in the centre\"\n    parts = []\n    for k, p in state.start.items():\n        nm = entities[k].name or k if k in entities else k\n        if 0 <= cx(p.box) <= 1:\n            parts.append(f\"{nm} {where(p.box)} of the frame\")\n    for e in state.events:\n        if e.describe:\n            parts.append(e.describe)\n    cam = {\"static\": \"static camera\", \"push_in\": \"slow camera push in\", \"pull_out\": \"slow camera pull out\"}.get(\n        state.camera.move, state.camera.move.replace(\"_\", \" \"))\n    parts.append(f\"{state.camera.size} shot, {cam}\")\n    return \", \".join(parts)\n", "stages.py": "\"\"\"Stage checkpoints: a stage's result is saved with a hash of everything that shaped it, so a rerun (after a Colab\ndisconnect, or with unchanged settings) reuses it instead of doing the work again; a changed input re-runs it.\n\n    sheet = stage(DIR, \"research\", {\"topic\": t, \"v\": 1}, lambda: do_research())\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib, json, time\nfrom pathlib import Path\n\n\ndef key_of(inputs) -> str:\n    return hashlib.sha256(json.dumps(inputs, sort_keys=True, default=str).encode()).hexdigest()[:16]\n\n\ndef stage(folder, name, inputs, fn, log=print):\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\n    f, k = folder / f\"{name}.json\", key_of(inputs)\n    if f.exists():\n        try:\n            d = json.loads(f.read_text())\n            if d.get(\"key\") == k:\n                log(f\"stage {name}: reused (saved {d.get('saved')}, key {k})\")\n                return d[\"result\"]\n        except Exception:\n            log(f\"stage {name}: checkpoint unreadable, running again\")\n    t0 = time.time()\n    result = fn()\n    tmp = f.with_suffix(\".tmp\")                      # write-then-rename: an interrupted save never looks complete\n    tmp.write_text(json.dumps({\"key\": k, \"inputs\": inputs, \"saved\": time.strftime(\"%Y-%m-%d %H:%M:%S\"),\n                               \"seconds\": round(time.time() - t0, 1), \"result\": result}, indent=1, default=str))\n    tmp.replace(f)\n    log(f\"stage {name}: done in {time.time() - t0:.0f}s (key {k})\")\n    return result\n", "story.py": "\"\"\"Two-friend storytelling: Story Director + Visual Story Director (fixed code; the LLM only writes data).\n\nPipeline (each stage small, structured, validated with Pydantic, retried with the error fed back):\n    facts (research.FactSheet)  \u2192  beats  \u2192  dialogue turns (each factual turn cites fact ids)\n    \u2192  visual intentions (per turn, independent of the turns' boundaries)  \u2192  checks  \u2192  plan (cast format)\nThe plan is the existing cast plan with one new recipe, \"story\": a SHOT is a dialogue scene (its lines are the\naudio units, placed narration-first as before) and `story.visuals` is that scene's own VISUAL timeline: each\nvisual starts at a point inside a turn ({\"line\": k, \"at\": 0..1}), so one turn can span a cutaway and a cut need\nnot happen at a line boundary. The audio track never depends on the visuals.\n\nVisual Story Director: maps each visual intention to the cheapest medium that is faithful, with an explicit\nfallback chain \u2014 character scene \u2192 licensed media \u2192 animated demonstration \u2192 AI video (only when allowed and\nwithin budget) \u2192 minimal graphic. Every decision is recorded (medium, why, fallback trail), and two independent\nstatistics are reported: NARRATIVE PURPOSE (what a visual is for) and RENDERING MEDIUM (how it was made).\n\"\"\"\nfrom __future__ import annotations\n\nimport json, re\nfrom typing import Literal, Optional\n\nfrom pydantic import BaseModel, Field, field_validator, model_validator\n\nimport characters as charbible\n\nBEATS = (\"observation\", \"question\", \"first_answer\", \"demonstration\", \"misconception\", \"correction\", \"reveal\",\n         \"real_world\", \"closing\")\nTURN_KINDS = (\"fact\", \"question\", \"analogy\", \"reaction\", \"misconception\", \"correction\", \"instruction\", \"observation\",\n              \"prediction\", \"challenge\", \"hypothetical\", \"opinion\")   # the last four: Arjun is more than a questioner\nCLAIM_FREE = (\"reaction\", \"instruction\", \"observation\", \"misconception\", \"prediction\", \"opinion\")  # not NLI-checked\n# conversational intentions (voice_direction.INTENTS): what the line is FOR, used by the voice performance director\nTURN_INTENTS = (\"genuine_curiosity\", \"thinking_aloud\", \"playful_disbelief\", \"gentle_correction\", \"explaining_observation\",\n                \"asking_confirmation\", \"quiet_realization\", \"enthusiastic_discovery\", \"sharing_surprising_fact\",\n                \"offering_reassurance\", \"reflective_explanation\", \"satisfying_revelation\", \"guiding_demo\", \"playful_remark\")\nFACTUAL_KINDS = (\"fact\", \"correction\")\nTONES = (\"curious\", \"explainer\", \"warm\", \"happy\", \"friendly\", \"awe\", \"serious\")\nINTENTS = (\"establishing\", \"two_friend\", \"closeup\", \"ots\", \"reaction\", \"real_event\", \"object_demo\", \"cutaway\",\n           \"reenactment\", \"analogy\", \"licensed_media\", \"animated_illustration\", \"minimal_overlay\")\nMEDIA = (\"character_scene\", \"licensed_media\", \"animated_demo\", \"ai_video\", \"presentation_graphic\")\nPRESENTATION = (\"minimal_overlay\",)                       # intents that are \"slides\" rather than scenes\n# shot-level beat used by the existing engine (voice direction BEAT_ROLE, validator S7)\nENGINE_BEAT = {\"observation\": \"hook\", \"question\": \"turn\", \"first_answer\": \"setup\", \"demonstration\": \"step\",\n               \"misconception\": \"turn\", \"correction\": \"resolution\", \"reveal\": \"reveal\", \"real_world\": \"step\",\n               \"closing\": \"payoff\"}\nMAX_TURN_WORDS = 30\nMIN_FACT_TURNS = 2                       # D8: at least this many sourced factual turns per story\nMAX_SENTENCE_WORDS = 22                  # = validate.py S4\nWORDS_PER_SECOND = 2.6\n\n\n# ------------------------------------------------------------------ the data\nclass Turn(BaseModel):\n    speaker: str\n    text: str\n    kind: Literal[TURN_KINDS] = \"observation\"\n    fact_ids: list[str] = Field(default_factory=list)\n    tone: Literal[TONES] = \"friendly\"\n    tts: Optional[str] = None\n    source_text: Optional[str] = None        # non-English turns: the English wording it translates (fact-checked)\n    intent: Optional[Literal[TURN_INTENTS]] = None   # why the character says it (else derived from the context)\n    expr: Optional[str] = None               # speaker's expression while saying it\n    gesture: Optional[str] = None\n\n    @field_validator(\"text\")\n    @classmethod\n    def short(cls, v):\n        v = v.strip()\n        if not v or len(v.split()) > MAX_TURN_WORDS:\n            raise ValueError(f\"each turn must have 1\u2013{MAX_TURN_WORDS} words (a friend, not a lecture)\")\n        for sent in re.split(r\"(?<=[.!?])\\s+\", v):           # the plan validator's S4 limit, enforced early\n            if len(sent.split()) > MAX_SENTENCE_WORDS:\n                raise ValueError(f\"sentence of {len(sent.split())} words (max {MAX_SENTENCE_WORDS}): split it\")\n        return v\n\n\nclass Visual(BaseModel):\n    id: str\n    intent: Literal[INTENTS]\n    turn: int = 0                            # index of the turn (in this scene) during which the visual starts\n    at: float = Field(0.0, ge=0.0, le=1.0)   # how far into that turn\n    subject: Optional[str] = None            # character the shot is about (closeup / ots / reaction)\n    template: Optional[str] = None           # demonstration template (demos.TEMPLATES)\n    description: str = \"\"                    # what should be seen (concrete nouns: also the licensed-media query)\n    lines: list[str] = Field(default_factory=list)     # minimal overlay text\n    chars: dict = Field(default_factory=dict)          # per character: expr / gesture / wear / hold\n    changes: list[dict] = Field(default_factory=list)\n    screen: Optional[str] = None\n    shot: Optional[str] = None               # explicit camera shot for character scenes\n    world: Optional[str] = None              # another place than the scene's (e.g. the screen they are watching)\n    camera: Optional[dict] = None            # storybook plans: {\"movement\": \"static|slow_push_in|slow_pull_back|pan_*\"}\n    environment_variant: Optional[str] = None   # storybook plans: a camera preset of the environment pack\n\n\nclass Scene(BaseModel):\n    id: str = Field(pattern=r\"^[a-z][a-z0-9_]{1,15}$\")\n    beat: Literal[BEATS]\n    world: str\n    turns: list[Turn] = Field(min_length=1, max_length=6)\n    visuals: list[Visual] = Field(min_length=1)\n    transition: Optional[str] = None         # \"time_skip\" / \"cut\" when the friends change place\n    sfx: list[dict] = Field(default_factory=list)      # optional sparse sound cues {\"type\", \"at\" s, \"dur\", \"gain_db\"}\n\n    @model_validator(mode=\"after\")\n    def visuals_inside(self):\n        prev = (-1, -1.0)\n        for v in self.visuals:\n            if not 0 <= v.turn < len(self.turns):\n                raise ValueError(f\"scene {self.id}: visual {v.id} starts in turn {v.turn}, which doesn't exist\")\n            if (v.turn, v.at) <= prev:\n                raise ValueError(f\"scene {self.id}: visuals must start in order (visual {v.id})\")\n            prev = (v.turn, v.at)\n        if (self.visuals[0].turn, self.visuals[0].at) != (0, 0.0):\n            raise ValueError(f\"scene {self.id}: the first visual must start with the first turn (turn 0, at 0)\")\n        return self\n\n\nclass StoryFact(BaseModel):\n    id: str\n    text: str\n    source_url: str\n    title: str = \"\"\n    revid: Optional[int] = None\n\n\nclass Story(BaseModel):\n    title: str\n    topic: str\n    language: str = \"en\"\n    logline: str = \"\"\n    facts: list[StoryFact]\n    scenes: list[Scene] = Field(min_length=3, max_length=14)\n    vocabulary: list[str] = Field(default_factory=list)\n\n\n# ------------------------------------------------------------------ LLM stages (small, structured)\nclass BeatOutline(BaseModel):\n    title: str\n    beats: list[dict] = Field(min_length=4, max_length=10)       # {\"beat\": BEATS, \"goal\": str, \"fact_ids\": [...]}\n\n    @field_validator(\"beats\")\n    @classmethod\n    def known(cls, v):\n        for b in v:\n            if b.get(\"beat\") not in BEATS:\n                raise ValueError(f\"unknown beat {b.get('beat')}; use one of {BEATS}\")\n        return v\n\n\nclass SceneDialogue(BaseModel):\n    world: str\n    turns: list[Turn] = Field(min_length=1, max_length=6)\n\n\nclass SceneVisuals(BaseModel):\n    visuals: list[Visual] = Field(min_length=1, max_length=4)\n\n\nSYSTEM = \"\"\"You write short educational scenes where two friends talk.\n{curious} ({curious_p}) notices things and asks; {expert} ({expert_p}) explains with everyday examples and small\ndemonstrations. Rules:\n- Use ONLY the numbered facts given. A turn of kind \"fact\" or \"correction\" must cite the facts that support it.\n- Never add a number, name, date or claim that is not in the cited facts.\n- Natural, short turns (at most 25 words). No lectures, no \"Wow!\"-only replies, no mocking.\n- {curious} may guess wrong (kind \"misconception\"); {expert} then corrects it with sourced facts (kind \"correction\").\n- Answer with ONE JSON object and nothing else.\"\"\"\n\n\ndef system_prompt(bible):\n    a, b = bible.by_role(\"curious\"), bible.by_role(\"expert\")\n    return SYSTEM.format(curious=a.name, curious_p=a.personality, expert=b.name, expert_p=b.personality)\n\n\ndef beats_prompt(topic, facts, target_s):\n    lines = \"\\n\".join(f\"{f.id}: {f.text}\" for f in facts)\n    return (f\"Topic: {topic}\\nTarget: about {target_s} seconds.\\nPlan 5\u20138 beats from: {', '.join(BEATS)}. Start with \"\n            \"an everyday observation, then the question. A misconception + correction is optional. End with closing.\\n\"\n            f\"Facts:\\n{lines}\\n\\nReturn: {{\\\"title\\\": \\\"max 8 words\\\", \\\"beats\\\": [{{\\\"beat\\\": \\\"observation\\\", \"\n            \"\\\"goal\\\": \\\"what the viewer learns or feels\\\", \\\"fact_ids\\\": [\\\"F1\\\"]}}]}}\")\n\n\ndef dialogue_prompt(bible, beat, facts, words, worlds):\n    a, b = bible.by_role(\"curious\"), bible.by_role(\"expert\")\n    lines = \"\\n\".join(f\"{f.id}: {f.text}\" for f in facts)\n    return (f\"Beat: {beat['beat']} \u2014 {beat['goal']}\\nAbout {words} spoken words in 2\u20134 turns.\\nSpeakers: \\\"{a.id}\\\" \"\n            f\"({a.name}), \\\"{b.id}\\\" ({b.name}).\\nPlace (world): one of {', '.join(worlds)}.\\nFacts:\\n{lines}\\n\\nReturn: \"\n            \"{\\\"world\\\": \\\"...\\\", \\\"turns\\\": [{\\\"speaker\\\": \\\"\" + a.id + \"\\\", \\\"text\\\": \\\"...\\\", \\\"kind\\\": \\\"question\\\", \"\n            \"\\\"fact_ids\\\": [], \\\"tone\\\": \\\"curious\\\"}]}\\nkinds: \" + \", \".join(TURN_KINDS) + \"; tones: \" + \", \".join(TONES))\n\n\ndef visuals_prompt(scene_turns, templates):\n    t = \"\\n\".join(f\"{i}: {x.speaker}: {x.text}\" for i, x in enumerate(scene_turns))\n    return (f\"Turns:\\n{t}\\n\\nChoose 1\u20133 visuals that SHOW what is being said (not slides). Intents: {', '.join(INTENTS)}. \"\n            f\"Demonstration templates: {', '.join(templates)}. A visual starts during a turn (\\\"turn\\\": index, \\\"at\\\": \"\n            \"0..1). The first starts at turn 0, at 0. Prefer the friends, real objects and demonstrations; use \"\n            \"\\\"minimal_overlay\\\" only for a key number or term.\\nReturn: {\\\"visuals\\\": [{\\\"id\\\": \\\"v1\\\", \\\"intent\\\": \"\n            \"\\\"two_friend\\\", \\\"turn\\\": 0, \\\"at\\\": 0.0, \\\"description\\\": \\\"...\\\"}]}\")\n\n\ndef make_story(topic, sheet, llm, bible, target_s=75, language=\"en\", log=print, max_facts=24, ask_json=None):\n    \"\"\"facts \u2192 beats \u2192 dialogue per beat \u2192 visual intentions per scene. Returns a validated Story (not yet checked\n    against the rules: call check_story / ground).\"\"\"\n    import demos\n    if ask_json is None:\n        from planner import ask_json\n    facts = sheet.facts[:max_facts]\n    sysp = system_prompt(bible)\n    outline = ask_json(llm, sysp, beats_prompt(topic, facts, target_s), BeatOutline, log=log)\n    worlds = (\"living_room\", \"cinema_seats\", \"cinema_screen\", \"lobby\", \"studio\")\n    per = max(int(target_s * WORDS_PER_SECOND / len(outline.beats)), 12)\n    by_id = {f.id: f for f in facts}\n    scenes = []\n    for k, b in enumerate(outline.beats):\n        rel = [by_id[i] for i in b.get(\"fact_ids\", []) if i in by_id] or facts[:6]\n        d = ask_json(llm, sysp, dialogue_prompt(bible, b, rel, per, worlds), SceneDialogue, log=log)\n        v = ask_json(llm, sysp, visuals_prompt(d.turns, demos.TEMPLATES), SceneVisuals, log=log)\n        scenes.append(Scene(id=f\"sc{k + 1:02d}\", beat=b[\"beat\"], world=d.world if d.world in worlds else \"studio\",\n                            turns=d.turns, visuals=v.visuals))\n    used = {i for s in scenes for t in s.turns for i in t.fact_ids}\n    return Story(title=outline.title, topic=topic, language=language,\n                 facts=[StoryFact(id=f.id, text=f.text, source_url=f.source_url, title=f.title, revid=f.revid)\n                        for f in facts if f.id in used], scenes=scenes)\n\n\n# ------------------------------------------------------------------ rule checks (no model)\ndef _issue(level, rule, scene, msg, fix=\"\"):\n    return {\"level\": level, \"rule\": rule, \"shot\": scene, \"message\": msg, \"fix\": fix}\n\n\nEMPTY_REACTIONS = re.compile(r\"^\\s*(wow|cool|oh|okay|ok|nice|amazing|really|great|whoa|i see)[\\s!.?,]*$\", re.I)\n\n\ndef check_story(story: Story, bible, target_s=None):\n    \"\"\"Dialogue quality, grounding structure, beat order and visual-variety rules. Returns validate-style issues.\"\"\"\n    out = []\n    ids = set(bible.characters)\n    cur, exp = bible.by_role(\"curious\").id, bible.by_role(\"expert\").id\n    fids = {f.id for f in story.facts}\n    turns = [(s, t) for s in story.scenes for t in s.turns]\n    for s, t in turns:\n        if t.speaker not in ids:\n            out.append(_issue(\"error\", \"D0\", s.id, f\"unknown speaker '{t.speaker}'\", f\"use {sorted(ids)}\"))\n        for f in t.fact_ids:\n            if f not in fids:\n                out.append(_issue(\"error\", \"D1\", s.id, f\"turn cites unknown fact '{f}'\"))\n        if t.kind in FACTUAL_KINDS and not t.fact_ids:\n            out.append(_issue(\"error\", \"D1\", s.id, f\"factual turn cites no fact: '{t.text[:60]}'\",\n                              \"cite the fact ids it states, or make it a question/reaction/analogy\"))\n        if t.kind == \"reaction\" and EMPTY_REACTIONS.match(t.text):\n            out.append(_issue(\"error\", \"D3\", s.id, f\"empty reaction '{t.text}'\",\n                              \"react with a real follow-up thought or question\"))\n    said = []\n    for s, t in turns:                                    # the viewer must not hear the same fact twice\n        if t.kind not in FACTUAL_KINDS:\n            continue\n        w = set(re.findall(r\"[a-z]{4,}\", t.text.lower()))\n        for s0, w0 in said:\n            if len(w & w0) / max(len(w | w0), 1) >= 0.8:\n                out.append(_issue(\"error\", \"D7\", s.id, f\"repeats what was already said in {s0}: '{t.text[:60]}'\",\n                                  \"say something new, or ask/react instead\"))\n                break\n        said.append((s.id, w))\n    for i, (s, t) in enumerate(turns):\n        if t.kind == \"misconception\":\n            # corrected by the other friend within the next 4 turns (a test may come first), with sourced facts\n            nxt = [x for _, x in turns[i + 1:i + 5]]\n            if not any(x.kind in FACTUAL_KINDS and x.speaker != t.speaker for x in nxt):\n                out.append(_issue(\"error\", \"D4\", s.id, \"a misconception is not corrected within the next four turns\",\n                                  \"the other friend corrects it with a sourced 'correction' or 'fact' turn\"))\n    run, prev = 0, None\n    for s, t in turns:\n        run = run + 1 if t.speaker == prev else 1\n        prev = t.speaker\n        if run > 2:\n            out.append(_issue(\"error\", \"D2\", s.id, f\"{run} turns in a row by the same friend (a monologue)\",\n                              \"let the other friend ask or react in between\"))\n    words = {c: sum(len(t.text.split()) for _, t in turns if t.speaker == c) for c in ids}\n    total = max(sum(words.values()), 1)\n    if words.get(exp, 0) / total > 0.78:\n        out.append(_issue(\"warn\", \"D2\", None, f\"{bible.characters[exp].name} speaks {words[exp] / total:.0%} of the words\",\n                          \"give the curious friend more real questions and guesses\"))\n    qs = sum(t.kind == \"question\" and t.speaker == cur for _, t in turns)\n    if qs < 2:\n        out.append(_issue(\"error\", \"D3\", None, f\"the curious friend asks only {qs} question(s)\", \"at least 2 real questions\"))\n    n_facts = sum(t.kind in FACTUAL_KINDS and bool(t.fact_ids) for _, t in turns)\n    if n_facts < MIN_FACT_TURNS:                          # an explainer that explains nothing is not a video we ship\n        out.append(_issue(\"error\", \"D8\", None, f\"only {n_facts} sourced factual turn(s) (min {MIN_FACT_TURNS})\",\n                          \"let the expert friend explain with the cited facts\"))\n    beats = [s.beat for s in story.scenes]\n    if \"question\" not in beats:\n        out.append(_issue(\"error\", \"D5\", None, \"no scene asks the central question (beat 'question')\"))\n    if beats[-1] != \"closing\":\n        out.append(_issue(\"error\", \"D5\", story.scenes[-1].id, \"the story must end with a 'closing' scene\"))\n    first_q = next((i for i, b in enumerate(beats) if b == \"question\"), 99)\n    first_fact = next((i for i, (s, t) in enumerate(turns) if t.kind in FACTUAL_KINDS), 99)\n    first_q_turn = sum(len(s.turns) for s in story.scenes[:first_q])\n    if first_fact < first_q_turn:\n        out.append(_issue(\"warn\", \"D5\", None, \"a fact is explained before the question is asked\",\n                          \"observation \u2192 question \u2192 answer\"))\n    if target_s:\n        est = total / WORDS_PER_SECOND + 0.6 * len(turns)\n        if not 0.7 * target_s <= est <= 1.35 * target_s:\n            out.append(_issue(\"warn\", \"D6\", None, f\"estimated length {est:.0f}s vs target {target_s}s\",\n                              \"add or remove a beat\"))\n    # visual rules (before the director: the intentions themselves)\n    vis = [(s, v) for s in story.scenes for v in s.visuals]\n    pres = sum(v.intent in PRESENTATION for _, v in vis)\n    if pres / max(len(vis), 1) > 0.2:\n        out.append(_issue(\"warn\", \"V5\", None, f\"{pres}/{len(vis)} visuals are text overlays (presentation graphics)\",\n                          \"show the friends, objects or a demonstration instead\"))\n    same, prevk = 1, None\n    for s, v in vis:\n        key = (v.intent, v.shot, s.world)\n        same = same + 1 if key == prevk else 1\n        prevk = key\n        if same >= 3:\n            out.append(_issue(\"warn\", \"V6\", s.id, f\"{same} visuals in a row are the same ({v.intent} in {s.world})\",\n                              \"change the angle, use a reaction or a cutaway\"))\n    for s, v in vis:\n        if v.intent == \"reenactment\" and not v.description:\n            out.append(_issue(\"warn\", \"V7\", s.id, f\"reenactment {v.id} has no description of what it re-creates\"))\n    return out\n\n\ndef ground(story: Story, nli, topic=\"\"):\n    \"\"\"Fact check every factual turn against the facts it cites (factcheck.check_sentences: numbers, names, NLI).\n    Analogies must be worded as comparisons; questions make no claim. Returns {turn key: result}.\"\"\"\n    import factcheck\n    by_id = {f.id: f.text for f in story.facts}\n    items = []\n    for s in story.scenes:\n        for k, t in enumerate(s.turns):\n            if t.kind in CLAIM_FREE:\n                continue                                    # no claim (a misconception is corrected by D4)\n            # the NLI model reads English: a translated turn is checked through the English wording it translates\n            items.append({\"id\": f\"{s.id}_{k + 1}\", \"text\": t.source_text or t.text, \"evidence\": [by_id[i] for i in t.fact_ids if i in by_id],\n                          \"question\": t.kind in (\"question\", \"challenge\"), \"analogy\": t.kind in (\"analogy\", \"hypothetical\")})\n    return factcheck.check_sentences(items, nli, topic=topic)\n\n\ndef enforce_grounding(st: Story, nli, bible, log=print):\n    \"\"\"Fact check, then DROP every claim-making turn that is not PASS (nothing unsupported is said). Visuals that\n    started in a dropped turn move to the next surviving turn. Returns (story, results, dropped keys).\"\"\"\n    res = ground(st, nli, topic=st.topic)\n    bad = {k for k, r in res.items() if r[\"status\"] != \"PASS\"}\n    if not bad:\n        return st, res, []\n    scenes = []\n    for s in st.scenes:\n        keep = [k for k in range(len(s.turns)) if f\"{s.id}_{k + 1}\" not in bad]\n        if not keep:\n            log(f\"story: scene {s.id} dropped (none of its turns is supported)\")\n            continue\n        new_of = {}\n        for k in range(len(s.turns)):                    # dropped turn \u2192 the next kept one (or the last kept)\n            new_of[k] = next((i for i, kk in enumerate(keep) if kk >= k), len(keep) - 1)\n        vis, seen = [], set()\n        for v in s.visuals:\n            nv = v.model_copy(update={\"turn\": new_of[v.turn], \"at\": v.at if v.turn in keep else 0.0})\n            if (nv.turn, nv.at) in seen or (vis and (nv.turn, nv.at) <= (vis[-1].turn, vis[-1].at)):\n                continue\n            seen.add((nv.turn, nv.at)); vis.append(nv)\n        vis[0] = vis[0].model_copy(update={\"turn\": 0, \"at\": 0.0})\n        scenes.append(s.model_copy(update={\"turns\": [s.turns[k] for k in keep], \"visuals\": vis}))\n    for k in sorted(bad):\n        log(f\"story: dropped unsupported turn {k}: {res[k]['text'][:80]!r} ({res[k]['why']})\")\n    out = Story.model_validate(st.model_copy(update={\"scenes\": scenes}).model_dump())\n    return out, res, sorted(bad)\n\n\n# ------------------------------------------------------------------ Visual Story Director\nINTENT_SHOT = {\"establishing\": \"establishing\", \"two_friend\": \"two_shot\", \"closeup\": \"closeup\", \"ots\": \"ots\",\n               \"reaction\": \"reaction\", \"reenactment\": \"wide\", \"cutaway\": \"wide\"}\nDEMO_FOR = {\"object_demo\", \"animated_illustration\", \"analogy\"}\nMEDIA_FOR = {\"real_event\", \"licensed_media\"}\n\n\nILLUSTRATION = {\"en\": \"Illustration\", \"te\": \"\u0c26\u0c43\u0c37\u0c4d\u0c1f\u0c3e\u0c02\u0c24\u0c02\"}     # on-screen tag of a re-enactment, per language\n\n\ndef direct(story: Story, bible, media=None, allow_ai=False, ai_budget=0):\n    \"\"\"Choose how every visual is made. media: {visual id: {\"file\", \"credit\", \"license\"}} for licensed images that\n    passed the licence + relevance gates. Returns {scene id: [visual spec for the plan]} and the decisions.\"\"\"\n    import demos\n    media = media or {}\n    plans, decisions = {}, []\n    ai_used = 0\n    for s in story.scenes:\n        out = []\n        speakers = [t.speaker for t in s.turns]\n        present = list(dict.fromkeys([bible.by_role(\"curious\").id, bible.by_role(\"expert\").id]))\n        for v in s.visuals:\n            trail, rec, spec, medium, label = [], None, None, None, None\n            want = v.intent\n            if want in MEDIA_FOR:\n                m = media.get(f\"{s.id}/{v.id}\") or media.get(v.id)      # \"scene/visual\" (ids repeat per scene)\n                if m:\n                    rec, medium = \"photo\", \"licensed_media\"\n                    spec = {\"file\": m[\"file\"], \"credit\": m[\"credit\"], \"move\": \"push_in\"}\n                else:\n                    trail.append(\"licensed media: none passed the licence/relevance gates\")\n                    if want == \"real_event\" and allow_ai and ai_used < ai_budget:\n                        trail.append(\"AI video: allowed, but AI cutaways inside a dialogue scene are not built yet\")\n                    want = \"object_demo\" if v.template else \"reenactment\" if want == \"real_event\" else \"two_friend\"\n            if rec is None and want in DEMO_FOR:\n                if v.template in demos.TEMPLATES:\n                    rec, medium = \"demo\", \"animated_demo\"\n                    spec = {\"template\": v.template, \"world\": v.world or s.world}\n                else:\n                    trail.append(f\"demonstration: no template for '{v.description[:40]}'\")\n                    want = \"two_friend\"\n            if rec is None and want == \"minimal_overlay\":\n                lines = [ln for ln in v.lines if len(ln.split()) <= 6][:3]\n                if lines:\n                    rec, medium = \"card\", \"presentation_graphic\"\n                    spec = {\"lines\": lines, \"headline\": True, \"reveal\": len(lines) > 1}\n                else:\n                    trail.append(\"overlay: no short text given\")\n                    want = \"two_friend\"\n            if rec is None:                                   # character scene (the default, never fails)\n                rec, medium = \"character\", \"character_scene\"\n                shot = v.shot or INTENT_SHOT.get(want, \"two_shot\")\n                subj = v.subject\n                if shot in (\"closeup\", \"ots\") and subj is None:\n                    subj = speakers[min(v.turn, len(speakers) - 1)]\n                if shot == \"reaction\" and subj is None:          # the one listening\n                    sp = speakers[min(v.turn, len(speakers) - 1)]\n                    subj = next(c for c in present if c != sp)\n                chars = {c: dict(v.chars.get(c, {})) for c in present}\n                spec = {\"world\": v.world or s.world, \"shot\": shot, \"chars\": chars}\n                if subj:\n                    spec[\"subject\"] = subj\n                if v.screen:\n                    spec[\"screen\"] = v.screen\n                if v.changes:\n                    spec[\"changes\"] = v.changes\n                if v.intent == \"reenactment\" or (v.intent == \"real_event\" and medium == \"character_scene\"):\n                    label = ILLUSTRATION.get(story.language, \"Illustration\")   # never shown as documentary footage\n                    spec[\"label\"] = label\n                if shot in (\"closeup\", \"reaction\"):\n                    spec[\"push\"] = True\n                if v.camera:\n                    spec[\"camera\"] = dict(v.camera)\n                if v.environment_variant:\n                    spec[\"environment_variant\"] = v.environment_variant\n            out.append({\"id\": v.id, \"intent\": v.intent, \"medium\": medium, \"recipe\": rec, rec: spec,\n                        \"start\": {\"line\": v.turn, \"at\": v.at}, \"description\": v.description,\n                        **({\"label\": label} if label else {}), **({\"fallback_trail\": trail} if trail else {})})\n            decisions.append({\"scene\": s.id, \"visual\": v.id, \"intent\": v.intent, \"medium\": medium, \"recipe\": rec,\n                              \"trail\": trail})\n        plans[s.id] = out\n    return plans, decisions\n\n\ndef statistics(rows):\n    \"\"\"Two independent statistics over visual screen time. rows: [{\"intent\", \"medium\", \"dur\"}].\"\"\"\n    tot = sum(r[\"dur\"] for r in rows) or 1.0\n    purpose, medium = {}, {}\n    for r in rows:\n        purpose[r[\"intent\"]] = purpose.get(r[\"intent\"], 0) + r[\"dur\"]\n        medium[r[\"medium\"]] = medium.get(r[\"medium\"], 0) + r[\"dur\"]\n    pct = lambda d: {k: round(100 * v / tot, 1) for k, v in sorted(d.items(), key=lambda kv: -kv[1])}\n    return {\"total_s\": round(tot, 2), \"narrative_purpose_pct\": pct(purpose), \"rendering_medium_pct\": pct(medium),\n            \"presentation_graphics_pct\": round(100 * medium.get(\"presentation_graphic\", 0) / tot, 1)}\n\n\n# ------------------------------------------------------------------ plan (existing cast format + \"story\" shots)\nTONE_TABLE = {\"curious\": {\"speed\": 0.98, \"pitch\": 0.0, \"gain_db\": 0.0, \"pause_after\": 0.45},\n              \"explainer\": {\"speed\": 1.0, \"pitch\": 0.0, \"gain_db\": 0.0, \"pause_after\": 0.4},\n              \"warm\": {\"speed\": 0.97, \"pitch\": 0.0, \"gain_db\": 0.0, \"pause_after\": 0.5},\n              \"happy\": {\"speed\": 1.0, \"pitch\": 0.0, \"gain_db\": 0.0, \"pause_after\": 0.4},\n              \"friendly\": {\"speed\": 1.0, \"pitch\": 0.0, \"gain_db\": 0.0, \"pause_after\": 0.35},\n              \"awe\": {\"speed\": 0.96, \"pitch\": 0.0, \"gain_db\": 0.0, \"pause_after\": 0.6},\n              \"serious\": {\"speed\": 0.97, \"pitch\": 0.0, \"gain_db\": 0.0, \"pause_after\": 0.5}}\nMOOD_FOR = {\"observation\": \"curious\", \"question\": \"curious\", \"first_answer\": \"calm\", \"demonstration\": \"busy\",\n            \"misconception\": \"curious\", \"correction\": \"calm\", \"reveal\": \"joy\", \"real_world\": \"calm\", \"closing\": \"joy\"}\nMOODS = {\"curious\": {\"pause_after\": 0.5, \"grade\": \"neutral\", \"move_amount\": 0.04,\n                     \"music\": {\"base\": 0.5, \"curious\": 0.7, \"tension\": 0.1, \"shimmer\": 0.3}},\n         \"calm\": {\"pause_after\": 0.45, \"grade\": \"neutral\", \"move_amount\": 0.04,\n                  \"music\": {\"base\": 0.4, \"curious\": 0.8, \"tension\": 0.0, \"shimmer\": 0.2}},\n         \"busy\": {\"pause_after\": 0.4, \"grade\": \"neutral\", \"move_amount\": 0.05,\n                  \"music\": {\"base\": 0.5, \"curious\": 0.6, \"tension\": 0.2, \"shimmer\": 0.1}},\n         \"joy\": {\"pause_after\": 0.45, \"grade\": \"neutral\", \"move_amount\": 0.05,\n                 \"music\": {\"base\": 0.3, \"curious\": 1.0, \"tension\": 0.0, \"shimmer\": 0.4}}}\n\n\ndef build_plan(story: Story, bible, project, media=None, allow_ai=False, ai_budget=0, plan_version=1, extra=None):\n    \"\"\"Story \u2192 cast-format plan the notebook runs (validated again there). Facts become sourced claims; each\n    factual line cites them; the visuals carry their intent and medium for the QC statistics.\"\"\"\n    visuals, decisions = direct(story, bible, media, allow_ai, ai_budget)\n    claim_of = {f.id: f\"c_{f.id.lower()}\" for f in story.facts}\n    chars = charbible.cast_characters(bible, story.language)\n    shots = []\n    for s in story.scenes:\n        lines, cited = [], []\n        for t in s.turns:\n            ln = {\"speaker\": t.speaker, \"tone\": t.tone, \"text\": t.text, \"kind\": t.kind}\n            tts = t.tts or charbible.speak_text(t.text, bible.characters[t.speaker].voices[story.language]) \\\n                if story.language in bible.characters[t.speaker].voices else t.tts\n            if tts:\n                ln[\"tts\"] = tts                          # pronunciation profile (e.g. 3D \u2192 \u0c24\u0c4d\u0c30\u0c40\u0c21\u0c40); subtitles keep text\n            if t.intent:\n                ln[\"intent\"] = t.intent\n            if t.fact_ids:\n                ln[\"claims\"] = [claim_of[f] for f in t.fact_ids if f in claim_of]\n                cited += ln[\"claims\"]\n            if t.expr or t.gesture:\n                ln[\"act\"] = {k: v for k, v in ((\"expr\", t.expr), (\"gesture\", t.gesture)) if v}\n            lines.append(ln)\n        vis = visuals[s.id]\n        _apply_acting(vis, s, bible)\n        shots.append({\"shot_id\": s.id, \"recipe\": \"story\", \"beat\": ENGINE_BEAT[s.beat], \"story_beat\": s.beat,\n                      \"mood\": MOOD_FOR[s.beat], \"location\": s.world, \"cast\": list(bible.characters),\n                      \"claims\": list(dict.fromkeys(cited)), \"lines\": lines,\n                      **({\"transition\": s.transition} if s.transition else {}),\n                      **({\"sfx\": s.sfx} if s.sfx else {}),\n                      \"story\": {\"world\": s.world, \"visuals\": vis}})\n    plan = {\n        \"plan_version\": plan_version, \"project\": project, \"title\": story.title, \"language\": story.language,\n        \"story_mode\": \"two_friends_cinematic\", \"visual_style\": \"stylized_2d\", \"license_strict\": True,\n        \"factual_dialogue\": True, \"logline\": story.logline,\n        \"notebook\": {\"emoji\": \"\ud83c\udfac\", \"name\": \"Two-Friend Explainer\", \"length\": \"about 1\u20131.5 min\"},\n        \"style\": {\"suffix\": \"stylized 2D animation, flat colours, two original cartoon friends\", \"negative\":\n                  \"text, watermark, photorealistic, deformed, blurry\"},\n        \"look\": {\"bg_top\": \"#1B2440\", \"bg_bottom\": \"#28406A\"},\n        \"characters\": chars, \"bible\": bible.model_dump(), \"tones\": dict(TONE_TABLE), \"moods\": dict(MOODS),\n        \"claims\": [{\"id\": claim_of[f.id], \"text\": f.text, \"sources\": [f.source_url],\n                    **({\"source_revision\": f.revid} if f.revid else {})} for f in story.facts],\n        \"vocabulary\": list(dict.fromkeys([c.name for c in bible.characters.values()] + story.vocabulary)),\n        \"entities\": {}, \"glossary\": {}, \"travel\": {}, \"shots\": shots, \"director\": decisions,\n        \"music\": {\"cues\": _music_cues(story)},\n        \"metadata\": _metadata(story, bible),\n    }\n    if extra:\n        plan.update(extra)\n    return plan\n\n\nMUSIC_FOR = {\"observation\": \"Investigations\", \"demonstration\": \"Easy Lemon\", \"reveal\": \"Carefree\"}\n\n\ndef _music_cues(story):\n    \"\"\"One licensed track per chapter: curious at the start, light during the demonstration, bright at the reveal.\"\"\"\n    cues, last = [], None\n    for s in story.scenes:\n        tr = MUSIC_FOR.get(s.beat)\n        if tr and tr != last:\n            cues.append({\"from\": s.id, \"track\": tr}); last = tr\n    if not cues or cues[0][\"from\"] != story.scenes[0].id:\n        cues.insert(0, {\"from\": story.scenes[0].id, \"track\": \"Investigations\"})\n    return cues\n\n\ndef _metadata(story, bible):\n    names = \" and \".join(c.name for c in bible.characters.values())\n    titles = sorted({f.title for f in story.facts if f.title})\n    disclaimer = (f\"Simplified explainer based on Wikipedia ({', '.join(titles)}). {names} are fictional, \"\n                  \"original animated characters; scenes marked 'Illustration' are re-enactments, not footage.\")\n    if story.language == \"te\":\n        tnames = \" \u0c2e\u0c30\u0c3f\u0c2f\u0c41 \".join(c.names.get(\"te\", c.name) for c in bible.characters.values())\n        disclaimer = (f\"\u0c35\u0c3f\u0c15\u0c40\u0c2a\u0c40\u0c21\u0c3f\u0c2f\u0c3e ({', '.join(titles)}) \u0c06\u0c27\u0c3e\u0c30\u0c02\u0c17\u0c3e \u0c38\u0c30\u0c33\u0c02\u0c17\u0c3e \u0c35\u0c3f\u0c35\u0c30\u0c3f\u0c02\u0c1a\u0c3f\u0c28 \u0c35\u0c40\u0c21\u0c3f\u0c2f\u0c4b. {tnames} \u0c15\u0c32\u0c4d\u0c2a\u0c3f\u0c24, \u0c38\u0c4a\u0c02\u0c24\u0c02\u0c17\u0c3e \"\n                      \"\u0c17\u0c40\u0c38\u0c3f\u0c28 \u0c2f\u0c3e\u0c28\u0c3f\u0c2e\u0c47\u0c37\u0c28\u0c4d \u0c2a\u0c3e\u0c24\u0c4d\u0c30\u0c32\u0c41; '\u0c26\u0c43\u0c37\u0c4d\u0c1f\u0c3e\u0c02\u0c24\u0c02' \u0c05\u0c28\u0c3f \u0c09\u0c28\u0c4d\u0c28 \u0c26\u0c43\u0c36\u0c4d\u0c2f\u0c3e\u0c32\u0c41 \u0c2a\u0c41\u0c28\u0c30\u0c4d\u0c28\u0c3f\u0c30\u0c4d\u0c2e\u0c3e\u0c23\u0c3e\u0c32\u0c41, \u0c28\u0c3f\u0c1c\u0c2e\u0c48\u0c28 \u0c2b\u0c41\u0c1f\u0c47\u0c1c\u0c4d \u0c15\u0c3e\u0c26\u0c41.\\n\" + disclaimer)\n    return {\"title\": story.title,\n            \"summary\": story.logline or f\"{names} work out: {story.topic}\",\n            \"disclaimer\": disclaimer,\n            \"category\": \"Education\", \"audience\": \"Not made for kids\",\n            \"tags\": list(dict.fromkeys([story.topic.lower().rstrip(\"?\"), \"science explained\", \"explainer\"]\n                                       + [v.lower() for v in story.vocabulary]))[:12]}\n\n\ndef _apply_acting(vis, scene, bible):\n    \"\"\"Turn-level acting (expression/gesture of the speaker) becomes timed 'changes' in the character visuals\n    that are on screen while that turn is spoken; the listener reacts with a nod at the start of each turn.\"\"\"\n    starts = [(v[\"start\"][\"line\"], v[\"start\"][\"at\"]) for v in vis] + [(len(scene.turns), 0.0)]\n    for i, v in enumerate(vis):\n        if v[\"recipe\"] != \"character\":\n            continue\n        a, b = starts[i], starts[i + 1]\n        span = max((b[0] + b[1]) - (a[0] + a[1]), 1e-6)\n        spec = v[\"character\"]\n        ch = list(spec.get(\"changes\", []))\n        for k, t in enumerate(scene.turns):\n            if not a[0] <= k < b[0] + (1 if b[1] > 0 else 0):\n                continue\n            tt = max(0.0, (k - (a[0] + a[1])) / span)\n            if tt > 1:\n                continue\n            listener = next((c for c in bible.characters if c != t.speaker), None)\n            st = {t.speaker: {k2: v2 for k2, v2 in ((\"expr\", t.expr), (\"gesture\", t.gesture or\n                                                                         (\"explain\" if t.kind in FACTUAL_KINDS else\n                                                                          \"point\" if t.kind == \"instruction\" else \"idle\")))\n                              if v2}}\n            if listener:\n                st[listener] = {\"gesture\": \"nod\" if k > 0 else \"idle\", \"expr\": \"listening\"}\n            ch.append({\"t\": round(min(tt, 1.0), 4), \"chars\": st})\n        spec[\"changes\"] = sorted(ch, key=lambda c: c[\"t\"])\n\n\ndef story_text(story: Story, bible):\n    out = [story.title.upper(), \"\"]\n    for s in story.scenes:\n        out.append(f\"[{s.id} \u00b7 {s.beat} \u00b7 {s.world}]\")\n        for t in s.turns:\n            name = bible.characters[t.speaker].name if t.speaker in bible.characters else t.speaker\n            out.append(f\"  {name.upper()}: {t.text}\" + (f\"   ({', '.join(t.fact_ids)})\" if t.fact_ids else \"\"))\n        out.append(\"\")\n    return \"\\n\".join(out)\n\n\n# ================================================================== v2: premise \u2192 outline \u2192 dialogue with memory\n# \u2192 critique \u2192 targeted revision \u2192 fact-aware repair. Each stage is small, structured (Pydantic) and cacheable; the\n# LLM writes data only. The critic's deterministic checks run without any model (CPU tests); the LLM critic is\n# optional and can only ADD weaknesses, never approve a story by itself.\nFORMATS = (\"everyday_mystery\", \"experiment\", \"misconception\", \"what_if\", \"journey\", \"historical_discovery\",\n           \"comparison\", \"observation_reveal\")\n\n\nclass Premise(BaseModel):\n    format: Literal[FORMATS]\n    opening: str                              # the concrete situation the viewer sees first\n    question: str                             # the central curiosity\n    relevance: str                            # why a viewer cares\n    motivation: str                           # why Arjun cares, in this moment\n    discovery: str                            # what they find out\n    misunderstanding: str = \"\"                # a plausible wrong idea to correct (optional)\n    key_fact: str                             # the most interesting VERIFIED fact (its id in fact_ids)\n    demo: str = \"\"                            # a physical demonstration opportunity\n    payoff: str                               # how the ending answers / reframes the opening\n    fact_ids: list[str] = Field(min_length=1)\n    risks: str = \"\"                           # what could become inaccurate\n\n\nclass Premises(BaseModel):\n    premises: list[Premise] = Field(min_length=1, max_length=3)\n\n\nclass OutlineBeat(BaseModel):\n    id: str = Field(pattern=r\"^b[0-9]{1,2}$\")\n    role: Literal[BEATS]\n    setting: str                              # a world of charanim.WORLDS\n    objective: str                            # what the characters want here\n    observation: str = \"\"\n    happens: str\n    changes: str                              # what is different afterwards (cause \u2192 effect)\n    learns: str = \"\"\n    open_question: str = \"\"                   # what pulls the viewer into the next beat\n    why_next: str = \"\"\n    fact_ids: list[str] = Field(default_factory=list)\n    demo: str = \"\"                            # demos.TEMPLATES name, or \"\"\n\n\nclass Outline(BaseModel):\n    title: str\n    beats: list[OutlineBeat] = Field(min_length=3, max_length=10)\n\n    @model_validator(mode=\"after\")\n    def shape(self):\n        if self.beats[-1].role != \"closing\":\n            raise ValueError(\"the last beat must be the closing\")\n        if len({b.id for b in self.beats}) != len(self.beats):\n            raise ValueError(\"beat ids must be unique\")\n        return self\n\n\nclass Weakness(BaseModel):\n    where: str                                # scene id or \"sc03_2\" (turn) or \"story\"\n    problem: str\n    evidence: str = \"\"\n    fix: str = \"\"\n    required: bool = True\n    rule: str = \"\"\n\n\nclass Critique(BaseModel):\n    weaknesses: list[Weakness] = Field(default_factory=list)\n\n\n# ------------------------------------------------------------------ Stage A: fact selection\ndef select_facts(facts, max_facts=16, max_words=45):   # cited Wikipedia sentences run 30\u201340 words\n    \"\"\"Facts a viewer can follow when spoken: short enough, no duplicates (first source wins).\"\"\"\n    out, seen = [], set()\n    for f in facts:\n        w = set(re.findall(r\"[a-z]{4,}\", f.text.lower()))\n        if len(f.text.split()) > max_words or any(len(w & s) / max(len(w | s), 1) > 0.7 for s in seen):\n            continue\n        out.append(f); seen.add(frozenset(w))\n        if len(out) >= max_facts:\n            break\n    return out\n\n\n# ------------------------------------------------------------------ Stage B/C: premises and their choice\nPREMISE_HINT = \"\"\"{\"premises\": [{\"format\": \"everyday_mystery|experiment|misconception|what_if|journey|historical_discovery|comparison|observation_reveal\",\n  \"opening\": \"a concrete moment the friends are in\", \"question\": \"the central question\", \"relevance\": \"why a viewer cares\",\n  \"motivation\": \"why Arjun cares right now\", \"discovery\": \"what they find out\", \"misunderstanding\": \"a plausible wrong idea (or empty)\",\n  \"key_fact\": \"the most surprising sourced fact, in plain words\", \"demo\": \"a physical demonstration they could do\",\n  \"payoff\": \"how the ending answers the opening\", \"fact_ids\": [\"F1\"], \"risks\": \"what could become inaccurate\"}]}\"\"\"\n\n\ndef premises_prompt(topic, facts, n, used_formats=()):\n    lines = \"\\n\".join(f\"{f.id}: {f.text}\" for f in facts)\n    avoid = f\" Avoid these formats used recently: {', '.join(used_formats)}.\" if used_formats else \"\"\n    return (f\"Topic: {topic}\\nPropose {n} DIFFERENT story concepts (different formats) for a short video where two \"\n            f\"friends discover this together. Use only these facts; never invent danger or evidence.{avoid}\\n\"\n            f\"Facts:\\n{lines}\\n\\nReturn JSON: {PREMISE_HINT}\")\n\n\ndef score_premise(p: Premise, fact_ids, templates=()):\n    \"\"\"Deterministic premise score (no model judges its own work). Returns (score, reasons). Unsupported \u2192 -inf.\"\"\"\n    reasons, sc = [], 0.0\n    missing = [f for f in p.fact_ids if f not in fact_ids]\n    if missing:\n        return float(\"-inf\"), [f\"cites unknown facts {missing}\"]\n    concrete = len(re.findall(r\"[a-z]{4,}\", p.opening.lower()))\n    if 4 <= concrete <= 30:\n        sc += 1; reasons.append(\"concrete opening\")\n    if p.question.strip().endswith(\"?\"):\n        sc += 1; reasons.append(\"a real question\")\n    if p.demo:\n        sc += 1; reasons.append(\"has a demonstration\")\n        if any(t.replace(\"_\", \" \") in p.demo.lower() or t.split(\"_\")[0] in p.demo.lower() for t in templates):\n            sc += 1; reasons.append(\"demonstration can be drawn\")\n    if p.misunderstanding:\n        sc += 0.5; reasons.append(\"a misconception to correct\")\n    if len(p.fact_ids) >= 3:\n        sc += 1; reasons.append(f\"{len(p.fact_ids)} sourced facts\")\n    q = set(re.findall(r\"[a-z]{4,}\", p.question.lower())); pay = set(re.findall(r\"[a-z]{4,}\", p.payoff.lower()))\n    if q & pay:\n        sc += 1; reasons.append(\"payoff answers the question\")\n    if p.format in (\"experiment\", \"what_if\", \"observation_reveal\", \"comparison\"):\n        sc += 0.5; reasons.append(\"the curious friend acts (predicts / tests / compares)\")\n    if re.search(r\"\\b(today we|did you know|let me explain|welcome)\\b\", p.opening, re.I):\n        sc -= 2; reasons.append(\"formulaic opening\")\n    return sc, reasons\n\n\ndef choose_premise(premises, fact_ids, templates=()):\n    scored = [(score_premise(p, fact_ids, templates), i, p) for i, p in enumerate(premises)]\n    (best_sc, why), i, p = max(scored, key=lambda x: (x[0][0], -x[1]))\n    if best_sc == float(\"-inf\"):\n        raise ValueError(\"no premise is supported by the facts\")\n    return p, [{\"index\": j, \"format\": q.format, \"score\": s, \"reasons\": r} for (s, r), j, q in scored]\n\n\n# ------------------------------------------------------------------ Stage D: outline\nOUTLINE_HINT = \"\"\"{\"title\": \"max 8 words\", \"beats\": [{\"id\": \"b1\", \"role\": \"observation\", \"setting\": \"cinema_seats\",\n  \"objective\": \"...\", \"observation\": \"...\", \"happens\": \"...\", \"changes\": \"...\", \"learns\": \"...\",\n  \"open_question\": \"...\", \"why_next\": \"...\", \"fact_ids\": [\"F1\"], \"demo\": \"finger_test or empty\"}]}\"\"\"\n\n\ndef outline_prompt(premise: Premise, facts, target_s, worlds, templates):\n    lines = \"\\n\".join(f\"{f.id}: {f.text}\" for f in facts)\n    return (f\"Premise ({premise.format}): opening: {premise.opening} | question: {premise.question} | discovery: \"\n            f\"{premise.discovery} | misunderstanding: {premise.misunderstanding} | demo: {premise.demo} | payoff: \"\n            f\"{premise.payoff}\\nPlan the story in 4\u20138 beats for about {target_s} seconds. Roles: {', '.join(BEATS)}. \"\n            \"Each beat must CAUSE the next (an observation, prediction, test or misunderstanding leads on) \u2014 never \"\n            \"'another interesting fact'. The last beat is 'closing' and answers the opening question.\\n\"\n            f\"Settings: {', '.join(worlds)}. Demonstrations that can be drawn: {', '.join(templates)}.\\n\"\n            f\"Facts:\\n{lines}\\n\\nReturn JSON: {OUTLINE_HINT}\")\n\n\n# ------------------------------------------------------------------ story state (memory between beats)\nclass StoryState:\n    \"\"\"What has happened so far, for the next beat's prompt and for continuity checks.\"\"\"\n    def __init__(self, bible):\n        self.cur, self.exp = bible.by_role(\"curious\"), bible.by_role(\"expert\")\n        self.facts, self.open_q, self.misconceptions, self.corrected, self.demos = [], [], [], [], []\n        self.location, self.recent = None, []\n\n    def update(self, scene: \"Scene\"):\n        if self.location and scene.world != self.location:\n            self.recent.append(f\"(they move from {self.location} to {scene.world})\")\n        self.location = scene.world\n        for t in scene.turns:\n            for f in t.fact_ids:\n                if f not in self.facts:\n                    self.facts.append(f)\n            if t.kind == \"question\" and t.speaker == self.cur.id:\n                self.open_q.append(t.text)\n            elif t.speaker == self.exp.id and t.kind in FACTUAL_KINDS and self.open_q:\n                self.open_q.pop(0)                          # the oldest question got its answer\n            if t.kind == \"misconception\":\n                self.misconceptions.append(t.text)\n            if t.kind == \"correction\" and self.misconceptions:\n                self.corrected.append(self.misconceptions.pop(0))\n            self.recent.append(f\"{t.speaker}: {t.text}\")\n        for v in scene.visuals:\n            if v.template and v.template not in self.demos:\n                self.demos.append(v.template)\n        self.recent = self.recent[-5:]\n\n    def summary(self):\n        return (f\"Location: {self.location or 'not set'}. Facts already explained: {', '.join(self.facts) or 'none'} \"\n                f\"(don't repeat them without adding something). {self.cur.name} still wonders: \"\n                f\"{' | '.join(self.open_q) or 'nothing open'}. {self.cur.name} still believes (uncorrected): \"\n                f\"{' | '.join(self.misconceptions) or 'nothing'}. Demonstrations done: {', '.join(self.demos) or 'none'}. \"\n                f\"Last lines: {' / '.join(self.recent) or '(start)'}\")\n\n\n# ------------------------------------------------------------------ Stage E: dialogue, beat by beat with memory\nDIALOGUE_RULES = \"\"\"Write it to be SPOKEN by two friends, not read:\n- {cur} is active: he observes, predicts, guesses wrong, proposes a test, notices contradictions, helps with the\n  demonstration, remembers earlier points. Not just \"Wow!\", \"Really?\", \"Tell me more\", \"What happens next?\".\n- {exp} doesn't lecture: she asks him to predict, demonstrates, uses everyday analogies, corrects gently, builds on\n  what he said, admits uncertainty when the facts don't say.\n- Short natural sentences, contractions, a little humour when it fits. No \"Today we'll learn\", \"Great question\",\n  \"Did you know\", \"Let me explain step by step\", \"In conclusion\", \"Another interesting fact\".\n- Every line does something: advances the story, explains, asks something relevant, reveals character, fixes a\n  misunderstanding, builds curiosity or pays off. No filler.\n- Turn kinds: fact | correction (must cite fact_ids) | question | challenge | prediction | misconception |\n  hypothetical (start with \"Imagine\" or \"What if\") | analogy (worded as a comparison) | observation | instruction |\n  reaction | opinion.\"\"\"\n\n\ndef dialogue_prompt_v2(bible, premise, beat: OutlineBeat, facts, words, state: StoryState):\n    lines = \"\\n\".join(f\"{f.id}: {f.text}\" for f in facts)\n    a, b = bible.by_role(\"curious\"), bible.by_role(\"expert\")\n    return (DIALOGUE_RULES.format(cur=a.name, exp=b.name) +\n            f\"\\n\\nCentral question: {premise.question}\\nStory so far: {state.summary()}\\n\"\n            f\"This beat ({beat.role}, setting {beat.setting}): objective: {beat.objective}; observation: \"\n            f\"{beat.observation}; what happens: {beat.happens}; what changes: {beat.changes}; viewer learns: \"\n            f\"{beat.learns}; leaves open: {beat.open_question}.\\nAbout {words} spoken words in 2\u20135 turns. Speakers: \"\n            f\"\\\"{a.id}\\\" ({a.name}), \\\"{b.id}\\\" ({b.name}).\\nFacts:\\n{lines}\\n\\nReturn JSON: {{\\\"world\\\": \\\"{beat.setting}\\\", \"\n            \"\\\"turns\\\": [{\\\"speaker\\\": \\\"...\\\", \\\"text\\\": \\\"...\\\", \\\"kind\\\": \\\"...\\\", \\\"fact_ids\\\": [], \\\"tone\\\": \\\"friendly\\\"}]}\"\n            f\"\\ntones: {', '.join(TONES)}\")\n\n\nWORLDS = (\"living_room\", \"cinema_seats\", \"cinema_screen\", \"lobby\", \"studio\")\n\n\ndef stage_premise(topic, facts, llm, bible, n_premises=2, used_formats=(), ask_json=None, log=print):\n    import demos\n    if ask_json is None:\n        from planner import ask_json\n    ps = ask_json(llm, system_prompt(bible), premises_prompt(topic, facts, n_premises, used_formats), Premises, log=log)\n    premise, scores = choose_premise(ps.premises[:max(n_premises, 1)], {f.id for f in facts}, demos.TEMPLATES)\n    return {\"premises\": [p.model_dump() for p in ps.premises], \"scores\": scores, \"premise\": premise.model_dump()}\n\n\ndef stage_outline(premise, facts, llm, bible, target_s, ask_json=None, log=print):\n    import demos\n    if ask_json is None:\n        from planner import ask_json\n    rel = [f for f in facts if f.id in premise.fact_ids] + [f for f in facts if f.id not in premise.fact_ids][:4]\n    return ask_json(llm, system_prompt(bible), outline_prompt(premise, rel, target_s, WORLDS, demos.TEMPLATES),\n                    Outline, log=log).model_dump()\n\n\ndef stage_dialogue(topic, premise, outline, facts, llm, bible, target_s, language=\"en\", ask_json=None, log=print):\n    \"\"\"Stage E: beat by beat with the story state (memory) in every prompt. Returns a Story.\"\"\"\n    import demos\n    if ask_json is None:\n        from planner import ask_json\n    sysp = system_prompt(bible)\n    te_note = \"\"\n    if language == \"te\":                                  # natural spoken Telugu writer (same schema)\n        import telugu_dialogue as td\n        sysp = td.system_prompt(bible)\n        te_note = (\"\\nWrite every turn's \\\"text\\\" in natural spoken Telugu. Every fact/correction turn also gives \"\n                   \"\\\"source_text\\\": the same claim in plain English using only the cited facts. Format: \"\n                   + td.DIALOGUE_HINT)\n    rel = [f for f in facts if f.id in premise.fact_ids] + [f for f in facts if f.id not in premise.fact_ids][:4]\n    state, scenes = StoryState(bible), []\n    per = max(int(target_s * WORDS_PER_SECOND / len(outline.beats)), 14)\n    by_id = {f.id: f for f in facts}\n    for k, b in enumerate(outline.beats):\n        bf = [by_id[i] for i in b.fact_ids if i in by_id] or rel[:4]\n        d = ask_json(llm, sysp, dialogue_prompt_v2(bible, premise, b, bf, per, state) + te_note, SceneDialogue, log=log)\n        v = ask_json(llm, sysp, visuals_prompt(d.turns, demos.TEMPLATES), SceneVisuals, log=log)\n        vis = list(v.visuals)\n        if b.demo in demos.TEMPLATES and not any(x.template == b.demo for x in vis):     # the planned demo is shown\n            last = vis[-1]\n            nxt = (max(last.turn, len(d.turns) - 1), 0.5)\n            if nxt <= (last.turn, last.at):\n                nxt = (last.turn, min(last.at + 0.25, 1.0))\n            if nxt > (last.turn, last.at):\n                vis.append(Visual(id=f\"v{len(vis) + 1}\", intent=\"object_demo\", turn=nxt[0], at=nxt[1],\n                                  template=b.demo, description=b.happens))\n        sc = Scene(id=f\"sc{k + 1:02d}\", beat=b.role, world=d.world if d.world in WORLDS else b.setting\n                   if b.setting in WORLDS else \"studio\", turns=d.turns, visuals=vis,\n                   transition=\"time_skip\" if scenes and scenes[-1].world != (d.world if d.world in WORLDS else b.setting) else None)\n        scenes.append(sc); state.update(sc)\n    used = {i for s in scenes for t in s.turns for i in t.fact_ids}\n    return Story(title=outline.title, topic=topic, language=language, logline=premise.opening,\n                 facts=[StoryFact(id=f.id, text=f.text, source_url=f.source_url, title=f.title, revid=f.revid)\n                        for f in facts if f.id in used], scenes=scenes)\n\n\ndef make_story_v2(topic, sheet, llm, bible, target_s=75, language=\"en\", log=print, ask_json=None,\n                  n_premises=2, max_revisions=1, used_formats=()):\n    \"\"\"Stages A\u2013G. Returns (story, record) where record keeps every intermediate artifact (premises + scores, the\n    chosen premise, outline, critique before/after) for caching and the story benchmark.\"\"\"\n    if ask_json is None:\n        from planner import ask_json\n    facts = select_facts(sheet.facts)                                                     # A\n    pr = stage_premise(topic, facts, llm, bible, n_premises, used_formats, ask_json, log)  # B + C\n    premise = Premise.model_validate(pr[\"premise\"]); ps = Premises(premises=pr[\"premises\"]); scores = pr[\"scores\"]\n    log(f\"story: premise '{premise.format}' chosen: {premise.question} ({scores})\")\n    outline = Outline.model_validate(stage_outline(premise, facts, llm, bible, target_s, ask_json, log))   # D\n    st = stage_dialogue(topic, premise, outline, facts, llm, bible, target_s, language, ask_json, log)      # E\n    before = critique(st, bible, llm=llm, ask_json=ask_json, log=log)                     # F (rules + LLM critic)\n    st, revised = revise(st, before, llm, bible, sheet, ask_json=ask_json, log=log, max_revisions=max_revisions)  # G\n    after = critique(st, bible)\n    record = {\"facts\": [f.id for f in facts], \"premises\": [p.model_dump() for p in ps.premises], \"scores\": scores,\n              \"premise\": premise.model_dump(), \"outline\": outline.model_dump(),\n              \"critique_before\": [w.model_dump() for w in before], \"revised_scenes\": revised,\n              \"critique_after\": [w.model_dump() for w in after]}\n    return st, record\n\n\n# ------------------------------------------------------------------ Stage F: critic\nFORMULAS = [r\"\\btoday (we|we'll|we will|i'll)\\b\", r\"\\bwelcome (back|to)\\b\", r\"\\bgreat question\\b\", r\"\\bdid you know\\b\",\n            r\"\\blet me explain\\b\", r\"\\bstep by step\\b\", r\"\\bin conclusion\\b\", r\"\\bhello (everyone|guys)\\b\"]\nARTIFICIAL = [r\"\\banother interesting fact\\b\", r\"\\bnow let me tell you\\b\", r\"\\bmoving on\\b\", r\"\\blet'?s move on\\b\",\n              r\"\\bnext,? let'?s\\b\", r\"\\bhere'?s another\\b\"]\nEMPTY_PROMPTS = re.compile(r\"^\\s*(wow|really|that'?s (amazing|cool|interesting)|tell me more|what happens next|\"\n                           r\"go on|and then)\\W*$\", re.I)\n\n\ndef _w(text):\n    return set(re.findall(r\"[a-z]{4,}\", text.lower()))\n\n\ndef critique(st: Story, bible, llm=None, ask_json=None, log=print):\n    \"\"\"Concrete weaknesses with evidence (deterministic). With an llm, its critique is ADDED (validated JSON).\"\"\"\n    out = []\n    cur, exp = bible.by_role(\"curious\").id, bible.by_role(\"expert\").id\n    turns = [(s, k, t) for s in st.scenes for k, t in enumerate(s.turns)]\n    tid = lambda s, k: f\"{s.id}_{k + 1}\"\n    for s, k, t in turns:\n        for pat in FORMULAS:\n            if re.search(pat, t.text, re.I):\n                out.append(Weakness(where=tid(s, k), rule=\"C1\", problem=\"formulaic phrase\", evidence=t.text,\n                                    fix=\"say it the way a friend would\"))\n        for pat in ARTIFICIAL:\n            if re.search(pat, t.text, re.I):\n                out.append(Weakness(where=tid(s, k), rule=\"C5\", problem=\"artificial transition\", evidence=t.text,\n                                    fix=\"let the next point arise from something they see, predict or misunderstand\"))\n        if t.speaker == cur and EMPTY_PROMPTS.match(t.text):\n            out.append(Weakness(where=tid(s, k), rule=\"C3\", problem=\"Arjun only prompts Meera to continue\",\n                                evidence=t.text, fix=\"give him an observation, a guess or a prediction\"))\n    run, start = 0, None                                   # C2: question \u2192 answer \u2192 question \u2192 answer ... loop\n    for i in range(1, len(turns)):\n        (s0, k0, a), (s1, k1, b) = turns[i - 1], turns[i]\n        qa = a.speaker == cur and a.kind == \"question\" and b.speaker == exp and b.kind in FACTUAL_KINDS\n        if qa:\n            run += 1; start = start or tid(s0, k0)\n            if run == 3:\n                out.append(Weakness(where=start, rule=\"C2\", problem=\"Q&A loop: three question\u2192fact exchanges in a row\",\n                                    evidence=\"; \".join(x.text[:40] for _, _, x in turns[i - 5:i + 1]),\n                                    fix=\"replace one exchange with a prediction, a test or a misunderstanding\"))\n        elif not (a.speaker == exp and b.speaker == cur):\n            run, start = 0, None\n    kinds = [t.kind for _, _, t in turns if t.speaker == cur]\n    active = {\"observation\", \"prediction\", \"challenge\", \"hypothetical\", \"misconception\", \"opinion\"}\n    if kinds and sum(x == \"question\" for x in kinds) / len(kinds) > 0.7:\n        out.append(Weakness(where=\"story\", rule=\"C3\", problem=f\"Arjun mostly asks ({kinds.count('question')}/{len(kinds)})\",\n                            evidence=\", \".join(kinds), fix=\"let him observe, predict, challenge or help demonstrate\"))\n    if not set(kinds) & active:\n        out.append(Weakness(where=\"story\", rule=\"C3\", problem=\"Arjun never observes, predicts or challenges\",\n                            evidence=\", \".join(kinds), fix=\"add one prediction or observation of his own\"))\n    words = {c: sum(len(t.text.split()) for _, _, t in turns if t.speaker == c) for c in (cur, exp)}\n    if words[exp] / max(sum(words.values()), 1) > 0.72:\n        out.append(Weakness(where=\"story\", rule=\"C4\", problem=\"Meera lectures\", required=False,\n                            evidence=f\"{words[exp] / max(sum(words.values()), 1):.0%} of the words\",\n                            fix=\"hand some discoveries to Arjun\"))\n    first = st.scenes[0].turns[0]\n    if first.kind in FACTUAL_KINDS or (first.kind == \"question\" and first.speaker == cur and\n                                        len(_w(first.text) & _w(st.topic)) >= 2 and not st.scenes[0].world):\n        out.append(Weakness(where=f\"{st.scenes[0].id}_1\", rule=\"C6\", problem=\"opens with the textbook question/fact\",\n                            evidence=first.text, fix=\"open on a concrete moment or observation\"))\n    opening = \" \".join(t.text for s in st.scenes[:2] for t in s.turns)\n    closing = \" \".join(t.text for t in st.scenes[-1].turns) + \" \" + \" \".join(t.text for t in st.scenes[-2].turns)\n    if not (_w(opening) & _w(closing)):\n        out.append(Weakness(where=st.scenes[-1].id, rule=\"C7\", problem=\"the ending doesn't pay off the opening\",\n                            evidence=closing[:120], fix=\"return to the opening moment with the new understanding\"))\n    if not any(v.template or v.intent in (\"object_demo\", \"animated_illustration\") for s in st.scenes for v in s.visuals) \\\n            and not any(t.kind == \"instruction\" for _, _, t in turns):\n        out.append(Weakness(where=\"story\", rule=\"C8\", problem=\"no demonstration at all\", evidence=\"\",\n                            fix=\"let them test or show the idea\"))\n    grams = {}\n    for s, k, t in turns:                                  # C9 repeated wording across lines\n        ws = re.findall(r\"[a-z']+\", t.text.lower())\n        for g in {\" \".join(ws[i:i + 4]) for i in range(len(ws) - 3)}:\n            grams.setdefault(g, []).append(tid(s, k))\n    for g, where in grams.items():\n        if len(set(where)) > 1 and len(_w(g)) >= 2:\n            out.append(Weakness(where=where[-1], rule=\"C9\", problem=\"repeated wording\", evidence=f\"'{g}' in {where}\",\n                                fix=\"vary it or cut the repeat\", required=False))\n    for i, (s, k, t) in enumerate(turns):                  # C10: a question nobody answers\n        if t.kind == \"question\" and t.speaker == cur:\n            nxt = [x for _, _, x in turns[i + 1:i + 3]]\n            if not any(x.speaker == exp and x.kind not in (\"question\",) for x in nxt):\n                out.append(Weakness(where=tid(s, k), rule=\"C10\", problem=\"question left without an answer\",\n                                    evidence=t.text, fix=\"answer it, or turn it into a test\"))\n    starts = [t.text.split()[0].lower().strip(\",.!?\") for _, _, t in turns if t.speaker == cur and t.kind == \"question\"]\n    for w0 in set(starts):\n        if starts.count(w0) >= 3:\n            out.append(Weakness(where=\"story\", rule=\"C11\", problem=f\"Arjun's questions keep starting with '{w0}'\",\n                                evidence=str(starts), fix=\"vary his rhythm\", required=False))\n    qs = [i for i, (_, _, t) in enumerate(turns) if t.kind == \"question\" and t.speaker == cur]\n    instant = [i for i in qs if i + 1 < len(turns) and turns[i + 1][2].speaker == exp\n               and turns[i + 1][2].kind in FACTUAL_KINDS]\n    if len(qs) >= 2 and len(instant) / len(qs) >= 0.5 and not {\"prediction\", \"hypothetical\"} & set(kinds):\n        out.append(Weakness(where=\"story\", rule=\"C12\", required=False,\n                            problem=f\"{len(instant)}/{len(qs)} questions are answered at once and Arjun never predicts\",\n                            evidence=\"; \".join(turns[i][2].text[:40] for i in instant),\n                            fix=\"let him guess or predict first, then test it\"))\n    unprompted = [tid(s, k) for i, (s, k, t) in enumerate(turns) if t.speaker == exp and t.kind in FACTUAL_KINDS\n                  and (i == 0 or turns[i - 1][2].speaker == exp)]\n    if unprompted:\n        out.append(Weakness(where=unprompted[0].split(\"_\")[0], rule=\"C13\", required=len(unprompted) >= 2,\n                            problem=\"an explanation arrives unprompted (nothing Arjun said or saw leads to it)\",\n                            evidence=\", \".join(unprompted), fix=\"let an observation, guess or test trigger it\"))\n    last = st.scenes[-1].turns[-1].text\n    if re.search(r\"^\\W*(\\w+\\s+){0,3}\\w+,\\s+(\\w+\\s+){0,3}\\w+,\\s+(and\\s+)?\", last) and \"?\" not in last:\n        out.append(Weakness(where=f\"{st.scenes[-1].id}_{len(st.scenes[-1].turns)}\", rule=\"C14\", required=False,\n                            problem=\"the ending is a recap list\", evidence=last,\n                            fix=\"end on a moment (a callback to the opening) instead of a summary\"))\n    if llm is not None and ask_json is not None:\n        try:\n            more = ask_json(llm, \"You are a demanding script editor. Find real weaknesses; never praise.\",\n                            critic_prompt(st, bible), Critique, log=log)\n            ids = {s.id for s in st.scenes} | {tid(s, k) for s, k, _ in turns} | {\"story\"}\n            out += [w.model_copy(update={\"rule\": w.rule or \"LLM\"}) for w in more.weaknesses if w.where in ids]\n        except Exception as e:\n            log(f\"story: LLM critic unavailable ({e}); deterministic critique only\")\n    return out\n\n\ndef critic_prompt(st: Story, bible):\n    script = \"\\n\".join(f\"{s.id}_{k + 1} [{s.beat}] {bible.characters[t.speaker].name}: {t.text}\"\n                       for s in st.scenes for k, t in enumerate(s.turns))\n    return (\"Judge this two-friend educational script. Is the opening interesting and concrete? Does each scene cause \"\n            \"the next? Does Arjun contribute ideas? Does Meera explain naturally? Repetition, artificial transitions, \"\n            \"formal wording, missing demonstrations, unprepared discoveries, an unsatisfying ending, generic Q&A?\\n\"\n            f\"{script}\\n\\nReturn JSON: {{\\\"weaknesses\\\": [{{\\\"where\\\": \\\"sc02 or sc02_1 or story\\\", \\\"problem\\\": \\\"...\\\", \"\n            \"\\\"evidence\\\": \\\"quote\\\", \\\"fix\\\": \\\"...\\\", \\\"required\\\": true}]}. An empty list only if there is truly nothing.\")\n\n\n# ------------------------------------------------------------------ Stage G: targeted revision\ndef revise(st: Story, weaknesses, llm, bible, sheet, ask_json=None, log=print, max_revisions=1):\n    \"\"\"Rewrite ONLY the scenes with required weaknesses (bounded); keep a rewrite only if it has fewer required\n    weaknesses and still passes the dialogue rules. Returns (story, revised scene ids).\"\"\"\n    if llm is None or max_revisions <= 0:\n        return st, []\n    if ask_json is None:\n        from planner import ask_json\n    by_scene = {}\n    for w in weaknesses:\n        if w.required:\n            sid = w.where.split(\"_\")[0] if w.where != \"story\" else None\n            by_scene.setdefault(sid, []).append(w)\n    targets = [s for s in st.scenes if s.id in by_scene]\n    if None in by_scene and not targets:                   # story-level problem: rewrite the weakest middle scene\n        targets = st.scenes[1:2]\n        by_scene[targets[0].id] = by_scene[None]\n    facts = {f.id: f for f in getattr(sheet, \"facts\", [])}\n    revised = []\n    for _ in range(max_revisions):\n        for s in targets:\n            ws = by_scene.get(s.id, [])\n            fl = \"\\n\".join(f\"{f.id}: {f.text}\" for i, f in facts.items() if i in {x for t in s.turns for x in t.fact_ids}) \\\n                or \"\\n\".join(f\"{f.id}: {f.text}\" for f in list(facts.values())[:6])\n            problems = \"\\n\".join(f\"- {w.problem}: {w.evidence} \u2192 {w.fix}\" for w in ws)\n            prompt = (DIALOGUE_RULES.format(cur=bible.by_role(\"curious\").name, exp=bible.by_role(\"expert\").name)\n                      + f\"\\n\\nRewrite this scene to fix ONLY these problems:\\n{problems}\\nScene ({s.beat}, {s.world}):\\n\"\n                      + \"\\n\".join(f\"{t.speaker} [{t.kind}] {t.text}\" for t in s.turns)\n                      + f\"\\nFacts:\\n{fl}\\n\\nReturn JSON: {{\\\"world\\\": \\\"{s.world}\\\", \\\"turns\\\": [...]}} (same format, \"\n                        f\"{len(s.turns)}\u00b11 turns).\")\n            try:\n                d = ask_json(llm, system_prompt(bible), prompt, SceneDialogue, log=log)\n            except Exception as e:\n                log(f\"story: revision of {s.id} failed ({e}); kept the original\"); continue\n            new_vis = [v.model_copy(update={\"turn\": min(v.turn, len(d.turns) - 1)}) for v in s.visuals]\n            dedup, last = [], None\n            for v in new_vis:\n                if last is None or (v.turn, v.at) > last:\n                    dedup.append(v); last = (v.turn, v.at)\n            cand_scene = s.model_copy(update={\"turns\": d.turns, \"visuals\": dedup})\n            cand = st.model_copy(update={\"scenes\": [cand_scene if x.id == s.id else x for x in st.scenes]})\n            try:\n                cand = Story.model_validate(cand.model_dump())\n            except Exception as e:\n                log(f\"story: revised {s.id} is invalid ({e}); kept the original\"); continue\n            req = lambda story_: sum(w.required for w in critique(story_, bible) if w.where.startswith(s.id))\n            errs = [i for i in check_story(cand, bible) if i[\"level\"] == \"error\"]\n            if req(cand) < req(st) and not errs:\n                st = cand; revised.append(s.id)\n                log(f\"story: revised {s.id} ({len(ws)} problems)\")\n            else:\n                log(f\"story: revision of {s.id} not better; kept the original\")\n    return st, revised\n\n\n# ------------------------------------------------------------------ fact-aware repair (replaces plain dropping)\nREWRITE_HINT = '{\"text\": \"...\", \"fact_ids\": [\"F1\"]}'\n\n\nclass TurnRewrite(BaseModel):\n    text: str\n    fact_ids: list[str] = Field(default_factory=list)\n\n\ndef repair_grounding(st: Story, nli, bible, llm=None, ask_json=None, log=print, max_rewrites=1):\n    \"\"\"1 find unsupported claims \u2192 2 bounded rewrite from the cited facts \u2192 3 re-check \u2192 4 still unsupported: remove\n    \u2192 5\u20136 remove what depended on it (the question it answered, a reaction to it) \u2192 7 re-check the story.\n    Returns (story, results, report).\"\"\"\n    res = ground(st, nli, topic=st.topic)\n    bad = {k for k, r in res.items() if r[\"status\"] != \"PASS\"}\n    report = {\"rewritten\": [], \"removed\": [], \"dependents_removed\": []}\n    if not bad:\n        return st, res, report\n    by_id = {f.id: f for f in st.facts}\n    scenes = {s.id: s for s in st.scenes}\n    if llm is not None:\n        if ask_json is None:\n            from planner import ask_json\n        for key in sorted(bad):\n            sid, k = key.rsplit(\"_\", 1); k = int(k) - 1\n            t = scenes[sid].turns[k]\n            ev = \"\\n\".join(f\"{i}: {by_id[i].text}\" for i in t.fact_ids if i in by_id)\n            for _ in range(max_rewrites):\n                try:\n                    rw = ask_json(llm, system_prompt(bible),\n                                  f\"Rewrite this line so it says ONLY what these facts support, in the same \"\n                                  f\"conversational voice ({bible.characters[t.speaker].name}, {t.kind}).\\nLine: {t.text}\\n\"\n                                  f\"Facts:\\n{ev}\\nReturn JSON: {REWRITE_HINT}\", TurnRewrite, log=log)\n                    nt = t.model_copy(update={\"text\": rw.text, \"fact_ids\": [i for i in rw.fact_ids if i in by_id] or t.fact_ids})\n                    Turn.model_validate(nt.model_dump())\n                except Exception as e:\n                    log(f\"story: rewrite of {key} failed ({e})\"); continue\n                import factcheck\n                r2 = factcheck.check_sentences([{\"id\": key, \"text\": nt.source_text or nt.text,\n                                                 \"evidence\": [by_id[i].text for i in nt.fact_ids if i in by_id],\n                                                 \"question\": nt.kind in (\"question\", \"challenge\"),\n                                                 \"analogy\": nt.kind in (\"analogy\", \"hypothetical\")}], nli, topic=st.topic)\n                if r2[key][\"status\"] == \"PASS\":\n                    turns = list(scenes[sid].turns); turns[k] = nt\n                    scenes[sid] = scenes[sid].model_copy(update={\"turns\": turns})\n                    report[\"rewritten\"].append({\"turn\": key, \"from\": t.text, \"to\": nt.text}); bad.discard(key)\n                    break\n    drop = set(bad)\n    for key in sorted(bad):                                # dependents of what is removed\n        sid, k = key.rsplit(\"_\", 1); k = int(k) - 1\n        turns = scenes[sid].turns\n        t = turns[k]\n        if k > 0 and turns[k - 1].kind == \"question\" and turns[k - 1].speaker != t.speaker:\n            drop.add(f\"{sid}_{k}\"); report[\"dependents_removed\"].append({\"turn\": f\"{sid}_{k}\", \"why\": f\"its answer {key} was removed\"})\n        if k + 1 < len(turns) and turns[k + 1].kind in (\"reaction\", \"observation\", \"opinion\") \\\n                and turns[k + 1].speaker != t.speaker:\n            drop.add(f\"{sid}_{k + 2}\"); report[\"dependents_removed\"].append({\"turn\": f\"{sid}_{k + 2}\", \"why\": f\"it reacts to {key}\"})\n        report[\"removed\"].append({\"turn\": key, \"text\": t.text, \"why\": res[key][\"why\"]})\n    st = Story.model_validate(st.model_copy(update={\"scenes\": [scenes[s.id] for s in st.scenes]}).model_dump())\n    if drop:\n        st = _remove_turns(st, drop, log)\n    for k in report[\"removed\"]:\n        log(f\"story: removed unsupported turn {k['turn']}: {k['text'][:80]!r} ({k['why']})\")\n    for k in report[\"dependents_removed\"]:\n        log(f\"story: removed {k['turn']} ({k['why']})\")\n    return st, ground(st, nli, topic=st.topic), report\n\n\ndef _remove_turns(st: Story, keys, log=print):\n    scenes = []\n    for s in st.scenes:\n        keep = [k for k in range(len(s.turns)) if f\"{s.id}_{k + 1}\" not in keys]\n        if not keep:\n            log(f\"story: scene {s.id} dropped (nothing left)\"); continue\n        new_of = {k: next((i for i, kk in enumerate(keep) if kk >= k), len(keep) - 1) for k in range(len(s.turns))}\n        vis = []\n        for v in s.visuals:\n            nv = v.model_copy(update={\"turn\": new_of[v.turn], \"at\": v.at if v.turn in keep else 0.0})\n            if not vis or (nv.turn, nv.at) > (vis[-1].turn, vis[-1].at):\n                vis.append(nv)\n        vis[0] = vis[0].model_copy(update={\"turn\": 0, \"at\": 0.0})\n        scenes.append(s.model_copy(update={\"turns\": [s.turns[k] for k in keep], \"visuals\": vis}))\n    return Story.model_validate(st.model_copy(update={\"scenes\": scenes}).model_dump())\n\n\n# ------------------------------------------------------------------ story benchmark helpers (old vs new script)\ndef story_metrics(st: Story, bible):\n    \"\"\"Structural numbers for the story benchmark (they inform, they don't decide: human ratings decide).\"\"\"\n    cur, exp = bible.by_role(\"curious\").id, bible.by_role(\"expert\").id\n    turns = [t for s in st.scenes for t in s.turns]\n    ck = critique(st, bible)\n    kinds = [t.kind for t in turns if t.speaker == cur]\n    words = {c: sum(len(t.text.split()) for t in turns if t.speaker == c) for c in (cur, exp)}\n    return {\"turns\": len(turns), \"words\": sum(words.values()),\n            \"curious_kinds\": {k: kinds.count(k) for k in sorted(set(kinds))},\n            \"curious_question_share\": round(kinds.count(\"question\") / max(len(kinds), 1), 2),\n            \"expert_word_share\": round(words[exp] / max(sum(words.values()), 1), 2),\n            \"demonstrations\": sum(1 for s in st.scenes for v in s.visuals if v.template),\n            \"required_weaknesses\": sum(w.required for w in ck), \"weaknesses\": [f\"{w.rule} {w.where}: {w.problem}\" for w in ck]}\n", "storybook.py": "\"\"\"Storybook character system: versioned character assets + a 2D cutout rig renderer (CPU, Pillow).\n\nAsset library (one folder per character version; never rewritten once it exists):\n  <root>/characters/<id>/<version>/manifest.json   identity, status (provisional | approved), provenance, palette,\n                                                   rig, poses, expressions, visemes, and every layer's file + sha256\n  <root>/characters/<id>/<version>/layers/*.png    cropped RGBA parts (offset recorded in the manifest)\n  <root>/characters/<id>/<version>/previews/*.png  contact sheets for the human approval step\n\nA video plan pins {id: {\"version\", \"manifest_sha256\"}} (plan[\"character_assets\"]); rendering refuses a library whose\nmanifest hash differs, so new artwork can never silently change an old video. A missing approved version is an\nerror, never a fallback to simple shapes. Provisional versions (storybook_art.py) can be rebuilt from code.\n\nRig (front view): root(hips) \u2192 torso \u2192 {neck, head, upper arms \u2192 forearms \u2192 hands}; the head carries the face,\nhair and glasses as one group. Joints have limits; poses are joint angles; transitions ease over ~0.45 s.\nFace: expression \u2192 eyes / brows / resting mouth; blinking on a per-character schedule; gaze moves the irises\n(clipped to the eye whites); `turn` (-1..1) shifts the features for a three-quarter look toward the other person.\nMouth: a viseme per frame from that speaker's own audio (silence \u2192 closed): only the speaker's mouth moves.\n\"\"\"\nfrom __future__ import annotations\n\nimport datetime, hashlib, json, math, os\nfrom collections import OrderedDict\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nRIG_VERSION = 1\nDEFAULT_RES = 1.6                     # stored pixels per design unit\nBLEND_S = 0.45                        # gesture transition time\n\n# ------------------------------------------------------------------------------------------- rig + library\nJOINT_LIMITS = {\"abduct\": (-35, 170), \"flex\": (-110, 160), \"wrist\": (-45, 45), \"head\": (-14, 14), \"lean\": (-8, 8)}\n\n# per side: a = lift outward (deg), b = elbow flex toward the body centre (deg), w = wrist, hand = hand drawing\nPOSES = {\n    \"idle\":       {\"L\": dict(a=7, b=10, w=0, hand=\"open\"), \"R\": dict(a=7, b=10, w=0, hand=\"open\")},\n    \"listen\":     {\"L\": dict(a=6, b=14, w=0, hand=\"open\"), \"R\": dict(a=6, b=14, w=0, hand=\"open\")},\n    \"explain\":    {\"L\": dict(a=7, b=10, w=0, hand=\"open\"), \"R\": dict(a=16, b=-62, w=-20, hand=\"open\"), \"lean\": 1.5},\n    \"open_hands\": {\"L\": dict(a=26, b=-48, w=-15, hand=\"open\"), \"R\": dict(a=26, b=-48, w=-15, hand=\"open\")},\n    \"point\":      {\"L\": dict(a=7, b=10, w=0, hand=\"open\"), \"R\": dict(a=82, b=-8, w=-10, hand=\"point\"), \"lean\": 2},\n    \"think\":      {\"L\": dict(a=-8, b=100, w=0, hand=\"hold\"), \"R\": dict(a=-26, b=160, w=30, hand=\"hold\"), \"head\": -5},\n    \"wave\":       {\"L\": dict(a=7, b=10, w=0, hand=\"open\"), \"R\": dict(a=100, b=-80, w=0, hand=\"wave\")},\n    \"surprise\":   {\"L\": dict(a=38, b=-35, w=-10, hand=\"open\"), \"R\": dict(a=38, b=-35, w=-10, hand=\"open\"), \"lean\": -3, \"head\": -3},\n    \"celebrate\":  {\"L\": dict(a=150, b=-15, w=0, hand=\"open\"), \"R\": dict(a=150, b=-15, w=0, hand=\"open\")},\n    \"duck\":       {\"L\": dict(a=72, b=-100, w=0, hand=\"open\"), \"R\": dict(a=72, b=-100, w=0, hand=\"open\"), \"drop\": 0.07, \"head\": 4},\n    \"hold\":       {\"L\": dict(a=7, b=10, w=0, hand=\"open\"), \"R\": dict(a=6, b=95, w=0, hand=\"hold\")},\n    \"show\":       {\"L\": dict(a=7, b=10, w=0, hand=\"open\"), \"R\": dict(a=20, b=135, w=0, hand=\"hold\")},\n    \"nod\":        {\"L\": dict(a=6, b=14, w=0, hand=\"open\"), \"R\": dict(a=6, b=14, w=0, hand=\"open\")},\n}\nGESTURE_ALIAS = {\"look_object\": \"listen\", \"react\": \"surprise\", \"step\": \"idle\"}\n\n# expression \u2192 eyes, brows, resting mouth, iris size\nEXPRESSIONS = {\n    \"neutral\":   (\"open\", \"neutral\", \"rest\", \"\"), \"listening\": (\"open\", \"neutral\", \"rest\", \"\"),\n    \"smile\":     (\"smile\", \"neutral\", \"smile\", \"\"), \"happy\": (\"smile\", \"raised\", \"smile\", \"\"),\n    \"curious\":   (\"open\", \"curious\", \"rest\", \"\"), \"confused\": (\"open\", \"furrow\", \"wavy\", \"\"),\n    \"surprised\": (\"wide\", \"high\", \"o_big\", \"small\"), \"excited\": (\"wide\", \"raised\", \"smile_talk\", \"\"),\n    \"thinking\":  (\"half\", \"furrow\", \"pursed\", \"\"), \"concerned\": (\"open\", \"sad\", \"frown\", \"\"),\n    \"laughing\":  (\"happy\", \"raised\", \"laugh\", \"\"), \"wink\": (\"wink\", \"raised\", \"smile\", \"\"),\n}\nSMILING = {\"smile\", \"happy\", \"excited\", \"laughing\", \"wink\"}\nVISEMES = (\"rest\", \"closed\", \"small\", \"wide\", \"round\", \"teeth\", \"smile_talk\")\n\nLAYER_NAMES = None                    # filled from storybook_art on build\n\n\ndef _sha(b):\n    return hashlib.sha256(b).hexdigest()\n\n\ndef manifest_sha(manifest_path):\n    return _sha(Path(manifest_path).read_bytes())\n\n\ndef build_character(root, cid, version=\"v1\", res=DEFAULT_RES, log=print):\n    \"\"\"Draw the provisional layers once and write the version folder. Never overwrites an existing version.\"\"\"\n    import storybook_art as art\n    vdir = Path(root) / \"characters\" / cid / version\n    if (vdir / \"manifest.json\").exists():\n        raise FileExistsError(f\"{vdir} already exists: asset versions are immutable (make a new version)\")\n    tmp = vdir.with_name(version + \".partial\")             # write to a temp folder, rename when complete\n    (tmp / \"layers\").mkdir(parents=True, exist_ok=True)\n    layers = {}\n    for name, im in art.layers(cid, res=res).items():\n        bb = im.getbbox()\n        if bb is None:\n            continue\n        crop = im.crop(bb)\n        f = tmp / \"layers\" / f\"{name}.png\"\n        crop.save(f, optimize=True)\n        layers[name] = {\"file\": f\"layers/{name}.png\", \"offset\": [bb[0] / res, bb[1] / res], \"size\": list(crop.size),\n                        \"sha256\": _sha(f.read_bytes())}\n    d = art.DESIGNS[cid]\n    man = {\n        \"character\": cid, \"version\": version, \"status\": \"provisional\", \"rig_version\": RIG_VERSION,\n        \"art_version\": art.ART_VERSION, \"res\": res, \"canvas\": list(art.CANVAS),\n        \"provenance\": {\"method\": \"procedural vector drawing (colab/engine/lib/storybook_art.py)\", \"model\": None,\n                       \"license\": \"project-original (code-drawn)\", \"created\": datetime.datetime.now().isoformat(timespec=\"seconds\"),\n                       \"approved_by\": None, \"approved_at\": None,\n                       \"note\": \"PROVISIONAL engineering art: replace with approved Character Studio artwork\"},\n        \"palette\": {k: v for k, v in d.items() if isinstance(v, str) and v.startswith(\"#\")},\n        \"identity\": {\"hair_style\": d[\"hair_style\"], \"accessory\": d[\"accessory\"]},\n        \"rig\": {\"joints\": {\"shoulder\": {s: list(art.SHOULDER[s]) for s in \"LR\"}, \"elbow\": {s: list(art.ELBOW[s]) for s in \"LR\"},\n                           \"wrist\": {s: list(art.WRIST[s]) for s in \"LR\"}, \"neck\": [500, 712], \"hips\": [500, 1180]},\n                \"limits\": JOINT_LIMITS, \"layer_order\": [\"hair_back\", \"legs\", \"neck\", \"torso\", \"upper_arm_L\", \"upper_arm_R\", \"head_group\",\n                                                         \"forearm_L\", \"hand_L\", \"forearm_R\", \"hand_R\", \"prop\"],\n                \"views\": [\"front\", \"three_quarter (feature shift)\", \"back\"]},\n        \"poses\": POSES, \"expressions\": {k: list(v) for k, v in EXPRESSIONS.items()}, \"visemes\": list(VISEMES),\n        \"layers\": layers,\n    }\n    (tmp / \"manifest.json\").write_text(json.dumps(man, indent=1))\n    if vdir.exists():\n        raise FileExistsError(str(vdir))\n    tmp.rename(vdir)\n    log(f\"character {cid} {version}: {len(layers)} layers written to {vdir}\")\n    return vdir\n\n\ndef verify(vdir):\n    \"\"\"Every layer present and unchanged (sha256) \u2192 [] or a list of problems.\"\"\"\n    vdir = Path(vdir)\n    try:\n        man = json.loads((vdir / \"manifest.json\").read_text())\n    except Exception as e:\n        return [f\"manifest unreadable: {e!r}\"]\n    bad = []\n    for name, L in man[\"layers\"].items():\n        f = vdir / L[\"file\"]\n        if not f.exists():\n            bad.append(f\"missing layer {name}\")\n        elif _sha(f.read_bytes()) != L[\"sha256\"]:\n            bad.append(f\"layer {name} changed since the manifest was written\")\n    return bad\n\n\ndef content_fingerprint(man):\n    \"\"\"sha256 of the artwork only (every layer hash + geometry), NOT of status/approval fields: approving a version\n    changes its manifest hash but never this fingerprint.\"\"\"\n    if man.get(\"asset_type\") == \"atlas_v1\":\n        import storybook_atlas\n        return storybook_atlas.content_sha(man)\n    body = {\"layers\": {k: v[\"sha256\"] for k, v in sorted(man[\"layers\"].items())}, \"rig\": man.get(\"rig\")}\n    return hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest()\n\n\ndef approval_blockers(vdir):\n    \"\"\"Why this version cannot be approved yet ([] = it can). Assets complete and unchanged, previews made,\n    model licence allowed in strict mode; an atlas must also cover blinking, every mouth shape and every pose.\"\"\"\n    vdir = Path(vdir)\n    out = list(verify(vdir))\n    if out:\n        return out\n    man = json.loads((vdir / \"manifest.json\").read_text())\n    if man.get(\"provenance\", {}).get(\"prototype_only\"):\n        out.append(\"prototype asset substitutions cannot be approved for production\")\n    if man.get(\"asset_type\") == \"atlas_v1\":\n        import storybook_atlas\n        cov = storybook_atlas.coverage(man)\n        out += [f\"pose '{p}' is not mapped\" for p, v in cov[\"poses_mapped\"].items() if v is None]\n        out += [f\"mouth shape '{v}' missing\" for v, ok in cov[\"visemes\"].items() if not ok]\n        out += [\"no blink (eyes-closed) patch\"] if not cov[\"blink\"] else []\n        previews = sorted((vdir / \"previews\").glob(\"*\")) if (vdir / \"previews\").exists() else []\n        if not previews:\n            out.append(\"no previews in previews/ (render them with colab/storybook_previews.py --atlas)\")\n        model = (man.get(\"provenance\") or {}).get(\"model\")\n        if model:\n            import registry\n            ok, why = registry.model_ok(model, strict=True)\n            if not ok:\n                out.append(f\"licence: {why}\")\n    return out\n\n\ndef approve(vdir, who, note=\"\"):\n    \"\"\"Human approval: records who/when in the manifest of a NEW version folder copy is not needed \u2014 approval is\n    the one allowed manifest edit (status provisional \u2192 approved). Layers can never change afterwards.\"\"\"\n    vdir = Path(vdir)\n    man = json.loads((vdir / \"manifest.json\").read_text())\n    if not who or not str(who).strip():\n        raise ValueError(\"approval needs the name of the person who looked at the previews\")\n    blockers = approval_blockers(vdir)\n    if blockers:\n        raise ValueError(f\"cannot approve: {blockers}\")\n    fp = content_fingerprint(man)\n    man[\"status\"] = \"approved\"\n    man[\"content_sha256\"] = fp\n    man[\"provenance\"].update(approved_by=who, approved_at=datetime.datetime.now().isoformat(timespec=\"seconds\"),\n                             approval_note=note)\n    (vdir / \"manifest.json\").write_text(json.dumps(man, indent=1))\n    return manifest_sha(vdir / \"manifest.json\")\n\n\nclass Character:\n    \"\"\"One character version loaded for rendering (sprites cached per output scale).\"\"\"\n\n    def __init__(self, vdir):\n        self.dir = Path(vdir)\n        self.man = json.loads((self.dir / \"manifest.json\").read_text())\n        self.id, self.version = self.man[\"character\"], self.man[\"version\"]\n        self.res = self.man[\"res\"]\n        self._img, self._scaled = {}, OrderedDict()\n        j = self.man[\"rig\"][\"joints\"]\n        self.sh, self.el, self.wr = j[\"shoulder\"], j[\"elbow\"], j[\"wrist\"]\n        self.neck, self.hips = j[\"neck\"], j[\"hips\"]\n\n    def has(self, name):\n        return name in self.man[\"layers\"]\n\n    def img(self, name):\n        if name not in self._img:\n            self._img[name] = Image.open(self.dir / self.man[\"layers\"][name][\"file\"]).convert(\"RGBA\")\n        return self._img[name]\n\n    def sprite(self, name, s):\n        \"\"\"(image at output scale s px/unit, offset in design units).\"\"\"\n        key = (name, round(s, 3))\n        if key in self._scaled:\n            self._scaled.move_to_end(key)\n            return self._scaled[key]\n        L = self.man[\"layers\"][name]\n        im = self.img(name)\n        f = s / self.res\n        w, h = max(1, int(round(im.width * f))), max(1, int(round(im.height * f)))\n        out = (im.resize((w, h), Image.LANCZOS), tuple(L[\"offset\"]))\n        self._scaled[key] = out\n        if len(self._scaled) > 400:\n            self._scaled.popitem(last=False)\n        return out\n\n\n_LIB = {}\n\n\ndef library_root(plan=None):\n    \"\"\"Where character/environment versions live: STORYBOOK_ASSETS (env), else the repo's colab/assets.\"\"\"\n    return Path(os.environ.get(\"STORYBOOK_ASSETS\") or Path(__file__).resolve().parents[2] / \"assets\")\n\n\ndef load(cid, version, root=None, expect_sha=None, expect_content=None):\n    root = Path(root or library_root())\n    key = (str(root.resolve()), cid, version, expect_sha, expect_content)\n    vdir = root / \"characters\" / cid / version\n    if not (vdir / \"manifest.json\").exists():\n        raise FileNotFoundError(f\"character asset {cid} {version} not found in {root} (build or download it; \"\n                                \"the renderer never substitutes simple shapes)\")\n    if expect_sha and manifest_sha(vdir / \"manifest.json\") != expect_sha:\n        raise ValueError(f\"character asset {cid} {version}: manifest differs from the one this plan was made with\")\n    man = json.loads((vdir / \"manifest.json\").read_text())\n    if expect_content and content_fingerprint(man) != expect_content:\n        raise ValueError(f\"character asset {cid} {version}: artwork differs from the one this plan was made with\")\n    if man.get(\"character\") != cid or man.get(\"version\") != version:\n        raise ValueError(\"character identity/version differs from its directory\")\n    def stat_signature(path):\n        st = path.stat()\n        return (st.st_size, st.st_mtime_ns, st.st_ctime_ns, st.st_ino)\n    signature = (manifest_sha(vdir / \"manifest.json\"),\n                 tuple((v[\"file\"], stat_signature(vdir / v[\"file\"])) for v in man[\"layers\"].values()))\n    if key in _LIB and getattr(_LIB[key], \"_verified_signature\", None) == signature:\n        return _LIB[key]\n    probs = verify(vdir)\n    if probs:\n        raise ValueError(f\"character asset {cid} {version} is damaged: {probs[:3]}\")\n    if man.get(\"asset_type\") == \"atlas_v1\":\n        import storybook_atlas\n        _LIB[key] = storybook_atlas.AtlasCharacter(vdir)\n    else:\n        _LIB[key] = Character(vdir)\n    _LIB[key]._verified_signature = signature\n    return _LIB[key]\n\n\ndef install_pack(data, root=None, log=print):\n    \"\"\"Unpack an asset pack (tar.gz bytes, or the same as base64 text) into the library \u2014 only folders that are\n    not there yet (an existing version is never overwritten); written to a staging folder, then moved into place.\"\"\"\n    if not data or (isinstance(data, str) and data.startswith(\"__\")):\n        return []\n    import base64, io, shutil, tarfile\n    raw = base64.b64decode(data) if isinstance(data, str) else data\n    root = Path(root or library_root())\n    stage = root / \".incoming\"\n    shutil.rmtree(stage, ignore_errors=True)\n    installed = []\n    with tarfile.open(fileobj=io.BytesIO(raw), mode=\"r:gz\") as tf:\n        for m in tf.getmembers():\n            top = Path(*Path(m.name).parts[:3])\n            if not (root / top / \"manifest.json\").exists():\n                tf.extract(m, stage, filter=\"data\")\n    for vdir in sorted(stage.glob(\"*/*/*\")):\n        dst = root / vdir.relative_to(stage)\n        if not (dst / \"manifest.json\").exists():\n            dst.parent.mkdir(parents=True, exist_ok=True)\n            shutil.move(str(vdir), str(dst))\n            installed.append(str(dst.relative_to(root)))\n            log(f\"storybook asset {dst.relative_to(root)} installed from the notebook\")\n    shutil.rmtree(stage, ignore_errors=True)\n    return installed\n\n\ndef _build_one(args):\n    root, cid, version = args\n    return str(build_character(root, cid, version, log=lambda *a: None))\n\n\ndef prepare_assets(plan, pack=None, root=None, log=print):\n    \"\"\"Make every asset version the plan pins available in the library (Drive), in this order:\n      1. already in the library \u2192 used as is (verified when loaded);\n      2. the plan's asset pack file next to the library (pack = {\"name\", \"sha256\"}; you upload it once) \u2192 installed\n         if its sha256 matches \u2014 byte-identical to the art the plan was checked against;\n      3. PROVISIONAL code-drawn versions \u2192 drawn here from the same code (once; characters in parallel; then cached);\n      4. an APPROVED version that is missing \u2192 error (approved art comes from the Character Studio on Drive).\n    Returns {asset: how} for the run log / QC.\"\"\"\n    import hashlib, time\n    import envfactory\n    if plan.get(\"cinematic_characters\"):\n        import cinematic_workflow as CW\n        CW.validate_plan(plan, root or library_root())\n    root = Path(root or library_root())\n    root.mkdir(parents=True, exist_ok=True)\n    chars = {c: a for c, a in (plan.get(\"character_assets\") or {}).items()}\n    envs = dict(plan.get(\"environments\") or {})\n    have = lambda kind, i, v: (root / kind / i / v / \"manifest.json\").exists()\n    report = {}\n    missing = [(\"characters\", c, a[\"version\"]) for c, a in chars.items() if not have(\"characters\", c, a[\"version\"])] + \\\n              [(\"environments\", e, v) for e, v in envs.items() if not have(\"environments\", e, v)]\n    if missing and pack and pack.get(\"name\"):\n        f = root / pack[\"name\"]\n        if f.exists():\n            data = f.read_bytes()\n            if hashlib.sha256(data).hexdigest() == pack.get(\"sha256\"):\n                for p in install_pack(data, root, log):\n                    report[p] = \"installed from the uploaded pack (byte-identical)\"\n            else:\n                log(f\"asset pack {f} does not match this plan (sha256): ignored\")\n    todo = []\n    for c, a in chars.items():\n        if have(\"characters\", c, a[\"version\"]):\n            report.setdefault(f\"characters/{c}/{a['version']}\", \"already on Drive\")\n        elif a.get(\"status\") == \"approved\":\n            raise RuntimeError(f\"approved character art {c} {a['version']} is not in {root}: copy it from the Character \"\n                               \"Studio output (videos never fall back to other art)\")\n        else:\n            todo.append((str(root), c, a[\"version\"]))\n    if todo:\n        t0 = time.time()\n        log(f\"drawing the provisional characters {[t[1] for t in todo]} once (about 6 min on Colab's CPU; then cached \"\n            f\"on Drive). Faster: upload {pack.get('name') if pack else 'the asset pack'} to {root}\")\n        from concurrent.futures import ProcessPoolExecutor\n        with ProcessPoolExecutor(max_workers=min(2, len(todo))) as ex:\n            for (r_, c, v), _ in zip(todo, ex.map(_build_one, todo)):\n                report[f\"characters/{c}/{v}\"] = f\"drawn from code ({time.time() - t0:.0f}s)\"\n    for e, v in envs.items():\n        if not have(\"environments\", e, v):\n            envfactory.build_environment(root, e, v, log=lambda *a: None)\n            report[f\"environments/{e}/{v}\"] = \"drawn from code\"\n        else:\n            report.setdefault(f\"environments/{e}/{v}\", \"already on Drive\")\n    for c, a in chars.items():                       # same bytes as the plan was checked with? (information)\n        m = root / \"characters\" / c / a[\"version\"] / \"manifest.json\"\n        if a.get(\"manifest_sha256\"):\n            report[f\"characters/{c}/{a['version']}\"] += \"; identical to the checked art\" if manifest_sha(m) == \\\n                a[\"manifest_sha256\"] else \"; redrawn (same code and version, not byte-identical)\"\n    return report\n\n\ndef ensure(cid, version, root=None, log=print):\n    \"\"\"Load a version; build it first if it is a missing PROVISIONAL (code-drawn) version.\"\"\"\n    root = Path(root or library_root())\n    vdir = root / \"characters\" / cid / version\n    if not (vdir / \"manifest.json\").exists():\n        build_character(root, cid, version, log=log)\n    return load(cid, version, root)\n\n\n# ------------------------------------------------------------------------------------------- geometry\ndef _rot(p, c, deg):\n    a = math.radians(deg)\n    x, y = p[0] - c[0], p[1] - c[1]\n    return (c[0] + x * math.cos(a) - y * math.sin(a), c[1] + x * math.sin(a) + y * math.cos(a))\n\n\ndef clamp(v, lim):\n    return max(lim[0], min(lim[1], v))\n\n\ndef ease(u):\n    u = max(0.0, min(1.0, u))\n    return u * u * (3 - 2 * u)\n\n\ndef blend_pose(p0, p1, u):\n    u = ease(u)\n    out = {}\n    for side in (\"L\", \"R\"):\n        a, b = p0[side], p1[side]\n        out[side] = {k: a[k] + (b[k] - a[k]) * u for k in (\"a\", \"b\", \"w\")}\n        out[side][\"hand\"] = b[\"hand\"] if u >= 0.5 else a[\"hand\"]\n    for k in (\"lean\", \"head\", \"drop\"):\n        out[k] = p0.get(k, 0) + (p1.get(k, 0) - p0.get(k, 0)) * u\n    return out\n\n\ndef pose_of(name):\n    return POSES.get(GESTURE_ALIAS.get(name, name), POSES[\"idle\"])\n\n\n# ------------------------------------------------------------------------------------------- drawing one character\nclass Painter:\n    \"\"\"Paints a Character at output scale s (px per design unit) onto a frame.\"\"\"\n\n    def __init__(self, ch, s):\n        self.ch, self.s = ch, s\n        self._rot = OrderedDict()\n        self._head = OrderedDict()\n\n    def _put(self, frame, im, x, y):\n        frame.alpha_composite(im, (int(round(x)), int(round(y))))\n\n    def _rotated(self, key, im, deg, pivot_px):\n        \"\"\"im rotated by deg (clockwise on screen) about pivot_px; returns (image, pivot position in the result).\"\"\"\n        if abs(deg) < 0.05:\n            return im, pivot_px\n        k = (key, round(deg * 4) / 4)\n        if k in self._rot:\n            self._rot.move_to_end(k)\n            r, p = self._rot[k]\n            return r, p\n        W, H = im.size\n        cx, cy = pivot_px\n        R = int(math.ceil(max(math.hypot(cx - x, cy - y) for x in (0, W) for y in (0, H)))) + 2\n        pad = Image.new(\"RGBA\", (2 * R, 2 * R), (0, 0, 0, 0))\n        pad.alpha_composite(im, (int(R - cx), int(R - cy)))\n        r = pad.rotate(-k[1], resample=Image.BICUBIC, center=(R, R))\n        bb = r.getbbox() or (0, 0, 1, 1)\n        r = r.crop(bb)\n        p = (R - bb[0], R - bb[1])\n        self._rot[k] = (r, p)\n        if len(self._rot) > 600:\n            self._rot.popitem(last=False)\n        return r, p\n\n    def part(self, frame, name, origin, M, deg, pivot_u):\n        \"\"\"Draw a layer whose design-unit pivot `pivot_u` lands on output pixel M(pivot_u), rotated by deg.\"\"\"\n        im, off = self.ch.sprite(name, self.s)\n        piv_px = ((pivot_u[0] - off[0]) * self.s, (pivot_u[1] - off[1]) * self.s)\n        r, p = self._rotated((name, round(self.s, 3)), im, deg, piv_px)\n        tx, ty = M(pivot_u)\n        self._put(frame, r, origin[0] + tx - p[0], origin[1] + ty - p[1])\n\n    def head_group(self, st, layer_set):\n        \"\"\"Face + hair + glasses composed into one sprite (cached per state), rotated as a whole later.\"\"\"\n        key = (round(self.s, 3), layer_set, st[\"eyes\"], st[\"brows\"], st[\"mouth\"], st[\"gaze\"], st[\"turn\"], st[\"glasses\"],\n               st[\"iris\"])\n        if key in self._head:\n            self._head.move_to_end(key)\n            return self._head[key]\n        ch, s = self.ch, self.s\n        x0, y0, x1, y1 = 180, 120, 840, 1180 if layer_set == \"back\" else 760   # design-unit box of the group\n        canvas = Image.new(\"RGBA\", (int((x1 - x0) * s), int((y1 - y0) * s)), (0, 0, 0, 0))\n\n        def put(name, dx=0.0, dy=0.0):\n            if not ch.has(name):\n                return\n            im, off = ch.sprite(name, s)\n            canvas.alpha_composite(im, (int(round((off[0] + dx - x0) * s)), int(round((off[1] + dy - y0) * s))))\n\n        if layer_set == \"back\":\n            put(\"hair_back\")\n        else:\n            fx = st[\"turn\"] * 26                       # three-quarter look: features slide toward the turn\n            put(\"head\"); put(\"cheeks\", fx * 0.8); put(\"nose\", fx * 1.1)\n            for side in (\"L\", \"R\"):\n                ev = st[\"eyes\"]\n                if ev == \"wink\":\n                    ev = \"open\" if side == \"L\" else \"happy\"\n                if ev in (\"open\", \"half\", \"wide\", \"smile\"):\n                    put(f\"eye_{side}_{ev}\", fx)\n                    iris_name = f\"iris_{side}\" + (\"_small\" if st[\"iris\"] == \"small\" else \"\")\n                    im, off = ch.sprite(iris_name, s)\n                    mim, moff = ch.sprite(f\"eyemask_{side}_{ev}\", s)\n                    gx, gy = st[\"gaze\"]\n                    tmp = Image.new(\"RGBA\", mim.size, (0, 0, 0, 0))\n                    tmp.alpha_composite(im, (int(round((off[0] + gx - moff[0]) * s)), int(round((off[1] + gy - moff[1]) * s))))\n                    a = np.minimum(np.asarray(tmp.getchannel(\"A\")), np.asarray(mim.getchannel(\"A\")))\n                    tmp.putalpha(Image.fromarray(a))\n                    canvas.alpha_composite(tmp, (int(round((moff[0] + fx - x0) * s)), int(round((moff[1] - y0) * s))))\n                    put(f\"lid_{side}_{ev}\", fx)        # lids and lashes over the iris\n                else:\n                    put(f\"eye_{side}_{ev}\", fx)\n            put(f\"brows_{st['brows']}\", fx)\n            put(f\"mouth_{st['mouth']}\", fx * 0.9)\n            put(\"hair_front\", st[\"turn\"] * 8)\n            if st[\"glasses\"]:\n                put(\"glasses\", fx)\n        out = (canvas, (x0, y0))\n        self._head[key] = out\n        if len(self._head) > 300:\n            self._head.popitem(last=False)\n        return out\n\n    def draw(self, frame, origin, pose, face, view=\"front\", hold=None, legs=True):\n        \"\"\"origin: output pixel of design point (0, 0) of this character (before the rig moves the root).\"\"\"\n        ch, s = self.ch, self.s\n        if view == \"back\":\n            im, off = ch.sprite(\"back\", s)\n            self._put(frame, im, origin[0] + off[0] * s, origin[1] + off[1] * s)\n            return\n        drop = pose.get(\"drop\", 0) * 1800\n        lean = clamp(pose.get(\"lean\", 0), JOINT_LIMITS[\"lean\"])\n        hips = ch.hips\n        o = (origin[0], origin[1] + drop * s)\n\n        def torsoM(p):                                  # torso space \u2192 output (lean about the hips)\n            q = _rot(p, hips, lean)\n            return (q[0] * s, q[1] * s)\n        # head transform: about the neck, plus lean, plus breathing/head offsets\n        hdeg = clamp(pose.get(\"head\", 0) + face.get(\"head_tilt\", 0), JOINT_LIMITS[\"head\"])\n        head_dy = face.get(\"head_dy\", 0)\n\n        def headM(p):\n            q = _rot(_rot(p, ch.neck, hdeg), hips, lean)\n            return (q[0] * s, (q[1] + head_dy) * s)\n        # arm chains\n        chains = {}\n        for side in (\"L\", \"R\"):\n            sgn = 1 if side == \"L\" else -1                # L lifts clockwise on screen, R counter-clockwise\n            a = clamp(pose[side][\"a\"], JOINT_LIMITS[\"abduct\"])\n            b = clamp(pose[side][\"b\"], JOINT_LIMITS[\"flex\"])\n            w = clamp(pose[side][\"w\"], JOINT_LIMITS[\"wrist\"])\n            up_deg = sgn * a + lean\n            fo_deg = up_deg - sgn * b\n            ha_deg = fo_deg - sgn * w\n            shoulder = torsoM(ch.sh[side])\n            el_u = _rot(ch.el[side], ch.sh[side], sgn * a)\n            el_u = _rot(el_u, hips, lean)\n            elbow = (el_u[0] * s, el_u[1] * s)\n            wr_u = _rot(_rot(ch.wr[side], ch.el[side], -sgn * b), ch.sh[side], sgn * a)\n            wr_u = _rot(wr_u, hips, lean)\n            wrist = (wr_u[0] * s, wr_u[1] * s)\n            chains[side] = (up_deg, fo_deg, ha_deg, shoulder, elbow, wrist, pose[side][\"hand\"])\n\n        def arm(side, part):\n            up_deg, fo_deg, ha_deg, shoulder, elbow, wrist, hand = chains[side]\n            if part == \"upper\":\n                self.part(frame, f\"upper_arm_{side}\", o, lambda p: shoulder, up_deg, ch.sh[side])\n            elif part == \"fore\":\n                self.part(frame, f\"forearm_{side}\", o, lambda p: elbow, fo_deg, ch.el[side])\n            else:\n                self.part(frame, f\"hand_{side}_{hand}\", o, lambda p: wrist, ha_deg, ch.wr[side])\n                if side == \"R\" and hold:\n                    self.part(frame, f\"prop_{hold}\", o, lambda p: wrist, ha_deg if hold != \"popcorn\" else 0, ch.wr[side])\n\n        hb, hb_org = self.head_group(face, \"back\")\n        piv = ((ch.neck[0] - hb_org[0]) * s, (ch.neck[1] - hb_org[1]) * s)\n        r, p = self._rotated((\"hb\", face[\"turn\"], round(s, 3)), hb, hdeg + lean, piv)\n        nk = headM(ch.neck)\n        self._put(frame, r, o[0] + nk[0] - p[0], o[1] + nk[1] - p[1])\n        if legs:                                        # legs stay planted (a duck bends the upper body down)\n            im, off = ch.sprite(\"legs\", s)\n            self._put(frame, im, origin[0] + off[0] * s, origin[1] + off[1] * s)\n        self.part(frame, \"neck\", o, torsoM, lean, ch.hips)\n        self.part(frame, \"torso\", o, torsoM, lean, ch.hips)\n        arm(\"L\", \"upper\"); arm(\"R\", \"upper\")\n        hf, hf_org = self.head_group(face, \"front\")\n        piv = ((ch.neck[0] - hf_org[0]) * s, (ch.neck[1] - hf_org[1]) * s)\n        key = (\"hf\", face[\"eyes\"], face[\"brows\"], face[\"mouth\"], face[\"gaze\"], face[\"turn\"], face[\"glasses\"], face[\"iris\"], round(s, 3))\n        r, p = self._rotated(key, hf, hdeg + lean, piv)\n        self._put(frame, r, o[0] + nk[0] - p[0], o[1] + nk[1] - p[1])\n        for side in (\"L\", \"R\"):\n            arm(side, \"fore\")\n            arm(side, \"hand\")\n\n\n# ------------------------------------------------------------------------------------------- timing helpers\ndef blink_closed(cid, t_abs):\n    rng = np.random.default_rng(int(hashlib.sha1(cid.encode()).hexdigest()[:8], 16))\n    t = rng.uniform(0.6, 2.0)\n    while t < t_abs + 1:\n        if t <= t_abs < t + 0.12:\n            return True\n        t += rng.uniform(2.4, 5.0)\n    return False\n\n\ndef idle_motion(cid, t_abs, speaking):\n    \"\"\"Small, slow, non-repeating head motion (two incommensurate sines); a bit livelier while speaking.\"\"\"\n    ph = (int(hashlib.sha1(cid.encode()).hexdigest()[:4], 16) % 628) / 100\n    amp = 1.6 if speaking else 1.0\n    tilt = amp * (1.1 * math.sin(2 * math.pi * t_abs / 4.7 + ph) + 0.6 * math.sin(2 * math.pi * t_abs / 2.9 + 2 * ph))\n    breathe = 2.2 * math.sin(2 * math.pi * t_abs / 3.8 + ph)\n    return tilt, breathe\n\n\ndef viseme_from_energy(e, prev):\n    \"\"\"Fallback when no spectral viseme track exists: loudness \u2192 mouth opening only.\"\"\"\n    if e < 0.08:\n        return \"rest\"\n    if e < 0.3:\n        return \"small\"\n    return \"wide\" if prev != \"wide\" or e > 0.6 else \"small\"\n\n\ndef visemes_from_audio(wav, sr, fps, n_frames=None):\n    \"\"\"Mouth shape per video frame from a speaker's own audio (language independent):\n    silence \u2192 rest; quiet \u2192 small; open vowels by spectral balance: low (o/u) \u2192 round, high (i/e, s) \u2192 teeth,\n    otherwise wide; very short dips inside speech \u2192 closed (m/b/p-like). Held \u2265 2 frames (no flicker).\"\"\"\n    w = wav.mean(1) if wav.ndim > 1 else wav\n    hop = sr / fps\n    n = n_frames if n_frames is not None else int(len(w) / hop)\n    rms, cen = np.zeros(n), np.zeros(n)\n    win = np.hanning(int(hop * 2) or 2)\n    for i in range(n):\n        a = int(i * hop)\n        x = w[a:a + len(win)]\n        if len(x) < len(win):\n            x = np.pad(x, (0, len(win) - len(x)))\n        rms[i] = math.sqrt(float(np.mean(x ** 2)) + 1e-12)\n        sp = np.abs(np.fft.rfft(x * win))\n        f = np.fft.rfftfreq(len(x), 1 / sr)\n        band = (f > 80) & (f < 6000)\n        cen[i] = float((sp[band] * f[band]).sum() / (sp[band].sum() + 1e-9))\n    ref = np.percentile(rms[rms > 0], 90) if np.any(rms > 0) else 1.0\n    e = np.clip(rms / max(ref, 1e-6), 0, 1.5)\n    out, prev, held = [], \"rest\", 0\n    for i in range(n):\n        if e[i] < 0.07:\n            v = \"rest\"\n        elif e[i] < 0.22:\n            v = \"closed\" if 0 < i < n - 1 and e[i - 1] > 0.3 and e[i + 1] > 0.3 else \"small\"\n        elif cen[i] < 900:\n            v = \"round\"\n        elif cen[i] > 2600:\n            v = \"teeth\"\n        else:\n            v = \"wide\"\n        if v != prev and held < 2 and prev != \"rest\" and v != \"rest\":\n            v = prev\n        held = held + 1 if v == prev else 1\n        out.append(v)\n        prev = v\n    return out, e\n", "storybook_art.py": "\"\"\"Provisional \"storybook\" artwork for Arjun and Meera: layered, rig-ready parts drawn once by code.\n\nWhy code: the real artwork is meant to come from the Character Studio notebook (FLUX.2 [klein] 4B on a Colab T4,\ncut into layers, approved by a person). Until that exists, this module draws a PROVISIONAL set in the same layer\nformat, so the asset library, rig, lip sync, environments, QC and whole videos can be built and tested now. It is\nmarked \"provisional\" in every manifest and is never reported as approved artwork.\n\nStyle: soft storybook shading (gradient fills, blurred shadow and highlight passes clipped to each shape, soft\ncoloured outlines), big expressive eyes with two highlights, rounded shapes from cubic B\u00e9zier paths \u2014 no stick limbs,\nno circle-and-line bodies. Every part is drawn complete, including the area another part normally hides (shoulder\ncaps, the top of the forearm, the neck under the collar), so rotating a joint never opens a seam.\n\nDesign units: a 1000 x 1800 canvas (front view, full body), stored at `res` pixels per unit (default 1.6).\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\nfrom PIL import Image, ImageChops, ImageDraw, ImageFilter\n\nART_VERSION = \"storybook-procedural-1\"\nCANVAS = (1000, 1800)\nSS = 2                                    # supersampling while drawing\n\n\ndef hexrgb(h):\n    h = h.lstrip(\"#\")\n    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))\n\n\ndef mix(a, b, t):\n    return tuple(int(round(x + (y - x) * t)) for x, y in zip(a, b))\n\n\n# ------------------------------------------------------------------------------------------- the two designs\nDESIGNS = {\n    \"arjun\": {\n        \"skin\": \"#B97A4F\", \"skin_shadow\": \"#94593A\", \"blush\": \"#E3876A\", \"lip\": \"#8E4A38\",\n        \"hair\": \"#2A1B15\", \"hair_light\": \"#55392B\", \"iris\": \"#5B3620\", \"iris_light\": \"#9A6A40\", \"brow\": \"#2A1B15\",\n        \"top\": \"#1E8C8B\", \"top_shadow\": \"#126767\", \"top_light\": \"#4DB8B2\",        # teal hoodie\n        \"tee\": \"#F6C343\", \"tee_shadow\": \"#D9A126\",                               # warm yellow T-shirt\n        \"pants\": \"#2F3547\", \"pants_shadow\": \"#21263A\",\n        \"shoe\": \"#E8573C\", \"shoe_shadow\": \"#B83C27\", \"sole\": \"#F4F1EA\",\n        \"hair_style\": \"wavy_short_curl\", \"lashes\": 0, \"accessory\": None,\n    },\n    \"meera\": {\n        \"skin\": \"#A8683F\", \"skin_shadow\": \"#86502F\", \"blush\": \"#E07C67\", \"lip\": \"#9A4A45\",\n        \"hair\": \"#23150F\", \"hair_light\": \"#4E3124\", \"iris\": \"#4A2A18\", \"iris_light\": \"#8C5A34\", \"brow\": \"#23150F\",\n        \"top\": \"#F27B6B\", \"top_shadow\": \"#CC5A50\", \"top_light\": \"#FFA08F\",        # coral jacket\n        \"tee\": \"#B9A2E6\", \"tee_shadow\": \"#977DCC\",                               # lavender top\n        \"pants\": \"#33405E\", \"pants_shadow\": \"#232D47\",\n        \"shoe\": \"#A98BE0\", \"shoe_shadow\": \"#7E62BA\", \"sole\": \"#F4F1EA\",\n        \"hair_style\": \"wavy_half_up\", \"lashes\": 2, \"accessory\": \"star_clip\",\n    },\n}\n\n# where things are (design units); left/right = SCREEN left/right in the front view\nHEAD_C = (500, 470)\nEYE_C = {\"L\": (402, 505), \"R\": (598, 505)}\nEYE_R = (60, 70)\nMOUTH_C = (500, 648)\nSHOULDER = {\"L\": (360, 815), \"R\": (640, 815)}\nELBOW = {\"L\": (338, 1000), \"R\": (662, 1000)}\nWRIST = {\"L\": (334, 1170), \"R\": (666, 1170)}\n\n\n# ------------------------------------------------------------------------------------------- drawing helpers\nclass Canvas:\n    def __init__(self, res):\n        self.res, self.k = res, res * SS\n        self.w, self.h = int(CANVAS[0] * self.k), int(CANVAS[1] * self.k)\n\n    def P(self, pts):\n        return [(x * self.k, y * self.k) for x, y in pts]\n\n    def new(self):\n        return Image.new(\"RGBA\", (self.w, self.h), (0, 0, 0, 0))\n\n    def mask(self, pts):\n        m = Image.new(\"L\", (self.w, self.h), 0)\n        ImageDraw.Draw(m).polygon(self.P(pts), fill=255)\n        return m\n\n    def ell_mask(self, c, rx, ry, angle=0.0):\n        return self.mask(ellipse_pts(c, rx, ry, angle))\n\n    def finish(self, im):\n        return im.resize((int(CANVAS[0] * self.res), int(CANVAS[1] * self.res)), Image.LANCZOS)\n\n\ndef bez(p0, p1, p2, p3, n=24):\n    out = []\n    for i in range(n + 1):\n        t = i / n\n        a, b, c, d = (1 - t) ** 3, 3 * (1 - t) ** 2 * t, 3 * (1 - t) * t ** 2, t ** 3\n        out.append((a * p0[0] + b * p1[0] + c * p2[0] + d * p3[0], a * p0[1] + b * p1[1] + c * p2[1] + d * p3[1]))\n    return out\n\n\ndef path(start, *segs, n=24):\n    \"\"\"start point + cubic segments (c1, c2, end) \u2192 polygon points.\"\"\"\n    pts, cur = [start], start\n    for c1, c2, end in segs:\n        pts += bez(cur, c1, c2, end, n)[1:]\n        cur = end\n    return pts\n\n\ndef ellipse_pts(c, rx, ry, angle=0.0, n=96):\n    ca, sa = math.cos(math.radians(angle)), math.sin(math.radians(angle))\n    return [(c[0] + rx * math.cos(2 * math.pi * i / n) * ca - ry * math.sin(2 * math.pi * i / n) * sa,\n             c[1] + rx * math.cos(2 * math.pi * i / n) * sa + ry * math.sin(2 * math.pi * i / n) * ca) for i in range(n)]\n\n\ndef capsule(a, b, r0, r1, n=40):\n    \"\"\"A limb: rounded at both ends, radius r0 at a and r1 at b (complete, so it can rotate without seams).\"\"\"\n    ang = math.atan2(b[1] - a[1], b[0] - a[0])\n    pts = []\n    for i in range(n + 1):\n        t = ang + math.pi / 2 + math.pi * i / n\n        pts.append((b[0] + r1 * math.cos(t - math.pi), b[1] + r1 * math.sin(t - math.pi)))\n    for i in range(n + 1):\n        t = ang + math.pi / 2 + math.pi * i / n\n        pts.append((a[0] + r0 * math.cos(t), a[1] + r0 * math.sin(t)))\n    return pts\n\n\ndef fill(cv, im, mask, color, shade=None, light=None, top=0.0, bot=1.0, outline=None, ow=4.0):\n    \"\"\"Paint `color` inside `mask`; `shade`: a darker colour toward the bottom/right (soft form shadow);\n    `light`: a soft highlight toward the top/left; outline: a soft coloured line just inside the edge.\"\"\"\n    col = Image.new(\"RGBA\", im.size, color + (255,))\n    if shade is not None:                                     # vertical gradient inside the shape's own box\n        bb = mask.getbbox()\n        if bb:\n            g = np.zeros((im.size[1],), np.float32)\n            y0, y1 = bb[1], bb[3]\n            ys = np.arange(im.size[1])\n            g = np.clip((ys - (y0 + top * (y1 - y0))) / max((bot - top) * (y1 - y0), 1), 0, 1) ** 1.4\n            grad = Image.fromarray((g[:, None] * 255).astype(np.uint8).repeat(im.size[0], 1), \"L\")\n            sh = Image.new(\"RGBA\", im.size, shade + (255,))\n            col = Image.composite(sh, col, grad)\n    layer = Image.new(\"RGBA\", im.size, (0, 0, 0, 0))\n    layer.paste(col, (0, 0), mask)\n    if light is not None:                                     # a blurred rim light from the upper left\n        off = ImageChops.offset(mask, int(14 * cv.k), int(18 * cv.k))\n        rim = ImageChops.subtract(mask, off).filter(ImageFilter.GaussianBlur(10 * cv.k))\n        rim = ImageChops.multiply(rim, mask)\n        lt = Image.new(\"RGBA\", im.size, light + (0,))\n        lt.putalpha(rim.point(lambda v: int(v * 0.75)))\n        layer.alpha_composite(lt)\n    if outline is not None:\n        er = mask.filter(ImageFilter.MinFilter(max(3, int(ow * cv.k) | 1)))\n        edge = ImageChops.subtract(mask, er).filter(ImageFilter.GaussianBlur(0.6 * cv.k))\n        ol = Image.new(\"RGBA\", im.size, outline + (0,))\n        ol.putalpha(edge)\n        layer.alpha_composite(ol)\n    im.alpha_composite(layer)\n\n\ndef soft(cv, im, mask, color, alpha=0.5, blur=12, clip=None, offset=(0, 0)):\n    \"\"\"A blurred shadow / glow, optionally clipped to another mask.\"\"\"\n    m = ImageChops.offset(mask, int(offset[0] * cv.k), int(offset[1] * cv.k)) if offset != (0, 0) else mask\n    m = m.filter(ImageFilter.GaussianBlur(blur * cv.k)).point(lambda v: int(v * alpha))\n    if clip is not None:\n        m = ImageChops.multiply(m, clip)\n    lay = Image.new(\"RGBA\", im.size, color + (0,))\n    lay.putalpha(m)\n    im.alpha_composite(lay)\n\n\ndef stroke(cv, im, pts, color, width, alpha=255):\n    d = ImageDraw.Draw(im)\n    w = max(1, int(width * cv.k))\n    d.line(cv.P(pts), fill=color + (alpha,), width=w, joint=\"curve\")\n    r = w / 2\n    for x, y in (cv.P([pts[0]])[0], cv.P([pts[-1]])[0]):\n        d.ellipse([x - r, y - r, x + r, y + r], fill=color + (alpha,))\n\n\n# ------------------------------------------------------------------------------------------- parts\ndef head_shape():\n    cx, cy = HEAD_C\n    return path((cx, cy - 250), ((cx + 150, cy - 250), (cx + 245, cy - 150), (cx + 245, cy - 10)),\n                ((cx + 245, cy + 120), (cx + 170, cy + 235), (cx, cy + 250)),\n                ((cx - 170, cy + 235), (cx - 245, cy + 120), (cx - 245, cy - 10)),\n                ((cx - 245, cy - 150), (cx - 150, cy - 250), (cx, cy - 250)))\n\n\ndef draw_head(cv, d):\n    im = cv.new()\n    skin, sh, ol = hexrgb(d[\"skin\"]), hexrgb(d[\"skin_shadow\"]), mix(hexrgb(d[\"skin_shadow\"]), (40, 20, 10), 0.35)\n    for sx in (-1, 1):                                        # ears (behind the face edge)\n        e = cv.ell_mask((500 + sx * 238, 510), 38, 52)\n        fill(cv, im, e, skin, shade=sh, outline=ol)\n        soft(cv, im, cv.ell_mask((500 + sx * 240, 512), 16, 26), sh, 0.55, 4)\n    m = cv.mask(head_shape())\n    fill(cv, im, m, skin, shade=sh, top=0.45, bot=1.25, outline=ol, ow=4)\n    soft(cv, im, cv.ell_mask((560, 360), 150, 110), (255, 230, 205), 0.22, 40, clip=m)       # forehead light\n    soft(cv, im, cv.mask(head_shape()), sh, 0.35, 18, clip=m, offset=(-22, -16))           # form shadow (right)\n    return cv.finish(im)\n\n\ndef draw_neck(cv, d):\n    im = cv.new()\n    m = cv.mask(path((452, 650), ((452, 700), (450, 760), (446, 800)), ((480, 815), (520, 815), (554, 800)),\n                     ((550, 760), (548, 700), (548, 650))))\n    fill(cv, im, m, hexrgb(d[\"skin_shadow\"]), shade=mix(hexrgb(d[\"skin_shadow\"]), (40, 20, 10), 0.25))\n    soft(cv, im, cv.ell_mask((500, 690), 70, 30), (40, 20, 10), 0.35, 10, clip=m)         # chin shadow\n    return cv.finish(im)\n\n\ndef draw_cheeks(cv, d):\n    im = cv.new()\n    for sx in (-1, 1):\n        soft(cv, im, cv.ell_mask((500 + sx * 158, 598), 40, 20), hexrgb(d[\"blush\"]), 0.38, 10)\n    return cv.finish(im)\n\n\ndef draw_nose(cv, d):\n    im = cv.new()\n    sh = hexrgb(d[\"skin_shadow\"])\n    soft(cv, im, cv.ell_mask((505, 590), 24, 16), sh, 0.65, 4)\n    soft(cv, im, cv.ell_mask((494, 580), 10, 7), (255, 235, 215), 0.5, 3)\n    return cv.finish(im)\n\n\ndef eye_sclera_pts(side, variant):\n    c = EYE_C[side]\n    rx, ry = EYE_R\n    if variant == \"wide\":\n        rx, ry = rx * 1.1, ry * 1.16\n    pts = ellipse_pts(c, rx, ry)\n    if variant == \"half\":                                    # upper lid lowered: cut the top\n        pts = [(x, max(y, c[1] - ry * 0.15)) for x, y in pts]\n    if variant == \"smile\":                                   # cheeks push the lower lid up\n        pts = [(x, min(y, c[1] + ry * 0.45 + 0.25 * abs(x - c[0]))) for x, y in pts]\n    return pts\n\n\ndef draw_eye(cv, d, side, variant, part=\"all\"):\n    \"\"\"One eye. part \"white\": the sclera only; \"lids\": lid lines and lashes only (drawn over the iris, which is a\n    separate layer clipped to `eyemask_*` while rendering); \"all\": closed / happy eyes (no iris).\"\"\"\n    im = cv.new()\n    line = mix(hexrgb(d[\"hair\"]), (0, 0, 0), 0.2)\n    c = EYE_C[side]\n    rx, ry = EYE_R\n    if variant in (\"closed\", \"happy\"):\n        if variant == \"closed\":\n            pts = bez((c[0] - rx, c[1] + 6), (c[0] - rx * 0.4, c[1] + ry * 0.45), (c[0] + rx * 0.4, c[1] + ry * 0.45),\n                      (c[0] + rx, c[1] + 6))\n        else:                                                # ^ shaped: laughing / very happy\n            pts = bez((c[0] - rx, c[1] + 16), (c[0] - rx * 0.45, c[1] - ry * 0.55), (c[0] + rx * 0.45, c[1] - ry * 0.55),\n                      (c[0] + rx, c[1] + 16))\n        stroke(cv, im, pts, line, 11)\n        if d[\"lashes\"]:\n            outer = pts[0] if side == \"L\" else pts[-1]\n            sx = -1 if side == \"L\" else 1\n            stroke(cv, im, [outer, (outer[0] + 22 * sx, outer[1] - 14)], line, 8)\n        return cv.finish(im)\n    pts = eye_sclera_pts(side, variant)\n    m = cv.mask(pts)\n    if part == \"white\":\n        fill(cv, im, m, (253, 252, 250), shade=(236, 234, 240), top=0.3, bot=1.2)\n        return cv.finish(im)\n    soft(cv, im, cv.mask([(x, y - 46) for x, y in pts]), (110, 80, 90), 0.28, 8, clip=m)   # lid shadow on the eye\n    top = [p for p in pts if p[1] <= c[1] + 8]\n    top.sort()\n    lid = Image.new(\"L\", (cv.w, cv.h), 0)\n    dl = ImageDraw.Draw(lid)\n    for i, (x, y) in enumerate(top):                           # thick in the middle, thin at the corners\n        u = i / max(len(top) - 1, 1)\n        r = (5 + 9 * math.sin(math.pi * min(max(u, 0.05), 0.95))) * cv.k\n        dl.ellipse([x * cv.k - r, (y - 2) * cv.k - r, x * cv.k + r, (y - 2) * cv.k + r], fill=255)\n    fill(cv, im, lid, line)\n    bottom = [p for p in pts if p[1] > c[1] + ry * 0.35]\n    bottom.sort()\n    if bottom:\n        stroke(cv, im, bottom, mix(line, hexrgb(d[\"skin\"]), 0.55), 4)\n    if d[\"lashes\"]:\n        outer = top[0] if side == \"L\" else top[-1]\n        sx = -1 if side == \"L\" else 1\n        for k in range(d[\"lashes\"]):\n            stroke(cv, im, [outer, (outer[0] + (24 - 6 * k) * sx, outer[1] - 16 - 12 * k)], line, 8)\n    return cv.finish(im)\n\n\ndef eye_mask(cv, side, variant):\n    m = cv.mask(eye_sclera_pts(side, variant))\n    m = m.filter(ImageFilter.MinFilter(max(3, int(5 * cv.k) | 1)))\n    return m.resize((int(CANVAS[0] * cv.res), int(CANVAS[1] * cv.res)), Image.LANCZOS)\n\n\ndef draw_iris(cv, d, side, small=False):\n    im = cv.new()\n    c = EYE_C[side]\n    r = 50 if not small else 36\n    iris = cv.ell_mask(c, r, r * 1.05)\n    fill(cv, im, iris, hexrgb(d[\"iris_light\"]), shade=hexrgb(d[\"iris\"]), top=0.0, bot=0.7)\n    soft(cv, im, cv.ell_mask(c, r, r * 1.05), (20, 10, 5), 0.6, 6, clip=iris, offset=(0, -10))\n    fill(cv, im, cv.ell_mask((c[0], c[1] + 2), r * 0.52, r * 0.54), (16, 10, 8))\n    fill(cv, im, cv.ell_mask((c[0] - r * 0.34, c[1] - r * 0.38), r * 0.34, r * 0.3), (255, 255, 255))   # highlights\n    fill(cv, im, cv.ell_mask((c[0] + r * 0.35, c[1] + r * 0.38), r * 0.13, r * 0.12), (255, 255, 255))\n    return cv.finish(im)\n\n\nBROWS = {   # (inner dy, mid dy, outer dy): negative = up\n    \"neutral\": (0, -8, 2), \"raised\": (-26, -34, -18), \"furrow\": (14, 4, -4), \"sad\": (-22, -10, 10),\n    \"curious\": None, \"high\": (-40, -48, -30),\n}\n\n\ndef draw_brows(cv, d, variant):\n    im = cv.new()\n    col = hexrgb(d[\"brow\"])\n    for side, sx in ((\"L\", -1), (\"R\", 1)):\n        shape = BROWS[variant]\n        if variant == \"curious\":                             # one brow up, one level: a questioning look\n            shape = BROWS[\"high\"] if side == \"R\" else BROWS[\"neutral\"]\n        c = (EYE_C[side][0], 412)\n        inner, mid, outer = shape\n        pts = bez((c[0] - 48 * sx, c[1] + inner), (c[0] - 15 * sx, c[1] + mid - 6), (c[0] + 20 * sx, c[1] + mid - 6),\n                  (c[0] + 52 * sx, c[1] + outer + 4))\n        m = Image.new(\"L\", (cv.w, cv.h), 0)\n        dd = ImageDraw.Draw(m)\n        for i, (x, y) in enumerate(pts):                      # tapered: thick inside, thin outside\n            r = (13 - 6 * i / len(pts)) * cv.k\n            dd.ellipse([x * cv.k - r, y * cv.k - r, x * cv.k + r, y * cv.k + r], fill=255)\n        fill(cv, im, m, col)\n    return cv.finish(im)\n\n\nMOUTHS = (\"rest\", \"closed\", \"small\", \"wide\", \"round\", \"teeth\", \"smile_talk\", \"smile\", \"frown\", \"wavy\", \"pursed\",\n          \"laugh\", \"o_big\")\n\n\ndef draw_mouth(cv, d, variant):\n    im = cv.new()\n    cx, cy = MOUTH_C\n    lip, dark = hexrgb(d[\"lip\"]), (70, 22, 26)\n    tongue, teeth = (226, 112, 110), (250, 248, 244)\n\n    def opening(pts, show_teeth=False, show_tongue=True):\n        m = cv.mask(pts)\n        fill(cv, im, m, dark)\n        bb = [min(p[0] for p in pts), min(p[1] for p in pts), max(p[0] for p in pts), max(p[1] for p in pts)]\n        if show_tongue:\n            t = cv.ell_mask(((bb[0] + bb[2]) / 2, bb[3] - 4), (bb[2] - bb[0]) * 0.32, (bb[3] - bb[1]) * 0.35)\n            fill(cv, im, Image.fromarray(np.minimum(np.asarray(t), np.asarray(m))), tongue)\n        if show_teeth:\n            tm = cv.mask([(bb[0], bb[1] - 4), (bb[2], bb[1] - 4), (bb[2], bb[1] + (bb[3] - bb[1]) * 0.3),\n                          (bb[0], bb[1] + (bb[3] - bb[1]) * 0.3)])\n            fill(cv, im, Image.fromarray(np.minimum(np.asarray(tm), np.asarray(m))), teeth)\n        stroke(cv, im, pts + [pts[0]], lip, 4)\n\n    if variant in (\"rest\", \"smile\"):\n        w, h = (46, 12) if variant == \"rest\" else (62, 22)\n        stroke(cv, im, bez((cx - w, cy - 4), (cx - w / 2, cy + h), (cx + w / 2, cy + h), (cx + w, cy - 4)), lip, 7)\n    elif variant == \"closed\":                                 # m / b / p: lips pressed\n        stroke(cv, im, bez((cx - 40, cy), (cx - 15, cy + 6), (cx + 15, cy + 6), (cx + 40, cy)), lip, 9)\n    elif variant == \"frown\":\n        stroke(cv, im, bez((cx - 40, cy + 10), (cx - 15, cy - 6), (cx + 15, cy - 6), (cx + 40, cy + 10)), lip, 7)\n    elif variant == \"wavy\":\n        stroke(cv, im, bez((cx - 44, cy + 4), (cx - 20, cy - 10), (cx + 10, cy + 14), (cx + 42, cy - 2)), lip, 7)\n    elif variant == \"pursed\":\n        fill(cv, im, cv.ell_mask((cx + 18, cy), 16, 12), lip)\n    elif variant == \"small\":\n        opening(ellipse_pts((cx, cy + 4), 24, 14), show_tongue=False)\n    elif variant == \"wide\":\n        opening(path((cx - 48, cy - 6), ((cx - 20, cy - 12), (cx + 20, cy - 12), (cx + 48, cy - 6)),\n                     ((cx + 44, cy + 34), (cx - 44, cy + 34), (cx - 48, cy - 6))), show_teeth=True)\n    elif variant == \"round\":\n        opening(ellipse_pts((cx, cy + 6), 22, 26))\n    elif variant == \"o_big\":\n        opening(ellipse_pts((cx, cy + 10), 30, 36))\n    elif variant == \"teeth\":                                  # ee: wide and flat, teeth visible\n        opening(path((cx - 52, cy - 2), ((cx - 20, cy - 8), (cx + 20, cy - 8), (cx + 52, cy - 2)),\n                     ((cx + 40, cy + 18), (cx - 40, cy + 18), (cx - 52, cy - 2))), show_teeth=True, show_tongue=False)\n    elif variant == \"smile_talk\":\n        opening(path((cx - 54, cy - 8), ((cx - 20, cy - 2), (cx + 20, cy - 2), (cx + 54, cy - 8)),\n                     ((cx + 40, cy + 30), (cx - 40, cy + 30), (cx - 54, cy - 8))), show_teeth=True)\n    elif variant == \"laugh\":\n        opening(path((cx - 64, cy - 12), ((cx - 25, cy - 4), (cx + 25, cy - 4), (cx + 64, cy - 12)),\n                     ((cx + 52, cy + 52), (cx - 52, cy + 52), (cx - 64, cy - 12))), show_teeth=True)\n    return cv.finish(im)\n\n\n# ---- hair\ndef draw_hair(cv, d, which):\n    im = cv.new()\n    hair, light = hexrgb(d[\"hair\"]), hexrgb(d[\"hair_light\"])\n    ol = mix(hair, (0, 0, 0), 0.4)\n    if d[\"hair_style\"] == \"wavy_short_curl\":\n        if which == \"back\":\n            m = cv.mask(path((240, 520), ((210, 330), (330, 190), (500, 188)), ((680, 186), (800, 320), (762, 520)),\n                             ((730, 470), (700, 420), (690, 400)), ((560, 420), (440, 420), (310, 400)),\n                             ((290, 430), (270, 470), (240, 520))))\n            fill(cv, im, m, mix(hair, (0, 0, 0), 0.15), outline=ol)\n            return cv.finish(im)\n        # front: wavy fringe + sides, and the signature curl at the top\n        pts = path((246, 520), ((226, 360), (300, 236), (420, 210)), ((520, 190), (650, 196), (720, 260)),\n                   ((770, 310), (770, 420), (756, 520)), ((735, 470), (722, 430), (705, 398)),\n                   ((680, 418), (650, 392), (635, 370)), ((600, 400), (560, 392), (545, 366)),\n                   ((515, 398), (470, 396), (455, 368)), ((420, 400), (380, 392), (368, 366)),\n                   ((330, 396), (300, 390), (292, 386)), ((282, 430), (268, 470), (246, 520)))\n        m = cv.mask(pts)\n        fill(cv, im, m, hair, shade=mix(hair, (0, 0, 0), 0.3), top=0.2, bot=1.2, outline=ol)\n        for a, b2, c2, e in (((330, 300), (380, 250), (450, 240), (500, 250)), ((540, 236), (600, 232), (650, 250), (690, 290)),\n                             ((300, 360), (320, 320), (350, 300), (390, 292))):\n            stroke(cv, im, bez(a, b2, c2, e), light, 9, 170)\n        curl = []                                            # the signature curl: a small spiral lock on top\n        for i in range(60):\n            t = i / 59\n            ang = math.pi * 0.9 + t * math.pi * 2.3\n            r = 46 * (1 - 0.7 * t)\n            curl.append((520 + 40 * t + r * math.cos(ang), 214 - 30 * t + r * math.sin(ang)))\n        stroke(cv, im, curl, hair, 26)\n        stroke(cv, im, curl[10:40], light, 7, 190)\n        return cv.finish(im)\n    # Meera: wavy hair, half-up (a small bun at the back-top), bangs swept to one side, star clip\n    if which == \"back\":\n        m = cv.mask(path((232, 470), ((200, 300), (330, 196), (500, 194)), ((680, 192), (800, 300), (768, 470)),\n                         ((800, 640), (840, 830), (790, 990)), ((750, 1040), (700, 1000), (690, 960)),\n                         ((640, 900), (660, 760), (640, 650)), ((560, 600), (440, 600), (360, 650)),\n                         ((340, 760), (360, 900), (310, 960)), ((300, 1000), (250, 1040), (210, 990)),\n                         ((160, 830), (200, 640), (232, 470))))\n        fill(cv, im, m, mix(hair, (0, 0, 0), 0.12), shade=mix(hair, (0, 0, 0), 0.35), outline=ol)\n        for x0 in (250, 300, 700, 750):                         # wave strands\n            sx = 1 if x0 < 500 else -1\n            stroke(cv, im, bez((x0, 640), (x0 - 30 * sx, 760), (x0 + 30 * sx, 860), (x0 - 10 * sx, 960)), light, 7, 150)\n        bun = cv.ell_mask((560, 200), 78, 64, -15)               # the half-up bun\n        fill(cv, im, bun, hair, shade=mix(hair, (0, 0, 0), 0.3), outline=ol)\n        stroke(cv, im, bez((505, 190), (540, 160), (590, 160), (620, 190)), light, 8, 170)\n        return cv.finish(im)\n    pts = path((236, 560), ((216, 370), (300, 236), (430, 212)), ((560, 190), (690, 214), (744, 300)),\n               ((786, 370), (780, 470), (764, 600)), ((748, 560), (735, 500), (720, 440)),\n               ((690, 420), (650, 380), (620, 330)), ((560, 380), (450, 410), (360, 420)),\n               ((330, 440), (300, 470), (296, 520)), ((280, 540), (262, 560), (236, 560)))\n    m = cv.mask(pts)\n    fill(cv, im, m, hair, shade=mix(hair, (0, 0, 0), 0.3), top=0.2, bot=1.2, outline=ol)\n    for a, b2, c2, e in (((340, 300), (420, 250), (520, 236), (600, 250)), ((630, 270), (690, 300), (720, 360), (730, 420)),\n                         ((300, 380), (330, 340), (380, 320), (440, 320))):\n        stroke(cv, im, bez(a, b2, c2, e), light, 9, 170)\n    if d[\"accessory\"] == \"star_clip\":                         # small yellow star hair clip (her signature)\n        cx, cy, R, r = 676, 300, 30, 13\n        star = [(cx + (R if i % 2 == 0 else r) * math.cos(-math.pi / 2 + i * math.pi / 5),\n                 cy + (R if i % 2 == 0 else r) * math.sin(-math.pi / 2 + i * math.pi / 5)) for i in range(10)]\n        fill(cv, im, cv.mask(star), (250, 205, 70), shade=(225, 150, 40), outline=(170, 100, 20))\n    return cv.finish(im)\n\n\n# ---- body\ndef draw_torso(cv, d):\n    im = cv.new()\n    top, sh, lt = hexrgb(d[\"top\"]), hexrgb(d[\"top_shadow\"]), hexrgb(d[\"top_light\"])\n    tee, tee_sh = hexrgb(d[\"tee\"]), hexrgb(d[\"tee_shadow\"])\n    ol = mix(sh, (0, 0, 0), 0.35)\n    body = path((400, 762), ((450, 744), (550, 744), (600, 762)), ((660, 772), (690, 800), (686, 860)),\n                ((684, 960), (664, 1100), (656, 1180)), ((560, 1196), (440, 1196), (344, 1180)),\n                ((336, 1100), (316, 960), (314, 860)), ((310, 800), (340, 772), (400, 762)))\n    m = cv.mask(body)\n    fill(cv, im, m, top, shade=sh, top=0.3, bot=1.3, light=lt, outline=ol)\n    v = path((452, 772), ((470, 790), (530, 790), (548, 772)), ((560, 900), (545, 1080), (540, 1192)),\n             ((510, 1196), (490, 1196), (460, 1192)), ((455, 1080), (440, 900), (452, 772)))\n    vm = cv.mask(v)\n    fill(cv, im, vm, tee, shade=tee_sh, top=0.4, bot=1.4)       # the shirt between the open jacket\n    soft(cv, im, vm, (0, 0, 0), 0.25, 8, clip=m, offset=(10, 0))\n    hood = path((400, 760), ((440, 730), (560, 730), (600, 760)), ((590, 800), (560, 812), (548, 790)),\n                ((530, 770), (470, 770), (452, 790)), ((440, 812), (410, 800), (400, 760)))\n    fill(cv, im, cv.mask(hood), mix(top, (255, 255, 255), 0.1), shade=sh, outline=ol)   # hood / collar\n    if d[\"hair_style\"] == \"wavy_short_curl\":\n        for x in (468, 532):                                    # hoodie drawstrings\n            stroke(cv, im, [(x, 800), (x - 4, 905)], (245, 245, 240), 7)\n            fill(cv, im, cv.ell_mask((x - 4, 912), 7, 10), (220, 220, 215))\n        stroke(cv, im, bez((370, 1070), (430, 1090), (570, 1090), (630, 1070)), ol, 4, 180)   # front pocket\n    else:\n        for y in (840, 920, 1000):                              # jacket buttons\n            fill(cv, im, cv.ell_mask((436, y), 9, 9), (250, 235, 220), outline=ol, ow=2)\n    stroke(cv, im, bez((350, 1160), (430, 1178), (570, 1178), (650, 1160)), ol, 5, 200)       # hem\n    return cv.finish(im)\n\n\ndef draw_legs(cv, d):\n    im = cv.new()\n    p, ps = hexrgb(d[\"pants\"]), hexrgb(d[\"pants_shadow\"])\n    ol = mix(ps, (0, 0, 0), 0.4)\n    for sx in (-1, 1):\n        x = 500 + sx * 78\n        leg = path((x - 70, 1170), ((x - 70, 1300), (x - 62, 1420), (x - 58, 1540)),\n                   ((x - 20, 1550), (x + 20, 1550), (x + 58, 1540)), ((x + 62, 1420), (x + 70, 1300), (x + 70, 1170)))\n        fill(cv, im, cv.mask(leg), p, shade=ps, top=0.2, bot=1.2, outline=ol)\n    for sx in (-1, 1):                                          # sneakers\n        x = 500 + sx * 82\n        shoe = path((x - 64, 1600), ((x - 70, 1550), (x - 40, 1525), (x, 1525)), ((x + 50, 1525), (x + 82, 1560), (x + 92, 1600)),\n                    ((x + 60, 1616), (x - 40, 1616), (x - 64, 1600)))\n        if sx < 0:\n            shoe = [(2 * x - px, py) for px, py in shoe]\n        fill(cv, im, cv.mask(shoe), hexrgb(d[\"shoe\"]), shade=hexrgb(d[\"shoe_shadow\"]), outline=mix(hexrgb(d[\"shoe_shadow\"]), (0, 0, 0), 0.4))\n        sole = [(px, max(py, 1596)) for px, py in shoe]\n        fill(cv, im, cv.mask(sole), hexrgb(d[\"sole\"]))\n        stroke(cv, im, [(x - 25, 1550), (x + 25, 1550)], hexrgb(d[\"sole\"]), 6, 220)\n    return cv.finish(im)\n\n\ndef draw_upper_arm(cv, d, side):\n    \"\"\"The sleeve from the shoulder to the elbow. It is drawn over the torso: its outline fades out around the\n    shoulder so the sleeve grows out of the jacket instead of showing a joint circle.\"\"\"\n    im = cv.new()\n    a, b = SHOULDER[side], ELBOW[side]\n    m = cv.mask(capsule(a, b, 50, 44))\n    fill(cv, im, m, hexrgb(d[\"top\"]), shade=hexrgb(d[\"top_shadow\"]), light=hexrgb(d[\"top_light\"]))\n    ol = mix(hexrgb(d[\"top_shadow\"]), (0, 0, 0), 0.35)\n    er = m.filter(ImageFilter.MinFilter(max(3, int(4 * cv.k) | 1)))\n    edge = ImageChops.subtract(m, er).filter(ImageFilter.GaussianBlur(0.6 * cv.k))\n    fade = cv.ell_mask(a, 70, 70).filter(ImageFilter.GaussianBlur(28 * cv.k))\n    edge = ImageChops.subtract(edge, fade)\n    lay = Image.new(\"RGBA\", im.size, ol + (0,)); lay.putalpha(edge)\n    im.alpha_composite(lay)\n    return cv.finish(im)\n\n\ndef draw_forearm(cv, d, side):\n    im = cv.new()\n    a, b = ELBOW[side], WRIST[side]\n    m = cv.mask(capsule(a, b, 44, 40))\n    fill(cv, im, m, hexrgb(d[\"top\"]), shade=hexrgb(d[\"top_shadow\"]), light=hexrgb(d[\"top_light\"]),\n         outline=mix(hexrgb(d[\"top_shadow\"]), (0, 0, 0), 0.35))\n    cuff = capsule((b[0] + (a[0] - b[0]) * 0.12, b[1] + (a[1] - b[1]) * 0.12), b, 42, 41)\n    fill(cv, im, cv.mask(cuff), mix(hexrgb(d[\"top_shadow\"]), (255, 255, 255), 0.08))\n    return cv.finish(im)\n\n\nHANDS = (\"open\", \"point\", \"wave\", \"hold\")\n\n\ndef draw_hand(cv, d, side, variant):\n    \"\"\"A soft cartoon hand at the wrist (pointing DOWN in the rest pose; the rig rotates it). One silhouette per hand:\n    palm, fingers and thumb are merged before painting, so there are no seams inside the hand.\"\"\"\n    im = cv.new()\n    skin, sh = hexrgb(d[\"skin\"]), hexrgb(d[\"skin_shadow\"])\n    ol = mix(sh, (40, 20, 10), 0.35)\n    wx, wy = WRIST[side]\n    sx = -1 if side == \"L\" else 1                               # thumb toward the body's centre\n    parts = [ellipse_pts((wx, wy + 42), 36, 40)]                # palm\n    parts.append(capsule((wx - 24 * sx, wy + 30), (wx - 46 * sx, wy + 62), 15, 12))   # thumb\n    if variant == \"point\":\n        parts.append(capsule((wx + 4 * sx, wy + 70), (wx + 8 * sx, wy + 138), 13, 11))\n        parts.append(ellipse_pts((wx + 2 * sx, wy + 76), 30, 22))                      # curled fingers\n    elif variant in (\"open\", \"wave\"):\n        spread = 1.0 if variant == \"wave\" else 0.45\n        for i, ang in enumerate((-18, -6, 6, 18)):\n            r = math.radians(ang * spread)\n            a = (wx + (i - 1.5) * 16, wy + 66)\n            b = (a[0] + 40 * math.sin(r), a[1] + (40 - 6 * abs(i - 1.5)) * math.cos(r))\n            parts.append(capsule(a, b, 11.5, 10.5))\n    else:                                                       # hold: a loose fist\n        parts.append(ellipse_pts((wx, wy + 70), 36, 26))\n    m = Image.new(\"L\", (cv.w, cv.h), 0)\n    for pts in parts:\n        m = ImageChops.lighter(m, cv.mask(pts))\n    fill(cv, im, m, skin, shade=sh, top=0.1, bot=1.3, outline=ol, ow=3.5)\n    if variant == \"hold\":\n        for i in range(3):\n            stroke(cv, im, [(wx - 18 + 18 * i, wy + 74), (wx - 16 + 18 * i, wy + 88)], ol, 3.5, 150)\n    return cv.finish(im)\n\n\ndef draw_glasses(cv, d):\n    im = cv.new()\n    frame = (32, 34, 40)\n    for side in (\"L\", \"R\"):\n        c = EYE_C[side]\n        lens = path((c[0] - 78, c[1] - 52), ((c[0] - 20, c[1] - 60), (c[0] + 20, c[1] - 60), (c[0] + 78, c[1] - 52)),\n                    ((c[0] + 82, c[1] + 10), (c[0] + 70, c[1] + 60), (c[0], c[1] + 62)),\n                    ((c[0] - 70, c[1] + 60), (c[0] - 82, c[1] + 10), (c[0] - 78, c[1] - 52)))\n        m = cv.mask(lens)\n        lay = Image.new(\"RGBA\", im.size, (60, 70, 95, 0)); lay.putalpha(m.point(lambda v: int(v * 0.26)))\n        im.alpha_composite(lay)\n        soft(cv, im, cv.ell_mask((c[0] - 30, c[1] - 30), 30, 14, -20), (255, 255, 255), 0.5, 4, clip=m)\n        stroke(cv, im, lens + [lens[0]], frame, 13)\n    stroke(cv, im, bez((480, 470), (490, 458), (510, 458), (520, 470)), frame, 13)\n    for sx in (-1, 1):\n        stroke(cv, im, [(500 + sx * 245, 480), (500 + sx * 262, 470)], frame, 12)\n    return cv.finish(im)\n\n\ndef draw_prop(cv, which):\n    \"\"\"Held props, drawn at the RIGHT wrist in the rest pose (they ride on the hand).\"\"\"\n    im = cv.new()\n    wx, wy = WRIST[\"R\"]\n    if which == \"popcorn\":\n        x, y = wx, wy + 40\n        box = [(x - 70, y - 120), (x + 70, y - 120), (x + 52, y + 60), (x - 52, y + 60)]\n        m = cv.mask(box)\n        fill(cv, im, m, (232, 62, 58), shade=(190, 40, 40))\n        for k in range(-2, 3):\n            stripe = [(x + 28 * k - 8, y - 120), (x + 28 * k + 8, y - 120), (x + 21 * k + 6, y + 60), (x + 21 * k - 6, y + 60)]\n            fill(cv, im, Image.fromarray(np.minimum(np.asarray(cv.mask(stripe)), np.asarray(m))), (250, 248, 240))\n        for k in range(9):\n            px, py = x - 64 + 16 * k, y - 132 - 16 * (k % 3)\n            fill(cv, im, cv.ell_mask((px, py), 22, 19), (255, 240, 190), shade=(236, 200, 120), outline=(210, 170, 90), ow=2)\n    elif which == \"3d_glasses\":\n        x, y = wx, wy + 10\n        for sx in (-1, 1):\n            lens = cv.ell_mask((x + sx * 52, y), 46, 34)\n            lay = Image.new(\"RGBA\", im.size, (60, 70, 95, 0)); lay.putalpha(lens.point(lambda v: int(v * 0.6)))\n            im.alpha_composite(lay)\n            stroke(cv, im, ellipse_pts((x + sx * 52, y), 46, 34), (32, 34, 40), 11)\n        stroke(cv, im, [(x - 8, y - 6), (x + 8, y - 6)], (32, 34, 40), 10)\n    return cv.finish(im)\n\n\ndef draw_back(cv, d):\n    \"\"\"Seen from behind (audience in the cinema, over-the-shoulder): hair, head and jacket back in one layer.\"\"\"\n    im = cv.new()\n    hair = hexrgb(d[\"hair\"])\n    fill(cv, im, cv.mask(path((400, 762), ((450, 744), (550, 744), (600, 762)), ((700, 800), (700, 1000), (690, 1196)),\n                              ((560, 1206), (440, 1206), (310, 1196)), ((300, 1000), (300, 800), (400, 762)))),\n         hexrgb(d[\"top\"]), shade=hexrgb(d[\"top_shadow\"]), outline=mix(hexrgb(d[\"top_shadow\"]), (0, 0, 0), 0.35))\n    fill(cv, im, cv.mask(head_shape()), hexrgb(d[\"skin_shadow\"]))\n    for sx in (-1, 1):\n        fill(cv, im, cv.ell_mask((500 + sx * 238, 510), 36, 50), hexrgb(d[\"skin_shadow\"]))\n    if d[\"hair_style\"] == \"wavy_half_up\":\n        back = path((228, 470), ((200, 250), (360, 190), (500, 190)), ((640, 190), (800, 250), (772, 470)),\n                    ((810, 700), (800, 900), (740, 1000)), ((600, 1040), (400, 1040), (260, 1000)),\n                    ((200, 900), (190, 700), (228, 470)))\n    else:\n        back = path((236, 560), ((210, 330), (330, 200), (500, 196)), ((670, 200), (790, 330), (764, 560)),\n                    ((700, 640), (300, 640), (236, 560)))\n    fill(cv, im, cv.mask(back), hair, shade=mix(hair, (0, 0, 0), 0.3), outline=mix(hair, (0, 0, 0), 0.4))\n    if d[\"hair_style\"] == \"wavy_half_up\":\n        fill(cv, im, cv.ell_mask((520, 260), 80, 66), mix(hair, (255, 255, 255), 0.05), outline=mix(hair, (0, 0, 0), 0.4))\n    return cv.finish(im)\n\n\nEYE_VARIANTS = (\"open\", \"half\", \"wide\", \"smile\", \"closed\", \"happy\")\nBROW_VARIANTS = tuple(BROWS)\n\n\ndef layers(cid, res=1.6):\n    \"\"\"Every layer of one character: {name: RGBA image (full canvas at `res` px per design unit)}.\"\"\"\n    d = DESIGNS[cid]\n    cv = Canvas(res)\n    out = {\"hair_back\": draw_hair(cv, d, \"back\"), \"legs\": draw_legs(cv, d), \"torso\": draw_torso(cv, d),\n           \"neck\": draw_neck(cv, d), \"head\": draw_head(cv, d), \"cheeks\": draw_cheeks(cv, d), \"nose\": draw_nose(cv, d),\n           \"hair_front\": draw_hair(cv, d, \"front\"), \"glasses\": draw_glasses(cv, d), \"back\": draw_back(cv, d)}\n    for side in (\"L\", \"R\"):\n        out[f\"upper_arm_{side}\"] = draw_upper_arm(cv, d, side)\n        out[f\"forearm_{side}\"] = draw_forearm(cv, d, side)\n        for h in HANDS:\n            out[f\"hand_{side}_{h}\"] = draw_hand(cv, d, side, h)\n        out[f\"iris_{side}\"] = draw_iris(cv, d, side)\n        out[f\"iris_{side}_small\"] = draw_iris(cv, d, side, small=True)\n        for v in EYE_VARIANTS:\n            if v in (\"open\", \"half\", \"wide\", \"smile\"):\n                out[f\"eye_{side}_{v}\"] = draw_eye(cv, d, side, v, \"white\")\n                out[f\"lid_{side}_{v}\"] = draw_eye(cv, d, side, v, \"lids\")\n            else:\n                out[f\"eye_{side}_{v}\"] = draw_eye(cv, d, side, v)\n            if v in (\"open\", \"half\", \"wide\", \"smile\"):\n                m = eye_mask(cv, side, v)\n                out[f\"eyemask_{side}_{v}\"] = Image.merge(\"RGBA\", (m, m, m, m))\n    for v in BROW_VARIANTS:\n        out[f\"brows_{v}\"] = draw_brows(cv, d, v)\n    for v in MOUTHS:\n        out[f\"mouth_{v}\"] = draw_mouth(cv, d, v)\n    for p in (\"popcorn\", \"3d_glasses\"):\n        out[f\"prop_{p}\"] = draw_prop(cv, p)\n    return out\n", "storybook_atlas.py": "\"\"\"Pose-atlas character assets (\"atlas_v1\"): cinematic AI artwork animated WITHOUT cutting it into joints.\n\nWhy (measured in reports/character_poc/approach_A_cutout_arm_test.png): rotating a cut-out arm of painted, 3D-style\nartwork leaves a hard cut down the torso, a broken shoulder and fabric shading that no longer matches the angle \u2014\nrepairing that for every pose is as much work as rendering the pose. So each pose is a COMPLETE image rendered by\nthe model from the approved master (reference-conditioned), and only small regions change at runtime:\n  - face patches (eyes-closed blink, mouth shapes, expressions), feathered, rendered by the same model from the same\n    master so the shading matches (no flat pasted cartoon mouth);\n  - whole-character motion: breathing (\u22640.6 % vertical), a slow sway (\u22640.6\u00b0 about the feet), eased crossfades\n    (BLEND_S) between poses, and three-quarter views when the character turns to the other person.\n\nManifest (in the same versioned, sha256-verified folder format as storybook.build_character):\n  \"asset_type\": \"atlas_v1\",\n  \"poses\":   {atlas_pose: {\"file\", \"view\": \"front|three_quarter_left|three_quarter_right|back\", \"feet\": [x, y],\n              \"top\": y, \"head\": [centre_x, top_y, chin_y], \"face\": {\"box\": [x0, y0, x1, y1]}, \"wear\": [...],\n              \"hold\": null|\"popcorn\"|...}}   \u2014 \"head\" places the character by its HEAD, so every camera preset\n              frames the face the same way whatever the body proportions are (else feet/top are used)\n  \"pose_map\": {storybook pose (13 names) \u2192 atlas_pose}   \u2190 explicit, reviewed substitutes, never implicit\n  \"face\":     {\"anchor_pose\": atlas_pose, \"box\": [...], \"blink\": file, \"visemes\": {viseme: file},\n               \"expressions\": {expression: file}}   (patches are RGBA crops of the anchor face box)\n  \"layers\":   every file + sha256 (verify/approve/load reuse storybook's functions)\nA plan that needs something the atlas does not have (a back view, a held prop, 3D glasses) gets a clear error from\ncheck_spec \u2014 the renderer never invents it and never falls back to the code-drawn characters.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib, json\nfrom collections import OrderedDict\nfrom pathlib import Path\n\nfrom PIL import Image\n\nimport storybook as SB\n\nASSET_TYPE = \"atlas_v1\"\nDESIGN_HEIGHT = 1426            # head top (190) \u2192 shoe soles (1616) in the storybook design units\nDESIGN_FEET = (500, 1616)\nDESIGN_HEAD = (500, 210, 720)   # head centre x, head top (hair), chin \u2014 the camera presets frame THIS head\nVIEWS = (\"front\", \"three_quarter_left\", \"three_quarter_right\", \"back\")\nMOUTH_OF_VISEME = {\"rest\": None, \"closed\": \"closed\", \"small\": \"small\", \"wide\": \"wide\", \"round\": \"round\",\n                   \"teeth\": \"teeth\", \"smile_talk\": \"smile_talk\"}\n\n\ndef is_atlas(man):\n    return man.get(\"asset_type\") == ASSET_TYPE\n\n\nclass AtlasCharacter:\n    \"\"\"An atlas_v1 version folder (loaded through storybook.load, which verifies every sha256 first).\"\"\"\n\n    def __init__(self, vdir):\n        self.dir = Path(vdir)\n        self.man = json.loads((self.dir / \"manifest.json\").read_text())\n        self.id, self.version = self.man[\"character\"], self.man[\"version\"]\n        self._img = {}\n\n    def has(self, name):\n        return name in self.man[\"layers\"]\n\n    def img(self, rel):\n        if rel not in self._img:\n            self._img[rel] = Image.open(self.dir / rel).convert(\"RGBA\")\n        return self._img[rel]\n\n    def wear_by_patch(self, w):\n        return w in (self.man.get(\"face\") or {}).get(\"wear\", {})\n\n    def atlas_pose(self, storybook_pose, view=\"front\", wear=(), hold=None):\n        \"\"\"The atlas pose image for a storybook pose in a view, or None if the atlas lacks it. Composition rules\n        (documented, not guesses): things worn on the face (3D glasses) are face patches on any FRONT pose and are\n        not visible from behind; a held prop uses this gesture's own image with the prop if there is one, otherwise\n        the prop's hold pose (holding popcorn \u2192 the popcorn pose); seen from behind, every gesture uses the back\n        image (a gesture is not visible from behind).\"\"\"\n        poses = self.man[\"poses\"]\n        base = self.man[\"pose_map\"].get(storybook_pose)\n        if base is None:\n            return None\n        need_wear = sorted(w for w in wear if not (view == \"back\" or (view == \"front\" and self.wear_by_patch(w))))\n\n        def match(k, h):\n            p = poses.get(k)\n            return p and p.get(\"view\", \"front\") == view and sorted(p.get(\"wear\", [])) == need_wear and p.get(\"hold\") == h\n\n        cands = [base] + [k for k, p in poses.items() if p.get(\"base\") in (storybook_pose, base)]\n        for k in cands:\n            if match(k, hold):\n                return k\n        if hold:                                   # the prop's own hold pose\n            for k, p in poses.items():\n                if p.get(\"hold\") == hold and p.get(\"base\") in (\"hold\", \"show\") and match(k, hold):\n                    return k\n        if view == \"back\":                         # from behind the gesture is not visible: the back image\n            for k, p in poses.items():\n                if p.get(\"view\") == \"back\" and p.get(\"base\") in (\"idle\", None) and match(k, hold):\n                    return k\n        return None\n\n    def missing(self, storybook_pose, view, wear, hold):\n        if self.atlas_pose(storybook_pose, view, wear, hold) is None:\n            need = [f\"pose '{storybook_pose}'\", f\"view '{view}'\"] + [f\"wearing {w}\" for w in wear] + \\\n                   ([f\"holding {hold}\"] if hold else [])\n            return (f\"{self.id} {self.version} (atlas) has no image for \" + \", \".join(need) +\n                    \" \u2014 add it with the Character Studio (a new version), or change the plan\")\n        return None\n\n\nclass AtlasPainter:\n    \"\"\"Same role as storybook.Painter, for atlas characters: draws one character at output scale s (px/design unit).\"\"\"\n\n    def __init__(self, ch, s):\n        self.ch, self.s = ch, s\n        self._cache = OrderedDict()\n\n    def _scaled(self, rel, k):\n        key = (rel, round(k, 4))\n        if key not in self._cache:\n            im = self.ch.img(rel)\n            self._cache[key] = im.resize((max(1, int(im.width * k)), max(1, int(im.height * k))), Image.LANCZOS)\n            if len(self._cache) > 64:\n                self._cache.popitem(last=False)\n        self._cache.move_to_end(key)\n        return self._cache[key]\n\n    def _pose_image(self, pid, face):\n        \"\"\"The pose image with the face patches for this frame (blink / expression / mouth) composited in place.\"\"\"\n        P = self.ch.man[\"poses\"][pid]\n        F = self.ch.man.get(\"face\", {})\n        im = self.ch.img(P[\"file\"])\n        patches = []\n        if F and P.get(\"face\") and P.get(\"view\", \"front\") == \"front\":\n            # order: expression (whole face) \u2192 blink (eyes) \u2192 worn items (eyes) \u2192 mouth shape (mouth)\n            expr = face.get(\"expr\")\n            if expr and expr in F.get(\"expressions\", {}):\n                patches.append(F[\"expressions\"][expr])\n            wearing = [w for w in face.get(\"wear\", ()) if w in F.get(\"wear\", {})]\n            if face.get(\"blink\") and F.get(\"blink\") and not wearing:     # (behind dark glasses a blink is invisible)\n                patches.append(F[\"blink\"])\n            patches += [F[\"wear\"][w] for w in wearing]\n            mouth = MOUTH_OF_VISEME.get(face.get(\"viseme\") or \"rest\")\n            if face.get(\"speaking\") and mouth and mouth in F.get(\"visemes\", {}):\n                patches.append(F[\"visemes\"][mouth])\n        if not patches:\n            return im\n        key = (\"composed\", pid, json.dumps(patches, sort_keys=True))\n        if key in self._cache:\n            self._cache.move_to_end(key)\n            return self._cache[key]\n        out = im.copy()\n        ax0, ay0, ax1, ay1 = F[\"box\"]\n        bx0, by0, bx1, by1 = P[\"face\"][\"box\"]\n        k = (bx1 - bx0) / (ax1 - ax0)                   # anchor face box \u2192 this pose's face box (similarity)\n        for ent in patches:\n            rel, pb = (ent[\"file\"], ent[\"box\"]) if isinstance(ent, dict) else (ent, F[\"box\"])\n            p = self.ch.img(rel)\n            if abs(k - 1) > 1e-3:\n                p = p.resize((max(1, int(p.width * k)), max(1, int(p.height * k))), Image.LANCZOS)\n            out.alpha_composite(p, (int(round(bx0 + (pb[0] - ax0) * k)), int(round(by0 + (pb[1] - ay0) * k))))\n        self._cache[key] = out\n        if len(self._cache) > 64:\n            self._cache.popitem(last=False)\n        return out\n\n    def _place(self, PP):\n        \"\"\"(scale image\u2192output, anchor in the image, the anchor's design point).\"\"\"\n        if PP.get(\"head\"):\n            cx, top, chin = PP[\"head\"]\n            return self.s * (DESIGN_HEAD[2] - DESIGN_HEAD[1]) / max(chin - top, 1), (cx, top), DESIGN_HEAD[:2]\n        return self.s * DESIGN_HEIGHT / max(PP[\"feet\"][1] - PP[\"top\"], 1), tuple(PP[\"feet\"]), DESIGN_FEET\n\n    def draw(self, frame, origin, pid, face, pid_prev=None, u=1.0, motion=(0.0, 0.0)):\n        \"\"\"origin: output pixel of design point (0, 0); pid/pid_prev: atlas poses, u: transition progress 0..1;\n        motion: (sway degrees, breathing fraction), both about the feet (they stay planted).\"\"\"\n        P = self.ch.man[\"poses\"][pid]\n        k = self._place(P)[0]\n        ims = [(self._pose_image(pid, face), P, 1.0)]\n        if pid_prev and pid_prev != pid and u < 1.0:\n            Pp = self.ch.man[\"poses\"][pid_prev]\n            e = SB.ease(u)\n            ims = [(self._pose_image(pid_prev, face), Pp, 1 - e), (ims[0][0], P, e)]\n        sway, breathe = motion\n        sway_q, breathe_q = round(sway * 10) / 10, round(breathe * 2000) / 2000      # quantized: frames reuse sprites\n        sprites = []\n        for im, PP, a in ims:\n            kk, anchor, dpt = self._place(PP)\n            key = (\"sprite\", id(im), round(kk, 4), sway_q, breathe_q)\n            if key in self._cache:\n                self._cache.move_to_end(key)\n                sc, feet, anc = self._cache[key]\n            else:\n                sk = (\"scaled\", id(im), round(kk, 4))\n                if sk not in self._cache:\n                    self._cache[sk] = im.resize((max(1, int(im.width * kk)), max(1, int(im.height * kk))), Image.LANCZOS)\n                sc = self._cache[sk]\n                stretch = 1 + breathe_q                 # breathing: stretch upward from the feet\n                if breathe_q:\n                    sc = sc.resize((sc.width, max(1, int(sc.height * stretch))), Image.BILINEAR)\n                feet = (PP[\"feet\"][0] * kk, PP[\"feet\"][1] * kk * stretch)\n                anc = (anchor[0] * kk, anchor[1] * kk * stretch)\n                if abs(sway_q) > 0.02:                  # sway about the feet\n                    sc = sc.rotate(-sway_q, resample=Image.BICUBIC, center=feet)\n                self._cache[key] = (sc, feet, anc)\n                while len(self._cache) > 160:\n                    self._cache.popitem(last=False)\n            ax, ay = origin[0] + dpt[0] * self.s, origin[1] + dpt[1] * self.s   # where the anchor lands\n            sprites.append((sc, (int(round(ax - anc[0])), int(round(ay - anc[1]))), a))\n        if len(sprites) == 1:\n            frame.alpha_composite(sprites[0][0], sprites[0][1]) if _inside(frame, *sprites[0][:2]) else \\\n                _paste_clipped(frame, *sprites[0][:2])\n            return k\n        # crossfade between two pose images (only during the ~0.45 s transition)\n        x0 = min(p[0] for _, p, _ in sprites); y0 = min(p[1] for _, p, _ in sprites)\n        x1 = max(p[0] + im.width for im, p, _ in sprites); y1 = max(p[1] + im.height for im, p, _ in sprites)\n        layers = []\n        for im, pos, a in sprites:\n            c = Image.new(\"RGBA\", (x1 - x0, y1 - y0), (0, 0, 0, 0))\n            c.alpha_composite(im, (pos[0] - x0, pos[1] - y0))\n            layers.append((c, a))\n        mix = Image.blend(layers[0][0], layers[1][0], layers[1][1])\n        _paste_clipped(frame, mix, (x0, y0))\n        return k\n\n\ndef _inside(frame, im, pos):\n    return pos[0] >= 0 and pos[1] >= 0 and pos[0] + im.width <= frame.width and pos[1] + im.height <= frame.height\n\n\ndef _paste_clipped(frame, im, pos):\n    \"\"\"alpha_composite that tolerates sprites partly outside the frame (close-ups).\"\"\"\n    x, y = pos\n    l, t = max(0, -x), max(0, -y)\n    r, b = min(im.width, frame.width - x), min(im.height, frame.height - y)\n    if r > l and b > t:\n        frame.alpha_composite(im.crop((l, t, r, b)), (x + l, y + t))\n\n\n# ------------------------------------------------------------------------------------------- building a version\ndef content_sha(man):\n    \"\"\"Fingerprint of the artwork only (every file hash + the pose/face geometry), independent of approval fields.\"\"\"\n    body = {\"layers\": {k: v[\"sha256\"] for k, v in sorted(man[\"layers\"].items())},\n            \"poses\": man.get(\"poses\"), \"pose_map\": man.get(\"pose_map\"), \"face\": man.get(\"face\")}\n    return hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest()\n\n\ndef import_atlas(src, root, cid, version, spec, provenance, log=print):\n    \"\"\"src: a folder with the files named in `spec` (poses/*.png, face/*.png); spec: {\"poses\", \"pose_map\", \"face\",\n    \"identity\", \"palette\"}. Writes a NEW immutable version (staging folder, then rename). Every storybook pose must be\n    mapped explicitly (to itself or to a reviewed substitute); every referenced file must exist and be non-empty.\"\"\"\n    import datetime, shutil\n    src, vdir = Path(src), Path(root) / \"characters\" / cid / version\n    if (vdir / \"manifest.json\").exists():\n        raise FileExistsError(f\"{vdir} exists: versions are immutable (make a new version)\")\n    unmapped = [p for p in SB.POSES if p not in spec[\"pose_map\"]]\n    if unmapped:\n        raise ValueError(f\"pose_map must map every storybook pose explicitly; missing: {unmapped}\")\n    bad_targets = [f\"{k}\u2192{v}\" for k, v in spec[\"pose_map\"].items() if v not in spec[\"poses\"]]\n    if bad_targets:\n        raise ValueError(f\"pose_map points to poses the atlas does not have: {bad_targets}\")\n    files = [p[\"file\"] for p in spec[\"poses\"].values()]\n    F = spec.get(\"face\") or {}\n    fname = lambda e: e[\"file\"] if isinstance(e, dict) else e\n    files += [fname(f) for f in [F.get(\"blink\")] + list(F.get(\"visemes\", {}).values()) +\n              list(F.get(\"expressions\", {}).values()) + list(F.get(\"wear\", {}).values()) if f]\n    tmp = vdir.with_name(version + \".partial\")\n    shutil.rmtree(tmp, ignore_errors=True)\n    layers = {}\n    for rel in files:\n        f = src / rel\n        if not f.exists():\n            raise FileNotFoundError(f\"atlas file missing: {rel}\")\n        im = Image.open(f).convert(\"RGBA\")\n        if im.getbbox() is None:\n            raise ValueError(f\"atlas file is empty (fully transparent): {rel}\")\n        dst = tmp / rel\n        dst.parent.mkdir(parents=True, exist_ok=True)\n        im.save(dst, optimize=True)\n        layers[rel] = {\"file\": rel, \"size\": list(im.size), \"sha256\": hashlib.sha256(dst.read_bytes()).hexdigest()}\n    man = {\"character\": cid, \"version\": version, \"status\": \"provisional\", \"asset_type\": ASSET_TYPE,\n           \"rig_version\": SB.RIG_VERSION, \"art_version\": f\"atlas:{provenance.get('model', '?')}\",\n           \"provenance\": dict(provenance, created=datetime.datetime.now().isoformat(timespec=\"seconds\"),\n                              approved_by=None, approved_at=None),\n           \"identity\": spec.get(\"identity\", {}), \"palette\": spec.get(\"palette\", {}),\n           \"poses\": spec[\"poses\"], \"pose_map\": spec[\"pose_map\"], \"face\": F, \"views\": sorted({p.get(\"view\", \"front\") for p in spec[\"poses\"].values()}),\n           \"expressions\": sorted(F.get(\"expressions\", {})), \"visemes\": sorted(F.get(\"visemes\", {})), \"layers\": layers}\n    man[\"content_sha256\"] = content_sha(man)\n    (tmp / \"manifest.json\").write_text(json.dumps(man, indent=1))\n    tmp.rename(vdir)\n    log(f\"atlas {cid} {version}: {len(layers)} files, {len(spec['poses'])} poses \u2192 {vdir}\")\n    return vdir\n\n\ndef coverage(man):\n    \"\"\"What the atlas provides vs what storybook plans can ask for (for the approval report).\"\"\"\n    F = man.get(\"face\", {})\n    return {\"poses_mapped\": {p: man[\"pose_map\"].get(p) for p in SB.POSES},\n            \"own_pose_images\": sorted({v for v in man[\"pose_map\"].values()}),\n            \"views\": sorted({p.get(\"view\", \"front\") for p in man[\"poses\"].values()}),\n            \"expressions\": {e: e in F.get(\"expressions\", {}) for e in SB.EXPRESSIONS},\n            \"visemes\": {v: (MOUTH_OF_VISEME[v] is None or MOUTH_OF_VISEME[v] in F.get(\"visemes\", {})) for v in SB.VISEMES},\n            \"blink\": bool(F.get(\"blink\"))}\n", "storybook_scene.py": "\"\"\"Storybook scene renderer: rigged characters inside a layered environment pack (drawkit kind \"character\").\n\nSame spec as charanim (world, shot, subject, chars, changes, screen, label, push, _speech, _t0, _dur, _n) plus\noptional storybook fields:\n  \"environment_variant\": a camera preset of the world's pack (else chosen from world + shot + subject)\n  \"camera\": {\"movement\": \"static|slow_push_in|slow_pull_back|pan_left|pan_right\"}  (default: static; push \u2192 push-in)\n  \"_visemes\": {char: [viseme per frame]}  (mouth shapes from that speaker's audio; else from \"_speech\" loudness)\nSelected by plan[\"visual_style_version\"] == \"storybook_v1\"; every other plan keeps the legacy charanim drawing.\nThe plan pins the asset versions (plan[\"character_assets\"], plan[\"environments\"]); a missing or changed asset is\nan error \u2014 never a silent fallback to simple shapes.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\nfrom collections import OrderedDict\n\nimport numpy as np\nfrom PIL import Image\n\nimport envfactory\nimport storybook as SB\n\nSTYLE = \"storybook_v1\"\n_PAINTERS = OrderedDict()\n_BG = OrderedDict()\n\n\ndef assets(plan):\n    root = SB.library_root(plan)\n    if plan.get(\"cinematic_characters\"):\n        import cinematic_workflow as CW\n        CW.validate_plan(plan, root)\n    chars = {}\n    for cid, a in (plan.get(\"character_assets\") or {}).items():\n        strict = a.get(\"status\") == \"approved\"           # approved art: byte-pinned; provisional may be redrawn\n        chars[cid] = SB.load(cid, a[\"version\"], root, expect_sha=a.get(\"manifest_sha256\") if strict else None,\n                             expect_content=a.get(\"content_sha256\") if strict else None)\n    return root, chars\n\n\ndef env_for(plan, world):\n    env = envfactory.WORLD_ENV.get(world)\n    if env is None:\n        raise ValueError(f\"no environment pack for world '{world}' (known: {sorted(envfactory.WORLD_ENV)})\")\n    ver = (plan.get(\"environments\") or {}).get(env)\n    if ver is None:\n        raise ValueError(f\"plan has no version for environment '{env}' (plan['environments'])\")\n    return envfactory.load(env, ver, SB.library_root(plan))\n\n\ndef preset_name(E, spec):\n    p = spec.get(\"environment_variant\") or E.preset_for(spec.get(\"world\", \"studio\"), spec.get(\"shot\", \"two_shot\"),\n                                                         spec.get(\"subject\"))\n    if p is None or p not in E.man[\"presets\"]:\n        raise ValueError(f\"environment {E.id}: no camera preset for world={spec.get('world')} shot={spec.get('shot')} \"\n                         f\"subject={spec.get('subject')}\")\n    return p\n\n\ndef _painter(ch, s):\n    key = (str(ch.dir.resolve()), ch.id, ch.version, ch.man.get(\"content_sha256\"), round(s, 2))\n    if key not in _PAINTERS:\n        _PAINTERS[key] = SB.Painter(ch, round(s, 2))\n        if len(_PAINTERS) > 12:\n            _PAINTERS.popitem(last=False)\n    _PAINTERS.move_to_end(key)\n    return _PAINTERS[key]\n\n\ndef _layer_view(E, name, box, out_size):\n    \"\"\"Crop + scale one master layer to the output (cached for repeated identical boxes).\"\"\"\n    key = (E.id, E.version, name, tuple(round(v, 1) for v in box), out_size)\n    if key in _BG:\n        _BG.move_to_end(key)\n        return _BG[key]\n    x, y, w, h = box\n    im = E.img(name).resize(out_size, Image.BILINEAR, box=(x, y, x + w, y + h))\n    _BG[key] = im\n    if len(_BG) > 60:\n        _BG.popitem(last=False)\n    return im\n\n\ndef _depth_box(base, cam, depth):\n    \"\"\"Parallax: a layer at `depth` moves/zooms `depth` times as much as the camera (1 = with the characters).\"\"\"\n    bx, by, bw, bh = base\n    cx, cy, cw, ch_ = cam\n    bcx, bcy = bx + bw / 2, by + bh / 2\n    ccx, ccy = cx + cw / 2, cy + ch_ / 2\n    zoom = bw / cw\n    z = 1 + (zoom - 1) * depth\n    w, h = bw / z, bh / z\n    mx, my = bcx + (ccx - bcx) * depth, bcy + (ccy - bcy) * depth\n    return (mx - w / 2, my - h / 2, w, h)\n\n\n# ------------------------------------------------------------------------------------------- per-character state\ndef timeline(spec, cid):\n    \"\"\"[(t, gesture, expr)] for one character from chars + changes (in order).\"\"\"\n    st = dict(spec.get(\"chars\", {}).get(cid, {}))\n    out = [(0.0, st.get(\"gesture\", \"idle\"), st.get(\"expr\"))]\n    for c in sorted(spec.get(\"changes\", []), key=lambda c: c[\"t\"]):\n        v = c.get(\"chars\", {}).get(cid)\n        if v:\n            g = v.get(\"gesture\", out[-1][1])\n            e = v.get(\"expr\", out[-1][2])\n            out.append((c[\"t\"], g, e))\n    return out\n\n\ndef state_at(spec, cid, t):\n    \"\"\"Current static state (wear/hold/expr/gesture) and the previous gesture with the change time.\"\"\"\n    st = dict(spec.get(\"chars\", {}).get(cid, {}))\n    since, prev_g = 0.0, st.get(\"gesture\", \"idle\")\n    for c in sorted(spec.get(\"changes\", []), key=lambda c: c[\"t\"]):\n        if c[\"t\"] <= t and cid in c.get(\"chars\", {}):\n            v = c[\"chars\"][cid]\n            if \"gesture\" in v and v[\"gesture\"] != st.get(\"gesture\", \"idle\"):\n                prev_g, since = st.get(\"gesture\", \"idle\"), c[\"t\"]\n            st.update(v)\n    return st, prev_g, since\n\n\ndef resolve_gesture(g, st, other_dx):\n    g = SB.GESTURE_ALIAS.get(g, g) or \"idle\"\n    if g in (\"idle\", \"listen\", \"nod\", \"explain\", \"open_hands\") and st.get(\"hold\") and g != \"explain\":\n        g = \"hold\"\n    pose = dict(SB.pose_of(g))\n    if g == \"point\" and other_dx < 0:                  # point toward the other person / the target on the left\n        pose = {\"L\": dict(SB.POSES[\"point\"][\"R\"]), \"R\": dict(SB.POSES[\"idle\"][\"R\"]), \"lean\": -2}\n    return g, pose\n\n\ndef mouth_for(expr, speaking, vis):\n    base = SB.EXPRESSIONS.get(expr or \"neutral\", SB.EXPRESSIONS[\"neutral\"])[2]\n    if not speaking:\n        return base\n    if vis in (\"rest\",):\n        return \"closed\" if base not in (\"smile\", \"laugh\") else \"smile\"\n    if expr in SB.SMILING and vis in (\"small\", \"wide\"):\n        return \"smile_talk\"\n    return vis\n\n\n# ------------------------------------------------------------------------------------------- the frame\ndef character_frame(spec, plan, t, w, h, bg=None, th=None):\n    import charanim                                     # shared helpers: movie on the screen, label tag\n    dur = float(spec.get(\"_dur\", 4.0))\n    n = max(int(spec.get(\"_n\", 1)), 1)\n    fi = min(int(round(t * (n - 1))), n - 1)\n    t_abs = float(spec.get(\"_t0\", 0.0)) + t * dur\n    root, chars = assets(plan)\n    E = env_for(plan, spec.get(\"world\", \"studio\"))\n    pname = preset_name(E, spec)\n    P = E.man[\"presets\"][pname]\n    S = E.man[\"sets\"][P[\"set\"]]\n    move = (spec.get(\"camera\") or {}).get(\"movement\") or (\"slow_push_in\" if spec.get(\"push\") else \"static\")\n    base = P[\"crop\"]\n    cam = envfactory.camera(P, t, move)\n    view = P.get(\"view\", \"front\")\n    screen = spec.get(\"screen\", \"plain\")\n    for c in sorted(spec.get(\"changes\", []), key=lambda c: c[\"t\"]):   # the screen can change during the shot\n        if c[\"t\"] <= t and c.get(\"screen\"):\n            screen = c[\"screen\"]\n    ids = [c for c in spec.get(\"chars\", {}) if c in chars] or list(chars)\n    layers = S[\"layers\"]\n    k_char = [i for i, (n_, _) in enumerate(layers) if n_ == \"@characters\"][0]\n    near = [i for i, (_, d) in enumerate(layers) if i > k_char and d > 1.0]   # foreground: own parallax\n    # 1. the static plate behind the characters (cached per preset + screen + size)\n    out = _plate(E, pname, P, S, layers[:k_char], screen, w, h).copy()\n    # 2. characters, then the layers between them and the foreground (light, haze) \u2014 at the base framing\n    _draw_characters(out, spec, plan, E, P, S, tuple(base), chars, ids, view, fi, t, t_abs, w, h)\n    for i in range(k_char + 1, len(layers)):\n        if i not in near:\n            out.alpha_composite(_layer_view(E, layers[i][0], tuple(base), (w, h)))\n    # 3. the camera move = one crop of the composed frame; the foreground moves faster (parallax)\n    if move != \"static\":\n        bx, by, bw, bh = base\n        cx, cy, cw, chh = cam\n        kx, ky = w / bw, h / bh\n        out = out.resize((w, h), Image.BILINEAR, box=((cx - bx) * kx, (cy - by) * ky, (cx - bx + cw) * kx, (cy - by + chh) * ky))\n    for i in near:\n        out.alpha_composite(_layer_view(E, layers[i][0], _depth_box(base, cam, layers[i][1]), (w, h)))\n    if spec.get(\"label\"):\n        charanim._tag(out, spec[\"label\"]) if w == 1920 else None\n    return np.asarray(out.convert(\"RGB\"))\n\n\n_PLATES = OrderedDict()\n\n\ndef _plate(E, pname, P, S, layers, screen, w, h):\n    import charanim\n    key = (E.id, E.version, pname, screen, w, h)\n    if key in _PLATES:\n        _PLATES.move_to_end(key)\n        return _PLATES[key]\n    base = tuple(P[\"crop\"])\n    out = Image.new(\"RGBA\", (w, h), (0, 0, 0, 255))\n    for lname, depth in layers:\n        if lname == \"@screen\":\n            sx0, sy0, sx1, sy1 = S[\"screen_rect\"]\n            kx, ky = w / base[2], h / base[3]\n            px0, py0 = int((sx0 - base[0]) * kx), int((sy0 - base[1]) * ky)\n            pw, ph = int((sx1 - sx0) * kx), int((sy1 - sy0) * ky)\n            out.alpha_composite(charanim.movie_frame(max(pw, 2), max(ph, 2), screen).convert(\"RGBA\"), (px0, py0))\n        else:\n            out.alpha_composite(_layer_view(E, lname, base, (w, h)))\n    _PLATES[key] = out\n    if len(_PLATES) > 16:\n        _PLATES.popitem(last=False)\n    return out\n\n\ndef _draw_characters(out, spec, plan, E, P, S, box, chars, ids, view, fi, t, t_abs, w, h):\n    kx = w / box[2]\n    light = S.get(\"lighting\", {})\n    speech = spec.get(\"_speech\", {})\n    vis_track = spec.get(\"_visemes\", {})\n    slots = P[\"slots\"]\n    present = [c for c in ids if c in slots]\n    for cid in present:\n        ch = chars[cid]\n        sx, sy, sc = slots[cid]\n        others = [slots[o][0] for o in present if o != cid] or [slots[o][0] for o in slots if o != cid]\n        other_dx = (others[0] - sx) if others else 0\n        s = sc * kx                                     # output px per design unit\n        if not (-600 * s < (sx - box[0]) * kx < w + 600 * s):\n            continue                                    # outside the frame (e.g. the other friend in a close-up)\n        st, prev_g, since = state_at(spec, cid, t)\n        e = (speech.get(cid) or [0.0])[fi] if speech.get(cid) else 0.0\n        vlist = vis_track.get(cid)\n        vis = vlist[fi] if vlist and fi < len(vlist) else SB.viseme_from_energy(e, \"rest\")\n        speaking = e > 0.06\n        g, pose = resolve_gesture(st.get(\"gesture\", \"idle\"), st, other_dx)\n        age = (t - since) * float(spec.get(\"_dur\", 4.0))\n        if age < SB.BLEND_S and prev_g != st.get(\"gesture\"):\n            _, p0 = resolve_gesture(prev_g, st, other_dx)\n            pose = SB.blend_pose(p0, pose, age / SB.BLEND_S)\n        else:\n            pose = SB.blend_pose(pose, pose, 1.0)\n        if g == \"wave\":                                 # a real wave: the forearm swings\n            pose[\"R\"][\"b\"] += 14 * math.sin(2 * math.pi * 2.2 * t_abs)\n        expr = st.get(\"expr\") or (\"listening\" if not speaking else \"neutral\")\n        if expr not in SB.EXPRESSIONS:\n            expr = \"neutral\"\n        eyes, brows, _, iris = SB.EXPRESSIONS[expr]\n        if SB.blink_closed(cid, t_abs) and eyes not in (\"closed\", \"happy\"):\n            eyes = \"closed\" if eyes != \"wink\" else \"wink\"\n        tilt, breathe = SB.idle_motion(cid, t_abs, speaking)\n        head_dy = breathe * 0.6\n        if g == \"nod\" and age < 1.6:\n            head_dy += 7 * abs(math.sin(math.pi * age / 0.4))\n        turn = 0.0 if view == \"back\" else max(-0.45, min(0.45, 0.35 * (1 if other_dx > 0 else -1 if other_dx < 0 else 0)))\n        gx = 9 if other_dx > 0 else -9 if other_dx < 0 else 0\n        gy = -3 if P.get(\"seated\") and not speaking else 0\n        if expr == \"thinking\":\n            gx, gy = 8 * (1 if other_dx >= 0 else -1), -12\n        face = {\"eyes\": eyes, \"brows\": brows, \"mouth\": mouth_for(expr, speaking, vis), \"gaze\": (gx, gy),\n                \"turn\": round(turn, 2), \"glasses\": \"3d_glasses\" in st.get(\"wear\", []), \"iris\": iris,\n                \"head_tilt\": tilt, \"head_dy\": head_dy}\n        pose[\"head\"] = pose.get(\"head\", 0)\n        # draw the character on its own canvas at a quantized scale, then light it and place it\n        s0 = round(s, 2)\n        cw_, chh = int(1000 * s0), int(1800 * s0)\n        canvas = Image.new(\"RGBA\", (cw_ + 2, chh + 2), (0, 0, 0, 0))\n        hold = st.get(\"hold\") if st.get(\"hold\") in (\"popcorn\", \"3d_glasses\") else None\n        if is_atlas(ch):                                # cinematic artwork: whole poses + face patches\n            _draw_atlas(canvas, ch, s0, st, g, prev_g, age, view, turn, expr, vis, speaking, cid, t_abs, tilt, breathe)\n        else:\n            pnt = _painter(ch, s0)\n            pnt.draw(canvas, (0, 0), pose, face, view=\"back\" if view == \"back\" else \"front\", hold=hold,\n                     legs=not P.get(\"seated\"))\n        tint = light.get(\"tint\")\n        if tint and light.get(\"tint_strength\", 0) > 0:      # match the scene's light colour (C-speed, no numpy)\n            from PIL import ImageChops\n            mul = ImageChops.multiply(canvas, Image.new(\"RGBA\", canvas.size, tuple(tint) + (255,)))\n            a_ = canvas.getchannel(\"A\")\n            canvas = Image.blend(canvas, mul, light[\"tint_strength\"])\n            canvas.putalpha(a_)\n        if abs(s / s0 - 1) > 0.002:\n            canvas = canvas.resize((max(1, int(canvas.width * s / s0)), max(1, int(canvas.height * s / s0))), Image.BILINEAR)\n        ox = (sx - box[0]) * kx - 500 * s\n        oy = (sy - box[1]) * kx - 1800 * s + breathe * s * 0.4\n        if not P.get(\"seated\") and view != \"back\":      # contact shadow on the floor\n            from PIL import ImageDraw, ImageFilter\n            sh = Image.new(\"RGBA\", (int(420 * s), int(80 * s)), (0, 0, 0, 0))\n            ImageDraw.Draw(sh).ellipse([0, 0, sh.width, sh.height], fill=(0, 0, 0, 110))\n            sh = sh.filter(ImageFilter.GaussianBlur(10 * s))\n            out.alpha_composite(sh, (int(ox + 290 * s), int(oy + 1570 * s)))\n        out.alpha_composite(canvas, (int(round(ox)), int(round(oy))))\n\n\ndef is_atlas(ch):\n    return getattr(ch, \"man\", {}).get(\"asset_type\") == \"atlas_v1\"\n\n\n_ATLAS_P = OrderedDict()\n\n\ndef atlas_view(ch, g, view, turn, wear, hold):\n    \"\"\"Keep conversational shots front-facing: atlas_v1 has face patches only for the front view.\n    Auto-selecting a three-quarter still disables mouth/blink animation. Back shots remain explicit.\n    Alternate-view facial animation needs a reviewed per-view face atlas before it can be enabled.\n    \"\"\"\n    return \"back\" if view == \"back\" else \"front\"\n\n\ndef _draw_atlas(canvas, ch, s0, st, g, prev_g, age, view, turn, expr, vis, speaking, cid, t_abs, tilt, breathe):\n    import storybook_atlas as SA\n    wear, hold = tuple(sorted(st.get(\"wear\", []))), st.get(\"hold\")\n    g = SB.GESTURE_ALIAS.get(g, g)\n    vw = atlas_view(ch, g, view, turn, wear, hold)\n    pid = ch.atlas_pose(g, vw, wear, hold)\n    if pid is None:\n        raise ValueError(ch.missing(g, vw, wear, hold))\n    prev = None\n    if age < SB.BLEND_S and prev_g and prev_g != g:\n        pg = SB.GESTURE_ALIAS.get(prev_g, prev_g)\n        prev = ch.atlas_pose(pg, atlas_view(ch, pg, view, turn, wear, hold), wear, hold)\n    key = (str(ch.dir.resolve()), ch.id, ch.version, ch.man.get(\"content_sha256\"), s0)\n    if key not in _ATLAS_P:\n        _ATLAS_P[key] = SA.AtlasPainter(ch, s0)\n        if len(_ATLAS_P) > 8:\n            _ATLAS_P.popitem(last=False)\n    face = {\"expr\": expr, \"viseme\": vis, \"speaking\": speaking, \"wear\": wear,\n            \"blink\": SB.blink_closed(cid, t_abs) and expr not in (\"laughing\", \"wink\")}\n    _ATLAS_P[key].draw(canvas, (0, 0), pid, face, pid_prev=prev, u=age / SB.BLEND_S if prev else 1.0,\n                       motion=(tilt * 0.35, 0.004 * breathe / 2.2))\n\n\n# ------------------------------------------------------------------------------------------- checks\ndef check_spec(spec, plan):\n    \"\"\"Clear errors for anything the storybook renderer can't draw faithfully.\"\"\"\n    out = []\n    try:\n        root, chars = assets(plan)\n    except Exception as e:\n        return [f\"character assets: {e}\"]\n    try:\n        E = env_for(plan, spec.get(\"world\", \"studio\"))\n        preset_name(E, spec)\n    except Exception as e:\n        out.append(str(e))\n    for cid, st in list(spec.get(\"chars\", {}).items()) + [(k, v) for c in spec.get(\"changes\", []) for k, v in c.get(\"chars\", {}).items()]:\n        if cid not in chars:\n            out.append(f\"unknown character '{cid}' (assets: {sorted(chars)})\")\n            continue\n        g = st.get(\"gesture\", \"idle\")\n        if SB.GESTURE_ALIAS.get(g, g) not in SB.POSES:\n            out.append(f\"{cid}: unsupported gesture '{g}' (rig poses: {sorted(SB.POSES)})\")\n        elif is_atlas(chars[cid]):                     # the atlas must hold this exact pose / view / wear / hold\n            try:\n                E_ = env_for(plan, spec.get(\"world\", \"studio\"))\n                back = E_.man[\"presets\"][preset_name(E_, spec)].get(\"view\") == \"back\"\n            except Exception:\n                back = False\n            full = dict(spec.get(\"chars\", {}).get(cid, {}), **st)\n            m = chars[cid].missing(SB.GESTURE_ALIAS.get(g, g), \"back\" if back else \"front\",\n                                   tuple(sorted(full.get(\"wear\", []))), full.get(\"hold\"))\n            if m:\n                out.append(m)\n        if st.get(\"expr\", \"neutral\") not in SB.EXPRESSIONS:\n            out.append(f\"{cid}: unknown expression '{st.get('expr')}'\")\n    mv = (spec.get(\"camera\") or {}).get(\"movement\")\n    if mv and mv not in (\"static\", \"slow_push_in\", \"slow_pull_back\", \"pan_left\", \"pan_right\"):\n        out.append(f\"unknown camera movement '{mv}'\")\n    return out\n", "subtitles.py": "\"\"\"Word-highlighted (\"karaoke\") subtitles as an ASS file, burned in by FFmpeg/libass in the final encode \u2014 no frame\nis drawn in Python. Each word lights up when it is spoken (timed from the speech recogniser's word timestamps);\nnumbers and key terms are emphasised. Short cues, safe margins, sizes relative to the frame (any resolution).\nFor scripts libass may not shape (Telugu), the notebook draws subtitles with Pillow instead (render cell).\n\"\"\"\nfrom __future__ import annotations\n\nimport re\n\n\ndef ass_time(t: float) -> str:\n    cs = max(0, int(round(t * 100)))\n    h, cs = divmod(cs, 360000); m, cs = divmod(cs, 6000); s, cs = divmod(cs, 100)\n    return f\"{h}:{m:02}:{s:02}.{cs:02}\"\n\n\ndef ass_color(hex_rgb: str, alpha: int = 0) -> str:\n    \"\"\"'#RRGGBB' \u2192 ASS &HAABBGGRR.\"\"\"\n    h = hex_rgb.lstrip(\"#\")\n    r, g, b = h[0:2], h[2:4], h[4:6]\n    return f\"&H{alpha:02X}{b}{g}{r}\".upper()\n\n\ndef _esc(w: str) -> str:\n    return w.replace(\"\\\\\", \"\\\\\\\\\").replace(\"{\", \"(\").replace(\"}\", \")\")\n\n\ndef write_ass(path, cues, W, H, font=\"DejaVu Sans\", colors=None, emphasis=(), accent=\"#FFC43D\", dim=\"#BFC6D0\"):\n    \"\"\"cues: [(start, end, [(word, word_start_s), ...], speaker)]. Words already spoken are shown in the speaker's\n    colour (white for the narrator), words still to come slightly dimmed; emphasised words (numbers, key terms)\n    in the accent colour and bold. Returns the number of dialogue lines.\"\"\"\n    colors = colors or {}\n    size, outline, margin = round(H * 0.055), max(2, round(H / 360)), round(H * 0.09)\n    emph = {e.lower() for e in emphasis}\n    head = f\"\"\"[Script Info]\nScriptType: v4.00+\nPlayResX: {W}\nPlayResY: {H}\nWrapStyle: 2\nScaledBorderAndShadow: yes\n\n[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\nStyle: Default,{font},{size},{ass_color('#FFFFFF')},{ass_color(dim)},{ass_color('#000000')},{ass_color('#000000', 0x80)},-1,0,0,0,100,100,0,0,1,{outline},0,2,{round(W * 0.06)},{round(W * 0.06)},{margin},1\n\n[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n\"\"\"\n    lines = []\n    for start, end, words, speaker in cues:\n        if not words:\n            continue\n        col = colors.get(speaker)\n        parts = [f\"{{\\\\1c{ass_color(col)}}}\" if col and col.upper() != \"#FFFFFF\" else \"\"]\n        t_prev = start\n        for i, (w, ws) in enumerate(words):\n            ws = min(max(ws, t_prev), end)\n            if i == 0 and ws > start:                    # silence before the first word: nothing lit yet\n                parts.append(f\"{{\\\\k{round((ws - start) * 100)}}}\")\n            nxt = words[i + 1][1] if i + 1 < len(words) else end\n            dur = max(1, round((max(nxt, ws) - ws) * 100))\n            key = re.sub(r\"[^\\w]\", \"\", w.lower())\n            if key in emph or re.search(r\"\\d\", w):\n                parts.append(f\"{{\\\\k{dur}\\\\1c{ass_color(accent)}\\\\b1}}{_esc(w)}{{\\\\1c{ass_color(col or '#FFFFFF')}}} \")\n            else:\n                parts.append(f\"{{\\\\k{dur}}}{_esc(w)} \")\n            t_prev = ws\n        lines.append(f\"Dialogue: 0,{ass_time(start)},{ass_time(end)},Default,{speaker},0,0,0,,{''.join(parts).rstrip()}\")\n    with open(path, \"w\", encoding=\"utf-8\") as f:\n        f.write(head + \"\\n\".join(lines) + \"\\n\")\n    return len(lines)\n", "telugu_dialogue.py": "\"\"\"Natural spoken Telugu for Arjun and Meera: writer prompt, Telugu critic, cross-language fact check, pronunciation.\n\nWriter: an extension of the Story Director's dialogue stage (story.py); same schema. A factual Telugu turn carries\n`source_text` = the canonical ENGLISH wording of the claim; the English NLI checks only that English text (its\ninputs are genuinely English). Whether the Telugu line SAYS the same thing is a separate, conservative check\n(`localization_check`): numbers, names and the claim's key terms must be present, else UNKNOWN \u2192 manual review.\nAn English-only entailment model is never run on Telugu text.\n\"\"\"\nfrom __future__ import annotations\n\nimport re, unicodedata\n\nTELUGU = re.compile(r\"[\u0c00-\u0c7f]\")\n\nSYSTEM = \"\"\"You are an expert Telugu conversational screenplay writer.\nWrite an educational conversation between {cur} and {exp} that sounds like two real Telugu-speaking friends.\nUse natural spoken Telugu (clear, contemporary, understood in Telangana and Andhra Pradesh) \u2014 not literary Telugu,\nnot a word-for-word translation of English, no exaggerated slang.\n{cur} is curious, observant and friendly: he notices things, predicts, sometimes misunderstands, tries small\nexperiments, remembers earlier points. Not every line of his is a question.\n{exp} is warm, knowledgeable and relaxed: she explains simply, uses everyday examples, asks him to predict, suggests\ndemonstrations, corrects gently. Never a newsreader, never long textbook paragraphs.\nBegin with an interesting observation or situation, build curiosity through actions and discoveries, end with a\nsatisfying realization. No repetitive questions, generic praise, artificial jokes, formal introductions or lectures.\nUse English words only where Telugu speakers naturally do (3D, screen, glasses, projector).\nEvery factual turn must be supported by the given facts: cite fact_ids and give \"source_text\" = the same claim in\nplain English using only those facts. Keep names, numbers, quantities and scientific meaning exactly.\nNo emotion instructions inside the spoken text. Return valid JSON only.\"\"\"\n\n\ndef system_prompt(bible, code_mix=None):\n    \"\"\"The writer's system prompt. code_mix (None = the original mostly-Telugu writer, unchanged) adds the\n    Telugu\u2013English style guide for that mode (CODE_MIX_MODES): short enough for the local Qwen3-4B planner.\"\"\"\n    a, b = bible.by_role(\"curious\"), bible.by_role(\"expert\")\n    out = SYSTEM.format(cur=a.names.get(\"te\", a.name), exp=b.names.get(\"te\", b.name))\n    if code_mix:\n        out = out.replace(\"Use English words only where Telugu speakers naturally do (3D, screen, glasses, projector).\\n\",\n                          \"\") + \"\\n\" + code_mix_prompt(code_mix)\n    return out\n\n\nDIALOGUE_HINT = ('{\"world\": \"...\", \"turns\": [{\"speaker\": \"arjun\", \"text\": \"<spoken Telugu>\", \"kind\": \"observation\", '\n                 '\"fact_ids\": [], \"source_text\": null, \"tone\": \"friendly\"}, {\"speaker\": \"meera\", \"text\": \"<Telugu>\", '\n                 '\"kind\": \"fact\", \"fact_ids\": [\"F1\"], \"source_text\": \"<the same claim in English>\", \"tone\": \"explainer\"}]}')\n\n\n# ------------------------------------------------------------------ Telugu critic\n# bookish / translated vocabulary that sounds like a textbook when spoken (with a spoken alternative)\nFORMAL = {\"\u0c2f\u0c4a\u0c15\u0c4d\u0c15\": \"the possessive ending (\u2026\u0c32\u0c4b / \u2026\u0c15\u0c3f)\", \"\u0c24\u0c4d\u0c30\u0c3f\u0c2e\u0c3f\u0c24\u0c40\u0c2f\": \"3D\", \"\u0c1a\u0c32\u0c28\u0c1a\u0c3f\u0c24\u0c4d\u0c30\u0c02\": \"\u0c38\u0c3f\u0c28\u0c3f\u0c2e\u0c3e\",\n          \"\u0c1a\u0c32\u0c28\u0c1a\u0c3f\u0c24\u0c4d\u0c30\u0c3e\u0c32\u0c41\": \"\u0c38\u0c3f\u0c28\u0c3f\u0c2e\u0c3e\u0c32\u0c41\", \"\u0c05\u0c28\u0c41\u0c2d\u0c42\u0c24\u0c3f\u0c28\u0c3f\": \"\u0c05\u0c28\u0c3f\u0c2a\u0c3f\u0c02\u0c1a\u0c21\u0c02\", \"\u0c17\u0c4d\u0c30\u0c39\u0c3f\u0c38\u0c4d\u0c24\u0c41\u0c02\u0c26\u0c3f\": \"\u0c05\u0c30\u0c4d\u0c25\u0c02 \u0c1a\u0c47\u0c38\u0c41\u0c15\u0c41\u0c02\u0c1f\u0c41\u0c02\u0c26\u0c3f / \u0c24\u0c46\u0c32\u0c41\u0c38\u0c41\u0c15\u0c41\u0c02\u0c1f\u0c41\u0c02\u0c26\u0c3f\",\n          \"\u0c2d\u0c3f\u0c28\u0c4d\u0c28\u0c2e\u0c48\u0c28\": \"\u0c35\u0c47\u0c30\u0c41\u0c35\u0c47\u0c30\u0c41\", \"\u0c15\u0c32\u0c3f\u0c17\u0c3f\u0c38\u0c4d\u0c24\u0c3e\u0c2f\u0c3f\": \"\u0c05\u0c28\u0c3f\u0c2a\u0c3f\u0c38\u0c4d\u0c24\u0c3e\u0c2f\u0c3f\", \"\u0c26\u0c4d\u0c35\u0c3e\u0c30\u0c3e\": \"\u0c24\u0c4b / \u0c28\u0c41\u0c02\u0c1a\u0c3f (often)\",\n          \"\u0c05\u0c28\u0c17\u0c3e\": \"\u0c05\u0c02\u0c1f\u0c47\", \"\u0c15\u0c3e\u0c35\u0c41\u0c28\": \"\u0c05\u0c02\u0c26\u0c41\u0c15\u0c47\", \"\u0c2e\u0c30\u0c3f\u0c2f\u0c41\": \"(join with -\u0c0a / \u0c07\u0c02\u0c15\u0c3e)\", \"\u0c09\u0c2a\u0c2f\u0c4b\u0c17\u0c3f\u0c02\u0c1a\u0c2c\u0c21\u0c41\u0c24\u0c41\u0c02\u0c26\u0c3f\": \"\u0c35\u0c3e\u0c21\u0c24\u0c3e\u0c30\u0c41\",\n          \"\u0c1a\u0c47\u0c2f\u0c2c\u0c21\u0c3f\u0c02\u0c26\u0c3f\": \"\u0c1a\u0c47\u0c36\u0c3e\u0c30\u0c41\", \"\u0c35\u0c40\u0c15\u0c4d\u0c37\u0c3f\u0c02\u0c1a\u0c41\": \"\u0c1a\u0c42\u0c21\u0c41\", \"\u0c2a\u0c4d\u0c30\u0c47\u0c15\u0c4d\u0c37\u0c15\u0c41\u0c32\u0c41\": \"\u0c1c\u0c28\u0c02 / \u0c1a\u0c42\u0c38\u0c47\u0c35\u0c3e\u0c33\u0c4d\u0c32\u0c41 (in dialogue)\",\n          \"\u0c1a\u0c26\u0c41\u0c28\u0c48\u0c28\": \"\u0c2b\u0c4d\u0c32\u0c3e\u0c1f\u0c4d\", \"\u0c26\u0c43\u0c36\u0c4d\u0c2f\u0c02\": \"\u0c15\u0c28\u0c3f\u0c2a\u0c3f\u0c02\u0c1a\u0c47\u0c26\u0c3f / \u0c35\u0c4d\u0c2f\u0c42\", \"\u0c1a\u0c3f\u0c24\u0c4d\u0c30\u0c3e\u0c32\u0c28\u0c41\": \"\u0c2c\u0c4a\u0c2e\u0c4d\u0c2e\u0c32\u0c4d\u0c28\u0c3f\", \"\u0c1a\u0c3f\u0c24\u0c4d\u0c30\u0c3e\u0c32\u0c41\": \"\u0c2c\u0c4a\u0c2e\u0c4d\u0c2e\u0c32\u0c41\",\n          \"\u0c1a\u0c3f\u0c24\u0c4d\u0c30\u0c3e\u0c28\u0c4d\u0c28\u0c3f\": \"\u0c2c\u0c4a\u0c2e\u0c4d\u0c2e\u0c28\u0c3f\", \"\u0c1a\u0c3f\u0c24\u0c4d\u0c30\u0c02\": \"\u0c2c\u0c4a\u0c2e\u0c4d\u0c2e\", \"\u0c26\u0c3f\u0c36\u0c32\": \"\u0c35\u0c48\u0c2a\u0c41\u0c32\", \"\u0c26\u0c3f\u0c36\u0c32\u0c4b\": \"\u0c35\u0c48\u0c2a\u0c41\"}\n# English idioms translated word for word (they don't exist in spoken Telugu)\nCALQUES = {\"\u0c24\u0c46\u0c32\u0c3f\u0c35\u0c48\u0c28 \u0c1c\u0c24\": \"'a clever pair of \u2026'\", \"\u0c38\u0c48\u0c28\u0c4d\u0c38\u0c4d \u0c15\u0c4b\u0c38\u0c02\": \"'for science!'\", \"\u0c28\u0c40 \u0c2a\u0c28\u0c3f\u0c28\u0c3f \u0c1a\u0c47\u0c2f\u0c3f\": \"'do your thing'\",\n           \"\u0c2e\u0c02\u0c1a\u0c3f \u0c2a\u0c4d\u0c30\u0c36\u0c4d\u0c28\": \"'good question'\", \"\u0c30\u0c4b\u0c1c\u0c41 \u0c1a\u0c3f\u0c35\u0c30\u0c32\u0c4b\": \"'at the end of the day'\", \"\u0c06\u0c28\u0c02\u0c26\u0c3f\u0c02\u0c1a\u0c02\u0c21\u0c3f\": \"'enjoy' (formal)\"}\nREACTIONS = (\"\u0c35\u0c3e\u0c35\u0c4d\", \"\u0c05\u0c26\u0c4d\u0c2d\u0c41\u0c24\u0c02\", \"\u0c05\u0c30\u0c47\", \"\u0c28\u0c3f\u0c1c\u0c02\u0c17\u0c3e\u0c28\u0c3e\", \"\u0c13\u0c39\u0c4d\", \"\u0c38\u0c42\u0c2a\u0c30\u0c4d\")\nFORMULAS_TE = (r\"\u0c08 \u0c30\u0c4b\u0c1c\u0c41 \u0c2e\u0c28\u0c02\", r\"\u0c28\u0c47\u0c30\u0c4d\u0c1a\u0c41\u0c15\u0c41\u0c02\u0c26\u0c3e\u0c02\", r\"\u0c1a\u0c3e\u0c32\u0c3e \u0c2e\u0c02\u0c1a\u0c3f \u0c2a\u0c4d\u0c30\u0c36\u0c4d\u0c28\", r\"\u0c2e\u0c40\u0c15\u0c41 \u0c24\u0c46\u0c32\u0c41\u0c38\u0c3e\", r\"\u0c2e\u0c41\u0c17\u0c3f\u0c02\u0c2a\u0c41\u0c32\u0c4b\", r\"\u0c38\u0c4d\u0c35\u0c3e\u0c17\u0c24\u0c02\")\n\n\ndef _words(text):\n    t = \"\".join(ch if unicodedata.category(ch)[0] in \"LMN\" or ch == \"'\" else \" \" for ch in text)\n    return [w for w in t.split() if w]\n\n\ndef english_share(text):\n    ws = _words(text)\n    return sum(1 for w in ws if re.match(r\"^[A-Za-z]\", w)) / max(len(ws), 1)\n\n\ndef critique(st, bible, max_words=24, code_mix=None):\n    \"\"\"Telugu-specific weaknesses with turn ids (story.Weakness-compatible dicts). code_mix=None keeps the original\n    rules exactly (T4 = English share); a code-mix mode replaces T4 with the structural Telugu\u2013English rules\n    (code_mix_critique: English-dominant line / awkward insertion / ...), never a fixed English ratio.\"\"\"\n    out = []\n    turns = [(s, k, t) for s in st.scenes for k, t in enumerate(s.turns)]\n    tid = lambda s, k: f\"{s.id}_{k + 1}\"\n    react = {}\n    for s, k, t in turns:\n        if not TELUGU.search(t.text):\n            out.append({\"where\": tid(s, k), \"rule\": \"T0\", \"problem\": \"not Telugu\", \"evidence\": t.text, \"required\": True,\n                        \"fix\": \"write the line in spoken Telugu\"})\n            continue\n        hits = [w for w in FORMAL if w in t.text]\n        if hits:\n            out.append({\"where\": tid(s, k), \"rule\": \"T1\", \"problem\": \"bookish / translated vocabulary\",\n                        \"evidence\": \", \".join(f\"{h} \u2192 {FORMAL[h]}\" for h in hits), \"required\": True,\n                        \"fix\": \"say it the way friends talk\"})\n        cal = [c for c in CALQUES if c in t.text]\n        if cal:\n            out.append({\"where\": tid(s, k), \"rule\": \"T8\", \"problem\": \"English idiom translated word for word\",\n                        \"evidence\": \", \".join(f\"{c} = {CALQUES[c]}\" for c in cal), \"required\": True,\n                        \"fix\": \"say what a Telugu friend would actually say here\"})\n        for pat in FORMULAS_TE:\n            if re.search(pat, t.text):\n                out.append({\"where\": tid(s, k), \"rule\": \"T2\", \"problem\": \"formulaic phrase\", \"evidence\": t.text,\n                            \"required\": True, \"fix\": \"start from the situation, not an announcement\"})\n        n = len(_words(t.text))\n        if n > max_words:\n            out.append({\"where\": tid(s, k), \"rule\": \"T3\", \"problem\": f\"long speech ({n} words)\", \"evidence\": t.text[:60],\n                        \"required\": t.speaker == bible.by_role(\"expert\").id, \"fix\": \"split it or let Arjun react\"})\n        sh = english_share(t.text)\n        if code_mix is None and sh > 0.35:\n            out.append({\"where\": tid(s, k), \"rule\": \"T4\", \"problem\": f\"too much English ({sh:.0%} of words)\",\n                        \"evidence\": t.text, \"required\": False, \"fix\": \"keep English only for words people really use\"})\n        for r in REACTIONS:\n            if t.text.startswith(r) or f\" {r}\" in t.text:\n                react.setdefault(r, []).append(tid(s, k))\n        if re.search(r\"<[^>]+>|\\(.*?(happily|sadly|laugh|\u0c28\u0c35\u0c4d\u0c35).*?\\)\", t.text):\n            out.append({\"where\": tid(s, k), \"rule\": \"T5\", \"problem\": \"stage direction inside spoken text\",\n                        \"evidence\": t.text, \"required\": True, \"fix\": \"put delivery in the style field, not the text\"})\n    for r, where in react.items():\n        if len(where) >= 2:\n            out.append({\"where\": where[-1], \"rule\": \"T6\", \"problem\": f\"reaction '{r}' repeated\", \"evidence\": str(where),\n                        \"required\": len(where) >= 3, \"fix\": \"react with what he actually noticed\"})\n    starts = [_words(t.text)[0] for _, _, t in turns if t.speaker == bible.by_role(\"curious\").id\n              and t.kind == \"question\" and _words(t.text)]\n    for w0 in set(starts):\n        if starts.count(w0) >= 3:\n            out.append({\"where\": \"story\", \"rule\": \"T7\", \"problem\": f\"Arjun's questions keep starting with '{w0}'\",\n                        \"evidence\": str(starts), \"required\": False, \"fix\": \"vary his rhythm\"})\n    if code_mix is not None:\n        out += code_mix_critique(st, bible, code_mix)\n    return out\n\n\n# ------------------------------------------------------------------ cross-language fact preservation\nTE_NUM = {\"\u0c12\u0c15\": 1, \"\u0c12\u0c15\u0c1f\u0c3f\": 1, \"\u0c30\u0c46\u0c02\u0c21\u0c41\": 2, \"\u0c2e\u0c42\u0c21\u0c41\": 3, \"\u0c28\u0c3e\u0c32\u0c41\u0c17\u0c41\": 4, \"\u0c10\u0c26\u0c41\": 5, \"\u0c06\u0c30\u0c41\": 6, \"\u0c0f\u0c21\u0c41\": 7, \"\u0c0e\u0c28\u0c3f\u0c2e\u0c3f\u0c26\u0c3f\": 8,\n          \"\u0c24\u0c4a\u0c2e\u0c4d\u0c2e\u0c3f\u0c26\u0c3f\": 9, \"\u0c2a\u0c26\u0c3f\": 10, \"\u0c35\u0c02\u0c26\": 100}\nEN_NUM = {\"one\": 1, \"two\": 2, \"three\": 3, \"four\": 4, \"five\": 5, \"six\": 6, \"seven\": 7, \"eight\": 8, \"nine\": 9, \"ten\": 10,\n          \"hundred\": 100, \"single\": 1, \"both\": 2}\n\n\ndef numbers(text, table):\n    out = {int(x) for x in re.findall(r\"\\d+\", text)}\n    for w in _words(text.lower()):\n        for k, v in table.items():\n            if w == k or (TELUGU.search(k) and w.startswith(k)):\n                out.add(v)\n    return out\n\n\ndef localization_check(te_text, en_text, glossary):\n    \"\"\"Does the Telugu line keep the English claim's numbers and key terms? glossary: {english term: [Telugu\n    renderings]}. PASS only when everything required is found; otherwise UNKNOWN (\u2192 manual review), never FAIL-safe\n    approval. Returns {\"status\", \"missing\", \"numbers\"}.\"\"\"\n    missing = []\n    en_l = en_text.lower()\n    for term, te_forms in glossary.items():\n        if re.search(rf\"(?<![a-z]){re.escape(term.lower())}(?![a-z])\", en_l):\n            if not any(f in te_text or (re.search(r\"[A-Za-z]\", f) and f.lower() in te_text.lower()) for f in te_forms):\n                missing.append(term)\n    n_en = numbers(en_text, EN_NUM) - {1, 2}       # \"one/two/both/single\" are too ambiguous across languages\n    n_te = numbers(te_text, TE_NUM) - {1, 2}\n    extra = sorted(n_te - n_en)\n    lost = sorted(n_en - n_te)\n    status = \"PASS\" if not missing and not extra and not lost else \"UNKNOWN\"\n    return {\"status\": status, \"missing_terms\": missing, \"numbers_lost\": lost, \"numbers_added\": extra}\n\n\n# ------------------------------------------------------------------ pronunciation per backend\n# Display text (subtitles) and TTS input are different strings: subtitles keep the natural code-mixed line\n# (\"\u0c07\u0c2a\u0c4d\u0c2a\u0c41\u0c21\u0c41 3D effect \u0c24\u0c17\u0c4d\u0c17\u0c3f\u0c2a\u0c4b\u0c2f\u0c3f\u0c02\u0c26\u0c3f\"), the TTS may get a respelling. Indic Parler is trained on Telugu script, so\n# English words are respelled the way Telugu speakers say them; Gemini gets the form that wins the listening\n# benchmark (TELUGU_PRON_BENCH), until then the display text. Every entry is a CANDIDATE until a native speaker\n# has listened (PRONUNCIATION_STATUS); only the words listed are changed \u2014 nothing is transliterated blindly.\n_PARLER_RESPELL = {\n    \"3D\": \"\u0c24\u0c4d\u0c30\u0c40\u0c21\u0c40\", \"effect\": \"\u0c0e\u0c2b\u0c46\u0c15\u0c4d\u0c1f\u0c4d\", \"screen\": \"\u0c38\u0c4d\u0c15\u0c4d\u0c30\u0c40\u0c28\u0c4d\", \"screens\": \"\u0c38\u0c4d\u0c15\u0c4d\u0c30\u0c40\u0c28\u0c4d\u0c38\u0c4d\", \"movie\": \"\u0c2e\u0c42\u0c35\u0c40\",\n    \"brain\": \"\u0c2c\u0c4d\u0c30\u0c46\u0c2f\u0c3f\u0c28\u0c4d\", \"projector\": \"\u0c2a\u0c4d\u0c30\u0c4a\u0c1c\u0c46\u0c15\u0c4d\u0c1f\u0c30\u0c4d\", \"polarization\": \"\u0c2a\u0c4b\u0c32\u0c30\u0c48\u0c1c\u0c47\u0c37\u0c28\u0c4d\", \"polarizing\": \"\u0c2a\u0c4b\u0c32\u0c30\u0c48\u0c1c\u0c3f\u0c02\u0c17\u0c4d\",\n    \"left eye\": \"\u0c32\u0c46\u0c2b\u0c4d\u0c1f\u0c4d \u0c10\", \"right eye\": \"\u0c30\u0c48\u0c1f\u0c4d \u0c10\", \"left lens\": \"\u0c32\u0c46\u0c2b\u0c4d\u0c1f\u0c4d \u0c32\u0c46\u0c28\u0c4d\u0c38\u0c4d\", \"right lens\": \"\u0c30\u0c48\u0c1f\u0c4d \u0c32\u0c46\u0c28\u0c4d\u0c38\u0c4d\",\n    \"eyes\": \"\u0c10\u0c38\u0c4d\", \"glasses\": \"\u0c17\u0c4d\u0c32\u0c3e\u0c38\u0c46\u0c38\u0c4d\", \"filter\": \"\u0c2b\u0c3f\u0c32\u0c4d\u0c1f\u0c30\u0c4d\", \"filters\": \"\u0c2b\u0c3f\u0c32\u0c4d\u0c1f\u0c30\u0c4d\u0c38\u0c4d\", \"test\": \"\u0c1f\u0c46\u0c38\u0c4d\u0c1f\u0c4d\",\n    \"popcorn\": \"\u0c2a\u0c3e\u0c2a\u0c4d\u200c\u0c15\u0c3e\u0c30\u0c4d\u0c28\u0c4d\", \"view\": \"\u0c35\u0c4d\u0c2f\u0c42\", \"views\": \"\u0c35\u0c4d\u0c2f\u0c42\u0c38\u0c4d\", \"picture\": \"\u0c2a\u0c3f\u0c15\u0c4d\u0c1a\u0c30\u0c4d\", \"pictures\": \"\u0c2a\u0c3f\u0c15\u0c4d\u0c1a\u0c30\u0c4d\u0c38\u0c4d\",\n    \"lens\": \"\u0c32\u0c46\u0c28\u0c4d\u0c38\u0c4d\", \"exactly\": \"\u0c0e\u0c17\u0c4d\u0c1c\u0c3e\u0c15\u0c4d\u0c1f\u0c4d\u200c\u0c32\u0c40\", \"correct\": \"\u0c15\u0c30\u0c46\u0c15\u0c4d\u0c1f\u0c4d\", \"seriously\": \"\u0c38\u0c40\u0c30\u0c3f\u0c2f\u0c38\u0c4d\u200c\u0c32\u0c40\",\n    \"actually\": \"\u0c2f\u0c3e\u0c15\u0c4d\u0c1a\u0c41\u0c35\u0c32\u0c4d\u0c32\u0c40\", \"wait\": \"\u0c35\u0c46\u0c2f\u0c3f\u0c1f\u0c4d\", \"slightly\": \"\u0c38\u0c4d\u0c32\u0c48\u0c1f\u0c4d\u200c\u0c32\u0c40\", \"different\": \"\u0c21\u0c3f\u0c2b\u0c30\u0c46\u0c02\u0c1f\u0c4d\",\n    \"drama\": \"\u0c21\u0c4d\u0c30\u0c3e\u0c2e\u0c3e\", \"scene\": \"\u0c38\u0c40\u0c28\u0c4d\", \"flat\": \"\u0c2b\u0c4d\u0c32\u0c3e\u0c1f\u0c4d\", \"depth\": \"\u0c21\u0c46\u0c2a\u0c4d\u0c24\u0c4d\", \"size\": \"\u0c38\u0c48\u0c1c\u0c4d\", \"real\": \"\u0c30\u0c3f\u0c2f\u0c32\u0c4d\",\n    \"act\": \"\u0c2f\u0c3e\u0c15\u0c4d\u0c1f\u0c4d\", \"miss\": \"\u0c2e\u0c3f\u0c38\u0c4d\", \"combine\": \"\u0c15\u0c02\u0c2c\u0c48\u0c28\u0c4d\", \"separate\": \"\u0c38\u0c46\u0c2a\u0c30\u0c47\u0c1f\u0c4d\", \"technology\": \"\u0c1f\u0c46\u0c15\u0c4d\u0c28\u0c3e\u0c32\u0c1c\u0c40\",\n    \"angle\": \"\u0c2f\u0c3e\u0c02\u0c17\u0c3f\u0c32\u0c4d\", \"angles\": \"\u0c2f\u0c3e\u0c02\u0c17\u0c3f\u0c32\u0c4d\u0c38\u0c4d\", \"clue\": \"\u0c15\u0c4d\u0c32\u0c42\", \"clues\": \"\u0c15\u0c4d\u0c32\u0c42\u0c38\u0c4d\", \"show\": \"\u0c37\u0c4b\", \"simple\": \"\u0c38\u0c3f\u0c02\u0c2a\u0c41\u0c32\u0c4d\",\n    \"feeling\": \"\u0c2b\u0c40\u0c32\u0c3f\u0c02\u0c17\u0c4d\", \"hero\": \"\u0c39\u0c40\u0c30\u0c4b\", \"logic\": \"\u0c32\u0c3e\u0c1c\u0c3f\u0c15\u0c4d\", \"problem\": \"\u0c2a\u0c4d\u0c30\u0c3e\u0c2c\u0c4d\u0c32\u0c2e\u0c4d\", \"try\": \"\u0c1f\u0c4d\u0c30\u0c48\",\n    \"super\": \"\u0c38\u0c42\u0c2a\u0c30\u0c4d\", \"full\": \"\u0c2b\u0c41\u0c32\u0c4d\", \"pass\": \"\u0c2a\u0c3e\u0c38\u0c4d\", \"check\": \"\u0c1a\u0c46\u0c15\u0c4d\", \"confirm\": \"\u0c15\u0c28\u0c4d\u0c2b\u0c30\u0c4d\u0c2e\u0c4d\", \"bro\": \"\u0c2c\u0c4d\u0c30\u0c4b\",\n    \"OK\": \"\u0c13\u0c15\u0c47\", \"okay\": \"\u0c13\u0c15\u0c47\", \"shadow\": \"\u0c37\u0c3e\u0c21\u0c4b\", \"shadows\": \"\u0c37\u0c3e\u0c21\u0c4b\u0c38\u0c4d\", \"TV\": \"\u0c1f\u0c40\u0c35\u0c40\", \"tiny\": \"\u0c1f\u0c48\u0c28\u0c40\",\n    \"mini\": \"\u0c2e\u0c3f\u0c28\u0c40\", \"nose\": \"\u0c28\u0c4b\u0c38\u0c4d\", \"basically\": \"\u0c2c\u0c47\u0c38\u0c3f\u0c15\u0c4d\u200c\u0c17\u0c3e\", \"literally\": \"\u0c32\u0c3f\u0c1f\u0c30\u0c32\u0c4d\u0c32\u0c40\", \"light\": \"\u0c32\u0c48\u0c1f\u0c4d\", \"feel\": \"\u0c2b\u0c40\u0c32\u0c4d\",\n    \"eye\": \"\u0c10\", \"lenses\": \"\u0c32\u0c46\u0c28\u0c4d\u0c38\u0c46\u0c38\u0c4d\",\n}\nPRONUNCIATION = {\"indic_parler\": {\"3D\": \"\u0c24\u0c4d\u0c30\u0c40\u0c21\u0c40\"}, \"gemini_free\": {},\n                 # code-mixed stories (plan \"code_mix\"): the full respelling table for the local voice\n                 \"indic_parler:code_mix\": dict(_PARLER_RESPELL),\n                 # Gemini candidate B for the listening benchmark: the same Telugu-script respelling\n                 \"gemini_free:telugu_script\": dict(_PARLER_RESPELL)}\nPRONUNCIATION_STATUS = {\n    \"indic_parler\": \"adopted for 3D only (earlier Telugu videos)\",\n    \"indic_parler:code_mix\": \"CANDIDATE \u2014 Telugu-script respelling; confirm by listening (TELUGU_PRON_BENCH)\",\n    \"gemini_free\": \"CANDIDATE \u2014 display text as written (Latin English words); confirm by listening\",\n    \"gemini_free:telugu_script\": \"CANDIDATE \u2014 Telugu-script respelling for Gemini; the benchmark decides\",\n}\n\n\ndef _sub(text, table):\n    \"\"\"Longest key first; Latin keys only where a Latin word starts/ends (Telugu endings may follow:\n    'eyes\u0c15\u0c3f' \u2192 '\u0c10\u0c38\u0c4d\u0c15\u0c3f'); case-insensitive for Latin.\"\"\"\n    out = text\n    for w in sorted(table, key=len, reverse=True):\n        say = table[w]\n        if re.search(r\"[A-Za-z0-9]\", w):\n            out = re.sub(rf\"(?<![A-Za-z]){re.escape(w)}(?![A-Za-z])\", lambda m, s=say: s, out, flags=re.I)\n        else:\n            out = out.replace(w, say)\n    return out\n\n\ndef tts_text(text, backend, extra=None):\n    \"\"\"The string sent to the TTS for this backend (subtitles always keep `text`).\"\"\"\n    return _sub(text, {**PRONUNCIATION.get(backend, {}), **(extra or {})})\n\n\ndef latin_left(text, backend, extra=None):\n    \"\"\"English (Latin) words the backend would still receive in Latin letters \u2192 a review note for Parler,\n    whose Telugu training doesn't cover Latin script (add the word to the table after listening).\"\"\"\n    return re.findall(r\"[A-Za-z][A-Za-z']*\", tts_text(text, backend, extra))\n\n\n# ================================================================== Telugu\u2013English code-mixing (Tenglish)\n# Style guide (docs/16_TENGLISH_STYLE_GUIDE.md has the long version and the research behind it):\n#  * Telugu is the grammatical base (matrix language): Telugu verbs, case endings, word order; a sentence ends in\n#    Telugu (verb / copula / particle) \u2014 \"3D effect \u0c24\u0c17\u0c4d\u0c17\u0c3f\u0c2a\u0c4b\u0c2f\u0c3f\u0c02\u0c26\u0c3f\", \"correct \u0c15\u0c26\u0c3e?\".\n#  * English supplies content words friends actually say in English (movie, screen, glasses, test, effect, brain,\n#    filter, projector, different) and a few discourse reactions (actually, wait, seriously, exactly, correct).\n#  * English verbs enter through Telugu light verbs: \"try \u0c1a\u0c47\u0c26\u0c4d\u0c26\u0c3e\u0c02\", \"combine \u0c1a\u0c47\u0c38\u0c4d\u0c24\u0c41\u0c02\u0c26\u0c3f\", \"miss \u0c05\u0c35\u0c41\u0c24\u0c41\u0c02\u0c26\u0c3f\";\n#    English nouns take Telugu endings: \"glasses\u0c32\u0c4b\", \"eyes\u0c15\u0c3f\", \"views\u0c28\u0c3f\".\n#  * English words are written in English letters in the display text; Telugu-script respelling is a TTS concern.\n#  * Not: whole English sentences, English function words inside a Telugu sentence (\"the\", \"is\", \"because\"),\n#    bookish Telugu, word-for-word idioms, forced slang, an English word in every line, academic English\n#    (\"perceive\", \"stereoscopic\") where a friend would say it simply. Some reactions stay fully Telugu.\nCODE_MIX_MODES = {\n    \"light\":   {\"label\": \"mostly Telugu with occasional English\", \"en_content_max\": 2,\n                \"note\": \"Mostly Telugu: at most one or two common English words in a line; many lines fully Telugu.\"},\n    \"natural\": {\"label\": \"balanced, natural Telugu\u2013English\", \"en_content_max\": 4,\n                \"note\": \"Balanced: English for everyday terms and a few reactions, Telugu for everything else.\"},\n    \"casual\":  {\"label\": \"urban casual Telugu\u2013English\", \"en_content_max\": 6,\n                \"note\": \"Urban casual: English terms and reactions are frequent, but the grammar stays Telugu.\"},\n}\n_GUIDE = \"\"\"Telugu\u2013English code-mixing ({label}). {note}\n- Telugu grammar always: Telugu verbs, case endings and word order; every sentence ends in Telugu.\n- English only for words friends really say in English (movie, screen, glasses, test, effect, brain, filter,\n  projector, different, actually, wait, seriously, exactly, correct). Write them in English letters.\n- English verbs with Telugu helpers: \"try \u0c1a\u0c47\u0c26\u0c4d\u0c26\u0c3e\u0c02\", \"combine \u0c1a\u0c47\u0c38\u0c4d\u0c24\u0c41\u0c02\u0c26\u0c3f\". English nouns take Telugu endings: \"glasses\u0c32\u0c4b\".\n- Never: a full English sentence, English \"the/is/of/because\" inside Telugu, bookish Telugu, translated idioms,\n  forced slang, English in every line. Some reactions stay fully Telugu (\"\u0c05\u0c2c\u0c4d\u0c2c\u0c3e!\", \"\u0c05\u0c35\u0c41\u0c28\u0c3e?\").\nGood: \"\u0c05\u0c30\u0c47! \u0c07\u0c2a\u0c4d\u0c2a\u0c41\u0c21\u0c41 3D effect \u0c24\u0c17\u0c4d\u0c17\u0c3f\u0c2a\u0c4b\u0c2f\u0c3f\u0c02\u0c26\u0c3f.\" | \"\u0c12\u0c15 \u0c1a\u0c3f\u0c28\u0c4d\u0c28 test \u0c1a\u0c47\u0c26\u0c4d\u0c26\u0c3e\u0c02.\" | \"Correct! \u0c2a\u0c4d\u0c30\u0c24\u0c3f lens\u0c15\u0c3f \u0c35\u0c47\u0c30\u0c47 filter \u0c09\u0c02\u0c1f\u0c41\u0c02\u0c26\u0c3f.\"\nBad: \"Actually the projector is showing two images.\" | \"\u0c24\u0c4d\u0c30\u0c3f\u0c2e\u0c3f\u0c24\u0c40\u0c2f \u0c1a\u0c32\u0c28\u0c1a\u0c3f\u0c24\u0c4d\u0c30\u0c02\" | \"Bro, literally full crazy vibe!\\\"\"\"\"\n\n\ndef code_mix_prompt(mode):\n    m = CODE_MIX_MODES[mode]\n    return _GUIDE.format(label=m[\"label\"], note=m[\"note\"])\n\n\n# English function words: inside a Telugu sentence they mean an English clause was pasted in (awkward insertion)\nEN_FUNCTION = {\"the\", \"a\", \"an\", \"is\", \"are\", \"was\", \"were\", \"am\", \"be\", \"been\", \"of\", \"to\", \"in\", \"on\", \"at\", \"for\",\n               \"with\", \"from\", \"by\", \"and\", \"or\", \"it\", \"its\", \"this\", \"that\", \"these\", \"those\", \"i\", \"you\", \"he\",\n               \"she\", \"we\", \"they\", \"my\", \"your\", \"his\", \"her\", \"our\", \"their\", \"me\", \"him\", \"us\", \"them\", \"do\",\n               \"does\", \"did\", \"will\", \"would\", \"can\", \"could\", \"should\", \"have\", \"has\", \"had\", \"not\", \"very\",\n               \"also\", \"because\", \"then\", \"than\", \"what\", \"why\", \"how\", \"when\", \"where\", \"which\", \"who\", \"just\",\n               \"about\", \"into\"}\n# discourse words Telugu speakers really use in English (not counted as awkward)\nEN_DISCOURSE = {\"so\", \"but\", \"ok\", \"okay\", \"actually\", \"basically\", \"wait\", \"seriously\", \"exactly\", \"correct\",\n                \"right\", \"sorry\", \"please\", \"bro\", \"super\", \"simple\", \"yes\", \"no\", \"literally\"}\n# academic English where a friend says it simply\nEN_ACADEMIC = {\"perceive\": \"feel \u0c05\u0c35\u0c41\u0c24\u0c41\u0c02\u0c26\u0c3f / \u0c24\u0c46\u0c32\u0c41\u0c38\u0c4d\u0c24\u0c41\u0c02\u0c26\u0c3f\", \"perception\": \"feeling\", \"stereoscopic\": \"3D\",\n               \"binocular\": \"\u0c30\u0c46\u0c02\u0c21\u0c41 \u0c15\u0c33\u0c4d\u0c32\u0c24\u0c4b\", \"monocular\": \"\u0c12\u0c15\u0c4d\u0c15 \u0c15\u0c28\u0c4d\u0c28\u0c41\u0c24\u0c4b\", \"retinal\": \"\u0c15\u0c02\u0c1f\u0c3f\u0c32\u0c4b\",\n               \"superimposed\": \"\u0c12\u0c15\u0c26\u0c3e\u0c28\u0c3f \u0c2e\u0c40\u0c26 \u0c12\u0c15\u0c1f\u0c3f\", \"designated\": \"\u0c26\u0c3e\u0c28\u0c3f\", \"visual angle\": \"size\",\n               \"polarized\": \"filter \u0c09\u0c28\u0c4d\u0c28\"}\n# English words written in Telugu letters: in a code-mixed display line they read better in English letters\nTRANSLITERATED_EN = {\"\u0c17\u0c4d\u0c32\u0c3e\u0c38\u0c46\u0c38\u0c4d\": \"glasses\", \"\u0c38\u0c4d\u0c15\u0c4d\u0c30\u0c40\u0c28\u0c4d\": \"screen\", \"\u0c1f\u0c46\u0c38\u0c4d\u0c1f\u0c4d\": \"test\", \"\u0c0e\u0c2b\u0c46\u0c15\u0c4d\u0c1f\u0c4d\": \"effect\",\n                     \"\u0c2a\u0c4d\u0c30\u0c4a\u0c1c\u0c46\u0c15\u0c4d\u0c1f\u0c30\u0c4d\": \"projector\", \"\u0c2b\u0c3f\u0c32\u0c4d\u0c1f\u0c30\u0c4d\": \"filter\", \"\u0c2a\u0c3e\u0c2a\u0c4d\u200c\u0c15\u0c3e\u0c30\u0c4d\u0c28\u0c4d\": \"popcorn\", \"\u0c32\u0c46\u0c28\u0c4d\u0c38\u0c4d\": \"lens\",\n                     \"\u0c2b\u0c4d\u0c32\u0c3e\u0c1f\u0c4d\": \"flat\", \"\u0c2b\u0c4d\u0c30\u0c47\u0c2e\u0c4d\": \"frame\", \"\u0c15\u0c30\u0c46\u0c15\u0c4d\u0c1f\u0c4d\": \"correct\", \"\u0c17\u0c46\u0c38\u0c4d\": \"guess\"}\nREACTIONS_EN = (\"exactly\", \"seriously\", \"actually\", \"wait\", \"correct\", \"super\", \"wow\", \"ok\", \"okay\")\n\n\ndef token_lang(w):\n    \"\"\"'te' | 'en' | 'mix' (English stem + Telugu ending: 'eyes\u0c15\u0c3f') | 'num'.\"\"\"\n    te, la = bool(TELUGU.search(w)), bool(re.search(r\"[A-Za-z]\", w))\n    if te and la:\n        return \"mix\"\n    if la:\n        return \"en\"\n    return \"te\" if te else \"num\"\n\n\ndef _sentences(text):\n    return [x for x in re.split(r\"(?<=[.!?\u2026])\\s+|\\.\\.\\.\", text) if _words(x)]\n\n\ndef line_profile(text):\n    \"\"\"Structure of one line: Telugu / English / mixed tokens, English function words, the longest run of\n    English words, whether each sentence ends in Telugu (Telugu grammar) \u2014 no ratio threshold involved.\"\"\"\n    ws = _words(text)\n    langs = [token_lang(w) for w in ws]\n    run = best = 0\n    for g in langs:\n        run = run + 1 if g == \"en\" else 0\n        best = max(best, run)\n    sents = []\n    for snt in _sentences(text):\n        sw = _words(snt)\n        sl = [token_lang(w) for w in sw]\n        fn = [w for w, g in zip(sw, sl) if g == \"en\" and w.lower() in EN_FUNCTION]\n        sents.append({\"text\": snt, \"n\": len(sw), \"ends_te\": sl[-1] in (\"te\", \"mix\"), \"fn\": fn,\n                      \"te\": sum(g in (\"te\", \"mix\") for g in sl), \"en\": sum(g == \"en\" for g in sl)})\n    en_words = [w for w, g in zip(ws, langs) if g in (\"en\", \"mix\")]\n    return {\"n\": len(ws), \"te\": langs.count(\"te\"), \"en\": langs.count(\"en\"), \"mix\": langs.count(\"mix\"),\n            \"en_run\": best, \"sentences\": sents,\n            \"en_content\": [w for w in en_words if re.sub(r\"[\u0c00-\u0c7f]\", \"\", w).lower() not in EN_FUNCTION | EN_DISCOURSE],\n            \"en_function\": [w for s_ in sents for w in s_[\"fn\"]], \"has_en\": bool(en_words)}\n\n\ndef english_dominant(text):\n    \"\"\"A sentence that is no longer Telugu: English grammar (an English function word AND an English ending), a run\n    of 6+ English words, or no Telugu at all in a sentence of 3+ words. Returns the offending sentences.\"\"\"\n    bad = []\n    for s_ in line_profile(text)[\"sentences\"]:\n        if s_[\"n\"] >= 3 and (s_[\"te\"] == 0 or (s_[\"fn\"] and not s_[\"ends_te\"])):\n            bad.append(s_[\"text\"])\n    p = line_profile(text)\n    if p[\"en_run\"] >= 6 and not bad:\n        bad.append(text)\n    return bad\n\n\ndef code_mix_critique(st, bible, mode):\n    \"\"\"The code-mixing rules (story.Weakness-compatible dicts):\n    T4  English-dominant sentence (required): the line stopped being Telugu.\n    T9  awkward insertion: English function words inside a Telugu sentence (required from 2, else a warning).\n    T10 English crowding for this mode (warning): more English content words in one line than the mode suggests.\n    T11 forced / missing mixing over the story (warnings): English in every line; or almost none in 'natural'/'casual'.\n    T12 English written in Telugu letters in the display text (warning; not in 'light' mode).\n    T13 academic English where a friend would say it simply (warning).\n    T6  (extended) the same English reaction repeated.\"\"\"\n    m = CODE_MIX_MODES[mode]\n    out, tid = [], (lambda s, k: f\"{s.id}_{k + 1}\")\n    turns = [(s, k, t) for s in st.scenes for k, t in enumerate(s.turns)]\n    react = {}\n    for s, k, t in turns:\n        if not TELUGU.search(t.text):\n            continue                                   # T0 already reports it\n        p = line_profile(t.text)\n        dom = english_dominant(t.text)\n        if dom:\n            out.append({\"where\": tid(s, k), \"rule\": \"T4\", \"problem\": \"English sentence, not Telugu with English words\",\n                        \"evidence\": \" | \".join(dom)[:120], \"required\": True,\n                        \"fix\": \"keep Telugu grammar: Telugu verb at the end, English only for the key words\"})\n        fn = [w for s_ in p[\"sentences\"] if s_[\"ends_te\"] for w in s_[\"fn\"]]\n        if fn:\n            out.append({\"where\": tid(s, k), \"rule\": \"T9\", \"problem\": \"awkward English insertion (function words)\",\n                        \"evidence\": \", \".join(fn), \"required\": len(fn) >= 2,\n                        \"fix\": \"say the small words in Telugu (the/is/of/because \u2192 -\u0c28\u0c3f, \u0c09\u0c02\u0c26\u0c3f, -\u0c32\u0c4b, \u0c0e\u0c02\u0c26\u0c41\u0c15\u0c02\u0c1f\u0c47)\"})\n        if len(p[\"en_content\"]) > m[\"en_content_max\"]:\n            out.append({\"where\": tid(s, k), \"rule\": \"T10\", \"problem\": f\"{len(p['en_content'])} English words in one line \"\n                        f\"(mode '{mode}': up to ~{m['en_content_max']})\", \"evidence\": \", \".join(p[\"en_content\"]),\n                        \"required\": False, \"fix\": \"keep English for the words friends really say in English\"})\n        tr = [f\"{w} \u2192 {e}\" for w, e in TRANSLITERATED_EN.items() if w in t.text]\n        if tr and mode != \"light\":                     # mostly-Telugu style may keep loanwords in Telugu letters\n            out.append({\"where\": tid(s, k), \"rule\": \"T12\", \"problem\": \"English word written in Telugu letters\",\n                        \"evidence\": \", \".join(tr), \"required\": False,\n                        \"fix\": \"write it in English in the display text (the TTS respelling is separate)\"})\n        ac = [f\"{w} \u2192 {e}\" for w, e in EN_ACADEMIC.items() if re.search(rf\"(?<![A-Za-z]){w}\", t.text, re.I)]\n        if ac:\n            out.append({\"where\": tid(s, k), \"rule\": \"T13\", \"problem\": \"academic English\", \"evidence\": \", \".join(ac),\n                        \"required\": False, \"fix\": \"say it the way a friend explains it\"})\n        first = _words(t.text)[:1]\n        for r in REACTIONS_EN:\n            if first and first[0].lower() == r:\n                react.setdefault(r, []).append(tid(s, k))\n    for r, where in react.items():\n        if len(where) >= 2:\n            out.append({\"where\": where[-1], \"rule\": \"T6\", \"problem\": f\"reaction '{r}' repeated\", \"evidence\": str(where),\n                        \"required\": len(where) >= 3, \"fix\": \"react with what was actually noticed\"})\n    lines = [t for _, _, t in turns if TELUGU.search(t.text)]\n    with_en = [t for t in lines if line_profile(t.text)[\"has_en\"]]\n    if lines and len(with_en) == len(lines) and mode != \"casual\":\n        out.append({\"where\": \"story\", \"rule\": \"T11\", \"problem\": \"English forced into every line\", \"evidence\":\n                    f\"{len(with_en)}/{len(lines)} lines\", \"required\": False,\n                    \"fix\": \"let some reactions stay fully Telugu\"})\n    if lines and mode in (\"natural\", \"casual\") and len(with_en) < 0.25 * len(lines):\n        out.append({\"where\": \"story\", \"rule\": \"T11\", \"problem\": \"hardly any code-mixing for this mode\", \"evidence\":\n                    f\"{len(with_en)}/{len(lines)} lines have an English word\", \"required\": False,\n                    \"fix\": \"use the English words friends really say (screen, test, effect, glasses...)\"})\n    return out\n\n\n# ------------------------------------------------------------------ the six-way evaluation (candidates A/B/C)\nCRITERIA = (\"naturalness\", \"clarity\", \"personality\", \"humour\", \"code_mixing\", \"science\")\nSCIENCE_POINTS = {        # brief \u00a73: what the explanation must contain, as the fact ids that support each point\n    \"two different views for the two eyes\": {\"F1\"},\n    \"the projector/glasses separate the views\": {\"F3\", \"F4\", \"F5\"},\n    \"the brain combines them into depth\": {\"F2\"},\n    \"one eye weakens stereo depth, other depth cues remain\": {\"F7\"},\n}\n\n\ndef evaluate(st, bible, mode, glossary=None):\n    \"\"\"Deterministic 0\u20135 proxies for the six criteria + the evidence behind each (a person still judges humour and\n    naturalness: these numbers only catch what is measurable). Returns {\"scores\", \"evidence\", \"issues\"}.\"\"\"\n    cur, exp = bible.by_role(\"curious\").id, bible.by_role(\"expert\").id\n    iss = critique(st, bible, code_mix=mode)\n    req = [w for w in iss if w[\"required\"]]\n    turns = [t for s in st.scenes for t in s.turns]\n    nat_bad = [w for w in iss if w[\"rule\"] in (\"T1\", \"T2\", \"T8\", \"T9\", \"T13\", \"T4\", \"T12\")]\n    words = [len(_words(t.text)) for t in turns]\n    long_ = [w for w in iss if w[\"rule\"] == \"T3\"]\n    ak = [t.kind for t in turns if t.speaker == cur]\n    mk = [t.kind for t in turns if t.speaker == exp]\n    guesses = sum(k in (\"misconception\", \"prediction\", \"hypothetical\") for k in ak)\n    tease = sum(k == \"reaction\" for k in mk)\n    pop0 = \"popcorn\" in \" \".join(t.text for t in st.scenes[0].turns).lower() or \"\u0c2a\u0c3e\u0c2a\u0c4d\u200c\u0c15\u0c3e\u0c30\u0c4d\u0c28\u0c4d\" in st.scenes[0].turns[0].text\n    pop1 = \"popcorn\" in \" \".join(t.text for t in st.scenes[-1].turns).lower() or \\\n        \"\u0c2a\u0c3e\u0c2a\u0c4d\u200c\u0c15\u0c3e\u0c30\u0c4d\u0c28\u0c4d\" in \" \".join(t.text for t in st.scenes[-1].turns)\n    reacts = [_words(t.text)[0].lower() for t in turns if t.kind == \"reaction\" and _words(t.text)]\n    prof = [line_profile(t.text) for t in turns]\n    mix_lines = sum(p[\"has_en\"] for p in prof)\n    te_only = sum(not p[\"has_en\"] for p in prof)\n    mix_bad = [w for w in iss if w[\"rule\"] in (\"T4\", \"T9\", \"T10\", \"T11\", \"T12\")]\n    cited = {f for t in turns for f in t.fact_ids}\n    covered = [k for k, ids in SCIENCE_POINTS.items() if ids & cited]\n    loc = {}\n    for s in st.scenes:\n        for k, t in enumerate(s.turns):\n            if t.fact_ids:\n                loc[f\"{s.id}_{k + 1}\"] = localization_check(t.text, t.source_text or \"\", glossary or {})[\"status\"] \\\n                    if t.source_text else \"UNKNOWN\"\n    clamp = lambda x: max(0, min(5, round(x, 1)))\n    scores = {\n        \"naturalness\": clamp(5 - sum(1.0 if w[\"required\"] else 0.25 for w in nat_bad)),\n        \"clarity\": clamp(5 - 1.0 * len(long_) - max(0, (sum(words) / max(len(words), 1)) - 12) * 0.3),\n        \"personality\": clamp(2 + min(guesses, 2) + min(tease, 1) - (ak.count(\"question\") / max(len(ak), 1) > 0.6)),\n        \"humour\": clamp(1 + 2 * pop0 + 2 * pop1 - (len(reacts) - len(set(reacts)))),\n        \"code_mixing\": clamp((5 if 0 < te_only < len(turns) else 3) - sum(1.0 if w[\"required\"] else 0.25 for w in mix_bad)),\n        \"science\": clamp(5 * len(covered) / len(SCIENCE_POINTS) - sum(v != \"PASS\" for v in loc.values())),\n    }\n    ev = {\"naturalness\": [f\"{w['rule']} {w['where']}: {w['problem']}\" for w in nat_bad],\n          \"clarity\": f\"{sum(words)} words, {sum(words) / max(len(words), 1):.1f} per turn, {len(long_)} long turns\",\n          \"personality\": f\"Arjun: {guesses} guesses/predictions, kinds {ak}; Meera: {tease} teasing reactions\",\n          \"humour\": f\"popcorn opening {pop0}, popcorn callback {pop1}, reaction openers {reacts}\",\n          \"code_mixing\": f\"{mix_lines} lines with English, {te_only} fully Telugu; \" +\n                         \"; \".join(f\"{w['rule']} {w['where']}\" for w in mix_bad),\n          \"science\": f\"points covered {len(covered)}/{len(SCIENCE_POINTS)}: {covered}; localization {loc}\"}\n    return {\"scores\": scores, \"evidence\": ev, \"issues\": iss, \"required\": req}\n\n\n# LLM rubric (optional, local Qwen3-4B): it can only ADD weaknesses; it never approves a story or a claim.\nRUBRIC_PROMPT = \"\"\"Rate this Telugu\u2013English dialogue between two friends (1\u20135 each) and list concrete problems.\nCriteria: naturalness (do Telugu friends talk like this?), clarity, personality (Arjun curious and funny,\nMeera relaxed and teasing), humour, code_mixing ({label}: Telugu grammar, natural English words only),\nscience (only what the cited facts say).\nReturn JSON: {{\"scores\": {{\"naturalness\": n, \"clarity\": n, \"personality\": n, \"humour\": n, \"code_mixing\": n,\n\"science\": n}}, \"problems\": [{{\"where\": \"sc01_2\", \"problem\": \"...\", \"fix\": \"...\"}}]}}\nDialogue:\n{dialogue}\"\"\"\n\n\ndef rubric_prompt(st, bible, mode):\n    lines = [f\"{s.id}_{k + 1} {bible.characters[t.speaker].name}: {t.text}\" for s in st.scenes\n             for k, t in enumerate(s.turns)]\n    return RUBRIC_PROMPT.format(label=CODE_MIX_MODES[mode][\"label\"], dialogue=\"\\n\".join(lines))\n\n\ndef llm_rubric(st, bible, mode, llm, ask_json):\n    \"\"\"The optional LLM opinion as extra (non-required) weaknesses; scores are reported, never used to approve.\"\"\"\n    from pydantic import BaseModel, Field\n\n    class Problem(BaseModel):\n        where: str = \"story\"\n        problem: str\n        fix: str = \"\"\n\n    class Rubric(BaseModel):\n        scores: dict = Field(default_factory=dict)\n        problems: list[Problem] = Field(default_factory=list)\n\n    r = ask_json(llm, \"You are a strict Telugu dialogue editor. Return valid JSON only.\",\n                 rubric_prompt(st, bible, mode), Rubric)\n    return {\"scores\": r.scores, \"issues\": [{\"where\": p.where, \"rule\": \"T-LLM\", \"problem\": p.problem, \"evidence\": \"\",\n                                            \"required\": False, \"fix\": p.fix} for p in r.problems]}\n\n\n# ------------------------------------------------------------------ speech check for code-mixed lines\n# Whisper writes an English word either in Latin (\"effect\") or in Telugu letters (\"\u0c0e\u0c2b\u0c46\u0c15\u0c4d\u0c1f\u0c41\"), and confuses Telugu\n# vowel lengths. Both sides are brought to one representation before comparing: known English words \u2192 their\n# Telugu-script respelling, then letters only; a consonant skeleton (vowel signs dropped) shows when the\n# \"error\" is only vowel noise. A bad score is a REVIEW flag for a person, never proof the voice is wrong.\n_VOWEL_SIGNS = re.compile(r\"[\u0c3e-\u0c4d\u0c55\u0c56\u0c01-\u0c03\u0c05-\u0c14]\")\n\n\ndef bilingual_key(text):\n    import turn_align\n    return turn_align.letters(_sub(text, _PARLER_RESPELL))\n\n\ndef _err(e, h):\n    import difflib\n    if not e:\n        return 0.0\n    m = sum(b.size for b in difflib.SequenceMatcher(None, e, h, autojunk=False).get_matching_blocks())\n    return round(1 - m / len(e), 3)\n\n\ndef compare_speech(expected, heard, threshold=0.35):\n    \"\"\"\u2192 {\"err\", \"err_skeleton\", \"status\": PASS | REVIEW | NO_ASR, \"why\"} for a code-mixed line.\"\"\"\n    if heard is None:\n        return {\"err\": None, \"err_skeleton\": None, \"status\": \"NO_ASR\", \"why\": \"no recogniser result\"}\n    e, h = bilingual_key(expected), bilingual_key(heard)\n    err = _err(e, h)\n    sk = _err(_VOWEL_SIGNS.sub(\"\", e), _VOWEL_SIGNS.sub(\"\", h))\n    if err <= threshold:\n        return {\"err\": err, \"err_skeleton\": sk, \"status\": \"PASS\", \"why\": \"\"}\n    if sk <= threshold:\n        return {\"err\": err, \"err_skeleton\": sk, \"status\": \"PASS\",\n                \"why\": \"consonants match; the difference is vowel-level recogniser noise\"}\n    return {\"err\": err, \"err_skeleton\": sk, \"status\": \"REVIEW\",\n            \"why\": \"the recogniser disagrees: a person listens (Whisper's Telugu/English mixing is weak)\"}\n\n\ndef bilingual_error(expected, heard):\n    \"\"\"Drop-in error for code-mixed lines: the smaller of the normalized letter error and the skeleton error.\"\"\"\n    r = compare_speech(expected, heard)\n    return None if r[\"err\"] is None else min(r[\"err\"], r[\"err_skeleton\"])\n", "trackers.py": "\"\"\"Trackers: turn sampled frames into observed boxes per entity, for motion_check.py.\n\nAll trackers return {entity: [(box, score) or None per frame]} with boxes normalised to 0..1.\nAssociation never looks at the PLANNED positions (that would bias the check towards passing): the first frame\ntakes the best-scoring detection, later frames take the detection that continues the previous box.\n\n- ColorTracker: exact colour blobs. Deterministic; used by the synthetic regression tests.\n- Owlv2Tracker: open-vocabulary detector (google/owlv2-base-patch16-ensemble, Apache-2.0, ~0.6 GB) queried with each\n  entity's short description. Runs on the T4 after the video model is unloaded.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef _link(cands_per_frame):\n    \"\"\"cands_per_frame: list of [(box, score)] -> one (box, score) or None per frame, by continuity.\"\"\"\n    out, prev = [], None\n    for cands in cands_per_frame:\n        if not cands:\n            out.append(None)\n            continue\n        if prev is None:\n            pick = max(cands, key=lambda c: c[1])\n        else:\n            pc = ((prev[0] + prev[2]) / 2, (prev[1] + prev[3]) / 2)\n            pick = min(cands, key=lambda c: abs((c[0][0] + c[0][2]) / 2 - pc[0]) + abs((c[0][1] + c[0][3]) / 2 - pc[1])\n                       - 0.05 * c[1])\n        out.append(pick)\n        prev = pick[0]\n    return out\n\n\nclass ColorTracker:\n    \"\"\"Finds pixels of an exact RGB colour per entity (synthetic test videos).\"\"\"\n    def __init__(self, colors: dict[str, tuple], tol: int = 10):\n        self.colors, self.tol = colors, tol\n\n    def track(self, frames: np.ndarray, queries=None):\n        n, h, w = frames.shape[:3]\n        out = {}\n        for name, col in self.colors.items():\n            seq = []\n            for f in frames:\n                m = (np.abs(f.astype(int) - np.array(col)).sum(axis=2) <= self.tol)\n                if m.sum() < 4:\n                    seq.append([])\n                    continue\n                ys, xs = np.nonzero(m)\n                seq.append([((xs.min() / w, ys.min() / h, (xs.max() + 1) / w, (ys.max() + 1) / h), 1.0)])\n            out[name] = _link(seq)\n        return out\n\n\ndef load_on(cls, mid, device, **kw):\n    \"\"\"from_pretrained straight onto the GPU (weights never sit in system RAM: the free Colab tier has ~12.7 GB);\n    on the CPU, or if that fails, one weight at a time through RAM (low_cpu_mem_usage) and then moved.\"\"\"\n    if device == \"cuda\":\n        try:\n            return cls.from_pretrained(mid, device_map={\"\": 0}, **kw).eval()\n        except Exception:\n            pass\n    return cls.from_pretrained(mid, low_cpu_mem_usage=True, **kw).to(device).eval()\n\n\nclass Owlv2Tracker:\n    \"\"\"Open-vocabulary detection on sampled frames, linked over time by continuity.\"\"\"\n    MODEL = \"google/owlv2-base-patch16-ensemble\"\n\n    def __init__(self, device=\"cuda\", cache_dir=None, threshold=0.12):\n        import torch\n        from transformers import Owlv2ForObjectDetection, Owlv2Processor\n        self.torch, self.device, self.th = torch, device, threshold\n        self.dt = torch.float16 if device == \"cuda\" else torch.float32\n        self.proc = Owlv2Processor.from_pretrained(self.MODEL, cache_dir=cache_dir)\n        self.model = load_on(Owlv2ForObjectDetection, self.MODEL, device, torch_dtype=self.dt, cache_dir=cache_dir)\n\n    def track(self, frames: np.ndarray, queries: dict[str, str]):\n        \"\"\"queries: {entity: \"a young woman in a teal hoodie\"}. Returns {entity: [(box, score) or None]}.\"\"\"\n        from PIL import Image\n        torch = self.torch\n        names = list(queries)\n        per = {k: [] for k in names}\n        for f in frames:\n            img = Image.fromarray(f)\n            inp = self.proc(text=[[queries[k] for k in names]], images=img, return_tensors=\"pt\")\n            inp = {k: (v.to(self.device, self.dt) if v.dtype.is_floating_point else v.to(self.device))\n                   for k, v in inp.items()}\n            with torch.no_grad():\n                out = self.model(**inp)\n            side = max(img.size)                    # OWLv2 pads to a square: boxes are relative to that square\n            post = getattr(self.proc, \"post_process_grounded_object_detection\", None) or \\\n                self.proc.post_process_object_detection           # renamed in transformers 5.x\n            res = post(out, threshold=self.th, target_sizes=torch.tensor([[side, side]]))[0]\n            w, h = img.size\n            for k in names:\n                per[k].append([])\n            for box, score, lab in zip(res[\"boxes\"].float().cpu().numpy(), res[\"scores\"].float().cpu().numpy(),\n                                       res[\"labels\"].cpu().numpy()):\n                x0, y0, x1, y1 = box\n                per[names[int(lab)]][-1].append(((max(x0, 0) / w, max(y0, 0) / h, min(x1, w) / w, min(y1, h) / h),\n                                                 float(score)))\n        return {k: _link(v) for k, v in per.items()}\n", "turn_align.py": "\"\"\"Split one multi-speaker TTS waveform back into the plan's dialogue turns (who speaks when).\n\nThe 2D characters need per-turn audio: who is speaking, when each turn starts and ends, and each turn's own\nloudness for the mouth. A two-speaker Gemini request returns ONE waveform, so:\n  1. pauses: 20 ms frames, silence = > 35 dB under the loud parts, \u2265 120 ms long;\n  2. expected cut points: from recogniser word times mapped onto the transcript (letters, difflib) when a speech\n     recogniser is available, else from each turn's share of the letters;\n  3. each cut snaps to the longest pause near its expected point; the cut is the middle of that pause;\n  4. checks: every cut found, cuts in order, every turn's duration plausible for its length (seconds per letter).\nAnything that fails is UNRESOLVED (\u2192 smaller request / per-turn synthesis / local voice). Nothing is invented: no\ntimestamp is ever placed where the audio has no pause.\n\"\"\"\nfrom __future__ import annotations\n\nimport difflib, re, unicodedata\n\nimport numpy as np\n\nFRAME_S = 0.02\n\n\ndef letters(text):\n    \"\"\"Letters only (any script; Telugu vowel signs kept), lower-case: what speech carries.\"\"\"\n    return \"\".join(ch for ch in text.lower() if unicodedata.category(ch)[0] in \"LM\" or ch.isdigit())\n\n\ndef quietest(wav, sr, t, win=0.15):\n    \"\"\"The quietest 20 ms point within \u00b1win s of t (where to cut when two speakers barely pause).\"\"\"\n    n = int(sr * FRAME_S)\n    a, b = max(int((t - win) * sr) // n, 0), min(int((t + win) * sr) // n, len(wav) // n - 1)\n    if b <= a:\n        return t\n    rms = [np.sqrt(np.mean(wav[i * n:(i + 1) * n] ** 2)) for i in range(a, b + 1)]\n    return (a + int(np.argmin(rms)) + 0.5) * FRAME_S\n\n\ndef pauses(wav, sr, min_s=0.08, drop_db=35.0):\n    \"\"\"[(start_s, end_s)] of silences inside the speech (not the lead-in / tail).\"\"\"\n    n = int(sr * FRAME_S)\n    k = len(wav) // n\n    if k < 3:\n        return []\n    rms = np.sqrt((wav[: k * n].reshape(k, n) ** 2).mean(1) + 1e-12)\n    db = 20 * np.log10(rms)\n    loud = np.percentile(db, 95)\n    quiet = db < loud - drop_db\n    voiced = np.where(~quiet)[0]\n    if not len(voiced):\n        return []\n    a, b = voiced[0], voiced[-1]\n    out, i = [], a\n    while i <= b:\n        if quiet[i]:\n            j = i\n            while j <= b and quiet[j]:\n                j += 1\n            if (j - i) * FRAME_S >= min_s:\n                out.append((i * FRAME_S, j * FRAME_S))\n            i = j\n        else:\n            i += 1\n    return out\n\n\ndef speech_span(wav, sr, drop_db=35.0):\n    n = int(sr * FRAME_S); k = len(wav) // n\n    db = 20 * np.log10(np.sqrt((wav[: k * n].reshape(k, n) ** 2).mean(1) + 1e-12))\n    v = np.where(db >= np.percentile(db, 95) - drop_db)[0]\n    return (v[0] * FRAME_S, (v[-1] + 1) * FRAME_S) if len(v) else (0.0, len(wav) / sr)\n\n\ndef expected_cuts(turn_texts, span, words=None):\n    \"\"\"Where each boundary should be. With recogniser words [[word, start, end]], from the words matched to each\n    turn's letters; else proportional to letters (Telugu letters \u2248 speaking time). Returns (cuts, source).\"\"\"\n    L = [letters(t) for t in turn_texts]\n    if words:\n        heard = [letters(w[0]) for w in words]\n        joined, owner = \"\", []\n        for i, h in enumerate(heard):\n            joined += h; owner += [i] * len(h)\n        target, turn_of = \"\", []\n        for k, x in enumerate(L):\n            target += x; turn_of += [k] * len(x)\n        sm = difflib.SequenceMatcher(None, target, joined, autojunk=False)\n        last_word = {}                                       # turn \u2192 last heard word matched inside it\n        first_word = {}\n        for a, b, size in sm.get_matching_blocks():\n            for j in range(size):\n                t_, w_ = turn_of[a + j], owner[b + j]\n                last_word[t_] = max(last_word.get(t_, -1), w_)\n                first_word[t_] = min(first_word.get(t_, 10 ** 9), w_)\n        cuts = []\n        for k in range(len(L) - 1):\n            if k in last_word and (k + 1) in first_word and last_word[k] < first_word[k + 1]:\n                cuts.append((words[last_word[k]][2] + words[first_word[k + 1]][1]) / 2)\n            else:\n                cuts.append(None)\n        if all(c is not None for c in cuts):\n            return cuts, \"recogniser\"\n    tot = sum(len(x) for x in L) or 1\n    s0, s1 = span\n    acc, cuts = 0, []\n    for x in L[:-1]:\n        acc += len(x); cuts.append(s0 + (s1 - s0) * acc / tot)\n    return cuts, \"letters\"\n\n\ndef align(wav, sr, turn_texts, words=None, sec_per_char=(0.03, 0.25), tol=None):\n    \"\"\"\u2192 {\"status\": \"PASS\"|\"UNRESOLVED\", \"spans\": [(start, end)] per turn, \"why\", \"source\"}.\"\"\"\n    if len(turn_texts) == 1:\n        s0, s1 = speech_span(wav, sr)\n        return {\"status\": \"PASS\", \"spans\": [(max(s0 - 0.05, 0), min(s1 + 0.05, len(wav) / sr))], \"why\": \"\", \"source\": \"single\"}\n    span = speech_span(wav, sr)\n    ps = pauses(wav, sr)\n    exp, src = expected_cuts(turn_texts, span, words)\n    tol = tol or max(0.6, 0.12 * (span[1] - span[0]) / max(len(turn_texts) - 1, 1))\n    cuts, used, prev, no_pause = [], set(), span[0], []\n    for k, e in enumerate(exp):\n        cands = [(i, p) for i, p in enumerate(ps) if i not in used and p[0] > prev and abs((p[0] + p[1]) / 2 - e) <= tol]\n        if not cands:\n            if src == \"recogniser\" and e > prev:\n                # quick turn-taking (no real pause): the recogniser's word times say where the speaker changes;\n                # cut at the quietest point there. The speaker check afterwards confirms who is talking.\n                q = quietest(wav, sr, e)\n                if q > prev:\n                    cuts.append((q, q)); prev = q; no_pause.append(k + 1)\n                    continue\n            return {\"status\": \"UNRESOLVED\", \"spans\": [], \"source\": src,\n                    \"why\": f\"no pause near the expected boundary {k + 1} ({e:.2f}s) \u2014 speaker change can't be placed\"}\n        i, p = max(cands, key=lambda c: (c[1][1] - c[1][0]) - 0.5 * abs((c[1][0] + c[1][1]) / 2 - e))\n        used.add(i); cuts.append(p); prev = p[1]\n    spans, start = [], max(span[0] - 0.05, 0.0)\n    for p in cuts:\n        pad = 0.03 if p[1] > p[0] else 0.0                 # no padding across a pause-less cut\n        spans.append((start, p[0] + pad)); start = p[1] - pad\n    spans.append((start, min(span[1] + 0.05, len(wav) / sr)))\n    for (a, b), t in zip(spans, turn_texts):\n        n = max(len(letters(t)), 1)\n        if not sec_per_char[0] <= (b - a) / n <= sec_per_char[1]:\n            return {\"status\": \"UNRESOLVED\", \"spans\": spans, \"source\": src,\n                    \"why\": f\"turn '{t[:30]}\u2026' would last {b - a:.2f}s for {n} letters (implausible)\"}\n    return {\"status\": \"PASS\", \"spans\": spans, \"source\": src, \"why\": \"\",\n            \"gaps\": [round(c[1] - c[0], 3) for c in cuts], \"cuts_without_pause\": no_pause}\n\n\ndef cut(wav, sr, spans, fade_s=0.008):\n    \"\"\"Per-turn pieces, with tiny edge fades (no clicks); silences inside the pieces trimmed by the spans.\"\"\"\n    out = []\n    for a, b in spans:\n        x = wav[int(a * sr):int(b * sr)].astype(np.float32).copy()\n        f = min(int(sr * fade_s), len(x) // 2)\n        if f:\n            r = np.linspace(0, 1, f, dtype=np.float32); x[:f] *= r; x[-f:] *= r[::-1]\n        out.append(x)\n    return out\n", "validate.py": "\"\"\"Scene validation: checks a video plan BEFORE any voice or GPU work.\n\nPure Python (no Colab, no GPU) so it is unit-tested locally (tests/test_validate.py).\nvalidate(plan, settings) -> list of issues; each issue is a dict:\n    {\"level\": \"error\" | \"warn\", \"rule\": \"P2\", \"shot\": \"sh05\", \"message\": \"...\", \"fix\": \"...\"}\nErrors stop the run before anything is generated (the plan must be fixed); warnings are reported.\n\nRule groups (the letters are used in reports and in the documentation):\n  S  storytelling / engagement   (research: Mayer's multimedia principles, Guo et al. 2014)\n  P  process and factual accuracy (claims with sources, conditional steps, numbers, invented internals)\n  A  narration \u2194 visual alignment (what is said must be what is shown)\n  V  visual consistency           (characters, props, places, time)\n\"\"\"\nimport re\n\n# ---------------------------------------------------------------- vocab\nHEDGES = {\"if\", \"may\", \"might\", \"some\", \"sometimes\", \"depending\", \"depends\", \"usually\", \"can\", \"could\",\n          \"often\", \"eligible\", \"unless\", \"when\", \"either\", \"or\", \"most\", \"many\", \"certain\", \"where\"}\n# Telugu: conditionals and \"may/can\" are word ENDINGS (\u0c09\u0c02\u0c1f\u0c47 = \"if it is\", \u0c1c\u0c30\u0c17\u0c4a\u0c1a\u0c4d\u0c1a\u0c41 = \"may happen\"), plus a few words\nTE_HEDGE_WORDS = {\"\u0c12\u0c15\u0c35\u0c47\u0c33\", \"\u0c15\u0c4a\u0c28\u0c4d\u0c28\u0c3f\u0c38\u0c3e\u0c30\u0c4d\u0c32\u0c41\", \"\u0c38\u0c3e\u0c27\u0c3e\u0c30\u0c23\u0c02\u0c17\u0c3e\", \"\u0c2c\u0c39\u0c41\u0c36\u0c3e\", \"\u0c32\u0c47\u0c26\u0c3e\", \"\u0c15\u0c4a\u0c28\u0c4d\u0c28\u0c3f\"}\nTE_HEDGE_ENDINGS = (\"\u0c24\u0c47\", \"\u0c1f\u0c47\", \"\u0c1a\u0c4d\u0c1a\u0c41\")\n\n\ndef hedged(text):\n    \"\"\"Does the narration word a conditional fact conditionally? (English words, or Telugu words / endings)\"\"\"\n    ws = set(words(text))\n    return bool((content_words(text) | ws) & HEDGES or ws & TE_HEDGE_WORDS\n                or any(w.endswith(TE_HEDGE_ENDINGS) for w in ws))\nTIME_UNITS = (r\"(seconds?|minutes?|hours?|days?|business days?|working days?|weeks?|months?\"\n              r\"|\u0c38\u0c46\u0c15\u0c28\u0c4d|\u0c28\u0c3f\u0c2e\u0c3f\u0c37|\u0c17\u0c02\u0c1f|\u0c30\u0c4b\u0c1c\u0c41|\u0c30\u0c4b\u0c1c\u0c41\u0c32|\u0c35\u0c3e\u0c30\u0c02|\u0c35\u0c3e\u0c30\u0c3e\u0c32|\u0c28\u0c46\u0c32)\")      # + Telugu stems (suffixes follow: \u0c30\u0c4b\u0c1c\u0c41\u0c32 \u0c2e\u0c41\u0c02\u0c26\u0c41)\nNUMBER_WORDS = {\"one\": 1, \"two\": 2, \"three\": 3, \"four\": 4, \"five\": 5, \"six\": 6, \"seven\": 7, \"eight\": 8, \"nine\": 9,\n                \"ten\": 10, \"eleven\": 11, \"twelve\": 12, \"fourteen\": 14, \"fifteen\": 15, \"twenty\": 20, \"thirty\": 30}\n# phrases that claim knowledge of a company's internal systems; allowed only with a cited claim\nINTERNAL_DETAIL = [\"algorithm\", \"database\", \"server\", \"servers\", \"machine learning\", \"ai model\", \"microservice\",\n                   \"internal system\", \"backend\", \"kafka\", \"redis\", \"api\", \"queue\", \"neural\"]\n# words a non-technical viewer may not know; each must be in the plan's glossary (explained with an analogy)\nJARGON = [\"acquirer\", \"issuer\", \"issuing bank\", \"card network\", \"payment gateway\", \"payment processor\",\n          \"settlement\", \"clearing\", \"authorization\", \"authorisation\", \"chargeback\", \"ledger\", \"ach\", \"neft\",\n          \"upi\", \"arn\", \"acquirer reference number\", \"reversal\", \"batch\", \"interchange\", \"api\", \"webhook\",\n          \"idempotency\", \"database\", \"microservice\", \"transaction id\"]\nSTOP = set(\"\"\"a an the and or but if then so of to in on at by for with from as is are was were be been being it its\nthis that these those you your yours he she they them his her their we our us i me my not no yes do does did done\ncan could may might will would shall should just only also very more most some any each every all into out up down\nover under again there here when where why how what which who whom whose than too now then still yet even\"\"\".split())\n\nDRAWN = {\"diagram\", \"ui\", \"journey\", \"card\", \"story\", \"character\", \"demo\"}\n\n\nimport unicodedata\n# a phrase can't start right after a letter/digit of any script or an Indic vowel sign (marks aren't \\w in Python)\n_NOT_START = r\"[\\w\\u0300-\\u036F\\u0900-\\u0DFF]\"\n\n\ndef words(text):\n    \"\"\"Words of any script (\"Riya's\" \u2192 riya's; Telugu keeps its vowel signs).\"\"\"\n    t = \"\".join(ch if ch == \"'\" or unicodedata.category(ch)[0] in \"LMN\" else \" \" for ch in text.lower())\n    return [w for w in t.split() if w.strip(\"'\")]\n\n\ndef content_words(text):\n    return {w.strip(\"'\") for w in words(text) if w not in STOP and len(w) > 2}\n\n\ndef numbers_in(text):\n    \"\"\"Numbers used with a time unit, e.g. '3-5 business days' -> {3, 5}; 'ten business days' -> {10}.\"\"\"\n    t = text.lower()\n    found = set()\n    for m in re.finditer(r\"(\\d+)\\s*(?:-|\u2013|to)\\s*(\\d+)\\s*\" + TIME_UNITS, t):\n        found |= {int(m.group(1)), int(m.group(2))}\n    for m in re.finditer(r\"(\\d+)\\s*\" + TIME_UNITS, t):\n        found.add(int(m.group(1)))\n    for w, n in NUMBER_WORDS.items():\n        if re.search(rf\"\\b{w}\\b\\s*\" + TIME_UNITS, t):\n            found.add(n)\n    return found\n\n\ndef all_numbers(text):\n    t = text.lower()\n    return {int(x) for x in re.findall(r\"\\d+\", t)} | {n for w, n in NUMBER_WORDS.items() if re.search(rf\"\\b{w}\\b\", t)}\n\n\ndef has_phrase(text, phrase):\n    \"\"\"Whole-word match. Telugu attaches case endings to words (PNR\u0c32\u0c4b = \"in the PNR\", \u0c30\u0c3f\u0c2f\u0c3e\u0c15\u0c3f = \"to Riya\"), so after a\n    phrase only a following LATIN letter/digit breaks the match (English \"car\" still doesn't match \"cart\").\"\"\"\n    phrase = phrase.lower()\n    right = r\"(?![a-z0-9])\" if phrase[-1:].isascii() else \"\"\n    return re.search(rf\"(?<!{_NOT_START}){re.escape(phrase)}{right}\", text.lower()) is not None\n\n\ndef visual_text(shot, plan):\n    \"\"\"Everything the viewer SEES in a shot, as text (for alignment checks).\"\"\"\n    parts = [shot.get(\"scene\", \"\"), shot.get(\"action\", \"\"), shot.get(\"shows\", \"\"), shot.get(\"label\", \"\"),\n             shot.get(\"title\", \"\")]\n    chars = plan.get(\"characters\", {})\n    for c in shot.get(\"cast\", []):\n        parts += [chars.get(c, {}).get(\"look\") or \"\", chars.get(c, {}).get(\"tag\") or \"\", chars.get(c, {}).get(\"name\", \"\")]\n    props = plan.get(\"props\", {})\n    for p in shot.get(\"props\", []):\n        parts += [props.get(p, {}).get(\"look\", \"\"), p]\n    for key in (\"ui\", \"journey\", \"card\", \"diagram\", \"stock\", \"character\", \"demo\"):\n        if key in shot:\n            parts.append(_flatten(shot[key]))\n    for v in (shot.get(\"story\") or {}).get(\"visuals\", []):        # two-friend scenes: every visual of the scene\n        parts += [v.get(\"description\", \"\"), v.get(\"intent\", \"\"), _flatten(v.get(v.get(\"recipe\"), {}))]\n        if v.get(\"recipe\") == \"demo\":\n            parts.append(DEMO_SHOWS.get(v[\"demo\"].get(\"template\"), \"\"))\n    if shot.get(\"recipe\") == \"journey\":\n        st = {s[\"id\"]: s for s in plan.get(\"journey\", {}).get(\"stations\", [])}\n        for sid in shot.get(\"journey\", {}).get(\"path\", []):\n            parts.append(st.get(sid, {}).get(\"label\", \"\") + \" \" + st.get(sid, {}).get(\"icon\", \"\"))\n    return \" \".join(parts)\n\n\n# what each drawn demonstration shows (demos.py), for the narration \u2194 visual check\nDEMO_SHOWS = {\"finger_test\": \"thumb finger hand left eye right eye close one eye jumps near far window\",\n              \"eye_views\": \"two eyes left eye right eye two pictures ball tree brain one picture depth\",\n              \"projector_glasses\": \"projector screen two pictures light glasses lenses filter left eye right eye \"\n                                   \"polarized polarization twist\"}\n\n\ndef _flatten(x):\n    if isinstance(x, dict):\n        return \" \".join(_flatten(v) for v in x.values())\n    if isinstance(x, list):\n        return \" \".join(_flatten(v) for v in x)\n    return str(x)\n\n\ndef spoken(shot):\n    return \" \".join(l.get(\"text\", \"\") for l in shot.get(\"lines\", []))\n\n\ndef entity_hits(text, entities):\n    \"\"\"Which plan entities (with synonyms) a text mentions.\"\"\"\n    out = set()\n    for name, syns in entities.items():\n        if any(has_phrase(text, s) for s in [name] + list(syns)):\n            out.add(name)\n    return out\n\n\n# ---------------------------------------------------------------- the checks\ndef validate(plan, settings=None):\n    s = {\"words_per_second\": 2.6, \"max_sentence_words\": 22, \"max_label_words\": 6, \"max_title_words\": 9,\n         \"max_diagram_share\": 0.25, \"max_diagram_run\": 2, \"max_same_visual_run\": 3, \"hook_seconds\": 10,\n         \"wpm_range\": (125, 185)}\n    s.update(plan.get(\"rules\", {}))\n    s.update(settings or {})\n    issues = []\n\n    def add(level, rule, shot, msg, fix=\"\"):\n        issues.append({\"level\": level, \"rule\": rule, \"shot\": shot, \"message\": msg, \"fix\": fix})\n\n    shots = plan.get(\"shots\", [])\n    if not shots:\n        add(\"error\", \"S0\", None, \"the plan has no shots\")\n        return issues\n    ids = [x[\"shot_id\"] for x in shots]\n    claims = {c[\"id\"]: c for c in plan.get(\"claims\", [])}\n    steps = {st[\"id\"]: st for st in plan.get(\"process\", [])}\n    step_order = [st[\"id\"] for st in plan.get(\"process\", [])]\n    entities = plan.get(\"entities\", {})\n    glossary = {k.lower(): v for k, v in plan.get(\"glossary\", {}).items()}\n    company = plan.get(\"real_company\")                      # topic is a real company: no invented internals\n\n    # ---- S: storytelling and engagement\n    first_lines = \" \".join(spoken(x) for x in shots[:2])\n    if \"?\" not in first_lines and not shots[0].get(\"hook\"):\n        add(\"warn\", \"S1\", ids[0], \"the first two shots don't pose a question (no hook)\",\n            \"open with the viewer's problem as a question, e.g. 'So why is the money not there yet?'\")\n    recipes = [x.get(\"recipe\", \"video\") for x in shots]\n    n_dg = sum(r == \"diagram\" for r in recipes)\n    if n_dg / len(shots) > s[\"max_diagram_share\"]:\n        add(\"error\", \"S2\", None, f\"{n_dg}/{len(shots)} shots are box-and-arrow diagrams (max \"\n            f\"{s['max_diagram_share']:.0%}): the video becomes a flowchart presentation\",\n            \"show the process with people, UI screens and the journey recipe instead\")\n    run = 0\n    for x, r in zip(shots, recipes):\n        run = run + 1 if r == \"diagram\" else 0\n        if run > s[\"max_diagram_run\"]:\n            add(\"error\", \"S2\", x[\"shot_id\"], f\"{run} diagram shots in a row\", \"break the run with a story or UI shot\")\n    same, prev = 1, None\n    for x in shots:\n        key = (x.get(\"recipe\", \"video\"), x.get(\"location\"),       # story scenes: also their own cuts\n               tuple(v.get(\"intent\") for v in (x.get(\"story\") or {}).get(\"visuals\", [])))\n        same = same + 1 if key == prev else 1\n        prev = key\n        if same > s[\"max_same_visual_run\"]:\n            add(\"warn\", \"S3\", x[\"shot_id\"], f\"{same} shots in a row look the same ({key[0]} at {key[1]})\",\n                \"change the angle, the place or the shot type to keep attention\")\n    for x in shots:\n        for l in x.get(\"lines\", []):\n            for sent in re.split(r\"(?<=[.!?])\\s+\", l.get(\"text\", \"\")):\n                n = len(words(sent))\n                if n > s[\"max_sentence_words\"]:\n                    add(\"error\", \"S4\", x[\"shot_id\"], f\"sentence of {n} words (max {s['max_sentence_words']}): '{sent[:60]}...'\",\n                        \"split it: one idea per sentence\")\n        if len(words(x.get(\"label\", \"\"))) > s[\"max_label_words\"]:\n            add(\"warn\", \"S5\", x[\"shot_id\"], f\"step label has {len(words(x['label']))} words (max {s['max_label_words']})\",\n                \"shorten it (on-screen text competes with narration)\")\n        if len(words(x.get(\"title\", \"\"))) > s[\"max_title_words\"]:\n            add(\"warn\", \"S5\", x[\"shot_id\"], \"title is long\", \"keep titles under ~9 words\")\n        txt = spoken(x).lower()\n        for term in JARGON:\n            if has_phrase(txt, term) and term not in glossary:\n                add(\"error\", \"S6\", x[\"shot_id\"], f\"jargon '{term}' is not in the plan's glossary\",\n                    \"say it in everyday words, or add it to 'glossary' with a simple analogy\")\n    seen_terms = set()\n    for x in shots:                                        # a glossary term must be explained where it first appears\n        txt = spoken(x).lower()\n        for term, g in glossary.items():\n            if has_phrase(txt, term) and term not in seen_terms:\n                seen_terms.add(term)\n                if not (x.get(\"explains\") == term or (x.get(\"explains\") or []) and term in x.get(\"explains\")):\n                    add(\"warn\", \"S6\", x[\"shot_id\"], f\"'{term}' first appears here without being explained\",\n                        f\"mark this shot \\\"explains\\\": \\\"{term}\\\" and give the analogy ({g})\")\n    if shots[-1].get(\"beat\") not in (\"payoff\", \"recap\"):\n        add(\"warn\", \"S7\", ids[-1], \"the last shot is not marked as the payoff/recap\",\n            \"end with the answer to the opening question (beat: \\\"payoff\\\")\")\n\n    # ---- P: process and factual accuracy\n    pos = -1\n    covered = set()\n    for x in shots:\n        st = x.get(\"step\")\n        if st is None:\n            continue\n        if st not in steps:\n            add(\"error\", \"P1\", x[\"shot_id\"], f\"unknown process step '{st}'\", f\"use one of {step_order}\")\n            continue\n        covered.add(st)\n        i = step_order.index(st)\n        if i < pos and not x.get(\"flashback\"):\n            add(\"error\", \"P1\", x[\"shot_id\"], f\"step '{st}' comes after a later step: the process runs backwards\",\n                \"reorder the shots, or mark the shot \\\"flashback\\\": true\")\n        pos = max(pos, i)\n        if steps[st].get(\"conditional\"):\n            if not hedged(spoken(x)) and not x.get(\"lines\") == []:\n                add(\"error\", \"P2\", x[\"shot_id\"], f\"step '{st}' only happens sometimes \"\n                    f\"({steps[st].get('condition', 'conditional')}) but the narration states it as always true\",\n                    \"word it conditionally: 'if\u2026', 'for some items\u2026', 'usually\u2026', 'depending on\u2026'\")\n    for st in step_order:\n        if not steps[st].get(\"conditional\") and not steps[st].get(\"optional\") and st not in covered:\n            add(\"error\", \"P3\", None, f\"required process step '{st}' ({steps[st].get('text', '')}) is never shown\",\n                \"add a shot for it, or don't call it a required step\")\n    for c in claims.values():\n        if not any(str(u).startswith(\"https://\") for u in c.get(\"sources\", [])):\n            add(\"error\", \"P4\", None, f\"claim {c['id']} has no source URL\", \"add the page where this fact is stated\")\n    for x in shots:\n        cited = x.get(\"claims\", [])\n        for cid in cited:\n            if cid not in claims:\n                add(\"error\", \"P4\", x[\"shot_id\"], f\"cites unknown claim '{cid}'\")\n        cited_text = \" \".join(claims[c][\"text\"] for c in cited if c in claims)\n        for l in x.get(\"lines\", []):\n            factual = l.get(\"speaker\") == \"narrator\" and not l.get(\"story\") and x.get(\"step\") is not None\n            # two-friend explainers: the friends' factual lines ARE the explanation, so they need sources too\n            friend_fact = bool(plan.get(\"factual_dialogue\")) and l.get(\"kind\") in (\"fact\", \"correction\")\n            if friend_fact:\n                for cid in l.get(\"claims\", []):\n                    if cid not in claims:\n                        add(\"error\", \"P4\", x[\"shot_id\"], f\"line cites unknown claim '{cid}'\")\n                if not l.get(\"claims\"):\n                    add(\"error\", \"P4\", x[\"shot_id\"], f\"a factual line cites no claim: '{l.get('text', '')[:60]}'\",\n                        \"add \\\"claims\\\": [ids] to the line, or make it a question / reaction / analogy\")\n                line_cited = \" \".join(claims[c][\"text\"] for c in l.get(\"claims\", []) if c in claims)\n                nums = {int(n) for n in re.findall(r\"\\d+\", l.get(\"text\", \"\"))}\n                if nums and not nums <= all_numbers(line_cited):\n                    add(\"error\", \"P5\", x[\"shot_id\"], f\"number(s) {sorted(nums)} are not in the line's cited claims\",\n                        \"only use numbers that appear in a sourced claim\")\n                continue\n            if factual and not cited:\n                add(\"error\", \"P4\", x[\"shot_id\"], \"a factual narrator line cites no claim\",\n                    \"add \\\"claims\\\": [ids] for the facts it states (or mark the line \\\"story\\\": true)\")\n            nums = numbers_in(l.get(\"text\", \"\")) if l.get(\"speaker\") == \"narrator\" else set()   # dialogue is fiction\n            if nums and not nums <= all_numbers(cited_text):\n                add(\"error\", \"P5\", x[\"shot_id\"], f\"timing {sorted(nums)} is not in the cited claims\",\n                    \"only use numbers that appear in a sourced claim\")\n            if company and l.get(\"speaker\") == \"narrator\":\n                for ph in INTERNAL_DETAIL:\n                    if has_phrase(l.get(\"text\", \"\"), ph) and not cited:\n                        add(\"error\", \"P6\", x[\"shot_id\"], f\"'{ph}' describes {company}'s internal systems without a source\",\n                            \"describe only what is publicly documented, or cite a claim\")\n    days = []\n    for x in shots:\n        if \"day\" in x:\n            if days and x[\"day\"] < days[-1][1] and not x.get(\"flashback\"):\n                add(\"error\", \"P7\", x[\"shot_id\"], f\"time runs backwards (day {x['day']} after day {days[-1][1]})\",\n                    \"fix the order or mark \\\"flashback\\\": true\")\n            days.append((x[\"shot_id\"], x[\"day\"]))\n\n    # ---- A: narration \u2194 visual alignment\n    for x in shots:\n        said, seen = spoken(x), visual_text(x, plan)\n        if not said.strip():\n            continue\n        missing = entity_hits(said, entities) - entity_hits(seen, entities) - set(x.get(\"offscreen\", []))\n        if missing:\n            add(\"error\", \"A1\", x[\"shot_id\"], f\"the narration talks about {sorted(missing)} but the shot doesn't show it\",\n                \"show it (scene/action/ui/journey), or list it in \\\"offscreen\\\" if it is only mentioned\")\n        if x.get(\"recipe\", \"video\") not in DRAWN and not (content_words(said) & content_words(seen)) and not entities:\n            add(\"warn\", \"A2\", x[\"shot_id\"], \"no word of the narration matches the visual description\",\n                \"make the visual show what is being said\")\n    for x in shots:\n        if x.get(\"recipe\", \"video\") in DRAWN | {\"video\"} and not (x.get(\"scene\") or x.get(\"shows\") or\n                                                                 x.get(\"recipe\") in DRAWN):\n            add(\"error\", \"A3\", x[\"shot_id\"], \"shot has no visual description\")\n\n    # ---- V: visual consistency\n    chars = plan.get(\"characters\", {})\n    for a, b in zip(shots, shots[1:]):\n        if a.get(\"location\") and b.get(\"location\") and a[\"location\"] != b[\"location\"] and not b.get(\"transition\"):\n            for who in set(a.get(\"cast\", [])) & set(b.get(\"cast\", [])):\n                trav = plan.get(\"travel\", {})\n                if b[\"location\"] not in trav.get(a[\"location\"], []):\n                    add(\"error\", \"V1\", b[\"shot_id\"], f\"{chars.get(who, {}).get('name', who)} jumps from \"\n                        f\"'{a['location']}' to '{b['location']}' with no transition\",\n                        \"add \\\"transition\\\": \\\"cut\\\"/\\\"time_skip\\\" with a reason, or a travel shot\")\n    props = plan.get(\"props\", {})\n    state = {}\n    for x in shots:\n        for p in x.get(\"props\", []):\n            if p not in props:\n                add(\"error\", \"V2\", x[\"shot_id\"], f\"unknown prop '{p}'\", \"define it in \\\"props\\\" with one fixed look\")\n        for p, st in x.get(\"prop_state\", {}).items():\n            path = props.get(p, {}).get(\"path\", [])\n            if st not in path:\n                add(\"error\", \"V3\", x[\"shot_id\"], f\"prop '{p}' state '{st}' is not in its path {path}\")\n                continue\n            if p in state and path.index(st) < path.index(state[p]) and not x.get(\"flashback\"):\n                add(\"error\", \"V3\", x[\"shot_id\"], f\"prop '{p}' goes back from '{state[p]}' to '{st}' \"\n                    \"(an object can't be in two places / move backwards)\")\n            state[p] = st\n    for x in shots:\n        for c in x.get(\"cast\", []):\n            if c not in chars:\n                add(\"error\", \"V4\", x[\"shot_id\"], f\"unknown character '{c}'\")\n    return issues\n\n\ndef errors(issues):\n    return [i for i in issues if i[\"level\"] == \"error\"]\n\n\ndef report(issues):\n    \"\"\"Readable text, one line per issue, errors first.\"\"\"\n    lines = []\n    for i in sorted(issues, key=lambda i: (i[\"level\"] != \"error\", i[\"shot\"] or \"\")):\n        lines.append(f\"{'\u274c' if i['level'] == 'error' else '\u26a0\ufe0f '} [{i['rule']}] {i['shot'] or 'plan'}: {i['message']}\"\n                     + (f\"  \u2192 {i['fix']}\" if i[\"fix\"] else \"\"))\n    return \"\\n\".join(lines)\n", "vlm_check.py": "\"\"\"Video-language-model event checks: did each required action actually happen, and when?\n\nThe VLM (default Qwen/Qwen3-VL-2B-Instruct, Apache-2.0; its technical report describes explicit text timestamps for\ntemporal grounding) gets sampled frames WITH their times and one narrow yes/no question per event. The answer must\nbe strict JSON with a time and what it saw; anything else becomes UNKNOWN, never PASS. Geometric checks\n(motion_check.py) stay the primary evidence for positions and directions; the VLM covers what boxes can't see\n(a hand placing a small case, a scan) and is cross-checked in acceptance.py.\n\nquestion()/parse() are pure Python and unit-tested; QwenJudge needs a GPU (not testable on the CPU machine).\n\"\"\"\nfrom __future__ import annotations\n\nimport json, re\n\nVERSION = 1\nQUESTION = \"\"\"You are checking an AI-generated video clip against a storyboard. Frames are listed with their times.\nQuestion: {q}\nAnswer ONLY with JSON: {{\"happened\": \"yes\" | \"no\" | \"unsure\", \"start_s\": number or null, \"end_s\": number or null,\n\"evidence\": \"one short sentence describing what you see\"}}\"\"\"\n\n\ndef question(event, names: dict[str, str]) -> str:\n    who = names.get(event.actor, event.actor)\n    obj = names.get(event.obj, event.obj) if event.obj else \"\"\n    tgt = names.get(event.target, event.target) if event.target else \"\"\n    return {\n        \"enter\": f\"Does {who} enter the frame?\",\n        \"walk_to\": f\"Does {who} walk towards {tgt or 'the destination'} and arrive there?\",\n        \"stop\": f\"Does {who} come to a stop?\",\n        \"take_out\": f\"Does {who} take {obj} out of a bag or pocket?\",\n        \"place\": f\"Does {who} put {obj} down on {tgt}, and does it stay there?\",\n        \"pick_up\": f\"Does {who} pick up {obj}?\",\n        \"hand_over\": f\"Does {who} hand {obj} to {tgt}?\",\n        \"scan\": f\"Does {who} scan {obj} with a handheld scanner?\",\n        \"open\": f\"Does {who} open {obj}?\",\n        \"sit\": f\"Does {who} sit down?\", \"stand\": f\"Does {who} stand up?\",\n        \"look_at\": f\"Does {who} look at {obj or tgt}?\",\n        \"talk\": f\"Does {who} speak (mouth moving)?\",\n        \"exit\": f\"Does {who} leave the frame?\",\n    }.get(event.type, event.describe or f\"Does {who} {event.type.replace('_', ' ')}?\")\n\n\ndef parse(text: str):\n    \"\"\"Strict parse of the VLM answer -> (status, evidence). Malformed or hedged answers -> UNKNOWN.\"\"\"\n    m = re.search(r\"\\{.*\\}\", text or \"\", re.S)\n    if not m:\n        return \"UNKNOWN\", {\"raw\": (text or \"\")[:200], \"why\": \"no JSON in the answer\"}\n    try:\n        d = json.loads(m.group(0))\n    except json.JSONDecodeError:\n        return \"UNKNOWN\", {\"raw\": text[:200], \"why\": \"invalid JSON\"}\n    h = str(d.get(\"happened\", \"\")).strip().lower()\n    ev = {\"start_s\": d.get(\"start_s\"), \"end_s\": d.get(\"end_s\"), \"evidence\": str(d.get(\"evidence\", \"\"))[:200]}\n    if h == \"yes\" and isinstance(d.get(\"end_s\"), (int, float)) and ev[\"evidence\"]:\n        return \"PASS\", ev\n    if h == \"no\" and ev[\"evidence\"]:\n        return \"FAIL\", ev\n    return \"UNKNOWN\", dict(ev, why=\"unsure, or 'yes' without a time and evidence\")\n\n\nclass QwenJudge:\n    \"\"\"Qwen3-VL on the GPU. Frames are passed as images, each preceded by its time.\"\"\"\n    MODEL = \"Qwen/Qwen3-VL-2B-Instruct\"\n\n    def __init__(self, device=\"cuda\", cache_dir=None, model=None):\n        import torch\n        from transformers import AutoProcessor, AutoModelForImageTextToText\n        self.torch, self.device = torch, device\n        mid = model or self.MODEL\n        self.proc = AutoProcessor.from_pretrained(mid, cache_dir=cache_dir)\n        from trackers import load_on            # straight onto the GPU: no 4.3 GB copy in system RAM\n        self.model = load_on(AutoModelForImageTextToText, mid, device,\n                             torch_dtype=torch.float16 if device == \"cuda\" else torch.float32, cache_dir=cache_dir)\n\n    def ask(self, frames, times, q: str, max_new_tokens=120) -> str:\n        from PIL import Image\n        content = []\n        for f, t in zip(frames, times):\n            content += [{\"type\": \"text\", \"text\": f\"t={t:.1f}s\"}, {\"type\": \"image\", \"image\": Image.fromarray(f)}]\n        content.append({\"type\": \"text\", \"text\": QUESTION.format(q=q)})\n        inp = self.proc.apply_chat_template([{\"role\": \"user\", \"content\": content}], tokenize=True,\n                                            add_generation_prompt=True, return_dict=True, return_tensors=\"pt\").to(self.device)\n        with self.torch.no_grad():\n            out = self.model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)\n        return self.proc.batch_decode(out[:, inp[\"input_ids\"].shape[1]:], skip_special_tokens=True)[0]\n", "voice_direction.py": "\"\"\"Voice direction: how each narration line should be DELIVERED, decided from the story, never from punctuation\nmarks or keywords. The words never change (emphasis words must be words of the line).\n\nWhere a line's direction comes from (first that applies):\n  1. \"voice_direction\" written in the plan (validated);\n  2. the author's tone label on the line (existing plans: curious, explainer, warm, worried, happy, friendly);\n  3. the line's narrative ROLE: the shot's beat (hook, setup, step, aside, turn, resolution, payoff) and what the\n     sentence does (asks, explains, reveals, concludes), with the neighbouring lines as context;\n  4. optionally a local LLM director (LLM_SYSTEM below) that reads the whole story and returns validated JSON.\nThen `smooth()` enforces the arc rules for every source: conservative intensity (profile range), small changes\ninside a beat, justified jumps only at beat boundaries, no runs of dramatic emotions, rare peaks.\n\"\"\"\nfrom __future__ import annotations\n\nimport json, re\nfrom typing import Literal\n\nfrom pydantic import BaseModel, Field, field_validator\n\nEmotion = Literal[\"neutral\", \"curious\", \"conversational\", \"suspenseful\", \"excited\", \"surprised\", \"serious\",\n                  \"empathetic\", \"reassuring\", \"confident\", \"reflective\", \"inspirational\"]\nRole = Literal[\"hook\", \"question\", \"explain\", \"reveal\", \"compare\", \"consequence\", \"conclude\", \"dialogue\"]\nDRAMATIC = {\"suspenseful\", \"excited\", \"surprised\", \"inspirational\"}\n# emotions that can sit next to each other (or blend) without sounding like a jump\nCOMPATIBLE = {\n    \"neutral\": {\"conversational\", \"curious\", \"serious\", \"confident\", \"reflective\", \"reassuring\"},\n    \"curious\": {\"neutral\", \"conversational\", \"suspenseful\", \"surprised\", \"excited\"},\n    \"conversational\": {\"neutral\", \"curious\", \"reassuring\", \"confident\", \"excited\"},\n    \"suspenseful\": {\"curious\", \"serious\", \"surprised\"},\n    \"excited\": {\"curious\", \"surprised\", \"inspirational\", \"conversational\", \"confident\"},\n    \"surprised\": {\"curious\", \"excited\", \"suspenseful\", \"serious\"},\n    \"serious\": {\"neutral\", \"empathetic\", \"suspenseful\", \"reflective\", \"confident\"},\n    \"empathetic\": {\"serious\", \"reassuring\", \"reflective\"},\n    \"reassuring\": {\"conversational\", \"empathetic\", \"confident\", \"neutral\"},\n    \"confident\": {\"neutral\", \"conversational\", \"reassuring\", \"inspirational\", \"reflective\", \"serious\"},\n    \"reflective\": {\"neutral\", \"serious\", \"confident\", \"inspirational\", \"empathetic\"},\n    \"inspirational\": {\"confident\", \"reflective\", \"excited\"},\n}\n\n\nclass VoiceDirection(BaseModel):\n    speaker_id: str = \"narrator\"\n    emotion: Emotion = \"neutral\"\n    blend: Emotion | None = None\n    intensity: float = Field(0.25, ge=0.0, le=1.0)\n    delivery: str = \"clear and natural\"\n    pace_multiplier: float = Field(1.0, ge=0.8, le=1.15)\n    energy: Literal[\"low\", \"medium\", \"high\"] = \"medium\"\n    emphasis_words: list[str] = Field(default_factory=list, max_length=3)\n    pause_before_ms: int = Field(0, ge=0, le=1200)\n    pause_after_ms: int = Field(450, ge=0, le=1500)\n    style_instruction: str = \"\"\n    transition: Literal[\"steady\", \"gradual\", \"shift\"] = \"steady\"\n    role: Role = \"explain\"\n    source: str = \"rules\"                       # plan | tone | rules | llm | dialogue (+ \"smoothed\")\n    intent: str | None = None                   # two-friend dialogue: the conversational intention of the line\n\n    @field_validator(\"blend\")\n    @classmethod\n    def blend_compatible(cls, v, info):\n        e = info.data.get(\"emotion\")\n        if v is not None and e is not None and v != e and v not in COMPATIBLE.get(e, set()):\n            raise ValueError(f\"blend {v} is not compatible with {e}\")\n        return v\n\n\nPROFILES = {\n    # base emotion, allowed intensity range, base pace, pause scale, emotions allowed, preferred backend, nonverbal\n    \"science_explainer\":     dict(base=\"conversational\", lo=0.15, hi=0.6, pace=1.0, pause=1.0, backend=\"qwen3-tts\",\n                                  allowed=None, nonverbal=[]),\n    \"beginner_education\":    dict(base=\"reassuring\", lo=0.1, hi=0.5, pace=0.96, pause=1.15, backend=\"qwen3-tts\",\n                                  allowed=None, nonverbal=[]),\n    \"cinematic_documentary\": dict(base=\"neutral\", lo=0.15, hi=0.7, pace=0.97, pause=1.2, backend=\"qwen3-tts\",\n                                  allowed=None, nonverbal=[]),\n    \"mystery_documentary\":   dict(base=\"curious\", lo=0.2, hi=0.7, pace=0.95, pause=1.25, backend=\"qwen3-tts\",\n                                  allowed=None, nonverbal=[]),\n    \"historical_narration\":  dict(base=\"reflective\", lo=0.1, hi=0.55, pace=0.96, pause=1.15, backend=\"qwen3-tts\",\n                                  allowed={\"neutral\", \"reflective\", \"serious\", \"curious\", \"confident\", \"empathetic\",\n                                           \"suspenseful\", \"surprised\"}, nonverbal=[]),\n    \"inspirational_story\":   dict(base=\"confident\", lo=0.2, hi=0.7, pace=1.0, pause=1.1, backend=\"qwen3-tts\",\n                                  allowed=None, nonverbal=[]),\n    # two friends talking: mostly natural, low intensity; stronger only on a few hero lines; pauses = response gaps\n    \"two_friends\":           dict(base=\"conversational\", lo=0.1, hi=0.5, pace=1.0, pause=1.0, backend=\"qwen3-tts\",\n                                  allowed=None, nonverbal=[]),\n}\n\n# the author's tone labels (existing plans) \u2192 (emotion, intensity, energy)\nTONE_MAP = {\"curious\": (\"curious\", 0.4, \"medium\"), \"explainer\": (\"conversational\", 0.25, \"medium\"),\n            \"warm\": (\"reassuring\", 0.3, \"medium\"), \"worried\": (\"serious\", 0.4, \"low\"),\n            \"happy\": (\"excited\", 0.45, \"high\"), \"friendly\": (\"conversational\", 0.35, \"medium\"),\n            \"sad\": (\"empathetic\", 0.4, \"low\"), \"tense\": (\"suspenseful\", 0.45, \"low\"),\n            \"awe\": (\"surprised\", 0.45, \"medium\"), \"serious\": (\"serious\", 0.35, \"low\")}\n# a shot's narrative beat \u2192 role, emotion, intensity, pace, pauses\nBEAT_ROLE = {\"hook\": (\"hook\", \"curious\", 0.45, 1.03, 0, 550), \"setup\": (\"explain\", \"conversational\", 0.3, 1.0, 0, 450),\n             \"step\": (\"explain\", \"conversational\", 0.25, 0.98, 0, 450), \"aside\": (\"explain\", \"neutral\", 0.2, 1.0, 0, 400),\n             \"turn\": (\"question\", \"suspenseful\", 0.45, 0.94, 150, 650), \"resolution\": (\"reveal\", \"surprised\", 0.45, 0.97, 250, 600),\n             \"reveal\": (\"reveal\", \"surprised\", 0.5, 0.96, 300, 650), \"consequence\": (\"consequence\", \"serious\", 0.4, 0.95, 100, 600),\n             \"payoff\": (\"conclude\", \"confident\", 0.35, 0.96, 150, 900)}\nEMOTION_WORDS = {\"neutral\": \"a clear, even tone\", \"curious\": \"quiet curiosity\", \"conversational\": \"a warm, conversational tone\",\n                 \"suspenseful\": \"restrained suspense\", \"excited\": \"controlled excitement\", \"surprised\": \"genuine but controlled surprise\",\n                 \"serious\": \"a serious, credible tone\", \"empathetic\": \"serious empathy and restrained emotion\",\n                 \"reassuring\": \"a calm, reassuring tone\", \"confident\": \"calm confidence\", \"reflective\": \"a thoughtful, reflective tone\",\n                 \"inspirational\": \"quiet inspiration\"}\n\n\ndef words_of(text):\n    return re.findall(r\"[A-Za-z0-9][A-Za-z0-9'\u2019-]*\", text)\n\n\ndef asks(text):\n    \"\"\"The sentence's FUNCTION (syntax), not its punctuation: does it ask something?\"\"\"\n    return bool(re.match(r\"^\\s*(what|why|how|who|where|when|which|will|would|can|could|do|does|did|is|are|was|were|\"\n                         r\"have|has|should|but what|so what|and what)\\b\", text, re.I)) or text.rstrip().endswith(\"?\")\n\n\ndef instruction(d: VoiceDirection, narrator_desc=None):\n    \"\"\"Natural-language direction for instruction-following TTS (built from the fields, never free-form).\"\"\"\n    pace = (\"slightly slower than usual\" if d.pace_multiplier < 0.97 else\n            \"slightly faster than usual\" if d.pace_multiplier > 1.03 else \"at a natural pace\")\n    if narrator_desc is None:\n        narrator_desc = \"the same narrator voice\" if d.speaker_id == \"narrator\" else \"the same voice\"\n    if d.intent in INTENT_WORDS:                       # a friend in a conversation, not a narrator reading\n        s = f\"Maintain {narrator_desc}. Talking to a friend, {INTENT_WORDS[d.intent]}, with {EMOTION_WORDS[d.emotion]}\"\n        s += f\", {pace}, {'subtle' if d.intensity < 0.35 else 'moderate'} expression\"\n        if d.emphasis_words:\n            s += \", lightly stressing \" + \", \".join(f\"'{w}'\" for w in d.emphasis_words)\n        return s + \". Casual and believable, like real speech, never theatrical or announcer-like.\"\n    s = f\"Maintain {narrator_desc}. Speak with {EMOTION_WORDS[d.emotion]}\"\n    if d.blend:\n        s += f\" and a touch of {EMOTION_WORDS[d.blend]}\"\n    s += f\", {pace}, {'subtle' if d.intensity < 0.35 else 'moderate' if d.intensity < 0.6 else 'clear'} expression\"\n    if d.emphasis_words:\n        s += \", gently emphasizing \" + \", \".join(f\"'{w}'\" for w in d.emphasis_words)\n    if d.role in (\"question\", \"hook\") and d.pause_before_ms:\n        s += \", with a natural pause before the final phrase\"\n    return s + \". Natural and believable, never theatrical.\"\n\n\ndef pick_emphasis(text, n=2):\n    \"\"\"Most informative words: numbers, names, long content words (deterministic; the LLM director may override).\"\"\"\n    stop = {\"about\", \"after\", \"their\", \"there\", \"these\", \"those\", \"which\", \"where\", \"while\", \"would\", \"could\",\n            \"should\", \"because\", \"before\", \"every\", \"other\", \"being\", \"still\", \"until\", \"since\", \"under\", \"into\"}\n    ws = words_of(text)\n    scored = []\n    for i, w in enumerate(ws):\n        lw = w.lower()\n        if lw in stop or len(lw) < 4 and not re.search(r\"\\d\", w):\n            continue\n        sc = (3 if re.search(r\"\\d\", w) else 0) + (2 if i > 0 and w[0].isupper() else 0) + min(len(w), 10) / 5\n        scored.append((sc, i, w))\n    return [w for _, _, w in sorted(sorted(scored, reverse=True)[:n], key=lambda x: x[1])]\n\n\ndef beats_of(shots):\n    \"\"\"Story beats: consecutive shots sharing the same (chapter, beat role); a beat may span several shots.\"\"\"\n    beats, cur, key = [], [], None\n    for s in shots:\n        k = (s.get(\"chapter\"), s.get(\"beat\"))\n        if cur and k != key:\n            beats.append(cur); cur = []\n        cur.append(s); key = k\n    if cur:\n        beats.append(cur)\n    return beats\n\n\ndef rule_directions(shots, profile=\"science_explainer\", narrator_only=False):\n    \"\"\"{line_id: VoiceDirection} from plan directions / author tones / narrative roles (+ context).\"\"\"\n    P = PROFILES[profile]\n    out = {}\n    beats = beats_of(shots)\n    for bi, beat in enumerate(beats):\n        last_beat = bi == len(beats) - 1\n        for s in beat:\n            for li, l in enumerate(s[\"lines\"]):\n                lid = l.get(\"id\") or f\"{s['shot_id']}_{li + 1}\"\n                if l.get(\"voice_direction\"):\n                    out[lid] = VoiceDirection(**{**l[\"voice_direction\"], \"source\": \"plan\"}); continue\n                if narrator_only and l.get(\"speaker\") != \"narrator\":\n                    continue\n                beat_name = s.get(\"beat\") or (\"hook\" if bi == 0 else \"payoff\" if last_beat else \"step\")\n                role, emo, inten, pace, pb, pa = BEAT_ROLE.get(beat_name, BEAT_ROLE[\"step\"])\n                src = \"rules\"\n                if l.get(\"speaker\", \"narrator\") != \"narrator\":\n                    role = \"dialogue\"\n                if asks(l[\"text\"]) and role not in (\"hook\",):\n                    role, emo, pb = \"question\", (\"suspenseful\" if beat_name in (\"turn\",) else \"curious\"), max(pb, 100)\n                if l.get(\"tone\") in TONE_MAP:                         # the author's intent outranks the rules\n                    emo, inten, energy = TONE_MAP[l[\"tone\"]]; src = \"tone\"\n                else:\n                    energy = \"low\" if emo in (\"serious\", \"empathetic\", \"suspenseful\", \"reflective\") else \\\n                        \"high\" if emo in (\"excited\",) else \"medium\"\n                if P[\"allowed\"] and emo not in P[\"allowed\"]:\n                    emo = P[\"base\"]\n                d = VoiceDirection(speaker_id=l.get(\"speaker\", \"narrator\"), emotion=emo, intensity=inten, role=role,\n                                   delivery=EMOTION_WORDS[emo], pace_multiplier=round(pace * P[\"pace\"], 3), energy=energy,\n                                   emphasis_words=pick_emphasis(l.get(\"tts\") or l[\"text\"]),\n                                   pause_before_ms=int(pb * P[\"pause\"]), pause_after_ms=int(pa * P[\"pause\"]),\n                                   transition=\"gradual\" if li or s is not beat[0] else \"shift\", source=src)\n                out[lid] = d\n    return out\n\n\ndef smooth(order, dirs, profile=\"science_explainer\", max_intensity=1.0):\n    \"\"\"Arc rules for ANY source. order: line ids in story order. max_intensity: a backend cap (measured: strong\n    instructions to Qwen3-TTS shifted the narrator's timbre and even changed a word). Returns (dirs, adjustments).\"\"\"\n    P, adj, run = dict(PROFILES[profile]), [], 0\n    P[\"hi\"] = min(P[\"hi\"], max_intensity)\n    peaks, last, runs = 0, {}, {}\n    for lid in order:\n        d = dirs.get(lid)\n        if d is None:\n            continue\n        # continuity is a property of ONE voice: Meera's calm correction must not be forced into Arjun's surprise\n        prev, run = last.get(d.speaker_id), runs.get(d.speaker_id, 0)\n        lo, hi = P[\"lo\"], P[\"hi\"]\n        if d.intensity > hi or d.intensity < lo:\n            adj.append(f\"{lid}: intensity {d.intensity} \u2192 within the {profile} range [{lo}, {hi}]\")\n            d.intensity = min(max(d.intensity, lo), hi)\n        if prev is not None:\n            same_beat = d.transition != \"shift\"\n            step = 0.15 if same_beat else 0.3\n            if abs(d.intensity - prev.intensity) > step:\n                new = round(prev.intensity + step * (1 if d.intensity > prev.intensity else -1), 2)\n                adj.append(f\"{lid}: intensity {d.intensity} \u2192 {new} (no sudden jump {'inside a beat' if same_beat else 'between beats'})\")\n                d.intensity = new\n            if same_beat and d.emotion != prev.emotion and d.emotion not in COMPATIBLE[prev.emotion] \\\n                    and d.source not in (\"plan\", \"dialogue\"):           # a friend's intention is a real change\n                adj.append(f\"{lid}: {d.emotion} \u2192 {prev.emotion} + blend (incompatible change inside a beat)\")\n                d.blend, d.emotion = (d.emotion if d.emotion in COMPATIBLE[prev.emotion] else None), prev.emotion\n        run = run + 1 if d.emotion in DRAMATIC else 0\n        if run > 2:                                               # no long runs of drama\n            adj.append(f\"{lid}: {d.emotion} \u2192 conversational (third dramatic line in a row)\")\n            d.emotion, d.intensity, run = \"conversational\", min(d.intensity, 0.35), 0\n        if d.intensity >= 0.6:\n            peaks += 1\n            if peaks > max(1, len(order) // 8):                   # peaks are rare\n                adj.append(f\"{lid}: intensity {d.intensity} \u2192 0.55 (too many peaks)\")\n                d.intensity = 0.55\n        if not d.style_instruction or d.source.startswith(\"dialogue\"):\n            d.style_instruction = instruction(d)\n        if adj and adj[-1].startswith(lid) and \"smoothed\" not in d.source:\n            d.source += \"+smoothed\"\n        last[d.speaker_id], runs[d.speaker_id] = d, run\n    return dirs, adj\n\n\ndef neutral_directions(order, dirs):\n    \"\"\"The neutral baseline for the A/B benchmark: same voice, no expression, no extra pauses.\"\"\"\n    return {k: VoiceDirection(speaker_id=d.speaker_id, emotion=\"neutral\", intensity=0.0, role=d.role,\n                              pause_before_ms=0, pause_after_ms=450, source=\"neutral-baseline\",\n                              style_instruction=\"\") for k, d in dirs.items()}\n\n\nLLM_SYSTEM = \"\"\"You are a documentary voice director. For each narration line, decide HOW it should be spoken so the\nstory feels alive but natural: read the whole story first, use each line's meaning and its neighbours, and keep one\nnarrator. Never change the words. Most lines are calm teaching (intensity 0.15\u20130.35); save stronger delivery for real\nturns and reveals; never alternate excited and calm every line; serious facts are never exciting.\nReturn ONE JSON object: {\"lines\": [{\"id\": \"...\", \"emotion\": one of [neutral, curious, conversational, suspenseful,\nexcited, surprised, serious, empathetic, reassuring, confident, reflective, inspirational], \"intensity\": 0-1,\n\"pace_multiplier\": 0.85-1.1, \"energy\": \"low|medium|high\", \"emphasis_words\": [up to 2 words copied from the line],\n\"pause_before_ms\": 0-600, \"pause_after_ms\": 200-1200, \"role\": \"hook|question|explain|reveal|compare|consequence|conclude|dialogue\",\n\"delivery\": \"a few words\"}]}\"\"\"\n\n\nclass _LLMOut(BaseModel):\n    lines: list[dict]\n\n\ndef llm_directions(llm, shots, base, ask_json, log=print):\n    \"\"\"The LLM proposes; every field is validated; emphasis words must be in the line; failures keep the rule\n    direction for that line (logged).\"\"\"\n    story = []\n    for s in shots:\n        for li, l in enumerate(s[\"lines\"]):\n            lid = l.get(\"id\") or f\"{s['shot_id']}_{li + 1}\"\n            story.append({\"id\": lid, \"beat\": s.get(\"beat\") or s.get(\"chapter\"), \"speaker\": l.get(\"speaker\", \"narrator\"),\n                          \"text\": l.get(\"tts\") or l[\"text\"]})\n    out = ask_json(llm, LLM_SYSTEM, \"Story, in order:\\n\" + json.dumps(story, ensure_ascii=False, indent=0), _LLMOut,\n                   log=log, max_new_tokens=2400)\n    texts = {x[\"id\"]: x[\"text\"] for x in story}\n    dirs = dict(base)\n    for row in out.lines:\n        lid = row.get(\"id\")\n        if lid not in texts or lid not in base:\n            continue\n        try:\n            words = {w.lower() for w in words_of(texts[lid])}\n            row[\"emphasis_words\"] = [w for w in row.get(\"emphasis_words\", []) if w.lower() in words][:3]\n            d = VoiceDirection(**{**base[lid].model_dump(), **{k: v for k, v in row.items() if k != \"id\"},\n                                  \"speaker_id\": base[lid].speaker_id, \"source\": \"llm\", \"style_instruction\": \"\"})\n            dirs[lid] = d\n        except Exception as e:\n            log(f\"voice director: line {lid} kept the rule direction ({str(e)[:120]})\")\n    return dirs\n\n\n# ---------------------------------------------------------------- two friends talking (story plans)\n# conversational intention \u2192 (emotion, intensity, pace, energy). Most lines stay natural and low-intensity;\n# the stronger ones are reserved for hero moments (discovery, revelation).\nINTENTS = {\n    \"genuine_curiosity\":       (\"curious\", 0.3, 1.0, \"medium\"),\n    \"thinking_aloud\":          (\"reflective\", 0.2, 0.96, \"low\"),\n    \"playful_disbelief\":       (\"surprised\", 0.35, 1.02, \"medium\"),\n    \"gentle_correction\":       (\"reassuring\", 0.25, 0.98, \"medium\"),\n    \"explaining_observation\":  (\"conversational\", 0.2, 1.0, \"medium\"),\n    \"asking_confirmation\":     (\"curious\", 0.25, 1.03, \"medium\"),\n    \"quiet_realization\":       (\"reflective\", 0.3, 0.95, \"low\"),\n    \"enthusiastic_discovery\":  (\"excited\", 0.45, 1.04, \"high\"),\n    \"sharing_surprising_fact\": (\"surprised\", 0.3, 1.0, \"medium\"),\n    \"offering_reassurance\":    (\"reassuring\", 0.25, 0.98, \"medium\"),\n    \"reflective_explanation\":  (\"reflective\", 0.2, 0.97, \"low\"),\n    \"satisfying_revelation\":   (\"confident\", 0.4, 0.97, \"medium\"),\n    \"guiding_demo\":            (\"conversational\", 0.3, 1.02, \"medium\"),\n    \"playful_remark\":          (\"conversational\", 0.3, 1.04, \"medium\"),\n}\nINTENT_WORDS = {\n    \"genuine_curiosity\": \"genuinely wondering about something\", \"thinking_aloud\": \"thinking it through out loud\",\n    \"playful_disbelief\": \"playfully doubtful\", \"gentle_correction\": \"gently putting a friend right\",\n    \"explaining_observation\": \"explaining what you both just saw\", \"asking_confirmation\": \"checking you understood\",\n    \"quiet_realization\": \"quietly realising something\", \"enthusiastic_discovery\": \"excited by what you just noticed\",\n    \"sharing_surprising_fact\": \"sharing something surprising\", \"offering_reassurance\": \"reassuring a friend\",\n    \"reflective_explanation\": \"explaining calmly\", \"satisfying_revelation\": \"revealing the answer, pleased\",\n    \"guiding_demo\": \"inviting your friend to try something\", \"playful_remark\": \"making a light, friendly remark\"}\nHERO_INTENTS = {\"enthusiastic_discovery\", \"satisfying_revelation\", \"playful_disbelief\"}\nHERO_BEATS = {\"observation\", \"question\", \"reveal\", \"closing\"}\n\n\ndef derive_intent(turn, prev, speaker_role, story_beat, first_in_scene):\n    \"\"\"The line's conversational intention from WHAT IT DOES and WHAT CAME JUST BEFORE (never punctuation alone).\n    turn/prev: plan lines (dicts with speaker, kind, text); speaker_role: curious | expert.\"\"\"\n    if turn.get(\"intent\") in INTENTS:\n        return turn[\"intent\"]\n    k = turn.get(\"kind\", \"observation\")\n    pk = (prev or {}).get(\"kind\")\n    other = prev is not None and prev.get(\"speaker\") != turn.get(\"speaker\")\n    follow = re.match(r\"^\\s*(so|and|then|wait|but|you mean|hold on)\\b\", turn.get(\"text\", \"\"), re.I)\n    if k == \"question\":\n        if other and pk in (\"fact\", \"correction\") and follow:\n            return \"asking_confirmation\"\n        if other and pk in (\"fact\", \"correction\", \"observation\") and story_beat in (\"reveal\", \"correction\"):\n            return \"playful_disbelief\" if follow else \"genuine_curiosity\"\n        return \"genuine_curiosity\"\n    if k in (\"misconception\", \"prediction\", \"hypothetical\"):\n        return \"thinking_aloud\"\n    if k == \"challenge\":\n        return \"playful_disbelief\"\n    if k == \"correction\":\n        return \"gentle_correction\"\n    if k == \"fact\":\n        if story_beat in (\"reveal\",) or (speaker_role == \"curious\" and other):\n            return \"satisfying_revelation\" if speaker_role == \"expert\" else \"quiet_realization\"\n        return \"explaining_observation\" if other and pk == \"question\" else \"reflective_explanation\"\n    if k == \"instruction\":\n        return \"guiding_demo\"\n    if k == \"analogy\":\n        return \"explaining_observation\"\n    if k in (\"reaction\", \"opinion\"):\n        return \"playful_remark\" if speaker_role == \"expert\" or story_beat == \"closing\" else \"quiet_realization\"\n    # observation\n    if speaker_role == \"curious\" and (pk == \"instruction\" or (first_in_scene and story_beat == \"observation\")):\n        return \"enthusiastic_discovery\"\n    if story_beat == \"closing\":\n        return \"offering_reassurance\" if speaker_role == \"expert\" else \"playful_remark\"\n    return \"explaining_observation\" if speaker_role == \"expert\" else \"thinking_aloud\"\n\n\ndef _jitter(key, lo, hi):\n    \"\"\"Deterministic value in [lo, hi] (no identical, mechanical gaps; the same line always gets the same gap).\"\"\"\n    import hashlib\n    f = int(hashlib.sha1(key.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF\n    return int(lo + (hi - lo) * f)\n\n\ndef response_gap_ms(turn, prev, intent, prev_intent, first_in_scene, key):\n    \"\"\"Silence before a line. Conversation research: the typical gap between speakers is ~200 ms (Stivers et al.\n    2009, PNAS, 10 languages); longer gaps carry meaning (thinking, trying a demonstration, a reveal).\"\"\"\n    if prev is None:\n        return 0\n    if prev.get(\"kind\") == \"instruction\":\n        return _jitter(key, 650, 950)                  # time to actually try it (also across a cut)\n    if first_in_scene:\n        return _jitter(key, 100, 250)                  # the shot's own lead-in (pre_s) adds ~300 ms\n    if intent == \"satisfying_revelation\":\n        return _jitter(key, 450, 650)\n    if intent in (\"quiet_realization\", \"thinking_aloud\"):\n        return _jitter(key, 400, 650)\n    if prev.get(\"speaker\") == turn.get(\"speaker\"):\n        return _jitter(key, 250, 450)                  # same person, next thought\n    if intent in (\"playful_disbelief\", \"enthusiastic_discovery\", \"playful_remark\"):\n        return _jitter(key, 120, 260)                  # quick reaction\n    if prev.get(\"kind\") == \"question\":\n        return _jitter(key, 200, 380)                  # answering\n    return _jitter(key, 180, 380)\n\n\ndef dialogue_directions(shots, characters):\n    \"\"\"{line_id: VoiceDirection} for two-friend plans: intention from the story context, response gaps instead of\n    narrator pauses (pause_after = 0), per-character baseline (the curious friend a little livelier).\"\"\"\n    out, prev, prev_intent = {}, None, None\n    for s in shots:\n        beat = s.get(\"story_beat\") or s.get(\"beat\") or \"\"\n        for li, l in enumerate(s[\"lines\"]):\n            lid = l.get(\"id\") or f\"{s['shot_id']}_{li + 1}\"\n            if l.get(\"voice_direction\"):\n                out[lid] = VoiceDirection(**{**l[\"voice_direction\"], \"source\": \"plan\"})\n                prev, prev_intent = l, out[lid].intent\n                continue\n            role = (characters.get(l.get(\"speaker\"), {}) or {}).get(\"role\", \"expert\")\n            intent = derive_intent(l, prev, role, beat, li == 0)\n            emo, inten, pace, energy = INTENTS[intent]\n            if role == \"curious\" and energy != \"low\":\n                inten, pace = min(inten + 0.05, 0.5), pace * 1.01       # Arjun: a bit livelier, not cartoonish\n            hero = intent in HERO_INTENTS and beat in HERO_BEATS\n            if not hero:\n                inten = min(inten, 0.35)\n            gap = response_gap_ms(l, prev, intent, prev_intent, li == 0, lid)\n            d = VoiceDirection(speaker_id=l.get(\"speaker\", \"narrator\"), emotion=emo, intensity=round(inten, 2),\n                               role=\"question\" if l.get(\"kind\") == \"question\" else \"dialogue\",\n                               delivery=INTENT_WORDS[intent], pace_multiplier=round(min(max(pace, 0.9), 1.08), 3),\n                               energy=energy, emphasis_words=pick_emphasis(l.get(\"tts\") or l[\"text\"], n=1),\n                               pause_before_ms=gap, pause_after_ms=0, transition=\"gradual\" if li else \"shift\",\n                               source=\"dialogue\", intent=intent)\n            d.style_instruction = instruction(d)\n            out[lid] = d\n            prev, prev_intent = l, intent\n    return out\n\n\ndef is_hero(line, shot, direction=None):\n    \"\"\"Lines that may get a second take: the opening, the central question, the reveal, the ending, and\n    discovery/revelation moments.\"\"\"\n    return (shot.get(\"story_beat\") in HERO_BEATS and line is shot[\"lines\"][0]) or \\\n        (direction is not None and direction.intent in HERO_INTENTS)\n", "voice_qc.py": "\"\"\"Voice quality control for narration chunks: objective measurements, each check PASS / FAIL / UNKNOWN.\n\nMeasured per chunk: spoken words vs the approved text (missing / repeated words), duration and speaking rate,\nRMS / peak / clipping, silence at the edges and long gaps inside, pitch distribution and jumps (librosa pYIN),\nand voice identity (ECAPA speaker embedding similarity to the narrator's reference; SpeechBrain, Apache-2.0).\nPitch and loudness are DIAGNOSTIC: variation is not proof that an emotion came across. Thresholds = TH (settings).\n\"\"\"\nfrom __future__ import annotations\n\nimport difflib, re\n\nimport numpy as np\n\nTH = {\"clip_level\": 0.995, \"clip_max_fraction\": 0.001, \"silence_db\": -45.0, \"edge_silence_max_s\": 1.0,\n      \"gap_max_s\": 1.2, \"rate_wps\": (1.4, 4.2), \"rate_sps\": (2.0, 7.0), \"pitch_jump_semitones\": 12.0,\n      \"min_rms_db\": -40.0,\n      \"speaker_min_similarity\": 0.80}\n# speaker_min_similarity: each chunk vs the leave-one-out mean of the narrator's other chunks. Calibrated on Kokoro\n# (6 voices, 4 sentences, 2 speeds): same voice \u2265 0.856, closest different voice (af_heart vs af_bella) \u2264 0.737.\n# Chunk-vs-chunk pairs overlap (0.69 vs 0.76), so single comparisons are NOT used. Re-check per backend.\n\n\ndef _db(x):\n    return 20 * np.log10(max(float(x), 1e-9))\n\n\nUNITS = {\"cm\": \"centimetres\", \"centimeters\": \"centimetres\", \"mm\": \"millimetres\", \"millimeters\": \"millimetres\",\n         \"km\": \"kilometres\", \"kilometers\": \"kilometres\", \"kg\": \"kilograms\", \"g\": \"grams\", \"m\": \"metres\",\n         \"meters\": \"metres\", \"mph\": \"miles per hour\", \"hz\": \"hertz\", \"s\": \"seconds\", \"min\": \"minutes\", \"hrs\": \"hours\"}\n\n\ndef syllables(text: str) -> int:\n    \"\"\"Rough English syllable count (vowel groups, silent final e): speaking rate in syllables/s doesn't punish a\n    sentence of short words the way words/s does ('So how does each eye get only its own picture?').\"\"\"\n    n = 0\n    for w in re.findall(r\"[a-z]+\", text.lower()):\n        k = len(re.findall(r\"[aeiouy]+\", w))\n        if w.endswith(\"e\") and not w.endswith((\"le\", \"ee\")) and k > 1:\n            k -= 1\n        n += max(k, 1)\n    return n\n\n\ndef measure(wav: np.ndarray, sr: int, words: int, th=TH, text: str | None = None) -> dict:\n    \"\"\"Signal measurements of one chunk (mono float32). With the text, the pace is also in syllables/s.\"\"\"\n    x = np.asarray(wav, dtype=np.float32).squeeze()\n    dur = len(x) / sr\n    out = {\"duration_s\": round(dur, 3), \"words\": words, \"empty\": len(x) < sr * 0.1}\n    if out[\"empty\"]:\n        return out\n    hop = int(sr * 0.02)\n    frames = x[: len(x) // hop * hop].reshape(-1, hop)\n    lev = 20 * np.log10(np.sqrt((frames ** 2).mean(axis=1)) + 1e-9)\n    loud = lev > th[\"silence_db\"]\n    out.update(rms_db=round(_db(np.sqrt((x ** 2).mean())), 1), peak_db=round(_db(np.abs(x).max()), 1),\n               clip_fraction=round(float((np.abs(x) >= th[\"clip_level\"]).mean()), 5))\n    if loud.any():\n        first, last = int(np.argmax(loud)), len(loud) - 1 - int(np.argmax(loud[::-1]))\n        out[\"lead_silence_s\"], out[\"tail_silence_s\"] = round(first * 0.02, 2), round((len(loud) - 1 - last) * 0.02, 2)\n        inner, gap, longest = loud[first:last + 1], 0, 0\n        for v in inner:\n            gap = 0 if v else gap + 1\n            longest = max(longest, gap)\n        out[\"longest_gap_s\"] = round(longest * 0.02, 2)\n        speech_s = (last - first + 1) * 0.02\n        out[\"speaking_rate_wps\"] = round(words / max(speech_s, 0.1), 2)\n        if text:\n            out[\"speaking_rate_sps\"] = round(syllables(text) / max(speech_s, 0.1), 2)\n    try:\n        import librosa\n        y = librosa.resample(x, orig_sr=sr, target_sr=16000) if sr != 16000 else x\n        f0, voiced, _ = librosa.pyin(y, fmin=60, fmax=400, sr=16000, frame_length=1024)\n        f0 = f0[voiced & np.isfinite(f0)]\n        if len(f0) > 5:\n            st = 12 * np.log2(f0 / np.median(f0))\n            out.update(f0_median_hz=round(float(np.median(f0)), 1), f0_spread_semitones=round(float(np.std(st)), 2),\n                       f0_max_jump_semitones=round(float(np.abs(np.diff(st)).max()), 1),\n                       voiced_fraction=round(float(voiced.mean()), 2))\n    except ImportError:\n        out[\"pitch\"] = \"librosa not available\"\n    return out\n\n\n# words that SOUND the same: a recogniser can't tell them apart, so hearing one for the other is not a missing word\nHOMOPHONES = [{\"eye\", \"i\", \"aye\"}, {\"eyes\", \"is\"}, {\"two\", \"to\", \"too\"}, {\"four\", \"for\"}, {\"right\", \"write\"},\n              {\"their\", \"there\", \"they're\"}, {\"here\", \"hear\"}, {\"one\", \"won\"}, {\"know\", \"no\"}, {\"see\", \"sea\"},\n              {\"whole\", \"hole\"}, {\"its\", \"it's\"}, {\"your\", \"you're\"}, {\"weight\", \"wait\"}, {\"by\", \"buy\", \"bye\"}]\n\n\ndef _homophones(a, b):\n    return a == b or any(a in g and b in g for g in HOMOPHONES)\n\n\ndef word_diff(expected: list[str], heard: list[str]) -> dict:\n    \"\"\"Missing / extra / repeated words between the approved text and what the recogniser heard (normalised words).\"\"\"\n    unit = lambda ws: [x for w in ws for x in UNITS.get(w, w).split()]      # \"6 cm\" was heard for \"six centimetres\"\n    expected, heard = unit(expected), unit(heard)\n    sm = difflib.SequenceMatcher(None, expected, heard, autojunk=False)\n    missing, extra, variants = [], [], []\n    for op, a0, a1, b0, b1 in sm.get_opcodes():\n        if op == \"replace\":\n            # the same sounds written differently is not a missing word: berth/birth, travelling/traveling,\n            # AM/\"a m\", waitlisted/\"weight listed\" (measured false alarms) \u2014 compared letter by letter\n            e_, h_ = \"\".join(expected[a0:a1]), \"\".join(heard[b0:b1])\n            same_sound = a1 - a0 == b1 - b0 and all(_homophones(x, y) for x, y in zip(expected[a0:a1], heard[b0:b1]))\n            if same_sound or difflib.SequenceMatcher(None, e_, h_, autojunk=False).ratio() >= 0.6:\n                variants.append((\" \".join(expected[a0:a1]), \" \".join(heard[b0:b1]))); continue\n        if op in (\"delete\", \"replace\"):\n            missing += expected[a0:a1]\n        if op in (\"insert\", \"replace\"):\n            extra += heard[b0:b1]\n    repeated = [w for i, w in enumerate(heard[1:], 1) if w == heard[i - 1] and len(w) > 2\n                and not any(expected[j] == expected[j - 1] == w for j in range(1, len(expected)))]\n    return {\"missing\": missing, \"extra\": extra, \"repeated\": repeated, \"heard_as\": variants}\n\n\ndef verdict(m: dict, speech_err: float | None, retry_at: float, similarity: float | None, diff: dict | None,\n            th=TH) -> dict:\n    \"\"\"Checks for one chunk \u2192 {check: {status, value}}; 'ok' = no FAIL (UNKNOWN is reported, the gate decides).\"\"\"\n    C = lambda st, v=None: {\"status\": st, \"value\": v}\n    c = {}\n    c[\"audio present\"] = C(\"FAIL\" if m.get(\"empty\") else \"PASS\", m.get(\"duration_s\"))\n    if m.get(\"empty\"):\n        return {\"checks\": c, \"ok\": False}\n    c[\"words correct (speech check)\"] = C(\"UNKNOWN\" if speech_err is None else \"PASS\" if speech_err <= retry_at else \"FAIL\",\n                                          speech_err)\n    if diff is not None:\n        c[\"no missing or repeated words\"] = C(\"PASS\" if not diff[\"missing\"] and not diff[\"repeated\"] else \"FAIL\", diff)\n    lo, hi = th[\"rate_sps\"] if \"speaking_rate_sps\" in m else th[\"rate_wps\"]\n    r = m.get(\"speaking_rate_sps\", m.get(\"speaking_rate_wps\"))\n    c[\"speaking rate natural\"] = C(\"UNKNOWN\" if r is None else \"PASS\" if lo <= r <= hi else \"FAIL\", r)\n    c[\"no clipping\"] = C(\"PASS\" if m[\"clip_fraction\"] <= th[\"clip_max_fraction\"] else \"FAIL\", m[\"clip_fraction\"])\n    c[\"audible level\"] = C(\"PASS\" if m[\"rms_db\"] >= th[\"min_rms_db\"] else \"FAIL\", m[\"rms_db\"])\n    c[\"no long silence inside\"] = C(\"PASS\" if m.get(\"longest_gap_s\", 0) <= th[\"gap_max_s\"] else \"FAIL\", m.get(\"longest_gap_s\"))\n    c[\"no long silence at the edges\"] = C(\"PASS\" if max(m.get(\"lead_silence_s\", 0), m.get(\"tail_silence_s\", 0))\n                                         <= th[\"edge_silence_max_s\"] else \"FAIL\",\n                                         [m.get(\"lead_silence_s\"), m.get(\"tail_silence_s\")])\n    j = m.get(\"f0_max_jump_semitones\")\n    c[\"no extreme pitch jumps (diagnostic)\"] = C(\"UNKNOWN\" if j is None else \"PASS\" if j <= th[\"pitch_jump_semitones\"]\n                                                 else \"FAIL\", j)\n    c[\"same narrator voice\"] = C(\"UNKNOWN\" if similarity is None else\n                                 \"PASS\" if similarity >= th[\"speaker_min_similarity\"] else \"FAIL\", similarity)\n    # diagnostic checks are reported, but never decide (pitch movement is not proof of anything)\n    return {\"checks\": c, \"ok\": not any(x[\"status\"] == \"FAIL\" for k, x in c.items() if \"(diagnostic)\" not in k)}\n\n\nclass SpeakerCheck:\n    \"\"\"ECAPA speaker embeddings (speechbrain/spkrec-ecapa-voxceleb, Apache-2.0), unit length. Used by\n    consistency(): a seed alone doesn't guarantee the same voice, so every chunk is measured.\"\"\"\n    MODEL = \"speechbrain/spkrec-ecapa-voxceleb\"\n\n    def __init__(self, savedir, device=\"auto\"):\n        from speechbrain.inference.speaker import EncoderClassifier\n        if device == \"auto\":                              # the GPU when there is one (small model, ~20 MB)\n            import torch\n            device = \"cuda:0\" if torch.cuda.is_available() else \"cpu\"   # speechbrain wants \"cuda:<index>\"\n        self.device = device\n        self.m = EncoderClassifier.from_hparams(self.MODEL, savedir=str(savedir), run_opts={\"device\": device})\n\n    def embed(self, wav, sr):\n        import torch\n        x = np.asarray(wav, dtype=np.float32).squeeze()\n        if sr != 16000:\n            import librosa\n            x = librosa.resample(x, orig_sr=sr, target_sr=16000)\n        with torch.inference_mode():\n            e = self.m.encode_batch(torch.from_numpy(x)[None]).squeeze().cpu().numpy()\n        return e / (np.linalg.norm(e) + 1e-9)\n\n\n\n# per backend: deterministic Kokoro varies little between chunks (calibrated: 0.80); Qwen3-TTS samples, so even\n# two plain takes of one speaker differ more (measured LOO 0.77\u20130.81 over 3 takes incl. styled ones): PROVISIONAL\n# 0.72 \u2014 recalibrate from VOICE_BENCH's voice_qc_report.json on the T4.\nSPEAKER_TH = {\"kokoro\": 0.80, \"qwen3-tts\": 0.72, \"parler\": 0.72, \"gemini\": 0.72}   # gemini: provisional, uncalibrated\n\n\nMIN_IDENTITY_S = 2.0      # shorter clips are not judged (and not used as reference)\nSHORT_CLIP_S, SHORT_TH = 3.5, 0.70\n# Calibration on two-friend dialogue (2026-10-18, 37 Kokoro lines, am_michael + af_heart, ECAPA leave-one-out):\n#   same voice: < 3 s min 0.684 (median 0.81); 3\u20133.5 s min 0.822; \u2265 3.5 s min 0.874. Other friend: max 0.18.\n# The 0.80 bar (calibrated on longer narrator chunks vs SIMILAR voices) only made false alarms on short lines.\n\n\ndef identity_threshold(backend: str, duration_s: float) -> float | None:\n    \"\"\"Pass mark for the identity similarity of one line (None = too short to judge).\"\"\"\n    if duration_s < MIN_IDENTITY_S:\n        return None\n    base = SPEAKER_TH.get(backend, TH[\"speaker_min_similarity\"])\n    return min(base, SHORT_TH) if duration_s < SHORT_CLIP_S else base\n\n\ndef consistency(embs: dict, th=TH, durations: dict | None = None) -> dict:\n    \"\"\"Narrator identity over the whole narration: each chunk vs the mean of ALL the speaker's OTHER chunks\n    (leave-one-out). {chunk id: similarity}; below th[\"speaker_min_similarity\"] = suspicious (regenerate/review).\n    Needs \u2265 3 chunks; with fewer, nothing can be judged (UNKNOWN). With durations, chunks shorter than\n    MIN_IDENTITY_S are not judged (None) and not used as reference.\"\"\"\n    short = {i for i, d in (durations or {}).items() if d < MIN_IDENTITY_S}\n    if short:\n        out = consistency({i: e for i, e in embs.items() if i not in short}, th)\n        return {i: out.get(i) for i in embs}\n    ids = list(embs)\n    if len(ids) < 3:\n        return {i: None for i in ids}\n    E = np.stack([embs[i] for i in ids])\n    total = E.sum(axis=0)\n    out = {}\n    for k, i in enumerate(ids):\n        r = (total - E[k]) / (len(ids) - 1)\n        out[i] = round(float(E[k] @ (r / (np.linalg.norm(r) + 1e-9))), 3)\n    return out\n\n\ndef conversation_diagnostics(lines, max_gap_s=1.5):\n    \"\"\"Timing of a dialogue as a whole. lines: [{\"id\", \"speaker\", \"start\", \"end\", \"rms_db\", \"sps\"}] in order.\n    Technical diagnostics only (a person still judges whether it sounds like a real conversation).\"\"\"\n    gaps = [round(b[\"start\"] - a[\"end\"], 3) for a, b in zip(lines, lines[1:])]\n    out = {\"n_gaps\": len(gaps), \"issues\": []}\n    if gaps:\n        g = np.array(gaps)\n        out.update(gap_mean_s=round(float(g.mean()), 3), gap_sd_s=round(float(g.std()), 3),\n                   gap_min_s=round(float(g.min()), 3), gap_max_s=round(float(g.max()), 3),\n                   gap_cv=round(float(g.std() / max(g.mean(), 1e-6)), 3))\n        if len(gaps) >= 6 and out[\"gap_cv\"] < 0.2:\n            out[\"issues\"].append(f\"mechanical rhythm: gaps vary only {out['gap_cv']:.0%} (sd/mean)\")\n        for a, b, x in zip(lines, lines[1:], gaps):\n            if x > max_gap_s:\n                out[\"issues\"].append(f\"dead air {x:.2f}s before {b['id']}\")\n            if x < -0.02:\n                out[\"issues\"].append(f\"overlap {-x:.2f}s: {a['id']} / {b['id']}\")\n    by = {}\n    for l in lines:\n        by.setdefault(l[\"speaker\"], []).append(l)\n    for sp, ls in by.items():\n        rates = [l[\"sps\"] for l in ls if l.get(\"sps\")]\n        if len(rates) >= 3:\n            med = float(np.median(rates))\n            for l in ls:\n                if l.get(\"sps\") and abs(l[\"sps\"] - med) / med > 0.35:\n                    out[\"issues\"].append(f\"{l['id']}: speaking rate {l['sps']:.1f} syll/s vs {sp}'s usual {med:.1f}\")\n        for a, b in zip(ls, ls[1:]):\n            if a.get(\"rms_db\") is not None and b.get(\"rms_db\") is not None and abs(a[\"rms_db\"] - b[\"rms_db\"]) > 4:\n                out[\"issues\"].append(f\"{b['id']}: level jump {b['rms_db'] - a['rms_db']:+.1f} dB vs {sp}'s previous line\")\n    return out\n\n\ndef pick_take(takes: list[dict], retry_at: float) -> dict:\n    \"\"\"Best of several takes of one line by the CHECKS only: first acceptable words (\u2264 retry_at wrong), then fewer\n    wrong words, then fewer failed checks; earlier take on a tie. Never by pitch variation or \"liveliness\".\"\"\"\n    return min(takes, key=lambda t: ((t.get(\"err\") or 0.0) > retry_at, t.get(\"err\") or 0.0, t.get(\"fails\", 0),\n                                     t.get(\"take\", 0)))\n"}""").items():
    (LIB_DIR / _name).write_text(_src)
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))
import validate, drawkit, musiclib, scene_state, layout_control, motion_check, trackers, vlm_check, acceptance, qclib, subtitles, stages, voice_direction, expressive_tts, voice_qc, registry, characters, charanim, demos, story, storybook_art, storybook, envfactory, storybook_scene
for _m in (validate, drawkit, musiclib, scene_state, layout_control, motion_check, trackers, vlm_check, acceptance, qclib, subtitles, stages, voice_direction, expressive_tts, voice_qc, registry, characters, charanim, demos, story, storybook_art, storybook, envfactory, storybook_scene):
    importlib.reload(_m)
log(f"engine library loaded from {LIB_DIR}: validate, drawkit, musiclib")


In [ ]:
import colab_budget
BUDGET = colab_budget.SessionBudget(MAX_SESSION_MINUTES)
# Include time already spent on preflight/install in the local session limit.
BUDGET.deadline = SESSION_STARTED + MAX_SESSION_MINUTES * 60
BUDGET.check('plan validation')


In [ ]:
BUDGET.check('plan')
# ==== 4. The story plan (written by Claude; edit plan.json on Drive to change it) ====
PLAN_EMBEDDED = json.loads(r"""{
 "plan_version": 5,
 "project": "three_d_friends_te_v5_cinematic",
 "title": "త్రీడీ సినిమాలో బొమ్మ మన దగ్గరికి వచ్చినట్టు ఎందుకు కనిపిస్తుంది?",
 "language": "te",
 "story_mode": "two_friends_cinematic",
 "visual_style": "cinematic_atlas",
 "license_strict": true,
 "factual_dialogue": true,
 "logline": "Popcorn దగ్గరికి వచ్చినట్టు ఎందుకు కనిపించిందన్న ప్రశ్నకు, రెండు కళ్ల చూపు మరియు పోలరైజ్డ్ అద్దాల పని ద్వారా సమాధానం; చివరలో చిన్న నవ్వు.",
 "notebook": {
  "emoji": "🎬",
  "name": "Two-Friend Explainer",
  "length": "about 1–1.5 min"
 },
 "style": {
  "suffix": "cinematic 3D-style illustrated characters, approved reusable RGBA atlas animation",
  "negative": "text, watermark, photorealistic, deformed, blurry"
 },
 "look": {
  "bg_top": "#1B2440",
  "bg_bottom": "#28406A"
 },
 "characters": {
  "arjun": {
   "name": "అర్జున్",
   "voice": "Prakash",
   "speed": 1.0,
   "pitch": 0.0,
   "fx": "none",
   "color": "#FFD166",
   "look": "original 2D character Arjun",
   "tag": "a boy with swept brown hair, red jacket, yellow striped shirt, teal trousers and sneakers",
   "role": "curious",
   "voice_style": "in a young, lively and curious voice at a moderate pace",
   "gemini_voice": "Achird"
  },
  "meera": {
   "name": "మీరా",
   "voice": "Lalitha",
   "speed": 1.0,
   "pitch": 0.0,
   "fx": "none",
   "color": "#7FDBDA",
   "look": "original 2D character Meera",
   "tag": "a girl with auburn ponytail, teal hair tie, large brown eyes and cream floral T-shirt",
   "role": "expert",
   "voice_style": "in a warm, clear and friendly explaining voice at a moderate pace",
   "gemini_voice": "Sulafat"
  }
 },
 "bible": {
  "version": 1,
  "relationship": "Close friends of the same age who hang out often. Arjun notices things and asks; Meera loves explaining with real examples and small demonstrations. She never mocks him and never lectures for long; he asks real follow-ups, sometimes with a plausible wrong guess, never just 'Wow'.",
  "characters": {
   "arjun": {
    "id": "arjun",
    "name": "Arjun",
    "names": {
     "te": "అర్జున్"
    },
    "role": "curious",
    "personality": "curious, observant, relatable, occasionally funny; asks the question the viewer is thinking",
    "look": {
     "skin": "#E5AD7B",
     "hair": "#79462D",
     "hair_style": "short_neat",
     "top": "#C83937",
     "top_detail": "#F8C454",
     "eyes": "#663C20",
     "glasses": false
    },
    "voices": {
     "en": {
      "backend": "kokoro",
      "voice": "am_michael",
      "style": "",
      "pitch": 0.0,
      "qwen_speaker": "aiden",
      "gemini_voice": null,
      "gemini_voice_status": "",
      "emotion_control": "none",
      "rate_range": [
       0.95,
       1.08
      ],
      "intensity_range": [
       0.1,
       0.5
      ],
      "accent": "American",
      "baseline": "friendly, curious, a little energetic; questions rise naturally, never cartoonish",
      "reference_sample": "",
      "pronunciation": {}
     },
     "te": {
      "backend": "parler",
      "voice": "Prakash",
      "style": "in a young, lively and curious voice at a moderate pace",
      "pitch": 0.0,
      "qwen_speaker": null,
      "gemini_voice": "Achird",
      "gemini_voice_status": "auditioned",
      "emotion_control": "description",
      "rate_range": [
       0.92,
       1.08
      ],
      "intensity_range": [
       0.1,
       0.5
      ],
      "accent": "",
      "baseline": "friendly and curious",
      "reference_sample": "",
      "pronunciation": {
       "3D": "త్రీడీ"
      }
     }
    },
    "expressions": [
     "neutral",
     "smile",
     "curious",
     "surprised",
     "confused",
     "thinking",
     "happy",
     "listening",
     "wink"
    ],
    "gestures": [
     "idle",
     "explain",
     "point",
     "think",
     "open_hands",
     "nod",
     "duck"
    ],
    "tag": "a boy with swept brown hair, red jacket, yellow striped shirt, teal trousers and sneakers",
    "color": "#FFD166"
   },
   "meera": {
    "id": "meera",
    "name": "Meera",
    "names": {
     "te": "మీరా"
    },
    "role": "expert",
    "personality": "knowledgeable, friendly, patient; simplifies with familiar examples and demonstrations",
    "look": {
     "skin": "#F0BE9E",
     "hair": "#984F2A",
     "hair_style": "long_tied",
     "top": "#F5EAD1",
     "top_detail": "#DB6572",
     "eyes": "#7C431E",
     "glasses": false
    },
    "voices": {
     "en": {
      "backend": "kokoro",
      "voice": "af_heart",
      "style": "",
      "pitch": 0.0,
      "qwen_speaker": null,
      "gemini_voice": null,
      "gemini_voice_status": "",
      "emotion_control": "none",
      "rate_range": [
       0.93,
       1.05
      ],
      "intensity_range": [
       0.1,
       0.45
      ],
      "accent": "American",
      "baseline": "warm, patient, confident; calm explanations that brighten on discoveries",
      "reference_sample": "",
      "pronunciation": {}
     },
     "te": {
      "backend": "parler",
      "voice": "Lalitha",
      "style": "in a warm, clear and friendly explaining voice at a moderate pace",
      "pitch": 0.0,
      "qwen_speaker": null,
      "gemini_voice": "Sulafat",
      "gemini_voice_status": "auditioned",
      "emotion_control": "description",
      "rate_range": [
       0.92,
       1.08
      ],
      "intensity_range": [
       0.1,
       0.5
      ],
      "accent": "",
      "baseline": "warm and clear",
      "reference_sample": "",
      "pronunciation": {
       "3D": "త్రీడీ"
      }
     }
    },
    "expressions": [
     "neutral",
     "smile",
     "curious",
     "surprised",
     "confused",
     "thinking",
     "happy",
     "listening",
     "wink"
    ],
    "gestures": [
     "idle",
     "explain",
     "point",
     "think",
     "open_hands",
     "nod",
     "duck"
    ],
    "tag": "a girl with auburn ponytail, teal hair tie, large brown eyes and cream floral T-shirt",
    "color": "#7FDBDA"
   }
  }
 },
 "tones": {
  "curious": {
   "speed": 0.98,
   "pitch": 0.0,
   "gain_db": 0.0,
   "pause_after": 0.45
  },
  "explainer": {
   "speed": 1.0,
   "pitch": 0.0,
   "gain_db": 0.0,
   "pause_after": 0.4
  },
  "warm": {
   "speed": 0.97,
   "pitch": 0.0,
   "gain_db": 0.0,
   "pause_after": 0.5
  },
  "happy": {
   "speed": 1.0,
   "pitch": 0.0,
   "gain_db": 0.0,
   "pause_after": 0.4
  },
  "friendly": {
   "speed": 1.0,
   "pitch": 0.0,
   "gain_db": 0.0,
   "pause_after": 0.35
  },
  "awe": {
   "speed": 0.96,
   "pitch": 0.0,
   "gain_db": 0.0,
   "pause_after": 0.6
  },
  "serious": {
   "speed": 0.97,
   "pitch": 0.0,
   "gain_db": 0.0,
   "pause_after": 0.5
  }
 },
 "moods": {
  "curious": {
   "pause_after": 0.5,
   "grade": "neutral",
   "move_amount": 0.04,
   "music": {
    "base": 0.5,
    "curious": 0.7,
    "tension": 0.1,
    "shimmer": 0.3
   }
  },
  "calm": {
   "pause_after": 0.45,
   "grade": "neutral",
   "move_amount": 0.04,
   "music": {
    "base": 0.4,
    "curious": 0.8,
    "tension": 0.0,
    "shimmer": 0.2
   }
  },
  "busy": {
   "pause_after": 0.4,
   "grade": "neutral",
   "move_amount": 0.05,
   "music": {
    "base": 0.5,
    "curious": 0.6,
    "tension": 0.2,
    "shimmer": 0.1
   }
  },
  "joy": {
   "pause_after": 0.45,
   "grade": "neutral",
   "move_amount": 0.05,
   "music": {
    "base": 0.3,
    "curious": 1.0,
    "tension": 0.0,
    "shimmer": 0.4
   }
  }
 },
 "claims": [
  {
   "id": "c_f1",
   "text": "In binocular depth vision, the sensation arises from processing differences in retinal images resulting from the two eyes looking from different, but similar, directions.",
   "sources": [
    "https://en.wikipedia.org/wiki/Stereopsis"
   ],
   "source_revision": 1377837508
  },
  {
   "id": "c_f2",
   "text": "The two images are combined in the brain to give the perception of stereopsis; when viewed, the human brain perceives the images as a single 3D view, giving the viewer the perception of 3D depth.",
   "sources": [
    "https://en.wikipedia.org/wiki/Stereoscopy"
   ],
   "source_revision": 1380706327
  },
  {
   "id": "c_f3",
   "text": "To present stereoscopic images and films, two images are projected superimposed onto the same screen or displayed through different polarizing filters.",
   "sources": [
    "https://en.wikipedia.org/wiki/Polarized_3D_system"
   ],
   "source_revision": 1377586129
  },
  {
   "id": "c_f4",
   "text": "The viewer wears low-cost eyeglasses with a polarizing filter for each eye. The left and right filters have different polarizations, so each eye receives only the image with the matching polarization.",
   "sources": [
    "https://en.wikipedia.org/wiki/Polarized_3D_system"
   ],
   "source_revision": 1377586129
  },
  {
   "id": "c_f7",
   "text": "Monocular cues provide depth information even when viewing a scene with only one eye. Monocular cues include relative size (distant objects subtend smaller visual angles than near objects), texture gradient, occlusion, linear perspective, contrast differences, and motion parallax.",
   "sources": [
    "https://en.wikipedia.org/wiki/Depth_perception"
   ],
   "source_revision": 1380662591
  }
 ],
 "vocabulary": [
  "Arjun",
  "Meera",
  "3D",
  "popcorn",
  "అర్జున్",
  "మీరా"
 ],
 "entities": {},
 "glossary": {
  "eyes": [
   "కళ్లు",
   "కళ్లూ",
   "కన్ను",
   "కంటి",
   "eyes"
  ],
  "eye": [
   "కన్ను",
   "కంటి",
   "eye"
  ],
  "different": [
   "వేరే",
   "different"
  ],
  "view": [
   "view",
   "బొమ్మ"
  ],
  "views": [
   "view",
   "బొమ్మ"
  ],
  "picture": [
   "picture",
   "బొమ్మ"
  ],
  "pictures": [
   "picture",
   "బొమ్మ"
  ],
  "brain": [
   "brain",
   "మెదడు"
  ],
  "combines": [
   "కలిపి",
   "combine"
  ],
  "depth": [
   "depth",
   "లోతు"
  ],
  "projector": [
   "projector",
   "Projector",
   "ప్రొజెక్టర్"
  ],
  "screen": [
   "screen",
   "స్క్రీన్",
   "తెర"
  ],
  "filter": [
   "filter",
   "ఫిల్టర్"
  ],
  "filters": [
   "filter",
   "ఫిల్టర్"
  ],
  "polarizing": [
   "polarizing",
   "పోలరైజింగ్"
  ],
  "lenses": [
   "lens",
   "అద్దం",
   "లెన్స్"
  ],
  "size": [
   "size",
   "Size",
   "పరిమాణం"
  ],
  "one eye": [
   "ఒక్క కన్ను",
   "ఒక కన్ను",
   "one eye"
  ],
  "flat": [
   "flat",
   "చదును"
  ],
  "image": [
   "view",
   "బొమ్మ"
  ]
 },
 "travel": {},
 "shots": [
  {
   "shot_id": "sc01",
   "recipe": "story",
   "beat": "hook",
   "story_beat": "observation",
   "mood": "curious",
   "location": "cinema_seats",
   "cast": [
    "arjun",
    "meera"
   ],
   "claims": [],
   "lines": [
    {
     "speaker": "arjun",
     "tone": "awe",
     "text": "అమ్మో! తెర మీద popcorn నా మీదకి వచ్చినట్టు ఎలా కనిపించింది?",
     "kind": "question",
     "tts": "అమ్మో! తెర మీద పాప్‌కార్న్ నా మీదకి వచ్చినట్టు ఎలా కనిపించింది?",
     "act": {
      "expr": "surprised",
      "gesture": "duck"
     }
    },
    {
     "speaker": "meera",
     "tone": "curious",
     "text": "అది తెలుసుకోవాలంటే… తెరనే కాదు, నీ రెండు కళ్లనీ గమనించాలి.",
     "kind": "instruction",
     "tts": "అది తెలుసుకోవాలంటే… తెరనే కాదు, నీ రెండు కళ్లనీ గమనించాలి."
    }
   ],
   "story": {
    "world": "cinema_seats",
    "visuals": [
     {
      "id": "v1",
      "intent": "establishing",
      "medium": "character_scene",
      "recipe": "character",
      "character": {
       "world": "cinema_seats",
       "shot": "establishing",
       "chars": {
        "arjun": {
         "wear": [
          "3d_glasses"
         ],
         "gesture": "duck",
         "expr": "surprised"
        },
        "meera": {
         "wear": [
          "3d_glasses"
         ]
        }
       },
       "screen": "clean+popcorn",
       "changes": [
        {
         "t": 0.0,
         "chars": {
          "arjun": {
           "expr": "surprised",
           "gesture": "duck"
          },
          "meera": {
           "gesture": "idle",
           "expr": "listening"
          }
         }
        }
       ]
      },
      "start": {
       "line": 0,
       "at": 0.0
      },
      "description": "Same cinema, same seats and glasses. Keep subtitles clear of faces."
     },
     {
      "id": "v2",
      "intent": "closeup",
      "medium": "character_scene",
      "recipe": "character",
      "character": {
       "world": "cinema_seats",
       "shot": "closeup",
       "chars": {
        "arjun": {
         "wear": [
          "3d_glasses"
         ]
        },
        "meera": {
         "wear": [
          "3d_glasses"
         ]
        }
       },
       "subject": "meera",
       "push": true,
       "changes": [
        {
         "t": 0.0,
         "chars": {
          "meera": {
           "gesture": "point"
          },
          "arjun": {
           "gesture": "nod",
           "expr": "listening"
          }
         }
        }
       ]
      },
      "start": {
       "line": 1,
       "at": 0.0
      },
      "description": "Same cinema, same seats and glasses. Keep subtitles clear of faces."
     }
    ]
   }
  },
  {
   "shot_id": "sc02",
   "recipe": "story",
   "beat": "turn",
   "story_beat": "question",
   "mood": "curious",
   "location": "cinema_seats",
   "cast": [
    "arjun",
    "meera"
   ],
   "claims": [],
   "lines": [
    {
     "speaker": "arjun",
     "tone": "curious",
     "text": "రెండు కళ్లతో చూసేది ఒకే బొమ్మ కాదా?",
     "kind": "question",
     "tts": "రెండు కళ్లతో చూసేది ఒకే బొమ్మ కాదా?"
    },
    {
     "speaker": "meera",
     "tone": "friendly",
     "text": "రెండు చూపుల్ని పక్కపక్కన చూస్తే తేడా తెలుస్తుంది. చూడు.",
     "kind": "instruction",
     "tts": "రెండు చూపుల్ని పక్కపక్కన చూస్తే తేడా తెలుస్తుంది. చూడు."
    }
   ],
   "story": {
    "world": "cinema_seats",
    "visuals": [
     {
      "id": "v1",
      "intent": "reaction",
      "medium": "character_scene",
      "recipe": "character",
      "character": {
       "world": "cinema_seats",
       "shot": "reaction",
       "chars": {
        "arjun": {
         "wear": [
          "3d_glasses"
         ]
        },
        "meera": {
         "wear": [
          "3d_glasses"
         ]
        }
       },
       "subject": "arjun",
       "push": true,
       "changes": [
        {
         "t": 0.0,
         "chars": {
          "arjun": {
           "gesture": "idle"
          },
          "meera": {
           "gesture": "idle",
           "expr": "listening"
          }
         }
        }
       ]
      },
      "start": {
       "line": 0,
       "at": 0.0
      },
      "description": "Same cinema, same seats and glasses. Keep subtitles clear of faces."
     },
     {
      "id": "v2",
      "intent": "two_friend",
      "medium": "character_scene",
      "recipe": "character",
      "character": {
       "world": "cinema_seats",
       "shot": "two_shot",
       "chars": {
        "arjun": {
         "wear": [
          "3d_glasses"
         ]
        },
        "meera": {
         "wear": [
          "3d_glasses"
         ]
        }
       },
       "changes": [
        {
         "t": 0.0,
         "chars": {
          "meera": {
           "gesture": "point"
          },
          "arjun": {
           "gesture": "nod",
           "expr": "listening"
          }
         }
        }
       ]
      },
      "start": {
       "line": 1,
       "at": 0.0
      },
      "description": "Same cinema, same seats and glasses. Keep subtitles clear of faces."
     }
    ]
   }
  },
  {
   "shot_id": "sc03",
   "recipe": "story",
   "beat": "step",
   "story_beat": "demonstration",
   "mood": "busy",
   "location": "cinema_seats",
   "cast": [
    "arjun",
    "meera"
   ],
   "claims": [
    "c_f1",
    "c_f2"
   ],
   "lines": [
    {
     "speaker": "meera",
     "tone": "explainer",
     "text": "రెండు కళ్లూ కొంచెం వేర్వేరు స్థానాల్లో ఉంటాయి కదా. అందుకే ఒక్కో కంటికి కొంచెం వేరే బొమ్మ కనిపిస్తుంది.",
     "kind": "fact",
     "tts": "రెండు కళ్లూ కొంచెం వేర్వేరు స్థానాల్లో ఉంటాయి కదా. అందుకే ఒక్కో కంటికి కొంచెం వేరే బొమ్మ కనిపిస్తుంది.",
     "intent": "explaining_observation",
     "claims": [
      "c_f1"
     ]
    },
    {
     "speaker": "arjun",
     "tone": "curious",
     "text": "అంటే ఒక కన్ను మూస్తే, ఆ రెండో చూపు ఉండదన్నమాట?",
     "kind": "prediction",
     "tts": "అంటే ఒక కన్ను మూస్తే, ఆ రెండో చూపు ఉండదన్నమాట?",
     "intent": "thinking_aloud",
     "act": {
      "expr": "thinking"
     }
    },
    {
     "speaker": "meera",
     "tone": "explainer",
     "text": "అవును. మెదడు రెండు కళ్ల బొమ్మల్లోని తేడాలను ఉపయోగించి లోతును గుర్తిస్తుంది.",
     "kind": "fact",
     "tts": "అవును. మెదడు రెండు కళ్ల బొమ్మల్లోని తేడాలను ఉపయోగించి లోతును గుర్తిస్తుంది.",
     "intent": "satisfying_revelation",
     "claims": [
      "c_f2"
     ]
    }
   ],
   "story": {
    "world": "cinema_seats",
    "visuals": [
     {
      "id": "v1",
      "intent": "closeup",
      "medium": "character_scene",
      "recipe": "character",
      "character": {
       "world": "cinema_seats",
       "shot": "closeup",
       "chars": {
        "arjun": {
         "wear": [
          "3d_glasses"
         ]
        },
        "meera": {
         "wear": [
          "3d_glasses"
         ]
        }
       },
       "subject": "meera",
       "screen": "clean+popcorn",
       "push": true,
       "changes": [
        {
         "t": 0.0,
         "chars": {
          "meera": {
           "gesture": "explain"
          },
          "arjun": {
           "gesture": "idle",
           "expr": "listening"
          }
         }
        }
       ]
      },
      "start": {
       "line": 0,
       "at": 0.0
      },
      "description": "Same seated friends, glasses remain worn; the screen image is an illustration of perceived depth, not a physical object entering the room."
     },
     {
      "id": "v2",
      "intent": "animated_illustration",
      "medium": "animated_demo",
      "recipe": "demo",
      "demo": {
       "template": "eye_views",
       "world": "cinema_seats"
      },
      "start": {
       "line": 0,
       "at": 0.3
      },
      "description": "Labelled left-eye and right-eye view diagram, followed by a depth-perception illustration. Not real footage. Keep this diagram visible through the final explanatory turn; no reaction cutaway until the next scene."
     }
    ]
   }
  },
  {
   "shot_id": "sc04",
   "recipe": "story",
   "beat": "resolution",
   "story_beat": "correction",
   "mood": "calm",
   "location": "cinema_seats",
   "cast": [
    "arjun",
    "meera"
   ],
   "claims": [
    "c_f3",
    "c_f4"
   ],
   "lines": [
    {
     "speaker": "arjun",
     "tone": "curious",
     "text": "మరి ఒకే తెర నుంచి ఒక్కో కంటికి వేరే బొమ్మ ఎలా వస్తుంది?",
     "kind": "question",
     "tts": "మరి ఒకే తెర నుంచి ఒక్కో కంటికి వేరే బొమ్మ ఎలా వస్తుంది?",
     "intent": "genuine_curiosity",
     "act": {
      "expr": "curious"
     }
    },
    {
     "speaker": "meera",
     "tone": "explainer",
     "text": "పోలరైజ్డ్ త్రీడీ పద్ధతిలో, రెండు బొమ్మలు వేర్వేరు పోలరైజేషన్‌తో ఒకే తెరపై పడతాయి.",
     "kind": "fact",
     "tts": "పోలరైజ్డ్ త్రీడీ పద్ధతిలో, రెండు బొమ్మలు వేర్వేరు పోలరైజేషన్‌తో ఒకే తెరపై పడతాయి.",
     "intent": "sharing_surprising_fact",
     "claims": [
      "c_f3"
     ]
    },
    {
     "speaker": "arjun",
     "tone": "awe",
     "text": "అంటే ఈ అద్దాలు ఏ బొమ్మ ఏ కంటికి వెళ్లాలో వేరు చేస్తాయా?",
     "kind": "prediction",
     "tts": "అంటే ఈ అద్దాలు ఏ బొమ్మ ఏ కంటికి వెళ్లాలో వేరు చేస్తాయా?",
     "intent": "asking_confirmation",
     "act": {
      "expr": "surprised"
     }
    },
    {
     "speaker": "meera",
     "tone": "explainer",
     "text": "అవును. ఒక్కో లెన్స్‌లోని ఫిల్టర్ దానికి సరిపోయే బొమ్మను ఆ కంటికి అందిస్తుంది.",
     "kind": "fact",
     "tts": "అవును. ఒక్కో లెన్స్‌లోని ఫిల్టర్ దానికి సరిపోయే బొమ్మను ఆ కంటికి అందిస్తుంది.",
     "intent": "reflective_explanation",
     "claims": [
      "c_f4"
     ]
    }
   ],
   "story": {
    "world": "cinema_seats",
    "visuals": [
     {
      "id": "v1",
      "intent": "two_friend",
      "medium": "character_scene",
      "recipe": "character",
      "character": {
       "world": "cinema_seats",
       "shot": "two_shot",
       "chars": {
        "arjun": {
         "wear": [
          "3d_glasses"
         ]
        },
        "meera": {
         "wear": [
          "3d_glasses"
         ]
        }
       },
       "changes": [
        {
         "t": 0.0,
         "chars": {
          "arjun": {
           "expr": "curious",
           "gesture": "idle"
          },
          "meera": {
           "gesture": "idle",
           "expr": "listening"
          }
         }
        },
        {
         "t": 0.8333,
         "chars": {
          "meera": {
           "gesture": "explain"
          },
          "arjun": {
           "gesture": "nod",
           "expr": "listening"
          }
         }
        }
       ]
      },
      "start": {
       "line": 0,
       "at": 0.0
      },
      "description": "Same seated friends, glasses remain worn; the screen image is an illustration of perceived depth, not a physical object entering the room."
     },
     {
      "id": "v2",
      "intent": "animated_illustration",
      "medium": "animated_demo",
      "recipe": "demo",
      "demo": {
       "template": "projector_glasses",
       "world": "cinema_seats"
      },
      "start": {
       "line": 1,
       "at": 0.2
      },
      "description": "Schematic of a polarized 3D system; consistently label the two image channels and matching eye filters. Do not imply every 3D system works this way. Keep this diagram visible through the final explanatory turn; no reaction cutaway until the next scene."
     }
    ]
   }
  },
  {
   "shot_id": "sc05",
   "recipe": "story",
   "beat": "reveal",
   "story_beat": "reveal",
   "mood": "joy",
   "location": "cinema_seats",
   "cast": [
    "arjun",
    "meera"
   ],
   "claims": [
    "c_f7",
    "c_f2"
   ],
   "lines": [
    {
     "speaker": "meera",
     "tone": "friendly",
     "text": "అద్దాలు అలాగే ఉంచు. ఒక్క కన్ను మూసి చూడు.",
     "kind": "instruction",
     "tts": "అద్దాలు అలాగే ఉంచు. ఒక్క కన్ను మూసి చూడు.",
     "intent": "guiding_demo",
     "act": {
      "expr": "smile"
     }
    },
    {
     "speaker": "arjun",
     "tone": "awe",
     "text": "నాకు దగ్గరికి వస్తున్నట్టు అనిపించడం తగ్గింది. కానీ బొమ్మలో ఇంకా కొంత లోతు కనిపిస్తోంది.",
     "kind": "observation",
     "tts": "నాకు దగ్గరికి వస్తున్నట్టు అనిపించడం తగ్గింది. కానీ బొమ్మలో ఇంకా కొంత లోతు కనిపిస్తోంది.",
     "intent": "enthusiastic_discovery",
     "act": {
      "expr": "curious"
     }
    },
    {
     "speaker": "meera",
     "tone": "explainer",
     "text": "అవును. వస్తువుల పరిమాణం లాంటి సూచనలతో ఒక్క కన్నుతోనూ లోతును అంచనా వేయగలం.",
     "kind": "fact",
     "tts": "అవును. వస్తువుల పరిమాణం లాంటి సూచనలతో ఒక్క కన్నుతోనూ లోతును అంచనా వేయగలం.",
     "intent": "explaining_observation",
     "claims": [
      "c_f7"
     ]
    },
    {
     "speaker": "arjun",
     "tone": "happy",
     "text": "ఓహో! బొమ్మ బయటికి రాలేదు. లోతు ఉన్నట్టు నాకు అనిపించింది!",
     "kind": "observation",
     "tts": "ఓహో! బొమ్మ బయటికి రాలేదు. లోతు ఉన్నట్టు నాకు అనిపించింది!",
     "intent": "quiet_realization",
     "claims": [
      "c_f2"
     ],
     "act": {
      "expr": "happy"
     }
    }
   ],
   "story": {
    "world": "cinema_seats",
    "visuals": [
     {
      "id": "v1",
      "intent": "two_friend",
      "medium": "character_scene",
      "recipe": "character",
      "character": {
       "world": "cinema_seats",
       "shot": "two_shot",
       "chars": {
        "arjun": {
         "wear": [
          "3d_glasses"
         ]
        },
        "meera": {
         "wear": [
          "3d_glasses"
         ]
        }
       },
       "changes": [
        {
         "t": 0.0,
         "chars": {
          "meera": {
           "expr": "smile",
           "gesture": "point"
          },
          "arjun": {
           "gesture": "idle",
           "expr": "listening"
          }
         }
        }
       ]
      },
      "start": {
       "line": 0,
       "at": 0.0
      },
      "description": "Same cinema, same seats and glasses. Keep subtitles clear of faces."
     },
     {
      "id": "v2",
      "intent": "animated_illustration",
      "medium": "animated_demo",
      "recipe": "demo",
      "demo": {
       "template": "eye_views",
       "world": "cinema_seats"
      },
      "start": {
       "line": 1,
       "at": 0.0
      },
      "description": "Hold the explanatory eye-view illustration through the monocular-depth qualification and final answer. This is a schematic, not a stereoscopic phone display."
     }
    ]
   }
  },
  {
   "shot_id": "sc06",
   "recipe": "story",
   "beat": "payoff",
   "story_beat": "closing",
   "mood": "joy",
   "location": "cinema_seats",
   "cast": [
    "arjun",
    "meera"
   ],
   "claims": [],
   "lines": [
    {
     "speaker": "meera",
     "tone": "happy",
     "text": "ఇప్పుడు సినిమా చూస్తావా, popcornకి దారి ఇస్తావా?",
     "kind": "reaction",
     "tts": "ఇప్పుడు సినిమా చూస్తావా, పాప్‌కార్న్కి దారి ఇస్తావా?",
     "intent": "playful_remark",
     "act": {
      "expr": "smile"
     }
    },
    {
     "speaker": "arjun",
     "tone": "happy",
     "text": "విషయం అర్థమైంది… కానీ తల మాత్రం వినట్లేదు!",
     "kind": "reaction",
     "tts": "విషయం అర్థమైంది… కానీ తల మాత్రం వినట్లేదు!",
     "intent": "playful_remark",
     "act": {
      "expr": "smile",
      "gesture": "duck"
     }
    }
   ],
   "story": {
    "world": "cinema_seats",
    "visuals": [
     {
      "id": "v1",
      "intent": "two_friend",
      "medium": "character_scene",
      "recipe": "character",
      "character": {
       "world": "cinema_seats",
       "shot": "two_shot",
       "chars": {
        "arjun": {
         "wear": [
          "3d_glasses"
         ]
        },
        "meera": {
         "wear": [
          "3d_glasses"
         ]
        }
       },
       "changes": [
        {
         "t": 0.0,
         "chars": {
          "meera": {
           "expr": "smile",
           "gesture": "idle"
          },
          "arjun": {
           "gesture": "idle",
           "expr": "listening"
          }
         }
        }
       ]
      },
      "start": {
       "line": 0,
       "at": 0.0
      },
      "description": "Same cinema, same seats and glasses. Keep subtitles clear of faces."
     },
     {
      "id": "v2",
      "intent": "two_friend",
      "medium": "character_scene",
      "recipe": "character",
      "character": {
       "world": "cinema_seats",
       "shot": "two_shot",
       "chars": {
        "arjun": {
         "wear": [
          "3d_glasses"
         ],
         "gesture": "duck",
         "expr": "smile"
        },
        "meera": {
         "wear": [
          "3d_glasses"
         ]
        }
       },
       "changes": [
        {
         "t": 0.0,
         "chars": {
          "arjun": {
           "expr": "smile",
           "gesture": "duck"
          },
          "meera": {
           "gesture": "nod",
           "expr": "listening"
          }
         }
        }
       ]
      },
      "start": {
       "line": 1,
       "at": 0.0
      },
      "description": "Same cinema, same seats and glasses. Keep subtitles clear of faces."
     }
    ]
   }
  }
 ],
 "director": [
  {
   "scene": "sc01",
   "visual": "v1",
   "intent": "establishing",
   "medium": "character_scene",
   "recipe": "character",
   "trail": []
  },
  {
   "scene": "sc01",
   "visual": "v2",
   "intent": "closeup",
   "medium": "character_scene",
   "recipe": "character",
   "trail": []
  },
  {
   "scene": "sc02",
   "visual": "v1",
   "intent": "reaction",
   "medium": "character_scene",
   "recipe": "character",
   "trail": []
  },
  {
   "scene": "sc02",
   "visual": "v2",
   "intent": "two_friend",
   "medium": "character_scene",
   "recipe": "character",
   "trail": []
  },
  {
   "scene": "sc03",
   "visual": "v1",
   "intent": "closeup",
   "medium": "character_scene",
   "recipe": "character",
   "trail": []
  },
  {
   "scene": "sc03",
   "visual": "v2",
   "intent": "animated_illustration",
   "medium": "animated_demo",
   "recipe": "demo",
   "trail": []
  },
  {
   "scene": "sc04",
   "visual": "v1",
   "intent": "two_friend",
   "medium": "character_scene",
   "recipe": "character",
   "trail": []
  },
  {
   "scene": "sc04",
   "visual": "v2",
   "intent": "animated_illustration",
   "medium": "animated_demo",
   "recipe": "demo",
   "trail": []
  },
  {
   "scene": "sc05",
   "visual": "v1",
   "intent": "two_friend",
   "medium": "character_scene",
   "recipe": "character",
   "trail": []
  },
  {
   "scene": "sc05",
   "visual": "v2",
   "intent": "animated_illustration",
   "medium": "animated_demo",
   "recipe": "demo",
   "trail": []
  },
  {
   "scene": "sc06",
   "visual": "v1",
   "intent": "two_friend",
   "medium": "character_scene",
   "recipe": "character",
   "trail": []
  },
  {
   "scene": "sc06",
   "visual": "v2",
   "intent": "two_friend",
   "medium": "character_scene",
   "recipe": "character",
   "trail": []
  }
 ],
 "music": {
  "mode": "procedural",
  "genre": "curiosity",
  "cues": []
 },
 "metadata": {
  "title": "త్రీడీ సినిమాలో బొమ్మ మన దగ్గరికి వచ్చినట్టు ఎందుకు కనిపిస్తుంది?",
  "summary": "Popcorn దగ్గరికి వచ్చినట్టు ఎందుకు కనిపించిందన్న ప్రశ్నకు, రెండు కళ్ల చూపు మరియు పోలరైజ్డ్ అద్దాల పని ద్వారా సమాధానం; చివరలో చిన్న నవ్వు.",
  "disclaimer": "వికీపీడియా (Depth perception, Polarized 3D system, Stereopsis, Stereoscopy) ఆధారంగా సరళంగా వివరించిన వీడియో. అర్జున్ మరియు మీరా కల్పిత, సొంతంగా గీసిన యానిమేషన్ పాత్రలు; 'దృష్టాంతం' అని ఉన్న దృశ్యాలు పునర్నిర్మాణాలు, నిజమైన ఫుటేజ్ కాదు.\nSimplified explainer based on Wikipedia (Depth perception, Polarized 3D system, Stereopsis, Stereoscopy). Arjun and Meera are fictional, original animated characters; scenes marked 'Illustration' are re-enactments, not footage.",
  "category": "Education",
  "audience": "Not made for kids",
  "tags": [
   "why do 3d movies look real? (telugu–english)",
   "science explained",
   "explainer",
   "3d",
   "popcorn",
   "అర్జున్",
   "మీరా"
  ]
 },
 "visual_style_version": "storybook_v1",
 "cinematic_characters": true,
 "character_assets": {
  "arjun": {
   "version": "cinematic-v2",
   "status": "pending_runtime_binding"
  },
  "meera": {
   "version": "cinematic-v2",
   "status": "pending_runtime_binding"
  }
 },
 "strict_open_source": true,
 "tts_backend": "indic_parler",
 "voice_policy": {
  "asr_retake_at": 0.6,
  "gemini_asr_repairs": false,
  "speech_compare": "bilingual"
 },
 "code_mix": "natural",
 "environments": {
  "cinema": "v1"
 },
 "story_revision": "three_d_friends_te_v5",
 "localization": {},
 "review": [
  "New v5 Telugu dialogue and pronunciation require review"
 ],
 "script_review_status": "pending"
}""")

PLAN = PLAN_EMBEDDED
# BEGIN RUNTIME CHARACTER BINDING
import cinematic_workflow as CW
os.environ['STORYBOOK_ASSETS'] = str(ASSETS)
PLAN = CW.bind_plan(PLAN_EMBEDDED, ASSETS, CHARACTER_VERSIONS)
# END RUNTIME CHARACTER BINDING
colab_budget.validate_target(PLAN)
colab_budget.validate_v4_continuity(PLAN)
plan_path = PROJ / 'plan.json'
if plan_path.exists() and json.loads(plan_path.read_text()) != PLAN:
    raise RuntimeError('Saved plan differs from this story or these character hashes. Keep the previous project '
                       'and choose a new PROJECT folder in Settings for changed assets/story.')
if not plan_path.exists():
    temporary = plan_path.with_suffix('.partial')
    temporary.write_text(json.dumps(PLAN, indent=2, ensure_ascii=False))
    temporary.replace(plan_path)

STYLE, NEG = PLAN["style"]["suffix"], PLAN["style"]["negative"]
SHOTS, MOODS, CHARS, TONES = PLAN["shots"], PLAN["moods"], PLAN["characters"], PLAN["tones"]
SFX_TYPES = {"boom", "rumble", "whoosh", "shimmer", "ding", "click", "pop", "coins", "rustle"}
DRAWN_RECIPES = {"diagram", "ui", "journey", "card", "story", "character", "demo"}   # made by code (no AI, no GPU)
if PLAN.get("bible"):                                  # two-friend videos: the plan's own Character Bible
    charanim.set_bible(characters.load(PLAN["bible"]))
if PLAN.get("visual_style_version"):                   # storybook: install the pinned asset versions before checks
    os.environ.setdefault("STORYBOOK_ASSETS", str(Path(DRIVE_ROOT) / "_assets"))
    STORYBOOK_PREP = storybook.prepare_assets(PLAN, root=ASSETS, log=log)
    print("Storybook assets: " + "; ".join(f"{k}: {v}" for k, v in STORYBOOK_PREP.items()))
VOICE_FX_TYPES, TONE_FX_TYPES = {"none", "sparkle", "old"}, {"shaky", "breathy"}
KOKORO_EN_VOICES = {"af_alloy", "af_aoede", "af_bella", "af_heart", "af_jessica", "af_kore", "af_nicole", "af_nova",
                    "af_river", "af_sarah", "af_sky", "am_adam", "am_echo", "am_eric", "am_fenrir", "am_liam",
                    "am_michael", "am_onyx", "am_puck", "am_santa", "bf_alice", "bf_emma", "bf_isabella", "bf_lily",
                    "bm_daniel", "bm_fable", "bm_george", "bm_lewis"}
def mood(shot):
    return MOODS[shot.get("mood", "calm")]

# checks: the plan may only use characters, voices, tones and effects this notebook knows
assert len({s["shot_id"] for s in SHOTS}) == len(SHOTS), "duplicate shot ids"
PLAN_LANGUAGE = PLAN.get("language", "en")
if PLAN_LANGUAGE == "en":
    VOICE_NAMES = KOKORO_EN_VOICES
else:                                                  # Indic Parler-TTS speakers trained for this language
    import parler_worker
    VOICE_NAMES = set(parler_worker.SPEAKERS.get(PLAN_LANGUAGE, []))
    assert VOICE_NAMES, f"no voice model configured for language '{PLAN_LANGUAGE}'"
for cid, c in CHARS.items():
    assert c["voice"] in VOICE_NAMES, f"{cid}: unknown voice {c['voice']}; choose from {sorted(VOICE_NAMES)}"
    assert c.get("fx", "none") in VOICE_FX_TYPES, f"{cid}: unknown voice effect {c.get('fx')}"
for tid, t in TONES.items():
    assert t.get("fx") in TONE_FX_TYPES | {None}, f"tone {tid}: unknown effect {t.get('fx')}"
for s in SHOTS:
    sid = s["shot_id"]
    assert s.get("mood", "calm") in MOODS, f"{sid}: unknown mood {s.get('mood')}"
    for c in s.get("cast", []):
        assert c in CHARS, f"{sid}: unknown character {c}"
    for l in s["lines"]:
        assert l["speaker"] in CHARS, f"{sid}: unknown speaker {l['speaker']}"
        assert l["tone"] in TONES, f"{sid}: unknown tone {l['tone']}"
    for c in s.get("sfx", []):
        assert c["type"] in SFX_TYPES, f"{sid}: unknown sound effect {c['type']}"

# ---- continuity check of the plan (before any GPU time is spent) ----
# 1. whoever speaks (except the narrator) must be on screen in that shot
# 2. nobody may vanish for one shot and come back in the same place (A-B-A), unless the shot is a cutaway
# 3. every shot must fit in ONE continuous AI clip (5 s, slowed down at most MAX_SLOWMO): a longer shot would need
#    a second, unrelated clip, and characters would jump or vanish in the middle of it
import unicodedata
EST_SEC_PER_LETTER = 0.1                             # Telugu speaking time per letter incl. vowel signs (conservative)
CLIP_SECONDS = 5.04                                  # Wan clips: 121 frames @ 24 fps or 81 frames @ 16 fps
SHOT_BUDGET = CLIP_SECONDS * MAX_SLOWMO              # longest shot that is still one continuous clip
def est_seconds(shot):
    """Conservative speaking-time estimate (the real one comes after the voices are made)."""
    t = float(shot.get("pre_s", 0.3))
    for l in shot["lines"]:
        c, tn = CHARS[l["speaker"]], TONES[l["tone"]]
        txt = l.get("tts") or l["text"]
        speed = c.get("speed", 1.0) * tn.get("speed", 1.0) * SPEED_SCALE
        if PLAN_LANGUAGE == "en":
            t += len(txt.split()) / (3.0 * speed)
        else:                                        # long, inflected words: estimate from letters instead
            t += EST_SEC_PER_LETTER * sum(1 for ch in txt if ch.isalnum() or unicodedata.category(ch)[0] == "M") / speed
        t += 0.45 * txt.count("...") + 0.12 * len(re.findall(r"[.!?,;:]", txt))
        t += float(l.get("pause_after", tn.get("pause_after", MOODS[shot.get("mood", "calm")]["pause_after"])))
    return t

problems, place = [], lambda s: s.get("location", s.get("scene", s["shot_id"]))
ARCH = PLAN.get("architecture", {"nodes": {}, "edges": []})
for s in SHOTS:
    if s.get("recipe", "video") not in {"video", "stock"} | DRAWN_RECIPES:
        problems.append(f"{s['shot_id']}: unknown recipe {s.get('recipe')} (use video, stock, ui, journey, card, "
                        "diagram, story, character or demo)")
    if s.get("recipe") == "stock":                  # licensed image: file + credit + an allowed licence, any length
        st = s.get("stock", {})
        if not st.get("file") or not (PROJ / "assets" / st["file"]).exists():
            problems.append(f"{s['shot_id']}: licensed image file missing (assets/{st.get('file')})")
        if not st.get("credit") or not st.get("license_ok"):
            problems.append(f"{s['shot_id']}: licensed image needs its credit and an allowed licence")
        continue
    if s.get("recipe") in ("story", "character", "demo"):  # two-friend scenes: drawn by code, any length
        for v in (s.get("story") or {}).get("visuals", []):
            if v.get("recipe") == "photo" and not os.path.isabs(v["photo"].get("file", "")):
                v["photo"]["file"] = str(PROJ / "assets" / v["photo"].get("file", ""))   # licensed image on Drive
            st_ = v.get("start", {})
            if not 0 <= st_.get("line", -1) < len(s["lines"]) or not 0 <= st_.get("at", -1) <= 1:
                problems.append(f"{s['shot_id']}: visual {v.get('id')} starts at {st_} (no such line/fraction)")
        problems += [f"{s['shot_id']}: {m}" for m in drawkit.check_spec(s["recipe"], s.get(s["recipe"], {}), PLAN)]
        for l in s["lines"]:
            if l["speaker"] != "narrator" and l["speaker"] not in s.get("cast", []):
                problems.append(f"{s['shot_id']}: {CHARS[l['speaker']]['name']} speaks but is not in the scene's cast")
        continue
    if s.get("recipe") in ("ui", "journey", "card"):        # drawn scenes: check the spec, no AI clip budget
        problems += [f"{s['shot_id']}: {m}" for m in drawkit.check_spec(s["recipe"], s.get(s["recipe"], {}), PLAN)]
        for l in s["lines"]:
            if l["speaker"] != "narrator" and l["speaker"] not in s.get("cast", []):
                problems.append(f"{s['shot_id']}: {CHARS[l['speaker']]['name']} speaks but is not on screen")
        continue
    if s.get("recipe") == "diagram":                # drawn shots: check they only use boxes that exist
        dg = s.get("diagram", {})
        used = set(dg.get("focus", [])) | ({*dg["show"]} if isinstance(dg.get("show"), list) else set())
        used |= {f[k] for f in dg.get("flows", []) for k in ("from", "to")}
        for n in sorted(used - set(ARCH["nodes"])):
            problems.append(f"{s['shot_id']}: diagram uses unknown box '{n}'")
        tab = dg.get("table")
        for c in (tab or {}).get("changes", []):
            if c["col"] not in tab["columns"] or c["row"] >= len(tab["rows"]):
                problems.append(f"{s['shot_id']}: table change {c} does not match the table")
        if s.get("cast"):
            problems.append(f"{s['shot_id']}: a diagram shot can't have a cast")
        continue
    for l in s["lines"]:
        if l["speaker"] != "narrator" and l["speaker"] not in s.get("cast", []):
            problems.append(f"{s['shot_id']}: {CHARS[l['speaker']]['name']} speaks but is not on screen (add to cast)")
    if est_seconds(s) > SHOT_BUDGET and not s.get("allow_long"):
        problems.append(f"{s['shot_id']}: ~{est_seconds(s):.1f}s of speech is longer than one clip "
                        f"({SHOT_BUDGET:.1f}s): split it into two shots")
for a, b in ARCH["edges"]:
    if a not in ARCH["nodes"] or b not in ARCH["nodes"]:
        problems.append(f"architecture: arrow {a} -> {b} uses an unknown box")
for a, b, c in zip(SHOTS, SHOTS[1:], SHOTS[2:]):
    if place(a) == place(b) == place(c) and not b.get("cutaway"):
        for who in set(a.get("cast", [])) & set(c.get("cast", [])) - set(b.get("cast", [])):
            problems.append(f"{b['shot_id']}: {CHARS[who]['name']} is in {a['shot_id']} and {c['shot_id']} at the same "
                            f"place but vanishes in {b['shot_id']} (add them, or mark the shot \"cutaway\": true)")
# ---- scene validation (engine/lib/validate.py): logic, facts, process accuracy, narration ↔ visual, consistency
ISSUES = validate.validate(PLAN, {"words_per_second": 2.6})
if PLAN.get("story_mode") == "two_friends_cinematic":
    # dialogue rules of the Story Director (sourced factual turns, no monologue, real questions, corrected guesses)
    # re-checked here on the plan as it will be made (plan.json may have been edited on Drive)
    try:
        _st = story.Story(
            title=PLAN["title"], topic=PLAN["title"], language=PLAN.get("language", "en"),
            facts=[story.StoryFact(id=c["id"], text=c["text"], source_url=c["sources"][0]) for c in PLAN.get("claims", [])],
            scenes=[story.Scene(id=s["shot_id"], beat=s.get("story_beat", "first_answer"), world=s.get("location", "studio"),
                                turns=[story.Turn(speaker=l["speaker"], text=l["text"], kind=l.get("kind", "observation"),
                                                  fact_ids=l.get("claims", []),
                                                  tone=l["tone"] if l["tone"] in story.TONES else "friendly")
                                       for l in s["lines"]],
                                visuals=[story.Visual(id=v["id"], intent=v["intent"], turn=v["start"]["line"],
                                                      at=v["start"]["at"]) for v in s["story"]["visuals"]])
                    for s in SHOTS if s.get("recipe") == "story"])
    except Exception as e:                             # pydantic ValidationError: report it like any plan problem
        _st = None
        ISSUES.append({"level": "error", "rule": "D0", "shot": None, "message": f"story structure: {e}"[:1500],
                       "fix": "see colab/engine/lib/story.py for the schema"})
    ISSUES += story.check_story(_st, charanim.bible()) if _st else []
    n_ai = sum(s.get("recipe", "video") == "video" for s in SHOTS)
    if n_ai > MAX_AI_VIDEO_SHOTS:
        ISSUES.append({"level": "error", "rule": "X6", "shot": None, "message": f"{n_ai} AI video shots in a story-mode "
                       f"plan (MAX_AI_VIDEO_SHOTS = {MAX_AI_VIDEO_SHOTS})", "fix": "use character scenes, demonstrations "
                       "or licensed media, or raise MAX_AI_VIDEO_SHOTS"})
# ---- scene-state engine (engine/lib/scene_state.py): positions, facing, camera side, ownership, ordered events.
# Simulated shot by shot; contradictions (crossing the line, reversals, objects changing hands without an event,
# acting from across the room, too many actions for one clip, actions ending at the cut) stop the run here.
STATES, SCENE_ENTS, _st_problems = scene_state.parse_plan(PLAN)
STATES = {st.shot_id: st for st in STATES}
_worlds, _st_issues = scene_state.check_sequence(list(STATES.values()), SCENE_ENTS)
ISSUES += [i.model_dump() for i in _st_problems + _st_issues]
for s in SHOTS:
    if s.get("state") and s.get("recipe", "video") == "video" and not s.get("fallback"):
        ISSUES.append({"level": "warn", "rule": "X5", "shot": s["shot_id"], "message": "AI shot with required actions has "
                       "no drawn \"fallback\": if its clip fails the checks, a still picture is used instead", "fix":
                       "add \"fallback\": {\"recipe\": \"card\", ...}"})
if STATES:
    print(f"Scene-state: {len(STATES)} shots with positions, camera side and ordered events; "
          f"{sum(len(st.events) for st in STATES.values())} events simulated")
ISSUES += [{"level": "error", "rule": "C", "shot": p.split(":")[0], "message": p, "fix": ""} for p in problems]
(PROJ / "render").mkdir(exist_ok=True)
(PROJ / "render" / "validation_report.json").write_text(json.dumps(ISSUES, indent=2, ensure_ascii=False))
(PROJ / "render" / "validation_report.txt").write_text(validate.report(ISSUES) or "No issues.")
if ISSUES:
    print("Scene validation:\n" + validate.report(ISSUES))
if validate.errors(ISSUES):
    log("plan validation errors:\n" + validate.report(validate.errors(ISSUES)), "ERROR")
    raise ValueError(f"The plan failed validation ({len(validate.errors(ISSUES))} errors, listed above and in "
                     "render/validation_report.txt). Nothing was generated: fix plan.json and run again.")
n_dg = sum(s.get("recipe") in DRAWN_RECIPES for s in SHOTS)
# names and special words for the speech check (it would otherwise hear "Zomato" as "the motto")
ASR_PROMPT = ", ".join(dict.fromkeys([c["name"] for k, c in CHARS.items() if k != "narrator"] + PLAN.get("vocabulary", [])))
log(f"speech-check vocabulary: {ASR_PROMPT}")
print(f"Continuity check of the plan: OK (every AI shot fits one {SHOT_BUDGET:.1f}s clip; speakers on screen; "
      f"nobody vanishes; {n_dg} drawn scenes checked; {len(validate.errors(ISSUES))} validation errors, "
      f"{len(ISSUES) - len(validate.errors(ISSUES))} warnings)")

# turn the plan into what the shared video engine uses: the same character description in every prompt
# keeps each character looking (roughly) the same from shot to shot; "must_show" is what the clip check looks for
for s in SHOTS:
    for k, l in enumerate(s["lines"], 1):
        l["id"] = f"{s['shot_id']}_{k}"
    if s.get("recipe") in DRAWN_RECIPES | {"stock"}:  # drawn scenes and licensed images: no AI prompts, never AI video
        s["must_show"] = []
        continue
    looks = [CHARS[c]["look"] for c in s.get("cast", []) if CHARS[c].get("look")]
    props = PLAN.get("props", {})
    looks += [props[p]["look"] for p in s.get("props", []) if p in props]   # same object, same words, every shot
    s["recipe"] = "video"
    s["image_prompt"] = ", ".join([s["scene"], *looks])
    s["motion_prompt"] = s["action"]
    if s["shot_id"] in STATES:                        # spatial wording from the scene-state (the layout video steers)
        s["motion_prompt"] = s["action"] + ", " + scene_state.prompt_hints(STATES[s["shot_id"]], SCENE_ENTS)
    s["must_show"] = [CHARS[c].get("tag") or CHARS[c]["look"].split(",")[0]
                      for c in s.get("cast", []) if CHARS[c].get("look") and not s.get("skip_check")]
    s["must_show"] += [props[p]["tag"] for p in s.get("props", [])            # small objects: only if "check": true
                       if props.get(p, {}).get("tag") and props[p].get("check") and not s.get("skip_check")]

n_words = sum(len(l["text"].split()) for s in SHOTS for l in s["lines"])
print(f'"{PLAN["title"]}": {len(SHOTS)} shots, {sum(len(s["lines"]) for s in SHOTS)} lines, {n_words} words')
for cid, c in CHARS.items():
    n = sum(l["speaker"] == cid for s in SHOTS for l in s["lines"])
    print(f"  {c['name']:<9} voice {c['voice']:<10} pitch {c.get('pitch', 0):+.1f}  effect {c.get('fx', 'none'):<8} {n} lines")
print("Story arc:", " → ".join(s.get("mood", "calm") for s in SHOTS))
log("Prompts:\n" + "\n".join(f"{s['shot_id']}: {s['image_prompt']} | {s['motion_prompt']}"
                              for s in SHOTS if s.get("motion_prompt")))
free_tier_estimate(sum(1 for s in SHOTS if s.get("motion_prompt")), sum(1 for s in SHOTS if s.get("recipe") in
                   globals().get("DRAWN_RECIPES", {"diagram"})))


In [ ]:
BUDGET.check('helpers')
# ==== 5. Helpers ====
def ff(*args, nice=False):
    """Run ffmpeg; every command, its time and any messages go to the log. Raise with the error text if it fails.
    nice=True: low CPU priority, so background edits don't slow down the GPU work."""
    cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-filter_threads", str(CPU_THREADS), "-filter_complex_threads", str(CPU_THREADS), *map(str, args)]
    if nice and shutil.which("nice"):
        cmd = ["nice", "-n", "10"] + cmd
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=BUDGET.timeout(600))
    log(f"ffmpeg ({time.time() - t0:.1f}s, exit {r.returncode}): " + " ".join(cmd)[:4000], "FFMPG")
    if r.stderr.strip():
        log("ffmpeg said: " + r.stderr[-3000:], "ERROR" if r.returncode else "WARN")
    if r.returncode:
        raise RuntimeError("ffmpeg failed:\n" + " ".join(cmd) + "\n" + r.stderr[-3000:])

import unicodedata
LANGUAGE = PLAN.get("language", "en")       # "en" (Kokoro voices) or "te" (Telugu: Indic Parler-TTS, Hind Guntur)
SCRIPT_FONTS = {"te": ("Hind Guntur", "https://github.com/google/fonts/raw/main/ofl/hindguntur/HindGuntur-{w}.ttf",
                       "ITF Hind Guntur (SIL Open Font License): Telugu + Latin, so 'PNR', 'WL 12', '₹' render too")}
SUB_FONT, FONTS_DIR = "DejaVu Sans", None   # subtitle font (libass) and where its file is

def setup_script_font():
    """Non-English video: one font with the script AND Latin letters for every drawn text and the subtitles.
    Downloaded once (OFL) to the model cache. Telugu needs complex shaping (raqm), or vowel signs break."""
    global SUB_FONT, FONTS_DIR
    if LANGUAGE == "en":
        return
    if LANGUAGE not in SCRIPT_FONTS:
        raise ValueError(f"language '{LANGUAGE}' has no font configured (SCRIPT_FONTS in the helpers cell)")
    from PIL import features
    if not features.check("raqm"):
        raise RuntimeError("Pillow can't shape Telugu text here (no raqm/fribidi): letters would be drawn broken. "
                           "Run: !apt-get -qq install libfribidi0 libraqm0, then Runtime → Restart session.")
    name, url, why = SCRIPT_FONTS[LANGUAGE]
    FONTS_DIR = MODEL_CACHE / "fonts"; FONTS_DIR.mkdir(parents=True, exist_ok=True)
    files = {}
    for w in ("Regular", "Bold"):
        f = FONTS_DIR / Path(url.format(w=w)).name
        if not f.exists() or f.stat().st_size < 50_000:
            urllib.request.urlretrieve(url.format(w=w), f.with_suffix(".part")); f.with_suffix(".part").rename(f)
        files[w] = f
    drawkit.set_script_font(files["Regular"], files["Bold"])
    SUB_FONT = name
    log(f"script font for '{LANGUAGE}': {name} ({why}); raqm shaping available")
    print(f"Language: {LANGUAGE} | text font: {name}")

import urllib.request
setup_script_font()

def sidecar(path, **info):
    Path(str(path) + ".json").write_text(json.dumps(info, indent=2))
    log(f"saved {Path(path).name}: " + json.dumps({k: v for k, v in info.items() if k != "words"})[:1500])

def read_sidecar(path):
    p = Path(str(path) + ".json")
    return json.loads(p.read_text()) if p.exists() else None

def free_gpu():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass
    try:                     # hand freed system RAM back to the OS (Python keeps it after big CPU-side loads otherwise)
        import ctypes; ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def ram_free_gb():
    """System RAM still available (GB); None if it can't be read."""
    try:
        import psutil
        return psutil.virtual_memory().available / 2**30
    except Exception:
        return None

def need_ram(gb, what):
    """Stop with a clear, logged error BEFORE a step that needs `gb` GB of system RAM when less is free: when Colab
    runs out of RAM it restarts the whole session and the error is lost."""
    free_gpu()
    free = ram_free_gb()
    log(f"RAM check for {what}: needs ~{gb:.1f} GB, {'?' if free is None else f'{free:.1f}'} GB free")
    if free is not None and free < gb + 1.0:          # 1 GB headroom for Python, FFmpeg and the notebook itself
        raise MemoryError(f"not enough system RAM for {what}: needs ~{gb:.1f} GB + 1 GB headroom, {free:.1f} GB free")

def gpu_mem():
    """'allocated / reserved / total GB' as PyTorch sees it (live tensors / held by the allocator / card)."""
    try:
        import torch
        if not torch.cuda.is_available():
            return "no GPU"
        a, r = torch.cuda.memory_allocated() / 2**30, torch.cuda.memory_reserved() / 2**30
        pk = torch.cuda.max_memory_reserved() / 2**30         # highest point since the last model release
        return (f"{a:.1f} allocated / {r:.1f} reserved / {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB"
                f" (peak {pk:.1f} GB)")
    except Exception:
        return "unknown"

def release_model(m, name="model"):
    """Really give a model's GPU memory back. `del` alone isn't enough when a library (accelerate hooks from
    device_map loading, a pipeline, a cache) still references the model: here every weight's storage is replaced by
    an empty CPU tensor, so the GPU memory is freed even if a hidden reference survives. Use only when done with it."""
    if m is None:
        return
    before = gpu_mem()
    try:
        import torch
        mods = [m] if isinstance(m, torch.nn.Module) else [c for c in getattr(m, "components", {}).values()
                                                           if isinstance(c, torch.nn.Module)]
        try:
            from accelerate.hooks import remove_hook_from_module
            for mod in mods:
                remove_hook_from_module(mod, recurse=True)
        except Exception:
            pass
        with torch.no_grad():
            for mod in mods:
                for t in list(mod.parameters()) + list(mod.buffers()):
                    t.data = torch.empty(0, dtype=t.dtype, device="cpu")
    except Exception:
        log_exc(f"release_model({name}) failed; falling back to garbage collection")
    free_gpu()
    log(f"released {name}: GPU {before} → {gpu_mem()}")
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()        # the next model's peak is measured on its own
    except Exception:
        pass

class GpuSampler:
    """Samples GPU utilisation and memory every second in the background (nvidia-smi), so 'is the GPU busy?' is
    answered by the whole interval, not by one snapshot. with GpuSampler() as gs: ...; gs.summary() -> dict."""
    def __init__(self, every=1.0):
        self.every, self.util, self.mem, self._stop = every, [], [], None

    def _run(self):
        while not self._stop.wait(self.every):
            try:
                r = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader,nounits"],
                                   capture_output=True, text=True, timeout=5)
                u, m = [float(x) for x in r.stdout.strip().splitlines()[0].split(",")]
                self.util.append(u); self.mem.append(m / 1024)
            except Exception:
                pass

    def __enter__(self):
        import threading
        self._stop = threading.Event()
        self._th = threading.Thread(target=self._run, daemon=True); self._th.start()
        return self

    def __exit__(self, *a):
        self._stop.set(); self._th.join(timeout=10)

    def summary(self):
        if not self.util:
            return {"gpu_util_avg": None, "gpu_util_samples": 0}
        busy = sum(u >= 50 for u in self.util) / len(self.util)
        return {"gpu_util_avg": round(sum(self.util) / len(self.util), 1), "gpu_busy_share": round(busy, 2),
                "gpu_mem_max_gb": round(max(self.mem), 2), "gpu_util_samples": len(self.util)}


def cuda_now():
    """Wall-clock time after all queued GPU work has finished (GPU calls return before the work is done)."""
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    except Exception:
        pass
    return time.time()


_ONES = ("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen "
         "sixteen seventeen eighteen nineteen").split()
_TENS = "_ _ twenty thirty forty fifty sixty seventy eighty ninety".split()

def num_words(n):
    """12 -> 'twelve', 100 -> 'one hundred' (so '100' and 'a hundred' count as the same words)."""
    if n < 20:
        return _ONES[n]
    if n < 100:
        return _TENS[n // 10] + ("" if n % 10 == 0 else " " + _ONES[n % 10])
    for size, name in [(10 ** 9, "billion"), (10 ** 6, "million"), (1000, "thousand"), (100, "hundred")]:
        if n >= size:
            return num_words(n // size) + " " + name + ("" if n % size == 0 else " " + num_words(n % size))

def norm_words(s):
    s = s.lower().replace("-", " ").replace("'", "").replace("\u2019", "")    # restaurant's == restaurants
    s = re.sub(r"(\d),(\d{3})", r"\1\2", s)                     # 1,000 -> 1000
    s = re.sub(r"\ba (hundred|thousand|million|billion)\b", r"one \1", s)
    s = re.sub(r"([a-z])(\d)", r"\1 \2", s); s = re.sub(r"(\d)([a-z])", r"\1 \2", s)
    s = re.sub(r"\d+", lambda m: num_words(int(m.group())) if len(m.group()) <= 12 else m.group(), s)
    # letters of ANY script are kept, with their vowel signs (Telugu "సీటు" is 4 code points; 2 are marks, which
    # [a-z0-9] — and even \w — would cut out); everything else is a word break
    return "".join(ch if ch == " " or unicodedata.category(ch)[0] in "LMN" else " " for ch in s).split()

def wer(ref, hyp):
    """Word error rate between two word lists."""
    dp = list(range(len(hyp) + 1))
    for i in range(1, len(ref) + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, len(hyp) + 1):
            prev, dp[j] = dp[j], min(dp[j] + 1, dp[j - 1] + 1, prev + (ref[i - 1] != hyp[j - 1]))
    return dp[-1] / max(len(ref), 1)

def speech_error(texts, heard):
    """How wrong the heard speech is, 0..1: the best of the word error rate and 2x the letter error rate (letters
    ignore spaces, so 'backend' / 'back end' or 'WebSocket' / 'web socket' don't count as mistakes).
    Other languages (Telugu): the letter error rate only — words are long and inflected, and Whisper's Telugu is far
    weaker than its English, so word-level scoring would flag correct speech; the threshold is SPEECH_RETRY_OTHER."""
    best = 1.0
    if LANGUAGE != "en" and (globals().get("PLAN") or {}).get("code_mix"):
        import telugu_dialogue                   # code-mixed Telugu: English words in either script, vowel noise
        return min(telugu_dialogue.bilingual_error(t, heard) for t in texts)
    hw = norm_words(heard)
    for t in texts:
        rw = norm_words(t)
        cer = wer(list("".join(rw)), list("".join(hw)))
        best = min(best, cer if LANGUAGE != "en" else min(wer(rw, hw), 2 * cer))
    return best

def ram(label=""):
    """Print system RAM and GPU memory use (Colab restarts the session if system RAM runs out)."""
    msg = resources()
    if msg:
        print(f"   {msg} {label}")

NVENC_OK = False
def pick_video_encoder():
    """NVIDIA hardware encoder (NVENC) when available, else the CPU x264 encoder."""
    global NVENC_OK
    try:
        test = WORK / "nvenc_test.mp4"
        ff("-f", "lavfi", "-i", "testsrc2=size=1280x720:rate=24", "-frames:v", 24,
           "-c:v", "h264_nvenc", "-pix_fmt", "yuv420p", test)
        print("Video encoding: GPU (NVENC)")
        NVENC_OK = True
        return ["-c:v", "h264_nvenc", "-preset", "p5", "-rc", "vbr", "-cq", "19", "-b:v", "0",
                "-profile:v", "high", "-pix_fmt", "yuv420p", "-r", FPS]
    except Exception:
        log_exc("NVENC not available")
        print("Video encoding: CPU (x264)")
        return ["-c:v", "libx264", "-preset", X264_PRESET, "-crf", "18", "-pix_fmt", "yuv420p", "-r", FPS]
VENC = pick_video_encoder()
if not NVENC_OK:
    VENC += ["-threads", str(CPU_THREADS)]
DEVICE_REPORT["shot + final video encoding"] = "GPU (NVENC)" if NVENC_OK else "CPU (x264)"

STATUS = {}          # shot_id -> how its visual was made (for the QC report)

# ---- Hugging Face models kept on Drive as REAL files ----
# The normal Hugging Face cache stores each file as a symlink into blobs/. Google Drive does not keep symlinks: in a
# later session the "file" is the link text ("../../blobs/…"), and e.g. faster-whisper fails with "Unsupported model
# binary version v774843950" (= the bytes "../."). So models cached on Drive are downloaded as plain files into
# MODEL_CACHE/hf_files/<repo> (snapshot_download with local_dir), checked, and re-downloaded if anything is broken.
WHISPER_REPOS = {"tiny": "Systran/faster-whisper-tiny", "tiny.en": "Systran/faster-whisper-tiny.en",
                 "small.en": "Systran/faster-whisper-small.en",
                 "large-v3-turbo": "mobiuslabsgmbh/faster-whisper-large-v3-turbo"}

def _broken_file(f):
    """A link that didn't survive Drive (tiny text file holding a path) or a file that can't be read."""
    try:
        if f.is_symlink():
            return not f.exists()
        if f.stat().st_size < 1024:                     # real small files are JSON/text, never a bare path
            head = f.read_bytes()[:3]
            return head == b"../" or head[:1] == b"/"
        return False
    except OSError:
        return True

def hf_files(repo, what="model", allow=None):
    """Folder with the repo's real files on Drive (downloaded once, at the licence-verified revision when known)."""
    d = MODEL_CACHE / "hf_files" / repo.replace("/", "--")
    files = [f for f in d.rglob("*") if f.is_file() and ".cache" not in f.parts] if d.exists() else []
    if (d / ".complete").exists() and files and not any(_broken_file(f) for f in files):
        return d
    if d.exists():
        log(f"{what}: cached files of {repo} are incomplete or broken links; downloading again", "WARN")
        shutil.rmtree(d, ignore_errors=True)
    from huggingface_hub import snapshot_download
    t0, rev = time.time(), registry.REVISIONS.get(repo)
    try:
        snapshot_download(repo, local_dir=str(d), revision=rev, allow_patterns=allow)
    except Exception:
        log_exc(f"{what}: download of {repo} at revision {rev} failed; trying the current revision")
        snapshot_download(repo, local_dir=str(d), allow_patterns=allow); rev = "main"
    bad = [str(f) for f in d.rglob("*") if f.is_file() and ".cache" not in f.parts and _broken_file(f)]
    if bad:
        raise RuntimeError(f"{what}: downloaded files of {repo} are broken: {bad[:3]}")
    (d / ".complete").write_text(json.dumps({"repo": repo, "revision": rev, "time": time.strftime("%Y-%m-%d %H:%M")}))
    log(f"{what}: {repo} saved as plain files in {d} ({time.time() - t0:.0f}s)")
    return d

# ---- licence gate: models marked "review" in the registry are refused in strict mode (decided BEFORE any download)
LICENSE_DECISIONS = []
def licence_gate():
    global VIDEO_SPEED, IMAGE_MODEL, CONTINUITY_CHECK
    strict = bool(globals().get("LICENSE_STRICT", False))
    uses = [("Kijai/WanVideo_comfy", "turbo LoRA", globals().get("VIDEO_SPEED") == "turbo"),
            ("stabilityai/stable-diffusion-xl-base-1.0", "SDXL keyframes", globals().get("IMAGE_MODEL") == "sdxl"),
            ("openai/clip-vit-base-patch32", "CLIP continuity check", bool(globals().get("CONTINUITY_CHECK")))]
    for mid, what, used in uses:
        ok, why = registry.model_ok(mid, strict=strict)
        LICENSE_DECISIONS.append({"model": mid, "use": what, "would_load": used, "allowed": ok, "why": why})
        if used and not ok:
            if what == "turbo LoRA":
                VIDEO_SPEED = "fast"           # 25 steps + First Block Cache: ~4x slower per clip, licence-clean
            elif what == "SDXL keyframes":
                IMAGE_MODEL = "procedural"      # fallback pictures are drawn instead
            else:
                CONTINUITY_CHECK = False        # scene-state shots keep the OWLv2 + VLM checks (Apache-2.0)
            print(f"🔒 LICENSE_STRICT: {what} refused ({why}) → " + {"turbo LoRA": "fast video mode",
                  "SDXL keyframes": "drawn fallback pictures", "CLIP continuity check": "check off"}[what])
        log(f"licence: {what}: {'allowed' if ok else 'refused'} ({why})")
licence_gate()

def preflight():
    """What this Colab session can do, measured (nothing assumed), saved to render/preflight.json."""
    import platform
    rep = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "gpu": GPU.get("name"),
           "vram_gb": GPU.get("vram_gb"), "compute_capability": GPU.get("sm"), "native_bf16": GPU.get("bf16"),
           "precision": ("bf16" if GPU.get("bf16") else "fp16") if GPU.get("name") else "fp32 (CPU)",
           "ram_free_gb": round(ram_free_gb(), 1), "disk_free_gb": round(free_disk_gb(), 1),
           "model_cache": str(MODEL_CACHE), "video_engine": VIDEO_ENGINE, "image_model": IMAGE_MODEL,
           "video_speed": globals().get("VIDEO_SPEED"), "run_mode": globals().get("RUN_MODE"),
           "story_mode": globals().get("STORY_MODE", "narrator"), "license_strict": globals().get("LICENSE_STRICT"),
           "licences": LICENSE_DECISIONS, "language": LANGUAGE, "cuda": False, "warnings": []}
    try:
        import torch
        rep["cuda"] = torch.cuda.is_available()
        rep["torch"] = torch.__version__
    except ImportError:
        rep["warnings"].append("torch is not installed: no GPU work possible")
    if LANGUAGE != "en" and not RENDER_CACHED_AUDIO_ONLY:  # the Telugu voice model is gated: say so now, not after an hour
        try:
            from google.colab import userdata
            has = bool(userdata.get("HF_TOKEN"))
        except Exception:
            has = bool(os.environ.get("HF_TOKEN"))
        rep["hf_token_present"] = has        # presence only; the value is never printed or logged
        if not has:
            rep["warnings"].append("LANGUAGE needs the gated Indic Parler-TTS model: add the HF_TOKEN secret "
                                   "(the voice cell stops with instructions otherwise; it never switches language)")
        elif (globals().get("STRICT_OPEN_SOURCE") or (globals().get("PLAN") or {}).get("code_mix")) and \
                os.environ.get("PARLER_SKIP_ACCESS_CHECK") != "1":
            try:                             # is the gate open for this token? (a HEAD request, nothing downloaded)
                import registry
                try:
                    from google.colab import userdata
                    tok_ = userdata.get("HF_TOKEN")
                except Exception:
                    tok_ = os.environ.get("HF_TOKEN")
                acc = registry.gated_access("ai4bharat/indic-parler-tts", tok_)
                tok_ = None
                rep["parler_access"] = {k: acc[k] for k in ("status", "why")}
                if acc["status"] != "PASS":
                    rep["warnings"].append(f"Indic Parler-TTS access: {acc['status']} ({acc['why']}). {acc['help']}")
            except Exception:
                log_exc("Parler access check failed")
    if rep["gpu"] and not rep["native_bf16"]:
        rep["notes"] = "T4-class GPU: Parler uses float32 for encoder stability; speech checks run on CPU int8"
    if rep["disk_free_gb"] < 20:
        rep["warnings"].append(f"only {rep['disk_free_gb']} GB disk free")
    if not rep["gpu"]:
        rep["warnings"].append("CPU-only run: matching cached narration required; scenes remain drawn from approved atlases")
    (PROJ / "render").mkdir(exist_ok=True)
    (PROJ / "render" / "preflight.json").write_text(json.dumps(rep, indent=1))
    log("preflight: " + json.dumps(rep)[:3000])
    print("Preflight: " + ", ".join(f"{k}={rep[k]}" for k in ("gpu", "vram_gb", "precision", "ram_free_gb",
                                                                "disk_free_gb", "story_mode", "license_strict"))
          + "".join(f"\n  ⚠️ {w}" for w in rep["warnings"]))
    return rep
PREFLIGHT = preflight()


In [ ]:
BUDGET.check('prefetch')
# ==== 5b. Start the big downloads in the background (they run while the voices are made) ====
# Only what is not cached yet: the text encoder (11.4 GB, only if some prompt has never been encoded) and the
# video model (only if it is not in the Drive model cache). The video step waits for them if they are not done.
import threading
def video_mode():
    if VIDEO_ENGINE != "wan22-5b":
        return "t2v"
    try:
        from diffusers import WanImageToVideoPipeline  # noqa: F401
        return "i2v"                                   # animates each shot's keyframe
    except Exception:
        log_exc("WanImageToVideoPipeline not available, using text-to-video")
        return "t2v"

VIDEO_MODE = video_mode() if VIDEO_ENGINE else None

def real_bf16():
    import torch
    return torch.cuda.get_device_capability(0)[0] >= 8           # T4 = 7.5 (no real bf16), L4/A100 = 8.x

VIDEO_MID = {"wan22-5b": "Wan-AI/Wan2.2-TI2V-5B-Diffusers", "wan21-1.3b": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"}
NEG_V = NEG + ", static, still image, frozen, slideshow"

def clip_prompt(shot, mode):
    return (f"{shot['motion_prompt']}, {STYLE}" if mode == "i2v"
            else f"{shot['image_prompt']}, {shot['motion_prompt']}, {STYLE}")

# Wan 2.1 and 2.2 share the same UMT5-XXL text encoder and tokenizer. The 2.2 repo stores it in bf16
# (11.4 GB download) while the 2.1 repo stores the same weights as fp32 (22.7 GB), so always take the small one.
TEXT_ENCODER_MIDS = ["Wan-AI/Wan2.2-TI2V-5B-Diffusers", "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"]
# encoded prompts, shared by ALL projects (in the model cache on Drive): any prompt encoded once, by any video,
# never needs the 11 GB text encoder again. Older runs kept them per project; those are still found.
EMB_CACHE = MODEL_CACHE / "prompt_cache"
OLD_EMB_CACHE = PROJ / "prompt_cache"

def emb_path(text):
    name = hashlib.sha1(text.encode()).hexdigest()[:16] + ".pt"
    return EMB_CACHE / name if (EMB_CACHE / name).exists() or not (OLD_EMB_CACHE / name).exists() else OLD_EMB_CACHE / name

def prefetch_jobs():
    """(what, repo, snapshot_download arguments) for everything the video step will need and doesn't have."""
    if not VIDEO_ENGINE or not any(s.get("motion_prompt") for s in SHOTS):
        return []
    jobs, mid = [], VIDEO_MID[VIDEO_ENGINE]
    prompts = {clip_prompt(s, VIDEO_MODE) for s in SHOTS if s.get("motion_prompt")} | {NEG_V}
    new = [t for t in prompts if not emb_path(t).exists()]
    if new:
        jobs.append((f"text encoder (11.4 GB) for {len(new)} new prompts", TEXT_ENCODER_MIDS[0],
                     {"allow_patterns": ["text_encoder/*", "tokenizer/*"]}))
    dt_name = "bfloat16" if GPU["bf16"] else "float16"
    states = globals().get("STATES") or {}
    vace = [s for s in SHOTS if s.get("motion_prompt") and VIDEO_RECIPE == "auto" and s["shot_id"] in states]
    if len(vace) < sum(1 for s in SHOTS if s.get("motion_prompt")):   # some shots use plain text-to-video
        skip = ["text_encoder/*"]
        if VIDEO_ENGINE == "wan21-1.3b" and (MODEL_CACHE / f"{VIDEO_ENGINE}-{dt_name}" / "complete").exists():
            skip += ["transformer/*", "vae/*"]    # loads from the Drive model cache: only small config files
        jobs.append((f"video model {mid}", mid, {"ignore_patterns": skip}))
    if vace:                                      # scene-state shots use the VACE layout model, not text-to-video
        skip = ["text_encoder/*"]
        if (MODEL_CACHE / f"wan21-vace-1.3b-{dt_name}" / "complete").exists():
            skip += ["transformer/*", "vae/*"]
        jobs.append(("VACE layout model Wan-AI/Wan2.1-VACE-1.3B-diffusers", "Wan-AI/Wan2.1-VACE-1.3B-diffusers",
                     {"ignore_patterns": skip}))
    return jobs

def _prefetch(jobs):
    from huggingface_hub import snapshot_download
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()                       # no download bars printed into the voice cell's output
    for what, repo, kw in jobs:
        t0 = time.time()
        try:
            snapshot_download(repo, **kw)
            log(f"prefetch: {what} ready in {time.time() - t0:.0f}s")
        except Exception:
            log_exc(f"prefetch of {what} failed (it is downloaded again when needed)")

def wait_for_prefetch():
    th = PREFETCH.get("thread")
    if th is not None and th.is_alive():
        t0 = time.time()
        print("Waiting for the background downloads to finish...")
        th.join()
        print(f"  waited {time.time() - t0:.0f}s")
    try:
        from huggingface_hub.utils import enable_progress_bars
        enable_progress_bars()
    except Exception:
        pass

PREFETCH = {}
_jobs = prefetch_jobs()
if _jobs:
    try:
        need_disk(20 if any("text encoder" in j[0] for j in _jobs) else 8, "the background downloads")
        PREFETCH["thread"] = threading.Thread(target=_prefetch, args=(_jobs,), daemon=True)
        PREFETCH["thread"].start()
        print("Downloading in the background while the voices are made:", "; ".join(j[0] for j in _jobs))
    except Exception as e:
        log_exc("background downloads not started")
        print("Background downloads not started (they happen in the video step instead):", repr(e)[:200])
else:
    print("Nothing to download in the background (no GPU, or everything is cached).")


In [ ]:
BUDGET.check('narration')
# ==== 6. Voices: each character has their own voice; each line is acted with a tone ====
# tone = speed + pitch + loudness + pause (+ an optional effect); character = voice + pitch + effect.
# Every line is checked by speech recognition and re-made if words come out wrong.
import numpy as np, soundfile as sf, urllib.request
SR = 24000
KOKORO_DIR = Path(os.environ.get("KOKORO_DIR", MODEL_CACHE / "kokoro")); KOKORO_DIR.mkdir(parents=True, exist_ok=True)
KOKORO_URL = "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/"
_tts, _asr, _asr_failed, ASR_ON_GPU = None, None, False, False

def kokoro_session(gpu):
    import onnxruntime as rt
    if gpu:
        # DEFAULT = a fixed convolution algorithm, no cuDNN search (the search fails on some T4 setups:
        # "Failed to initialize CUDNN Frontend ... HEURISTIC_QUERY_FAILED")
        providers = [("CUDAExecutionProvider", {"cudnn_conv_algo_search": "DEFAULT"}), "CPUExecutionProvider"]
    else:
        providers = ["CPUExecutionProvider"]
    sess = rt.InferenceSession(str(KOKORO_DIR / "kokoro-v1.0.onnx"), providers=providers)
    if gpu:
        sess.disable_fallback()                    # fail loudly instead of switching to CPU behind our back
    log(f"Kokoro session providers: {sess.get_providers()}")
    return sess

def use_cpu_voice(reason):
    """Switch the voice model to the CPU (small model: a few minutes for the whole narration)."""
    global _tts
    from kokoro_onnx import Kokoro
    LOG.write(f"Voice model: GPU failed, switching to CPU. Reason:\n{reason}", "ERROR")
    _tts = Kokoro.from_session(kokoro_session(False), str(KOKORO_DIR / "voices-v1.0.bin"))
    _tts.on_gpu = False
    print("Voice model running on: CPU (the GPU attempt failed; full error in the log file)")
    DEVICE_REPORT["voice (Kokoro)"] = "CPU (GPU attempt failed)"

def get_tts():
    global _tts
    if _tts is None and LANGUAGE != "en":
        if RENDER_CACHED_AUDIO_ONLY:
            raise RuntimeError('A narration WAV or matching sidecar is missing/stale. Allocate a GPU and set '
                               'RENDER_CACHED_AUDIO_ONLY=False to synthesize it.')
        BUDGET.check('Telugu speech model')
        if not GPU['name']:
            raise RuntimeError('New Telugu speech needs an allocated GPU; cached rendering remains available.')
        import torch
        if torch.cuda.mem_get_info()[0] < 9 * 2**30:
            raise RuntimeError('Need at least 9 GiB free GPU memory before loading Parler. Restart the runtime.')
        _tts = ParlerVoice()
    if _tts is None:
        for f in ["kokoro-v1.0.onnx", "voices-v1.0.bin"]:          # ~350 MB, Apache-2.0 model
            if not (KOKORO_DIR / f).exists():                       # once; then kept in the model cache
                print("Downloading", f, "...")
                urllib.request.urlretrieve(KOKORO_URL + f, KOKORO_DIR / (f + ".part"))
                (KOKORO_DIR / (f + ".part")).rename(KOKORO_DIR / f)
        from kokoro_onnx import Kokoro
        import onnxruntime as rt
        log(f"onnxruntime {rt.__version__}, available providers: {rt.get_available_providers()}")
        if "CUDAExecutionProvider" in rt.get_available_providers():
            try:
                rt.preload_dlls()                      # use the CUDA libraries that come with PyTorch
            except Exception:
                log_exc("onnxruntime preload_dlls failed")
            try:
                _tts = Kokoro.from_session(kokoro_session(True), str(KOKORO_DIR / "voices-v1.0.bin"))
                _tts.on_gpu = True
                _tts.create("Testing the voice on the graphics card, with a sentence of ordinary length "
                            "so that every layer of the model runs once.", voice=VOICE, speed=1.0, lang="en-us")
                print("Voice model running on: GPU")
                DEVICE_REPORT["voice (Kokoro)"] = "GPU"
            except Exception:
                use_cpu_voice(traceback.format_exc())
        else:
            use_cpu_voice("onnxruntime has no CUDA support in this session")
        assert VOICE in _tts.get_voices(), f"unknown voice {VOICE}; choose from {_tts.get_voices()}"
    return _tts

def speak(text, speed, voice=None, attempt=0, instruct=""):
    """Kokoro TTS (English); Qwen3-TTS for "qwen:<speaker>" voices (expressive narration, `instruct` = the voice
    direction); Indic Parler-TTS for other languages (attempt = which take: a new seed per retry).
    Kokoro: if the GPU fails on some sentence later on, finish on the CPU instead of stopping."""
    if voice and voice.startswith("gemini:"):          # Telugu via Gemini free tier (made by the voice pass)
        return GEMINI_SPEAK(text, voice.split(":", 1)[1], attempt)
    if (voice or "").startswith("qwen:"):
        return get_qwen_tts().create(text, voice, speed, attempt, instruct)
    if LANGUAGE != "en":
        return get_tts().create(text, voice=voice or VOICE, speed=speed, attempt=attempt)
    try:
        return get_tts().create(text, voice=voice or VOICE, speed=speed, lang="en-us")
    except Exception:
        if not getattr(_tts, "on_gpu", False):
            raise
        use_cpu_voice(traceback.format_exc())
        return _tts.create(text, voice=voice or VOICE, speed=speed, lang="en-us")

def preload_cuda_libs():
    """faster-whisper's GPU backend (CTranslate2 4.x) needs libcublas.so.12 + cuDNN 9. On Colab they live inside the
    pip packages nvidia-cublas-cu12 / nvidia-cudnn-cu12 (installed with torch), outside the loader's search path, so
    the GPU attempt fails ("Library libcublas.so.12 is not found or cannot be loaded") and the check silently ran on
    the CPU. Loading them by full path first (RTLD_GLOBAL) makes them findable. Harmless if absent."""
    import ctypes, glob, importlib.util
    done = []
    for pkg in ("nvidia.cublas", "nvidia.cudnn"):
        try:
            spec = importlib.util.find_spec(pkg)
        except Exception:
            spec = None
        for root in (list(spec.submodule_search_locations or []) if spec else []):
            for lib in sorted(glob.glob(os.path.join(root, "lib", "lib*.so*"))):
                try:
                    ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL); done.append(os.path.basename(lib))
                except OSError:
                    pass
    log(f"CUDA libraries preloaded for faster-whisper: {done or 'none found'}")
    return done

def get_asr():
    # Keep VRAM available for Parler float32 on a T4. This small model is only a diagnostic.
    global _asr, _asr_failed, ASR_ON_GPU
    if _asr_failed:
        return None
    if _asr is None:
        BUDGET.check('speech check')
        try:
            from faster_whisper import WhisperModel
            src = str(hf_files('Systran/faster-whisper-small', 'small multilingual speech check'))
            _asr = WhisperModel(src, device='cpu', compute_type='int8', cpu_threads=CPU_THREADS)
            DEVICE_REPORT['speech check (Whisper)'] = 'CPU (small multilingual, int8)'
            print('Speech check: CPU int8; listen to Telugu pronunciation before sharing.')
        except Exception:
            log_exc('speech check unavailable; report remains unverified')
            _asr_failed = True
    return _asr

def listen(path):
    """Transcribe a file -> (text, [[word, start, end], ...]) or (None, None)."""
    global _asr_failed
    asr = get_asr()
    if asr is None:
        return None, None
    try:
        # decode with ffmpeg to 16 kHz mono (avoids a faster-whisper / PyAV version clash)
        p16 = Path(str(path) + ".16k.wav")
        ff("-i", path, "-ar", 16000, "-ac", 1, p16)
        audio, _ = sf.read(str(p16), dtype="float32")
        # names the recogniser can't know (Zomato, Ananya...) are given up front, so it doesn't hear "the motto"
        # beam 5 everywhere: a greedy search was measured only ~6 % faster on CPU (encoder + word timing dominate)
        segs, _ = asr.transcribe(audio, language=LANGUAGE, word_timestamps=True, beam_size=5,
                                 initial_prompt=globals().get("ASR_PROMPT") or None)
        words = [[w.word.strip(), round(w.start, 3), round(w.end, 3)] for seg in segs for w in (seg.words or [])]
        return " ".join(w[0] for w in words), words
    except Exception as e:
        log_exc("speech check failed")
        _asr_failed = True
        print("Speech check failed, continuing without it:", repr(e)[:200])
        return None, None

# ---- other languages (Telugu): Indic Parler-TTS (Apache-2.0) in its own Python environment ----
if LANGUAGE != "en":
    if ASR_MODEL.endswith(".en"):                     # English-only recogniser can't check Telugu
        ASR_MODEL = ASR_MODEL_MULTI
    WER_RETRY = SPEECH_RETRY_OTHER                    # letter error rate; Whisper is much weaker on Telugu than English
    log(f"language {LANGUAGE}: voices = Indic Parler-TTS, speech check = {ASR_MODEL} (letter error, retry above "
        f"{WER_RETRY:.0%})")

def hf_token():
    """Hugging Face token from a Colab secret (🔑 in the left bar) or the HF_TOKEN environment variable. Never in code."""
    tok, why = os.environ.get("HF_TOKEN"), "no HF_TOKEN environment variable"
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception as e:
            log(f"no HF_TOKEN Colab secret: {e!r}", "WARN")
            why = {"SecretNotFoundError": "there is no secret named exactly HF_TOKEN",
                   "NotebookAccessError": "the HF_TOKEN secret exists but 'Notebook access' is OFF for this notebook "
                                          "(or the access pop-up was dismissed)"}.get(type(e).__name__, repr(e)[:200])
    if not tok:
        raise RuntimeError(
            f"Colab gave no Hugging Face token ({why}).\n"
            "The Telugu voice model (ai4bharat/indic-parler-tts) needs a free Hugging Face token:\n"
            "  1. Log in at huggingface.co, open huggingface.co/ai4bharat/indic-parler-tts and accept the terms "
            "(approved instantly).\n  2. huggingface.co/settings/tokens → New token → type 'Read' → copy it.\n"
            "  3. In Colab: 🔑 Secrets (left bar) → Add new secret → name HF_TOKEN, paste the token, turn ON "
            "'Notebook access'.\n  4. Run this cell again (the voice environment is already set up: no reinstall).")
    return tok.strip()                               # pasted with a space or line break = an invalid token (401)

def check_hf_access(tok, repo):
    """Before starting the voice model: is the token valid, and does its account have access to the gated model?
    Hugging Face answers 401 for an invalid token and 403 for terms not accepted; each gets its own fix here.
    The token itself is never printed or logged."""
    if os.environ.get("PARLER_SKIP_ACCESS_CHECK") == "1":       # tests with a stand-in worker
        return
    from huggingface_hub import HfApi
    api = HfApi()
    try:
        name = api.whoami(token=tok).get("name", "?")
    except Exception as e:
        log(f"Hugging Face token check failed: {type(e).__name__}: {str(e)[:300]}", "ERROR")
        raise RuntimeError(
            f"The HF_TOKEN secret is not a valid Hugging Face token (it starts with {tok[:3]!r}, {len(tok)} characters; "
            "a real one starts with 'hf_' and is about 37 characters).\n"
            "  → huggingface.co/settings/tokens → Create new token → type 'Read' → copy the token VALUE (hf_…), not its name.\n"
            "  → Colab 🔑 Secrets → HF_TOKEN → replace the value (no spaces) → Notebook access ON → re-run this cell.")
    try:
        api.auth_check(repo, token=tok)
    except Exception as e:
        log(f"Hugging Face access check for {repo} failed: {type(e).__name__}: {str(e)[:300]}", "ERROR")
        raise RuntimeError(
            f"The token works (Hugging Face account '{name}'), but that account can't download {repo}.\n"
            f"  → While logged in as '{name}', open huggingface.co/{repo} and accept the terms (approved instantly).\n"
            "  → If the token is 'Fine-grained', tick 'Read access to contents of all public gated repos you can access' "
            "(or make a 'Read' token instead).\n  → Re-run this cell.")
    log(f"Hugging Face: account '{name}' has access to {repo}")

def install_tool_env(env, requirements, module, expect_tf, label):
    """One-time per session (~2–4 min): a virtual environment that sees Colab's torch/numpy but has its own
    transformers (the version its tool pins); exact versions from the worker's REQUIREMENTS, installed with
    --no-deps (nothing else is resolved, downloaded or upgraded). Used by the Telugu and the expressive voices."""
    t0, py = time.time(), Path(env) / "bin" / "python"
    print(f"Setting up the {label} environment (one time per session, ~3 min)...")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "--system-site-packages", "--python", sys.executable,
                    str(env)], check=True, capture_output=True)
    req = Path(env) / "requirements.txt"; req.write_text(requirements)
    r = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", str(py), "--no-deps",
                        "-r", str(req)], capture_output=True, text=True)
    log(f"{label} env install ({time.time() - t0:.0f}s, exit {r.returncode}):\n{r.stdout[-3000:]}\n{r.stderr[-3000:]}")
    if r.returncode:
        raise RuntimeError(f"{label} environment could not be installed (details in the log): "
                           + r.stderr.strip().splitlines()[-1][:300])
    chk = subprocess.run([str(py), "-I", "-c", f"import transformers, torch, {module}; "
                          "print(transformers.__version__, torch.__version__)"], capture_output=True, text=True)
    log(f"{label} env check: {chk.stdout.strip()} {chk.stderr[-1500:]}")
    if chk.returncode or not chk.stdout.startswith(expect_tf):
        raise RuntimeError(f"{label} environment is broken (details in the log): " + chk.stderr[-300:])
    (Path(env) / "ready").write_text(chk.stdout)
    print(f"  ready in {time.time() - t0:.0f}s ({chk.stdout.strip()})")

class ParlerVoice:
    """Indic Parler-TTS behind the same interface as Kokoro (create, get_voices). parler-tts pins transformers 4.46,
    the notebook needs 5.x, so it runs in its own environment (Colab's torch, its own transformers) as ONE worker
    process: the model loads once, each sentence is a JSON request. Each character = a trained speaker name + a
    description; each retry = a new seed (Parler samples, so a retry is a genuinely new take)."""
    ENV = Path(os.environ.get("PARLER_ENV", str(WORK.parent / "tts_parler")))

    def __init__(self):
        import parler_worker as pw
        self.pw, self.proc = pw, None
        self.py = self.ENV / "bin" / "python"
        t0 = time.time()
        if not (self.ENV / "ready").exists():
            self.install()
        tok = hf_token()
        check_hf_access(tok, os.environ.get("PARLER_MODEL", pw.MODEL))
        env = dict(os.environ, HF_TOKEN=tok, HF_HOME=str(WORK.parent / "hf_parler"), PYTHONWARNINGS="ignore",
                   TOKENIZERS_PARALLELISM="false")
        dev = "cuda" if GPU["name"] else "cpu"
        dt = "bfloat16" if GPU.get("bf16") else "float32"   # T4: fp16 overflows in the T5 description encoder
        worker = os.environ.get("PARLER_WORKER") or pw.__file__
        self.err_path = Path(LOG.path).with_name("parler_worker.log")         # next to the run log (moved to Drive with it)
        self.err_start = self.err_path.stat().st_size if self.err_path.exists() else 0
        self.err = open(self.err_path, "a")
        self.proc = subprocess.Popen([str(self.py), "-I", worker, os.environ.get("PARLER_MODEL", pw.MODEL), dev, dt],
                                     stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=self.err, text=True,
                                     env=env, bufsize=1)
        try:
            ready = self._read(timeout=BUDGET.timeout(1200))  # first session downloads model weights
        except BaseException:
            self.close()
            raise
        if not ready.get("ready"):
            self.close()
            raise RuntimeError(f"Telugu voice model did not start: {ready} (details: parler_worker.log next to the run log)")
        self.sr = ready["sr"]
        log(f"Parler worker ready in {time.time() - t0:.0f}s: {ready}")
        DEVICE_REPORT["voice (Indic Parler, own process)"] = f"{'GPU' if ready['device'] == 'cuda' else 'CPU'} ({ready['dtype']})"
        print(f"Voice model: Indic Parler-TTS ({ready['device']}, {ready['dtype']}, transformers "
              f"{ready.get('transformers')} in its own environment)")

    def install(self):
        install_tool_env(self.ENV, self.pw.REQUIREMENTS, "parler_tts", "4.46", "Telugu voice")

    def _read(self, timeout=600):
        import select
        r, _, _ = select.select([self.proc.stdout], [], [], BUDGET.timeout(timeout))
        if not r:
            self.close()
            raise RuntimeError(f"Telugu voice model gave no answer in {timeout}s (parler_worker.log next to the run log)")
        while True:
            line = self.proc.stdout.readline()
            if not line:
                raise RuntimeError(self.why_stopped())
            try:
                return json.loads(line)
            except ValueError:
                log(f"parler worker (not protocol): {line.strip()[:300]}", "WARN")

    def why_stopped(self):
        """The worker exited: show its real error on screen (last lines of its log) and what to do about it."""
        try:
            self.proc.wait(timeout=10)
        except Exception:
            pass
        self.err.flush()
        try:
            with open(self.err_path, "rb") as f:
                f.seek(self.err_start); tail = f.read().decode("utf-8", "replace")
        except Exception as e:
            tail = f"(could not read {self.err_path}: {e!r})"
        lines = [l for l in tail.splitlines() if l.strip() and "Warning" not in l and not l.startswith("  warnings.warn")]
        last = "\n".join(lines[-12:])
        low = tail.lower()
        if "gated" in low or " 401" in low or " 403" in low or "unauthorized" in low or "access to model" in low:
            fix = ("Hugging Face refused the download. Open huggingface.co/ai4bharat/indic-parler-tts while logged in and "
                   "ACCEPT the terms; use a token of type 'Read' (a 'Fine-grained' token needs 'Read access to contents "
                   "of all public gated repos you can access' ticked). Then re-run this cell.")
        elif "out of memory" in low:
            fix = "The GPU or RAM was full when the voice model loaded: Runtime → Restart session, then Run all."
        elif "modulenotfounderror" in low or "importerror" in low or "undefined symbol" in low:
            (self.ENV / "ready").unlink(missing_ok=True)
            fix = "The voice environment is broken: it is reinstalled when you re-run this cell. Send this error if it repeats."
        else:
            fix = "Send this error message (and parler_worker.log next to the run log) to Claude."
        log(f"parler worker stopped (exit {self.proc.returncode}):\n{tail[-6000:]}", "ERROR")
        return (f"Telugu voice model stopped (exit code {self.proc.returncode}).\n{fix}\n\nLast lines of its log "
                f"({self.err_path}):\n{last}")

    def get_voices(self):
        return self.pw.SPEAKERS.get(LANGUAGE, [])

    def _one(self, text, desc, seed):
        out = WORK / f"parler_{hashlib.sha1(f'{text}|{desc}|{seed}'.encode()).hexdigest()[:12]}.wav"
        self.proc.stdin.write(json.dumps({"text": text, "description": desc, "seed": seed, "out": str(out)}) + "\n")
        self.proc.stdin.flush()
        r = self._read()
        if not r.get("ok"):
            raise RuntimeError(f"Telugu voice failed: {r.get('error')}")
        return out, r

    def create(self, text, voice, speed=1.0, attempt=0):
        """Sentence by sentence (Parler is most reliable on short inputs), each checked for a plausible length (catches
        cut-off or rambling takes: those get a new seed, up to 3 tries), joined with short pauses; then the line's
        speed (tempo, pitch kept) and Kokoro's sample rate, so everything downstream is unchanged."""
        name, _, style = voice.partition("|")
        desc = (f"{name} speaks {style or 'in a clear, natural, moderately paced voice'}. The recording is of very "
                "high quality, with the speaker's voice sounding clear and very close up, with no background noise.")
        parts = [x.strip() for x in re.split(r"(?<=[.?!।])\s+", text) if x.strip()]
        pieces = []
        for k, part in enumerate(parts):
            letters = sum(1 for ch in part if not ch.isspace())
            for t in range(3):
                seed = int(hashlib.sha1(f"{part}|{name}".encode()).hexdigest()[:6], 16) + 1000 * attempt + 97 * t
                try:
                    out, r = self._one(part, desc, seed)
                except RuntimeError as e:                 # a sampled take can fail inside the model (seen on Colab:
                    if t == 2:                            # "IndexError: tuple index out of range"): new seed
                        raise
                    log(f"Parler take for {part[:40]!r} failed with seed {seed}: {e}; new seed", "WARN")
                    continue
                per = r["seconds"] / max(letters, 1)
                if PARLER_SEC_PER_CHAR[0] <= per <= PARLER_SEC_PER_CHAR[1]:
                    break
                log(f"Parler take for {part[:40]!r}: {r['seconds']}s for {letters} letters ({per:.3f} s/letter) — "
                    "implausible, new seed", "WARN")
            pieces.append(out)
        lst = WORK / f"parler_concat_{os.getpid()}.txt"
        gap = WORK / "parler_gap.wav"
        if not gap.exists():
            ff("-f", "lavfi", "-i", f"anullsrc=r={self.sr}:cl=mono", "-t", 0.18, gap)
        lst.write_text("".join(f"file '{p}'\n" + (f"file '{gap}'\n" if i < len(pieces) - 1 else "")
                               for i, p in enumerate(pieces)))
        joined = WORK / f"parler_line_{os.getpid()}.wav"
        ff("-f", "concat", "-safe", 0, "-i", lst, "-af", f"atempo={min(max(speed, 0.5), 2.0):.4f},aresample={SR}",
           "-ac", 1, "-ar", SR, joined)
        wav, sr = sf.read(str(joined), dtype="float32")
        return wav, sr

    def close(self):
        """Stop the worker: it is a separate process, so its GPU memory is only freed when it exits."""
        if self.proc and self.proc.poll() is None:
            try:
                self.proc.stdin.close(); self.proc.wait(timeout=10)
            except Exception:
                self.proc.kill()
                self.proc.wait(timeout=10)
        if self.proc:
            for stream in (self.proc.stdin, self.proc.stdout):
                if stream is not None and not stream.closed:
                    try:
                        stream.close()
                    except OSError:
                        pass
        if getattr(self, 'err', None) is not None:
            self.err.close()
        self.proc = None

class QwenVoice(ParlerVoice):
    """Expressive English narration: Qwen3-TTS 1.7B CustomVoice (Apache-2.0) in its own environment (qwen-tts pins
    transformers 4.57). ONE fixed premium speaker = the narrator's identity; the voice direction becomes the
    `instruct` (style only). Precision QWEN_TTS_DTYPE (T4: float32 unless VOICE_BENCH shows float16 is stable).
    A whole line (1–3 sentences) is one request, so its prosody stays natural; retries = new seeds."""
    ENV = Path(os.environ.get("QWEN_TTS_ENV", str(WORK.parent / "tts_qwen")))

    def __init__(self):
        import qwen_tts_worker as qw
        self.pw, self.proc = qw, None
        self.py = self.ENV / "bin" / "python"
        t0 = time.time()
        if not (self.ENV / "ready").exists():
            install_tool_env(self.ENV, qw.REQUIREMENTS, "qwen_tts", "4.57", "expressive voice")
        dev = "cuda" if GPU["name"] else "cpu"
        dt = QWEN_TTS_DTYPE if dev == "cuda" else "bfloat16"
        env = dict(os.environ, HF_HOME=str(WORK.parent / "hf_qwen_tts"), PYTHONWARNINGS="ignore", TOKENIZERS_PARALLELISM="false")
        worker = os.environ.get("QWEN_TTS_WORKER") or qw.__file__
        self.err_path = Path(LOG.path).with_name("qwen_tts_worker.log")
        self.err_start = self.err_path.stat().st_size if self.err_path.exists() else 0
        self.err = open(self.err_path, "a")
        self.model = os.environ.get("QWEN_TTS_MODEL", qw.MODEL)
        self.proc = subprocess.Popen([str(self.py), "-I", worker, self.model, dev, dt], stdin=subprocess.PIPE,
                                     stdout=subprocess.PIPE, stderr=self.err, text=True, env=env, bufsize=1)
        ready = self._read(timeout=2400)
        if not ready.get("ready"):
            raise RuntimeError(f"expressive voice model did not start: {ready}")
        self.speakers, self.instruct_ok, self.info = ready.get("speakers", []), ready.get("instruct", False), ready
        log(f"Qwen3-TTS worker ready in {time.time() - t0:.0f}s: {ready}")
        DEVICE_REPORT["voice (Qwen3-TTS, own process)"] = f"{'GPU' if ready['device'] == 'cuda' else 'CPU'} ({ready['dtype']})"
        print(f"Voice model: Qwen3-TTS ({self.model.split('/')[-1]}, {ready['device']}, {ready['dtype']}, "
              f"{'instruction control' if self.instruct_ok else 'NO instruction control: directions not applied'})")

    def get_voices(self):
        return [f"qwen:{s}" for s in self.speakers]

    def create(self, text, voice, speed=1.0, attempt=0, instruct=""):
        spk = voice.split(":", 1)[1]
        if spk not in self.speakers:
            raise ValueError(f"Qwen3-TTS has no speaker '{spk}' (available: {self.speakers})")
        seed = int(hashlib.sha1(f"{text}|{spk}".encode()).hexdigest()[:6], 16) + 1000 * attempt
        out = WORK / f"qwen_{hashlib.sha1(f'{text}|{spk}|{instruct}|{seed}'.encode()).hexdigest()[:12]}.wav"
        self.proc.stdin.write(json.dumps({"text": text, "speaker": spk, "language": "English",
                                          "instruct": instruct if self.instruct_ok else "", "seed": seed,
                                          "out": str(out)}) + "\n")
        self.proc.stdin.flush()
        r = self._read(timeout=900)
        if not r.get("ok"):
            raise RuntimeError(f"expressive voice failed: {r.get('error')}")
        LAST_TTS_INFO.update(r)
        wav, sr = sf.read(str(out), dtype="float32")
        if sr != SR:                                   # Qwen3-TTS is 24 kHz like Kokoro; resample only if not
            res = WORK / f"{out.stem}_{SR}.wav"
            ff("-i", out, "-ar", SR, "-ac", 1, res); wav, sr = sf.read(str(res), dtype="float32")
        return wav, sr

_qwen_tts = None
LAST_TTS_INFO = {}
def get_qwen_tts():
    global _qwen_tts
    if _qwen_tts is None:
        _qwen_tts = QwenVoice()
    return _qwen_tts

def close_tts():
    global _qwen_tts, _tts
    for t_ in (_tts, _qwen_tts):
        if t_ is not None and hasattr(t_, "close"):
            t_.close()
    _qwen_tts = _tts = None

BUDGET.on_exhausted = close_tts

VOICE_FX = {"none":    "",
            "sparkle": "chorus=0.6:0.9:50|60:0.4|0.32:0.25|0.4:2|1.3,aecho=0.8:0.7:40:0.25",   # magical shimmer
            "old":     "lowpass=f=7000,bass=g=2:f=180"}                                         # warmer, older
TONE_FX = {"shaky":   "tremolo=f=5.5:d=0.15",           # trembling (sad / scared)
           "breathy": "highpass=f=200,treble=g=2"}      # airy, whisper-like

# ---- expressive narration: a delivery direction per line, from the story (voice_direction.py) ----
# Words never change. Kokoro can only follow the pace (+ real pauses); Qwen3-TTS follows the whole direction as an
# instruction. What each backend did with each field is reported (prosody_report.json), never assumed.
DIRECTIONS, VQC = {}, {}
if EXPRESSIVE:
    _order = [l["id"] for s_ in SHOTS for l in s_["lines"]]
    _dialogue = PLAN.get("story_mode") == "two_friends_cinematic"
    _profile = "two_friends" if _dialogue else VOICE_PROFILE
    def _direct():
        # two friends: intention from the conversation + response gaps; narrators: beat roles + narration pauses
        d = voice_direction.dialogue_directions(SHOTS, CHARS) if _dialogue else \
            voice_direction.rule_directions(SHOTS, VOICE_PROFILE)
        if VOICE_DIRECTOR == "llm":                     # a local model reads the whole story (GPU, freed after)
            import planner
            llm = planner.LocalLLM(DIRECTOR_MODEL, device="cuda" if GPU["name"] else "cpu",
                                   cache_dir=str(WORK.parent / "hf_planner"), log=log)
            try:
                d = voice_direction.llm_directions(llm, SHOTS, d, planner.ask_json, log)
            finally:
                del llm; free_gpu()
        d, adj = voice_direction.smooth(_order, d, _profile,
                                        EXPRESSIVE_MAX_INTENSITY if VOICE_BACKEND != "kokoro" else 1.0)
        return {"directions": {k: v.model_dump() for k, v in d.items()}, "adjustments": adj}
    _vd = stages.stage(PROJ / "stages", "voice_direction", {
        "lines": [(l["id"], l.get("tts") or l["text"], l.get("tone"), s_.get("beat"), s_.get("chapter"),
                   l.get("voice_direction"), l.get("speaker"), l.get("kind"), l.get("intent"), s_.get("story_beat"))
                  for s_ in SHOTS for l in s_["lines"]],
        "profile": _profile, "director": VOICE_DIRECTOR, "backend": VOICE_BACKEND,
        "cap": EXPRESSIVE_MAX_INTENSITY, "v": 3}, _direct, log)
    DIRECTIONS = {k: voice_direction.VoiceDirection(**v) for k, v in _vd["directions"].items()}
    (PROJ / "render").mkdir(exist_ok=True)
    (PROJ / "render" / "voice_direction.json").write_text(json.dumps(_vd, indent=1))
    print(f"Voice direction ({_profile}, {VOICE_DIRECTOR}): " + " → ".join(
        f"{DIRECTIONS[i].intent or DIRECTIONS[i].emotion}{'*' if DIRECTIONS[i].intensity >= 0.5 else ''}"
        for i in _order if i in DIRECTIONS))
    for a_ in _vd["adjustments"]:
        log("voice direction smoothed: " + a_)

# ---- VOICE_BENCH: the same script, neutral vs expressive, per backend (measured, blind A/B), then stop ----
BENCH_SCRIPT = [   # hook → setup → build-up → suspense question → reveal → calm conclusion (a demo, not published)
    ("hook", "What happens when a giant star finally runs out of fuel?"),
    ("setup", "For millions of years, the star burns hydrogen, pushing outward against its own gravity."),
    ("step", "But when the fuel runs out, nothing holds gravity back anymore."),
    ("turn", "So what happens to all that matter, falling inward at once?"),
    ("resolution", "It collapses into a point so dense that not even light can escape."),
    ("payoff", "Every black hole began as a star, just like the ones you can see tonight.")]
VOICE_BENCH_BACKENDS = ["kokoro", "qwen3-tts"]
# two-friend dialogue benchmark (story plans): one short scene covering the intentions the brief lists. Telugu uses
# lines with the same INTENTIONS, written as natural spoken Telugu (not a word-for-word translation).
DIALOGUE_BENCH = {
    "en": [("meera", "explaining_observation", "fact", "When you look at something close, each of your eyes sees it from a slightly different angle."),
           ("arjun", "genuine_curiosity", "question", "Wait, so my left eye and my right eye see different pictures right now?"),
           ("meera", "reflective_explanation", "fact", "Pretty much... and your brain quietly turns them into one."),
           ("arjun", "playful_disbelief", "observation", "No way. That's why my thumb jumped when I closed one eye!"),
           ("meera", "gentle_correction", "correction", "Almost. Your thumb doesn't move. You just switched which eye was looking."),
           ("arjun", "playful_remark", "reaction", "So I've been watching two movies at once my whole life."),
           ("meera", "satisfying_revelation", "fact", "And that's the whole trick of 3D glasses: each eye gets only its own picture."),
           ("meera", "reflective_explanation", "fact", "The projector puts both pictures on one screen, each lens lets only one through, "
                                                        "and your brain builds the depth from the difference.")],
    "te": [("meera", "explaining_observation", "fact", "ఏదైనా దగ్గరగా ఉన్నదాన్ని చూసినప్పుడు, నీ రెండు కళ్లూ దాన్ని కొంచెం వేర్వేరు కోణాల నుంచి చూస్తాయి."),
           ("arjun", "genuine_curiosity", "question", "ఆగు, అంటే ఇప్పుడు నా ఎడమ కన్నూ కుడి కన్నూ వేర్వేరు చిత్రాలు చూస్తున్నాయా?"),
           ("meera", "reflective_explanation", "fact", "అవును... కానీ నీ మెదడు వాటిని నిశ్శబ్దంగా ఒకటిగా కలిపేస్తుంది."),
           ("arjun", "playful_disbelief", "observation", "అయ్యో! అందుకేనా ఒక కన్ను మూస్తే నా బొటనవేలు దూకింది!"),
           ("meera", "gentle_correction", "correction", "దాదాపు. బొటనవేలు కదలదు, చూసే కన్ను మారుతుంది అంతే."),
           ("arjun", "playful_remark", "reaction", "అంటే నేను జీవితాంతం ఒకేసారి రెండు సినిమాలు చూస్తున్నానన్నమాట."),
           ("meera", "satisfying_revelation", "fact", "త్రీడీ గ్లాసెస్ రహస్యం అదే: ప్రతి కంటికి దాని చిత్రం మాత్రమే చేరుతుంది."),
           ("meera", "reflective_explanation", "fact", "ప్రొజెక్టర్ రెండు చిత్రాలనూ ఒకే స్క్రీన్ మీద వేస్తుంది, ప్రతి లెన్స్ ఒకదాన్నే పంపుతుంది, "
                                                        "అప్పుడు నీ మెదడు ఆ తేడా నుంచి లోతును తానే తయారు చేస్తుంది.")]}
QWEN_BENCH_SPEAKERS = {"arjun": "aiden", "meera": "serena"}   # Meera: no native-English female preset exists in
                                                             # Qwen3-TTS CustomVoice; "serena" is a CANDIDATE to judge


def run_dialogue_bench():
    """VOICE_BENCH for two-friend plans: old (narrator-style) vs new (conversation) direction on Kokoro; Qwen3-TTS
    with conversational instructions; Parler for Telugu. One TTS family at a time. Measured: load/generation time,
    real-time factor, peak VRAM, peak RAM, audio length, speech check, identity per character, QC. Every backend
    gets a status: PASS / FAIL / NOT_TESTED / RESOURCE_UNAVAILABLE / LICENSE_BLOCKED / ACCESS_REQUIRED."""
    import random, resource
    out_dir = PROJ / "render" / "voice_bench"; out_dir.mkdir(parents=True, exist_ok=True)
    lang = LANGUAGE if LANGUAGE in DIALOGUE_BENCH else "en"
    script = DIALOGUE_BENCH[lang]
    shot = {"shot_id": "db", "story_beat": "first_answer",
            "lines": [{"id": f"db_{i + 1}", "speaker": sp, "kind": k, "intent": it, "text": t, "tone": "friendly"}
                      for i, (sp, it, k, t) in enumerate(script)]}
    order = [l["id"] for l in shot["lines"]]
    new, _ = voice_direction.smooth(order, voice_direction.dialogue_directions([shot], CHARS), "two_friends")
    old = voice_direction.rule_directions([dict(shot, beat="step", chapter="c")], VOICE_PROFILE)
    old, _ = voice_direction.smooth(order, old, VOICE_PROFILE)
    configs = ([("kokoro", "old_direction", old), ("kokoro", "new_direction", new), ("qwen3-tts", "new_direction", new)]
               if lang == "en" else [("parler", "new_direction", new)])
    try:
        spk = voice_qc.SpeakerCheck(WORK.parent / "ecapa")
    except Exception:
        log_exc("speaker check unavailable in the voice benchmark"); spk = None
    report, samples = {}, []
    for be, variant, dirs in configs:
        key = f"{be}_{variant}"
        mid = {"kokoro": "hexgrad/Kokoro-82M", "qwen3-tts": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
               "parler": "ai4bharat/indic-parler-tts"}[be]
        ok_lic, why = registry.model_ok(mid, strict=LICENSE_STRICT)
        rec = {"backend": be, "model": mid, "revision": registry.REVISIONS.get(mid), "licence": why,
               "device": "cuda" if GPU["name"] else "cpu"}
        if not ok_lic:
            report[key] = dict(rec, status="LICENSE_BLOCKED"); continue
        if be == "qwen3-tts" and not GPU["name"]:
            report[key] = dict(rec, status="RESOURCE_UNAVAILABLE",
                               why="no GPU: measured ~4–12 min per line on CPU; run the benchmark on a T4"); continue
        if be == "parler":
            try:
                hf_token()
            except Exception as e:
                report[key] = dict(rec, status="ACCESS_REQUIRED", why=str(e)[:300]); continue
        t0, chunks, rows, embs = time.time(), [], {}, {}
        try:
            for l in shot["lines"]:
                d, c = dirs[l["id"]], CHARS[l["speaker"]]
                if be == "qwen3-tts":
                    voice = f"qwen:{c.get('qwen_speaker') or QWEN_BENCH_SPEAKERS[l['speaker']]}"
                    instr, speed = expressive_tts.qwen_params(d, voice)["instruct"], 1.0
                elif be == "parler":
                    voice = "|".join([c["voice"], ", ".join(x for x in (c.get("voice_style"), expressive_tts.parler_style(d)) if x)])
                    instr, speed = "", 1.0
                else:
                    voice, instr = c["voice"], ""
                    speed = expressive_tts.kokoro_params(d, c.get("speed", 1.0))["speed"]
                t1 = time.time()
                wav, sr = speak(l["text"], speed, voice, attempt=0, instruct=instr)
                gen = time.time() - t1
                tmp = out_dir / f"{key}_{l['id']}.wav"; sf.write(tmp, np.asarray(wav, dtype=np.float32), SR)
                heard, _ = listen(tmp)
                err = None if heard is None else speech_error([l["text"]], heard)
                vq = check_line(tmp, l["text"], heard, err)
                vq.update(speaker=l["speaker"], intent=d.intent, gen_s=round(gen, 2),
                          rtf=round(gen / max(len(wav) / SR, 0.1), 2), instruct=instr,
                          peak_vram_gb=LAST_TTS_INFO.get("peak_vram_gb") if be == "qwen3-tts" else None)
                rows[l["id"]] = vq
                if spk:
                    embs.setdefault(l["speaker"], {})[l["id"]] = spk.embed(wav, SR)
                chunks.append((wav, d.pause_before_ms / 1000, d.pause_after_ms / 1000))
            for sp_, e_ in embs.items():
                for lid, sim in voice_qc.consistency(e_).items():
                    rows[lid]["identity"] = sim
            f = assemble(chunks, out_dir / f"dialogue_{key}.wav")
            samples.append((key, f))
            fails = {lid: [c_ for c_, x in r["checks"].items() if x["status"] == "FAIL" and "(diagnostic)" not in c_]
                     for lid, r in rows.items()}
            report[key] = dict(rec, status="FAIL" if any(fails.values()) else "PASS",
                               total_s=round(time.time() - t0, 1), audio_s=round(sum(len(c[0]) for c in chunks) / SR, 1),
                               peak_ram_gb=round(max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
                                                     resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss) / 2**20, 2),
                               lines=rows, fails=fails, human_naturalness=None)
            print(f"  {key}: {report[key]['status']} {report[key]['total_s']:.0f}s for {report[key]['audio_s']}s of speech")
        except Exception as e:
            log_exc(f"voice bench {key} failed"); report[key] = dict(rec, status="FAIL", error=repr(e)[:300])
            print(f"  {key}: FAILED {repr(e)[:200]}")
        if be != "kokoro":
            close_tts()                                    # one TTS model family at a time
    for k_, r_ in report.items():
        if r_["status"] not in ("PASS", "FAIL"):
            print(f"  {k_}: {r_['status']} — {r_.get('why', r_.get('licence', ''))[:160]}")
    (out_dir / "dialogue_bench.json").write_text(json.dumps(report, indent=1, default=str))
    rnd, key_ = random.Random(), {}
    rnd.shuffle(samples)
    sheet = ["# Blind listening sheet (dialogue benchmark)", "", "Rate each sample 1–5, THEN open blind_test_key.json.", "",
             "| Sample | Human-like | Pronunciation | Conversational flow | Emotion fits | Voices consistent | Overall |",
             "|---|---|---|---|---|---|---|"]
    try:
        from IPython.display import Audio, display
        for lab, (k_, f_) in zip("ABCDEFGH", samples):
            key_[lab] = k_; sheet.append(f"| {lab} | | | | | | |")
            print(f"Sample {lab}:"); display(Audio(str(f_)))
    except Exception:
        for lab, (k_, f_) in zip("ABCDEFGH", samples):
            key_[lab] = k_; sheet.append(f"| {lab} | | | | | | |")
        log_exc("audio players unavailable (not in a notebook)")
    (out_dir / "blind_test_key.json").write_text(json.dumps({k: str(v) for k, v in key_.items()}, indent=1))
    (out_dir / "listening_sheet.md").write_text("\n".join(sheet) + "\n")
    raise VoiceBenchDone(f"Dialogue voice benchmark finished: {out_dir}. Listen and fill listening_sheet.md before "
                         "opening blind_test_key.json; then set VOICE_BENCH = False.")


class VoiceBenchDone(Exception):
    """Raised on purpose after the voice benchmark so 'Run all' stops before making the video."""


def assemble(chunks, out):
    """Chunks [(wav, pause_before_s, pause_after_s)] → one track: real silences, 8 ms edge fades, no per-chunk
    loudness normalisation (the intended dynamics stay)."""
    parts = []
    for w, pb, pa in chunks:
        w = np.asarray(w, dtype=np.float32).copy(); n_ = min(int(SR * 0.008), len(w) // 2)
        if n_:
            r_ = np.linspace(0, 1, n_, dtype=np.float32); w[:n_] *= r_; w[-n_:] *= r_[::-1]
        parts += [np.zeros(int(SR * pb), np.float32), w, np.zeros(int(SR * pa), np.float32)]
    sf.write(out, np.concatenate(parts), SR)
    return out


def run_voice_bench():
    import random
    out_dir = PROJ / "render" / "voice_bench"; out_dir.mkdir(parents=True, exist_ok=True)
    shots = [{"shot_id": f"b{i + 1}", "beat": beat, "chapter": beat, "lines": [{"id": f"b{i + 1}_1", "speaker": "narrator",
              "text": t}]} for i, (beat, t) in enumerate(BENCH_SCRIPT)]
    order = [s_["lines"][0]["id"] for s_ in shots]
    expressive, adj = voice_direction.smooth(order, voice_direction.rule_directions(shots, VOICE_PROFILE), VOICE_PROFILE)
    neutral = voice_direction.neutral_directions(order, expressive)
    text = {s_["lines"][0]["id"]: s_["lines"][0]["text"] for s_ in shots}
    (out_dir / "voice_direction.json").write_text(json.dumps({k: v.model_dump() for k, v in expressive.items()}, indent=1))
    try:
        spk = voice_qc.SpeakerCheck(WORK.parent / "ecapa")
    except Exception:
        log_exc("speaker check unavailable in the voice benchmark"); spk = None
    report, prosody, samples = {}, {}, []
    for be in VOICE_BENCH_BACKENDS:
        voice = f"qwen:{NARRATOR_QWEN_SPEAKER}" if be == "qwen3-tts" else CHARS["narrator"]["voice"]
        for variant, dirs in (("neutral", neutral), ("expressive", expressive)):
            key = f"{be}_{variant}"
            t0, chunks, rows, embs = time.time(), [], {}, {}
            try:
                if be == "qwen3-tts":
                    get_qwen_tts()
                load_s = time.time() - t0
                for lid in order:
                    d = dirs[lid]
                    instr = expressive_tts.qwen_params(d, voice)["instruct"] if be == "qwen3-tts" else ""
                    speed = expressive_tts.kokoro_params(d, 1.0)["speed"] if be == "kokoro" else 1.0
                    t1 = time.time()
                    wav, sr = speak(text[lid], speed, voice, attempt=0, instruct=instr)
                    gen = time.time() - t1
                    tmp = out_dir / f"{key}_{lid}.wav"; sf.write(tmp, np.asarray(wav, dtype=np.float32), SR)
                    heard, _ = listen(tmp)
                    err = None if heard is None else speech_error([text[lid]], heard)
                    vq = check_line(tmp, text[lid], heard, err)
                    vq.update(gen_s=round(gen, 2), rtf=round(gen / max(len(wav) / SR, 0.1), 2),
                              peak_vram_gb=LAST_TTS_INFO.get("peak_vram_gb") if be == "qwen3-tts" else None)
                    rows[lid] = vq
                    if spk:
                        embs[lid] = spk.embed(wav, SR)
                    chunks.append((wav, d.pause_before_ms / 1000, d.pause_after_ms / 1000))
                    prosody[f"{key}/{lid}"] = {"direction": d.model_dump(), "applied": expressive_tts.report(d, be),
                                               "instruct": instr, "speed": speed}
                if spk:
                    for lid, sim in voice_qc.consistency(embs).items():
                        rows[lid]["checks"]["same narrator voice"] = {"status": "UNKNOWN" if sim is None else
                            "PASS" if sim >= voice_qc.SPEAKER_TH.get(be, 0.8) else "FAIL", "value": sim}
                f = assemble(chunks, out_dir / f"narration_{variant}_{be}.wav")
                samples.append((key, f))
                report[key] = {"load_s": round(load_s, 1), "total_s": round(time.time() - t0, 1),
                               "audio_s": round(sum(len(c[0]) for c in chunks) / SR, 1), "lines": rows,
                               "fails": {lid: [c for c, x in r["checks"].items() if x["status"] == "FAIL"] for lid, r in rows.items()}}
                print(f"  {key}: {report[key]['total_s']:.0f}s for {report[key]['audio_s']}s of speech | fails: "
                      + (", ".join(f"{k}:{v}" for k, v in report[key]["fails"].items() if v) or "none"))
            except Exception as e:
                log_exc(f"voice bench {key} failed"); report[key] = {"error": repr(e)[:300]}
                print(f"  {key}: FAILED {repr(e)[:200]}")
        if be == "qwen3-tts":
            close_tts()                                        # one TTS model family at a time
    (out_dir / "prosody_report.json").write_text(json.dumps(prosody, indent=1, default=str))
    (out_dir / "voice_qc_report.json").write_text(json.dumps(report, indent=1, default=str))
    # blind A/B: each backend's neutral and expressive take in random order; the key is saved, not shown
    rnd = random.Random(); key_ = {}
    try:
        from IPython.display import Audio, display
        for be in VOICE_BENCH_BACKENDS:
            pair = [x for x in samples if x[0].startswith(be)]
            rnd.shuffle(pair)
            for lab, (k_, f_) in zip("AB", pair):
                key_[f"{be} {lab}"] = k_
                print(f"{be} — sample {lab}:"); display(Audio(str(f_)))
    except Exception:
        log_exc("audio preview unavailable")
    (out_dir / "blind_test_key.json").write_text(json.dumps(key_, indent=1))
    raise VoiceBenchDone(f"Voice benchmark finished: {out_dir}. Listen to A/B before opening blind_test_key.json; "
                         "then set VOICE_BENCH = False and VOICE_BACKEND to the winner.")


def backend_of(voice):
    return ("qwen3-tts" if voice.startswith("qwen:") else "gemini" if voice.startswith("gemini:")
            else "parler" if LANGUAGE != "en" else "kokoro")
RETAKE_BACKENDS = ("qwen3-tts", "parler")   # local sampling backends: a new seed is a genuinely new take. Gemini lines
                                            # come only from the planned two-speaker pass (budget, conversational flow)

def line_settings(l):
    """Everything that decides how a line sounds (also the cache key: change any of it and the line is remade)."""
    c, t = CHARS[l["speaker"]], TONES[l["tone"]]
    d = DIRECTIONS.get(l["id"])
    # with a direction, emotion is never faked by DSP: no tone pitch shift / gain / tremolo (character pitch stays:
    # that is the character's identity, not an emotion)
    pitch = round(c.get("pitch", 0) + (0 if d else t.get("pitch", 0)), 2)
    gain = round(c.get("gain_db", 0) + (0 if d else t.get("gain_db", 0)), 2)
    chain = []
    if abs(pitch) > 0.01:                             # pitch shift that keeps the length (and word timings)
        r = 2 ** (pitch / 12)
        chain += [f"asetrate={round(SR * r)}", f"aresample={SR}", f"atempo={1 / r:.5f}"]
    for fx in [VOICE_FX[c.get("fx", "none")], "" if d else TONE_FX.get(t.get("fx"), "")]:
        if fx:
            chain.append(fx)
    if gain:
        chain.append(f"volume={gain}dB")
    chain.append("alimiter=limit=0.95")
    if GEMINI_ACTIVE and c.get("gemini_voice"):      # Telugu via Gemini free tier (its own pronunciation rules)
        return {"text": gem_text(l["text"]), "voice": f"gemini:{c['gemini_voice']}",
                "speed": 1.0, "fx": "alimiter=limit=0.95", **({"direction": d.model_dump(), "instruct": ""} if d else {})}
    if LANGUAGE != "en":
        voice = "|".join([c["voice"], ", ".join(x for x in (c.get("voice_style"),
                                                           expressive_tts.parler_style(d) if d else t.get("describe")) if x)])
    elif EXPRESSIVE and VOICE_BACKEND == "qwen3-tts" and l["speaker"] == "narrator":
        voice = f"qwen:{NARRATOR_QWEN_SPEAKER}"         # one fixed premium speaker = the narrator's identity
    elif EXPRESSIVE and VOICE_BACKEND == "qwen3-tts" and c.get("qwen_speaker"):
        voice = f"qwen:{c['qwen_speaker']}"             # a friend with an approved Qwen speaker in the Bible
    else:
        voice = c["voice"]
    want = {"text": l.get("tts") or l["text"], "voice": voice,
            "speed": round(c.get("speed", 1.0) * (d.pace_multiplier if d else t.get("speed", 1.0)) * SPEED_SCALE, 3),
            "fx": ",".join(chain)}
    if d:
        want["direction"] = d.model_dump()
        want["instruct"] = expressive_tts.qwen_params(d, voice)["instruct"] if voice.startswith("qwen:") else ""
    return want

def check_line(path, text, heard, err):
    """Voice QC of one finished line (voice_qc.py): signal checks + missing/repeated words."""
    wav, sr = sf.read(str(path), dtype="float32")
    m = voice_qc.measure(wav, sr, len(text.split()), text=text)
    diff = voice_qc.word_diff(norm_words(text), norm_words(heard)) if heard else None
    return voice_qc.verdict(m, err, WER_RETRY, None, diff) | {"metrics": m}

# ---- Telugu via Google Gemini FREE TIER (optional): preflight → two-speaker chunks → split per turn → checks.
# Only the approved dialogue text (+ short style words) is sent; story, facts and checks stay local. If ANY condition
# or line fails, the whole video uses the local Indic Parler voices (one voice per character per video).
import telugu_dialogue, gemini_tts, importlib
GEMINI_ACTIVE, GEMINI_REPORT, GEMINI_CLIENT, GEMINI_KEY = False, None, None, ""
def gemini_wanted():
    if LANGUAGE != "te":
        return False
    if globals().get("STRICT_OPEN_SOURCE"):          # strict open source: Gemini only on an explicit opt-in
        return TELUGU_TTS_BACKEND == "gemini_free"
    return TELUGU_TTS_BACKEND == "gemini_free" or (TELUGU_TTS_BACKEND == "auto" and PLAN.get("tts_backend") == "gemini_free")
def gem_text(text):
    """What Gemini reads (subtitles keep `text`): the form chosen by the listening benchmark."""
    form = "gemini_free" if globals().get("GEMINI_TRANSCRIPT_FORM", "display") == "display" else "gemini_free:telugu_script"
    return telugu_dialogue.tts_text(text, form)
VOICE_POLICY = PLAN.get("voice_policy", {})         # per-story voice rules (absent: the earlier behaviour)
CODE_MIX = PLAN.get("code_mix")
def gemini_key():
    """GEMINI_API_KEY from Colab Secrets (or the environment). Never printed, logged or saved."""
    try:
        from google.colab import userdata
        k = userdata.get("GEMINI_API_KEY")
    except Exception:
        k = None
    return (k or os.environ.get("GEMINI_API_KEY", "")).strip()
GEMINI_CFG = dict(enabled=gemini_wanted(), model=GEMINI_TTS_MODEL, free_tier_confirmed=GEMINI_FREE_TIER_CONFIRMED,
                  free_tier_only=GEMINI_FREE_TIER_ONLY, max_videos_per_day=MAX_VIDEOS_PER_DAY,
                  max_calls_per_video=MAX_GEMINI_TTS_CALLS_PER_VIDEO, max_retries=MAX_GEMINI_RETRIES_PER_SEGMENT,
                  mode=GEMINI_MODE, reserve_calls=2, asr_repairs=VOICE_POLICY.get("gemini_asr_repairs", True))
GEMINI_LEDGER = gemini_tts.Ledger(Path(DRIVE_ROOT) / "_gemini_usage.json")
GEMINI_CACHE = PROJ / "gemini_audio"

def gemini_client():
    """The Gemini REST client (no package to install; tests inject a stand-in)."""
    global GEMINI_CLIENT, GEMINI_KEY
    if GEMINI_CLIENT is None:
        GEMINI_KEY = gemini_key()
        if not GEMINI_KEY:
            return None
        try:                                           # REST with Colab's own `requests`: nothing is installed
            GEMINI_CLIENT = (globals().get("GEMINI_CLIENT_FACTORY") or gemini_tts.make_client)(GEMINI_KEY)
        except Exception as e:
            log("Gemini client could not start: " + gemini_tts.redact(e, GEMINI_KEY)[:300], "ERROR")
    return GEMINI_CLIENT

def GEMINI_SPEAK(text, voice, attempt=0):
    """One line on demand (only if a Gemini line must be remade): cached, budgeted, never a paid route."""
    who = next(k for k, c in CHARS.items() if c.get("gemini_voice") == voice)
    p_, info = gemini_tts.synthesize(gemini_client(), GEMINI_CFG, [{"speaker": who, "text": text}], {who: voice},
                                     GEMINI_CACHE, GEMINI_LEDGER, PROJECT, GEMINI_KEY, log, label=f"single {attempt}")
    tmp_ = WORK / f"gemini_single_{os.getpid()}.wav"; ff("-i", p_, "-ar", SR, "-ac", 1, tmp_)
    return sf.read(str(tmp_), dtype="float32")

def run_gemini_voices():
    global GEMINI_ACTIVE, GEMINI_REPORT
    ok_scope, why_scope = registry.hosted_ok(GEMINI_TTS_MODEL, "tts")
    lines = [{"id": l["id"], "scene": s_["shot_id"], "speaker": l["speaker"], "intent": l.get("intent") or
              (DIRECTIONS[l["id"]].intent if l["id"] in DIRECTIONS else None),
              "text": gem_text(l["text"])} for s_ in SHOTS for l in s_["lines"]]
    voices = {k: c["gemini_voice"] for k, c in CHARS.items() if c.get("gemini_voice")}
    done = all((read_sidecar(PROJ / "narration" / f"{l['id']}.wav") or {}).get("backend") == "gemini"
               and (read_sidecar(PROJ / "narration" / f"{l['id']}.wav") or {}).get("text") == l["text"]
               and (PROJ / "narration" / f"{l['id']}.wav").exists() for l in lines)
    if done and (PROJ / "render" / "gemini_report.json").exists():
        GEMINI_REPORT = json.loads((PROJ / "render" / "gemini_report.json").read_text()); GEMINI_ACTIVE = True
        print("Telugu voices: Gemini lines already made (cache), no API call"); return
    pf = gemini_tts.preflight(GEMINI_CFG, gemini_key(), GEMINI_LEDGER, PROJECT,
                              gemini_client() if gemini_key() else None)
    if not ok_scope:
        pf["checks"]["hosted exception scope"] = ("POLICY_BLOCKED", why_scope); pf["status"] = "POLICY_BLOCKED"
    print("Gemini free-tier preflight: " + pf["status"])
    for k_, (st_, why_) in pf["checks"].items():
        print(f"  {'✅' if st_ == 'PASS' else '❌'} {k_}: {st_}" + (f" — {why_}" if why_ and st_ != "PASS" else ""))
    GEMINI_REPORT = {"status": "FALLBACK", "preflight": pf, "why": f"preflight {pf['status']}"}
    if pf["status"] == "PASS":
        asr = globals().get("GEMINI_ASR_FN") or listen
        spk = globals().get("GEMINI_SPEAKER_FN")          # (tests inject a stand-in; normally ECAPA)
        if spk is None:
            try:
                _sc = voice_qc.SpeakerCheck(WORK.parent / "ecapa"); spk = _sc.embed
            except Exception:
                log_exc("speaker check unavailable for the Gemini attribution check")
        t0 = time.time()
        rep = gemini_tts.run_voices(lines, voices, GEMINI_CFG, gemini_client(), GEMINI_CACHE, GEMINI_LEDGER, PROJECT,
                                    WORK / "gemini_lines", asr_fn=asr, key=GEMINI_KEY, log=log,
                                    retry_at=SPEECH_RETRY_OTHER, speaker_fn=spk,
                                    err_fn=telugu_dialogue.bilingual_error if CODE_MIX else None)
        rep.update(preflight=pf, seconds=round(time.time() - t0, 1))
        GEMINI_REPORT = rep
    if GEMINI_REPORT["status"] == "PASS":
        GEMINI_ACTIVE = True
        DEVICE_REPORT["voice (Gemini free tier)"] = "Google's servers (no local GPU or RAM)"
        for l in lines:
            if l["id"] in GEMINI_REPORT["gaps"] and l["id"] in DIRECTIONS:      # keep the model's own response timing
                DIRECTIONS[l["id"]].pause_before_ms = min(GEMINI_REPORT["gaps"][l["id"]], 1200)
        by_line = {l["id"]: l for s_ in SHOTS for l in s_["lines"]}
        for lid, m in GEMINI_REPORT["lines"].items():
            out = PROJ / "narration" / f"{lid}.wav"
            ff("-i", WORK / "gemini_lines" / f"{lid}.wav", "-af", "alimiter=limit=0.95", "-ar", SR, "-ac", 1, out)
            want = line_settings(by_line[lid])
            sidecar(out, **want, backend="gemini", model=GEMINI_TTS_MODEL, speaker=by_line[lid]["speaker"],
                    tone=by_line[lid]["tone"], words=m.get("words") or [], heard=m.get("heard"), wer=m.get("err"),
                    flagged=bool(m.get("review")), how=m.get("how"),
                    takes={"takes": [{"take": 1}], "selected_take_id": f"{lid}_t1"})
        print(f"Telugu voices: Gemini {GEMINI_TTS_MODEL} — {GEMINI_REPORT['calls']} API calls for this video, "
              f"{len(GEMINI_REPORT['chunks'])} conversation chunks ({GEMINI_REPORT['seconds']}s)")
    else:
        print(f"Telugu voices: local Indic Parler-TTS (Gemini not used: {GEMINI_REPORT['why'][:300]})")
    for ch_ in GEMINI_REPORT.get("chunks", []):              # what happened to each conversation chunk
        print(f"  chunk {ch_['lines'][0]}..{ch_['lines'][-1]}: split {ch_.get('align', '-')}"
              + (f" ({ch_['why']})" if ch_.get("why") else "")
              + (f", cut without pause at {ch_['cuts_without_pause']}" if ch_.get("cuts_without_pause") else "")
              + (f", repaired {ch_['repaired']}" if ch_.get("repaired") else "")
              + (f", flagged for listening {ch_['flagged_for_review']}" if ch_.get("flagged_for_review") else "")
              + (f", made per line" if ch_.get("mode") == "per-turn" else ""))
    (PROJ / "render").mkdir(exist_ok=True)
    (PROJ / "render" / "gemini_report.json").write_text(json.dumps(GEMINI_REPORT, indent=1, default=str, ensure_ascii=False))
    (PROJ / "render" / "gemini_usage_today.json").write_text(json.dumps(GEMINI_LEDGER.today(), indent=1))

# ---- TELUGU_VOICE_BENCH / TELUGU_VOICE_AUDITION (one-off, then stop): same Telugu dialogue, several voices ----
AUDITION_PAIRS = [("Puck", "Kore"), ("Achird", "Sulafat"), ("Fenrir", "Aoede"), ("Zubenelgenubi", "Vindemiatrix")]
def run_telugu_bench():
    import random
    out_dir = PROJ / "render" / "telugu_voice_bench"; out_dir.mkdir(parents=True, exist_ok=True)
    first = [s_ for s_ in SHOTS][:3]                       # opening, question, demonstration (+ explanation)
    lines = [{"id": l["id"], "scene": s_["shot_id"], "speaker": l["speaker"], "text": l["text"], "tone": l["tone"],
              "intent": DIRECTIONS[l["id"]].intent if l["id"] in DIRECTIONS else None} for s_ in first for l in s_["lines"]]
    rows, samples = {}, []
    project = f"bench:{PROJECT}"
    cfg = dict(GEMINI_CFG, enabled=True, max_calls_per_video=MAX_GEMINI_BENCH_CALLS)
    pf = gemini_tts.preflight(cfg, gemini_key(), GEMINI_LEDGER, project, gemini_client() if gemini_key() else None)
    def assemble_lines(paths, gaps, out):
        parts = []
        for f_, g_ in zip(paths, gaps):
            w_, r_ = sf.read(str(f_), dtype="float32")
            if r_ != SR:
                t_ = WORK / "bench_rs.wav"; ff("-i", f_, "-ar", SR, "-ac", 1, t_); w_, _ = sf.read(str(t_), dtype="float32")
            parts += [np.zeros(int(SR * g_), np.float32), w_]
        sf.write(out, np.concatenate(parts), SR); return out
    if TELUGU_VOICE_BENCH:
        # A. local Indic Parler (existing voices, per line, the director's gaps)
        try:
            t0, paths, errs = time.time(), [], {}
            for l in lines:
                c = CHARS[l["speaker"]]
                wav, sr = speak(telugu_dialogue.tts_text(l["text"], "indic_parler"), 1.0,
                                "|".join([c["voice"], c.get("voice_style") or ""]), attempt=0)
                f_ = out_dir / f"parler_{l['id']}.wav"; sf.write(f_, np.asarray(wav, np.float32), SR); paths.append(f_)
                h_, _ = listen(f_); errs[l["id"]] = None if h_ is None else gemini_tts.letter_error(l["text"], h_)
            gaps = [0] + [DIRECTIONS[l["id"]].pause_before_ms / 1000 if l["id"] in DIRECTIONS else 0.3 for l in lines[1:]]
            samples.append(("indic_parler", assemble_lines(paths, gaps, out_dir / "A_parler.wav")))
            rows["indic_parler"] = {"status": "PASS", "seconds": round(time.time() - t0, 1), "api_calls": 0, "letter_error": errs}
        except Exception as e:
            log_exc("Telugu bench: Parler failed")
            rows["indic_parler"] = {"status": "ACCESS_REQUIRED" if "HF_TOKEN" in str(e) or "gated" in str(e) else "FAIL",
                                    "why": str(e)[:200]}
        close_tts()
        # B/C/D. Gemini (two-speaker chunk, per turn, optional Flash-Lite)
        variants = [("gemini_flash_scene", GEMINI_TTS_MODEL, "scene"), ("gemini_flash_turn", GEMINI_TTS_MODEL, "turn")]
        if TELUGU_BENCH_LITE:
            variants.append(("gemini_lite_scene", "gemini-3.8-flash-lite-tts", "scene"))
        voices = {k: c["gemini_voice"] for k, c in CHARS.items() if c.get("gemini_voice")}
        for name, model, mode in variants:
            if pf["status"] != "PASS":
                rows[name] = {"status": pf["status"], "why": {k: v for k, v in pf["checks"].items() if v[0] != "PASS"}}
                continue
            t0 = time.time()
            cfg_v = dict(cfg, model=model, mode=mode, reserve_calls=0)
            calls0 = GEMINI_LEDGER.calls(project)
            rep = gemini_tts.run_voices([dict(l, text=gem_text(l["text"])) for l in lines],
                                        voices, cfg_v, gemini_client(), GEMINI_CACHE, GEMINI_LEDGER, project,
                                        out_dir / name, asr_fn=listen, key=GEMINI_KEY, log=log, retry_at=SPEECH_RETRY_OTHER)
            st_ = "PASS" if rep["status"] == "PASS" else ("FREE_QUOTA_EXHAUSTED" if "QUOTA" in rep["why"] else
                                                          "ACCESS_REQUIRED" if "ACCESS" in rep["why"] else "FAIL")
            rows[name] = {"status": st_, "why": rep["why"], "seconds": round(time.time() - t0, 1),
                          "api_calls": GEMINI_LEDGER.calls(project) - calls0, "chunks": rep["chunks"],
                          "letter_error": {k: v.get("err") for k, v in rep["lines"].items()}}
            if rep["status"] == "PASS":
                gaps = [0] + [rep["gaps"].get(l["id"], 300) / 1000 for l in lines[1:]]
                samples.append((name, assemble_lines([out_dir / name / f"{l['id']}.wav" for l in lines], gaps,
                                                     out_dir / f"{name}.wav")))
    if TELUGU_VOICE_AUDITION and pf["status"] == "PASS":
        ex = lines[:4]
        for a_, m_ in AUDITION_PAIRS:
            try:
                p_, info = gemini_tts.synthesize(gemini_client(), dict(cfg, mode="scene"),
                                                 [{"speaker": l["speaker"], "text": gem_text(l["text"])}
                                                  for l in ex], {"arjun": a_, "meera": m_}, GEMINI_CACHE, GEMINI_LEDGER,
                                                 project, GEMINI_KEY, log, label=f"audition {a_}/{m_}")
                samples.append((f"audition Arjun={a_} Meera={m_}", p_))
                rows[f"audition {a_}/{m_}"] = {"status": "PASS", "duration_s": info["duration_s"]}
            except gemini_tts.GeminiBlocked as e:
                rows[f"audition {a_}/{m_}"] = {"status": e.status, "why": e.why}; break
    elif TELUGU_VOICE_AUDITION:
        rows["audition"] = {"status": pf["status"], "why": {k: v for k, v in pf["checks"].items() if v[0] != "PASS"}}
    (out_dir / "bench_report.json").write_text(json.dumps({"preflight": pf, "results": rows,
        "dialogue": [(l["id"], l["speaker"], l["text"]) for l in lines]}, indent=1, ensure_ascii=False, default=str))
    bench = [x for x in samples if not x[0].startswith("audition")]
    random.Random().shuffle(bench)
    key_ = {}
    sheet = ["# Telugu listening sheet (native speakers, 1–5)", "", "Dialogue:", ""] + \
            [f"- {CHARS[l['speaker']]['name']}: {l['text']}" for l in lines] + ["",
            "| Sample | Pronunciation | Natural delivery | Emotion fits | Voice consistency | Characters believable | "
            "Timing | Overall | Sounds like two real people? | Notes |", "|---|---|---|---|---|---|---|---|---|---|"]
    try:
        from IPython.display import Audio, display
        print("Telugu dialogue:\n" + "\n".join(f"  {CHARS[l['speaker']]['name']}: {l['text']}" for l in lines))
    except Exception:
        Audio = display = None
    for lab, (name, f_) in zip("ABCDEFGH", bench):
        key_[lab] = name; sheet.append(f"| {lab} | | | | | | | | | |")
        if display:
            print(f"Sample {lab}:"); display(Audio(str(f_)))
    for name, f_ in samples:
        if name.startswith("audition"):
            sheet.append(f"| {name} | | | | | | | | | |")
            if display:
                print(name + ":"); display(Audio(str(f_)))
    (out_dir / "blind_key.json").write_text(json.dumps(key_, indent=1))
    (out_dir / "listening_sheet.md").write_text("\n".join(sheet) + "\n")
    for k_, r_ in rows.items():
        print(f"  {k_}: {r_['status']}" + (f" — {str(r_.get('why'))[:150]}" if r_["status"] != "PASS" else
                                           f" ({r_.get('api_calls', 0)} API calls)"))
    raise VoiceBenchDone(f"Telugu voice benchmark/audition finished: {out_dir}. Native speakers fill "
                         "listening_sheet.md before opening blind_key.json; then set the flags back to False.")

# ---- TELUGU_PRON_BENCH (one-off, then stop): the same short code-mixed exchange in each pronunciation form ----
PRON_QUESTIONS = ["Is this how friends actually talk?", "Are the English words natural?", "Are both voices understandable?",
                  "Do the reactions sound genuine?", "Are any pronunciations strange? (which words)"]
def run_pron_bench():
    import random
    out_dir = PROJ / "render" / "pron_bench"; out_dir.mkdir(parents=True, exist_ok=True)
    allL = [dict(l, scene=s_["shot_id"]) for s_ in SHOTS for l in s_["lines"]]
    lines = allL[:4]                                     # the opening exchange: popcorn, test, 3D effect
    rows, samples = {}, []
    forms = [("parler_respelled", "indic_parler:code_mix"), ("parler_latin", None)]
    try:
        acc = registry.gated_access("ai4bharat/indic-parler-tts", hf_token())
    except Exception as e:
        acc = {"status": "TOKEN_MISSING", "why": str(e)[:200], "help": registry.ACCESS_HELP}
    for name, form in forms:
        if acc["status"] != "PASS":
            rows[name] = {"status": acc["status"], "why": acc["why"]}; continue
        try:
            paths = []
            for l in lines:
                c = CHARS[l["speaker"]]
                txt = telugu_dialogue.tts_text(l["text"], form) if form else l["text"]
                wav, sr = speak(txt, 1.0, "|".join([c["voice"], c.get("voice_style") or ""]), attempt=0)
                f_ = out_dir / f"{name}_{l['id']}.wav"; sf.write(f_, np.asarray(wav, np.float32), SR); paths.append(f_)
            gap = np.zeros(int(SR * 0.35), np.float32)
            mix = np.concatenate([x for f_ in paths for x in (sf.read(str(f_), dtype="float32")[0], gap)])
            sf.write(out_dir / f"{name}.wav", mix, SR); samples.append((name, out_dir / f"{name}.wav"))
            rows[name] = {"status": "PASS", "tts_text": [telugu_dialogue.tts_text(l["text"], form) if form else l["text"] for l in lines]}
        except Exception as e:
            log_exc(f"pronunciation bench {name} failed"); rows[name] = {"status": "FAIL", "why": repr(e)[:200]}
    close_tts()
    if gemini_wanted():                                  # opt-in only: 2 scene requests under the benchmark budget
        cfg = dict(GEMINI_CFG, enabled=True, max_calls_per_video=MAX_GEMINI_BENCH_CALLS, mode="scene")
        project = f"bench:{PROJECT}"
        pf = gemini_tts.preflight(cfg, gemini_key(), GEMINI_LEDGER, project, gemini_client() if gemini_key() else None)
        voices = {k: c["gemini_voice"] for k, c in CHARS.items() if c.get("gemini_voice")}
        for name, form in (("gemini_display", "gemini_free"), ("gemini_telugu_script", "gemini_free:telugu_script")):
            if pf["status"] != "PASS":
                rows[name] = {"status": pf["status"]}; continue
            try:
                p_, info = gemini_tts.synthesize(gemini_client(), cfg, [{"speaker": l["speaker"],
                                                 "text": telugu_dialogue.tts_text(l["text"], form)} for l in lines],
                                                 voices, GEMINI_CACHE, GEMINI_LEDGER, project, GEMINI_KEY, log, label=f"pron {name}")
                samples.append((name, p_)); rows[name] = {"status": "PASS", "cache_hit": info.get("cache_hit")}
            except gemini_tts.GeminiBlocked as e:
                rows[name] = {"status": e.status, "why": e.why}; break
    random.Random().shuffle(samples)
    key_ = {lab: n for lab, (n, _) in zip("ABCDEFGH", samples)}
    sheet = ["# Telugu–English pronunciation listening sheet (native speakers)", "",
             "Same four lines in each sample. Answer 1–5 (5 = yes, completely) and name strange words.", ""] + \
            [f"- {CHARS[l['speaker']]['name']}: {l['text']}" for l in lines] + ["",
             "| Sample | " + " | ".join(PRON_QUESTIONS) + " | Notes |", "|---" * (len(PRON_QUESTIONS) + 2) + "|"] + \
            [f"| {lab} |" + " |" * (len(PRON_QUESTIONS) + 1) for lab in key_]
    (out_dir / "listening_sheet.md").write_text("\n".join(sheet) + "\n")
    (out_dir / "blind_key.json").write_text(json.dumps(key_, indent=1))
    (out_dir / "pron_report.json").write_text(json.dumps({"parler_access": acc, "results": rows}, indent=1,
                                                          ensure_ascii=False, default=str))
    try:
        from IPython.display import Audio, display
        for lab, (n, f_) in zip("ABCDEFGH", samples):
            print(f"Sample {lab}:"); display(Audio(str(f_)))
    except Exception:
        pass
    for k_, r_ in rows.items():
        print(f"  {k_}: {r_['status']}" + (f" — {r_.get('why', '')}" if r_["status"] != "PASS" else ""))
    if acc["status"] != "PASS":
        print(acc["help"])
    raise VoiceBenchDone(f"pronunciation benchmark finished: {out_dir}. Fill listening_sheet.md, then open blind_key.json; "
                         "set the winning form (GEMINI_TRANSCRIPT_FORM / the Parler table) and TELUGU_PRON_BENCH = False.")

def parler_gate():
    """Before any local Telugu synthesis (and before the benchmarks): is the gated Indic Parler-TTS open for this
    token? Uses the worker's own check (whoami + gated-repo access; 401 and 403 each get their fix) — it stops with
    the steps, never bypasses the gate and never switches a character to another voice."""
    tok_ = hf_token()                                    # no token: stops with the Colab Secrets how-to
    check_hf_access(tok_, os.environ.get("PARLER_MODEL", "ai4bharat/indic-parler-tts"))
    tok_ = None
    DEVICE_REPORT["Parler access"] = "checked before synthesis"
    return "PASS"

if VOICE_BENCH:
    run_dialogue_bench() if PLAN.get("story_mode") == "two_friends_cinematic" else run_voice_bench()
if TELUGU_VOICE_BENCH or TELUGU_VOICE_AUDITION:
    run_telugu_bench()
if TELUGU_PRON_BENCH:
    run_pron_bench()
if gemini_wanted():
    run_gemini_voices()
elif LANGUAGE == "te" and STRICT_OPEN_SOURCE:
    print("Telugu voices: local Indic Parler-TTS (STRICT_OPEN_SOURCE: Gemini is opt-in — set TELUGU_TTS_BACKEND = "
          "\"gemini_free\" to use it)")
if not RENDER_CACHED_AUDIO_ONLY and LANGUAGE != "en" and not GEMINI_ACTIVE and (STRICT_OPEN_SOURCE or CODE_MIX) and any(
        not (PROJ / "narration" / f"{l['id']}.wav").exists() for s_ in SHOTS for l in s_["lines"]):
    PARLER_ACCESS = parler_gate()                        # only when a line still has to be made locally
RETAKE_AT = VOICE_POLICY.get("asr_retake_at", WER_RETRY)  # code-mixed Telugu: only a clearly wrong take is remade;
                                                          # a doubtful one is kept and flagged (WER_RETRY) for listening
FLAGGED, _voices_checked, TAKES = [], False, {}
for shot in SHOTS:
    for l in shot["lines"]:
        BUDGET.check("next narration line")
        want = line_settings(l)
        who = CHARS[l["speaker"]]["name"]
        out = PROJ / "narration" / f"{l['id']}.wav"
        old = read_sidecar(out)
        if out.exists() and old and all(old.get(k) == v for k, v in want.items()) \
                and (old.get("words") is not None or get_asr() is None):
            if old.get("flagged"):
                FLAGGED.append(l["id"])
            if len((old.get("takes") or {}).get("takes", [])) > 1:
                TAKES[l["id"]] = old["takes"]                 # approved takes are reused from the cache
            continue                                          # already made with these settings
        if not _voices_checked:
            missing = {c["voice"] for c in CHARS.values()} - set(get_tts().get_voices())
            assert not missing, f"voices not in this voice model: {missing} (available: {get_tts().get_voices()})"
            _voices_checked = True
        best = None
        # Kokoro gives the same audio for the same input, so retries change the speed slightly
        for attempt, sp in enumerate([want["speed"], round(want["speed"] - 0.05, 3), round(want["speed"] + 0.04, 3)][:MAX_TTS_ATTEMPTS]):
            BUDGET.check("next speech take")
            wav, sr = speak(want["text"], sp, want["voice"], attempt=attempt, instruct=want.get("instruct", ""))
            assert sr == SR, sr
            tmp = WORK / f"{l['id']}_try{attempt}.wav"
            sf.write(tmp, np.asarray(wav, dtype=np.float32), SR)
            heard, words = listen(tmp)
            err = None if heard is None else speech_error([l["text"], want["text"]], heard)
            if best is None or (err is not None and err < best["err"]):
                best = {"err": err if err is not None else 0.0, "tmp": tmp, "words": words, "speed": sp, "heard": heard}
            if err is None or err <= RETAKE_AT:
                break
            print(f"  {l['id']} ({who}): speech check heard {err:.0%} wrong words, trying again")
            log(f"    heard: {heard!r} | expected: {l['text']!r}", "WARN")
        ff("-i", best["tmp"], "-af", want["fx"], "-ar", SR, "-ac", 1, out)      # character voice + tone
        d_ = DIRECTIONS.get(l["id"])
        vq = check_line(out, want["text"], best["heard"], best["err"] if best["heard"] is not None else None)
        bad = [k for k, x in vq["checks"].items() if x["status"] == "FAIL" and not k.startswith(("words", "no missing"))
               and "(diagnostic)" not in k]
        if ALLOW_CALMER_RETAKE and bad and d_ is not None and backend_of(want["voice"]) in RETAKE_BACKENDS and d_.intensity > 0.15:
            # expressive take failed a signal check (odd pace, long gap, clipping...): once more, toned down
            calmer = d_.model_copy(update={"intensity": round(d_.intensity * 0.5, 2), "style_instruction": ""})
            calmer.style_instruction = voice_direction.instruction(calmer)
            wav2, _ = speak(want["text"], want["speed"], want["voice"], attempt=5, instruct=calmer.style_instruction)
            tmp2 = WORK / f"{l['id']}_calmer.wav"; sf.write(tmp2, np.asarray(wav2, dtype=np.float32), SR)
            heard2, words2 = listen(tmp2)
            err2 = None if heard2 is None else speech_error([l["text"], want["text"]], heard2)
            out2 = WORK / f"{l['id']}_calmer_fx.wav"; ff("-i", tmp2, "-af", want["fx"], "-ar", SR, "-ac", 1, out2)
            vq2 = check_line(out2, want["text"], heard2, err2)
            n_fail = lambda v: sum(x["status"] == "FAIL" for x in v["checks"].values())
            log(f"{l['id']}: expressive take failed {bad}; toned-down retake fails {n_fail(vq2)} vs {n_fail(vq)}")
            if n_fail(vq2) < n_fail(vq):
                shutil.copy(out2, out); vq = vq2
                best.update(heard=heard2, words=words2, err=err2 if err2 is not None else 0.0)
                vq["toned_down"] = calmer.intensity
        # ---- hero lines (opening, central question, reveal, ending): a bounded second take on backends whose takes
        # differ (Qwen3-TTS, Parler). Chosen by the checks only (fewer wrong words, then fewer failed checks), never
        # by "more pitch variation"; both takes are kept for the A/B listen below. Kokoro is deterministic: 1 take.
        takes = [{"take": 1, "err": best["err"] if best["heard"] is not None else None,
                  "fails": sum(x["status"] == "FAIL" for x in vq["checks"].values())}]
        n_takes = MAX_VOICE_TAKES_HERO if voice_direction.is_hero(l, shot, d_) else MAX_VOICE_TAKES_NORMAL
        if n_takes > 1 and get_asr() is not None and not ASR_ON_GPU:
            n_takes = 1                                   # each extra take costs a slow CPU speech check: skipped
            log(f"{l['id']}: speech check on CPU → no extra hero take")
        if n_takes > 1 and backend_of(want["voice"]) in RETAKE_BACKENDS:
            tdir = PROJ / "narration" / "takes"; tdir.mkdir(exist_ok=True)
            shutil.copy(out, tdir / f"{l['id']}_t1.wav")
            for k_ in range(2, n_takes + 1):
                try:                                      # an extra take is optional: its failure never stops the run
                    wk, _ = speak(want["text"], want["speed"], want["voice"], attempt=10 + k_, instruct=want.get("instruct", ""))
                except Exception as e:
                    log_exc(f"{l['id']}: extra take {k_} failed; keeping the takes already made")
                    takes.append({"take": k_, "err": None, "fails": 99, "error": repr(e)[:200]})
                    continue
                tk = WORK / f"{l['id']}_take{k_}.wav"; sf.write(tk, np.asarray(wk, dtype=np.float32), SR)
                tkf = tdir / f"{l['id']}_t{k_}.wav"; ff("-i", tk, "-af", want["fx"], "-ar", SR, "-ac", 1, tkf)
                hk, wdk = listen(tkf)
                ek = None if hk is None else speech_error([l["text"], want["text"]], hk)
                vqk = check_line(tkf, want["text"], hk, ek)
                takes.append({"take": k_, "err": ek, "fails": sum(x["status"] == "FAIL" for x in vqk["checks"].values())})
                if voice_qc.pick_take(takes, WER_RETRY) is takes[-1]:
                    shutil.copy(tkf, out); vq = vqk
                    best.update(heard=hk, words=wdk, err=ek if ek is not None else 0.0)
            sel = voice_qc.pick_take([t for t in takes if "error" not in t], WER_RETRY)["take"]
            log(f"{l['id']}: hero line, {len(takes)} takes {takes} → take {sel}")
            TAKES[l["id"]] = {"takes": takes, "selected_take_id": f"{l['id']}_t{sel}"}
        VQC[l["id"]] = vq
        flagged = best["heard"] is not None and best["err"] > WER_RETRY
        if flagged:
            FLAGGED.append(l["id"])
        sidecar(out, model={"kokoro": "kokoro-82M (onnx v1.0)", "parler": "ai4bharat/indic-parler-tts",
                            "qwen3-tts": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
                            "gemini": GEMINI_TTS_MODEL}[backend_of(want["voice"])],
                backend=backend_of(want["voice"]), applied=expressive_tts.report(d_, backend_of(want["voice"])),
                voice_qc=vq, tts_info=dict(LAST_TTS_INFO) if backend_of(want["voice"]) == "qwen3-tts" else None, **want, speaker=l["speaker"], tone=l["tone"],
                takes=TAKES.get(l["id"], {"takes": [{"take": 1}], "selected_take_id": f"{l['id']}_t1"}),
                speed_used=best["speed"], wer=None if best["heard"] is None else round(best["err"], 3),
                heard=best["heard"], words=best["words"], flagged=flagged)
        info = sf.info(str(out))
        print(f"  {l['id']} {who:<9} [{l['tone']}] {info.frames / info.samplerate:.1f}s"
              + ("" if best["heard"] is None else f"  speech check {best['err']:.0%} wrong words")
              + ("  ⚠️ CHECK THIS LINE" if flagged else ""))
# ---- the same narrator throughout? (ECAPA leave-one-out similarity; calibrated in tests/test_voice.py) ----
if EXPRESSIVE:
    # Do not co-load ECAPA with Parler on a free T4 or start extra identity retakes.
    DEVICE_REPORT['speaker identity'] = 'Not run: manual listening required in this preset'
    print('Speaker identity requires listening review; ECAPA is omitted from the free-tier preset.')
    _lines = [l for s_ in SHOTS for l in s_["lines"]]
    _sc = {l["id"]: read_sidecar(PROJ / "narration" / f"{l['id']}.wav") or {} for l in _lines}
    (PROJ / "render" / "prosody_report.json").write_text(json.dumps({
        "backend": {k: expressive_tts.BACKENDS.get(v.get("backend", "kokoro"), {}) for k, v in _sc.items()},
        "lines": {k: {"text": v.get("text"), "direction": v.get("direction"), "applied": v.get("applied"),
                      "instruct": v.get("instruct"), "tts_info": v.get("tts_info")} for k, v in _sc.items()}},
        indent=1, default=str))
    (PROJ / "render" / "voice_qc_report.json").write_text(json.dumps(
        {k: VQC.get(k) or v.get("voice_qc") for k, v in _sc.items()}, indent=1, default=str))
    _fails = {k: [c for c, x in (VQC.get(k) or v.get("voice_qc") or {"checks": {}})["checks"].items()
                  if x["status"] == "FAIL" and "(diagnostic)" not in c] for k, v in _sc.items()}
    print("Voice QC: " + ("all lines pass" if not any(_fails.values()) else
                          "; ".join(f"{k}: {', '.join(f)}" for k, f in _fails.items() if f)))
close_tts(); _tts = _asr = None; free_gpu()          # free voice + speech models before the big image/video models
if TAKES:                                             # A/B listening for hero lines (inline, no web server)
    (PROJ / "render" / "voice_takes.json").write_text(json.dumps(TAKES, indent=1))
    try:
        from IPython.display import Audio, display
        for lid_, t_ in TAKES.items():
            print(f"{lid_}: selected {t_['selected_take_id']} (checks: {t_['takes']})")
            for x_ in t_["takes"]:
                if "error" not in x_:
                    display(Audio(str(PROJ / "narration" / "takes" / f"{lid_}_t{x_['take']}.wav")))
    except Exception:
        log_exc("inline A/B players unavailable (not in a notebook)")
print("Voices ready." + (f" ⚠️ Listen to: {FLAGGED}" if FLAGGED else "")); ram()


In [ ]:
BUDGET.check('timeline')
# ==== 7. Timeline: shot lengths come from the lines; pauses come from the tone (or the line) ====
PRE, TAIL = 0.3, 1.5                                 # lead-in per shot, extra hold at the very end
t, TIMELINE, PLACED = 0.0, [], []
for i, shot in enumerate(SHOTS):
    m, start = mood(shot), t
    cur = start + float(shot.get("pre_s", PRE))
    for l in shot["lines"]:
        out = PROJ / "narration" / f"{l['id']}.wav"
        info = sf.info(str(out)); d = info.frames / info.samplerate
        dv = (globals().get("DIRECTIONS") or {}).get(l["id"])
        if dv is not None:
            cur += dv.pause_before_ms / 1000            # a meaningful silence before a reveal / question
        PLACED.append({"id": l["id"], "speaker": l["speaker"], "tone": l["tone"], "text": l["text"],
                       "start": cur, "end": cur + d, "words": (read_sidecar(out) or {}).get("words")})
        pause = l.get("pause_after", dv.pause_after_ms / 1000 if dv is not None else
                      TONES[l["tone"]].get("pause_after", m["pause_after"]))
        cur += d + float(pause)                       # breathing space after the line
    dur = max(cur - start, float(shot.get("min_s", 0)))
    if i == len(SHOTS) - 1:
        dur += TAIL
    dur = round(dur * FPS) / FPS                    # whole frames
    TIMELINE.append({"shot_id": shot["shot_id"], "mood": shot.get("mood", "calm"), "start": round(start, 3),
                     "dur": dur, "frames": round(dur * FPS)})
    t = start + dur
TOTAL = t
if TOTAL > MAX_VIDEO_DURATION_SECONDS:
    raise RuntimeError(f'Measured narration makes a {TOTAL:.1f}s video, over the {MAX_VIDEO_DURATION_SECONDS}s '
                       'local limit. Review the saved narration before changing the budget.')

# all voices on one track, each line at its exact time (never stretched)
track = np.zeros(int(math.ceil(TOTAL * SR)) + SR // 10, dtype=np.float32)
for p in PLACED:
    w, _ = sf.read(str(PROJ / "narration" / f"{p['id']}.wav"), dtype="float32")
    n_ = min(int(SR * 0.008), len(w) // 2)            # 8 ms fades at the chunk's (silent) edges: no clicks
    if n_:
        w = w.copy(); ramp = np.linspace(0, 1, n_, dtype=np.float32); w[:n_] *= ramp; w[-n_:] *= ramp[::-1]
    a = int(p["start"] * SR); track[a:a + len(w)] += w[: len(track) - a]
sf.write(WORK / "narration_full.wav", track, SR)

# subtitles: at most 42 characters per cue, timed to the spoken words, coloured per character (narrator in italics)
def _lines(text, maxlen=42):
    out, cur = [], ""
    for wd in text.split():
        if cur and len(cur) + 1 + len(wd) > maxlen:
            out.append(cur); cur = wd
        else:
            cur = (cur + " " + wd).strip()
    return out + ([cur] if cur else [])

def token_times(p):
    """Start time (relative to the line) of every word of the script text."""
    toks, dur = p["text"].split(), p["end"] - p["start"]
    cp = np.cumsum([0] + [len(tk) + 1 for tk in toks])[:-1].astype(float)
    total = cp[-1] + len(toks[-1])
    xs, ys = [0.0], [0.0]
    if p.get("words"):                               # match script words to what the recogniser heard
        dn = [" ".join(norm_words(tk)) for tk in toks]
        an = [" ".join(norm_words(w[0])) for w in p["words"]]
        for a, b, size in difflib.SequenceMatcher(None, dn, an, autojunk=False).get_matching_blocks():
            for k in range(size):
                if cp[a + k] > xs[-1]:
                    xs.append(cp[a + k]); ys.append(p["words"][b + k][1])
    xs.append(total); ys.append(dur)
    ys = np.maximum.accumulate(np.clip(ys, 0, dur))
    return np.interp(cp, xs, ys)                     # unmatched words: interpolated by position

def _ts(x):
    ms = int(round(x * 1000)); h, ms = divmod(ms, 3600000); m_, ms = divmod(ms, 60000); s_, ms = divmod(ms, 1000)
    return f"{h:02}:{m_:02}:{s_:02},{ms:03}"

def styled(txt, speaker):
    c = CHARS[speaker]
    return f"<i>{txt}</i>" if speaker == "narrator" else f'<font color="{c.get("color", "#FFFFFF")}">{txt}</font>'

cues, word_timed, ass_cues = [], 0, []
for p in PLACED:
    tt, k, lines = token_times(p), 0, _lines(p["text"])
    word_timed += bool(p.get("words"))
    starts = []
    for l in lines:
        starts.append(p["start"] + (0.0 if k == 0 else tt[k])); k += len(l.split())
    ends = starts[1:] + [p["end"]]
    cues += [(a, max(b, a + 0.6), styled(l, p["speaker"])) for a, b, l in zip(starts, ends, lines)]
    k = 0                                            # the same cues, word by word, for the highlighted ASS subtitles
    for a, b, l in zip(starts, ends, lines):
        ws = l.split()
        ass_cues.append((a, max(b, a + 0.6), [(w, p["start"] + tt[k + j]) for j, w in enumerate(ws)],
                         p.get("speaker", "narrator")))
        k += len(ws)
srt = "\n".join(f"{k}\n{_ts(a)} --> {_ts(b)}\n{txt}\n" for k, (a, b, txt) in enumerate(cues, 1))
(WORK / "subtitles.srt").write_text(srt); (PROJ / "render" / "subtitles.srt").write_text(srt)
_emph = [w for w in PLAN.get("vocabulary", []) if " " not in w]
_cols = {k: c.get("color", "#FFFFFF") for k, c in (CHARS.items() if "CHARS" in globals() else [])}
n_ass = subtitles.write_ass(WORK / "subtitles.ass", ass_cues, W, H, font=SUB_FONT, colors=_cols, emphasis=_emph)
shutil.copy(WORK / "subtitles.ass", PROJ / "render" / "subtitles.ass")

# the script, screenplay style (handy for the YouTube description or for checking the story)
script = [PLAN["title"].upper(), ""]
for shot in SHOTS:
    script.append(f"[{shot['shot_id']}: " + (shot.get("scene") or f"{shot.get('recipe')}: {shot.get('label', '')}") + "]")
    for l in shot["lines"]:
        script.append(f"  {CHARS[l['speaker']]['name'].upper()} ({l['tone']}): {l['text']}")
    script.append("")
(PROJ / "render" / "script.txt").write_text("\n".join(script))

def describe(shot):
    """What the viewer sees, in one line (for the storyboard)."""
    r = shot.get("recipe", "video")
    if r == "video":
        return f"AI video: {shot['scene']}. {shot['action']}"
    if r == "ui":
        heads = [w.get("text") for sc in shot["ui"]["screens"] for w in sc if w.get("type") == "header"]
        acts = [e["do"] + (f" {e.get('target', '')}" if e.get("target") else "") for e in shot["ui"].get("events", [])]
        return f"App screen ({' → '.join(heads)}): " + ", ".join(acts)
    if r == "journey":
        names = {s_["id"]: s_["label"] for s_ in PLAN["journey"]["stations"]}
        return "Journey: " + " → ".join(names[x] for x in shot["journey"].get("path", [])) + (
            f" (calendar: {shot['journey']['calendar']['days']} days)" if shot["journey"].get("calendar") else "")
    if r == "card":
        return "Card: " + " / ".join(shot["card"]["lines"])
    if r == "stock":
        return f"Licensed image ({shot['stock']['license']}): {shot['stock'].get('description', '')} — {shot['stock']['credit']}"
    if r == "story":                                  # the scene's own visual timeline (cuts inside the dialogue)
        return " → ".join(f"[{v['intent']} · {v['medium']}] {v.get('description', '')}"
                          + (f" ({v['label']})" if v.get("label") else "") for v in shot["story"]["visuals"])
    return r

_cues = musiclib.arrange(PLAN, TIMELINE, TOTAL) if MUSIC_STYLE == "library" else []
def cue_at(t):
    return next((c["track"] for c in _cues if c["start"] <= t < c["end"]), "generated score" if not _cues else "")
_by_shot = {}
for i in globals().get("ISSUES", []):
    _by_shot.setdefault(i["shot"], []).append(f"{i['level']} {i['rule']}")
claims_by_id = {c["id"]: c for c in PLAN.get("claims", [])}
sb = [f"# Storyboard: {PLAN['title']}", "", f"Total {TOTAL:.1f} s, {len(SHOTS)} scenes. Generated by the notebook "
      "from the plan (do not edit; edit plan.json).", "",
      "| # | Time | Beat / step | What the viewer sees | Narration / dialogue | Music · SFX | Sources | Checks |",
      "|---|---|---|---|---|---|---|---|"]
for row, shot in zip(TIMELINE, SHOTS):
    said = " ".join(f"**{CHARS[l['speaker']]['name']}** ({l['tone']}): {l['text']}" for l in shot["lines"])
    sfx_ = ", ".join(c["type"] for c in shot.get("sfx", []))
    src = ", ".join(shot.get("claims", []))
    sb.append(f"| {shot['shot_id']} | {row['start']:.1f}–{row['start'] + row['dur']:.1f}s "
              f"| {shot.get('beat', '')} {('· ' + shot['step']) if shot.get('step') else ''} "
              f"| {describe(shot)}{(' · label: ' + shot['label']) if shot.get('label') else ''} | {said} "
              f"| {cue_at(row['start'] + 0.01)}{(' · ' + sfx_) if sfx_ else ''} | {src} "
              f"| {', '.join(_by_shot.get(shot['shot_id'], [])) or 'ok'} |")
if PLAN.get("claims"):
    sb += ["", "## Sources", ""] + [f"- **{c['id']}**: {c['text']} — " + ", ".join(c["sources"]) for c in PLAN["claims"]]
(PROJ / "render" / "storyboard.md").write_text("\n".join(sb) + "\n")
(PROJ / "render" / "timeline.json").write_text(json.dumps({"total_s": TOTAL, "shots": TIMELINE,
    "lines": [{k: v for k, v in p.items() if k != "words"} for p in PLACED]}, indent=2))
for row, shot in zip(TIMELINE, SHOTS):
    who = ", ".join(CHARS[l["speaker"]]["name"] for l in shot["lines"])
    long = row["dur"] > SHOT_BUDGET + 0.6 and not shot.get("allow_long") and shot.get("recipe") not in DRAWN_RECIPES | {"stock"}
    print(f"  {row['shot_id']}: {row['start']:6.2f}s  +{row['dur']:.2f}s  [{row['mood']}]  {who}"
          + (f"  ⚠️ longer than one clip ({SHOT_BUDGET:.1f}s): the last frame will be held; split this shot" if long else ""))
# short-form limit: two-friend videos (MAX_VIDEO_DURATION_SECONDS) or a plan's own "max_duration_s"
MAX_DURATION = PLAN.get("max_duration_s") or (globals().get("MAX_VIDEO_DURATION_SECONDS")
                                               if PLAN.get("story_mode") == "two_friends_cinematic" else None)
if MAX_DURATION and TOTAL > MAX_DURATION:
    # never sped up or cut to fit: the script must be shortened (fact-preserving revision), then Run all again
    print(f"⚠️ {TOTAL:.1f}s is longer than the {MAX_DURATION}s limit: shorten the script (remove or merge a beat, "
          "keep every sourced fact) — the final check will not approve it as is.")
    log(f"video {TOTAL:.1f}s exceeds the {MAX_DURATION}s limit", "WARN")
print(f"Total length: {TOTAL:.1f}s, {len(cues)} subtitle cues "
      f"({word_timed}/{len(PLACED)} lines timed word by word)")


In [ ]:
BUDGET.check('diagrams')
# ==== 7b. Diagram shots: system-design animations drawn here (no AI, no GPU) ====
# Boxes for services / databases / queues, arrows, requests travelling between them, database rows changing state
# and code notes. Text is exact and sharp, and a shot takes seconds. Layout: 4 columns x 4 rows of boxes on the
# left, a panel (table / code) on the right. Frames are streamed straight into ffmpeg: no video is held in RAM.
# Finished diagrams are cached on Drive and rendered in the background while the GPU makes the AI clips.
import threading
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageDraw, ImageFont
DIAGRAM_VERSION = 1
DW, DH = 1920, 1080
KIND_COLORS = {"client": (59, 130, 246), "service": (139, 92, 246), "store": (16, 185, 129),
               "queue": (245, 158, 11), "cache": (239, 68, 68), "external": (120, 128, 145)}
BG, BOX_W, BOX_H = (13, 17, 33), 250, 86
PACKET, PULSE, TRAVEL = (255, 214, 102), (255, 214, 102), 0.9

def dfont(size, mono=False, bold=True):
    names = (["DejaVuSansMono-Bold.ttf", "DejaVuSansMono.ttf"] if mono else ["DejaVuSans-Bold.ttf", "DejaVuSans.ttf"])
    for n in (names if bold else names[::-1]):
        p = drawkit.font_file(n)                     # system DejaVu, else matplotlib's copy (always on Colab)
        if p:
            return ImageFont.truetype(p, size)
    return ImageFont.load_default(size=size)
F_NODE, F_SUB, F_PKT, F_TAB, F_TABH, F_NOTE = dfont(26), dfont(19, bold=False), dfont(22, mono=True), \
    dfont(23, mono=True, bold=False), dfont(23, mono=True), dfont(22, mono=True, bold=False)

def mix(c, k):                                       # blend a colour toward the background (k=0 colour, 1 = bg)
    return tuple(int(a * (1 - k) + b * k) for a, b in zip(c, BG))

_BG_IMG = None
def background():
    global _BG_IMG
    if _BG_IMG is None:
        y = np.linspace(0, 1, DH)[:, None, None]
        top, bot = np.array([18, 24, 48]), np.array([8, 10, 22])
        _BG_IMG = Image.fromarray(np.broadcast_to(top * (1 - y) + bot * y, (DH, DW, 3)).astype(np.uint8))
    return _BG_IMG

def border_point(a, b, pad=8):
    """Where the line from box centre a toward b leaves box a."""
    (ax, ay), (bx, by) = a, b
    dx, dy = bx - ax, by - ay
    if dx == 0 and dy == 0:
        return a
    t = min((BOX_W / 2 + pad) / abs(dx) if dx else 1e9, (BOX_H / 2 + pad) / abs(dy) if dy else 1e9)
    return ax + dx * t, ay + dy * t

def arrow(d, p, q, color, width):
    d.line([p, q], fill=color, width=width)
    ang = math.atan2(q[1] - p[1], q[0] - p[0])
    h = 10 + width * 2
    d.polygon([q, (q[0] - h * math.cos(ang - 0.4), q[1] - h * math.sin(ang - 0.4)),
               (q[0] - h * math.cos(ang + 0.4), q[1] - h * math.sin(ang + 0.4))], fill=color)

def draw_node(d, spec, state):
    x, y = spec["x"], spec["y"]
    col = KIND_COLORS[spec["kind"]]
    k = {"ghost": 0.88, "dim": 0.62, "normal": 0.0, "focus": 0.0}[state]
    body, edge = mix(mix(col, 0.72), k), mix(col, k)
    box = [x - BOX_W // 2, y - BOX_H // 2, x + BOX_W // 2, y + BOX_H // 2]
    d.rounded_rectangle(box, radius=16, fill=body, outline=edge, width=6 if state == "focus" else 3)
    if spec["kind"] in ("store", "cache"):               # a little database "cylinder" mark
        d.ellipse([box[0] + 12, box[1] + 10, box[0] + 36, box[1] + 20], outline=edge, width=2)
    txt = mix((255, 255, 255), k)
    sub = spec.get("sub")
    d.text((x, y - (11 if sub else 0)), spec["label"], font=F_NODE, fill=txt, anchor="mm")
    if sub:
        d.text((x, y + 22), sub, font=F_SUB, fill=mix((200, 210, 230), k), anchor="mm")

def table_img(tab, values, hot):
    """The right-hand table panel: title, header, rows; `hot` = cells changed in the last moment."""
    w, rh = 620, 46
    cols = tab["columns"]
    h = 58 + rh * (1 + len(values))
    im = Image.new("RGB", (w, h), (22, 28, 52)); d = ImageDraw.Draw(im)
    d.rounded_rectangle([0, 0, w - 1, h - 1], radius=14, outline=(16, 185, 129), width=3)
    d.text((18, 30), "TABLE  " + tab["name"], font=F_TABH, fill=(110, 231, 183), anchor="lm")
    cw = (w - 24) // len(cols)
    for j, c in enumerate(cols):
        d.text((14 + j * cw, 58 + rh // 2), c, font=F_TABH, fill=(180, 190, 215), anchor="lm")
    d.line([12, 58 + rh, w - 12, 58 + rh], fill=(60, 70, 100), width=2)
    for i, row in enumerate(values):
        yy = 58 + rh * (i + 1)
        for j, v in enumerate(row):
            if (i, j) in hot:
                d.rounded_rectangle([10 + j * cw, yy + 5, 6 + (j + 1) * cw, yy + rh - 5], radius=8, fill=(120, 95, 20))
            d.text((14 + j * cw, yy + rh // 2), str(v), font=F_TAB,
                   fill=(255, 230, 150) if (i, j) in hot else (230, 235, 245), anchor="lm")
    return im

def note_img(lines):
    w, lh = 620, 34
    h = 52 + lh * len(lines)
    im = Image.new("RGB", (w, h), (16, 20, 36)); d = ImageDraw.Draw(im)
    d.rounded_rectangle([0, 0, w - 1, h - 1], radius=14, outline=(139, 92, 246), width=3)
    d.text((18, 26), "CODE", font=F_TABH, fill=(196, 181, 253), anchor="lm")
    for i, l in enumerate(lines):
        d.text((18, 52 + lh * i + lh // 2), l, font=F_NOTE, fill=(230, 235, 245), anchor="lm")
    return im

def diagram_timing(dg, dur):
    """Start time of each flow, table change and node reveal (spread over the shot unless given)."""
    flows, changes = dg.get("flows", []), (dg.get("table") or {}).get("changes", [])
    span0, span1 = 0.4, max(dur - 1.2, 0.8)
    ft = [f.get("at", span0 + (span1 - span0) * i / max(len(flows), 1)) for i, f in enumerate(flows)]
    ct = [c.get("at", span0 + 0.6 + (span1 - span0) * (i + 0.5) / max(len(changes), 1)) for i, c in enumerate(changes)]
    return ft, ct

def render_diagram(shot, frames, out):
    arch, dg = PLAN["architecture"], shot["diagram"]
    nodes, dur = arch["nodes"], frames / FPS
    show = set(nodes) if dg.get("show", "all") == "all" else set(dg["show"])
    focus = set(dg.get("focus", []))
    flows = dg.get("flows", [])
    ft, ct = diagram_timing(dg, dur)
    reveal = [n for n in nodes if n in show] if dg.get("reveal") else []
    rt = {n: 0.2 + i * min(0.5, dur * 0.6 / max(len(reveal), 1)) for i, n in enumerate(reveal)}
    tab = dg.get("table")
    note = note_img(dg["note"]) if dg.get("note") else None
    base_cache = {}

    def base(t):
        shown = {n for n in show if n not in rt or t >= rt[n]}
        tstate = sum(t >= c for c in ct)
        hot = frozenset((c["row"], tab["columns"].index(c["col"])) for c, c0 in zip(tab["changes"], ct)
                        if c0 <= t < c0 + 1.2) if tab else frozenset()
        key = (frozenset(shown), tstate, hot)
        if key not in base_cache:
            im = background().copy(); d = ImageDraw.Draw(im)
            for a, b in arch["edges"]:
                if a in shown and b in shown:
                    p, q = border_point((nodes[a]["x"], nodes[a]["y"]), (nodes[b]["x"], nodes[b]["y"])), \
                           border_point((nodes[b]["x"], nodes[b]["y"]), (nodes[a]["x"], nodes[a]["y"]))
                    lit = not focus or (a in focus and b in focus)
                    arrow(d, p, q, (95, 110, 150) if lit else (45, 52, 78), 3)
            for n, spec in nodes.items():
                st = "ghost" if n not in shown else ("focus" if n in focus else ("dim" if focus else "normal"))
                draw_node(d, spec, st)
            y0 = 180
            if tab:
                vals = [list(r) for r in tab["rows"]]
                for c in tab.get("changes", [])[:tstate]:
                    vals[c["row"]][tab["columns"].index(c["col"])] = c["value"]
                ti = table_img(tab, vals, hot); im.paste(ti, (1270, y0)); y0 += ti.height + 30
            if note is not None:
                im.paste(note, (1270, y0))
            if len(base_cache) > 64:
                base_cache.clear()
            base_cache[key] = im
        return base_cache[key]

    cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",
           "-s", f"{DW}x{DH}", "-r", str(FPS), "-i", "-", *X264_DRAWN, str(out)]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
    try:
        for i in range(frames):
            t = i / FPS
            im = base(t).copy(); d = ImageDraw.Draw(im)
            for f, t0 in zip(flows, ft):
                a, b = nodes[f["from"]], nodes[f["to"]]
                p, q = border_point((a["x"], a["y"]), (b["x"], b["y"])), border_point((b["x"], b["y"]), (a["x"], a["y"]))
                if t0 <= t < t0 + TRAVEL:                  # the request travelling along the arrow
                    u = (t - t0) / TRAVEL; u = u * u * (3 - 2 * u)
                    arrow(d, p, q, (255, 214, 102), 4)
                    cx, cy = p[0] + (q[0] - p[0]) * u, p[1] + (q[1] - p[1]) * u
                    d.ellipse([cx - 12, cy - 12, cx + 12, cy + 12], fill=PACKET, outline=(255, 255, 255), width=2)
                    if f.get("label"):
                        bb = d.textbbox((cx, cy - 34), f["label"], font=F_PKT, anchor="mm")
                        d.rounded_rectangle([bb[0] - 10, bb[1] - 6, bb[2] + 10, bb[3] + 6], radius=10,
                                            fill=(30, 34, 58), outline=PACKET, width=2)
                        d.text((cx, cy - 34), f["label"], font=F_PKT, fill=(255, 236, 170), anchor="mm")
                elif t0 + TRAVEL <= t < t0 + TRAVEL + 0.5:  # the receiving box lights up
                    x, y = b["x"], b["y"]
                    d.rounded_rectangle([x - BOX_W // 2 - 6, y - BOX_H // 2 - 6, x + BOX_W // 2 + 6, y + BOX_H // 2 + 6],
                                        radius=20, outline=PULSE, width=5)
            proc.stdin.write(im.tobytes())
        proc.stdin.close()
        err = proc.stderr.read().decode(errors="replace")
        if proc.wait():
            raise RuntimeError("diagram encoding failed: " + err[-1500:])
    finally:
        if proc.poll() is None:
            proc.kill()
        proc.wait(timeout=10)
        for stream in (proc.stdin, proc.stderr):
            if stream is not None and not stream.closed:
                try:
                    stream.close()
                except OSError:
                    pass

# ---- every drawn scene: diagram (above), and app screens / journey / cards from engine/lib/drawkit.py ----
DRAWN = globals().get("DRAWN_RECIPES", {"diagram"})
DRAWKIT_VERSION = 3                                   # bump when drawkit's look changes, to redraw every scene
CHARANIM_VERSION = 4                                  # bump when the friends' drawings / demos change

# ---- storybook plans: the approved (or provisional) character and environment asset versions, once per session ----
STORYBOOK = PLAN.get("visual_style_version") == "storybook_v1"
STORYBOOK_VERSION = 1                                 # bump when the storybook renderer's drawing changes
STORYBOOK_REPORT = {}
if STORYBOOK:
    import storybook as SB, envfactory
    os.environ.setdefault("STORYBOOK_ASSETS", str(Path(DRIVE_ROOT) / "_assets"))   # the library lives on Drive
    _t0 = time.time()
    for _cid, _a in PLAN["character_assets"].items():
        _vdir = SB.library_root() / "characters" / _cid / _a["version"]
        if _a.get("status") == "approved":
            if not (_vdir / "manifest.json").exists():
                raise RuntimeError(f"approved character asset {_cid} {_a['version']} is not on Drive ({_vdir}): copy it "
                                   "from the Character Studio output; the video never falls back to simple shapes")
            SB.load(_cid, _a["version"], expect_sha=_a.get("manifest_sha256"), expect_content=_a.get("content_sha256"))
        else:                                         # provisional, code-drawn: prepared in the plan cell, cached on Drive
            SB.ensure(_cid, _a["version"], log=log)
        _m = json.loads((_vdir / "manifest.json").read_text())
        STORYBOOK_REPORT[_cid] = {"version": _a["version"], "status": _m["status"], "art_version": _m["art_version"],
                                  "verify": SB.verify(_vdir), "layers": len(_m["layers"])}
    for _e, _ver in PLAN["environments"].items():
        envfactory.ensure(_e, _ver, SB.library_root(), log=log)
        STORYBOOK_REPORT[f"env:{_e}"] = {"version": _ver, "verify": envfactory.verify(SB.library_root() / "environments" / _e / _ver)}
    log(f"storybook assets ready in {time.time() - _t0:.1f}s: {json.dumps(STORYBOOK_REPORT)[:1500]}")
    print("Storybook characters: " + ", ".join(f"{k} {v['version']} ({v.get('status', 'pack')})" for k, v in STORYBOOK_REPORT.items()))
    DEVICE_REPORT["storybook characters + environments"] = "CPU (Pillow), assets cached on Drive"
STORYBOOK_MOUTH = {}                                  # shot -> mouth check (listener mouths, speech coverage)

# ---- two-friend scenes: the scene's VISUAL timeline, cut inside the dialogue (the audio is already placed) ----
STORY_SPECS, STORY_ROWS = {}, {}                      # shot -> drawable spec / [visual, intent, medium, start, dur]
_TL = {r["shot_id"]: r for r in TIMELINE}

def speech_energy(shot, frames):
    """Per character, per video frame: how loud that character is speaking (0..1). Each line is normalised by its
    own loud parts, so a quiet voice still opens the mouth; silence (pauses, the other speaker) is 0."""
    t0, out = _TL[shot["shot_id"]]["start"], {}
    placed = {p["id"]: p for p in PLACED}
    for l in shot["lines"]:
        p = placed.get(l["id"])
        if p is None:
            continue
        w, sr_ = sf.read(str(PROJ / "narration" / f"{l['id']}.wav"), dtype="float32")
        w = w.mean(1) if w.ndim > 1 else w
        n = int(len(w) / (sr_ / FPS))
        rms = np.array([np.sqrt(np.mean(w[int(i * sr_ / FPS):int((i + 1) * sr_ / FPS)] ** 2) + 1e-12) for i in range(n)])
        ref = np.percentile(rms, 90) if n else 1.0
        e = np.clip((rms / max(ref, 1e-6) - 0.08) / 0.92, 0, 1)
        e = np.maximum(e, np.r_[0, e[:-1]] * 0.6)        # a little hold: no flicker between syllables
        arr = out.setdefault(l["speaker"], np.zeros(frames, np.float32))
        a = int(round((p["start"] - t0) * FPS))
        k = max(0, min(len(e), frames - a))
        if k > 0 and a >= 0:
            arr[a:a + k] = np.maximum(arr[a:a + k], e[:k])
    return out

def speech_visemes(shot, frames):
    """Per character, per video frame: the mouth shape from that speaker's own audio (storybook plans). Silence and
    the other speaker's lines are "rest" — a listener's mouth never moves with someone else's voice."""
    t0, out = _TL[shot["shot_id"]]["start"], {}
    placed = {p["id"]: p for p in PLACED}
    for l in shot["lines"]:
        p = placed.get(l["id"])
        if p is None:
            continue
        w, sr_ = sf.read(str(PROJ / "narration" / f"{l['id']}.wav"), dtype="float32")
        vv, _ = SB.visemes_from_audio(w, sr_, FPS)
        arr = out.setdefault(l["speaker"], ["rest"] * frames)
        a = int(round((p["start"] - t0) * FPS))
        for i, v in enumerate(vv):
            if 0 <= a + i < frames and v != "rest":
                arr[a + i] = v
    return out

def story_spec(shot, frames):
    sid = shot["shot_id"]
    if sid in STORY_SPECS and STORY_SPECS[sid][0] == frames:
        return STORY_SPECS[sid][1]
    t0, placed = _TL[sid]["start"], {p["id"]: p for p in PLACED}
    vis = shot["story"]["visuals"]
    starts = []
    for v in vis:                                      # a visual starts at a point INSIDE a line
        p = placed[shot["lines"][v["start"]["line"]]["id"]]
        starts.append((p["start"] - t0) + v["start"]["at"] * (p["end"] - p["start"]))
    starts[0] = 0.0
    fr = [min(max(int(round(x * FPS)), 0), frames) for x in starts] + [frames]
    for i in range(1, len(fr)):
        fr[i] = max(fr[i], fr[i - 1])
    energy = speech_energy(shot, frames) if globals().get("LIP_SYNC", "energy") == "energy" else {}
    vis_all = speech_visemes(shot, frames) if STORYBOOK and energy else {}
    for c, vv in vis_all.items():                       # one silence gate for both tracks: the loudness decides
        e = energy.get(c, np.zeros(frames))
        vis_all[c] = [v if e[i] > 0.06 else "rest" for i, v in enumerate(vv)]
    visuals, rows = [], []
    for v, a, b in zip(vis, fr, fr[1:]):
        if b <= a:
            log(f"{sid}: visual {v['id']} gets no time (starts too close to the next one); skipped", "WARN")
            continue
        spec = dict(v[v["recipe"]])
        if v["recipe"] == "character":
            spec.update(_t0=round(t0 + a / FPS, 3), _dur=round((b - a) / FPS, 3),
                        _speech={c: [round(float(x), 2) for x in e[a:b]] for c, e in energy.items()})
            if vis_all:
                spec["_visemes"] = {c: list(vv[a:b]) for c, vv in vis_all.items()}
        visuals.append({"id": v["id"], "recipe": v["recipe"], v["recipe"]: spec, "frames": b - a})
        rows.append({"shot": sid, "visual": v["id"], "intent": v.get("intent", "?"), "medium": v.get("medium", "?"),
                     "recipe": v["recipe"], "start": round(t0 + a / FPS, 2), "dur": round((b - a) / FPS, 2),
                     "label": v.get("label", ""), "description": v.get("description", "")})
    if vis_all:                                        # mouth check: only the speaker's mouth moves
        rep_ = {}
        for c, vv in vis_all.items():
            e = energy.get(c, np.zeros(frames))
            talk = [i for i in range(frames) if e[i] > 0.06]
            rep_[c] = {"speech_frames": len(talk), "open_in_speech": sum(vv[i] not in ("rest", "closed") for i in talk),
                       "open_in_silence": sum(1 for i in range(frames) if e[i] <= 0.06 and vv[i] not in ("rest",))}
        STORYBOOK_MOUTH[sid] = rep_
    STORY_SPECS[sid], STORY_ROWS[sid] = (frames, {"visuals": visuals}), rows
    return {"visuals": visuals}

# drawn frames are encoded on the GPU (NVENC) when the session has one: the CPU stays free for drawing
X264_DRAWN = ["-threads", str(CPU_THREADS), "-c:v", "libx264", "-preset", "veryfast", "-crf", "16", "-tune", "animation", "-pix_fmt", "yuv420p"]
NVENC_DRAWN = ["-c:v", "h264_nvenc", "-preset", "p4", "-rc", "vbr", "-cq", "16", "-b:v", "0", "-pix_fmt", "yuv420p"]
DRAWN_VENC = NVENC_DRAWN if globals().get("NVENC_OK") else X264_DRAWN
log(f"drawn scenes are encoded with {'NVENC (GPU)' if DRAWN_VENC is NVENC_DRAWN else 'x264 (CPU)'}")
DEVICE_REPORT["drawn scenes: drawing (Pillow)"] = "CPU (Pillow has no GPU path)"
DEVICE_REPORT["drawn scenes: encoding"] = "GPU (NVENC)" if DRAWN_VENC is NVENC_DRAWN else "CPU (x264)"

def render_drawn(shot, frames, out, venc=None):
    kind = shot.get("recipe")
    if kind == "diagram":
        return render_diagram(shot, frames, out)
    venc = venc or DRAWN_VENC
    try:
        return _render_drawn(shot, frames, out, venc)
    except colab_budget.BudgetExceeded:
        raise
    except RuntimeError:
        if venc is X264_DRAWN:
            raise
        log_exc(f"{shot['shot_id']}: GPU encoder failed for a drawn scene, encoding it on the CPU")
        return _render_drawn(shot, frames, out, X264_DRAWN)

def _render_drawn(shot, frames, out, venc):
    kind = shot.get("recipe")
    cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",
           "-s", f"{W}x{H}", "-r", str(FPS), "-i", "-", *venc, str(out)]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
    try:
        spec = story_spec(shot, frames) if kind == "story" else \
            dict(shot[kind], _has_label=bool(shot.get("label")))      # layout keeps clear of the step label
        for frame in drawkit.render(kind, spec, PLAN, frames, FPS, W, H):
            BUDGET.check("drawing frames")
            proc.stdin.write(frame.tobytes())
        proc.stdin.close()
        err = proc.stderr.read().decode(errors="replace")
        if proc.wait():
            raise RuntimeError(f"{kind} encoding failed: " + err[-1500:])
    finally:
        if proc.poll() is None:
            proc.kill()
        proc.wait(timeout=10)
        for stream in (proc.stdin, proc.stderr):
            if stream is not None and not stream.closed:
                try:
                    stream.close()
                except OSError:
                    pass

DIAGRAM_DIR = PROJ / "diagrams"; DIAGRAM_DIR.mkdir(exist_ok=True)
DIAGRAM_FILES = {}

def diagram_path(shot, frames):
    kind = shot.get("recipe")
    shared_bits = PLAN.get("architecture") if kind == "diagram" else [PLAN.get("journey"), PLAN.get("look")]
    if kind in ("story", "character", "demo"):         # the friends' look + this scene's cut points and mouths
        shared_bits = [shared_bits, PLAN.get("bible"), CHARANIM_VERSION]
        if STORYBOOK:                                  # + the exact asset versions and the renderer version
            shared_bits = shared_bits + [PLAN.get("character_assets"), PLAN.get("environments"), STORYBOOK_VERSION]
    body = story_spec(shot, frames) if kind == "story" else shot[kind]
    key = hashlib.sha1(json.dumps([DIAGRAM_VERSION, DRAWKIT_VERSION, kind, shared_bits, body, bool(shot.get("label")),
                                   frames, FPS, W, H],
                                  sort_keys=True).encode()).hexdigest()[:12]
    return DIAGRAM_DIR / f"{shot['shot_id']}_{key}.mp4"

def make_diagram(shot, row):
    out = diagram_path(shot, row["frames"])
    if out.exists():
        return out, True
    t0, tmp = time.time(), WORK / f"diagram_{shot['shot_id']}.mp4"
    render_drawn(shot, row["frames"], tmp)
    shutil.copy(tmp, out)
    for old in DIAGRAM_DIR.glob(f"{shot['shot_id']}_*.mp4"):
        if old != out:
            old.unlink()
    log(f"{shot['shot_id']}: {shot.get('recipe')} drawn in {time.time() - t0:.1f}s ({row['frames']} frames)")
    return out, False

DIAGRAM_SHOTS = [(s, r) for s, r in zip(SHOTS, TIMELINE) if s.get("recipe") in DRAWN]
def _diagram_job(shot, row):
    try:
        DIAGRAM_FILES[shot["shot_id"]] = make_diagram(shot, row)[0]
    except Exception:
        log_exc(f"{shot['shot_id']}: diagram failed (it is retried in the edit step)")

DIAGRAM_POOL, DIAGRAM_JOBS = ThreadPoolExecutor(max_workers=1), []
for s, r in DIAGRAM_SHOTS:
    DIAGRAM_JOBS.append(DIAGRAM_POOL.submit(_diagram_job, s, r))

def wait_for_diagrams():
    t0 = time.time()
    for j in DIAGRAM_JOBS:
        j.result()
    for s, r in DIAGRAM_SHOTS:                          # anything that failed in the background: try once more here
        if s["shot_id"] not in DIAGRAM_FILES:
            DIAGRAM_FILES[s["shot_id"]] = make_diagram(s, r)[0]
    if DIAGRAM_SHOTS:
        log(f"waited {time.time() - t0:.1f}s for {len(DIAGRAM_SHOTS)} diagram shots")
print(f"{len(DIAGRAM_SHOTS)} drawn scenes (app screens, journeys, cards, diagrams) are being drawn in the "
      "background (no GPU needed)." if DIAGRAM_SHOTS else "No drawn scenes in this plan.")


In [ ]:
BUDGET.check('keyframes')
# ==== 8. Keyframe images: only made when needed ====
# L4/A100 (image-to-video): every shot starts from a keyframe picture, so they are made now.
# T4 (text-to-video): the video model doesn't use them, so they are skipped (saves the ~7 GB SDXL download, its RAM
# and ~10 min); a keyframe is only made later for a shot whose video failed, as its fallback picture.
from PIL import Image, ImageDraw, ImageFilter, ImageOps

def procedural_space(seed, w=1344, h=768):
    """Fallback image when no GPU is available: stars + soft nebula."""
    rng = np.random.default_rng(seed)
    img = np.zeros((h, w, 3), np.float32)
    for _ in range(6):
        cx, cy, r = rng.uniform(0, w), rng.uniform(0, h), rng.uniform(120, 380)
        col = rng.uniform(0.1, 0.6, 3)
        yy, xx = np.ogrid[:h, :w]
        img += col * np.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * r * r)))[..., None] * 0.5
    ys, xs = rng.integers(0, h, 1500), rng.integers(0, w, 1500)
    img[ys, xs] = rng.uniform(0.5, 1.0, (1500, 1))
    return Image.fromarray((np.clip(img, 0, 1) * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(0.6))

def load_image_pipe():
    global IMAGE_MODEL
    try:
        import torch
        from diffusers import DiffusionPipeline
    except Exception as e:
        log_exc("diffusers import failed")
        raise RuntimeError("Could not load the image library (diffusers). Click Runtime → Restart session, "
                           "then Runtime → Run all. If it happens again, send this error to Claude:\n" + repr(e)[:500])
    if IMAGE_MODEL == "z-image-turbo":
        try:
            p = DiffusionPipeline.from_pretrained("Tongyi-MAI/Z-Image-Turbo", torch_dtype=torch.bfloat16)
            p.enable_model_cpu_offload()
            return p
        except Exception as e:
            log_exc("Z-Image-Turbo could not load")
            print("Z-Image-Turbo could not load:", repr(e)[:300], "\n→ falling back to SDXL")
            IMAGE_MODEL = "sdxl"
    kw = dict(torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    try:                                           # load straight onto the GPU (keeps system RAM free on a T4)
        p = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", device_map="cuda", **kw)
    except Exception:
        log_exc("SDXL device_map='cuda' load failed, loading normally")
        p = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", **kw)
    return p.to("cuda")

def needs_image(shot):
    out = PROJ / "keyframes" / f"{shot['shot_id']}.png"
    sc = read_sidecar(out)
    return (not out.exists() or sc is None or sc.get("prompt") != f"{shot['image_prompt']}, {STYLE}"
            or sc.get("seed") != shot["seed"] or (sc.get("model") == "procedural" and IMAGE_MODEL != "procedural"))

def make_keyframes(shots):
    todo = [s for s in shots if s.get("image_prompt") and needs_image(s)]
    if not todo:
        return
    pipe = load_image_pipe() if IMAGE_MODEL != "procedural" else None
    for shot in todo:
        out = PROJ / "keyframes" / f"{shot['shot_id']}.png"
        prompt = f"{shot['image_prompt']}, {STYLE}"
        t0, used = time.time(), IMAGE_MODEL
        try:
            if pipe is None:
                raise RuntimeError("no GPU image model")
            import torch
            g = torch.Generator("cpu").manual_seed(shot["seed"])
            if IMAGE_MODEL == "z-image-turbo":
                img = pipe(prompt=prompt, height=768, width=1344, num_inference_steps=9,
                           guidance_scale=0.0, generator=g).images[0]
            else:
                img = pipe(prompt=prompt, negative_prompt=NEG, height=768, width=1344,
                           num_inference_steps=30, guidance_scale=6.0, generator=g).images[0]
        except Exception as e:
            log_exc(f"{shot['shot_id']}: image generation failed")
            if pipe is not None:
                print(f"  {shot['shot_id']}: image generation failed ({repr(e)[:200]}), using procedural fallback")
            img, used = procedural_space(shot["seed"]), "procedural"
        img.save(out)
        sidecar(out, model=used, prompt=prompt, seed=shot["seed"], seconds=round(time.time() - t0, 1))
        print(f"  {shot['shot_id']}: {used} ({time.time() - t0:.0f}s)")
    del pipe; free_gpu()
    print("Keyframes ready."); ram()

if VIDEO_MODE == "t2v":
    print("Text-to-video mode: keyframe pictures skipped (only made later as fallbacks for shots without video).")
else:
    make_keyframes(SHOTS)


In [ ]:
BUDGET.check('editlib')
# ==== 8b. Editing: each shot is cut to its length, slowed if needed, colour graded and titled ====
# A finished shot is cached on Drive (segments/), keyed by everything that shapes it, so re-runs only re-edit
# shots that changed. While the GPU makes clips, finished shots are edited in the background on the CPU.
from PIL import ImageFont
from concurrent.futures import ThreadPoolExecutor
GRADES = {"neutral":  "eq=contrast=1.0",
          "cool":     "colorbalance=bs=0.06:bm=0.03:bh=0.02",
          "contrast": "eq=contrast=1.08:saturation=1.05",
          "hot":      "eq=contrast=1.12:saturation=1.15,colorbalance=rh=0.05:rm=0.04",
          "deep":     "eq=contrast=1.1:brightness=-0.03:saturation=0.9"}

def zoompan(move, n, amt):
    N = max(n - 1, 1)
    cx, cy = "iw/2-(iw/zoom/2)", "ih/2-(ih/zoom/2)"
    z, x, y = {"push_in":   (f"1+{amt}*on/{N}", cx, cy),
               "pull_out":  (f"{1 + amt}-{amt}*on/{N}", cx, cy),
               "pan_right": (f"{1 + amt}", f"(iw-iw/zoom)*on/{N}", cy),
               "pan_left":  (f"{1 + amt}", f"(iw-iw/zoom)*(1-on/{N})", cy),
               "tilt_up":   (f"{1 + amt}", cx, f"(ih-ih/zoom)*(1-on/{N})"),
               "tilt_down": (f"{1 + amt}", cx, f"(ih-ih/zoom)*on/{N}")}.get(move, ("1.0", "0", "0"))
    return f"zoompan=z='{z}':x='{x}':y='{y}':d={n}:s={W}x{H}:fps={FPS}"

def look(shot, dur, d=0.6):
    f = [GRADES[shot.get("grade", mood(shot)["grade"])]]
    if shot.get("flash_in"):  f.append("fade=t=in:st=0:d=0.35:color=white")
    elif shot.get("fade_in"): f.append(f"fade=t=in:st=0:d={d}")
    if shot.get("fade_out"):  f.append(f"fade=t=out:st={max(dur - d, 0):.3f}:d={d}")
    return f

def font(size):
    for p in [drawkit.font_file("DejaVuSans-Bold.ttf"), "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]:
        if p and os.path.exists(p):
            return ImageFont.truetype(p, size)
    log("no TrueType font found: titles use Pillow's default font", "WARN")
    return ImageFont.load_default(size=size)

def overlay_png(shot, kind):
    """Transparent PNG laid over the moving video: "title" (big, centred) or "label" (step badge, top left).
    Text is drawn here, never by the AI model, so it is always sharp and spelled right."""
    im = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    d = ImageDraw.Draw(im)
    if kind == "title":
        f, words, lines = font(84), shot["title"].split(), [""]
        for wd in words:                              # wrap so a long title never runs off the screen
            trial = (lines[-1] + " " + wd).strip()
            if lines[-1] and d.textlength(trial, font=f) > W - 240:
                lines.append(wd)
            else:
                lines[-1] = trial
        text = "\n".join(lines)
        box = d.multiline_textbbox((0, 0), text, font=f, spacing=18, align="center")
        xy = ((W - (box[2] - box[0])) // 2 - box[0], (H - (box[3] - box[1])) // 2 - box[1])
        d.multiline_text(xy, text, font=f, fill=(245, 245, 250, 255), spacing=18, align="center",
                         stroke_width=3, stroke_fill=(0, 0, 0, 230))
    elif kind == "credit":                            # licensed image credit (title, author, source, licence), top right
        f = font(max(18, H // 50))
        text = shot["stock"]["credit"]
        while d.textlength(text, font=f) > W * 0.6 and len(text) > 20:
            text = text[:-4].rstrip() + "…"
        tw = d.textlength(text, font=f)
        x, y, pad = W - tw - 40, 28, 10
        d.rounded_rectangle([x - pad, y - pad // 2, x + tw + pad, y + f.size + pad], radius=8, fill=(0, 0, 0, 140))
        d.text((x, y), text, font=f, fill=(235, 235, 240, 255))
    else:
        f, x, y, pad = font(44), 60, 56, 22
        box = d.textbbox((0, 0), shot["label"], font=f)
        tw, th = box[2] - box[0], box[3] - box[1]
        d.rounded_rectangle([x, y, x + tw + 2 * pad + 16, y + th + 2 * pad], radius=18, fill=(14, 14, 20, 200))
        d.rounded_rectangle([x, y, x + 10, y + th + 2 * pad], radius=5, fill=(232, 72, 85, 255))
        d.text((x + 16 + pad - box[0], y + pad - box[1]), shot["label"], font=f, fill=(255, 255, 255, 255))
    out = WORK / f"{shot['shot_id']}_{kind}.png"; im.save(out); return out

def still_frame(shot):
    kf = PROJ / "keyframes" / f"{shot['shot_id']}.png"
    return kf if kf.exists() else None

SEG_DIR = PROJ / "segments"; SEG_DIR.mkdir(exist_ok=True)
if not BURN_SUBS:                    # segments are joined without re-encoding: they need the final quality
    SEG_VENC = VENC
elif NVENC_OK:                       # segments are re-encoded once more with the subtitles: fast, near-lossless
    SEG_VENC = ["-c:v", "h264_nvenc", "-preset", "p2", "-rc", "vbr", "-cq", "16", "-b:v", "0", "-pix_fmt", "yuv420p", "-r", FPS]
else:
    SEG_VENC = ["-c:v", "libx264", "-preset", "veryfast", "-crf", "16", "-pix_fmt", "yuv420p", "-r", FPS]

HOLDS = {}                           # shot -> seconds its last frame is frozen (should be ~0)
REVIEW = {}                          # shot -> why a person should look at it before publishing
def segment_recipe(shot, row):
    """ffmpeg inputs + filter graph for one shot, and what it depends on (for the cache key)."""
    sid, n, dur = shot["shot_id"], row["frames"], row["dur"]
    info = clip_info(shot) if shot.get("motion_prompt") else None
    inputs, k = [], 0
    fb = None
    if shot.get("motion_prompt") and not info and "logic_shot" in globals() and logic_shot(shot):
        # scene-state shot without an accepted/approved clip: never used silently; fallback + listed for review
        REVIEW[sid] = logic_usable(shot)[1]
        if shot.get("fallback"):
            info = {"use_fallback": True}
    if info and info.get("use_fallback") and shot.get("fallback"):
        # the AI clip failed the continuity / relevance check even after a remake: use the plan's drawn fallback
        fb = {"shot_id": f"{sid}_fallback", **shot["fallback"]}
        DIAGRAM_FILES[sid] = make_diagram(fb, row)[0]
        REVIEW.setdefault(sid, "AI clip didn't show the right scene/characters; drawn fallback used")
    if shot.get("recipe") == "stock":                     # licensed real image (Commons), fitted + slow camera move
        src = PROJ / "assets" / shot["stock"]["file"]
        inputs = ["-i", src]
        amt = shot.get("move_amount", mood(shot)["move_amount"])
        # fitted inside the frame over a blurred copy of itself: nothing of the picture is cropped away
        fc = ("[0:v]format=rgb24,split[bg][fg];"
              f"[bg]scale={2 * W}:{2 * H}:force_original_aspect_ratio=increase,crop={2 * W}:{2 * H},boxblur=30:3,"
              "eq=brightness=-0.12[b];"
              f"[fg]scale={2 * W}:{2 * H}:force_original_aspect_ratio=decrease[f];"
              "[b][f]overlay=(W-w)/2:(H-h)/2," + ",".join([zoompan(shot.get("move", "push_in"), n, amt),
                                                          *look(shot, dur), "format=yuv420p"]) + "[v]")
        k, how, info = 1, f"licensed image ({shot['stock'].get('license', '?')}) + camera move", None
    elif shot.get("recipe") in DRAWN or fb:               # drawn scene (no AI), exact length
        dg = DIAGRAM_FILES[sid]
        inputs = ["-i", dg]
        fc = "[0:v]" + ",".join([f"scale={W}:{H}:flags=lanczos", "tpad=stop_mode=clone:stop_duration=30",
                                 *look(shot, dur)[1:], "format=yuv420p"]) + "[v]"
        k, how = 1, (f"{fb['recipe']} fallback (AI clip rejected, review)" if fb else f"{shot['recipe']} (drawn, no AI)")
        info = None
    elif info:                                            # real AI video
        parts = [PROJ / "clips" / p for p in info["parts"]]
        for p in parts:
            inputs += ["-i", p]
        src_len = len(parts) * info["frames_per_part"] / info["fps"]
        stretch = min(max(dur / src_len, 1.0), MAX_SLOWMO)
        HOLDS[sid] = round(max(0.0, dur - src_len * stretch), 2)   # seconds the last frame is held (a freeze)
        chain = []
        if stretch > 1.01:
            chain.append(f"setpts={stretch:.4f}*PTS")
        if (stretch > 1.01 or info["fps"] != FPS) and SMOOTH_MOTION == "mci":
            chain.append(f"minterpolate=fps={FPS}:mi_mode=mci:mc_mode=aobmc:vsbmc=1")   # can warp shapes
        elif (stretch > 1.01 or info["fps"] != FPS) and SMOOTH_MOTION == "blend":
            chain.append(f"minterpolate=fps={FPS}:mi_mode=blend")                    # soft cross-fade, no warping
        else:
            chain.append(f"fps={FPS}")
        chain += [f"scale={W}:{H}:force_original_aspect_ratio=increase:flags=lanczos", f"crop={W}:{H}"]
        if SHARPEN > 0:                                   # the AI clips are small (832x480): sharpen after upscaling
            chain.append(f"unsharp=5:5:{SHARPEN}:5:5:0")
        chain += ["tpad=stop_mode=clone:stop_duration=30", *look(shot, dur), "format=yuv420p"]
        head = "".join(f"[{i}:v]" for i in range(len(parts)))
        fc = (f"{head}concat=n={len(parts)}:v=1:a=0," if len(parts) > 1 else "[0:v]") + ",".join(chain) + "[v]"
        k = len(parts)
        how = f"AI video ({info['engine']}, {len(parts)} clip{'s' if len(parts) > 1 else ''}" + \
              (f", slowed {stretch:.2f}x" if stretch > 1.01 else "") + ")"
    else:                                                 # fallback: camera move over the still
        src = still_frame(shot)
        if src is None:
            src = WORK / f"{sid}_blank.png"; Image.new("RGB", (W, H), (5, 5, 12)).save(src)
        inputs = ["-i", src]
        amt = shot.get("move_amount", mood(shot)["move_amount"])
        fc = ("[0:v]" + ",".join([f"scale={2 * W}:{2 * H}:force_original_aspect_ratio=increase", f"crop={2 * W}:{2 * H}",
                                  zoompan(shot.get("move", "push_in"), n, amt), *look(shot, dur), "format=yuv420p"]) + "[v]")
        k = 1
        how = f"⚠️ still image + camera move (no AI video for this shot)"
    vout = "[v]"
    for kind in ("label", "title", "credit"):
        if not (shot.get(kind) or (kind == "credit" and shot.get("recipe") == "stock")):
            continue
        inputs += ["-loop", "1", "-i", overlay_png(shot, kind)]
        fades = (f"fade=t=in:st=0.6:d=0.8:alpha=1,fade=t=out:st={max(dur - 1.2, 1.5):.3f}:d=0.7:alpha=1"
                 if kind == "title" else f"fade=t=in:st=0.15:d=0.4:alpha=1,fade=t=out:st={max(dur - 0.35, 0.6):.3f}:d=0.3:alpha=1")
        fc += f";[{k}:v]format=rgba,{fades}[o{k}];{vout}[o{k}]overlay=0:0:shortest=0[v{k}]"
        vout = f"[v{k}]"; k += 1
    if shot.get("recipe") == "stock":
        deps = shot["stock"]["file"] + "|" + shot["stock"]["credit"]   # file name = content hash of the image
    elif shot.get("recipe") in DRAWN or fb:
        deps = DIAGRAM_FILES[sid].name                    # the file name carries the diagram's content hash
    else:
        dep = (PROJ / "clips" / f"{sid}.json") if info else (PROJ / "keyframes" / f"{sid}.png.json")
        deps = dep.read_text() if dep.exists() else "none"
    return inputs, fc, vout, how, bool(info), deps

SEG_RECIPE_VERSION = 3               # bump when the editing code changes, to re-edit every shot
def render_segment(shot, row, background=False):
    """Edit one shot (or reuse the cached result). Returns (file, how it was made, is AI video, was cached)."""
    sid = shot["shot_id"]
    inputs, fc, vout, how, is_video, deps = segment_recipe(shot, row)
    key = hashlib.sha1(json.dumps([SEG_RECIPE_VERSION, [str(x) for x in inputs], fc, row["frames"],
                                   [str(x) for x in SEG_VENC], shot.get("title"), shot.get("label"), deps]).encode()).hexdigest()[:12]
    out = SEG_DIR / f"{sid}_{key}.mp4"
    if out.exists():
        return out, how, is_video, True
    tmp = WORK / f"seg_{sid}.mp4"
    ff(*inputs, "-filter_complex", fc, "-map", vout, "-frames:v", row["frames"], *SEG_VENC, "-an", tmp, nice=background)
    shutil.copy(tmp, out)
    for old in SEG_DIR.glob(f"{sid}_*.mp4"):
        if old != out:
            old.unlink()
    log(f"{sid}: edited ({'background' if background else 'foreground'}) -> {out.name}")
    return out, how, is_video, False

EDITOR, EDIT_JOBS = ThreadPoolExecutor(max_workers=1), []
TL_ROW = {r["shot_id"]: r for r in TIMELINE}
def edit_in_background(shot):
    def job():
        try:
            return render_segment(shot, TL_ROW[shot["shot_id"]], background=True)
        except Exception:
            log_exc(f"{shot['shot_id']}: background edit failed (it is retried in the edit step)")
    EDIT_JOBS.append(EDITOR.submit(job))


In [ ]:
BUDGET.check('logic')
# ==== 8c. Logical consistency: controlled generation + checks for shots with a scene-state ====
# Shots with a "state" block in the plan are generated with Wan2.1-VACE-1.3B, steered by a layout video drawn from the
# plan (white background, one coloured box per person/object, moving where the plan says). Every clip is then checked
# against the SAME plan: people/objects detected with OWLv2, camera motion measured, positions, direction, stops,
# hand-offs and order checked (engine/lib/motion_check.py), and each required action asked to a video-language model
# (Qwen3-VL-2B). Failures get a TARGETED repair (stronger layout, simpler layout, static camera, new seed, fewer frames)
# within REPAIR_BUDGET; whatever still fails or is uncertain goes to REVIEW and is NOT used unless approved in
# approvals.json — the editor uses the shot's drawn fallback instead. Nothing fails silently.
VACE_MID = "Wan-AI/Wan2.1-VACE-1.3B-diffusers"            # Apache-2.0; same text encoder and VAE as Wan2.1-T2V-1.3B
VACE_NATIVE = (16, 81, 832, 480)                          # fps, max frames, width, height
GEN_CODE_VERSION = 1                                      # bump when generation code changes (regenerates)
CHECK_FRAMES = 12                                         # frames sampled for detection and the VLM
STATES = globals().get("STATES", {})
SCENE_ENTS = globals().get("SCENE_ENTS", {})
APPROVALS_PATH = PROJ / "approvals.json"
APPROVALS = json.loads(APPROVALS_PATH.read_text()) if APPROVALS_PATH.exists() else {}
LOGIC_DIR = PROJ / "render" / "review"; LOGIC_DIR.mkdir(parents=True, exist_ok=True)
STATIC_SUFFIX = ", static camera, locked-off tripod shot"
_TRACKER, _JUDGE, _LOADER_ERRORS = None, None, {}
LOCAL_HF = WORK.parent / "hf_checkers"     # checkers (OWLv2 0.6 GB, Qwen3-VL-2B ~4.3 GB) stay on the session disk, not
                                           # on Drive: a free 15 GB Drive is already used by the video model caches


def logic_shot(shot):
    return VIDEO_RECIPE == "auto" and shot["shot_id"] in STATES


def logic_frames(shot):
    fps, most = VACE_NATIVE[:2]
    return frames_for(TL[shot["shot_id"]]["dur"], fps, most) if FRAMES_ON_DEMAND else most


def logic_prompt(shot, static=False):
    return clip_prompt(shot, "t2v") + (STATIC_SUFFIX if static else "")


def _hash(o):
    return hashlib.sha256(json.dumps(o, sort_keys=True, default=str).encode()).hexdigest()[:16]


def logic_gen_key(shot):
    st = STATES[shot["shot_id"]]
    return acceptance.generation_key(
        model=VACE_MID, recipe="vace_layout", steps=TURBO_STEPS if VIDEO_SPEED == "turbo" else CLIP_STEPS,
        lora=TURBO_LORA[1] if VIDEO_SPEED == "turbo" else None, frames=logic_frames(shot), size=VACE_NATIVE[2:],
        seed=shot["seed"], prompt=logic_prompt(shot), negative=NEG_V, control=_hash(st.model_dump()),
        code_version=GEN_CODE_VERSION, extra={"budget": REPAIR_BUDGET})


def logic_val_key(shot):
    st = STATES[shot["shot_id"]]
    return acceptance.validation_key(
        logic_gen_key(shot), {"state": st.model_dump(), "queries": entity_queries(st)},
        {"scene_state": scene_state.VERSION, "motion": motion_check.VERSION, "tracker": trackers.Owlv2Tracker.MODEL,
         "vlm": (vlm_check.VERSION, vlm_check.QwenJudge.MODEL) if VLM_CHECK else None, "logic": LOGIC_CHECK},
        motion_check.TH)


def entity_queries(st):
    """Detector text query per entity: plan scene_entities[*].query, else the character/prop tag, else the name."""
    q = {}
    for k in set(st.start) | {e.actor for e in st.events} | {e.obj for e in st.events if e.obj}:
        ent = (PLAN.get("scene_entities") or {}).get(k, {})
        q[k] = ent.get("query") or (CHARS.get(k, {}) if "CHARS" in globals() else {}).get("tag") or \
            (PLAN.get("props", {}).get(k, {}).get("tag")) or ent.get("name") or k
    return q


def logic_usable(shot):
    sc = read_sidecar(PROJ / "clips" / shot["shot_id"]) or {}
    ok, why = acceptance.usable(dict(sc, shot_id=shot["shot_id"]), logic_gen_key(shot), logic_val_key(shot), APPROVALS)
    if ok and not all((PROJ / "clips" / p).exists() for p in sc.get("parts", [])):
        return False, "clip file missing: regenerate"
    return ok, why


CHECKER_LOADS = {"n": 0, "s": 0.0}


def _load_checkers():
    global _TRACKER, _JUDGE
    dev = "cuda" if GPU["name"] else "cpu"
    if (LOGIC_CHECK and _TRACKER is None and "tracker" not in _LOADER_ERRORS) or \
            (VLM_CHECK and _JUDGE is None and "vlm" not in _LOADER_ERRORS):
        CHECKER_LOADS["n"] += 1
    t0 = time.time()
    if LOGIC_CHECK and _TRACKER is None and "tracker" not in _LOADER_ERRORS:
        try:
            _TRACKER = trackers.Owlv2Tracker(device=dev, cache_dir=str(LOCAL_HF))
            log(f"OWLv2 detector loaded, GPU {gpu_mem()}"); ram("(OWLv2 detector loaded)")
        except Exception:
            log_exc("OWLv2 could not load"); _LOADER_ERRORS["tracker"] = "detector could not load"
    if VLM_CHECK and _JUDGE is None and "vlm" not in _LOADER_ERRORS:
        try:
            _JUDGE = vlm_check.QwenJudge(device=dev, cache_dir=str(LOCAL_HF))
            log(f"Qwen3-VL judge loaded, GPU {gpu_mem()}"); ram("(Qwen3-VL judge loaded)")
        except Exception:
            log_exc("Qwen3-VL could not load"); _LOADER_ERRORS["vlm"] = "video-language model could not load"
    CHECKER_LOADS["s"] += time.time() - t0


def logic_checks(shot, arr, fps):
    """Every check for one clip: list of CheckResult (PASS / FAIL / UNKNOWN, with evidence)."""
    sid, st = shot["shot_id"], STATES[shot["shot_id"]]
    n = len(arr)
    idx = np.linspace(0, n - 1, min(CHECK_FRAMES, n)).round().astype(int)
    exp_all = layout_control.expected_tracks(st, n)
    exp = {k: [v[i] for i in idx] for k, v in exp_all.items()}
    res = []
    if not LOGIC_CHECK:
        return [motion_check.CheckResult("logic_check", "UNKNOWN", None, "LOGIC_CHECK is off: clip not verified")]
    _load_checkers()
    if _TRACKER is None:
        return [motion_check.CheckResult("validator_error", "UNKNOWN", None, _LOADER_ERRORS.get("tracker", "no detector"))]
    obs = _TRACKER.track(arr[idx], entity_queries(st))
    near_obs = [obs_frame for obs_frame in ([b[0] for b in (obs[k][j] for k in obs) if b] for j in range(len(idx)))]
    # camera motion is measured on the BACKGROUND: blank where subjects are detected (nearest sampled frame) AND where
    # the plan puts every person / prop / moving thing in that exact frame (the detector can miss someone; the plan
    # can't). Static scenery (anchors such as a train coach) stays in: it is what shows a real camera move.
    movers = {k for k, v in exp_all.items()
              if getattr(SCENE_ENTS.get(k), "kind", "character") != "anchor" or (v and v[0] != v[-1])}
    masks = [near_obs[int(np.argmin(np.abs(idx - i)))] + [exp_all[k][i] for k in movers if exp_all[k][i]]
             for i in range(n)]
    shift, rel = motion_check.estimate_camera_motion(arr, masks, scale=8)
    src_len = n / fps
    stretch = min(max(TL[sid]["dur"] / src_len, 1.0), MAX_SLOWMO)
    res += motion_check.run_checks(motion_check.validate_clip, st, exp, obs, shift[idx], rel[idx], SCENE_ENTS,
                                   clip_seconds=src_len, used_seconds=TL[sid]["dur"], stretch=stretch)
    # actions boxes can't prove (hands, small objects, scans): ask the VLM, cross-checked with the geometry
    for e in st.events:
        # every required action that isn't pure walking/standing is asked: an unchecked required action would
        # otherwise pass silently (e.g. "the ticket checker looks at her phone")
        if not e.critical or e.type in {"stop", "idle"}:
            continue
        motion = e.type in scene_state.LOCOMOTION
        if motion:
            # walking / entering / exiting is measured by geometry (start, end, direction of the actor). Before,
            # "enter"/"exit" looked for an object hand-off result (never there) → geometry UNKNOWN → the VLM alone
            # decided, and a "no" from a 2B model on 12 frames triggered a blind reseed.
            mine = [r for r in res if r.entity == e.actor and r.check in ("start_position", "end_position", "direction")]
            gst = ("FAIL" if any(r.status == "FAIL" for r in mine) else
                   "PASS" if mine and all(r.status == "PASS" for r in mine) else "UNKNOWN")
            geo = motion_check.CheckResult("geometry", gst, e.actor, "; ".join(f"{r.check} {r.status}" for r in mine))
            if gst != "UNKNOWN":       # geometry measured it (as walk_to always was): no second opinion needed
                res.append(motion_check.CheckResult("event_motion", gst, e.actor, f"{e.id} ({e.type}): geometry {gst}",
                                                    {"event": e.id, "geometry": geo.reason}))
                continue
        else:
            geo = next((r for r in res if r.check == "object_transfer" and r.entity == e.obj), None)
        if not VLM_CHECK:
            vlm_status, ev = "UNKNOWN", {"why": "VLM_CHECK is off"}
        elif _JUDGE is None:
            vlm_status, ev = "UNKNOWN", {"why": _LOADER_ERRORS.get("vlm", "no VLM")}
        else:
            try:
                names = {k: v for k, v in entity_queries(st).items()}
                ans = _JUDGE.ask(arr[idx], [i / fps for i in idx], vlm_check.question(e, names))
                vlm_status, ev = vlm_check.parse(ans)
                ev["question"] = vlm_check.question(e, names)
            except Exception as ex:
                log_exc(f"{sid}: VLM question failed"); vlm_status, ev = "UNKNOWN", {"why": repr(ex)[:200]}
        geo_status = geo.status if geo else "UNKNOWN"
        # geometry is primary; the VLM can confirm what boxes can't see; a disagreement is never a pass
        final = ("FAIL" if geo_status == "FAIL" else
                 "UNKNOWN" if geo_status == "PASS" and vlm_status == "FAIL" else
                 "PASS" if "PASS" in (geo_status, vlm_status) and "FAIL" not in (geo_status, vlm_status) else
                 vlm_status)
        if geo and geo.status == "UNKNOWN" and final == "PASS":
            geo.critical = False                                  # superseded by the VLM's timed evidence
        if final == "PASS" and isinstance(ev.get("end_s"), (int, float)):
            # the VLM's time is the timing evidence when boxes can't time the action (small objects)
            for j, r in enumerate(res):
                if r.check == "completes_before_cut" and r.entity == e.id and r.status == "UNKNOWN":
                    res[j] = motion_check.check_completes_before_cut(e.id, min(ev["end_s"] / src_len, 1.0), src_len,
                                                                     TL[sid]["dur"], stretch)
        res.append(motion_check.CheckResult("event_motion" if motion else "event", final, e.actor,
                                            f"{e.id} ({e.type}): geometry {geo_status}, VLM {vlm_status}",
                                            dict(ev, event=e.id)))
    return res


CLIP_TIMING = {}     # shot_id -> seconds per stage of its latest generation (sidecar, perf.json, benchmark)


def make_vace_clip(pipe_v, shot, emb, params):
    """One VACE generation steered by the plan's layout video. Every stage is timed with the GPU synchronized."""
    import torch
    st = STATES[shot["shot_id"]]
    fps, _, w, h = VACE_NATIVE
    nf = params["frames"]
    t0 = cuda_now()
    tracks = layout_control.expected_tracks(st, nf)
    ctrl = layout_control.control_video(tracks, w, h, skip=params["skip"])
    mask = [Image.new("L", (w, h), 255)] * nf
    seed = shot["seed"] + params["seed_off"]
    torch.manual_seed(seed)
    vt = getattr(getattr(pipe_v, "vae", None), "timing", None)
    v0 = dict(vt) if vt else None
    steps = []
    def _step(pipe, i, t, kw):
        steps.append(cuda_now())
        return {}
    t1 = cuda_now()
    with GpuSampler() as gs:
        out = pipe_v(video=ctrl, mask=mask, conditioning_scale=params["cs"],
                     prompt_embeds=emb[logic_prompt(shot, params["static"])].to("cuda"),
                     negative_prompt_embeds=emb[NEG_V].to("cuda"), height=h, width=w, num_frames=nf,
                     num_inference_steps=pipe_v.speed["steps"], guidance_scale=pipe_v.speed["guidance"],
                     generator=torch.Generator("cpu").manual_seed(seed), output_type="pt",
                     callback_on_step_end=_step).frames[0]
    t2 = cuda_now()
    if not torch.isfinite(out).all():
        raise RuntimeError("the model produced a broken clip (NaN)")
    arr = (out.clamp(0, 1) * 255).round().to(torch.uint8).permute(0, 2, 3, 1).cpu().numpy()
    del out
    t3 = cuda_now()
    tm = {"frames": nf, "steps": pipe_v.speed["steps"], "speed": pipe_v.speed["speed"],
          "layout_video_s": round(t1 - t0, 1), "pipeline_s": round(t2 - t1, 1), "to_frames_s": round(t3 - t2, 1)}
    if vt:
        enc, dec = vt["encode_s"] - v0["encode_s"], vt["decode_s"] - v0["decode_s"]
        start = vt["last_encode_end"] or t1
        tm.update(vae_encode_s=round(enc, 1), vae_decode_s=round(dec, 1),
                  vae_encodes=vt["encode_calls"] - v0["encode_calls"], zero_latent_reused=vt["zero_reused"] - v0["zero_reused"],
                  vae_tiles=bool(getattr(pipe_v.vae, "use_tiling", False)))
        if steps:
            per = np.diff([start] + steps)
            tm.update(denoise_s=round(steps[-1] - start, 1), step_s=[round(float(x), 1) for x in per],
                      step_avg_s=round(float(per.mean()), 1))
        tm["other_s"] = round(tm["pipeline_s"] - enc - dec - tm.get("denoise_s", 0.0), 1)
    tm.update(gs.summary())
    CLIP_TIMING[shot["shot_id"]] = tm
    log(f"{shot['shot_id']}: VACE layout clip {nf} frames, conditioning {params['cs']}, seed {seed}, "
        f"skip {sorted(params['skip'])}, static {params['static']} | timing {json.dumps(tm)}")
    return arr, fps


def precheck(shot):
    """Deterministic checks of the PLAN before any GPU time is spent: problems here make a clip unverifiable no
    matter how well it is generated (the detector can't see a tiny subject; a subject that is on screen too briefly
    fails 'presence' by construction). Returns warnings (shown, logged, not blocking)."""
    st = STATES[shot["shot_id"]]
    n = logic_frames(shot)
    tr = layout_control.expected_tracks(st, n)
    out = []
    for e in st.events:
        if not e.critical:
            continue
        for k in {e.actor, e.obj} - {None}:
            boxes = tr.get(k) or []
            vis = [b for b in boxes if b and b[2] > 0.02 and b[0] < 0.98]
            share = len(vis) / max(len(boxes), 1)
            if share < motion_check.TH["visible"] and not getattr(SCENE_ENTS.get(k), "small", False):
                out.append(f"{k} is on screen in only {share:.0%} of the planned clip (presence needs "
                           f"{motion_check.TH['visible']:.0%})")
            if vis and k == e.actor and min((b[2] - b[0]) * (b[3] - b[1]) for b in vis) < 0.01:
                out.append(f"{k}'s box is under 1% of the frame at some point: the detector may miss it")
    return out


def make_vace_clip_safe(pipe_v, shot, emb, params):
    """make_vace_clip, but an out-of-memory error doesn't lose the shot: the memory state is logged, memory is freed
    and the clip is retried once with 25 % fewer frames (attention memory grows with the square of the frame count)."""
    import torch
    try:
        return make_vace_clip(pipe_v, shot, emb, params)
    except torch.cuda.OutOfMemoryError:
        try:
            summary = torch.cuda.memory_summary(abbreviated=True)
        except Exception:
            summary = "(memory summary unavailable)"
        log(f"{shot['shot_id']}: out of GPU memory ({gpu_mem()})\n" + summary, "ERROR")
        free_gpu()
        params = apply_repair("fewer_frames", params, STATES[shot["shot_id"]])
        print(f"    {shot['shot_id']}: GPU memory full → retrying with {params['frames']} frames")
        return make_vace_clip(pipe_v, shot, emb, params)


def release_checkers():
    """Give the checkers' GPU memory back before the video model is loaded again."""
    global _TRACKER, _JUDGE
    for obj, name in ((_TRACKER, "OWLv2"), (_JUDGE, "Qwen3-VL")):
        if obj is not None:
            release_model(getattr(obj, "model", None), name)
    _TRACKER = _JUDGE = None
    free_gpu()


def apply_repair(rep, params, st):
    """Targeted repair: change exactly what the failure points at."""
    if rep == "layout_strength":
        params["cs"] = min(params["cs"] + 0.25, 1.5)
    elif rep == "layout_simplify":                          # keep only moving subjects and what they interact with
        keep = {e.actor for e in st.events} | {e.obj for e in st.events if e.obj} | {e.target for e in st.events if e.target}
        params["skip"] = set(st.start) - keep
    elif rep == "static_camera":
        params["static"] = True
    elif rep == "reseed":
        params["seed_off"] += 1000
    elif rep == "fewer_frames":
        f = int(params["frames"] * 0.75)                       # 25 % shorter, kept at Wan's 4k+1 frame counts
        params["frames"] = max(33, 4 * ((f - 1) // 4) + 1)
    return params


def review_assets(shot, arr, results, decision):
    """Pictures for a person: 4 frames of the clip, and the plan's intended layout (first → last positions)."""
    sid, st = shot["shot_id"], STATES[shot["shot_id"]]
    try:
        n = len(arr)
        strip = np.concatenate([np.asarray(Image.fromarray(arr[i]).resize((416, 240)))
                                for i in np.linspace(0, n - 1, 4).round().astype(int)], axis=1)
        Image.fromarray(strip).save(LOGIC_DIR / f"{sid}.jpg", quality=85)
        names = {k: (CHARS.get(k, {}).get("name") if "CHARS" in globals() else None) or k for k in st.start}
        layout_control.composition_guide(st, layout_control.expected_tracks(st, 9), 832, 480, names).save(
            LOGIC_DIR / f"{sid}_plan.png")
        (LOGIC_DIR / f"{sid}.txt").write_text(
            f"{sid}: {decision.status}\nfailed: " + "; ".join(decision.reasons) + "\nuncertain: " + "; ".join(decision.unknown)
            + "\n\n" + "\n".join(f"{r.status:7} {r.check} {r.entity or ''}: {r.reason}" for r in results))
    except Exception:
        log_exc(f"{sid}: could not save review pictures")


def decode_clip(path):
    """mp4 -> uint8 frames (for re-validating a cached clip without regenerating it)."""
    probe = subprocess.run(["ffmpeg", "-i", str(path)], capture_output=True, text=True).stderr
    m = re.search(r", (\d+)x(\d+)[, ]", probe)
    w, h = int(m.group(1)), int(m.group(2))
    raw = subprocess.run(["ffmpeg", "-loglevel", "error", "-i", str(path), "-f", "rawvideo", "-pix_fmt", "rgb24", "-"],
                         capture_output=True).stdout
    return np.frombuffer(raw, np.uint8).reshape(-1, h, w, 3)


print(f"Logical consistency: {len(STATES)} shots with a scene-state"
      + (f" → VACE layout control + checks (LOGIC_CHECK={LOGIC_CHECK}, VLM_CHECK={VLM_CHECK}, "
         f"REPAIR_BUDGET={REPAIR_BUDGET})" if STATES else "; other AI shots keep plain text-to-video (not logically verified)"))


In [ ]:
BUDGET.check('bench')
# ==== 8d. Benchmark (only when BENCHMARK = True in the settings) ====
# Times ONE scene-state shot (BENCH_SHOT) under each configuration in BENCH_CONFIGS: same shot, seed, frame count and
# prompts, so the numbers are comparable. Every result is checked by the same validators (QC decision = quality
# proxy) and saved for a visual comparison. Writes render/bench/benchmark.md + benchmark.json, then stops the
# notebook (no video is made). Set BENCHMARK = False afterwards for normal runs.
BENCH = {
    "baseline":    {"speed": "fast", "vae_tiling": True, "zero_cache": False,
                    "about": "the old setting: 25 steps x 2 passes (guidance) + First Block Cache, VAE in tiles"},
    "turbo_tiled": {"speed": "turbo", "vae_tiling": True, "zero_cache": False,
                    "about": "turbo LoRA (4 steps, 1 pass) only; VAE still in tiles"},
    "optimized":   {"speed": "turbo", "vae_tiling": "auto", "zero_cache": True,
                    "about": "turbo + untiled VAE (tiles only after out-of-memory) + zero-latent reuse"},
}


class BenchmarkDone(Exception):
    """Raised on purpose after a benchmark so 'Run all' stops before making the video."""


def run_benchmark():
    global VIDEO_SPEED, VAE_TILING, VAE_ZERO_CACHE
    import torch
    shot = next((s for s in SHOTS if s["shot_id"] == BENCH_SHOT), None)
    if shot is None or not logic_shot(shot):
        print(f"BENCHMARK: {BENCH_SHOT} is not a scene-state shot of this plan; choose one of {sorted(STATES)}")
        return
    if VIDEO_ENGINE is None:
        print("BENCHMARK needs a GPU runtime"); return
    out = PROJ / "render" / "bench"; out.mkdir(parents=True, exist_ok=True)
    keep = (VIDEO_SPEED, VAE_TILING, VAE_ZERO_CACHE)
    print(f"Benchmark of {BENCH_SHOT} ({logic_frames(shot)} frames, {VACE_NATIVE[2]}x{VACE_NATIVE[3]}) on {GPU['name']}: "
          f"{', '.join(BENCH_CONFIGS)}")
    emb = encode_prompts(sorted({logic_prompt(shot), NEG_V}))
    rows = []
    for name in BENCH_CONFIGS:
        cfg = BENCH[name]
        VIDEO_SPEED, VAE_TILING, VAE_ZERO_CACHE = cfg["speed"], cfg["vae_tiling"], cfg["zero_cache"]
        row = {"config": name, "about": cfg["about"]}
        pipe = None
        try:
            free_gpu(); torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            pipe = load_video_model("vace")
            row["load_s"] = round(time.time() - t0, 1)
            row["speed_used"], row["turbo_error"] = pipe.speed["speed"], pipe.speed.get("turbo_error")
            if cfg["speed"] == "turbo" and pipe.speed["speed"] != "turbo":
                row["note"] = "turbo failed to load: not measured (see turbo_error)"
                print(f"  {name}: turbo failed ({row['turbo_error']}); skipped")
                continue
            torch.cuda.reset_peak_memory_stats()
            params = {"cs": 1.0, "skip": set(), "static": False, "seed_off": 0, "frames": logic_frames(shot)}
            t1 = time.time()
            arr, fps = make_vace_clip(pipe, shot, emb, params)
            path = out / f"{BENCH_SHOT}_{name}.mp4"
            write_clip(arr, fps, path)
            row["clip_total_s"] = round(time.time() - t1, 1)
            row["peak_vram_gb"] = round(torch.cuda.max_memory_reserved() / 2**30, 2)
            row.update(CLIP_TIMING.get(BENCH_SHOT, {}))
            row["path"] = str(path)
            Image.fromarray(np.concatenate([arr[i] for i in np.linspace(0, len(arr) - 1, 4).round().astype(int)],
                                           axis=1)).resize((1664, 240)).save(out / f"{BENCH_SHOT}_{name}.jpg", quality=85)
            print(f"  {name}: {row['clip_total_s'] / 60:.1f} min for the clip {timing_line(row)}, "
                  f"peak GPU memory {row['peak_vram_gb']} GB")
        except Exception as e:
            log_exc(f"benchmark {name} failed"); row["error"] = repr(e)[:300]
            print(f"  {name}: FAILED {repr(e)[:200]}")
        finally:
            rows.append(row)
            if pipe is not None:
                release_model(pipe, "VACE (benchmark)"); del pipe
            free_gpu()
    VIDEO_SPEED, VAE_TILING, VAE_ZERO_CACHE = keep
    for row in rows:                                   # same checks for every result: quality is compared, not assumed
        if row.get("path"):
            try:
                res = logic_checks(shot, decode_clip(Path(row["path"])), VACE_NATIVE[0])
                d = acceptance.decide(res, [], 0)
                row.update(qc=d.status, qc_failed=d.reasons, qc_uncertain=d.unknown)
            except Exception as e:
                log_exc("benchmark check failed"); row["qc"] = f"check failed: {repr(e)[:100]}"
    release_checkers()
    base = next((r for r in rows if r["config"] == "baseline" and r.get("clip_total_s")), None)
    cols = [("Model initialisation (s)", "load_s"), ("Diffusion (s)", "denoise_s"), ("Seconds per step", "step_avg_s"),
            ("VAE encode (s)", "vae_encode_s"), ("VAE decode (s)", "vae_decode_s"), ("Total clip (s)", "clip_total_s"),
            ("Peak GPU memory (GB)", "peak_vram_gb"), ("Average GPU utilisation (%)", "gpu_util_avg"),
            ("QC decision", "qc")]
    md = [f"# Benchmark: {BENCH_SHOT} on {GPU['name']} ({time.strftime('%Y-%m-%d %H:%M')})", "",
          "| Measurement | " + " | ".join(r["config"] for r in rows) + " |", "|---|" + "---|" * len(rows)]
    for label, k in cols:
        md.append(f"| {label} | " + " | ".join(str(r.get(k, "—")) for r in rows) + " |")
    if base:
        md.append("| Speed-up vs baseline (total clip, measured) | " + " | ".join(
            f"{base['clip_total_s'] / r['clip_total_s']:.1f}x" if r.get("clip_total_s") else "—" for r in rows) + " |")
    md += ["", *[f"- **{r['config']}**: {r['about']}" + (f" — {r.get('note') or r.get('error')}" if r.get('note') or r.get('error') else "")
                 for r in rows], "", "Clips and 4-frame strips for a visual comparison are in this folder."]
    (out / "benchmark.md").write_text("\n".join(md))
    (out / "benchmark.json").write_text(json.dumps(rows, indent=2, default=str))
    print("\n".join(md))
    raise BenchmarkDone(f"Benchmark finished: {out / 'benchmark.md'}. Set BENCHMARK = False to make the video.")


In [ ]:
BUDGET.check('clips')
# ==== 9. AI video for every shot (Wan 2.2 on L4/A100, Wan 2.1 small model on T4) ====
NATIVE = {"wan22-5b": (24, 121, 1280, 704), "wan21-1.3b": (16, 81, 832, 480)}   # fps, frames, width, height
TL = {r["shot_id"]: r for r in TIMELINE}

def clip_key(shot, mode):
    k = {"motion_prompt": shot["motion_prompt"], "image_prompt": shot["image_prompt"], "seed": shot["seed"]}
    if mode == "i2v":                                 # only image-to-video clips depend on the keyframe
        k["keyframe_prompt"] = (read_sidecar(PROJ / "keyframes" / f"{shot['shot_id']}.png") or {}).get("prompt")
    return k

def clip_info(shot):
    """The finished clip set for this shot, or None if missing / made from different settings.
    Scene-state shots: only when generated from the same request AND accepted by the checks (or approved)."""
    if logic_shot(shot):
        return read_sidecar(PROJ / "clips" / shot["shot_id"]) if logic_usable(shot)[0] else None
    sc = read_sidecar(PROJ / "clips" / shot["shot_id"])
    if not sc or any(sc.get(k) != v for k, v in clip_key(shot, sc.get("mode")).items()):
        return None
    if sc.get("writer", 1) < CLIP_WRITER:            # made by the old writer: colours are inverted, remake it
        return None
    return sc if all((PROJ / "clips" / p).exists() for p in sc["parts"]) else None

def encode_prompts(texts):
    """Encoded prompts come from the Drive cache when possible. The rest go through Wan's huge text encoder
    (UMT5-XXL), loaded straight onto the GPU once, then freed. Keeping it in system RAM crashed the free T4."""
    import torch
    EMB_CACHE.mkdir(parents=True, exist_ok=True)
    out = {}
    for t in texts:
        if emb_path(t).exists():
            try:
                e = torch.load(emb_path(t)).float()
                out[t] = torch.cat([e, e.new_zeros(512 - len(e), e.shape[1])]).unsqueeze(0)   # same as diffusers
            except Exception:
                log_exc(f"cached prompt {emb_path(t).name} unreadable, encoding again")
    missing = [t for t in texts if t not in out]
    print(f"Prompts: {len(out)} from the Drive cache, {len(missing)} to encode.")
    if not missing:
        return out
    from transformers import AutoTokenizer, UMT5EncoderModel
    from diffusers.pipelines.wan.pipeline_wan import prompt_clean
    need_disk(13, "the text encoder download (11.4 GB)", purge=[([VIDEO_MID["wan21-1.3b"]], ["transformer", "vae"])])
    print("Downloading the text encoder (11 GB, one time per session; takes ~5–10 min)...")
    for mid in TEXT_ENCODER_MIDS:
        try:
            tok = AutoTokenizer.from_pretrained(mid, subfolder="tokenizer")
            for dt in ([torch.bfloat16] if real_bf16() else [torch.float16, torch.bfloat16]):
                te = UMT5EncoderModel.from_pretrained(mid, subfolder="text_encoder", torch_dtype=dt,
                                                      device_map={"": 0}, low_cpu_mem_usage=True).eval()
                ram("(text encoder on GPU)")
                new, ok = {}, True
                with torch.no_grad():
                    for t in missing:
                        ti = tok([prompt_clean(t)], padding="max_length", max_length=512, truncation=True,
                                 add_special_tokens=True, return_attention_mask=True, return_tensors="pt")
                        n = int(ti.attention_mask.sum())
                        e = te(ti.input_ids.to("cuda"), ti.attention_mask.to("cuda")).last_hidden_state[0, :n].float().cpu()
                        if not torch.isfinite(e).all():
                            ok = False; break
                        new[t] = e
                release_model(te, "text encoder"); del te       # really free its ~11 GB before the video model
                if ok:
                    for t, e in new.items():
                        torch.save(e.half(), emb_path(t))
                        out[t] = torch.cat([e, e.new_zeros(512 - len(e), e.shape[1])]).unsqueeze(0)
                    # the prompts are saved on Drive now, so the 11 GB text encoder can leave the Colab disk
                    print(f"Prompts saved; freed {purge_hf(TEXT_ENCODER_MIDS, ['text_encoder']):.1f} GB of disk.")
                    return out
                print(f"  text encoder overflowed in {dt}, retrying")
        except Exception as e:
            log_exc(f"text encoder from {mid} failed")
            print(f"  text encoder from {mid} failed ({repr(e)[:200]}), trying the next copy")
            free_gpu()
    raise RuntimeError("text encoder could not encode the prompts")

# Speed-up for the T4 model (Wan 2.1 1.3B): the Self-Forcing DMD distillation as a LoRA (Apache-2.0,
# github.com/guandeh17/Self-Forcing; LoRA extracted by Kijai). 4 steps and no negative-prompt pass instead of
# 25 steps x 2 passes: ~10x less work per clip.
TURBO_LORA = ("Kijai/WanVideo_comfy", "LoRAs/Wan2_1_self_forcing_1_3B/Wan2_1_self_forcing_dmd_1_3B_lora_rank_32_fp16.safetensors")
TURBO_STEPS = 4

def turbo_scheduler():
    """Self-Forcing's own sampling: timesteps 1000/750/500/250 warped by shift 5; at each step predict the clean
    video, then re-noise to the next level (stochastic_sampling)."""
    from diffusers import FlowMatchEulerDiscreteScheduler
    class TurboScheduler(FlowMatchEulerDiscreteScheduler):
        def set_timesteps(self, num_inference_steps=None, device=None, **kw):
            n = num_inference_steps
            return super().set_timesteps(sigmas=list(1.0 - np.arange(n) / n), device=device)
    return TurboScheduler(shift=5.0, stochastic_sampling=True)

def turbo_lora_file():
    """The turbo LoRA as a plain local file. Downloaded to the session disk (Hugging Face's cache uses symlinks and
    lock files, which a Google Drive folder doesn't reliably support), then kept on Drive as ONE plain file so the
    next session copies it instead of downloading. Returns the local path."""
    from huggingface_hub import hf_hub_download
    name = Path(TURBO_LORA[1]).name
    keep, local = MODEL_CACHE / "lora" / name, WORK.parent / "lora" / name
    if local.exists():
        return local
    local.parent.mkdir(parents=True, exist_ok=True)
    if keep.exists() and keep.stat().st_size > 50e6:
        shutil.copy(keep, local)
        return local
    got = Path(hf_hub_download(TURBO_LORA[0], TURBO_LORA[1], cache_dir=str(WORK.parent / "hf_lora")))
    shutil.copy(got, local)
    try:
        keep.parent.mkdir(parents=True, exist_ok=True); shutil.copy(local, keep)
    except Exception:
        log_exc("could not keep the turbo LoRA on Drive (it is downloaded again next session)")
    return local

def fuse_turbo(p):
    """Fuse the Self-Forcing turbo LoRA into the transformer and PROVE it worked: the base blocks must change and
    (for VACE) the layout-control blocks must not. Wan2.1-VACE-1.3B's base blocks are bit-identical to Wan2.1-T2V-1.3B
    (checked: 4 sampled tensors, relative difference 0), so this T2V-distilled LoRA lands on the network it was
    distilled from; VACE's own blocks are untouched. Raises with the failing stage named."""
    import torch
    from safetensors.torch import load_file
    stage, t0 = "download", time.time()
    try:
        path = turbo_lora_file()
        stage = "read"; sd = load_file(str(path))
        tr = p.transformer
        base = tr.blocks[0].attn1.to_q.weight
        vace = getattr(tr, "vace_blocks", None)
        vace_w = vace[0].proj_out.weight if vace is not None else None
        before = (base.detach().float().sum().item(), None if vace_w is None else vace_w.detach().float().sum().item())
        stage = "load"; p.load_lora_weights(sd, adapter_name="turbo")
        stage = "fuse"; p.fuse_lora(lora_scale=1.0)
        stage = "unload"; p.unload_lora_weights()        # weights stay fused; removes the extra LoRA layers
        stage = "verify"
        after = (tr.blocks[0].attn1.to_q.weight.detach().float().sum().item(),
                 None if vace_w is None else tr.vace_blocks[0].proj_out.weight.detach().float().sum().item())
        if after[0] == before[0]:
            raise RuntimeError("the LoRA was loaded but the base weights did not change (nothing was fused)")
        if before[1] is not None and abs(after[1] - before[1]) > 1e-3 * max(abs(before[1]), 1.0):
            raise RuntimeError("the LoRA changed VACE's layout-control blocks (it must not)")
        stage = "scheduler"; p.scheduler = turbo_scheduler()
    except Exception as e:
        e.turbo_stage = stage
        raise
    log(f"turbo LoRA fused and verified in {time.time() - t0:.1f}s ({path.name}); scheduler {type(p.scheduler).__name__}")

def speed_up(p):
    """Apply VIDEO_SPEED. Returns the settings the clips are made with. Never stops the run: on any failure it
    logs the full error, SHOWS the reason, and falls back to the next slower mode (recorded in each clip's sidecar)."""
    mode = VIDEO_SPEED
    if mode == "turbo" and VIDEO_ENGINE != "wan21-1.3b":
        log("turbo LoRA only exists for Wan 2.1 1.3B; using fast mode")
        mode = "fast"
    turbo_error = None
    if mode == "turbo":
        try:
            fuse_turbo(p)
            print(f"Speed: turbo ({TURBO_STEPS} steps, Self-Forcing distilled model, verified)")
            return {"speed": "turbo", "steps": TURBO_STEPS, "guidance": 1.0}
        except Exception as e:
            log_exc("turbo LoRA failed, using fast mode")
            turbo_error = f"{getattr(e, 'turbo_stage', '?')}: {type(e).__name__}: {str(e)[:300]}"
            print(f"⚠️ Turbo mode FAILED at step {turbo_error}\n   → using fast mode instead (~6x slower per clip). "
                  "Full traceback in the log: search 'turbo LoRA failed'.")
            mode = "fast"
            try:                                           # a half-loaded adapter must not stay in the model
                p.unload_lora_weights()
            except Exception:
                pass
    if mode == "fast":
        try:
            from diffusers.hooks import FirstBlockCacheConfig
            p.transformer.enable_cache(FirstBlockCacheConfig(threshold=CACHE_THRESHOLD))
            print(f"Speed: fast ({CLIP_STEPS} steps + First Block Cache, threshold {CACHE_THRESHOLD})")
            return {"speed": "fast", "steps": CLIP_STEPS, "guidance": 5.0, "turbo_error": turbo_error}
        except Exception:
            log_exc("First Block Cache failed, using quality mode")
    print(f"Speed: quality ({CLIP_STEPS} full steps)")
    return {"speed": "quality", "steps": CLIP_STEPS, "guidance": 5.0, "turbo_error": turbo_error}

def weights_gb(path, kw, default):
    """Size of a model's weights at the dtype it is loaded in (what a CPU-side load holds in RAM at its peak)."""
    try:
        d = Path(path) / kw.get("subfolder", "")
        files = list(d.glob("*.safetensors")) if d.is_dir() else []
        if files:
            return sum(f.stat().st_size for f in files) / 2**30
    except Exception:
        pass
    return default

def load_to_gpu(cls, path, ram_gb=4.5, **kw):
    """Load a model straight onto the GPU (system RAM stays free). If that fails, the fallback goes through system
    RAM: only after checking there is room (ram_gb = its size there; VACE in half precision ≈ 3.6 GB), one weight at
    a time (low_cpu_mem_usage), then moved to the GPU at once. Never the unchecked full-RAM load that crashed Colab."""
    ram(f"(before loading {cls.__name__})")
    try:
        m = cls.from_pretrained(path, device_map={"": 0}, **kw)
        ram(f"({cls.__name__} on the GPU)")
        return m
    except Exception:
        log_exc(f"{cls.__name__}: direct GPU load failed, loading through system RAM")
    need_ram(weights_gb(path, kw, ram_gb), f"{cls.__name__} (loaded through system RAM)")
    m = cls.from_pretrained(path, low_cpu_mem_usage=True, **kw)
    if GPU["name"]:
        m = m.to("cuda")
    free_gpu()
    ram(f"({cls.__name__} moved to the GPU)")
    return m

def video_parts(mid, dt, tr_cls=None, name=None):
    """Transformer + VAE of a 1.3B model (Wan2.1 T2V, or VACE). From the Drive model cache when it is there (half
    precision, no download); otherwise from Hugging Face (full precision), then saved to the cache for next time."""
    import torch
    from diffusers import AutoencoderKLWan, WanTransformer3DModel
    tr_cls = tr_cls or WanTransformer3DModel
    cdir = MODEL_CACHE / f"{name or VIDEO_ENGINE}-{str(dt).split('.')[-1]}"
    if (cdir / "complete").exists():
        try:
            t0 = time.time()
            tr = load_to_gpu(tr_cls, cdir / "transformer", torch_dtype=dt)
            vae = load_to_gpu(AutoencoderKLWan, cdir / "vae", ram_gb=0.6, torch_dtype=torch.float32)
            print(f"Video model loaded from the model cache in {time.time() - t0:.0f}s (no download)")
            return tr, vae
        except Exception:
            log_exc("model cache unreadable, downloading again")
            shutil.rmtree(cdir, ignore_errors=True)
    need_disk(8, "the video model download (6.2 GB)", purge=[(TEXT_ENCODER_MIDS, ["text_encoder"])])
    tr = load_to_gpu(tr_cls, mid, subfolder="transformer", torch_dtype=dt)
    vae = load_to_gpu(AutoencoderKLWan, mid, ram_gb=0.6, subfolder="vae", torch_dtype=torch.float32)
    try:
        MODEL_CACHE.mkdir(parents=True, exist_ok=True)
        drive_free = shutil.disk_usage(MODEL_CACHE).free / 2**30
        log(f"model cache: {drive_free:.1f} GB free where it is stored ({MODEL_CACHE})")
        if drive_free > 4:                              # the cache itself is ~3.3 GB
            t0 = time.time()
            tr.save_pretrained(cdir / "transformer", max_shard_size="1GB")   # 1 GB at a time: low RAM
            vae.save_pretrained(cdir / "vae")
            (cdir / "complete").write_text(mid)
            print(f"Saved the video model to the model cache in {time.time() - t0:.0f}s: next sessions skip the download")
            purge_hf([mid], ["transformer", "vae"])  # the model is in GPU memory and in the cache: free the disk
        else:
            print(f"⚠️ Only {drive_free:.1f} GB free on Google Drive: the video model cache (3.3 GB) was NOT saved, so "
                  "every new session downloads the model again (~4 min). Free up 4 GB on Drive to fix this.")
    except Exception:
        log_exc("could not save the model cache")
        shutil.rmtree(cdir, ignore_errors=True)
    return tr, vae

ZERO_LATENTS = MODEL_CACHE / "vae_zero_latents"   # ~17 MB per clip length; kept on Drive across rounds and sessions

def vae_guard(vae, ident="wan21-vae"):
    """Wrap a Wan VAE's encode/decode (VACE runs 2 encodes + 1 decode of the whole clip per generation, in fp32,
    OUTSIDE the step progress bar):
      - timing: synchronized seconds per call → vae.timing (shown per clip, written to perf.json);
      - VAE_TILING="auto": untiled first (tiles overlap, ~2.5x the pixels at 832x480); on out-of-memory the SAME call
        is retried with tiles and tiles stay on — the clip is never lost; True = always tiles (the old behaviour);
      - VAE_ZERO_CACHE: VACE's "inactive" video is all zeros (our mask covers the whole frame), so its latent is the
        same for every clip of that size: computed once, saved, then reused (bit-identical, one VAE pass saved per
        clip; the file survives the VACE reloads between rounds and the next sessions)."""
    if getattr(vae, "_guarded", False):
        return vae
    import torch
    enc, dec = vae.encode, vae.decode
    vae.timing = {"encode_s": 0.0, "decode_s": 0.0, "encode_calls": 0, "decode_calls": 0, "zero_reused": 0,
                  "tiles_after_oom": False, "last_encode_end": None}
    if VAE_TILING is True:
        vae.enable_tiling()

    def run(fn, kind, x, *a, **kw):
        t0 = cuda_now()
        try:
            out = fn(x, *a, **kw)
        except torch.cuda.OutOfMemoryError:
            if getattr(vae, "use_tiling", False):
                raise
            log(f"VAE {kind} without tiles: out of GPU memory ({gpu_mem()}) → tiles from now on", "WARN")
            free_gpu(); vae.enable_tiling(); vae.timing["tiles_after_oom"] = True
            out = fn(x, *a, **kw)
        t1 = cuda_now()
        vae.timing[f"{kind}_s"] += t1 - t0; vae.timing[f"{kind}_calls"] += 1
        if kind == "encode":
            vae.timing["last_encode_end"] = t1
        return out

    def encode(x, *a, **kw):
        if VAE_ZERO_CACHE and torch.is_tensor(x) and not a and kw.get("return_dict", True) and not bool(x.any()):
            tiles = bool(getattr(vae, "use_tiling", False))
            key = hashlib.sha1(f"{ident}|{tuple(x.shape)}|{x.dtype}|tiles={tiles}".encode()).hexdigest()[:16]
            f = ZERO_LATENTS / f"{key}.pt"
            if f.exists():
                try:
                    from diffusers.models.autoencoders.vae import DiagonalGaussianDistribution
                    from diffusers.models.modeling_outputs import AutoencoderKLOutput
                    h = torch.load(f).to(x.device)
                    vae.timing["zero_reused"] += 1; vae.timing["last_encode_end"] = cuda_now()
                    return AutoencoderKLOutput(latent_dist=DiagonalGaussianDistribution(h))
                except Exception:
                    log_exc(f"cached zero latent {f.name} unreadable, encoding again")
            out = run(enc, "encode", x, *a, **kw)
            try:
                ZERO_LATENTS.mkdir(parents=True, exist_ok=True)
                torch.save(out.latent_dist.parameters.detach().cpu(), f)
            except Exception:
                log_exc("could not save the zero latent (computed again next time)")
            return out
        return run(enc, "encode", x, *a, **kw)

    vae.encode, vae.decode = encode, (lambda z, *a, **kw: run(dec, "decode", z, *a, **kw))
    vae._guarded = True
    return vae

def load_video_model(mode):
    """Video model WITHOUT the text encoder (prompts are pre-encoded)."""
    import torch
    from diffusers import AutoencoderKLWan, WanPipeline
    mid = VIDEO_MID[VIDEO_ENGINE]
    dt = torch.bfloat16 if real_bf16() else torch.float16
    if mode == "vace":                   # layout-controlled generation for scene-state shots (any GPU)
        from diffusers import WanVACEPipeline, WanVACETransformer3DModel
        tr, vae = video_parts(VACE_MID, dt, WanVACETransformer3DModel, "wan21-vace-1.3b")
        p = WanVACEPipeline.from_pretrained(VACE_MID, transformer=tr, vae=vae, text_encoder=None, torch_dtype=dt)
        p.to("cuda")
        vae_guard(p.vae, f"{VACE_MID}|{p.vae.dtype}")   # timed; untiled first, tiles only after out-of-memory; zero-latent reuse
        p.speed = speed_up(p)            # turbo LoRA fuses into VACE's base blocks (verified in fuse_turbo)
        ram("(VACE layout model loaded)")
        return p
    if mode == "i2v":
        from diffusers import WanImageToVideoPipeline as cls
    else:
        cls = WanPipeline
    if VIDEO_ENGINE == "wan22-5b":
        vae = AutoencoderKLWan.from_pretrained(mid, subfolder="vae", torch_dtype=torch.float32)
        p = cls.from_pretrained(mid, vae=vae, text_encoder=None, torch_dtype=dt)
        p.enable_model_cpu_offload()     # 5B: L4/A100 sessions have 50–80 GB RAM, so offloading is safe
    else:                                # 1.3B (~3 GB) sits on the T4 GPU: keeps system RAM free
        tr, vae = video_parts(mid, dt)
        # transformer and VAE are passed in, so only the small config files are downloaded
        p = cls.from_pretrained(mid, transformer=tr, vae=vae, text_encoder=None, torch_dtype=dt)
        p.to("cuda")
    p.speed = speed_up(p)
    ram("(video model loaded)")
    return p

CLIP_WRITER = 2   # 1 = diffusers export_to_video (bug: it multiplies 8-bit frames by 255 -> photo-negative colours)

def write_clip(arr, fps, path):
    """8-bit RGB frames (frames, H, W, 3) -> near-lossless H.264 file. Our own writer: diffusers' export_to_video
    expects 0..1 floats and turned our 0..255 frames into a colour negative, and compressed them heavily."""
    assert arr.dtype == np.uint8 and arr.ndim == 4 and arr.shape[-1] == 3, (arr.dtype, arr.shape)
    n, h, w, _ = arr.shape
    cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",
           "-s", f"{w}x{h}", "-r", str(fps), "-i", "-", "-c:v", "libx264", "-preset", "medium", "-crf", "12",
           "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(path)]
    t0 = time.time()
    r = subprocess.run(cmd, input=np.ascontiguousarray(arr).tobytes(), capture_output=True)
    log(f"clip written ({time.time() - t0:.1f}s, exit {r.returncode}): {n} frames {w}x{h} -> {Path(path).name}", "FFMPG")
    if r.returncode:
        raise RuntimeError("writing the clip failed: " + r.stderr.decode(errors="replace")[-2000:])

def check_clip_writer():
    """Self-test: known colours in, same colours out. Stops the run instead of making wrong-coloured videos."""
    ref = np.zeros((8, 64, 64, 3), np.uint8)
    ref[:, :, :32] = (200, 80, 30)                   # fox orange
    ref[:, :, 32:] = (20, 30, 60)                    # night blue
    p = WORK / "writer_test.mp4"
    write_clip(ref, 16, p)
    raw = subprocess.run(["ffmpeg", "-loglevel", "error", "-i", str(p), "-f", "rawvideo", "-pix_fmt", "rgb24", "-"],
                         capture_output=True).stdout
    back = np.frombuffer(raw, np.uint8).reshape(-1, 64, 64, 3)
    err = np.abs(back[:, 8:24, 8:24].astype(int).mean((0, 1, 2)) - [200, 80, 30]).max()
    err = max(err, np.abs(back[:, 8:24, 40:56].astype(int).mean((0, 1, 2)) - [20, 30, 60]).max())
    log(f"clip writer self-test: max colour error {err:.1f}")
    assert err < 6, f"clip writer changes colours (error {err:.1f}); not making any clips"

# ---- continuity check: is every character who should be on screen actually in the clip? ----
# CLIP (openai/clip-vit-base-patch32, MIT, ~0.6 GB) compares 8 frames of the clip with "a scene with <character>"
# and with "the same scene with nobody in it". A character counts as seen in a frame unless the second clearly wins.
PRESENCE_MODEL, PRESENCE_MIN = "openai/clip-vit-base-patch32", 0.75   # seen in at least 6 of 8 frames
PRESENCE_FRAME_P = 0.2      # per frame: below this the character is clearly missing (tested: absent ≈ 0.01, present ≈ 0.5)
_presence = None

def presence_model():
    global _presence
    if _presence is None:
        import torch
        from transformers import CLIPModel, CLIPProcessor
        dev = "cuda" if GPU["name"] else "cpu"
        dt = torch.float16 if dev == "cuda" else torch.float32
        src = str(hf_files(PRESENCE_MODEL, "continuity checker",     # real files on Drive (no broken symlinks);
                           allow=["*.json", "*.txt", "pytorch_model.bin"]))  # PyTorch weights only (not TF/Flax copies)
        kw = dict(torch_dtype=dt)
        try:                             # straight onto the GPU: no copy in system RAM
            m = CLIPModel.from_pretrained(src, device_map={"": 0} if dev == "cuda" else None, **kw).eval()
        except Exception:
            log_exc("CLIP: direct GPU load failed, loading through system RAM")
            need_ram(1.0, "the CLIP continuity checker")
            m = CLIPModel.from_pretrained(src, low_cpu_mem_usage=True, **kw).to(dev).eval()
        _presence = (m, CLIPProcessor.from_pretrained(src), dev, dt)
        log(f"continuity checker loaded ({PRESENCE_MODEL} on {dev})")
    return _presence

def presence(arr, tags, scene):
    """{tag: {"seen": share of sampled frames where it is visible, "mean": average probability}}."""
    import torch
    m, proc, dev, dt = presence_model()
    idx = np.linspace(0, len(arr) - 1, 8).round().astype(int)
    # the two descriptions differ ONLY in whether the character is there (tag first: CLIP reads at most 77 tokens)
    texts = [f"{t}, in {scene}" for t in tags] + [f"{scene}, with nobody in it"]
    inp = proc(text=texts, images=[Image.fromarray(arr[i]) for i in idx], return_tensors="pt",
               padding=True, truncation=True)
    inp = {k: (v.to(dev, dt) if k == "pixel_values" else v.to(dev)) for k, v in inp.items()}
    with torch.no_grad():
        logits = m(**inp).logits_per_image.float().cpu()                     # (8 frames, tags + 1)
    out = {}
    for j, t in enumerate(tags):
        p = torch.softmax(logits[:, [j, -1]], dim=1)[:, 0].numpy()
        out[t] = {"seen": round(float((p > PRESENCE_FRAME_P).mean()), 3), "mean": round(float(p.mean()), 3)}
    return out

def continuity_score(pres):
    if not pres:
        return 1.0
    return min(v["seen"] for v in pres.values()) + 0.01 * min(v["mean"] for v in pres.values())

RELEVANCE_MIN = 0.5          # the clip must match its OWN scene better than any other place in ≥ half the frames

def relevance(arr, own, others):
    """Narration ↔ visual check at render time: share of sampled frames where CLIP matches the shot's own scene
    description better than every scene description from OTHER places in the video."""
    import torch
    if not others:
        return None
    m, proc, dev, dt = presence_model()
    idx = np.linspace(0, len(arr) - 1, 8).round().astype(int)
    inp = proc(text=[own] + others, images=[Image.fromarray(arr[i]) for i in idx], return_tensors="pt",
               padding=True, truncation=True)
    inp = {k: (v.to(dev, dt) if k == "pixel_values" else v.to(dev)) for k, v in inp.items()}
    with torch.no_grad():
        logits = m(**inp).logits_per_image.float().cpu()                     # (8 frames, 1 + others)
    win = (logits.argmax(dim=1) == 0).float()
    return {"seen": round(float(win.mean()), 3), "mean": round(float(torch.softmax(logits, 1)[:, 0].mean()), 3)}

def distractors(shot):
    """Scene descriptions of AI shots set somewhere else (same-place shots are allowed to look alike)."""
    here = shot.get("location", shot.get("scene"))
    out = []
    for s in SHOTS:
        if s.get("motion_prompt") and s.get("location", s.get("scene")) != here and s.get("scene") not in out:
            out.append(s["scene"])
    return out[:12]

def missing_from(pres):
    return [t for t, v in (pres or {}).items()
            if v["seen"] < (RELEVANCE_MIN if t.startswith("scene:") else PRESENCE_MIN)]

def frames_for(dur, fps, most, max_stretch=1.15, fewest=33):
    """Frames to generate for a shot of `dur` seconds: enough to cover it when slowed down at most 1.15x.
    Generating fewer frames is much faster (attention cost grows with the square of the frame count).
    Wan needs 4k+1 frames."""
    f = math.ceil(dur * fps / max_stretch) + 1
    f = 4 * math.ceil((f - 1) / 4) + 1
    return max(fewest, min(most, f))

def make_clip(pipe_v, mode, shot, image, part, emb, frames=None):
    import torch
    fps, nf, w, h = NATIVE[VIDEO_ENGINE]
    nf = frames or nf
    torch.manual_seed(shot["seed"] + part)            # turbo re-noises between steps with the global generator
    kw = dict(prompt_embeds=emb[clip_prompt(shot, mode)].to("cuda"), negative_prompt_embeds=emb[NEG_V].to("cuda"),
              height=h, width=w, num_frames=nf, num_inference_steps=pipe_v.speed["steps"],
              guidance_scale=pipe_v.speed["guidance"], generator=torch.Generator("cpu").manual_seed(shot["seed"] + part))
    log(f"{shot['shot_id']} part {part}: {pipe_v.speed} {w}x{h} {nf} frames")
    if mode == "i2v":
        kw["image"] = ImageOps.fit(image.convert("RGB"), (w, h))
    # frames stay on the GPU as a tensor and become 8-bit there: 4x less system RAM than a float32 array
    vid = pipe_v(**kw, output_type="pt").frames[0]                  # (frames, 3, H, W), values 0..1
    if not torch.isfinite(vid).all():
        raise RuntimeError("the model produced a broken clip (NaN)")
    arr = (vid.clamp(0, 1) * 255).round().to(torch.uint8).permute(0, 2, 3, 1).cpu().numpy()
    del vid
    if arr.mean() < 2.5:
        raise RuntimeError("the model produced a black clip")
    return arr, fps, nf

PERF = {}

def timing_line(tm):
    """'(VAE encode 1.2 + 25 steps x 30.1 s + VAE decode 2.3 min, GPU 97% busy)' for the screen."""
    if not tm.get("step_avg_s"):
        return ""
    parts = [f"VAE in {tm.get('vae_encode_s', 0) / 60:.1f}", f"{tm.get('steps')} steps x {tm['step_avg_s']:.0f} s",
             f"VAE out {tm.get('vae_decode_s', 0) / 60:.1f} min"]
    util = f", GPU {tm['gpu_util_avg']:.0f}% busy" if tm.get("gpu_util_avg") is not None else ""
    return "(" + " + ".join(parts) + util + ")"

def perf_report():
    """Where the scene-state time went: per clip and overall. Screen table + render/perf.json (for comparisons)."""
    if not PERF.get("clips"):
        return
    PERF["total_s"] = round(time.time() - PERF["t0"], 1)
    PERF["checker_loads"] = CHECKER_LOADS.get("n", 0)
    keys = ["layout_video_s", "vae_encode_s", "denoise_s", "vae_decode_s", "to_frames_s", "write_s", "check_s"]
    print("\nWhere the time went (seconds; GPU-synchronized):")
    print("  shot  rnd speed  steps " + " ".join(f"{k[:-2]:>12}" for k in keys) + "  s/step  GPU%  decision")
    for c in PERF["clips"]:
        print(f"  {c['shot']:5} {c['round']:>3} {str(c.get('speed')):6} {str(c.get('steps')):>5} "
              + " ".join(f"{c.get(k, 0) or 0:12.0f}" for k in keys)
              + f"  {c.get('step_avg_s') or 0:6.1f}  {c.get('gpu_util_avg') or 0:4.0f}  {c['decision']}")
    print(f"  model loads: VACE x{PERF['vace_loads']} ({PERF['vace_load_s']:.0f}s), checkers x{PERF['checker_loads']} "
          f"({CHECKER_LOADS.get('s', 0):.0f}s), releases {PERF['release_s']:.0f}s | rounds {PERF['rounds']} | "
          f"regenerations {PERF['regenerations']}/{PERF['generations']} | total {PERF['total_s'] / 60:.1f} min")
    try:
        (PROJ / "render").mkdir(parents=True, exist_ok=True)
        (PROJ / "render" / "perf.json").write_text(json.dumps(PERF, indent=2, default=str))
    except Exception:
        log_exc("could not write perf.json")
    log("perf: " + json.dumps(PERF, default=str)[:4000])

CLIPS_T0 = time.time()
def gpu_budget_spent():
    """MAX_GPU_MINUTES: once clip making has run this long, no new clip is started (the rest use their fallback)."""
    used = (time.time() - CLIPS_T0) / 60
    if used >= globals().get("MAX_GPU_MINUTES", 1e9):
        log(f"GPU budget used: {used:.0f} of {MAX_GPU_MINUTES} min; no new clips", "WARN")
        return True
    return False

video_shots = [s for s in SHOTS if s.get("motion_prompt")]
logic_all = [s for s in video_shots if logic_shot(s)]
def _logic_state(s):
    sc = read_sidecar(PROJ / "clips" / s["shot_id"]) or {}
    if sc.get("gen_key") != logic_gen_key(s) or not all((PROJ / "clips" / p).exists() for p in sc.get("parts", ["-"])):
        return "generate"
    return "ok" if sc.get("val_key") == logic_val_key(s) else "recheck"
logic_todo = [s for s in logic_all if _logic_state(s) == "generate"]
logic_recheck = [s for s in logic_all if _logic_state(s) == "recheck"]
if BENCHMARK:
    run_benchmark()          # times one shot under each setting, writes render/bench/benchmark.md, then stops
todo = [s for s in video_shots if not logic_shot(s) and clip_info(s) is None]
if logic_recheck:
    print(f"Re-checking {len(logic_recheck)} cached clips because the plan or the checks changed: "
          f"{[s['shot_id'] for s in logic_recheck]}")
if todo and VIDEO_ENGINE is None:
    print("No GPU for video: these shots will be camera moves over stills:", [s["shot_id"] for s in todo])
elif todo:
    check_clip_writer()
    bad = [s["shot_id"] for s in todo if (read_sidecar(PROJ / "clips" / s["shot_id"]) or {}).get("writer", 2) < CLIP_WRITER]
    if bad:
        print(f"Remaking {len(bad)} clips that were saved with inverted colours by the old writer: {bad}")
    for s in video_shots:                            # shots finished in earlier runs: edit them while the GPU works
        if s not in todo:
            edit_in_background(s)
    pipe_v, mode, emb = None, VIDEO_MODE, None
    wait_for_prefetch()
    try:
        emb = encode_prompts(sorted({clip_prompt(s, mode) for s in todo} | {NEG_V}))
        pipe_v = load_video_model(mode)
        print(f"Video model loaded ({VIDEO_ENGINE}, {mode}). {len(todo)} shots to make "
              f"(each shows a {pipe_v.speed['steps']}-step progress bar; finished shots are saved to Drive as they complete).")
    except Exception as e:
        log_exc("video model could not load")
        print("❌ Video model could not load:", repr(e)[:500])
    spent = []
    for i, shot in enumerate(todo, 1):
        if pipe_v is None:
            break
        if gpu_budget_spent():
            print(f"⏱️ MAX_GPU_MINUTES ({MAX_GPU_MINUTES}) reached: the remaining {len(todo) - i + 1} shots use stills/fallbacks")
            break
        sid, t0 = shot["shot_id"], time.time()
        fps, nf = NATIVE[VIDEO_ENGINE][:2]
        # A long line gets more clips (max 3) instead of a frozen last frame:
        # - image-to-video: each clip continues from the last frame of the one before (one continuous shot)
        # - text-to-video with characters: ONE clip only. A second clip would be an unrelated new take and the
        #   characters would jump, change or vanish mid-shot (story plans are checked to fit one clip)
        # - text-to-video, scenery only (space, maps...): extra takes are fine: a cut between two views of a nebula
        #   looks natural, and nothing can "vanish"
        parts_needed = max(1, min(3, math.ceil(TL[sid]["dur"] / (nf / fps * MAX_SLOWMO))))
        if mode != "i2v" and shot.get("must_show"):
            parts_needed = 1
        tags = list(shot.get("must_show") or []) if CONTINUITY_CHECK else []
        frames = frames_for(TL[sid]["dur"], fps, nf) if (FRAMES_ON_DEMAND and parts_needed == 1) else None
        img = Image.open(PROJ / "keyframes" / f"{sid}.png") if mode == "i2v" else None
        names = []
        try:
            checks = []
            for part in range(1, parts_needed + 1):
                best = None
                for attempt in range(1 + (CONTINUITY_RETRIES if (tags or CONTINUITY_CHECK) else 0)):
                    arr, fps, nf = make_clip(pipe_v, mode, shot, img, part + 1000 * attempt, emb, frames)
                    pres = None
                    if tags:
                        try:
                            pres = presence(arr, tags, shot.get("scene", shot["image_prompt"]))
                        except Exception:
                            log_exc("continuity check failed; it is switched off for this run")
                            tags = []
                    if CONTINUITY_CHECK and shot.get("scene"):
                        try:                                  # does the clip show THIS scene, not another one?
                            rel = relevance(arr, shot["scene"], distractors(shot))
                            if rel is not None:
                                pres = dict(pres or {}, **{"scene: " + shot["scene"][:40]: rel})
                        except Exception:
                            log_exc("scene relevance check failed")
                    log(f"{sid} part {part} attempt {attempt + 1}: presence {pres}")
                    if best is None or continuity_score(pres) > continuity_score(best[1]):
                        best = (arr, pres, attempt)
                    del arr
                    people = [m for m in missing_from(pres) if not m.startswith("scene:")]
                    if not people:                    # scene-only doubts: flag for review, don't spend GPU time
                        break
                    if attempt < CONTINUITY_RETRIES:
                        print(f"    {sid}: not on screen enough: {people} → remaking with a new seed")
                arr, pres, attempt = best; best = None
                checks.append({"part": part, "presence": pres, "attempts": attempt + 1, "missing": missing_from(pres)})
                if missing_from(pres):
                    print(f"    ⚠️ {sid}: the check isn't sure these are visible: {missing_from(pres)}"
                          + (" → its drawn fallback scene will be used" if shot.get("fallback") else " → kept the best take")
                          + f"; look at render/review/{sid}.jpg")
                name = f"{sid}_p{part}.mp4"
                write_clip(arr, fps, WORK / name)
                shutil.copy(WORK / name, PROJ / "clips" / name)
                names.append(name)
                img = Image.fromarray(arr[-1])
                del arr; free_gpu()
            sidecar(PROJ / "clips" / sid, **clip_key(shot, mode), parts=names, fps=fps, frames_per_part=nf,
                    writer=CLIP_WRITER, continuity=checks,
                    use_fallback=bool(shot.get("fallback")) and any(c["missing"] for c in checks), engine=VIDEO_ENGINE, mode=mode, steps=pipe_v.speed["steps"], speed=pipe_v.speed["speed"], gpu=GPU["name"],
                    seconds=round(time.time() - t0, 1))
            edit_in_background(shot)                  # cut + grade this shot on the CPU while the GPU goes on
            spent.append(time.time() - t0)
            left = sum(spent) / len(spent) * (len(todo) - i) / 60
            print(f"  [{i}/{len(todo)}] {sid}: {len(names)} clip(s) in {spent[-1] / 60:.1f} min"
                  f" | about {left:.0f} min left for the other shots")
            ram()
        except Exception as e:
            log_exc(f"{sid}: video failed")
            print(f"  [{i}/{len(todo)}] {sid}: video failed ({repr(e)[:300]}); this shot will use a camera move")
            free_gpu()
    release_model(pipe_v, "video model"); release_model((_presence or [None])[0], "CLIP")
    del pipe_v, emb; _presence = None; free_gpu()

# ---- scene-state shots: VACE layout control, every clip checked against the plan, targeted repairs ----
# Runs in ROUNDS so the 15 GB T4 only ever holds one model family: (A) VACE makes a candidate for every pending
# shot → VACE released; (B) the checkers (OWLv2 + Qwen3-VL) check every candidate → checkers released; shots that
# need a targeted repair go to the next round (at most REPAIR_BUDGET more rounds).
if (logic_todo or logic_recheck) and VIDEO_ENGINE is None:
    print("No GPU: scene-state shots can't be generated or checked now; they use their fallback until a GPU run:",
          [s["shot_id"] for s in logic_todo + logic_recheck])
elif logic_todo or logic_recheck:
    import torch
    TRACK = {s["shot_id"]: {"params": {"cs": 1.0, "skip": set(), "static": False, "seed_off": 0, "frames": logic_frames(s)},
                            "tried": [], "attempts": [], "best": None, "t0": time.time(), "gen_error": None}
             for s in logic_todo}
    pending, emb = list(logic_todo), None
    for _s in pending:
        for _w in precheck(_s):
            print(f"  ⚠️ plan check {_s['shot_id']}: {_w}"); log(f"plan check {_s['shot_id']}: {_w}", "WARN")
    PERF.update(t0=time.time(), vace_loads=0, vace_load_s=0.0, checker_load_s=0.0, release_s=0.0, rounds=0,
                generations=0, regenerations=0, clips=[])
    if pending:
        check_clip_writer()
        wait_for_prefetch()
        try:
            emb = encode_prompts(sorted({logic_prompt(s) for s in pending} | {logic_prompt(s, True) for s in pending}
                                        | {NEG_V}))
            log(f"GPU after the prompts: {gpu_mem()}")
        except Exception as e:
            log_exc("prompts could not be encoded"); print("❌ Prompts could not be encoded; scene-state shots use their fallback")
            pending = []

    def finish(shot):
        """Write the final clip + sidecar for a shot whose rounds are over."""
        sid, T = shot["shot_id"], TRACK[shot["shot_id"]]
        if T["best"] is None:
            print(f"  {sid}: no usable candidate ({T['gen_error'] or 'not generated'}); its fallback will be used")
            return
        path, fps, n, results, dec, _ = T["best"]
        final = dec if dec.status == "ACCEPT" else acceptance.decide(results, T["tried"], 0)
        name = f"{sid}_p1.mp4"
        shutil.copy(path, PROJ / "clips" / name)
        if final.status != "ACCEPT":
            review_assets(shot, decode_clip(path), results, final)
        sidecar(PROJ / "clips" / sid, gen_key=logic_gen_key(shot), val_key=logic_val_key(shot), decision=final.status,
                parts=[name], fps=fps, frames_per_part=n, writer=CLIP_WRITER, engine="wan21-vace-1.3b", mode="vace",
                gpu=GPU["name"], checks=[r.to_dict() for r in results], attempts=T["attempts"], repairs=T["tried"],
                failed=final.reasons, uncertain=final.unknown, seconds=round(time.time() - T["t0"], 1),
                motion_prompt=shot["motion_prompt"], image_prompt=shot["image_prompt"], seed=shot["seed"],
                speed=T.get("speed"))
        mark = "✅ accepted" if final.status == "ACCEPT" else f"⚠️ {final.status}: not used unless approved"
        print(f"  {sid}: {mark} after {len(T['attempts'])} attempt(s)"
              + ("" if final.status == "ACCEPT" else f" | see render/review/{sid}.*"))
        if final.status == "ACCEPT":
            edit_in_background(shot)

    rnd = 0
    while pending:
        rnd += 1
        # (A) generation: only VACE on the GPU
        try:
            tl = time.time()
            pipe_v = load_video_model("vace")
            PERF["vace_loads"] += 1; PERF["vace_load_s"] += time.time() - tl; PERF["rounds"] = rnd
            log(f"round {rnd}: VACE loaded in {time.time() - tl:.0f}s, GPU {gpu_mem()}")
            if rnd == 1:
                print(f"VACE layout model loaded. {len(pending)} scene-state shots; each is checked against the plan "
                      f"(positions, direction, stops, hand-offs, order) and repaired up to {REPAIR_BUDGET}x.")
        except Exception as e:
            log_exc("VACE model could not load")
            print("❌ VACE model could not load (details in the log); these shots use their fallback:", repr(e)[:300])
            break
        cands = {}
        for shot in pending:
            sid, T = shot["shot_id"], TRACK[shot["shot_id"]]
            if gpu_budget_spent():                        # no clip → finish() gives the shot its drawn fallback
                REVIEW[sid] = f"not made: MAX_GPU_MINUTES ({MAX_GPU_MINUTES}) reached"
                continue
            t0 = time.time()
            try:
                arr, fps = make_vace_clip_safe(pipe_v, shot, emb, T["params"])
                path = WORK / f"cand_{sid}_{len(T['attempts']) + 1}.mp4"
                tw = time.time(); write_clip(arr, fps, path)
                tm = dict(CLIP_TIMING.get(sid, {}), write_s=round(time.time() - tw, 1), clip_total_s=round(time.time() - t0, 1))
                CLIP_TIMING[sid] = tm; T["speed"] = pipe_v.speed.get("speed")
                PERF["generations"] += 1; PERF["regenerations"] += rnd > 1
                cands[sid] = (path, fps, len(arr))
                del arr
                print(f"  round {rnd} · {sid}: clip made in {(time.time() - t0) / 60:.1f} min " + timing_line(tm))
            except Exception as e:
                log_exc(f"{sid}: video failed"); T["gen_error"] = repr(e)[:200]
                print(f"  round {rnd} · {sid}: video failed ({repr(e)[:200]})")
            free_gpu()
        tr0 = time.time(); release_model(pipe_v, "VACE"); del pipe_v; PERF["release_s"] += time.time() - tr0
        # (B) checking: only the checkers on the GPU
        next_pending = []
        for shot in pending:
            sid, T = shot["shot_id"], TRACK[shot["shot_id"]]
            if sid not in cands:
                finish(shot)
                continue
            path, fps, n = cands[sid]
            tc = time.time(); results = logic_checks(shot, decode_clip(path), fps); check_s = round(time.time() - tc, 1)
            dec = acceptance.decide(results, T["tried"], REPAIR_BUDGET)
            n_fail = sum(r.status == "FAIL" and r.critical for r in results)
            tm = dict(CLIP_TIMING.get(sid, {}), check_s=check_s)
            PERF["clips"].append(dict(tm, shot=sid, round=rnd, decision=dec.status, repair=dec.repair))
            T["attempts"].append({"params": {k: (sorted(v) if isinstance(v, set) else v) for k, v in T["params"].items()},
                                  "decision": dec.status, "failed": dec.reasons, "uncertain": dec.unknown, "timing": tm})
            log(f"{sid} attempt {len(T['attempts'])}: {dec.status} {dec.repair or ''} | failed {dec.reasons} | "
                f"uncertain {dec.unknown}")
            best_ok = T["best"] is not None and T["best"][4].status == "ACCEPT"
            if not best_ok and (T["best"] is None or dec.status == "ACCEPT" or n_fail < T["best"][5]):
                T["best"] = (path, fps, n, results, dec, n_fail)    # fewest critical failures (for review only)
            if dec.status == "REPAIR":
                print(f"    {sid}: {dec.reasons[0] if dec.reasons else 'check failed'} → repair: {dec.repair}")
                T["params"] = apply_repair(dec.repair, T["params"], STATES[sid]); T["tried"].append(dec.repair)
                next_pending.append(shot)
            else:
                finish(shot)
        tr0 = time.time(); release_checkers(); PERF["release_s"] += time.time() - tr0
        pending = next_pending
        ram()
    for shot in pending:                              # rounds stopped early (model failed to load)
        finish(shot)
    for shot in logic_recheck:                        # cached clip, changed plan/checks: check again, no regeneration
        sid = shot["shot_id"]
        try:
            sc = read_sidecar(PROJ / "clips" / sid)
            arr = decode_clip(PROJ / "clips" / sc["parts"][0])
            results = logic_checks(shot, arr, sc.get("fps", 16))
            final = acceptance.decide(results, [], 0)
            if final.status != "ACCEPT":
                review_assets(shot, arr, results, final)
            sc.update(val_key=logic_val_key(shot), decision=final.status, checks=[r.to_dict() for r in results],
                      failed=final.reasons, uncertain=final.unknown)
            sidecar(PROJ / "clips" / sid, **sc)
            print(f"  {sid}: re-checked → {final.status}")
        except Exception:
            log_exc(f"{sid}: re-check failed")
            print(f"  {sid}: re-check failed (details in the log); its fallback is used")
    release_checkers()
    del emb; free_gpu()
    log(f"scene-state shots done, GPU {gpu_mem()}")
    perf_report()
ready = sum(clip_info(s) is not None for s in video_shots)
make_keyframes([s for s in SHOTS if not (s.get("motion_prompt") and clip_info(s))      # fallback pictures, if any
                and not (logic_shot(s) and s.get("fallback"))])                     # (drawn fallback needs none)
print(f"{ready}/{len(video_shots)} shots have real AI video."
      + ("" if ready == len(video_shots) else "  Re-run this notebook later to make the missing ones."))


In [ ]:
BUDGET.check('music')
# ==== 10. Music: licensed library tracks per chapter (default), or a generated score as fallback ====
MSR = 48000
def synth(out, inputs, fc, dur):
    args = []
    for src in inputs:
        args += ["-f", "lavfi", "-i", src.format(d=f"{dur:.3f}")]
    ff(*args, "-filter_complex", fc, "-ar", MSR, "-ac", 2, "-t", f"{dur:.3f}", out)
    a, _ = sf.read(str(out), dtype="float32", always_2d=True)
    return a

def curious_layer(dur, bpm=92, seed=7):
    """Playful, inquisitive motif: plucked-string arpeggios (Karplus-Strong), a soft bass pluck, music-box bells."""
    rng = np.random.default_rng(seed)
    eighth = 60 / bpm / 2
    chords = [(38, [57, 62, 65, 69, 76]),      # Dm9
              (43, [62, 67, 70, 74, 76]),      # Gm6
              (34, [62, 65, 69, 72, 77]),      # Bbmaj7
              (33, [64, 67, 69, 73, 76])]      # A7
    patterns = [[0, 2, 1, 3, -1, 2, 4, 1], [3, 1, -1, 2, 0, 2, -1, 4],
                [0, 1, 2, 4, 3, -1, 2, 1], [4, 2, -1, 1, 3, 1, 0, -1]]
    cache = {}
    def pluck(midi, length=0.9, damp=3.0):
        if midi not in cache:
            f = 440 * 2 ** ((midi - 69) / 12); N = max(int(MSR / f), 2); n = int(length * MSR)
            buf, out, pos = rng.uniform(-1, 1, N).astype(np.float32), np.empty(n, np.float32), 0
            while pos < n:
                k = min(N, n - pos); out[pos:pos + k] = buf[:k]
                buf = 0.996 * 0.5 * (buf + np.roll(buf, -1)); pos += k
            cache[midi] = out * np.exp(-np.arange(n) / MSR * damp)
        return cache[midi]
    def bell(midi, length=1.8):
        f = 440 * 2 ** ((midi - 69) / 12); t = np.arange(int(length * MSR)) / MSR
        return ((np.sin(2 * np.pi * f * t) + 0.3 * np.sin(2 * np.pi * 2 * f * t)
                 + 0.1 * np.sin(2 * np.pi * 3.01 * f * t)) * np.exp(-t * 2.5) * 0.35).astype(np.float32)
    out = np.zeros((int((dur + 3) * MSR), 2), np.float32)
    def place(sig, t, gain, pan):
        a = int(t * MSR); seg = out[a:a + len(sig)]
        seg[:, 0] += sig[:len(seg)] * gain * (1 - pan); seg[:, 1] += sig[:len(seg)] * gain * pan
    step = 0
    while step * eighth < dur:
        bar = step // 8
        root, tones = chords[bar % 4]
        idx = patterns[(bar // 4 + bar) % 4][step % 8]
        t = step * eighth + rng.uniform(-0.008, 0.008)                    # tiny human timing variation
        if step % 8 == 0:
            place(pluck(root + 12, 1.6, 1.5), t, 0.55, 0.5)                # bass
        if idx >= 0:
            place(pluck(tones[idx]), t, rng.uniform(0.22, 0.38), rng.uniform(0.25, 0.75))
        if step % 16 == 4 and bar % 2 == 1:
            place(bell(tones[-1] + 12), t, 0.5, rng.uniform(0.3, 0.7))   # music-box sparkle
        step += 1
    raw = WORK / "m_curious_raw.wav"
    sf.write(raw, out[: int(dur * MSR)], MSR)
    ff("-i", raw, "-af", "aecho=0.8:0.5:120|260:0.25|0.15", WORK / "m_curious.wav")
    a, _ = sf.read(str(WORK / "m_curious.wav"), dtype="float32", always_2d=True)
    return a

def rms_norm(a, target):
    return a * (target / max(float(np.sqrt((a ** 2).mean())), 1e-6))

def envelope(key, n_samples, smooth_s=1.5):
    """Per-shot target level for one layer, smoothed so changes swell instead of jumping."""
    rate = 100
    tgt = np.zeros(int(TOTAL * rate) + 1, np.float32)
    for shot, row in zip(SHOTS, TIMELINE):
        lv = {**mood(shot)["music"], **shot.get("music", {})}.get(key, 0.0)
        if key == "curious" and MUSIC_STYLE not in ("curious", "library"):
            lv = 0.0
        tgt[int(row["start"] * rate): int((row["start"] + row["dur"]) * rate) + 1] = lv
    k = int(smooth_s * rate)
    sm = np.convolve(np.pad(tgt, (k, k), mode="edge"), np.ones(k) / k, mode="same")[k:-k]
    return np.interp(np.arange(n_samples) / MSR, np.arange(len(sm)) / rate, sm).astype(np.float32)

D = TOTAL + 1.0
MUSIC_SOURCE, MUSIC_CUES, MUSIC_CREDITS = "generated", [], ""

def lufs(path):
    """Integrated loudness of a file (EBU R128)."""
    r = subprocess.run(["ffmpeg", "-hide_banner", "-nostats", "-i", str(path), "-af", "ebur128", "-f", "null", "-"],
                       capture_output=True, text=True)
    m = re.findall(r"I:\s+(-?\d+\.\d) LUFS", r.stderr)
    return float(m[-1]) if m else None

def fetch_track(title):
    """Licensed track from the library, downloaded once into the model cache (Drive) and reused."""
    import urllib.request
    f = MODEL_CACHE / "music" / f"{title}.mp3"
    f.parent.mkdir(parents=True, exist_ok=True)
    want = musiclib.TRACKS[title]["bytes"]
    if f.exists() and f.stat().st_size == want:
        return f
    t0 = time.time()
    req = urllib.request.Request(musiclib.url(title), headers={"User-Agent": "Mozilla/5.0 (ai-video notebook)"})
    with urllib.request.urlopen(req, timeout=60) as r, open(f.with_suffix(".part"), "wb") as out:
        shutil.copyfileobj(r, out)
    got = f.with_suffix(".part").stat().st_size
    if got < 100_000:
        raise RuntimeError(f"download of '{title}' looks broken ({got} bytes)")
    if got != want:
        log(f"'{title}': {got} bytes, catalogue says {want} (file may have been updated); using it", "WARN")
    f.with_suffix(".part").rename(f)
    log(f"music '{title}' downloaded in {time.time() - t0:.1f}s ({got / 2**20:.1f} MB)")
    return f

def library_music():
    """One licensed track per chapter, 2 s crossfades at chapter changes (never inside a scene)."""
    cues = musiclib.arrange(PLAN, TIMELINE, TOTAL)
    # story-derived restraint: under mostly serious / empathetic narration the music steps back 4 dB (it never
    # gets LOUDER because the voice is excited; the sidechain ducking keeps every word clear either way)
    dirs = globals().get("DIRECTIONS") or {}
    for c in cues:
        inside = [dirs[p["id"]].emotion for p in PLACED if p["id"] in dirs and c["start"] <= p["start"] < c["end"]]
        if inside and sum(e in ("serious", "empathetic") for e in inside) / len(inside) >= 0.5:
            c["gain_db"] = c.get("gain_db", 0.0) - 4.0
            log(f"music: '{c['track']}' {c['start']:.1f}–{c['end']:.1f}s 4 dB lower (serious narration)")
    XF, n = 2.0, int(D * MSR)
    bed = np.zeros((n, 2), np.float32)
    for i, c in enumerate(cues):
        a0, a1 = max(c["start"] - XF / 2, 0.0), min(c["end"] + XF / 2, D)
        seg = WORK / f"m_cue{i}.wav"
        ff("-stream_loop", "-1", "-ss", f"{c.get('offset', 0.0):.2f}", "-i", fetch_track(c["track"]),
           "-t", f"{a1 - a0:.3f}", "-ar", MSR, "-ac", 2, seg)
        a, _ = sf.read(str(seg), dtype="float32", always_2d=True)
        a = a * 10 ** (c.get("gain_db", 0.0) / 20)
        k = min(int(XF * MSR), len(a) // 2)
        if i > 0:
            a[:k] *= np.linspace(0, 1, k)[:, None]
        if i < len(cues) - 1:
            a[-k:] *= np.linspace(1, 0, k)[:, None]
        s0 = int(a0 * MSR); bed[s0:s0 + len(a)] += a[: n - s0]
        print(f"  music {c['start']:6.1f}s–{c['end']:6.1f}s: \"{c['track']}\" ({musiclib.TRACKS[c['track']]['feel']})")
    return bed, cues

user_track = next((p for p in [PROJ / "music.mp3", PROJ / "music.wav"] if p.exists()), None)
mus = None
if not user_track and MUSIC_STYLE == "library":
    try:
        mus, MUSIC_CUES = library_music()
        MUSIC_SOURCE, MUSIC_CREDITS = "library", musiclib.credits(MUSIC_CUES)
    except Exception as e:
        log_exc("music library failed; using the generated score instead")
        print("⚠️ Music library unavailable (details in the log); using the generated score instead.")
if mus is not None:
    pass
elif user_track:
    print("Using your music:", user_track.name, "(make sure you have the rights to it)")
    ff("-stream_loop", "-1", "-i", user_track, "-t", f"{D:.3f}", "-ar", MSR, "-ac", 2, WORK / "user_music.wav")
    mus, _ = sf.read(str(WORK / "user_music.wav"), dtype="float32", always_2d=True)
    n = len(mus)
    level = np.clip(envelope("base", n) + envelope("curious", n) + 0.6 * envelope("tension", n)
                    + 0.4 * envelope("shimmer", n), 0.15, 1.0)
    mus = mus * level[:, None]
else:
    layers = {
        "base": synth(WORK / "m_base.wav",
            ["sine=frequency=55:sample_rate=48000:duration={d}", "sine=frequency=82.41:sample_rate=48000:duration={d}",
             "sine=frequency=110:sample_rate=48000:duration={d}", "anoisesrc=color=brown:sample_rate=48000:duration={d}:amplitude=0.3"],
            "[0]volume=0.5[a];[1]volume=0.3,tremolo=f=0.1:d=0.6[b];[2]volume=0.15,tremolo=f=0.15:d=0.8[c];"
            "[3]lowpass=f=300,volume=0.4[e];[a][b][c][e]amix=inputs=4:normalize=0,"
            "aecho=0.8:0.7:1000|1800:0.3|0.25,pan=stereo|c0=c0|c1=c0", D),
        "tension": synth(WORK / "m_tension.wav",
            ["sine=frequency=41.2:sample_rate=48000:duration={d}", "sine=frequency=58.27:sample_rate=48000:duration={d}",
             "anoisesrc=color=brown:sample_rate=48000:duration={d}:amplitude=0.5"],
            "[0]volume=0.6,tremolo=f=1.6:d=0.9[a];[1]volume=0.25,tremolo=f=1.6:d=0.9[b];"
            "[2]lowpass=f=180,volume=0.5,tremolo=f=1.6:d=0.7[c];[a][b][c]amix=inputs=3:normalize=0,pan=stereo|c0=c0|c1=c0", D),
        "shimmer": synth(WORK / "m_shimmer.wav",
            ["sine=frequency=440:sample_rate=48000:duration={d}", "sine=frequency=659.25:sample_rate=48000:duration={d}",
             "sine=frequency=880:sample_rate=48000:duration={d}"],
            "[0]volume=0.10,tremolo=f=0.2:d=0.6[a];[1]volume=0.07,tremolo=f=0.27:d=0.6[b];[2]volume=0.04,tremolo=f=0.33:d=0.7[c];"
            "[a][b][c]amix=inputs=3:normalize=0,aecho=0.8:0.8:700|1300:0.35|0.25,pan=stereo|c0=c0|c1=c0", D),
        "curious": curious_layer(D),
    }
    targets = {"base": 0.10, "tension": 0.10, "shimmer": 0.06, "curious": 0.12}   # loudness balance between layers
    n = min(len(a) for a in layers.values())
    mus = sum(rms_norm(a[:n], targets[k]) * envelope(k, n)[:, None] for k, a in layers.items())
    print(f"Generated a 4-layer score that follows the moods (style: {MUSIC_STYLE}; royalty-free, made here).")
if MUSIC_SOURCE == "library":
    # level from the VOICE: the bed sits MUSIC_UNDER_VOICE_DB under the voice (ducking adds more while someone talks)
    sf.write(WORK / "music_bed.wav", mus, MSR)
    v, m = lufs(WORK / "narration_full.wav"), lufs(WORK / "music_bed.wav")
    if v is not None and m is not None:
        target = v - MUSIC_UNDER_VOICE_DB - 20 * math.log10(max(MUSIC_GAIN, 1e-3))
        mus = mus * 10 ** ((target - m) / 20)
        log(f"music level: voice {v} LUFS, bed {m} → {target:.1f} LUFS before the {MUSIC_GAIN} gain and ducking")
    peak = float(np.abs(mus).max())
    if peak > 0.98:
        mus = mus * (0.98 / peak)
else:
    mus = mus / max(float(np.abs(mus).max()), 1e-6) * 0.8
fi, fo = int(3 * MSR), int(4 * MSR)
mus[:fi] *= np.linspace(0, 1, fi)[:, None]
mus = mus[: int(TOTAL * MSR)]
mus[-fo:] *= np.linspace(1, 0, min(fo, len(mus)))[:, None]
sf.write(WORK / "music.wav", mus, MSR)
(PROJ / "render" / "music_credits.txt").write_text(MUSIC_CREDITS or "Generated in the notebook (no licence needed).")


In [ ]:
BUDGET.check('sfx')
# ==== 11. Sound effects at story moments (generated here, no licences needed) ====
def make_sfx(kind, dur, out):
    d = f"{dur:.3f}"
    if kind == "boom":
        synth(out, ["anoisesrc=color=brown:sample_rate=48000:duration={d}:amplitude=1", "sine=frequency=45:sample_rate=48000:duration={d}"],
              f"[0]lowpass=f=160,volume=1.2[a];[1]volume=0.8[b];[a][b]amix=inputs=2:normalize=0,"
              f"afade=t=out:st=0.05:d={dur - 0.05:.3f}:curve=exp,pan=stereo|c0=c0|c1=c0", dur)
    elif kind == "rumble":
        synth(out, ["anoisesrc=color=brown:sample_rate=48000:duration={d}:amplitude=0.8"],
              f"[0]lowpass=f=90,volume=1.5,tremolo=f=6:d=0.3,afade=t=in:d={dur / 3:.3f},"
              f"afade=t=out:st={dur * 2 / 3:.3f}:d={dur / 3:.3f},pan=stereo|c0=c0|c1=c0", dur)
    elif kind == "whoosh":
        synth(out, ["anoisesrc=color=pink:sample_rate=48000:duration={d}:amplitude=0.8"],
              f"[0]highpass=f=300,lowpass=f=3500,afade=t=in:d={dur * 0.6:.3f}:curve=exp,"
              f"afade=t=out:st={dur * 0.6:.3f}:d={dur * 0.4:.3f},pan=stereo|c0=c0|c1=c0", dur)
    elif kind == "click":                                                # finger tap on a phone button
        synth(out, ["anoisesrc=color=white:sample_rate=48000:duration={d}:amplitude=0.6"],
              f"[0]highpass=f=2500,lowpass=f=9000,afade=t=out:st=0:d={min(dur, 0.05):.3f}:curve=exp,"
              f"volume=0.8,pan=stereo|c0=c0|c1=c0", dur)
    elif kind == "pop":                                                  # notification / item appearing
        synth(out, ["sine=frequency=880:sample_rate=48000:duration={d}"],
              f"[0]vibrato=f=12:d=0.3,afade=t=in:d=0.01,afade=t=out:st=0.02:d={max(dur - 0.02, 0.04):.3f}:curve=exp,"
              f"volume=0.6,pan=stereo|c0=c0|c1=c0", dur)
    elif kind == "coins":                                                # money arriving
        synth(out, ["sine=frequency=2093:sample_rate=48000:duration={d}", "sine=frequency=2637:sample_rate=48000:duration={d}",
                    "sine=frequency=3136:sample_rate=48000:duration={d}"],
              f"[0]afade=t=out:st=0:d=0.25:curve=exp[a];[1]adelay=90|90,afade=t=out:st=0.09:d=0.25:curve=exp[b];"
              f"[2]adelay=180|180,afade=t=out:st=0.18:d={max(dur - 0.18, 0.1):.3f}:curve=exp[c];"
              f"[a][b][c]amix=inputs=3:normalize=0,volume=0.35,pan=stereo|c0=c0|c1=c0", dur)
    elif kind == "ding":                                                 # app notification
        synth(out, ["sine=frequency=1318.5:sample_rate=48000:duration={d}", "sine=frequency=1975.5:sample_rate=48000:duration={d}"],
              f"[0]volume=0.5,afade=t=out:st=0.0:d={dur:.3f}:curve=exp[a];"
              f"[1]volume=0.35,adelay=110|110,afade=t=out:st=0.11:d={max(dur - 0.11, 0.05):.3f}:curve=exp[b];"
              f"[a][b]amix=inputs=2:normalize=0,pan=stereo|c0=c0|c1=c0", dur)
    elif kind == "rustle":                                               # popcorn / paper bag: soft crackle
        synth(out, ["anoisesrc=color=white:sample_rate=48000:duration={d}:amplitude=0.5"],
              f"[0]highpass=f=1800,lowpass=f=7000,tremolo=f=23:d=0.9,tremolo=f=7:d=0.6,afade=t=in:d=0.04,"
              f"afade=t=out:st={dur * 0.5:.3f}:d={dur * 0.5:.3f},volume=0.7,pan=stereo|c0=c0|c1=c0", dur)
    elif kind == "shimmer":
        synth(out, ["sine=frequency=1318.5:sample_rate=48000:duration={d}", "sine=frequency=1760:sample_rate=48000:duration={d}"],
              f"[0]volume=0.3,tremolo=f=7:d=0.5[a];[1]volume=0.2,tremolo=f=9:d=0.5[b];[a][b]amix=inputs=2:normalize=0,"
              f"aecho=0.8:0.8:400|900:0.3|0.2,afade=t=in:d={dur / 2:.3f},afade=t=out:st={dur / 2:.3f}:d={dur / 2:.3f},"
              f"pan=stereo|c0=c0|c1=c0", dur)
    a, _ = sf.read(str(out), dtype="float32", always_2d=True)
    return a

fx = np.zeros((int(TOTAL * MSR) + MSR, 2), np.float32)
n_fx = 0
for shot, row in zip(SHOTS, TIMELINE):
    for k, cue in enumerate(shot.get("sfx", [])):
        a = make_sfx(cue["type"], float(cue.get("dur", 3.0)), WORK / f"sfx_{shot['shot_id']}_{k}.wav")
        a *= 10 ** ((cue.get("gain_db", -12) + SFX_GAIN_DB) / 20)
        s0 = int((row["start"] + cue.get("at", 0.0)) * MSR)
        seg = fx[s0:s0 + len(a)]; seg += a[:len(seg)]
        n_fx += 1
        print(f"  {shot['shot_id']}: {cue['type']} at {row['start'] + cue.get('at', 0.0):.1f}s")
sf.write(WORK / "sfx_track.wav", fx[: int(TOTAL * MSR)], MSR)
print(f"{n_fx} sound effects placed.")


In [ ]:
BUDGET.check('render')
# ==== 12. Edit: every shot (cached ones are reused) → join → mix voices, music, effects → final.mp4 ====
t_wait = time.time()
for j in EDIT_JOBS:                                   # let background edits finish first
    j.result()
wait_for_diagrams()
if EDIT_JOBS:
    log(f"waited {time.time() - t_wait:.1f}s for {len(EDIT_JOBS)} background edits")
segs, n_video, n_cached = [], 0, 0
for shot, row in zip(SHOTS, TIMELINE):
    out, how, is_video, cached = render_segment(shot, row)
    n_video += is_video; n_cached += cached
    hold = HOLDS.get(shot["shot_id"], 0)
    STATUS[shot["shot_id"]] = f"{how} [{row['mood']}]" + (f"  ⚠️ last frame held {hold:.1f}s" if hold > 0.6 else "")
    segs.append(out)
    print(f"  {shot['shot_id']}: {STATUS[shot['shot_id']]}" + ("  (already edited)" if cached else ""))
N_VIDEO_SHOTS = sum(s.get("recipe", "video") == "video" for s in SHOTS)   # drawn and licensed-image shots aren't AI video
print(f"{n_video}/{N_VIDEO_SHOTS} video shots are real AI video, {len(SHOTS) - N_VIDEO_SHOTS} drawn or licensed-image scenes. "
      f"{n_cached} shots reused from the edit cache.")

lst = WORK / "segments.txt"
lst.write_text("".join(f"file '{p}'\n" for p in segs))
video = WORK / "video_only.mp4"
ff("-f", "concat", "-safe", "0", "-i", lst, "-c", "copy", video)

final = WORK / "final.mp4"
def duck_filter():
    """The music ducking (narration = sidechain control signal); the QC rebuilds the same bed to measure it."""
    return "sidechaincompress=" + ":".join(f"{k}={v}" for k, v in DUCK.items())

AF = "aformat=sample_fmts=fltp:sample_rates=48000:channel_layouts=stereo"   # FFmpeg 4.4 (Colab's Ubuntu 22.04) can't
#                                     negotiate formats for sidechaincompress by itself: state them on every input
fc = (f"[1:a]aresample=48000,pan=stereo|c0=c0|c1=c0,{AF},asplit=2[vo][sc];"
      f"[2:a]aresample=48000,{AF},volume={MUSIC_GAIN}[m];"
      f"[m][sc]{duck_filter()}[duck];"
      f"[3:a]aresample=48000,{AF}[fx];"
      "[vo][duck][fx]amix=inputs=3:duration=first:normalize=0,"
      "loudnorm=I=-14:TP=-1.5:LRA=11,aresample=48000[a]")
def drawn_subtitles(srt_path):
    """Subtitles drawn by Pillow + raqm (the same shaping as the cards) as ONE transparent image stream, for scripts
    the FFmpeg subtitle burner may not shape: libass in the tested FFmpeg 4.4 build detached Telugu vowel signs
    (మ ీరు instead of మీరు). Same place, size, outline and speaker colours as the English subtitles; no italics
    (Telugu has no true italic). Returns the concat file for FFmpeg's image-sequence input."""
    from PIL import ImageDraw
    ts = lambda x: (lambda h, m, r: int(h) * 3600 + int(m) * 60 + float(r.replace(",", ".")))(*x.split(":"))
    blocks = re.findall(r"^\d+\n(\S+) --> (\S+)\n(.*?)(?=\n\n|\Z)", srt_path.read_text(), re.S | re.M)
    d = WORK / "subs_png"; shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    blank = d / "blank.png"; Image.new("RGBA", (W, H), (0, 0, 0, 0)).save(blank)
    f = drawkit.font(round(H * 0.052), bold=True)
    items, t = [], 0.0
    for k, (a, b, txt) in enumerate(blocks):
        a, b = max(ts(a), t), ts(b)
        if b <= a:
            continue
        m = re.search(r'color="(#[0-9A-Fa-f]{6})"', txt)
        plain = re.sub(r"<[^>]+>", "", txt).strip()
        im = Image.new("RGBA", (W, H), (0, 0, 0, 0))
        ImageDraw.Draw(im).text((W / 2, H - H * 30 / 288), plain, font=f, fill=m.group(1) if m else "#FFFFFF",
                                anchor="ms", stroke_width=max(2, H // 270), stroke_fill="black")
        png = d / f"{k:04d}.png"; im.save(png)
        if a > t:
            items.append((blank, a - t))
        items.append((png, b - a)); t = b
    items += [(blank, 1.0), (blank, None)]          # the concat demuxer ignores the last entry's duration
    lst = d / "subs.txt"
    lst.write_text("".join(f"file '{p}'\n" + (f"duration {du:.3f}\n" if du is not None else "") for p, du in items))
    log(f"drawn subtitles: {len(blocks)} cues, font {SUB_FONT}")
    return lst

extra_in = []
if BURN_SUBS and LANGUAGE != "en":                    # non-Latin script: draw the subtitles ourselves (shaped)
    extra_in = ["-f", "concat", "-safe", 0, "-i", drawn_subtitles(WORK / "subtitles.srt")]
    fc += ";[0:v][4:v]overlay=0:0:eof_action=pass:format=auto[v]"
    vmap, vcodec = "[v]", VENC
elif BURN_SUBS and SUB_STYLE == "karaoke":           # word-highlighted ASS (libass), one pass in the final encode
    fc += f";[0:v]ass={WORK / 'subtitles.ass'}" + (f":fontsdir={FONTS_DIR}" if FONTS_DIR else "") + "[v]"
    vmap, vcodec = "[v]", VENC
elif BURN_SUBS:
    fc += (f";[0:v]subtitles={WORK / 'subtitles.srt'}" + (f":fontsdir={FONTS_DIR}" if FONTS_DIR else "")
           + f":force_style='FontName={SUB_FONT},FontSize=18,Outline=1,Shadow=0,MarginV=30'[v]")
    vmap, vcodec = "[v]", VENC
else:
    vmap, vcodec = "0:v", ["-c:v", "copy"]
ff("-i", video, "-i", WORK / "narration_full.wav", "-i", WORK / "music.wav", "-i", WORK / "sfx_track.wav", *extra_in,
   "-filter_complex", fc, "-map", vmap, "-map", "[a]", *vcodec, "-c:a", "aac", "-b:a", "192k",
   "-movflags", "+faststart", "-shortest", final)
shutil.copy(final, PROJ / "render" / "final.mp4")
print("✅ Saved:", PROJ / "render" / "final.mp4")


In [ ]:
BUDGET.check('qc')
# ==== 13. Quality checks ====
final = PROJ / "render" / "final.mp4"
qc = {"shots": STATUS, "ai_video_shots": f"{n_video}/{N_VIDEO_SHOTS}", "expected_s": round(TOTAL, 2), "story_arc": [r["mood"] for r in TIMELINE]}
side = {p["id"]: read_sidecar(PROJ / "narration" / f"{p['id']}.wav") or {} for p in PLACED}
qc["speech_check"] = {k: v.get("wer") for k, v in side.items()}
qc["lines_to_listen_to"] = FLAGGED
qc["frozen_last_frame_s"] = {k: v for k, v in HOLDS.items() if v > 0}
qc["continuity"] = {s["shot_id"]: (read_sidecar(PROJ / "clips" / s["shot_id"]) or {}).get("continuity")
                    for s in SHOTS if s.get("must_show")}
qc["continuity_warnings"] = {sid: [m for c in (v or []) for m in c["missing"]]
                             for sid, v in qc["continuity"].items() if any(c["missing"] for c in (v or []))}
_iss = globals().get("ISSUES", [])
qc["validation"] = {"errors": len([i for i in _iss if i["level"] == "error"]),
                    "warnings": [f"[{i['rule']}] {i['shot'] or 'plan'}: {i['message']}" for i in _iss if i["level"] == "warn"]}
qc["review"] = dict(REVIEW, **{sid: "still missing after a remake: " + ", ".join(m)
                               for sid, m in qc.get("continuity_warnings", {}).items() if sid not in REVIEW})
REVIEW_DIR = PROJ / "render" / "review"; REVIEW_DIR.mkdir(parents=True, exist_ok=True)
for _sid in qc["review"]:
    _sc = read_sidecar(PROJ / "clips" / _sid) or {}
    if not _sc.get("parts"):
        continue
    try:
        _n = max(int(_sc.get("frames_per_part", 81)) // 4, 1)
        ff("-i", PROJ / "clips" / _sc["parts"][0], "-vf", f"select=not(mod(n\\,{_n})),scale=480:-2,tile=4x1",
           "-frames:v", 1, "-q:v", 3, REVIEW_DIR / f"{_sid}.jpg")
        _scores = "; ".join(f"{k}: seen in {v['seen']:.0%} of frames" for c in _sc.get("continuity", [])
                            for k, v in (c.get("presence") or {}).items())
        qc["review"][_sid] += f" | picture: render/review/{_sid}.jpg | scores: {_scores}"
    except Exception:
        log_exc(f"{_sid}: could not make the review picture")
if qc["review"]:
    print("Shots to look at before publishing (pictures in render/review/):")
    for _sid, _why in qc["review"].items():
        print(f"  {_sid}: {_why}")
qc["music"] = {"source": MUSIC_SOURCE, "cues": MUSIC_CUES, "credits": MUSIC_CREDITS}
# logical consistency: per scene-state shot, the decision and the evidence
_logic = {}
for s in SHOTS:
    if "logic_shot" in globals() and logic_shot(s):
        sc = read_sidecar(PROJ / "clips" / s["shot_id"]) or {}
        ok, why = logic_usable(s)
        _logic[s["shot_id"]] = {"used_in_video": ok, "why": why, "decision": sc.get("decision"),
                                "failed": sc.get("failed", []), "uncertain": sc.get("uncertain", []),
                                "repairs": sc.get("repairs", []), "attempts": len(sc.get("attempts", []))}
qc["logic"] = _logic
qc["not_logically_verified"] = [s["shot_id"] for s in SHOTS if s.get("recipe", "video") == "video"
                                and not ("logic_shot" in globals() and logic_shot(s))]
qc["subtitles_word_timed"] = f"{sum(bool(p.get('words')) for p in PLACED)}/{len(PLACED)}"
# ---- measurements of the rendered file (qclib: ffprobe + FFmpeg detectors) ----
TH = dict(qclib.THRESHOLDS, loudness_target_lufs=-14.0)
C = qclib.check
checks = {}
try:
    pr = qclib.probe(final)
    v, a = pr["video"] or {}, pr["audio"] or {}
    qc["probe"] = pr
    ok = (v.get("codec") == "h264" and v.get("pix_fmt") == "yuv420p" and a.get("codec") == "aac"
          and (v.get("width"), v.get("height")) == (W, H) and abs(v.get("fps", 0) - FPS) < 0.01)
    checks["playable H.264/AAC stream, right size and fps"] = C("PASS" if ok else "FAIL", evidence={"video": v, "audio": a},
        fix=f"expected h264 yuv420p {W}x{H} @ {FPS} fps + AAC")
    drift = abs(pr["duration"] - TOTAL)
    checks["no duration drift (video vs timeline)"] = C("PASS" if drift <= TH["duration_drift_s"] else "FAIL",
        evidence={"file_s": round(pr["duration"], 2), "timeline_s": round(TOTAL, 2)}, fix="re-run the render cell")
    cut = (v.get("duration", 0) - a.get("duration", 0)) > 0.3
    last_line_end = max((pp["end"] for pp in PLACED), default=0.0)
    checks["ending not cut off (audio covers the video, last line ends in time)"] = C(
        "FAIL" if cut or last_line_end > pr["duration"] - 0.2 else "PASS",
        evidence={"audio_s": a.get("duration"), "video_s": v.get("duration"), "last_line_end_s": round(last_line_end, 2)})
except Exception as e:
    log_exc("ffprobe of the final video failed")
    checks["playable H.264/AAC stream, right size and fps"] = C("UNKNOWN", evidence=repr(e)[:200], fix="check FFmpeg")
try:
    ev = qclib.video_events(final, TH)
    qc["black_segments"], qc["freezes"] = ev["black"], ev["freezes"]
    bad_black = [(x, y) for x, y in ev["black"] if not (x < 0.7 or y > TOTAL - 1.0)]   # fades at start/end are planned
    checks["no unplanned black frames"] = C("FAIL" if bad_black else "PASS", evidence=bad_black)
    # freezes only matter inside AI-video shots: drawn cards and stills hold still on purpose
    frozen = [r["shot_id"] for r, sh in zip(TIMELINE, SHOTS) if sh.get("recipe", "video") == "video"
              and qclib.overlaps(ev["freezes"], r["start"], r["start"] + r["dur"], 0.6)]
    checks["no frozen frames inside AI-video shots"] = C("FAIL" if frozen else "PASS", shots=frozen,
        evidence=ev["freezes"], fix="regenerate those clips or use their fallback")
except Exception as e:
    log_exc("video event detection failed")
    checks["no unplanned black frames"] = C("UNKNOWN", evidence=repr(e)[:200])
try:
    au = qclib.audio_stats(final, TH)
    qc["audio"] = au; qc["loudness_lufs"] = au["integrated_lufs"]
    lu = au["integrated_lufs"]
    checks["loudness on target"] = C("UNKNOWN" if lu is None else
        "PASS" if abs(lu - TH["loudness_target_lufs"]) <= TH["loudness_tolerance_lu"] else "FAIL",
        evidence={"integrated_lufs": lu, "target": TH["loudness_target_lufs"]})
    tp = au["true_peak_dbtp"]
    checks["no clipping (true peak)"] = C("UNKNOWN" if tp is None else "PASS" if tp <= TH["true_peak_max_dbtp"] else "FAIL",
                                          evidence={"true_peak_dbtp": tp, "max": TH["true_peak_max_dbtp"]})
    checks["no unintended silence"] = C("FAIL" if au["silences"] else "PASS", evidence=au["silences"],
                                        fix="a gap in narration and music: check the timeline and music cues")
except Exception as e:
    log_exc("audio analysis failed")
    checks["loudness on target"] = C("UNKNOWN", evidence=repr(e)[:200])
try:                                                  # the ducked music bed, rebuilt with the same filter, measured
    duck_wav = WORK / "qc_music_ducked.wav"
    ff("-i", WORK / "narration_full.wav", "-i", WORK / "music.wav", "-filter_complex",
       f"[0:a]aresample=48000,pan=stereo|c0=c0|c1=c0,{AF}[sc];[1:a]aresample=48000,{AF},volume={MUSIC_GAIN}[m];"
       f"[m][sc]{duck_filter()}[d]", "-map", "[d]", duck_wav)
    dk = qclib.ducking(WORK / "narration_full.wav", duck_wav)
    qc["ducking"] = dk
    checks["music ducks under the voice"] = C("UNKNOWN" if dk["reduction_db"] is None else
        "PASS" if dk["reduction_db"] >= TH["duck_min_db"] else "FAIL", evidence=dk,
        fix="raise DUCK ratio or lower DUCK threshold")
except Exception as e:
    log_exc("ducking check failed")
    checks["music ducks under the voice"] = C("UNKNOWN", evidence=repr(e)[:200])
try:
    sheet = qclib.contact_sheet(final, PROJ / "render" / "contact_sheet.jpg")
    qc["contact_sheet"] = str(sheet)
except Exception:
    log_exc("contact sheet failed")

# ---- checks from the pipeline's own records ----
checks["narration complete and heard correctly"] = C("PASS" if not FLAGGED and all(
    v is not None for v in qc["speech_check"].values()) else "FAIL" if FLAGGED else "UNKNOWN",
    shots=FLAGGED, evidence=qc["speech_check"], fix="listen to these lines; re-run the narration cell")
checks["every video shot is verified AI video (no unchecked stand-ins)"] = C(
    "PASS" if n_video == N_VIDEO_SHOTS else "FAIL", evidence=STATUS,
    shots=[s["shot_id"] for s in SHOTS if s.get("recipe", "video") == "video" and "AI video" not in STATUS.get(s["shot_id"], "")])
checks["no frozen last frames (clip shorter than its shot)"] = C(
    "PASS" if all(v <= 0.6 for v in HOLDS.values()) else "FAIL", shots=[k for k, v in HOLDS.items() if v > 0.6], evidence=HOLDS)
checks["characters stay on screen"] = C("PASS" if not qc.get("continuity_warnings") else "FAIL",
                                       shots=list(qc.get("continuity_warnings", {})), evidence=qc.get("continuity_warnings"))
checks["music licensed and credited"] = C("PASS" if MUSIC_SOURCE != "library" or bool(MUSIC_CREDITS) else "FAIL",
                                          evidence=MUSIC_CREDITS)
checks["nothing flagged for review"] = C("PASS" if not qc.get("review") else "FAIL", shots=list(qc.get("review", {})),
                                         evidence=qc.get("review"), fix="look at render/review/ and approve or fix")
checks["every scene-state shot verified or approved"] = C(
    "PASS" if all(x["used_in_video"] for x in qc.get("logic", {}).values()) else "FAIL",
    shots=[k for k, x in qc.get("logic", {}).items() if not x["used_in_video"]], evidence=qc.get("logic"))
checks["plan validation: no errors"] = C("PASS" if not qc["validation"]["errors"] else "FAIL", evidence=qc["validation"])
checks["AI shots without logical verification (information)"] = C(
    "PASS" if not qc["not_logically_verified"] else "UNKNOWN", mandatory=False, shots=qc["not_logically_verified"])
if globals().get("FACTCHECK") is not None:            # topic videos: every narration sentence vs its sources
    bad = {k: x for k, x in FACTCHECK.items() if x["status"] != "PASS"}
    checks["every narrated fact supported by its sources"] = C("PASS" if not bad else "FAIL", shots=sorted(bad),
        evidence=bad, fix="rewrite or remove these sentences (render/factcheck.json)")
if globals().get("ASSET_MANIFEST") is not None:      # licensed images: allowed licence + provenance recorded
    badl = [a["id"] for a in ASSET_MANIFEST if not a.get("license_ok")]
    checks["every image has an allowed licence and its credit"] = C("PASS" if not badl else "FAIL", shots=badl,
        evidence={a["id"]: a.get("license") for a in ASSET_MANIFEST})
if PLAN.get("story_mode") == "two_friends_cinematic":  # ---- storytelling checks (two-friend videos) ----
    MEDIUM_OF = {"video": "ai_video", "stock": "licensed_media", "card": "presentation_graphic", "ui": "presentation_graphic",
                 "diagram": "presentation_graphic", "journey": "animated_demo", "character": "character_scene",
                 "demo": "animated_demo"}
    _rows = []
    for s, r in zip(SHOTS, TIMELINE):
        if s.get("recipe") == "story":
            _rows += STORY_ROWS.get(s["shot_id"], [])
        else:
            _rows.append({"shot": s["shot_id"], "visual": s["shot_id"], "intent": s.get("intent", s.get("recipe", "video")),
                          "medium": MEDIUM_OF.get(s.get("recipe", "video"), "?"), "start": r["start"], "dur": r["dur"],
                          "label": s.get("label", ""), "description": describe(s)})
    stats = story.statistics(_rows)
    qc["story_statistics"] = stats
    (PROJ / "render" / "story_stats.json").write_text(json.dumps({"statistics": stats, "visuals": _rows}, indent=1))
    _ov = [(a["id"], b["id"]) for a, b in zip(PLACED, PLACED[1:]) if b["start"] < a["end"] - 0.02]
    _order = [p_["id"] for p_ in PLACED] == [l["id"] for s in SHOTS for l in s["lines"]]
    checks["dialogue: turns in script order, no two friends talking at once"] = C(
        "PASS" if not _ov and _order else "FAIL", evidence={"overlaps": _ov, "in_order": _order})
    _fact = [l for s in SHOTS for l in s["lines"] if l.get("kind") in story.FACTUAL_KINDS]
    _claims = {c["id"]: c for c in PLAN.get("claims", [])}
    _ungrounded = [l["id"] for l in _fact if not l.get("claims") or not all(
        any(str(u).startswith("https://") for u in _claims.get(c, {}).get("sources", [])) for c in l["claims"])]
    checks["every factual dialogue line cites a sourced fact"] = C("PASS" if not _ungrounded else "FAIL",
        shots=_ungrounded, evidence={"factual_lines": len(_fact)}, fix="cite a claim with a source URL or rephrase")
    checks["presentation graphics at most 20% of screen time"] = C(
        "PASS" if stats["presentation_graphics_pct"] <= 20 else "FAIL", evidence=stats["rendering_medium_pct"],
        fix="replace cards/diagrams with the friends, objects or a demonstration")
    _unl = [f"{r['shot']}/{r['visual']}" for r in _rows if r["intent"] in ("reenactment", "real_event")
            and r["medium"] in ("character_scene", "ai_video") and r.get("label") != "Illustration"]
    checks["re-enactments are labelled 'Illustration' (never shown as real footage)"] = C(
        "PASS" if not _unl else "FAIL", shots=_unl)
    _beats = [s.get("story_beat") for s in SHOTS if s.get("story_beat")]
    _miss = [b for b in ("question", "closing") if b not in _beats]
    _answered = any(b in _beats for b in ("first_answer", "demonstration", "correction", "reveal"))
    checks["story complete: question, answer, closing"] = C("PASS" if not _miss and _answered else "FAIL",
        evidence={"beats": _beats, "missing": _miss, "answered": _answered})
    _long = [f"{r['shot']}/{r['visual']} {r['dur']}s" for r in _rows if r["dur"] > 12]
    checks["visual variety: no single visual longer than 12 s (information)"] = C(
        "PASS" if not _long else "UNKNOWN", mandatory=False, shots=_long)
    checks["holistic review by a person: contact sheet + dialogue/shot timeline (information)"] = C(
        "UNKNOWN", mandatory=False, evidence="render/contact_sheet.jpg, render/dialogue_shot_timeline.md")
    _cl = []
    for p_ in PLACED:
        _m = ((side.get(p_["id"]) or {}).get("voice_qc") or {}).get("metrics", {})
        _cl.append({"id": p_["id"], "speaker": p_["speaker"], "start": p_["start"], "end": p_["end"],
                    "rms_db": _m.get("rms_db"), "sps": _m.get("speaking_rate_sps")})
    _cd = voice_qc.conversation_diagnostics(_cl)
    qc["conversation_timing"] = _cd
    checks["conversation timing: no dead air, overlaps, mechanical gaps or level jumps (diagnostic)"] = C(
        "PASS" if not _cd["issues"] else "UNKNOWN", mandatory=False, evidence=_cd,
        fix="listen to these places; a person decides whether it sounds like a real conversation")
    _md = [f"# Dialogue and shot timeline: {PLAN['title']}", "", "## Audio timeline (dialogue turns)", "",
           "| Start | End | Speaker | Line | Kind | Sources |", "|---|---|---|---|---|---|"]
    _lk = {l["id"]: l for s in SHOTS for l in s["lines"]}
    for p_ in PLACED:
        l = _lk[p_["id"]]
        _md.append(f"| {p_['start']:.2f} | {p_['end']:.2f} | {CHARS[p_['speaker']]['name']} | {p_['text']} | "
                   f"{l.get('kind', '')} | {', '.join(l.get('claims', []))} |")
    _md += ["", "## Visual timeline (independent of the turns)", "",
            "| Start | Length | Scene / visual | Purpose | Medium | What is seen |", "|---|---|---|---|---|---|"]
    for r in _rows:
        _md.append(f"| {r['start']:.2f} | {r['dur']:.2f} | {r['shot']}/{r['visual']} | {r['intent']} | {r['medium']} | "
                   f"{r.get('description', '')}{(' (' + r['label'] + ')') if r.get('label') else ''} |")
    _md += ["", "## Statistics (screen time)", "", "**Narrative purpose:** " + ", ".join(
        f"{k} {v}%" for k, v in stats["narrative_purpose_pct"].items()), "", "**Rendering medium:** " + ", ".join(
        f"{k} {v}%" for k, v in stats["rendering_medium_pct"].items())]
    (PROJ / "render" / "dialogue_shot_timeline.md").write_text("\n".join(_md) + "\n")
    print("Story statistics — purpose: " + ", ".join(f"{k} {v}%" for k, v in stats["narrative_purpose_pct"].items())
          + "\n                   medium:  " + ", ".join(f"{k} {v}%" for k, v in stats["rendering_medium_pct"].items()))
if globals().get("MAX_DURATION"):                     # short-form limit (timeline cell), measured on the real file
    _dur = (qc.get("probe") or {}).get("duration") or TOTAL
    checks[f"video no longer than {MAX_DURATION} s"] = C(
        "PASS" if _dur <= MAX_DURATION + 0.5 else "FAIL", evidence={"duration_s": round(_dur, 2)},
        fix="shorten the script with a fact-preserving revision (never speed up the voices or cut the ending)")
if globals().get("STORYBOOK"):                        # illustrated characters + environment packs
    _sr = globals().get("STORYBOOK_REPORT", {})
    _bad = {k: v["verify"] for k, v in _sr.items() if v.get("verify")}
    checks["storybook assets: exact versions present, every layer unchanged"] = C(
        "PASS" if _sr and not _bad else "FAIL", evidence=_sr, fix="restore the asset version folder on Drive")
    _prov = sorted(k for k, v in _sr.items() if not k.startswith("env:") and v.get("status") != "approved")
    checks["character artwork approved by a person (not provisional)"] = C(
        "PASS" if _sr and not _prov else "UNKNOWN", shots=_prov,
        evidence={k: v.get("status") for k, v in _sr.items() if not k.startswith("env:")},
        fix="MANUAL_REVIEW_REQUIRED: approve a Character Studio version (colab/approve_character.py), then pin it")
    _mo = globals().get("STORYBOOK_MOUTH", {})
    _silent_open = {sid: {c: r["open_in_silence"] for c, r in v.items() if r["open_in_silence"]} for sid, v in _mo.items()}
    _silent_open = {k: v for k, v in _silent_open.items() if v}
    _sp = sum(r["speech_frames"] for v in _mo.values() for r in v.values())
    _op = sum(r["open_in_speech"] for v in _mo.values() for r in v.values())
    checks["lip sync: only the speaker's mouth moves, and it moves while speaking"] = C(
        "UNKNOWN" if not _mo else "FAIL" if _silent_open or (_sp and _op / _sp < 0.3) else "PASS",
        shots=list(_silent_open), evidence={"speech_frames": _sp, "open_mouth_in_speech": _op, "open_in_silence": _silent_open},
        fix="check the viseme track (storybook.visemes_from_audio) for these scenes")
if PLAN.get("localization"):                          # Telugu lines vs their verified English claims
    _loc = PLAN["localization"]
    _rev = [k for k, r in _loc.items() if r.get("status") != "PASS"]
    checks["Telugu factual lines keep the verified claims (numbers, names, key terms)"] = C(
        "PASS" if not _rev else "UNKNOWN", shots=_rev, evidence=_loc,
        fix="MANUAL_REVIEW_REQUIRED: a Telugu speaker compares these lines with their source_text")
_gr = globals().get("GEMINI_REPORT")
if _gr:                                                # the hosted free-tier TTS exception, if it was used
    qc["gemini_tts"] = {k: v for k, v in _gr.items() if k != "lines"}
    _used = _gr.get("status") == "PASS"
    checks["Gemini free-tier TTS: approved model, within the local budget (information)"] = C(
        "PASS" if (not _used or (GEMINI_TTS_MODEL in gemini_tts.FREE_TTS_MODELS and
                                 _gr.get("calls", 0) <= MAX_GEMINI_TTS_CALLS_PER_VIDEO)) else "FAIL",
        mandatory=False, evidence={"used": _used, "calls": _gr.get("calls"), "why_not": _gr.get("why")})
    if _used:
        _mis = (_gr.get("attribution") or {}).get("misattributed", [])
        checks["Gemini voices: every turn split, attributed to the right friend and heard correctly"] = C(
            "PASS" if not _mis and all(c.get("align", "PASS") == "PASS" or c.get("mode") == "per-turn"
                                       for c in _gr.get("chunks", [])) else "FAIL",
            evidence={"chunks": _gr.get("chunks"), "attribution": _gr.get("attribution")})
qc["devices"] = dict(DEVICE_REPORT)                   # where each step actually ran, and the whole-run peaks
qc["peaks"] = dict(RUN_PEAKS)
print("\nWhere each step ran:")
for k_, v_ in DEVICE_REPORT.items():
    print(f"  {k_}: {v_}")
print(f"  peak system RAM {RUN_PEAKS['ram_gb']} GB | peak GPU memory {RUN_PEAKS['gpu_gb']} GB (all processes)")
checks['Speaker identity (manual listening needed)'] = C(
    'UNKNOWN', mandatory=False, evidence='ECAPA and its extra retakes are omitted from the free-tier preset.')
if PLAN.get('story_revision') == 'three_d_friends_te_v5':
    checks['Revised Telugu script and pronunciation reviewed'] = C(
        'UNKNOWN', evidence='New v5 dialogue needs fluent-speaker listening and factual review; old approvals do not apply.')
qc["checks"] = checks
qc["approved"] = qclib.approved(checks)
qc["thresholds"] = TH
(PROJ / "render" / "qc_report.json").write_text(json.dumps(qc, indent=2, default=str))
log("qc: " + json.dumps({k: c["status"] for k, c in checks.items()}))
print("\nFinal checks:")
for k, c in checks.items():
    mark = {"PASS": "✅", "FAIL": "❌", "UNKNOWN": "❔"}[c["status"]]
    print(f"  {mark} {k}" + ("" if c["mandatory"] else " (info)") + (f"  → {', '.join(c['shots'])}" if c["shots"] else ""))
print(f"\n{'APPROVED ✅ every mandatory check passed' if qc['approved'] else 'NOT APPROVED ⚠️ fix or review the ❌/❔ items'}"
      f" | report: render/qc_report.json | contact sheet: render/contact_sheet.jpg")


In [ ]:
BUDGET.check('metadata')
# ==== 14. YouTube text (paste these into YouTube Studio) ====
md_ = PLAN["metadata"]
voices = ", ".join(f"{c['name']} ({c['voice']})" for c in CHARS.values())
sources = sorted({u for c in PLAN.get("claims", []) for u in c["sources"]})
desc = (md_["summary"] + "\n\n"
        + ("Sources:\n" + "\n".join(sources) + "\n\n" if sources else "")
        + (md_["disclaimer"] + "\n" if md_.get("disclaimer") else "")
        + f"Voices: Indic Parler-TTS stock Telugu speakers: {voices}.\n"
        "About this video: the voices are AI-generated (synthetic)" + ("" if all(sh.get("recipe") in ("card", "stock", "ui", "journey", "diagram") for sh in SHOTS) else " and so is the animation") + ". "
        "No real people or voices were imitated.")
if md_.get("image_credits"):                         # CC BY images: title, author, source, licence (TASL)
    desc += "\n\nImages (Wikimedia Commons):\n" + "\n".join(md_["image_credits"])
if MUSIC_CREDITS:                                    # CC BY 4.0 requires this credit (incompetech's format)
    desc += "\n\nMusic:\n" + MUSIC_CREDITS
meta = {"title": md_["title"], "description": desc, "tags": md_["tags"],
        "category": md_.get("category", "Film & Animation"), "visibility": "Private",
        "altered_or_synthetic_content": "Yes",
        "made_for_kids": md_.get("audience", "Decide honestly in YouTube Studio")}
(PROJ / "render" / "metadata.json").write_text(json.dumps(meta, indent=2))
print("TITLE:\n" + meta["title"] + "\n\nDESCRIPTION:\n" + desc + "\n\nTAGS:\n" + ", ".join(meta["tags"]))
print(f"\nUpload as PRIVATE, answer YES to 'Altered or synthetic content'. Audience: {meta['made_for_kids']}.")


In [ ]:
BUDGET.check('preview')
# ==== 15. Preview (small copy, so the notebook stays light) ====
from IPython.display import Video, display
prev = WORK / "preview.mp4"
ff("-i", PROJ / "render" / "final.mp4", "-vf", "scale=854:-2", "-c:v", "libx264", "-crf", "28",
   "-preset", "veryfast", "-c:a", "aac", "-b:a", "96k", prev)
display(Video(str(prev), embed=True, width=720))
